In [ ]:
# ============================================================
# PHASE 17.07A — RESEARCH EVIDENCE INVENTORY
# New notebook: Delhi_AQI_Research_Paper_Completion_Phase_17_07
#
# Goal:
#   1. Reconnect Google Drive safely.
#   2. Verify frozen models and feature contract.
#   3. Inventory existing research artifacts.
#   4. Identify evidence gaps before new experiments.
#
# Restrictions:
#   No model retraining.
#   No final-holdout data reads.
#   No modification of frozen deployment artifacts.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd

# ------------------------------------------------------------
# 1. Locate the project or mount Google Drive safely
# ------------------------------------------------------------

ROOT_CANDIDATES = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive2/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive3/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/google_drive/MyDrive/Delhi-AQI-Prediction"),
]

def valid_project_root(root):
    package = root / "deployment/phase16_33_frozen_model_package"
    required = [
        package / "config/feature_contract.json",
        package / "models/phase15_10_final_tuned_extra_trees_trainval.joblib",
        package / "models/phase15_10_final_tuned_random_forest_trainval.joblib",
    ]
    return all(path.is_file() for path in required)

ROOT = next(
    (path for path in ROOT_CANDIDATES if valid_project_root(path)),
    None,
)

if ROOT is None:
    from google.colab import drive

    mount_points = [
        Path("/content/gdrive"),
        Path("/content/drive2"),
        Path("/content/drive3"),
        Path("/content/google_drive"),
        Path("/content/drive_mount"),
        Path("/content/drive"),
    ]

    for mount in mount_points:
        # Never remount over a non-empty folder or delete existing files.
        if mount.is_symlink():
            continue

        if mount.exists():
            if not mount.is_dir():
                continue
            try:
                if any(mount.iterdir()):
                    continue
            except OSError:
                continue
        else:
            mount.mkdir(parents=True, exist_ok=True)

        try:
            print(f"Attempting Google Drive mount at {mount} ...")
            drive.mount(str(mount))
        except Exception as exc:
            print(f"Mount attempt failed: {type(exc).__name__}")
            continue

        ROOT = next(
            (path for path in ROOT_CANDIDATES if valid_project_root(path)),
            None,
        )
        if ROOT is not None:
            break

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate the Delhi AQI project and its frozen model files. "
        "Check the Drive mount before continuing."
    )

DEPLOY = ROOT / "deployment"
PACKAGE = DEPLOY / "phase16_33_frozen_model_package"
CONFIG = PACKAGE / "config"
DATA = ROOT / "data"
PROCESSED = DATA / "processed"

OUT = DEPLOY / "tests/phase17_07A_research_evidence_audit"
OUT.mkdir(parents=True, exist_ok=True)

INVENTORY_CSV = OUT / "phase17_07A_artifact_inventory.csv"
AUDIT_JSON = OUT / "phase17_07A_research_gap_audit.json"

print("=" * 72)
print("PHASE 17.07A — RESEARCH EVIDENCE INVENTORY")
print("=" * 72)
print(f"Project root: {ROOT}")

# ------------------------------------------------------------
# 2. Verify frozen feature contract and model hashes
# ------------------------------------------------------------

EXPECTED_HASHES = {
    "extra_trees":
        "f0b3d43a56a7d5e4f08554f37ff234e1b5e012b5d5d69f18c750844b36cbe72d",
    "random_forest":
        "538b5ff34152d19221518e831528093b7bcc57e2b05f82669718f540b514abd8",
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

FEATURE_FILE = CONFIG / "feature_contract.json"

feature_contract = read_json(FEATURE_FILE)
feature_order = feature_contract.get("feature_order", [])

if len(feature_order) != 73 or len(set(feature_order)) != 73:
    raise RuntimeError("Frozen feature contract must contain 73 unique features.")

model_paths = {
    "extra_trees":
        PACKAGE / "models/phase15_10_final_tuned_extra_trees_trainval.joblib",
    "random_forest":
        PACKAGE / "models/phase15_10_final_tuned_random_forest_trainval.joblib",
}

model_hashes = {
    name: sha256_file(path)
    for name, path in model_paths.items()
}

if model_hashes != EXPECTED_HASHES:
    raise RuntimeError(
        "Frozen model SHA-256 mismatch. Stop and investigate before proceeding."
    )

print("Frozen feature count:", len(feature_order))
print("Frozen model hashes verified: True")
print("StationId feature position:", feature_order.index("StationId") + 1)

# ------------------------------------------------------------
# 3. Inventory artifact paths and sizes
#    Do not load data or prediction rows.
# ------------------------------------------------------------

SEARCH_ROOTS = [
    ROOT / "notebooks",
    ROOT / "outputs",
    DEPLOY,
]

# Include processed-data file names as metadata only, one level deep.
artifact_paths = []

for folder in SEARCH_ROOTS:
    if not folder.exists():
        continue

    for path in folder.rglob("*"):
        if not path.is_file():
            continue

        if any(
            part in {
                ".git", "__pycache__", ".ipynb_checkpoints",
                ".venv", "venv", "node_modules",
                "bundle", "verification_extract",
            }
            for part in path.parts
        ):
            continue

        if path.suffix.lower() in {
            ".ipynb", ".json", ".csv", ".parquet",
            ".xlsx", ".md", ".txt", ".joblib",
            ".pkl", ".pdf", ".png", ".jpg", ".zip",
        }:
            artifact_paths.append(path)

if PROCESSED.exists():
    for path in PROCESSED.iterdir():
        if path.is_file() and path.suffix.lower() in {
            ".parquet", ".csv", ".json", ".pkl",
        }:
            artifact_paths.append(path)

artifact_paths = sorted(set(artifact_paths))

inventory_rows = []

for path in artifact_paths:
    try:
        size_bytes = path.stat().st_size
        relative_path = str(path.relative_to(ROOT))
    except (OSError, ValueError):
        continue

    inventory_rows.append({
        "relative_path": relative_path,
        "filename": path.name,
        "extension": path.suffix.lower(),
        "size_bytes": size_bytes,
    })

inventory_df = pd.DataFrame(inventory_rows)

if inventory_df.empty:
    inventory_df = pd.DataFrame(
        columns=["relative_path", "filename", "extension", "size_bytes"]
    )
else:
    inventory_df = (
        inventory_df.drop_duplicates("relative_path")
        .sort_values("relative_path")
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 4. Categorize possible evidence for each paper requirement
# ------------------------------------------------------------

TOPICS = {
    "chronological_model_comparison": [
        "competition", "comparison", "metric", "performance", "phase15_",
    ],
    "validation_tuning_ensemble": [
        "tuning", "tuned", "validation", "ensemble", "weight",
    ],
    "holdout_and_persistence": [
        "holdout", "persistence", "baseline", "phase15_12", "prediction",
    ],
    "incremental_value_and_ablation": [
        "ablation", "feature_reduction", "top_7", "top_10", "top_20",
        "aqi_only", "persistence",
    ],
    "bootstrap_uncertainty_intervals": [
        "bootstrap", "confidence_interval", "uncertainty",
        "prediction_interval", "coverage_interval",
    ],
    "error_regimes_transitions": [
        "stratification", "regime", "station", "monthly", "season",
        "jump", "transition", "lockdown",
    ],
    "feature_attribution_and_shap": [
        "importance", "permutation", "shap", "xai", "explain",
    ],
    "deep_learning_alternatives": [
        "lstm", "cnn", "timegan", "vmd", "deep_learning",
    ],
    "deployment_and_reproducibility": [
        "phase17_06", "deployment", "inference", "release", "smoke",
    ],
}

topic_matches = {}

for topic, keywords in TOPICS.items():
    matched = inventory_df[
        inventory_df["relative_path"].str.lower().apply(
            lambda name: any(keyword in name for keyword in keywords)
        )
    ]

    topic_matches[topic] = matched["relative_path"].tolist()

# Index the top-level keys of small JSON reports for discovery only.
# Do not read model artifacts, Parquet rows, or CSV/holdout predictions.
json_report_metadata = []

for _, row in inventory_df[
    inventory_df["extension"] == ".json"
].iterrows():
    filename = row["filename"].lower()

    if not any(token in filename for token in [
        "report", "audit", "metric", "result", "summary",
        "manifest", "evaluation", "performance",
    ]):
        continue

    if row["size_bytes"] > 2_000_000:
        continue

    path = ROOT / row["relative_path"]

    try:
        content = read_json(path)
    except Exception:
        continue

    if not isinstance(content, dict):
        continue

    # Only store report identifiers and keys, not arbitrary contents.
    json_report_metadata.append({
        "relative_path": row["relative_path"],
        "top_level_keys": list(content.keys())[:60],
        "phase": content.get("phase"),
        "decision": content.get("decision"),
        "rows_predicted": content.get("rows_predicted"),
        "final_holdout_rows_used": content.get("final_holdout_rows_used"),
    })

# ------------------------------------------------------------
# 5. Save inventory and research-gap report
# ------------------------------------------------------------

inventory_df.to_csv(INVENTORY_CSV, index=False)

research_requirements = {
    "4.1_chronological_comparison":
        "Find the complete candidate-model table and persistence scores.",
    "4.2_validation_ensemble_selection":
        "Recover before/after tuning metrics and the validation-selected weights.",
    "4.3_frozen_holdout":
        "Recover ET, RF, ensemble, and persistence metrics without re-evaluating on the holdout.",
    "4.4_incremental_value":
        "Check for AQI-only, AQI-plus-lags, AQI-plus-rolling, and full-feature ablation results.",
    "4.5_uncertainty":
        "Find day-block bootstrap, paired deltas, 95% intervals, and prediction-interval evidence.",
    "4.6_error_regimes":
        "Find AQI-band, monthly/seasonal, lockdown, and station-level stratification.",
    "4.7_abrupt_transitions":
        "Find large-jump frequency, squared-error contribution, and missed deterioration cases.",
    "4.8_feature_attribution":
        "Find permutation-importance tables and grouped contributions.",
    "4.9_deep_learning":
        "Find exploratory LSTM, CNN-BiLSTM, VMD-LSTM, and TimeGAN reports.",
    "4.10_robustness_and_operational":
        "Find feature-reduction, latency, reproducibility, and deployment evidence.",
}

gap_report = {}

for section, description in research_requirements.items():
    topic_key = {
        "4.1_chronological_comparison": "chronological_model_comparison",
        "4.2_validation_ensemble_selection": "validation_tuning_ensemble",
        "4.3_frozen_holdout": "holdout_and_persistence",
        "4.4_incremental_value": "incremental_value_and_ablation",
        "4.5_uncertainty": "bootstrap_uncertainty_intervals",
        "4.6_error_regimes": "error_regimes_transitions",
        "4.7_abrupt_transitions": "error_regimes_transitions",
        "4.8_feature_attribution": "feature_attribution_and_shap",
        "4.9_deep_learning": "deep_learning_alternatives",
        "4.10_robustness_and_operational": "deployment_and_reproducibility",
    }[section]

    matches = topic_matches.get(topic_key, [])

    gap_report[section] = {
        "requirement": description,
        "candidate_artifact_count": len(matches),
        "candidate_artifacts": matches[:60],
        "status": (
            "CANDIDATES_FOUND_REVIEW_BEFORE_RERUNNING"
            if matches
            else "NO_FILENAME_MATCH_REQUIRES_FURTHER_INVESTIGATION"
        ),
    }

audit = {
    "phase": "17.07A",
    "decision": "RESEARCH_EVIDENCE_INVENTORY_COMPLETE",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(ROOT),
    "frozen_feature_count": len(feature_order),
    "frozen_model_sha256": model_hashes,
    "artifact_path_count": len(inventory_df),
    "inventory_csv": str(INVENTORY_CSV),
    "json_report_metadata": json_report_metadata,
    "topic_candidate_artifacts": topic_matches,
    "research_gap_inventory": gap_report,
    "constraints": {
        "dataset_rows_loaded": False,
        "final_holdout_data_read": False,
        "models_retrained": False,
        "frozen_model_files_modified": False,
        "production_enabled": False,
        "production_status": "NO_VALIDATED_LIVE_AQI_SOURCE",
    },
    "next_phase": (
        "17.07B — map existing metrics and lock the research protocol before "
        "running persistence, ablation, bootstrap, or transition experiments."
    ),
}

with open(AUDIT_JSON, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 6. Print a concise summary
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("PHASE 17.07A — FINAL RESULT")
print("=" * 72)
print("Decision:", audit["decision"])
print("Artifacts inventoried:", len(inventory_df))
print("Small JSON reports indexed:", len(json_report_metadata))

for section, details in gap_report.items():
    print(
        f"{section}: {details['candidate_artifact_count']} candidate artifacts"
    )

print("\nInventory CSV:", INVENTORY_CSV)
print("Research gap JSON:", AUDIT_JSON)
print("Final holdout data read: False")
print("Models retrained: False")
print("Production enabled: False")
print("Production status: NO_VALIDATED_LIVE_AQI_SOURCE")
print("=" * 72)

In [ ]:
# ============================================================
# PHASE 17.07B — EXISTING METRICS REGISTRY + LOCKED PAPER PROTOCOL
#
# Purpose:
#   1. Extract saved scalar metrics from small JSON reports.
#   2. Inventory prediction and evaluation files by schema/metadata.
#   3. Identify previously saved paper evidence.
#   4. Freeze the remaining research protocol before experiments.
#
# Restrictions:
#   No model training or tuning.
#   No Parquet/CSV data rows loaded.
#   No final-holdout observations accessed.
#   No frozen artifacts modified.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import pandas as pd

# ------------------------------------------------------------
# 1. Reconnect to the existing project path
# ------------------------------------------------------------

ROOT_CANDIDATES = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive2/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive3/MyDrive/Delhi-AQI-Prediction"),
]

def valid_project_root(root):
    package = root / "deployment/phase16_33_frozen_model_package"
    return all(p.is_file() for p in [
        package / "config/feature_contract.json",
        package / "models/phase15_10_final_tuned_extra_trees_trainval.joblib",
        package / "models/phase15_10_final_tuned_random_forest_trainval.joblib",
    ])

ROOT = next((p for p in ROOT_CANDIDATES if valid_project_root(p)), None)

if ROOT is None:
    from google.colab import drive

    for mount in [
        Path("/content/gdrive"),
        Path("/content/drive2"),
        Path("/content/drive3"),
        Path("/content/google_drive"),
        Path("/content/drive_mount"),
        Path("/content/drive"),
    ]:
        if mount.is_symlink():
            continue
        if mount.exists() and (
            not mount.is_dir() or any(mount.iterdir())
        ):
            continue
        if not mount.exists():
            mount.mkdir(parents=True, exist_ok=True)

        try:
            drive.mount(str(mount))
        except Exception:
            continue

        ROOT = next((p for p in ROOT_CANDIDATES if valid_project_root(p)), None)
        if ROOT is not None:
            break

if ROOT is None:
    raise FileNotFoundError("Verified Delhi AQI project not found in Google Drive.")

DEPLOY = ROOT / "deployment"
PACKAGE = DEPLOY / "phase16_33_frozen_model_package"
CONFIG = PACKAGE / "config"
DATA = ROOT / "data"
PROCESSED = DATA / "processed"

OUT = DEPLOY / "tests/phase17_07B_metrics_registry_protocol"
OUT.mkdir(parents=True, exist_ok=True)

METRICS_CSV = OUT / "phase17_07B_saved_scalar_metrics_registry.csv"
SCHEMA_CSV = OUT / "phase17_07B_evaluation_artifact_schema_registry.csv"
REPORT_INDEX_JSON = OUT / "phase17_07B_report_index.json"
PROTOCOL_JSON = OUT / "phase17_07B_locked_research_protocol.json"
SUMMARY_JSON = OUT / "phase17_07B_research_readiness_summary.json"

# ------------------------------------------------------------
# 2. Verify frozen contract and models
# ------------------------------------------------------------

EXPECTED_HASHES = {
    "extra_trees":
        "f0b3d43a56a7d5e4f08554f37ff234e1b5e012b5d5d69f18c750844b36cbe72d",
    "random_forest":
        "538b5ff34152d19221518e831528093b7bcc57e2b05f82669718f540b514abd8",
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

feature_contract_path = CONFIG / "feature_contract.json"
feature_contract = read_json(feature_contract_path)
FEATURE_ORDER = feature_contract.get("feature_order", [])

if len(FEATURE_ORDER) != 73 or len(set(FEATURE_ORDER)) != 73:
    raise RuntimeError("Frozen contract must contain exactly 73 unique features.")

MODEL_PATHS = {
    "extra_trees":
        PACKAGE / "models/phase15_10_final_tuned_extra_trees_trainval.joblib",
    "random_forest":
        PACKAGE / "models/phase15_10_final_tuned_random_forest_trainval.joblib",
}

MODEL_HASHES = {
    name: sha256_file(path)
    for name, path in MODEL_PATHS.items()
}

if MODEL_HASHES != EXPECTED_HASHES:
    raise RuntimeError("Frozen model hash mismatch. Research audit aborted.")

print("=" * 76)
print("PHASE 17.07B — METRICS REGISTRY + LOCKED PAPER PROTOCOL")
print("=" * 76)
print(f"[1] Project root: {ROOT}")
print(f"[1] Frozen features: {len(FEATURE_ORDER)}")
print("[1] Frozen ET/RF model hashes verified.")

# ------------------------------------------------------------
# 3. Inventory existing experiment artifacts
#    Record paths and sizes only. Do not load data tables.
# ------------------------------------------------------------

SEARCH_ROOTS = [
    ROOT / "notebooks",
    ROOT / "outputs",
    DEPLOY,
    PROCESSED,
]

SKIP_PARTS = {
    ".git", "__pycache__", ".ipynb_checkpoints", ".venv",
    "venv", "node_modules", "bundle", "verification_extract",
}

ARTIFACT_EXTENSIONS = {
    ".json", ".csv", ".parquet", ".ipynb", ".xlsx",
    ".md", ".txt", ".pdf", ".joblib", ".pkl",
}

ARTIFACTS = []

for base in SEARCH_ROOTS:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in ARTIFACT_EXTENSIONS:
            continue
        if any(part in SKIP_PARTS for part in path.parts):
            continue

        try:
            relative = str(path.relative_to(ROOT))
            size = path.stat().st_size
        except (OSError, ValueError):
            continue

        ARTIFACTS.append({
            "path": path,
            "relative_path": relative,
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": size,
        })

ARTIFACTS = list({
    record["relative_path"]: record
    for record in ARTIFACTS
}.values())

print(f"[2] Artifact paths indexed: {len(ARTIFACTS)}")

# ------------------------------------------------------------
# 4. Extract actual scalar metrics from small JSON reports only
# ------------------------------------------------------------

METRIC_KEY_PATTERN = re.compile(
    r"(^|_)(mae|rmse|r2|r_squared|r2_score|"
    r"accuracy|within_?5|within_?10|within_?20|"
    r"weight|delta|improvement|improvement_pct|"
    r"baseline|persistence|target|decision|phase|"
    r"rows_predicted|n_rows|n_samples|n_stations|"
    r"tolerance|bootstrap|ci_lower|ci_upper|"
    r"confidence_interval|prediction_interval|"
    r"holdout_rows_used|production_enabled|production_status)"
    r"($|_)",
    re.IGNORECASE,
)

SCALAR_METRICS = []
REPORT_INDEX = []
JSON_LIMIT_BYTES = 5_000_000

def collect_metric_scalars(obj, file_path, prefix="", depth=0):
    # Traverse dictionaries only; deliberately ignore lists/arrays because
    # they can contain row-level predictions or observations.
    if depth > 12 or not isinstance(obj, dict):
        return

    for key, value in obj.items():
        key_str = str(key)
        full_key = f"{prefix}.{key_str}" if prefix else key_str
        normalized_key = re.sub(r"[^a-z0-9]+", "_", key_str.lower()).strip("_")

        if (
            METRIC_KEY_PATTERN.search(normalized_key)
            and isinstance(value, (str, int, float, bool))
        ):
            SCALAR_METRICS.append({
                "relative_path": file_path,
                "metric_key": full_key,
                "metric_value": value,
                "value_type": type(value).__name__,
            })

        if isinstance(value, dict):
            collect_metric_scalars(
                value,
                file_path,
                prefix=full_key,
                depth=depth + 1,
            )

JSON_FILES = [
    a for a in ARTIFACTS
    if a["extension"] == ".json"
    and a["size_bytes"] <= JSON_LIMIT_BYTES
    and any(token in a["relative_path"].lower() for token in [
        "report", "result", "metric", "audit", "summary",
        "evaluation", "performance", "phase15", "phase16", "phase17",
    ])
]

for artifact in JSON_FILES:
    path = artifact["path"]
    try:
        data = read_json(path)
    except Exception as exc:
        REPORT_INDEX.append({
            "relative_path": artifact["relative_path"],
            "read_status": "JSON_PARSE_FAILED",
            "error_type": type(exc).__name__,
        })
        continue

    if not isinstance(data, dict):
        REPORT_INDEX.append({
            "relative_path": artifact["relative_path"],
            "read_status": "NOT_A_JSON_OBJECT",
        })
        continue

    # Store top-level keys and report identity only.
    index_record = {
        "relative_path": artifact["relative_path"],
        "read_status": "OK",
        "top_level_keys": list(data.keys())[:100],
        "phase": data.get("phase"),
        "decision": data.get("decision"),
        "rows_predicted": data.get("rows_predicted"),
        "final_holdout_rows_used": data.get("final_holdout_rows_used"),
        "production_enabled": data.get("production_enabled"),
        "production_status": data.get("production_status"),
    }
    REPORT_INDEX.append(index_record)

    # Only scalar report metrics are extracted; row arrays are not traversed.
    collect_metric_scalars(data, artifact["relative_path"])

metrics_df = pd.DataFrame(SCALAR_METRICS)
if metrics_df.empty:
    metrics_df = pd.DataFrame(
        columns=["relative_path", "metric_key", "metric_value", "value_type"]
    )
else:
    metrics_df = metrics_df.drop_duplicates().sort_values(
        ["relative_path", "metric_key"]
    )

metrics_df.to_csv(METRICS_CSV, index=False)
save_json(REPORT_INDEX_JSON, REPORT_INDEX)

print(f"[3] Small JSON reports inspected: {len(REPORT_INDEX)}")
print(f"[3] Scalar metric/status fields indexed: {len(metrics_df)}")

# ------------------------------------------------------------
# 5. Inspect data artifact schemas WITHOUT reading data rows
# ------------------------------------------------------------

SCHEMA_ROWS = []

CSV_NAME_FILTER = re.compile(
    r"metric|result|prediction|holdout|baseline|competition|"
    r"feature|importance|bootstrap|error|evaluation|phase15|phase16|"
    r"split|assignment|ablation|transition|stratification",
    re.IGNORECASE,
)

try:
    import pyarrow.parquet as pq
except Exception:
    pq = None

for artifact in ARTIFACTS:
    path = artifact["path"]
    ext = artifact["extension"]
    lower_path = artifact["relative_path"].lower()

    if ext == ".parquet":
        entry = {
            "relative_path": artifact["relative_path"],
            "format": "parquet",
            "size_bytes": artifact["size_bytes"],
            "metadata_read_only": True,
            "row_count": None,
            "columns": None,
            "schema_status": None,
        }

        if pq is None:
            entry["schema_status"] = "PYARROW_NOT_AVAILABLE"
        else:
            try:
                parquet_file = pq.ParquetFile(path)
                entry["row_count"] = parquet_file.metadata.num_rows
                entry["columns"] = parquet_file.schema.names
                entry["schema_status"] = "FOOTER_METADATA_READ"
            except Exception as exc:
                entry["schema_status"] = f"METADATA_READ_FAILED:{type(exc).__name__}"

        SCHEMA_ROWS.append(entry)

    elif ext == ".csv" and CSV_NAME_FILTER.search(lower_path):
        # Read only the header, never any CSV data rows.
        entry = {
            "relative_path": artifact["relative_path"],
            "format": "csv",
            "size_bytes": artifact["size_bytes"],
            "metadata_read_only": True,
            "row_count": None,
            "columns": None,
            "schema_status": None,
        }

        if artifact["size_bytes"] > 200_000_000:
            entry["schema_status"] = "HEADER_SKIPPED_FILE_OVER_200MB"
        else:
            try:
                header = pd.read_csv(path, nrows=0)
                entry["columns"] = list(header.columns)
                entry["schema_status"] = "HEADER_ONLY"
            except Exception as exc:
                entry["schema_status"] = f"HEADER_READ_FAILED:{type(exc).__name__}"

        SCHEMA_ROWS.append(entry)

schema_df = pd.DataFrame(SCHEMA_ROWS)
if schema_df.empty:
    schema_df = pd.DataFrame(
        columns=[
            "relative_path", "format", "size_bytes",
            "metadata_read_only", "row_count", "columns", "schema_status",
        ]
    )

schema_df.to_csv(SCHEMA_CSV, index=False)

print(f"[4] Data artifact schemas indexed: {len(schema_df)}")
print("[4] Parquet rows read: 0; CSV rows read: 0.")

# ------------------------------------------------------------
# 6. Detect candidate prediction, split and feature-ablation artifacts
# ------------------------------------------------------------

def path_contains(artifact, terms):
    value = artifact["relative_path"].lower()
    return any(term.lower() in value for term in terms)

candidate_categories = {
    "prediction_files": [
        a["relative_path"] for a in ARTIFACTS
        if path_contains(a, ["prediction", "predictions"])
        and a["extension"] in {".csv", ".parquet"}
    ],
    "holdout_files": [
        a["relative_path"] for a in ARTIFACTS
        if path_contains(a, ["holdout", "phase15_12", "final_test"])
        and a["extension"] in {".csv", ".parquet", ".json"}
    ],
    "persistence_files": [
        a["relative_path"] for a in ARTIFACTS
        if path_contains(a, ["persistence", "baseline"])
    ],
    "feature_ablation_files": [
        a["relative_path"] for a in ARTIFACTS
        if path_contains(a, ["ablation", "feature_reduction", "top_7", "top_10", "top_20", "aqi_only"])
    ],
    "bootstrap_files": [
        a["relative_path"] for a in ARTIFACTS
        if path_contains(a, ["bootstrap", "confidence_interval", "uncertainty"])
    ],
    "transition_files": [
        a["relative_path"] for a in ARTIFACTS
        if path_contains(a, ["transition", "jump", "stratification", "regime"])
    ],
    "importance_files": [
        a["relative_path"] for a in ARTIFACTS
        if path_contains(a, ["importance", "permutation", "shap", "xai"])
    ],
    "model_competition_files": [
        a["relative_path"] for a in ARTIFACTS
        if path_contains(a, ["competition", "model_metrics", "leaderboard"])
    ],
}

# Narrow to existing actual schema entries for useful file-column evidence.
schema_by_path = {
    row["relative_path"]: row
    for row in SCHEMA_ROWS
}

# Files with likely prediction payload columns.
prediction_schema_candidates = []
for row in SCHEMA_ROWS:
    columns = row.get("columns") or []
    lowered = [str(c).lower() for c in columns]

    if any(("prediction" in c or c.startswith("aqi_pred")) for c in lowered):
        prediction_schema_candidates.append({
            "relative_path": row["relative_path"],
            "format": row["format"],
            "row_count_metadata": row.get("row_count"),
            "columns": columns,
        })

# Identify the known split assignment artifact if present.
split_candidates = [
    a["relative_path"] for a in ARTIFACTS
    if "phase15_1_time_split_assignments.parquet" in a["relative_path"].lower()
]

# ------------------------------------------------------------
# 7. Save the pre-registered paper protocol
# ------------------------------------------------------------

# These are protocol commitments, not claims that the experiments already passed.
protocol = {
    "protocol_id": "Delhi-AQI-Paper-Protocol-17.07B",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "objective": "Forecast AQI_t_plus_1 using information available through time t.",
    "target": "AQI_t_plus_1",
    "station_scope": [
        "DL031", "DL002", "DL021", "DL028", "DL019",
        "DL010", "DL034", "DL007", "DL013", "DL004",
    ],
    "frozen_feature_count": 73,
    "frozen_feature_order_sha256": hashlib.sha256(
        json.dumps(FEATURE_ORDER, separators=(",", ":")).encode("utf-8")
    ).hexdigest(),
    "frozen_models": {
        "extra_trees_sha256": MODEL_HASHES["extra_trees"],
        "random_forest_sha256": MODEL_HASHES["random_forest"],
        "ensemble_weights": {
            "extra_trees": 0.594466,
            "random_forest": 0.405534,
        },
        "retraining_frozen_models_prohibited": True,
    },
    "chronological_split": {
        "assignment_artifact_candidates": split_candidates,
        "expected_total_rows": 304313,
        "expected_train_rows": 166845,
        "expected_validation_rows": 69097,
        "expected_final_holdout_rows": 68352,
        "expected_gap_rows": 19,
        "rule": (
            "Verify the saved split assignment metadata before the next model experiment. "
            "Do not randomly shuffle or alter the split."
        ),
    },
    "primary_metrics": [
        "MAE", "RMSE", "R2",
        "accuracy_within_5_AQI_points",
        "accuracy_within_10_AQI_points",
        "accuracy_within_20_AQI_points",
    ],
    "primary_comparisons": {
        "section_4_1": [
            "All compatible candidate models under the same 10-station chronological protocol",
            "Persistence baseline",
        ],
        "section_4_2": [
            "Untuned versus tuned candidates on validation",
            "Validation-optimized Extra Trees/Random Forest ensemble",
            "Explain validation selection and limited ensemble improvement",
        ],
        "section_4_3": [
            "Frozen Extra Trees",
            "Frozen Random Forest",
            "Frozen weighted ensemble",
            "Persistence",
        ],
        "section_4_4": [
            "Persistence",
            "AQI(t)-only supervised baseline",
            "AQI(t) plus lag features",
            "AQI(t) plus rolling statistics",
            "Full-feature model",
            "Full frozen ensemble",
        ],
        "section_4_5": [
            "Paired day-block bootstrap for principal metric differences",
            "MAE and RMSE differences, 95% confidence intervals and effect size",
            "Use the same holdout timestamps for paired comparisons",
            "Do not interpret hourly rows as independent observations",
        ],
        "section_4_6": [
            "Target AQI bands: <100, 100-199, 200-299, 300-399, >=400",
            "Seasons: winter Dec-Feb, summer Mar-May, monsoon Jun-Sep, post-monsoon Oct-Nov",
            "Month and station strata",
            "Pre/during/post lockdown only if exact historical dates and the split support the comparison",
            "Actual absolute AQI-change bins: <=5, (5,10], (10,20], >20",
        ],
        "section_4_7": [
            "Proportion of hours with absolute AQI change >20",
            "Contribution to total squared error",
            "Deterioration versus improvement cases",
            "MAE/RMSE on jump versus non-jump hours",
            "Large-deterioration false-negative rate if a detection rule is pre-specified",
        ],
        "section_4_8": [
            "Saved feature-importance results and repeatability",
            "AQI(t), AQI rolling and lagged features",
            "Grouped temporal, pollutant and contextual importance",
        ],
        "section_4_9": [
            "LSTM, CNN-BiLSTM, VMD-LSTM, TimeGAN as exploratory methods",
            "Do not directly rank results generated under incompatible scopes/protocols",
        ],
        "section_4_10": [
            "73-feature, top-20, top-10, top-7, and AQI-only variants",
            "Feature rankings and model choice must be determined using training/validation only",
            "Inference latency, missing-input rejection, reproducibility and live-data readiness",
            "Prediction intervals calibrated from non-holdout residuals, where feasible",
        ],
    },
    "feature_ablation_rules": {
        "feature_ranking_source": "Training/validation only",
        "hyperparameter_selection_source": "Validation only",
        "fit_policy_for_new_ablation_models": (
            "Fit candidates on training data; make all selection decisions on validation; "
            "refit the selected candidate on train+validation before its final holdout evaluation."
        ),
        "current_feature_semantics": (
            "Confirm the exact AQI(t) source column from the feature-engineering code before "
            "defining the AQI-only model. Do not infer semantics from a feature name alone."
        ),
        "missing_input_policy": (
            "Do not impute features in a way that violates the frozen production contract."
        ),
    },
    "uncertainty_rules": {
        "primary_inference": "Paired day-block bootstrap",
        "bootstrap_block_definition": (
            "Resample contiguous 7-day blocks and preserve all stations and paired model "
            "predictions within each selected temporal block, subject to verification of "
            "the saved timestamp structure."
        ),
        "interval_level": 0.95,
        "replicates": 10000,
        "prediction_interval_level": 0.90,
        "prediction_interval_calibration": (
            "Use training/validation residuals only to set interval calibration; evaluate "
            "coverage and width on the locked holdout once."
        ),
    },
    "final_holdout_policy": {
        "holdout_may_be_used_for_model_selection": False,
        "holdout_may_be_used_for_feature_ranking": False,
        "holdout_may_be_used_for_hyperparameter_tuning": False,
        "evaluate_once_after_protocol_lock": True,
        "existing_saved_holdout_metrics_may_be_recovered": True,
        "do_not_reuse_holdout_for_new_selection_decisions": True,
    },
    "deployment_policy": {
        "live_source_validated": False,
        "production_enabled": False,
        "production_status": "NO_VALIDATED_LIVE_AQI_SOURCE",
        "release_gate_must_remain_closed": True,
    },
}

# ------------------------------------------------------------
# 8. Save concise readiness summary
# ------------------------------------------------------------

gap_summary = {
    "4.1_chronological_model_comparison": len(candidate_categories["model_competition_files"]),
    "4.2_validation_tuning_ensemble": len(candidate_categories["model_competition_files"]),
    "4.3_holdout_files": len(candidate_categories["holdout_files"]),
    "4.4_persistence_files": len(candidate_categories["persistence_files"]),
    "4.4_ablation_files": len(candidate_categories["feature_ablation_files"]),
    "4.5_bootstrap_files": len(candidate_categories["bootstrap_files"]),
    "4.6_4.7_transition_regime_files": len(candidate_categories["transition_files"]),
    "4.8_feature_attribution_files": len(candidate_categories["importance_files"]),
    "prediction_schema_candidates": len(prediction_schema_candidates),
}

summary = {
    "phase": "17.07B",
    "decision": "METRICS_REGISTRY_AND_PROTOCOL_SAVED",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(ROOT),
    "artifact_paths_indexed": len(ARTIFACTS),
    "small_json_reports_indexed": len(REPORT_INDEX),
    "scalar_report_fields_indexed": len(metrics_df),
    "schema_entries_indexed": len(schema_df),
    "candidate_counts": gap_summary,
    "prediction_schema_candidates": prediction_schema_candidates[:100],
    "artifact_categories": candidate_categories,
    "outputs": {
        "scalar_metrics_csv": str(METRICS_CSV),
        "schema_registry_csv": str(SCHEMA_CSV),
        "report_index_json": str(REPORT_INDEX_JSON),
        "locked_protocol_json": str(PROTOCOL_JSON),
        "readiness_summary_json": str(SUMMARY_JSON),
    },
    "final_holdout_rows_read": 0,
    "dataset_rows_read": 0,
    "models_retrained": False,
    "production_enabled": False,
    "production_status": "NO_VALIDATED_LIVE_AQI_SOURCE",
}

save_json(PROTOCOL_JSON, protocol)
save_json(SUMMARY_JSON, summary)

# ------------------------------------------------------------
# 9. Print concise results and the next priority
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("PHASE 17.07B — FINAL RESULT")
print("=" * 76)
print("Decision:", summary["decision"])
print("JSON report fields extracted:", len(metrics_df))
print("Schema entries indexed:", len(schema_df))
print("\nCandidate artifact counts:")
for name, count in gap_summary.items():
    print(f"  {name}: {count}")

print("\nPrediction files with relevant output columns:")
for item in prediction_schema_candidates[:15]:
    print(
        f"  {item['relative_path']} | rows(metadata)={item['row_count_metadata']} "
        f"| columns={item['columns'][:12]}"
    )

print("\nSaved artifacts:")
print("  Metrics registry:", METRICS_CSV)
print("  Schema registry:", SCHEMA_CSV)
print("  Report index:", REPORT_INDEX_JSON)
print("  Locked protocol:", PROTOCOL_JSON)
print("  Readiness summary:", SUMMARY_JSON)

print("\nFinal holdout rows read: 0")
print("Dataset rows read: 0")
print("Models retrained: False")
print("Production enabled: False")
print("Production status: NO_VALIDATED_LIVE_AQI_SOURCE")
print("=" * 76)

In [ ]:
# ============================================================
# PHASE 17.07C — FROZEN HOLDOUT METRICS RECOVERY
#
# Reads the existing saved holdout predictions once.
# Computes ET, RF, ensemble and persistence metrics.
# Also calculates station-level performance and improvement.
#
# No retraining. No retuning. No feature selection.
# Production remains disabled.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Resolve the existing Drive project
# ------------------------------------------------------------

ROOT_CANDIDATES = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive2/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive3/MyDrive/Delhi-AQI-Prediction"),
]

ROOT = next(
    (
        p for p in ROOT_CANDIDATES
        if (
            p / "outputs/reports/phase15_12_final_holdout/"
                "phase15_12_final_holdout_predictions.parquet"
        ).is_file()
    ),
    None,
)

if ROOT is None:
    from google.colab import drive

    for mount in [
        Path("/content/gdrive"),
        Path("/content/drive2"),
        Path("/content/drive3"),
        Path("/content/google_drive"),
        Path("/content/drive_mount"),
        Path("/content/drive"),
    ]:
        if mount.is_symlink():
            continue

        if mount.exists() and (
            not mount.is_dir() or any(mount.iterdir())
        ):
            continue

        if not mount.exists():
            mount.mkdir(parents=True, exist_ok=True)

        try:
            drive.mount(str(mount))
        except Exception:
            continue

        ROOT = next(
            (
                p for p in ROOT_CANDIDATES
                if (
                    p / "outputs/reports/phase15_12_final_holdout/"
                        "phase15_12_final_holdout_predictions.parquet"
                ).is_file()
            ),
            None,
        )
        if ROOT is not None:
            break

if ROOT is None:
    raise FileNotFoundError(
        "Cannot locate the saved Phase 15.12 final holdout prediction artifact."
    )

DEPLOY = ROOT / "deployment"
PACKAGE = DEPLOY / "phase16_33_frozen_model_package"
CONFIG = PACKAGE / "config"

HOLDOUT_FILE = (
    ROOT / "outputs/reports/phase15_12_final_holdout/"
    "phase15_12_final_holdout_predictions.parquet"
)

EXPECTED_HASHES = {
    "extra_trees":
        "f0b3d43a56a7d5e4f08554f37ff234e1b5e012b5d5d69f18c750844b36cbe72d",
    "random_forest":
        "538b5ff34152d19221518e831528093b7bcc57e2b05f82669718f540b514abd8",
}

EXPECTED_STATIONS = [
    "DL031", "DL002", "DL021", "DL028", "DL019",
    "DL010", "DL034", "DL007", "DL013", "DL004",
]

OUT = ROOT / "outputs/research_paper/phase17_07C_holdout_metrics"
OUT.mkdir(parents=True, exist_ok=True)

OVERALL_CSV = OUT / "phase17_07C_overall_holdout_metrics.csv"
STATION_CSV = OUT / "phase17_07C_station_level_metrics.csv"
REPORT_JSON = OUT / "phase17_07C_frozen_holdout_metrics_report.json"
SUMMARY_MD = OUT / "PHASE_17_07C_HOLDOUT_METRICS.md"

PREDICTION_COLUMNS = {
    "Persistence": "Persistence_Prediction",
    "Extra Trees": "Tuned_Extra_Trees_Prediction",
    "Random Forest": "Tuned_Random_Forest_Prediction",
    "Weighted ensemble": "Final_Ensemble_Prediction",
}

WEIGHTS = {
    "Extra Trees": 0.594466,
    "Random Forest": 0.405534,
}

print("=" * 76)
print("PHASE 17.07C — FROZEN HOLDOUT METRICS RECOVERY")
print("=" * 76)
print(f"[1] Project root: {ROOT}")

# ------------------------------------------------------------
# 2. Verify the frozen models and feature contract
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

model_paths = {
    "extra_trees":
        PACKAGE / "models/phase15_10_final_tuned_extra_trees_trainval.joblib",
    "random_forest":
        PACKAGE / "models/phase15_10_final_tuned_random_forest_trainval.joblib",
}

model_hashes = {
    name: sha256_file(path)
    for name, path in model_paths.items()
}

if model_hashes != EXPECTED_HASHES:
    raise RuntimeError("Frozen model hash mismatch; metric recovery aborted.")

feature_contract = read_json(CONFIG / "feature_contract.json")
feature_order = feature_contract.get("feature_order", [])

if len(feature_order) != 73 or len(set(feature_order)) != 73:
    raise RuntimeError("Frozen feature contract is not exactly 73 unique features.")

print("[2] Frozen model hashes verified.")
print("[2] Frozen feature contract verified.")

# ------------------------------------------------------------
# 3. Read the existing holdout prediction artifact
# ------------------------------------------------------------

required_columns = [
    "Datetime",
    "StationId",
    "AQI_t_plus_1",
    *PREDICTION_COLUMNS.values(),
]

holdout = pd.read_parquet(HOLDOUT_FILE)

missing_columns = [
    col for col in required_columns if col not in holdout.columns
]
if missing_columns:
    raise ValueError(
        f"Holdout artifact is missing required columns: {missing_columns}"
    )

holdout = holdout[required_columns].copy()

if len(holdout) != 68352:
    raise ValueError(
        f"Expected 68,352 saved holdout rows; found {len(holdout)}. "
        "Investigate before reporting results."
    )

holdout["StationId"] = holdout["StationId"].astype(str)
holdout["Datetime"] = pd.to_datetime(holdout["Datetime"], errors="raise")

if set(holdout["StationId"].unique()) != set(EXPECTED_STATIONS):
    raise ValueError(
        "Station set differs from the frozen 10-station evaluation scope."
    )

duplicate_station_times = int(
    holdout.duplicated(["StationId", "Datetime"]).sum()
)

if duplicate_station_times:
    raise ValueError(
        f"Found {duplicate_station_times} duplicate StationId-Datetime rows."
    )

numeric_cols = [
    "AQI_t_plus_1",
    *PREDICTION_COLUMNS.values(),
]

for col in numeric_cols:
    holdout[col] = pd.to_numeric(holdout[col], errors="raise")

values = holdout[numeric_cols].to_numpy(dtype=float)

if not np.isfinite(values).all():
    raise ValueError("Holdout actuals or predictions contain missing/non-finite values.")

print(f"[3] Saved holdout rows loaded: {len(holdout):,}")
print(f"[3] Unique stations: {holdout['StationId'].nunique()}")
print(f"[3] Duplicate station-time rows: {duplicate_station_times}")
print(
    f"[3] Time range: {holdout['Datetime'].min()} to "
    f"{holdout['Datetime'].max()}"
)

# ------------------------------------------------------------
# 4. Define metrics consistently for all four predictions
# ------------------------------------------------------------

def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    if len(actual) == 0 or len(actual) != len(predicted):
        raise ValueError("Actual and predicted arrays must have matching, non-zero length.")

    error = predicted - actual
    absolute_error = np.abs(error)
    squared_error = error ** 2

    denominator = np.sum((actual - np.mean(actual)) ** 2)
    r2 = (
        float(1 - np.sum(squared_error) / denominator)
        if denominator > 0 else np.nan
    )

    return {
        "N": int(len(actual)),
        "MAE": float(np.mean(absolute_error)),
        "RMSE": float(np.sqrt(np.mean(squared_error))),
        "R2": r2,
        "Within_5_AQI_pct": float(100 * np.mean(absolute_error <= 5)),
        "Within_10_AQI_pct": float(100 * np.mean(absolute_error <= 10)),
        "Within_20_AQI_pct": float(100 * np.mean(absolute_error <= 20)),
        "Bias_prediction_minus_actual": float(np.mean(error)),
    }

actual = holdout["AQI_t_plus_1"].to_numpy(dtype=float)

overall_rows = []
for model_name, prediction_col in PREDICTION_COLUMNS.items():
    predictions = holdout[prediction_col].to_numpy(dtype=float)

    row = {
        "Model": model_name,
        **calculate_metrics(actual, predictions),
    }
    overall_rows.append(row)

overall = pd.DataFrame(overall_rows)

# ------------------------------------------------------------
# 5. Compare the final ensemble with persistence
# ------------------------------------------------------------

baseline = overall.loc[overall["Model"] == "Persistence"].iloc[0]

overall["MAE_improvement_vs_persistence"] = (
    baseline["MAE"] - overall["MAE"]
)
overall["MAE_improvement_vs_persistence_pct"] = (
    100 * (baseline["MAE"] - overall["MAE"]) / baseline["MAE"]
)
overall["RMSE_improvement_vs_persistence"] = (
    baseline["RMSE"] - overall["RMSE"]
)
overall["RMSE_improvement_vs_persistence_pct"] = (
    100 * (baseline["RMSE"] - overall["RMSE"]) / baseline["RMSE"]
)

# ------------------------------------------------------------
# 6. Verify the ensemble formula from saved component predictions
# ------------------------------------------------------------

et_pred = holdout["Tuned_Extra_Trees_Prediction"].to_numpy(dtype=float)
rf_pred = holdout["Tuned_Random_Forest_Prediction"].to_numpy(dtype=float)
ensemble_pred = holdout["Final_Ensemble_Prediction"].to_numpy(dtype=float)

ensemble_from_weights = (
    WEIGHTS["Extra Trees"] * et_pred
    + WEIGHTS["Random Forest"] * rf_pred
)

ensemble_formula_max_abs_difference = float(
    np.max(np.abs(ensemble_from_weights - ensemble_pred))
)

ENSEMBLE_FORMULA_TOLERANCE = 1e-8
ensemble_formula_pass = (
    ensemble_formula_max_abs_difference <= ENSEMBLE_FORMULA_TOLERANCE
)

print(
    "[4] Ensemble formula max absolute difference: "
    f"{ensemble_formula_max_abs_difference:.12g}"
)

# ------------------------------------------------------------
# 7. Station-level results for persistence, ET, RF and ensemble
# ------------------------------------------------------------

station_rows = []

for station_id, group in holdout.groupby("StationId", sort=True):
    station_actual = group["AQI_t_plus_1"].to_numpy(dtype=float)
    persistence_pred = group["Persistence_Prediction"].to_numpy(dtype=float)
    persistence_metrics = calculate_metrics(station_actual, persistence_pred)

    for model_name, prediction_col in PREDICTION_COLUMNS.items():
        model_pred = group[prediction_col].to_numpy(dtype=float)
        model_metrics = calculate_metrics(station_actual, model_pred)

        station_rows.append({
            "StationId": station_id,
            "Model": model_name,
            **model_metrics,
            "MAE_improvement_vs_persistence": (
                persistence_metrics["MAE"] - model_metrics["MAE"]
            ),
            "MAE_improvement_vs_persistence_pct": (
                100
                * (persistence_metrics["MAE"] - model_metrics["MAE"])
                / persistence_metrics["MAE"]
                if persistence_metrics["MAE"] != 0 else np.nan
            ),
            "RMSE_improvement_vs_persistence": (
                persistence_metrics["RMSE"] - model_metrics["RMSE"]
            ),
            "RMSE_improvement_vs_persistence_pct": (
                100
                * (persistence_metrics["RMSE"] - model_metrics["RMSE"])
                / persistence_metrics["RMSE"]
                if persistence_metrics["RMSE"] != 0 else np.nan
            ),
        })

station_metrics = pd.DataFrame(station_rows)

# ------------------------------------------------------------
# 8. Consistency checks against the previously recorded ensemble result
# ------------------------------------------------------------

ensemble_row = overall.loc[
    overall["Model"] == "Weighted ensemble"
].iloc[0]

previously_reported_ensemble = {
    "MAE": 3.216121,
    "RMSE": 7.658390,
    "R2": 0.996634,
    "Within_5_AQI_pct": 85.249883,
    "Within_10_AQI_pct": 94.605864,
    "Within_20_AQI_pct": 98.090765,
}

metric_consistency = {}
for metric, expected in previously_reported_ensemble.items():
    observed = float(ensemble_row[metric])
    metric_consistency[metric] = {
        "saved_summary_value": expected,
        "recovered_value": observed,
        "absolute_difference": abs(observed - expected),
        "consistent_with_rounding": abs(observed - expected) <= 0.001,
    }

consistency_pass = all(
    entry["consistent_with_rounding"]
    for entry in metric_consistency.values()
)

# ------------------------------------------------------------
# 9. Save tables and report
# ------------------------------------------------------------

overall.to_csv(OVERALL_CSV, index=False)
station_metrics.to_csv(STATION_CSV, index=False)

station_ensemble = station_metrics[
    station_metrics["Model"] == "Weighted ensemble"
].copy()

station_improvement_positive_mae = int(
    (station_ensemble["MAE_improvement_vs_persistence"] > 0).sum()
)
station_improvement_positive_rmse = int(
    (station_ensemble["RMSE_improvement_vs_persistence"] > 0).sum()
)

report = {
    "phase": "17.07C",
    "decision": (
        "FROZEN_HOLDOUT_METRICS_RECOVERED"
        if ensemble_formula_pass and consistency_pass
        else "FROZEN_HOLDOUT_METRICS_RECOVERED_WITH_REVIEW_FLAGS"
    ),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": str(HOLDOUT_FILE),
    "source_sha256": sha256_file(HOLDOUT_FILE),
    "n_rows": int(len(holdout)),
    "n_stations": int(holdout["StationId"].nunique()),
    "time_start": str(holdout["Datetime"].min()),
    "time_end": str(holdout["Datetime"].max()),
    "duplicate_station_datetime_rows": duplicate_station_times,
    "frozen_model_hashes": model_hashes,
    "ensemble_weights": WEIGHTS,
    "ensemble_formula_max_abs_difference": ensemble_formula_max_abs_difference,
    "ensemble_formula_pass": ensemble_formula_pass,
    "ensemble_metric_consistency": metric_consistency,
    "ensemble_metric_consistency_pass": consistency_pass,
    "overall_metrics": overall.to_dict(orient="records"),
    "station_ensemble_advantage_over_persistence": {
        "stations_with_lower_mae": station_improvement_positive_mae,
        "stations_with_lower_rmse": station_improvement_positive_rmse,
        "total_stations": int(station_ensemble["StationId"].nunique()),
    },
    "outputs": {
        "overall_metrics_csv": str(OVERALL_CSV),
        "station_metrics_csv": str(STATION_CSV),
    },
    "analysis_limits": [
        "This step recovers point metrics from a previously saved holdout prediction file.",
        "It does not retrain or tune any model.",
        "It does not use the holdout for feature ranking or model selection.",
        "Confidence intervals and statistical significance are not computed in this phase.",
        "The holdout should not be repeatedly used to choose among newly designed ablation models.",
    ],
    "final_holdout_rows_used": int(len(holdout)),
    "models_retrained": False,
    "production_enabled": False,
    "production_status": "NO_VALIDATED_LIVE_AQI_SOURCE",
}

with open(REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

summary_lines = [
    "# Phase 17.07C — Frozen Holdout Metrics",
    "",
    f"**Decision:** {report['decision']}",
    "",
    f"- Rows: {len(holdout):,}",
    f"- Stations: {holdout['StationId'].nunique()}",
    f"- Period: {holdout['Datetime'].min()} to {holdout['Datetime'].max()}",
    f"- Ensemble formula parity: {ensemble_formula_pass}",
    f"- Previously reported ensemble metrics consistent: {consistency_pass}",
    "",
    "## Overall metrics",
    "",
    overall.to_markdown(index=False, floatfmt=".6f"),
    "",
    "## Weighted ensemble by station",
    "",
    station_ensemble[
        [
            "StationId", "N", "MAE", "RMSE", "R2",
            "MAE_improvement_vs_persistence_pct",
            "RMSE_improvement_vs_persistence_pct",
        ]
    ].to_markdown(index=False, floatfmt=".4f"),
    "",
    "## Interpretation restriction",
    "",
    "These are recovered point estimates from the existing frozen holdout predictions.",
    "They are not uncertainty intervals and do not support a claim of statistical",
    "significance by themselves. The final holdout is not to be used for new model",
    "selection, feature ranking or hyperparameter tuning.",
    "",
    "Production remains disabled because the live input source is not validated.",
]

SUMMARY_MD.write_text("\n".join(summary_lines), encoding="utf-8")

# ------------------------------------------------------------
# 10. Print paper-facing tables
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("PHASE 17.07C — OVERALL HOLDOUT METRICS")
print("=" * 76)
print(
    overall[
        [
            "Model", "N", "MAE", "RMSE", "R2",
            "Within_5_AQI_pct",
            "Within_10_AQI_pct",
            "Within_20_AQI_pct",
            "MAE_improvement_vs_persistence_pct",
            "RMSE_improvement_vs_persistence_pct",
        ]
    ].round(6).to_string(index=False)
)

print("\n" + "=" * 76)
print("WEIGHTED ENSEMBLE — STATION-LEVEL PERSISTENCE COMPARISON")
print("=" * 76)
print(
    station_ensemble[
        [
            "StationId", "N", "MAE", "RMSE", "R2",
            "MAE_improvement_vs_persistence_pct",
            "RMSE_improvement_vs_persistence_pct",
        ]
    ].round(4).to_string(index=False)
)

print("\n" + "=" * 76)
print("PHASE 17.07C — FINAL RESULT")
print("=" * 76)
print("Decision:", report["decision"])
print("Ensemble formula parity:", ensemble_formula_pass)
print("Previously reported ensemble metric consistency:", consistency_pass)
print(
    "Stations where ensemble beats persistence on RMSE:",
    f"{station_improvement_positive_rmse}/10",
)
print("Overall metrics CSV:", OVERALL_CSV)
print("Station metrics CSV:", STATION_CSV)
print("JSON report:", REPORT_JSON)
print("Paper summary:", SUMMARY_MD)
print("Models retrained: False")
print("Production enabled: False")
print("Production status: NO_VALIDATED_LIVE_AQI_SOURCE")
print("=" * 76)

In [ ]:
# ============================================================
# PHASE 17.07C-R1 — REPAIR HOLDOUT METRICS RECOVERY
# Diagnose missing/infinite values; use common valid rows only.
# No retraining, tuning, imputation, or model selection.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd

# 1. Find the project and verify frozen artifacts.
ROOT_CANDIDATES = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive2/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive3/MyDrive/Delhi-AQI-Prediction"),
]

HOLDOUT_REL = (
    "outputs/reports/phase15_12_final_holdout/"
    "phase15_12_final_holdout_predictions.parquet"
)

ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / HOLDOUT_REL).is_file()),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Cannot find the existing frozen holdout prediction artifact. "
        "Reconnect Google Drive and rerun this cell."
    )

DEPLOY = ROOT / "deployment"
PACKAGE = DEPLOY / "phase16_33_frozen_model_package"
CONFIG = PACKAGE / "config"
HOLDOUT_FILE = ROOT / HOLDOUT_REL

OUT = (
    ROOT / "outputs/research_paper/"
    "phase17_07C_R1_holdout_metrics_repair"
)
OUT.mkdir(parents=True, exist_ok=True)

INVALID_ROWS_CSV = OUT / "phase17_07C_R1_invalid_rows_audit.csv"
MISSINGNESS_CSV = OUT / "phase17_07C_R1_column_quality_audit.csv"
OVERALL_CSV = OUT / "phase17_07C_R1_overall_metrics.csv"
STATION_CSV = OUT / "phase17_07C_R1_station_metrics.csv"
REPORT_JSON = OUT / "phase17_07C_R1_holdout_metrics_report.json"

EXPECTED_HASHES = {
    "extra_trees":
        "f0b3d43a56a7d5e4f08554f37ff234e1b5e012b5d5d69f18c750844b36cbe72d",
    "random_forest":
        "538b5ff34152d19221518e831528093b7bcc57e2b05f82669718f540b514abd8",
}

PREDICTIONS = {
    "Persistence": "Persistence_Prediction",
    "Extra Trees": "Tuned_Extra_Trees_Prediction",
    "Random Forest": "Tuned_Random_Forest_Prediction",
    "Weighted ensemble": "Final_Ensemble_Prediction",
}
ACTUAL_COL = "AQI_t_plus_1"
TIME_COL = "Datetime"
STATION_COL = "StationId"

WEIGHTS = {"Extra Trees": 0.594466, "Random Forest": 0.405534}
EXPECTED_HOLDOUT_ROWS = 68352

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def metrics(actual, prediction):
    actual = np.asarray(actual, dtype=float)
    prediction = np.asarray(prediction, dtype=float)

    error = prediction - actual
    abs_error = np.abs(error)
    squared_error = error ** 2
    variance = np.sum((actual - np.mean(actual)) ** 2)

    return {
        "N": int(len(actual)),
        "MAE": float(np.mean(abs_error)),
        "RMSE": float(np.sqrt(np.mean(squared_error))),
        "R2": (
            float(1 - np.sum(squared_error) / variance)
            if variance > 0 else np.nan
        ),
        "Within_5_AQI_pct": float(100 * np.mean(abs_error <= 5)),
        "Within_10_AQI_pct": float(100 * np.mean(abs_error <= 10)),
        "Within_20_AQI_pct": float(100 * np.mean(abs_error <= 20)),
        "Bias_prediction_minus_actual": float(np.mean(error)),
    }

print("=" * 72)
print("PHASE 17.07C-R1 — HOLDOUT METRICS REPAIR")
print("=" * 72)

# 2. Verify frozen model hashes and the 73-feature contract.
feature_contract = read_json(CONFIG / "feature_contract.json")
feature_order = feature_contract.get("feature_order", [])
require(
    len(feature_order) == 73 and len(set(feature_order)) == 73,
    "Frozen feature contract is not 73 unique features.",
)

model_paths = {
    "extra_trees": PACKAGE / "models/phase15_10_final_tuned_extra_trees_trainval.joblib",
    "random_forest": PACKAGE / "models/phase15_10_final_tuned_random_forest_trainval.joblib",
}
model_hashes = {name: sha256_file(path) for name, path in model_paths.items()}
require(model_hashes == EXPECTED_HASHES, "Frozen model hash mismatch.")

print("[1] Frozen model hashes and feature contract verified.")

# 3. Load only the already-saved holdout predictions.
df = pd.read_parquet(HOLDOUT_FILE)

required_cols = [
    TIME_COL, STATION_COL, ACTUAL_COL, *PREDICTIONS.values()
]
missing_cols = [c for c in required_cols if c not in df.columns]
require(not missing_cols, f"Required columns missing: {missing_cols}")

df = df[required_cols].copy()
original_rows = len(df)

require(
    original_rows == EXPECTED_HOLDOUT_ROWS,
    f"Expected {EXPECTED_HOLDOUT_ROWS} rows, received {original_rows}. "
    "Do not proceed until the artifact discrepancy is understood.",
)

df[STATION_COL] = df[STATION_COL].astype("string")
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

if df.duplicated([STATION_COL, TIME_COL]).any():
    raise ValueError("Duplicate station-time rows found; investigate before metrics.")

# 4. Convert numeric values carefully and audit NaN/inf separately.
NUMERIC_COLS = [ACTUAL_COL, *PREDICTIONS.values()]
quality_rows = []
invalid_masks = {}

for col in NUMERIC_COLS:
    numeric = pd.to_numeric(df[col], errors="coerce")
    missing_or_non_numeric = numeric.isna()
    infinite = pd.Series(
        np.isinf(numeric.to_numpy(dtype=float, na_value=np.nan)),
        index=df.index
    )

    invalid = missing_or_non_numeric | infinite
    invalid_masks[col] = invalid

    quality_rows.append({
        "column": col,
        "total_rows": len(df),
        "missing_or_non_numeric": int(missing_or_non_numeric.sum()),
        "infinite_values": int(infinite.sum()),
        "total_invalid": int(invalid.sum()),
        "valid_rows": int((~invalid).sum()),
    })

    df[col] = numeric

quality = pd.DataFrame(quality_rows)
quality.to_csv(MISSINGNESS_CSV, index=False)

# Also mark timestamp/station metadata issues.
metadata_invalid = df[TIME_COL].isna() | df[STATION_COL].isna()

any_numeric_invalid = pd.Series(False, index=df.index)
for invalid_mask in invalid_masks.values():
    any_numeric_invalid |= invalid_mask

invalid_row_mask = metadata_invalid | any_numeric_invalid

invalid_rows = df.loc[
    invalid_row_mask, [TIME_COL, STATION_COL]
].copy()

invalid_rows.insert(
    0, "original_row_position",
    np.flatnonzero(invalid_row_mask.to_numpy())
)

for col, mask in invalid_masks.items():
    invalid_rows[f"invalid_{col}"] = mask.loc[invalid_row_mask].to_numpy()

invalid_rows["invalid_datetime_or_station"] = (
    metadata_invalid.loc[invalid_row_mask].to_numpy()
)
invalid_rows.to_csv(INVALID_ROWS_CSV, index=False)

print("\n[2] Per-column quality audit:")
print(quality.to_string(index=False))
print(f"\nRows with any invalid required field: {int(invalid_row_mask.sum())}")

# 5. Use a common set for every model, including persistence.
# Do not impute or independently select different rows for each model.
valid_mask = ~invalid_row_mask
valid = df.loc[valid_mask].copy()

if valid.empty:
    report = {
        "phase": "17.07C-R1",
        "decision": "NO_VALID_COMMON_HOLDOUT_ROWS",
        "total_rows": original_rows,
        "valid_common_rows": 0,
        "invalid_rows": int(invalid_row_mask.sum()),
        "column_quality": quality.to_dict(orient="records"),
        "invalid_rows_csv": str(INVALID_ROWS_CSV),
        "column_quality_csv": str(MISSINGNESS_CSV),
        "production_enabled": False,
        "production_status": "NO_VALIDATED_LIVE_AQI_SOURCE",
    }
    with open(REPORT_JSON, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    raise RuntimeError(
        "No common valid rows. Review the quality and invalid-row CSVs."
    )

valid["StationId"] = valid[STATION_COL].astype(str)

actual = valid[ACTUAL_COL].to_numpy(dtype=float)
overall_rows = []

for model_name, col in PREDICTIONS.items():
    model_pred = valid[col].to_numpy(dtype=float)
    overall_rows.append({
        "Model": model_name,
        **metrics(actual, model_pred),
    })

overall = pd.DataFrame(overall_rows)

# 6. Compute improvement over persistence on exactly the same rows.
persistence = overall.loc[overall["Model"] == "Persistence"].iloc[0]

overall["MAE_improvement_vs_persistence"] = (
    persistence["MAE"] - overall["MAE"]
)
overall["MAE_improvement_vs_persistence_pct"] = (
    100 * (persistence["MAE"] - overall["MAE"]) / persistence["MAE"]
    if persistence["MAE"] != 0 else np.nan
)
overall["RMSE_improvement_vs_persistence"] = (
    persistence["RMSE"] - overall["RMSE"]
)
overall["RMSE_improvement_vs_persistence_pct"] = (
    100 * (persistence["RMSE"] - overall["RMSE"]) / persistence["RMSE"]
    if persistence["RMSE"] != 0 else np.nan
)

# 7. Verify that the stored ensemble follows the frozen weighted formula.
formula_prediction = (
    WEIGHTS["Extra Trees"] * valid[PREDICTIONS["Extra Trees"]].to_numpy(dtype=float)
    + WEIGHTS["Random Forest"] * valid[PREDICTIONS["Random Forest"]].to_numpy(dtype=float)
)

ensemble_prediction = valid[PREDICTIONS["Weighted ensemble"]].to_numpy(dtype=float)
formula_diff = np.abs(formula_prediction - ensemble_prediction)

ensemble_formula_max_diff = float(formula_diff.max())
ensemble_formula_pass = ensemble_formula_max_diff <= 1e-8

# 8. Station-level comparisons on common valid rows only.
station_rows = []

for station_id, group in valid.groupby("StationId", sort=True):
    station_actual = group[ACTUAL_COL].to_numpy(dtype=float)
    baseline_metrics = metrics(
        station_actual, group[PREDICTIONS["Persistence"]].to_numpy(dtype=float)
    )

    for model_name, col in PREDICTIONS.items():
        current_metrics = metrics(station_actual, group[col].to_numpy(dtype=float))

        station_rows.append({
            "StationId": station_id,
            "Model": model_name,
            **current_metrics,
            "MAE_improvement_vs_persistence_pct": (
                100 * (baseline_metrics["MAE"] - current_metrics["MAE"])
                / baseline_metrics["MAE"]
                if baseline_metrics["MAE"] != 0 else np.nan
            ),
            "RMSE_improvement_vs_persistence_pct": (
                100 * (baseline_metrics["RMSE"] - current_metrics["RMSE"])
                / baseline_metrics["RMSE"]
                if baseline_metrics["RMSE"] != 0 else np.nan
            ),
        })

station_metrics = pd.DataFrame(station_rows)

# 9. Compare ensemble estimates with the previously reported figures.
# A discrepancy does not trigger model selection; it triggers review.
known_ensemble = {
    "MAE": 3.216121,
    "RMSE": 7.658390,
    "R2": 0.996634,
    "Within_5_AQI_pct": 85.249883,
    "Within_10_AQI_pct": 94.605864,
    "Within_20_AQI_pct": 98.090765,
}

ensemble_row = overall.loc[
    overall["Model"] == "Weighted ensemble"
].iloc[0]

known_consistency = {}
for metric_name, old_value in known_ensemble.items():
    new_value = float(ensemble_row[metric_name])
    known_consistency[metric_name] = {
        "previously_reported": old_value,
        "recovered_on_common_valid_rows": new_value,
        "absolute_difference": abs(new_value - old_value),
        "comparable_row_count": len(valid) == EXPECTED_HOLDOUT_ROWS,
    }

full_support = len(valid) == EXPECTED_HOLDOUT_ROWS
consistent_with_previous = (
    full_support
    and all(
        item["absolute_difference"] <= 0.001
        for item in known_consistency.values()
    )
)

overall.to_csv(OVERALL_CSV, index=False)
station_metrics.to_csv(STATION_CSV, index=False)

ensemble_by_station = station_metrics[
    station_metrics["Model"] == "Weighted ensemble"
]
stations_ensemble_better_rmse = int(
    (ensemble_by_station["RMSE_improvement_vs_persistence_pct"] > 0).sum()
)

decision = (
    "FULL_HOLDOUT_METRICS_RECOVERED"
    if full_support and ensemble_formula_pass and consistent_with_previous
    else "COMMON_VALID_HOLDOUT_METRICS_RECOVERED_REQUIRES_REVIEW"
)

report = {
    "phase": "17.07C-R1",
    "decision": decision,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "holdout_file": str(HOLDOUT_FILE),
    "holdout_file_sha256": sha256_file(HOLDOUT_FILE),
    "original_rows": int(original_rows),
    "valid_common_rows": int(len(valid)),
    "excluded_rows": int(original_rows - len(valid)),
    "valid_row_coverage_pct": float(100 * len(valid) / original_rows),
    "invalid_rows_csv": str(INVALID_ROWS_CSV),
    "column_quality_csv": str(MISSINGNESS_CSV),
    "column_quality": quality.to_dict(orient="records"),
    "frozen_model_hashes": model_hashes,
    "ensemble_weights": WEIGHTS,
    "ensemble_formula_max_abs_difference": ensemble_formula_max_diff,
    "ensemble_formula_pass": ensemble_formula_pass,
    "overall_metrics": overall.to_dict(orient="records"),
    "station_metrics_csv": str(STATION_CSV),
    "station_metrics": station_metrics.to_dict(orient="records"),
    "ensemble_vs_previous_summary": known_consistency,
    "previous_summary_consistency_pass": consistent_with_previous,
    "ensemble_improves_RMSE_over_persistence_at_stations": stations_ensemble_better_rmse,
    "total_stations_in_valid_subset": int(valid["StationId"].nunique()),
    "restriction": (
        "Metrics use a single common subset of valid rows. Do not compare these "
        "point estimates with full-row results without considering row coverage. "
        "No invalid values were imputed."
    ),
    "models_retrained": False,
    "model_selection_on_holdout": False,
    "production_enabled": False,
    "production_status": "NO_VALIDATED_LIVE_AQI_SOURCE",
}

with open(REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 72)
print("PHASE 17.07C-R1 — OVERALL METRICS")
print("=" * 72)
print(f"Rows in original artifact: {original_rows:,}")
print(f"Common valid rows used: {len(valid):,}")
print(f"Rows excluded from every model comparison: {original_rows - len(valid):,}")
print(f"Valid row coverage: {100 * len(valid) / original_rows:.4f}%")
print()
print(
    overall[
        [
            "Model", "N", "MAE", "RMSE", "R2",
            "Within_5_AQI_pct", "Within_10_AQI_pct", "Within_20_AQI_pct",
            "MAE_improvement_vs_persistence_pct",
            "RMSE_improvement_vs_persistence_pct",
        ]
    ].round(6).to_string(index=False)
)

print("\nEnsemble formula max difference:", f"{ensemble_formula_max_diff:.12g}")
print("Ensemble formula pass:", ensemble_formula_pass)
print("Consistent with previous ensemble summary on full support:", consistent_with_previous)
print("Stations where ensemble beats persistence on RMSE:",
      f"{stations_ensemble_better_rmse}/{valid['StationId'].nunique()}")

print("\nQuality audit:", MISSINGNESS_CSV)
print("Invalid rows audit:", INVALID_ROWS_CSV)
print("Overall metrics CSV:", OVERALL_CSV)
print("Station metrics CSV:", STATION_CSV)
print("Report JSON:", REPORT_JSON)
print("Models retrained: False")
print("Production enabled: False")
print("Production status: NO_VALIDATED_LIVE_AQI_SOURCE")
print("=" * 72)

In [ ]:
# PHASE 17.07C-R2 — FULL-SUPPORT AND MATCHED-SUPPORT RECONCILIATION
# Read-only analysis: no model retraining, tuning, or holdout-based selection.

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=" * 76)
print("PHASE 17.07C-R2 — HOLDOUT SUPPORT RECONCILIATION")
print("=" * 76)

# 1. Resolve project and artifacts
POSSIBLE_ROOTS = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in POSSIBLE_ROOTS if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Check the Google Drive mount and project folder."
    )

HOLDOUT_PATH = (
    ROOT / "outputs/reports/phase15_12_final_holdout/"
    "phase15_12_final_holdout_predictions.parquet"
)
if not HOLDOUT_PATH.exists():
    raise FileNotFoundError(f"Missing frozen holdout artifact: {HOLDOUT_PATH}")

OUT = ROOT / "outputs/research_paper/phase17_07C_R2_support_reconciliation"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(HOLDOUT_PATH)

ACTUAL = "AQI_t_plus_1"
ET = "Tuned_Extra_Trees_Prediction"
RF = "Tuned_Random_Forest_Prediction"
ENS = "Final_Ensemble_Prediction"
PERSIST = "Persistence_Prediction"

required = [ACTUAL, ET, RF, ENS, PERSIST]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise KeyError(f"Expected columns missing from holdout artifact: {missing_cols}")

# Preserve source data. Use separate numeric analysis columns.
x = df[required].copy()
for c in required:
    x[c] = pd.to_numeric(x[c], errors="coerce")

finite = pd.DataFrame(
    {c: np.isfinite(x[c].to_numpy(dtype=float)) for c in required},
    index=x.index
)

# Actual plus each model's own available predictions.
model_cols = {
    "Persistence": PERSIST,
    "Extra Trees": ET,
    "Random Forest": RF,
    "Weighted ensemble": ENS,
}

def metrics_for(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    err = pred - actual
    return {
        "N": int(len(actual)),
        "MAE": float(mean_absolute_error(actual, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, pred))),
        "R2": float(r2_score(actual, pred)),
        "Within_5_AQI_pct": float(np.mean(np.abs(err) <= 5) * 100),
        "Within_10_AQI_pct": float(np.mean(np.abs(err) <= 10) * 100),
        "Within_20_AQI_pct": float(np.mean(np.abs(err) <= 20) * 100),
    }

# 2. Per-model maximum available support
full_support_rows = []
support_masks = {}

for model, pred_col in model_cols.items():
    mask = finite[ACTUAL] & finite[pred_col]
    support_masks[model] = mask
    m = metrics_for(x.loc[mask, ACTUAL], x.loc[mask, pred_col])
    m["Model"] = model
    m["Support"] = "All valid rows for this model"
    full_support_rows.append(m)

full_support = pd.DataFrame(full_support_rows)[
    ["Model", "Support", "N", "MAE", "RMSE", "R2",
     "Within_5_AQI_pct", "Within_10_AQI_pct", "Within_20_AQI_pct"]
]

# 3. Strict common support: all five required columns must be finite.
common_mask = finite[required].all(axis=1)
common_rows = []
for model, pred_col in model_cols.items():
    mask = common_mask
    m = metrics_for(x.loc[mask, ACTUAL], x.loc[mask, pred_col])
    m["Model"] = model
    m["Support"] = "Same rows for all four models"
    common_rows.append(m)

common_support = pd.DataFrame(common_rows)[
    ["Model", "Support", "N", "MAE", "RMSE", "R2",
     "Within_5_AQI_pct", "Within_10_AQI_pct", "Within_20_AQI_pct"]
]

# 4. Quantify effects of excluding rows whose persistence prediction is missing.
ensemble_valid = support_masks["Weighted ensemble"]
persistence_valid = support_masks["Persistence"]
persistence_missing_but_ensemble_valid = ensemble_valid & ~persistence_valid

subset_rows = []
for subset_name, mask in [
    ("All ensemble-valid rows", ensemble_valid),
    ("Rows with valid persistence", ensemble_valid & persistence_valid),
    ("Rows with missing persistence", persistence_missing_but_ensemble_valid),
]:
    n = int(mask.sum())
    if n == 0:
        subset_rows.append({"Subset": subset_name, "N": 0})
        continue
    actual = x.loc[mask, ACTUAL].to_numpy(dtype=float)
    pred = x.loc[mask, ENS].to_numpy(dtype=float)
    m = metrics_for(actual, pred)
    m["Subset"] = subset_name
    subset_rows.append(m)

subset_table = pd.DataFrame(subset_rows)
subset_cols = ["Subset", "N", "MAE", "RMSE", "R2",
               "Within_5_AQI_pct", "Within_10_AQI_pct", "Within_20_AQI_pct"]
subset_table = subset_table[[c for c in subset_cols if c in subset_table.columns]]

# 5. Fair paired comparison against persistence, on common support only.
common_by_model = {
    model: metrics_for(x.loc[common_mask, ACTUAL], x.loc[common_mask, pred_col])
    for model, pred_col in model_cols.items()
}
persistence_mae = common_by_model["Persistence"]["MAE"]
persistence_rmse = common_by_model["Persistence"]["RMSE"]

paired_rows = []
for model in ["Extra Trees", "Random Forest", "Weighted ensemble"]:
    m = common_by_model[model].copy()
    m["Model"] = model
    m["MAE_improvement_vs_persistence_pct"] = (
        (persistence_mae - m["MAE"]) / persistence_mae * 100
    )
    m["RMSE_improvement_vs_persistence_pct"] = (
        (persistence_rmse - m["RMSE"]) / persistence_rmse * 100
    )
    paired_rows.append(m)

paired_vs_persistence = pd.DataFrame(paired_rows)[
    ["Model", "N", "MAE", "RMSE", "R2",
     "Within_5_AQI_pct", "Within_10_AQI_pct", "Within_20_AQI_pct",
     "MAE_improvement_vs_persistence_pct",
     "RMSE_improvement_vs_persistence_pct"]
]

# 6. Formula precision diagnostic. Use the documented rounded weights only.
ET_WEIGHT_ROUNDED = 0.594466
RF_WEIGHT_ROUNDED = 0.405534
rounded_formula = (
    ET_WEIGHT_ROUNDED * x[ET].to_numpy(dtype=float)
    + RF_WEIGHT_ROUNDED * x[RF].to_numpy(dtype=float)
)
stored_ensemble = x[ENS].to_numpy(dtype=float)
formula_mask = finite[[ET, RF, ENS]].all(axis=1).to_numpy()

formula_diff = np.abs(rounded_formula[formula_mask] - stored_ensemble[formula_mask])
formula_summary = {
    "ET_weight_used_for_diagnostic": ET_WEIGHT_ROUNDED,
    "RF_weight_used_for_diagnostic": RF_WEIGHT_ROUNDED,
    "weights_sum": ET_WEIGHT_ROUNDED + RF_WEIGHT_ROUNDED,
    "rows_checked": int(formula_mask.sum()),
    "max_absolute_difference": float(formula_diff.max()) if len(formula_diff) else None,
    "mean_absolute_difference": float(formula_diff.mean()) if len(formula_diff) else None,
    "pass_at_1e-5_tolerance": bool(
        len(formula_diff) and formula_diff.max() <= 1e-5
    ),
    "note": (
        "This is a precision diagnostic with displayed rounded weights, "
        "not permission to refit or alter the frozen ensemble."
    )
}

# 7. Search small JSON reports for previously recorded holdout metrics / full weights.
# This is a text audit only; it does not use the holdout to select a model.
search_roots = [ROOT / "outputs", ROOT / "deployment"]
needles = ["3.216121", "7.658390", "0.996634", "0.594466", "0.405534"]
found_references = []

for search_root in search_roots:
    if not search_root.exists():
        continue
    for p in search_root.rglob("*.json"):
        try:
            if p.stat().st_size > 2_000_000:
                continue
            raw = p.read_text(encoding="utf-8", errors="ignore")
            hits = [needle for needle in needles if needle in raw]
            if hits:
                found_references.append({
                    "path": str(p),
                    "matched_values": "; ".join(hits),
                    "excerpt": raw[:500].replace("\n", " ")
                })
        except (OSError, PermissionError):
            pass

references_df = pd.DataFrame(found_references)

# 8. Save tables and audit report.
full_support.to_csv(OUT / "phase17_07C_R2_model_specific_full_support.csv", index=False)
common_support.to_csv(OUT / "phase17_07C_R2_all_models_common_support.csv", index=False)
paired_vs_persistence.to_csv(
    OUT / "phase17_07C_R2_paired_vs_persistence.csv", index=False
)
subset_table.to_csv(OUT / "phase17_07C_R2_ensemble_missing_persistence_sensitivity.csv", index=False)
references_df.to_csv(OUT / "phase17_07C_R2_prior_report_reference_search.csv", index=False)

report = {
    "phase": "17.07C-R2",
    "source_holdout_rows": int(len(df)),
    "all_model_common_rows": int(common_mask.sum()),
    "rows_with_missing_persistence": int((~finite[PERSIST]).sum()),
    "ensemble_rows_excluded_from_common_support": int(
        persistence_missing_but_ensemble_valid.sum()
    ),
    "model_specific_full_support_metrics": full_support.to_dict(orient="records"),
    "common_support_metrics": common_support.to_dict(orient="records"),
    "paired_vs_persistence": paired_vs_persistence.to_dict(orient="records"),
    "ensemble_subset_sensitivity": subset_table.to_dict(orient="records"),
    "rounded_weight_formula_diagnostic": formula_summary,
    "prior_report_references_found": int(len(references_df)),
    "models_retrained": False,
    "production_enabled": False,
}
(OUT / "phase17_07C_R2_support_reconciliation.json").write_text(
    json.dumps(report, indent=2, allow_nan=False), encoding="utf-8"
)

print("\n[1] MODEL-SPECIFIC FULL SUPPORT")
print(full_support.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

print("\n[2] ALL MODELS ON IDENTICAL COMMON SUPPORT")
print(common_support.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

print("\n[3] PAIRED IMPROVEMENT VS PERSISTENCE — COMMON SUPPORT")
print(paired_vs_persistence.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

print("\n[4] ENSEMBLE SENSITIVITY TO THE 67 MISSING-PERSISTENCE ROWS")
print(subset_table.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

print("\n[5] ROUNDED-WEIGHT FORMULA DIAGNOSTIC")
print(json.dumps(formula_summary, indent=2))

print("\n[6] PRIOR METRIC / WEIGHT REPORT REFERENCES")
if len(references_df):
    print(references_df[["path", "matched_values"]].to_string(index=False))
else:
    print("No matching small JSON reports found by the exact-value scan.")

print("\nSaved reconciliation folder:")
print(OUT)
print("Models retrained: False")
print("Production enabled: False")
print("=" * 76)

In [ ]:
# PHASE 17.07D — INVESTIGATE MISSING PERSISTENCE / LARGE ENSEMBLE ERRORS

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 76)
print("PHASE 17.07D — DIFFICULT HOLDOUT CASES")
print("=" * 76)

# 1. Resolve the existing project directory safely.
ROOT_CANDIDATES = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in ROOT_CANDIDATES if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the existing Delhi-AQI-Prediction folder.")

HOLDOUT_PATH = (
    ROOT / "outputs/reports/phase15_12_final_holdout/"
    "phase15_12_final_holdout_predictions.parquet"
)
FEATURE_PATH = ROOT / "data/processed/delhi_10_station_feature_engineered.parquet"

for p in [HOLDOUT_PATH, FEATURE_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required artifact not found: {p}")

OUT = ROOT / "outputs/research_paper/phase17_07D_difficult_case_audit"
OUT.mkdir(parents=True, exist_ok=True)

# 2. Load the frozen holdout predictions and identify missing persistence values.
pred = pd.read_parquet(HOLDOUT_PATH)

required = [
    "Datetime", "StationId", "AQI_t_plus_1",
    "Tuned_Extra_Trees_Prediction",
    "Tuned_Random_Forest_Prediction",
    "Final_Ensemble_Prediction",
    "Persistence_Prediction",
]
missing_columns = [c for c in required if c not in pred.columns]
if missing_columns:
    raise KeyError(f"Holdout file is missing columns: {missing_columns}")

pred["Datetime"] = pd.to_datetime(pred["Datetime"], errors="coerce")
pred["StationId"] = pred["StationId"].astype("string")

for c in required[2:]:
    pred[c] = pd.to_numeric(pred[c], errors="coerce")

missing_persistence = pred[
    pred["Persistence_Prediction"].isna()
    | ~np.isfinite(pred["Persistence_Prediction"].fillna(np.nan))
].copy()

missing_persistence["Ensemble_Error"] = (
    missing_persistence["Final_Ensemble_Prediction"]
    - missing_persistence["AQI_t_plus_1"]
)
missing_persistence["Ensemble_Absolute_Error"] = (
    missing_persistence["Ensemble_Error"].abs()
)

print(f"\n[1] Holdout rows: {len(pred):,}")
print(f"Rows with missing persistence prediction: {len(missing_persistence):,}")
if len(missing_persistence) == 0:
    raise RuntimeError("No missing persistence rows found; review the source artifact.")

# 3. Inspect AQI-related fields in the saved feature matrix.
features = pd.read_parquet(FEATURE_PATH)

for c in ["Datetime", "StationId"]:
    if c not in features.columns:
        raise KeyError(f"Feature matrix has no required join key: {c}")

features["Datetime"] = pd.to_datetime(features["Datetime"], errors="coerce")
features["StationId"] = features["StationId"].astype("string")

aqi_feature_cols = [
    c for c in features.columns
    if "AQI" in c.upper()
    and c not in {"Datetime", "StationId", "AQI_t_plus_1"}
]

# Keep the diagnostic join compact.
feature_subset = features[["Datetime", "StationId"] + aqi_feature_cols].copy()
feature_subset = feature_subset.rename(
    columns={c: f"feature__{c}" for c in aqi_feature_cols}
)

# Check key uniqueness before joining; never silently deduplicate source data.
duplicate_keys = int(
    feature_subset.duplicated(["Datetime", "StationId"], keep=False).sum()
)
if duplicate_keys:
    raise ValueError(
        f"Feature matrix contains {duplicate_keys} rows with duplicate station-time keys. "
        "Inspect keys before proceeding with the join."
    )

diagnostic = missing_persistence.merge(
    feature_subset,
    on=["Datetime", "StationId"],
    how="left",
    validate="many_to_one",
    indicator=True
)

print(f"Feature-matrix rows matched: {(diagnostic['_merge'] == 'both').sum():,}")
print(f"Feature-matrix rows unmatched: {(diagnostic['_merge'] != 'both').sum():,}")
print("\nAQI-like feature columns found:")
print(aqi_feature_cols if aqi_feature_cols else "None found")

# 4. Summarize errors, station concentration, and timing.
diagnostic["Target_Actual_AQI"] = diagnostic["AQI_t_plus_1"]
diagnostic["Ensemble_Prediction"] = diagnostic["Final_Ensemble_Prediction"]

print("\n[2] Actual target and ensemble error on missing-persistence rows")
summary_cols = [
    "AQI_t_plus_1", "Final_Ensemble_Prediction",
    "Ensemble_Error", "Ensemble_Absolute_Error"
]
print(diagnostic[summary_cols].describe().to_string())

print("\n[3] Counts and ensemble error by station")
station_summary = (
    diagnostic.groupby("StationId", dropna=False)
    .agg(
        Rows=("AQI_t_plus_1", "size"),
        Mean_Actual_AQI=("AQI_t_plus_1", "mean"),
        Mean_Absolute_Error=("Ensemble_Absolute_Error", "mean"),
        Max_Absolute_Error=("Ensemble_Absolute_Error", "max"),
    )
    .reset_index()
    .sort_values(["Rows", "Mean_Absolute_Error"], ascending=False)
)
print(station_summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\n[4] Largest ensemble errors among the missing-persistence rows")
show_cols = [
    "Datetime", "StationId", "AQI_t_plus_1",
    "Final_Ensemble_Prediction", "Ensemble_Error",
    "Ensemble_Absolute_Error", "_merge"
]
show_cols += [
    f"feature__{c}" for c in aqi_feature_cols
    if f"feature__{c}" in diagnostic.columns
]
largest_errors = diagnostic.sort_values(
    "Ensemble_Absolute_Error", ascending=False
).head(20)

print(largest_errors[show_cols].to_string(index=False))

# 5. Save detailed row-level and station-level audits.
diagnostic.drop(columns=["_merge"]).to_csv(
    OUT / "phase17_07D_missing_persistence_case_details.csv", index=False
)
station_summary.to_csv(
    OUT / "phase17_07D_station_summary.csv", index=False
)
largest_errors[show_cols].to_csv(
    OUT / "phase17_07D_largest_ensemble_errors.csv", index=False
)

report = {
    "phase": "17.07D",
    "holdout_rows": int(len(pred)),
    "rows_missing_persistence": int(len(missing_persistence)),
    "feature_matrix_rows_matched": int((diagnostic["_merge"] == "both").sum()),
    "feature_matrix_rows_unmatched": int((diagnostic["_merge"] != "both").sum()),
    "aqi_feature_columns": aqi_feature_cols,
    "missing_persistence_station_counts": {
        str(k): int(v)
        for k, v in diagnostic["StationId"].value_counts(dropna=False).items()
    },
    "missing_persistence_ensemble_mae": float(
        diagnostic["Ensemble_Absolute_Error"].mean()
    ),
    "missing_persistence_ensemble_rmse": float(
        np.sqrt(np.mean(diagnostic["Ensemble_Error"].dropna() ** 2))
    ),
    "models_retrained": False,
    "holdout_used_for_model_selection": False,
    "production_enabled": False,
}
(OUT / "phase17_07D_case_audit_report.json").write_text(
    json.dumps(report, indent=2, allow_nan=False), encoding="utf-8"
)

print("\nSaved audit folder:")
print(OUT)
print("Models retrained: False")
print("Holdout used for model selection: False")
print("Production enabled: False")
print("=" * 76)

In [ ]:
# PHASE 17.07E — FROZEN-FEATURE MISSINGNESS AUDIT

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 76)
print("PHASE 17.07E — FEATURE MISSINGNESS AUDIT")
print("=" * 76)

# 1. Locate the existing project and frozen artifacts.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next((p for p in roots if p.exists() and (p / "outputs").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project folder not found.")

PRED_PATH = (
    ROOT / "outputs/reports/phase15_12_final_holdout/"
    "phase15_12_final_holdout_predictions.parquet"
)
FEATURE_PATH = ROOT / "data/processed/delhi_10_station_feature_engineered.parquet"
CONTRACT_PATH = (
    ROOT / "deployment/phase16_33_frozen_model_package/config/feature_contract.json"
)

for p in [PRED_PATH, FEATURE_PATH, CONTRACT_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required artifact missing: {p}")

OUT = ROOT / "outputs/research_paper/phase17_07E_feature_missingness_audit"
OUT.mkdir(parents=True, exist_ok=True)

# 2. Read the frozen feature contract and discover its 73 feature names.
contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
candidates = []

def walk_contract(obj, path="root"):
    if isinstance(obj, dict):
        for k, v in obj.items():
            walk_contract(v, f"{path}.{k}")
    elif isinstance(obj, list):
        if (
            len(obj) == 73
            and all(isinstance(v, str) for v in obj)
            and "StationId" in obj
        ):
            candidates.append((path, obj))
        for i, v in enumerate(obj):
            if isinstance(v, (dict, list)):
                walk_contract(v, f"{path}[{i}]")

walk_contract(contract)

if not candidates:
    raise ValueError(
        "Could not automatically identify the 73-feature list in feature_contract.json. "
        f"Top-level contract type: {type(contract).__name__}; "
        f"keys: {list(contract.keys()) if isinstance(contract, dict) else 'not a dict'}"
    )

# Prefer a contract field explicitly named like a feature list.
preferred = [
    item for item in candidates
    if any(term in item[0].lower() for term in ["feature", "input", "column"])
]
contract_path, FEATURE_COLS = (preferred or candidates)[0]
if len(FEATURE_COLS) != 73 or len(set(FEATURE_COLS)) != 73:
    raise ValueError("The discovered frozen feature list is not 73 unique features.")

print(f"[1] Frozen contract list: {contract_path}")
print(f"Feature count: {len(FEATURE_COLS)}")

# 3. Load artifacts and standardize join keys.
pred = pd.read_parquet(PRED_PATH)
features = pd.read_parquet(FEATURE_PATH)

for name, frame in [("holdout predictions", pred), ("feature matrix", features)]:
    missing_keys = {"Datetime", "StationId"} - set(frame.columns)
    if missing_keys:
        raise KeyError(f"{name} lacks join keys: {sorted(missing_keys)}")

pred["Datetime"] = pd.to_datetime(pred["Datetime"], errors="coerce")
features["Datetime"] = pd.to_datetime(features["Datetime"], errors="coerce")
pred["StationId"] = pred["StationId"].astype("string")
features["StationId"] = features["StationId"].astype("string")

missing_features = [c for c in FEATURE_COLS if c not in features.columns]
if missing_features:
    raise KeyError(
        f"{len(missing_features)} frozen features not found in the feature matrix: "
        f"{missing_features}"
    )

if features.duplicated(["Datetime", "StationId"]).any():
    raise ValueError("Feature matrix has duplicate station-time keys; stop for review.")

# Use only the keys and the exact frozen input features.
frozen_inputs = features[["Datetime", "StationId"] + FEATURE_COLS].copy()

# Validate feature rows for the entire holdout, not just the 67 cases.
audit = pred.merge(
    frozen_inputs,
    on=["Datetime", "StationId"],
    how="left",
    validate="many_to_one",
    indicator=True
)

matched = audit["_merge"].eq("both")
if not matched.all():
    print(f"WARNING: {int((~matched).sum())} holdout rows did not match the feature matrix.")

# 4. Count missing features per row, without filling any values.
feature_block = audit[FEATURE_COLS]
missing_mask = feature_block.isna()

audit["Missing_Feature_Count"] = missing_mask.sum(axis=1)
audit["Missing_Feature_Pct"] = audit["Missing_Feature_Count"] / len(FEATURE_COLS) * 100

aqi_features = [c for c in FEATURE_COLS if c.upper().startswith("AQI")]
if "AQI" not in FEATURE_COLS:
    raise ValueError("The frozen feature contract does not contain current AQI ('AQI').")

audit["Missing_AQI_Feature_Count"] = (
    missing_mask[aqi_features].sum(axis=1) if aqi_features else 0
)
audit["Current_AQI_Missing"] = audit["AQI"].isna()

persist_missing = (
    audit["Persistence_Prediction"].isna()
    | ~np.isfinite(
        pd.to_numeric(audit["Persistence_Prediction"], errors="coerce")
        .to_numpy(dtype=float, na_value=np.nan)
    )
)
hard = audit.loc[persist_missing].copy()

actual_col = "AQI_t_plus_1"
ensemble_col = "Final_Ensemble_Prediction"
hard["Ensemble_Error"] = hard[ensemble_col] - hard[actual_col]
hard["Ensemble_Absolute_Error"] = hard["Ensemble_Error"].abs()

# 5. Compare each feature's missingness in the 67 cases versus all holdout rows.
feature_rows = []
for c in FEATURE_COLS:
    overall_rate = float(audit[c].isna().mean() * 100)
    hard_rate = float(hard[c].isna().mean() * 100) if len(hard) else np.nan
    feature_rows.append({
        "Feature": c,
        "Missing_Overall_N": int(audit[c].isna().sum()),
        "Missing_Overall_Pct": overall_rate,
        "Missing_Hard_Cases_N": int(hard[c].isna().sum()),
        "Missing_Hard_Cases_Pct": hard_rate,
        "Hard_Case_Missingness_Excess_pp": hard_rate - overall_rate,
    })

feature_missingness = pd.DataFrame(feature_rows).sort_values(
    "Hard_Case_Missingness_Excess_pp", ascending=False
)

# 6. Check error severity against the number of missing input features.
bins = [-1, 0, 5, 10, 20, 35, 73]
labels = ["0 missing", "1–5 missing", "6–10 missing",
          "11–20 missing", "21–35 missing", "36–73 missing"]
audit["Missing_Feature_Band"] = pd.cut(
    audit["Missing_Feature_Count"], bins=bins, labels=labels
)

severity_rows = []
for band, g in audit.groupby("Missing_Feature_Band", observed=False):
    g = g.dropna(subset=[actual_col, ensemble_col])
    if len(g) == 0:
        continue
    err = (
        pd.to_numeric(g[ensemble_col], errors="coerce")
        - pd.to_numeric(g[actual_col], errors="coerce")
    ).to_numpy(dtype=float)
    err = err[np.isfinite(err)]
    if len(err) == 0:
        continue
    severity_rows.append({
        "Missing_Feature_Band": str(band),
        "N": int(len(err)),
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "Mean_Missing_Feature_Count": float(g["Missing_Feature_Count"].mean()),
    })

severity = pd.DataFrame(severity_rows)

# 7. Summarize whether current AQI is missing in the difficult cases.
current_aqi_summary = {
    "holdout_rows": int(len(audit)),
    "feature_rows_matched": int(matched.sum()),
    "feature_rows_unmatched": int((~matched).sum()),
    "frozen_feature_count": int(len(FEATURE_COLS)),
    "rows_missing_persistence": int(len(hard)),
    "hard_cases_current_AQI_missing": int(hard["AQI"].isna().sum()),
    "hard_cases_current_AQI_missing_pct": (
        float(hard["AQI"].isna().mean() * 100) if len(hard) else None
    ),
    "hard_cases_mean_missing_features": (
        float(hard["Missing_Feature_Count"].mean()) if len(hard) else None
    ),
    "hard_cases_median_missing_features": (
        float(hard["Missing_Feature_Count"].median()) if len(hard) else None
    ),
    "holdout_mean_missing_features": float(audit["Missing_Feature_Count"].mean()),
    "holdout_median_missing_features": float(audit["Missing_Feature_Count"].median()),
    "models_retrained": False,
    "feature_values_imputed": False,
    "holdout_used_for_model_selection": False,
    "production_enabled": False,
}

# 8. Save detailed, reproducible audits.
feature_missingness.to_csv(
    OUT / "phase17_07E_feature_missingness_comparison.csv", index=False
)
severity.to_csv(
    OUT / "phase17_07E_error_by_missing_feature_band.csv", index=False
)

key_cols = [
    "Datetime", "StationId", actual_col, ensemble_col,
    "Persistence_Prediction", "Ensemble_Error", "Ensemble_Absolute_Error",
    "Missing_Feature_Count", "Missing_Feature_Pct",
    "Missing_AQI_Feature_Count", "Current_AQI_Missing", "_merge"
]
hard[key_cols + FEATURE_COLS].to_csv(
    OUT / "phase17_07E_difficult_case_feature_values.csv", index=False
)

(OUT / "phase17_07E_missingness_report.json").write_text(
    json.dumps(current_aqi_summary, indent=2, allow_nan=False),
    encoding="utf-8"
)

print("\n[2] MISSINGNESS SUMMARY")
print(json.dumps(current_aqi_summary, indent=2))

print("\n[3] FEATURES WITH THE LARGEST EXCESS MISSINGNESS IN THE 67 CASES")
print(
    feature_missingness.head(20).to_string(
        index=False, float_format=lambda v: f"{v:.3f}"
    )
)

print("\n[4] ENSEMBLE ERROR BY NUMBER OF MISSING INPUT FEATURES")
if len(severity):
    print(severity.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
else:
    print("No valid error groups were available.")

print("\n[5] CURRENT AQI AND SELECTED LAGS IN THE DIFFICULT CASES")
inspect_cols = [
    c for c in [
        "Datetime", "StationId", actual_col, ensemble_col,
        "AQI", "AQI_lag_1h", "AQI_lag_2h", "AQI_lag_12h",
        "AQI_rolling_mean_3h", "AQI_rolling_mean_12h",
        "Missing_Feature_Count", "Ensemble_Absolute_Error"
    ] if c in hard.columns
]
print(
    hard.sort_values("Ensemble_Absolute_Error", ascending=False)[inspect_cols]
    .head(15)
    .to_string(index=False)
)

print("\nSaved audit folder:")
print(OUT)
print("Models retrained: False")
print("Feature values imputed: False")
print("Production enabled: False")
print("=" * 76)

In [ ]:
# PHASE 17.07E-R1 — CORRECTED FROZEN-FEATURE MISSINGNESS AUDIT

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 76)
print("PHASE 17.07E-R1 — FEATURE MISSINGNESS AUDIT")
print("=" * 76)

# 1. Locate the existing project and artifacts.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in roots if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project folder not found.")

PRED_PATH = (
    ROOT / "outputs/reports/phase15_12_final_holdout/"
    "phase15_12_final_holdout_predictions.parquet"
)
FEATURE_PATH = ROOT / "data/processed/delhi_10_station_feature_engineered.parquet"
CONTRACT_PATH = (
    ROOT / "deployment/phase16_33_frozen_model_package/config/feature_contract.json"
)

for path in [PRED_PATH, FEATURE_PATH, CONTRACT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact missing: {path}")

OUT = ROOT / "outputs/research_paper/phase17_07E_R1_feature_missingness_audit"
OUT.mkdir(parents=True, exist_ok=True)

# 2. Read the exact frozen 73-feature list.
contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
candidates = []

def walk_contract(obj, path="root"):
    if isinstance(obj, dict):
        for key, value in obj.items():
            walk_contract(value, f"{path}.{key}")
    elif isinstance(obj, list):
        if (
            len(obj) == 73
            and all(isinstance(value, str) for value in obj)
            and "StationId" in obj
        ):
            candidates.append((path, obj))
        for i, value in enumerate(obj):
            if isinstance(value, (dict, list)):
                walk_contract(value, f"{path}[{i}]")

walk_contract(contract)

if not candidates:
    raise ValueError("Could not locate the frozen 73-feature list.")

preferred = [
    item for item in candidates
    if any(s in item[0].lower() for s in ["feature", "input", "column"])
]
contract_path, FEATURE_COLS = (preferred or candidates)[0]

if len(FEATURE_COLS) != 73 or len(set(FEATURE_COLS)) != 73:
    raise ValueError("Frozen feature list must contain 73 unique feature names.")

if "StationId" not in FEATURE_COLS:
    raise ValueError("StationId is unexpectedly absent from the frozen contract.")

print(f"[1] Contract list: {contract_path}")
print(f"Frozen features: {len(FEATURE_COLS)}")
print(f"StationId feature position: {FEATURE_COLS.index('StationId') + 1}")

# 3. Load artifacts and validate column names.
pred = pd.read_parquet(PRED_PATH)
features = pd.read_parquet(FEATURE_PATH)

for label, frame in [("holdout predictions", pred), ("feature matrix", features)]:
    if not frame.columns.is_unique:
        duplicates = frame.columns[frame.columns.duplicated()].tolist()
        raise ValueError(
            f"{label} itself contains duplicate column labels: {duplicates}"
        )

required_pred_cols = [
    "Datetime", "StationId", "AQI_t_plus_1",
    "Tuned_Extra_Trees_Prediction",
    "Tuned_Random_Forest_Prediction",
    "Final_Ensemble_Prediction",
    "Persistence_Prediction",
]

for label, frame, required in [
    ("predictions", pred, required_pred_cols),
    ("feature matrix", features, ["Datetime", "StationId"] + FEATURE_COLS),
]:
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise KeyError(f"{label} is missing required columns: {missing}")

pred["Datetime"] = pd.to_datetime(pred["Datetime"], errors="coerce")
features["Datetime"] = pd.to_datetime(features["Datetime"], errors="coerce")
pred["StationId"] = pred["StationId"].astype("string")
features["StationId"] = features["StationId"].astype("string")

for col in required_pred_cols[2:]:
    pred[col] = pd.to_numeric(pred[col], errors="coerce")

# 4. CRITICAL FIX:
# StationId is already in FEATURE_COLS, so select it only once.
# Datetime is a join key, not one of the 73 model features.
right_cols = ["Datetime"] + [
    col for col in FEATURE_COLS if col != "Datetime"
]
if len(right_cols) != 74 or len(set(right_cols)) != 74:
    raise ValueError(
        f"Expected 74 unique columns (Datetime plus 73 features); got {len(right_cols)}."
    )

frozen_inputs = features[right_cols].copy()

if frozen_inputs.duplicated(["Datetime", "StationId"]).any():
    duplicate_count = int(
        frozen_inputs.duplicated(["Datetime", "StationId"], keep=False).sum()
    )
    raise ValueError(
        f"Feature matrix has {duplicate_count} rows with duplicate station-time keys."
    )

# Use only the required holdout columns to avoid accidental column-name collisions.
pred_base = pred[required_pred_cols].copy()

audit = pred_base.merge(
    frozen_inputs,
    on=["Datetime", "StationId"],
    how="left",
    validate="many_to_one",
    indicator=True
)

matched = audit["_merge"].eq("both")
print(f"\n[2] Holdout rows: {len(audit):,}")
print(f"Feature rows matched: {int(matched.sum()):,}")
print(f"Feature rows unmatched: {int((~matched).sum()):,}")

# 5. Quantify missing inputs across the exact frozen feature contract.
missing_mask = audit[FEATURE_COLS].isna()
audit["Missing_Feature_Count"] = missing_mask.sum(axis=1)
audit["Missing_Feature_Pct"] = (
    audit["Missing_Feature_Count"] / len(FEATURE_COLS) * 100
)

aqi_features = [c for c in FEATURE_COLS if c.upper().startswith("AQI")]
audit["Missing_AQI_Feature_Count"] = (
    missing_mask[aqi_features].sum(axis=1) if aqi_features else 0
)
audit["Current_AQI_Missing"] = audit["AQI"].isna()

persistence = pd.to_numeric(
    audit["Persistence_Prediction"], errors="coerce"
)
persist_values = persistence.to_numpy(dtype=float, na_value=np.nan)
persistence_missing = ~np.isfinite(persist_values)

hard = audit.loc[persistence_missing].copy()
hard["Ensemble_Error"] = (
    hard["Final_Ensemble_Prediction"] - hard["AQI_t_plus_1"]
)
hard["Ensemble_Absolute_Error"] = hard["Ensemble_Error"].abs()

print(f"Rows missing persistence predictions: {len(hard):,}")

# 6. Compare feature-level missingness in the difficult rows versus the full holdout.
feature_rows = []
for col in FEATURE_COLS:
    overall_pct = float(audit[col].isna().mean() * 100)
    hard_pct = float(hard[col].isna().mean() * 100) if len(hard) else 0.0
    feature_rows.append({
        "Feature": col,
        "Missing_Overall_N": int(audit[col].isna().sum()),
        "Missing_Overall_Pct": overall_pct,
        "Missing_Hard_Cases_N": int(hard[col].isna().sum()),
        "Missing_Hard_Cases_Pct": hard_pct,
        "Hard_Case_Missingness_Excess_pp": hard_pct - overall_pct,
    })

feature_missingness = pd.DataFrame(feature_rows).sort_values(
    "Hard_Case_Missingness_Excess_pp", ascending=False
)

# 7. Assess errors by missing-feature count.
bins = [-1, 0, 5, 10, 20, 35, 73]
labels = [
    "0 missing", "1-5 missing", "6-10 missing",
    "11-20 missing", "21-35 missing", "36-73 missing"
]
audit["Missing_Feature_Band"] = pd.cut(
    audit["Missing_Feature_Count"], bins=bins, labels=labels
)

severity_rows = []
for band, group in audit.groupby("Missing_Feature_Band", observed=False):
    actual = pd.to_numeric(group["AQI_t_plus_1"], errors="coerce").to_numpy(
        dtype=float, na_value=np.nan
    )
    predicted = pd.to_numeric(
        group["Final_Ensemble_Prediction"], errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)

    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual, predicted = actual[valid], predicted[valid]
    if len(actual) == 0:
        continue

    error = predicted - actual
    severity_rows.append({
        "Missing_Feature_Band": str(band),
        "N": int(len(error)),
        "MAE": float(np.mean(np.abs(error))),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "Mean_Missing_Feature_Count": float(
            group["Missing_Feature_Count"].mean()
        ),
    })

severity = pd.DataFrame(severity_rows)

# 8. Summarize the difficult subset.
report = {
    "phase": "17.07E-R1",
    "contract_path": contract_path,
    "frozen_feature_count": int(len(FEATURE_COLS)),
    "holdout_rows": int(len(audit)),
    "feature_rows_matched": int(matched.sum()),
    "feature_rows_unmatched": int((~matched).sum()),
    "rows_missing_persistence": int(len(hard)),
    "hard_cases_current_AQI_missing": int(hard["AQI"].isna().sum()),
    "hard_cases_current_AQI_missing_pct": (
        float(hard["AQI"].isna().mean() * 100) if len(hard) else 0.0
    ),
    "hard_cases_mean_missing_features": (
        float(hard["Missing_Feature_Count"].mean()) if len(hard) else 0.0
    ),
    "hard_cases_median_missing_features": (
        float(hard["Missing_Feature_Count"].median()) if len(hard) else 0.0
    ),
    "holdout_mean_missing_features": float(
        audit["Missing_Feature_Count"].mean()
    ),
    "holdout_median_missing_features": float(
        audit["Missing_Feature_Count"].median()
    ),
    "models_retrained": False,
    "feature_values_imputed": False,
    "holdout_used_for_model_selection": False,
    "production_enabled": False,
}

# 9. Save the audits without changing original project files.
feature_missingness.to_csv(
    OUT / "phase17_07E_R1_feature_missingness_comparison.csv", index=False
)
severity.to_csv(
    OUT / "phase17_07E_R1_error_by_missing_feature_band.csv", index=False
)

key_cols = [
    "Datetime", "StationId", "AQI_t_plus_1",
    "Final_Ensemble_Prediction", "Persistence_Prediction",
    "Ensemble_Error", "Ensemble_Absolute_Error",
    "Missing_Feature_Count", "Missing_Feature_Pct",
    "Missing_AQI_Feature_Count", "Current_AQI_Missing", "_merge"
]
hard_export_cols = list(dict.fromkeys(key_cols + FEATURE_COLS))
hard.loc[:, hard_export_cols].to_csv(
    OUT / "phase17_07E_R1_difficult_case_feature_values.csv", index=False
)

(OUT / "phase17_07E_R1_missingness_report.json").write_text(
    json.dumps(report, indent=2, allow_nan=False),
    encoding="utf-8"
)

print("\n[3] MISSINGNESS SUMMARY")
print(json.dumps(report, indent=2))

print("\n[4] FEATURES WITH HIGHEST EXCESS MISSINGNESS IN DIFFICULT CASES")
print(
    feature_missingness.head(20).to_string(
        index=False, float_format=lambda v: f"{v:.3f}"
    )
)

print("\n[5] ENSEMBLE ERROR BY NUMBER OF MISSING INPUT FEATURES")
if len(severity):
    print(severity.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
else:
    print("No valid error groups were available.")

print("\n[6] LARGEST ERRORS AND SELECTED AQI FEATURES")
inspect_cols = [
    c for c in [
        "Datetime", "StationId", "AQI_t_plus_1",
        "Final_Ensemble_Prediction", "AQI",
        "AQI_lag_1h", "AQI_lag_2h", "AQI_lag_12h",
        "AQI_rolling_mean_3h", "AQI_rolling_mean_12h",
        "Missing_Feature_Count", "Ensemble_Absolute_Error"
    ] if c in hard.columns
]
print(
    hard.sort_values("Ensemble_Absolute_Error", ascending=False)[inspect_cols]
    .head(15).to_string(index=False)
)

print("\nSaved audit folder:")
print(OUT)
print("Models retrained: False")
print("Feature values imputed: False")
print("Production enabled: False")
print("=" * 76)

In [ ]:
# PHASE 17.07F — STRATIFIED HOLDOUT AND ABRUPT-TRANSITION ANALYSIS

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

print("=" * 78)
print("PHASE 17.07F — STRATIFIED HOLDOUT AND TRANSITION ANALYSIS")
print("=" * 78)

# 1. Resolve existing project files safely.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next((p for p in roots if p.exists() and (p / "outputs").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Project folder not found.")

PRED_PATH = (
    ROOT / "outputs/reports/phase15_12_final_holdout/"
    "phase15_12_final_holdout_predictions.parquet"
)
FEATURE_PATH = ROOT / "data/processed/delhi_10_station_feature_engineered.parquet"

for p in [PRED_PATH, FEATURE_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required artifact not found: {p}")

OUT = ROOT / "outputs/research_paper/phase17_07F_stratified_holdout"
OUT.mkdir(parents=True, exist_ok=True)

pred = pd.read_parquet(PRED_PATH)
features = pd.read_parquet(FEATURE_PATH)

ACTUAL = "AQI_t_plus_1"
MODEL_COLS = {
    "Persistence": "Persistence_Prediction",
    "Extra Trees": "Tuned_Extra_Trees_Prediction",
    "Random Forest": "Tuned_Random_Forest_Prediction",
    "Weighted ensemble": "Final_Ensemble_Prediction",
}

required = ["Datetime", "StationId", ACTUAL] + list(MODEL_COLS.values())
missing = [c for c in required if c not in pred.columns]
if missing:
    raise KeyError(f"Missing columns in holdout prediction file: {missing}")

for c in ["Datetime", "StationId", "AQI"]:
    if c not in features.columns:
        raise KeyError(f"Feature matrix missing required column: {c}")

pred = pred[required].copy()
pred["Datetime"] = pd.to_datetime(pred["Datetime"], errors="coerce")
pred["StationId"] = pred["StationId"].astype("string")
features["Datetime"] = pd.to_datetime(features["Datetime"], errors="coerce")
features["StationId"] = features["StationId"].astype("string")
features["AQI"] = pd.to_numeric(features["AQI"], errors="coerce")

for c in [ACTUAL] + list(MODEL_COLS.values()):
    pred[c] = pd.to_numeric(pred[c], errors="coerce")

# The Datetime key represents time t; target is AQI at t+1 hour.
current_aqi = features[["Datetime", "StationId", "AQI"]].rename(
    columns={"AQI": "AQI_current"}
)

if current_aqi.duplicated(["Datetime", "StationId"]).any():
    raise ValueError("Duplicate station-time keys in feature matrix.")

df = pred.merge(
    current_aqi,
    on=["Datetime", "StationId"],
    how="left",
    validate="many_to_one",
    indicator=True
)

if not df["_merge"].eq("both").all():
    raise ValueError("Some holdout rows did not match the feature matrix.")

df.drop(columns="_merge", inplace=True)
df["Target_Datetime"] = df["Datetime"] + pd.Timedelta(hours=1)
df["Target_Month"] = df["Target_Datetime"].dt.strftime("%Y-%m")
month = df["Target_Datetime"].dt.month

df["Season"] = np.select(
    [month.isin([12, 1, 2]),
     month.isin([3, 4, 5]),
     month.isin([6, 7, 8, 9]),
     month.isin([10, 11])],
    ["Winter", "Pre-monsoon", "Monsoon", "Post-monsoon"],
    default="Unknown"
)

df["Target_AQI_Band"] = pd.cut(
    df[ACTUAL],
    bins=[-np.inf, 100, 200, 300, 400, np.inf],
    labels=["<100", "100-199", "200-299", "300-399", ">=400"],
    right=False
)

# 2. Robust metrics; R2 is undefined for a constant target group.
def calculate_metrics(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    good = np.isfinite(y) & np.isfinite(yhat)
    y, yhat = y[good], yhat[good]

    if len(y) == 0:
        return {
            "N": 0, "MAE": None, "RMSE": None, "R2": None,
            "Within_5_AQI_pct": None,
            "Within_10_AQI_pct": None,
            "Within_20_AQI_pct": None,
        }

    error = yhat - y
    sse = float(np.sum(error ** 2))
    sst = float(np.sum((y - np.mean(y)) ** 2))
    r2 = float(1 - sse / sst) if len(y) > 1 and sst > 0 else None

    return {
        "N": int(len(y)),
        "MAE": float(np.mean(np.abs(error))),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "R2": r2,
        "Within_5_AQI_pct": float(np.mean(np.abs(error) <= 5) * 100),
        "Within_10_AQI_pct": float(np.mean(np.abs(error) <= 10) * 100),
        "Within_20_AQI_pct": float(np.mean(np.abs(error) <= 20) * 100),
    }

def metrics_by_group(frame, group_col, label):
    rows = []
    for group_name, g in frame.groupby(group_col, observed=False, dropna=False):
        for model, prediction_col in MODEL_COLS.items():
            m = calculate_metrics(g[ACTUAL], g[prediction_col])
            rows.append({
                "Grouping": label,
                "Group": str(group_name),
                "Model": model,
                **m
            })
    return pd.DataFrame(rows)

# 3. AQI-severity, station, season and monthly performance.
aqi_band_metrics = metrics_by_group(df, "Target_AQI_Band", "Target AQI band")
station_metrics = metrics_by_group(df, "StationId", "Station")
season_metrics = metrics_by_group(df, "Season", "Season")
month_metrics = metrics_by_group(df, "Target_Month", "Target month")

# 4. Actual and predicted AQI changes from t to t+1.
# Transition analysis necessarily excludes rows with missing current AQI.
df["Actual_Change"] = df[ACTUAL] - df["AQI_current"]

for model, pred_col in MODEL_COLS.items():
    df[f"{model}_Predicted_Change"] = df[pred_col] - df["AQI_current"]

delta = df["Actual_Change"]
df["Transition_Band"] = np.select(
    [
        delta < -20,
        (delta >= -20) & (delta < -5),
        (delta >= -5) & (delta <= 5),
        (delta > 5) & (delta <= 20),
        delta > 20
    ],
    [
        "Sharp improvement (<-20)",
        "Moderate improvement (-20 to -5)",
        "Stable (-5 to +5)",
        "Moderate deterioration (+5 to +20)",
        "Sharp deterioration (>+20)"
    ],
    default="Unknown"
)

transition_df = df[df["AQI_current"].notna() & df[ACTUAL].notna()].copy()
transition_metrics = metrics_by_group(
    transition_df, "Transition_Band", "Actual AQI change"
)

# 5. Measure large-transition detection and missed deteriorations.
transition_rows = []
for model in ["Extra Trees", "Random Forest", "Weighted ensemble", "Persistence"]:
    change_col = f"{model}_Predicted_Change"
    valid = transition_df[
        transition_df["Actual_Change"].notna()
        & transition_df[change_col].notna()
    ].copy()

    actual_change = valid["Actual_Change"].to_numpy(dtype=float)
    predicted_change = valid[change_col].to_numpy(dtype=float)

    for event, actual_event, predicted_event in [
        ("Sharp deterioration (>+20)", actual_change > 20, predicted_change > 20),
        ("Sharp improvement (<-20)", actual_change < -20, predicted_change < -20),
        ("Any abrupt change (absolute >20)",
         np.abs(actual_change) > 20, np.abs(predicted_change) > 20)
    ]:
        tp = int(np.sum(actual_event & predicted_event))
        actual_positive = int(np.sum(actual_event))
        predicted_positive = int(np.sum(predicted_event))
        transition_rows.append({
            "Model": model,
            "Event": event,
            "Eligible_Rows": int(len(valid)),
            "Actual_Event_Rows": actual_positive,
            "Predicted_Event_Rows": predicted_positive,
            "True_Positive": tp,
            "Missed_Events": int(actual_positive - tp),
            "Recall_pct": (
                float(tp / actual_positive * 100) if actual_positive else None
            ),
            "Precision_pct": (
                float(tp / predicted_positive * 100)
                if predicted_positive else None
            )
        })

transition_detection = pd.DataFrame(transition_rows)

# 6. Quantify concentration of the ensemble's squared error.
ensemble_error = df["Final_Ensemble_Prediction"] - df[ACTUAL]
squared_error = ensemble_error ** 2
overall_sse = float(squared_error.sum(skipna=True))
overall_n = int(squared_error.notna().sum())

top_n = max(1, math.ceil(0.025 * overall_n))
top_indices = squared_error.nlargest(top_n).index
top_sse = float(squared_error.loc[top_indices].sum())

missing_current_mask = df["AQI_current"].isna()
missing_current_sse = float(squared_error.loc[missing_current_mask].sum())

known_change = df["Actual_Change"].notna()
abrupt_mask = known_change & (df["Actual_Change"].abs() > 20)
abrupt_sse = float(squared_error.loc[abrupt_mask].sum())

sse_concentration = pd.DataFrame([
    {
        "Subset": "Top 2.5% by ensemble squared error",
        "Rows": int(len(top_indices)),
        "Share_of_all_holdout_rows_pct": float(len(top_indices) / overall_n * 100),
        "SSE": top_sse,
        "Share_of_total_ensemble_SSE_pct": float(top_sse / overall_sse * 100),
    },
    {
        "Subset": "Rows missing current AQI",
        "Rows": int(missing_current_mask.sum()),
        "Share_of_all_holdout_rows_pct": float(missing_current_mask.mean() * 100),
        "SSE": missing_current_sse,
        "Share_of_total_ensemble_SSE_pct": float(
            missing_current_sse / overall_sse * 100
        ),
    },
    {
        "Subset": "Rows with absolute actual change >20 AQI",
        "Rows": int(abrupt_mask.sum()),
        "Share_of_all_holdout_rows_pct": float(abrupt_mask.sum() / overall_n * 100),
        "SSE": abrupt_sse,
        "Share_of_total_ensemble_SSE_pct": float(abrupt_sse / overall_sse * 100),
    }
])

# 7. Save tables and summary report.
tables = {
    "aqi_band": aqi_band_metrics,
    "station": station_metrics,
    "season": season_metrics,
    "month": month_metrics,
    "transition": transition_metrics,
    "transition_detection": transition_detection,
    "sse_concentration": sse_concentration,
}

for name, table in tables.items():
    table.to_csv(OUT / f"phase17_07F_{name}_metrics.csv", index=False)

report = {
    "phase": "17.07F",
    "holdout_rows": int(len(df)),
    "rows_with_current_AQI": int(df["AQI_current"].notna().sum()),
    "rows_missing_current_AQI": int(df["AQI_current"].isna().sum()),
    "transition_analysis_rows": int(len(transition_df)),
    "top_2_5pct_error_rows": int(top_n),
    "total_ensemble_SSE": overall_sse,
    "sse_concentration": sse_concentration.to_dict(orient="records"),
    "models_retrained": False,
    "holdout_used_for_model_selection": False,
    "production_enabled": False,
}
(OUT / "phase17_07F_summary.json").write_text(
    json.dumps(report, indent=2, allow_nan=False), encoding="utf-8"
)

print("\n[1] PERFORMANCE BY TARGET AQI BAND")
print(aqi_band_metrics.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\n[2] PERFORMANCE BY STATION — ENSEMBLE ONLY")
print(
    station_metrics[station_metrics["Model"] == "Weighted ensemble"]
    .to_string(index=False, float_format=lambda v: f"{v:.3f}")
)

print("\n[3] PERFORMANCE BY SEASON — ENSEMBLE ONLY")
print(
    season_metrics[season_metrics["Model"] == "Weighted ensemble"]
    .to_string(index=False, float_format=lambda v: f"{v:.3f}")
)

print("\n[4] PERFORMANCE BY ACTUAL AQI CHANGE")
print(transition_metrics.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\n[5] LARGE-TRANSITION DETECTION")
print(transition_detection.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\n[6] ERROR CONCENTRATION")
print(sse_concentration.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\nSaved to:", OUT)
print("Models retrained: False")
print("Holdout used for model selection: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07G — PAIRED DAY-BLOCK BOOTSTRAP
# Descriptive uncertainty analysis only; no model selection or retraining.

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 78)
print("PHASE 17.07G — PAIRED DAY-BLOCK BOOTSTRAP")
print("=" * 78)

# 1. Locate and load the frozen holdout artifact.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next((p for p in roots if p.exists() and (p / "outputs").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project folder not found.")

HOLDOUT_PATH = (
    ROOT / "outputs/reports/phase15_12_final_holdout/"
    "phase15_12_final_holdout_predictions.parquet"
)
if not HOLDOUT_PATH.exists():
    raise FileNotFoundError(f"Frozen holdout predictions not found: {HOLDOUT_PATH}")

OUT = ROOT / "outputs/research_paper/phase17_07G_day_block_bootstrap"
OUT.mkdir(parents=True, exist_ok=True)

ACTUAL = "AQI_t_plus_1"
MODEL_COLS = {
    "Persistence": "Persistence_Prediction",
    "Extra Trees": "Tuned_Extra_Trees_Prediction",
    "Random Forest": "Tuned_Random_Forest_Prediction",
    "Weighted ensemble": "Final_Ensemble_Prediction",
}
PRED_COLS = list(MODEL_COLS.values())
NUMERIC_COLS = [ACTUAL] + PRED_COLS

df = pd.read_parquet(HOLDOUT_PATH)
required = ["Datetime", "StationId"] + NUMERIC_COLS
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Required columns missing from holdout file: {missing}")

df = df[required].copy()
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")

for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 2. Create a common complete-case support for a fair paired comparison.
numeric_values = df[NUMERIC_COLS].to_numpy(dtype=float)
complete_mask = (
    df["Datetime"].notna().to_numpy()
    & np.isfinite(numeric_values).all(axis=1)
)
common = df.loc[complete_mask].copy()

if common.empty:
    raise ValueError("No complete common-support rows are available.")

common["Day"] = common["Datetime"].dt.normalize()

print(f"[1] Original holdout rows: {len(df):,}")
print(f"Common complete-case rows: {len(common):,}")
print(f"Rows excluded from paired comparisons: {len(df) - len(common):,}")
print(f"Distinct day blocks: {common['Day'].nunique():,}")

if common["Day"].nunique() < 30:
    raise ValueError("Too few distinct day blocks for this planned bootstrap.")

# 3. Calculate each day's aggregate error contributions.
# Resampling whole days retains the within-day observations across stations.
daily_records = []

for day, g in common.groupby("Day", sort=True):
    y = g[ACTUAL].to_numpy(dtype=float)
    e_persist = g[MODEL_COLS["Persistence"]].to_numpy(dtype=float) - y
    abs_persist = np.abs(e_persist)

    record = {
        "Day": day,
        "N": int(len(g)),
        "Persistence_abs_sum": float(abs_persist.sum()),
        "Persistence_sq_sum": float(np.square(e_persist).sum()),
    }

    for tol in [5, 10, 20]:
        record[f"Persistence_within_{tol}"] = int((abs_persist <= tol).sum())

    for model in ["Extra Trees", "Random Forest", "Weighted ensemble"]:
        pred = g[MODEL_COLS[model]].to_numpy(dtype=float)
        error = pred - y
        abs_error = np.abs(error)

        record[f"{model}_abs_sum"] = float(abs_error.sum())
        record[f"{model}_sq_sum"] = float(np.square(error).sum())
        record[f"{model}_wins"] = int((abs_persist > abs_error).sum())

        for tol in [5, 10, 20]:
            record[f"{model}_within_{tol}"] = int((abs_error <= tol).sum())

    daily_records.append(record)

daily = pd.DataFrame(daily_records).sort_values("Day").reset_index(drop=True)

# 4. Point estimates from the entire matched support.
n_total = int(daily["N"].sum())
point_rows = []

for model in MODEL_COLS:
    abs_sum_col = (
        "Persistence_abs_sum" if model == "Persistence"
        else f"{model}_abs_sum"
    )
    sq_sum_col = (
        "Persistence_sq_sum" if model == "Persistence"
        else f"{model}_sq_sum"
    )

    mae = float(daily[abs_sum_col].sum() / n_total)
    rmse = float(np.sqrt(daily[sq_sum_col].sum() / n_total))

    row = {"Model": model, "N": n_total, "MAE": mae, "RMSE": rmse}
    for tol in [5, 10, 20]:
        within_col = f"Persistence_within_{tol}" if model == "Persistence" else f"{model}_within_{tol}"
        row[f"Within_{tol}_AQI_pct"] = float(daily[within_col].sum() / n_total * 100)
    point_rows.append(row)

point_metrics = pd.DataFrame(point_rows)

# 5. Bootstrap by resampling entire dates with replacement.
# Seed makes the confidence intervals reproducible.
N_BOOT = 5000
SEED = 20261023
rng = np.random.default_rng(SEED)

n_days = len(daily)
day_counts = daily["N"].to_numpy(dtype=float)

# Each bootstrap replicate samples n_days day blocks with replacement.
sampled_day_indices = rng.integers(
    low=0, high=n_days, size=(N_BOOT, n_days)
)
bootstrap_row_counts = day_counts[sampled_day_indices].sum(axis=1)

if np.any(bootstrap_row_counts <= 0):
    raise RuntimeError("A bootstrap replicate unexpectedly contains no rows.")

base_abs = daily["Persistence_abs_sum"].to_numpy(dtype=float)
base_sq = daily["Persistence_sq_sum"].to_numpy(dtype=float)
base_abs_boot = base_abs[sampled_day_indices].sum(axis=1)
base_sq_boot = base_sq[sampled_day_indices].sum(axis=1)

base_mae_boot = base_abs_boot / bootstrap_row_counts
base_rmse_boot = np.sqrt(base_sq_boot / bootstrap_row_counts)

def ci(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return [None, None]
    return [
        float(np.percentile(values, 2.5)),
        float(np.percentile(values, 97.5))
    ]

def get_model_point(model, metric):
    row = point_metrics.loc[point_metrics["Model"].eq(model)].iloc[0]
    return float(row[metric])

bootstrap_rows = []

for model in ["Extra Trees", "Random Forest", "Weighted ensemble"]:
    abs_sums = daily[f"{model}_abs_sum"].to_numpy(dtype=float)
    sq_sums = daily[f"{model}_sq_sum"].to_numpy(dtype=float)

    model_abs_boot = abs_sums[sampled_day_indices].sum(axis=1)
    model_sq_boot = sq_sums[sampled_day_indices].sum(axis=1)

    model_mae_boot = model_abs_boot / bootstrap_row_counts
    model_rmse_boot = np.sqrt(model_sq_boot / bootstrap_row_counts)

    # Positive deltas mean the model improves on persistence.
    delta_mae_boot = base_mae_boot - model_mae_boot
    delta_rmse_boot = base_rmse_boot - model_rmse_boot

    relative_mae_boot = (
        delta_mae_boot / np.maximum(base_mae_boot, 1e-12) * 100
    )
    relative_rmse_boot = (
        delta_rmse_boot / np.maximum(base_rmse_boot, 1e-12) * 100
    )

    wins_boot = (
        daily[f"{model}_wins"].to_numpy(dtype=float)[sampled_day_indices].sum(axis=1)
        / bootstrap_row_counts * 100
    )

    tol_differences = {}
    for tol in [5, 10, 20]:
        model_within = (
            daily[f"{model}_within_{tol}"].to_numpy(dtype=float)[sampled_day_indices]
            .sum(axis=1) / bootstrap_row_counts * 100
        )
        base_within = (
            daily[f"Persistence_within_{tol}"].to_numpy(dtype=float)[sampled_day_indices]
            .sum(axis=1) / bootstrap_row_counts * 100
        )
        tol_differences[tol] = model_within - base_within

    point_mae = get_model_point("Persistence", "MAE") - get_model_point(model, "MAE")
    point_rmse = get_model_point("Persistence", "RMSE") - get_model_point(model, "RMSE")

    point_persist_mae = get_model_point("Persistence", "MAE")
    point_persist_rmse = get_model_point("Persistence", "RMSE")

    bootstrap_rows.append({
        "Model": model,
        "N": n_total,
        "Day_Blocks": n_days,
        "Persistence_MAE": point_persist_mae,
        "Model_MAE": get_model_point(model, "MAE"),
        "Delta_MAE_Persistence_minus_Model": point_mae,
        "Delta_MAE_95CI_Lower": ci(delta_mae_boot)[0],
        "Delta_MAE_95CI_Upper": ci(delta_mae_boot)[1],
        "Relative_MAE_Improvement_pct": (
            point_mae / point_persist_mae * 100
        ),
        "Relative_MAE_Improvement_95CI_Lower_pct": ci(relative_mae_boot)[0],
        "Relative_MAE_Improvement_95CI_Upper_pct": ci(relative_mae_boot)[1],
        "Persistence_RMSE": point_persist_rmse,
        "Model_RMSE": get_model_point(model, "RMSE"),
        "Delta_RMSE_Persistence_minus_Model": point_rmse,
        "Delta_RMSE_95CI_Lower": ci(delta_rmse_boot)[0],
        "Delta_RMSE_95CI_Upper": ci(delta_rmse_boot)[1],
        "Relative_RMSE_Improvement_pct": (
            point_rmse / point_persist_rmse * 100
        ),
        "Relative_RMSE_Improvement_95CI_Lower_pct": ci(relative_rmse_boot)[0],
        "Relative_RMSE_Improvement_95CI_Upper_pct": ci(relative_rmse_boot)[1],
        "Bootstrap_Pct_Replicates_MAE_Improvement_Positive": float(
            np.mean(delta_mae_boot > 0) * 100
        ),
        "Bootstrap_Pct_Replicates_RMSE_Improvement_Positive": float(
            np.mean(delta_rmse_boot > 0) * 100
        ),
        "Pct_Rows_With_Lower_Absolute_Error": float(
            (daily[f"{model}_wins"].sum() / n_total) * 100
        ),
        "Pct_Rows_With_Lower_Absolute_Error_95CI_Lower": ci(wins_boot)[0],
        "Pct_Rows_With_Lower_Absolute_Error_95CI_Upper": ci(wins_boot)[1],
        "Delta_Within_5_AQI_percentage_points": float(
            get_model_point(model, "Within_5_AQI_pct")
            - get_model_point("Persistence", "Within_5_AQI_pct")
        ),
        "Delta_Within_5_AQI_95CI_Lower_pp": ci(tol_differences[5])[0],
        "Delta_Within_5_AQI_95CI_Upper_pp": ci(tol_differences[5])[1],
        "Delta_Within_10_AQI_percentage_points": float(
            get_model_point(model, "Within_10_AQI_pct")
            - get_model_point("Persistence", "Within_10_AQI_pct")
        ),
        "Delta_Within_10_AQI_95CI_Lower_pp": ci(tol_differences[10])[0],
        "Delta_Within_10_AQI_95CI_Upper_pp": ci(tol_differences[10])[1],
        "Delta_Within_20_AQI_percentage_points": float(
            get_model_point(model, "Within_20_AQI_pct")
            - get_model_point("Persistence", "Within_20_AQI_pct")
        ),
        "Delta_Within_20_AQI_95CI_Lower_pp": ci(tol_differences[20])[0],
        "Delta_Within_20_AQI_95CI_Upper_pp": ci(tol_differences[20])[1],
    })

bootstrap_results = pd.DataFrame(bootstrap_rows)

# 6. Save the reproducible tables and methodology metadata.
point_metrics.to_csv(
    OUT / "phase17_07G_common_support_point_metrics.csv", index=False
)
daily.to_csv(
    OUT / "phase17_07G_daily_error_contributions.csv", index=False
)
bootstrap_results.to_csv(
    OUT / "phase17_07G_paired_day_block_bootstrap_results.csv", index=False
)

report = {
    "phase": "17.07G",
    "bootstrap_method": (
        "Paired cluster bootstrap: resample distinct calendar days with replacement; "
        "retain all available station-hour rows within each selected day."
    ),
    "interpretation": (
        "Positive Delta_MAE_Persistence_minus_Model and "
        "Delta_RMSE_Persistence_minus_Model indicate improvement over persistence."
    ),
    "bootstrap_replicates": N_BOOT,
    "random_seed": SEED,
    "source_holdout_rows": int(len(df)),
    "common_support_rows": n_total,
    "excluded_rows": int(len(df) - len(common)),
    "day_blocks": int(n_days),
    "point_metrics": point_metrics.to_dict(orient="records"),
    "paired_comparisons": bootstrap_results.to_dict(orient="records"),
    "limitations": [
        "Day-block resampling preserves within-day clustering, but not dependence "
        "between adjacent days.",
        "Intervals describe uncertainty for this held-out period and do not establish "
        "performance on future live data.",
        "Holdout results were not used for model selection or tuning."
    ],
    "models_retrained": False,
    "holdout_used_for_model_selection": False,
    "production_enabled": False
}

(OUT / "phase17_07G_bootstrap_report.json").write_text(
    json.dumps(report, indent=2, allow_nan=False),
    encoding="utf-8"
)

print("\n[1] POINT METRICS ON THE IDENTICAL COMMON SUPPORT")
print(point_metrics.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\n[2] PAIRED DAY-BLOCK BOOTSTRAP — 95% CONFIDENCE INTERVALS")
display_cols = [
    "Model", "N",
    "Delta_MAE_Persistence_minus_Model",
    "Delta_MAE_95CI_Lower", "Delta_MAE_95CI_Upper",
    "Relative_MAE_Improvement_pct",
    "Relative_MAE_Improvement_95CI_Lower_pct",
    "Relative_MAE_Improvement_95CI_Upper_pct",
    "Delta_RMSE_Persistence_minus_Model",
    "Delta_RMSE_95CI_Lower", "Delta_RMSE_95CI_Upper",
    "Relative_RMSE_Improvement_pct",
    "Relative_RMSE_Improvement_95CI_Lower_pct",
    "Relative_RMSE_Improvement_95CI_Upper_pct",
    "Bootstrap_Pct_Replicates_MAE_Improvement_Positive",
    "Bootstrap_Pct_Replicates_RMSE_Improvement_Positive",
]
print(
    bootstrap_results[display_cols].to_string(
        index=False, float_format=lambda v: f"{v:.4f}"
    )
)

print("\n[3] PAIRED ACCURACY GAINS")
acc_cols = [
    "Model",
    "Pct_Rows_With_Lower_Absolute_Error",
    "Pct_Rows_With_Lower_Absolute_Error_95CI_Lower",
    "Pct_Rows_With_Lower_Absolute_Error_95CI_Upper",
    "Delta_Within_10_AQI_percentage_points",
    "Delta_Within_10_AQI_95CI_Lower_pp",
    "Delta_Within_10_AQI_95CI_Upper_pp",
]
print(
    bootstrap_results[acc_cols].to_string(
        index=False, float_format=lambda v: f"{v:.4f}"
    )
)

print("\nSaved results:", OUT)
print("Models retrained: False")
print("Holdout used for model selection: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07H — RECOVER MODEL COMPARISON, ABLATION AND XAI EVIDENCE
# Read-only artifact inventory and report inspection.

from pathlib import Path
import json
import pandas as pd

print("=" * 78)
print("PHASE 17.07H — RESEARCH EVIDENCE RECOVERY")
print("=" * 78)

# 1. Resolve the existing project root.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next((p for p in roots if p.exists() and (p / "outputs").exists()), None)

if ROOT is None:
    raise FileNotFoundError(
        "Delhi-AQI-Prediction not found. Check the existing Google Drive mount."
    )

OUT = ROOT / "outputs/research_paper/phase17_07H_evidence_recovery"
OUT.mkdir(parents=True, exist_ok=True)

# 2. Identify relevant saved research artifacts by filename.
keywords = [
    "ablation", "feature_importance", "importance", "permutation",
    "xai", "competition", "model_comparison", "baseline",
    "tuning", "feature_rank", "feature_selection", "validation"
]
allowed_extensions = {".json", ".csv", ".tsv", ".parquet"}

candidates = []
for path in ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in allowed_extensions:
        continue

    name = path.name.lower()
    if any(keyword in name for keyword in keywords):
        try:
            size = path.stat().st_size
        except OSError:
            continue

        candidates.append({
            "path": str(path),
            "relative_path": str(path.relative_to(ROOT)),
            "extension": path.suffix.lower(),
            "size_bytes": size,
            "matched_keywords": "; ".join(k for k in keywords if k in name),
        })

catalog = pd.DataFrame(candidates)
if len(catalog):
    catalog = catalog.sort_values(
        ["matched_keywords", "relative_path"]
    ).reset_index(drop=True)

catalog.to_csv(OUT / "phase17_07H_candidate_artifact_catalog.csv", index=False)

print(f"\n[1] Candidate files found: {len(catalog)}")
if len(catalog):
    print(catalog[["relative_path", "size_bytes", "matched_keywords"]]
          .to_string(index=False, max_rows=120))
else:
    print("No matching files were found by filename.")

# 3. Inspect small JSON reports and CSV/TSV table headers.
# Large files and general prediction Parquets are not opened.
MAX_JSON_BYTES = 3_000_000
MAX_TABLE_BYTES = 10_000_000
MAX_TABLES_TO_PRINT = 15

json_findings = []
table_findings = []
tables_printed = 0

def collect_scalar_fields(obj, prefix=""):
    """Collect scalar report fields without dumping entire JSON objects."""
    found = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            key_path = f"{prefix}.{key}" if prefix else str(key)
            if isinstance(value, (str, int, float, bool)) or value is None:
                found.append({
                    "field": key_path,
                    "value": value
                })
            elif isinstance(value, (dict, list)):
                found.extend(collect_scalar_fields(value, key_path))
    elif isinstance(obj, list):
        # Limit traversal of long arrays while keeping the report manageable.
        for i, value in enumerate(obj[:100]):
            if isinstance(value, (dict, list)):
                found.extend(collect_scalar_fields(value, f"{prefix}[{i}]"))
    return found

for record in catalog.to_dict(orient="records"):
    path = Path(record["path"])
    ext = record["extension"]
    size = record["size_bytes"]
    relative = record["relative_path"]

    # JSON: capture scalar evidence and relevant metric/feature values.
    if ext == ".json" and size <= MAX_JSON_BYTES:
        try:
            obj = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
            scalars = collect_scalar_fields(obj)

            relevant = [
                item for item in scalars
                if any(term in item["field"].lower() for term in [
                    "mae", "rmse", "r2", "feature", "importance",
                    "permutation", "ablation", "baseline", "weight",
                    "validation", "model", "metric", "rank", "column"
                ])
            ]

            json_findings.append({
                "relative_path": relative,
                "size_bytes": size,
                "relevant_scalar_field_count": len(relevant),
                "fields": relevant[:100]
            })
        except Exception as e:
            json_findings.append({
                "relative_path": relative,
                "size_bytes": size,
                "read_error": str(e)
            })

    # CSV/TSV: read a bounded sample only.
    elif ext in {".csv", ".tsv"} and size <= MAX_TABLE_BYTES:
        try:
            sep = "\t" if ext == ".tsv" else ","
            sample = pd.read_csv(path, sep=sep, nrows=12)
            table_findings.append({
                "relative_path": relative,
                "size_bytes": size,
                "columns": list(sample.columns),
                "sample_rows": sample.head(8).to_dict(orient="records")
            })

            if tables_printed < MAX_TABLES_TO_PRINT:
                print(f"\n[TABLE] {relative}")
                print("Columns:", list(sample.columns))
                if len(sample):
                    print(sample.head(8).to_string(index=False))
                else:
                    print("No data rows in sample.")
                tables_printed += 1
        except Exception as e:
            table_findings.append({
                "relative_path": relative,
                "size_bytes": size,
                "read_error": str(e)
            })

# 4. Save compact JSON evidence without changing any source artifacts.
(OUT / "phase17_07H_json_report_fields.json").write_text(
    json.dumps(json_findings, indent=2, default=str, allow_nan=False),
    encoding="utf-8"
)

(OUT / "phase17_07H_small_table_schemas_and_samples.json").write_text(
    json.dumps(table_findings, indent=2, default=str, allow_nan=False),
    encoding="utf-8"
)

# 5. Show especially relevant report summaries.
print("\n[2] MOST RELEVANT SAVED REPORTS")
for report in json_findings:
    relevant_fields = report.get("fields", [])
    if not relevant_fields:
        continue

    print(f"\n--- {report['relative_path']} ---")
    for item in relevant_fields[:30]:
        print(f"{item['field']}: {item['value']}")

print("\n[3] EVIDENCE FILES SAVED TO")
print(OUT)

print("\nSource Parquet prediction tables were not loaded by this inventory step.")
print("Models retrained: False")
print("Holdout used for model selection: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07H-R1 — SAFE JSON SAVE + RESEARCH EVIDENCE SUMMARY
# Reads existing artifacts only; converts NaN/Inf to JSON null during export.

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

print("=" * 78)
print("PHASE 17.07H-R1 — EVIDENCE RECOVERY REPAIR")
print("=" * 78)

# 1. Resolve the current project and existing evidence folder.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in roots if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Existing Delhi-AQI-Prediction project folder not found.")

OUT = ROOT / "outputs/research_paper/phase17_07H_evidence_recovery"
OUT.mkdir(parents=True, exist_ok=True)

catalog_path = OUT / "phase17_07H_candidate_artifact_catalog.csv"
if not catalog_path.exists():
    raise FileNotFoundError(
        f"Candidate catalog missing: {catalog_path}. "
        "Rerun the Phase 17.07H inventory cell first."
    )

catalog = pd.read_csv(catalog_path)

# 2. Make any sampled values safe for strict JSON.
def json_safe(value):
    if value is None or value is pd.NA or value is pd.NaT:
        return None

    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return json_safe(value.item())

    if isinstance(value, float):
        return value if math.isfinite(value) else None

    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}

    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]

    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())

    if isinstance(value, (str, int, bool)):
        return value

    # Handle other scalar missing values without applying pd.isna to containers.
    try:
        missing = pd.isna(value)
        if isinstance(missing, (bool, np.bool_)) and missing:
            return None
    except (TypeError, ValueError):
        pass

    return str(value)

# 3. Reconstruct compact JSON and table-sample evidence from the saved catalog.
def collect_scalar_fields(obj, prefix=""):
    found = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            key_path = f"{prefix}.{key}" if prefix else str(key)

            if isinstance(value, dict):
                found.extend(collect_scalar_fields(value, key_path))
            elif isinstance(value, list):
                found.extend(collect_scalar_fields(value, key_path))
            else:
                found.append({
                    "field": key_path,
                    "value": json_safe(value)
                })

    elif isinstance(obj, list):
        for i, value in enumerate(obj[:100]):
            if isinstance(value, (dict, list)):
                found.extend(collect_scalar_fields(value, f"{prefix}[{i}]"))

    return found

json_findings = []
table_findings = []

for record in catalog.to_dict(orient="records"):
    path = Path(record["path"])
    ext = str(record["extension"]).lower()
    size = int(record["size_bytes"])
    relative = record["relative_path"]

    if not path.exists():
        continue

    # Small JSON reports: collect relevant metric/configuration fields.
    if ext == ".json" and size <= 3_000_000:
        try:
            obj = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
            scalar_fields = collect_scalar_fields(obj)

            relevant = [
                item for item in scalar_fields
                if any(term in item["field"].lower() for term in [
                    "mae", "rmse", "r2", "feature", "importance",
                    "permutation", "ablation", "baseline", "weight",
                    "validation", "model", "metric", "rank"
                ])
            ]

            json_findings.append({
                "relative_path": relative,
                "size_bytes": size,
                "relevant_scalar_field_count": len(relevant),
                "fields": relevant[:120]
            })
        except Exception as exc:
            json_findings.append({
                "relative_path": relative,
                "size_bytes": size,
                "read_error": str(exc)
            })

    # Small CSV/TSV reports: save schema and a bounded sample.
    elif ext in {".csv", ".tsv"} and size <= 10_000_000:
        try:
            sep = "\t" if ext == ".tsv" else ","
            sample = pd.read_csv(path, sep=sep, nrows=12)

            table_findings.append({
                "relative_path": relative,
                "size_bytes": size,
                "columns": list(sample.columns),
                "sample_rows": sample.head(8).to_dict(orient="records")
            })
        except Exception as exc:
            table_findings.append({
                "relative_path": relative,
                "size_bytes": size,
                "read_error": str(exc)
            })

# 4. Save with NaN and Inf replaced by null. These are derived audit files only.
(OUT / "phase17_07H_json_report_fields.json").write_text(
    json.dumps(json_safe(json_findings), indent=2, allow_nan=False),
    encoding="utf-8"
)

(OUT / "phase17_07H_small_table_schemas_and_samples.json").write_text(
    json.dumps(json_safe(table_findings), indent=2, allow_nan=False),
    encoding="utf-8"
)

print(f"[1] Candidate artifact entries: {len(catalog)}")
print(f"Small JSON report summaries: {len(json_findings)}")
print(f"Small table summaries: {len(table_findings)}")
print("Strict JSON export: PASS")

# 5. Read key research reports individually; don't combine different evaluation scopes.
selected = [
    ("outputs/reports/phase15_7_final_model_comparison.csv", 20),
    ("outputs/FINAL_RESULTS/00_MASTER/model_comparison_master.csv", 20),
    ("outputs/FINAL_RESULTS/03_MODEL_COMPARISON/comprehensive_model_comparison.csv", 20),
    ("outputs/FINAL_RESULTS/01_TRAINING_AND_VALIDATION/validation_model_ranking.csv", 20),
    ("outputs/reports/phase15_10_ensemble_validation_summary.csv", 20),
    ("outputs/reports/phase15_8_top_two_validation_metrics.csv", 20),
    ("outputs/reports/phase15_9_ensemble_validation_results.csv", 20),
    ("outputs/reports/phase14_10_ablation_leaderboard.csv", 20),
    ("outputs/reports/phase15_11_fast_xai/phase15_11_grouped_feature_importance.csv", 25),
    ("outputs/reports/phase15_11_fast_xai/phase15_11_ensemble_permutation_importance.csv", 25),
    ("outputs/reports/phase15_11_fast_xai/phase15_11_extra_trees_native_importance.csv", 15),
    ("outputs/reports/phase15_11_fast_xai/phase15_11_random_forest_native_importance.csv", 15),
]

recovered_tables = []

print("\n[2] KEY SAVED TABLES")
for relative, max_rows in selected:
    path = ROOT / relative
    if not path.exists():
        print(f"\n[MISSING] {relative}")
        continue

    try:
        frame = pd.read_csv(path)
        recovered_tables.append({
            "relative_path": relative,
            "rows": int(len(frame)),
            "columns": list(frame.columns)
        })

        print(f"\n--- {relative} ---")
        print(f"Rows: {len(frame)} | Columns: {list(frame.columns)}")
        if len(frame):
            print(frame.head(max_rows).to_string(index=False))
        else:
            print("Table has no data rows.")
    except Exception as exc:
        print(f"\n[READ ERROR] {relative}: {exc}")

# 6. Summarize the saved fast-XAI report without assuming importance values.
xai_json_path = ROOT / "outputs/reports/phase15_11_fast_xai/phase15_11_fast_xai_summary.json"
if xai_json_path.exists():
    try:
        xai_report = json.loads(xai_json_path.read_text(encoding="utf-8"))
        (OUT / "phase17_07H_R1_fast_xai_source_summary.json").write_text(
            json.dumps(json_safe(xai_report), indent=2, allow_nan=False),
            encoding="utf-8"
        )
        print("\n[3] FAST-XAI SUMMARY TOP-LEVEL FIELDS")
        if isinstance(xai_report, dict):
            for key, value in xai_report.items():
                if isinstance(value, (dict, list)):
                    print(f"{key}: [{type(value).__name__}, {len(value)} items]")
                else:
                    print(f"{key}: {value}")
    except Exception as exc:
        print(f"Could not read fast-XAI summary: {exc}")

(OUT / "phase17_07H_R1_recovered_table_index.json").write_text(
    json.dumps(json_safe(recovered_tables), indent=2, allow_nan=False),
    encoding="utf-8"
)

print("\n[4] REPAIR COMPLETE")
print("Saved folder:", OUT)
print("Original reports modified: False")
print("Models retrained: False")
print("Holdout used for model selection: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07I — CONSOLIDATED RESEARCH PAPER EVIDENCE PACKAGE

from pathlib import Path
import json
import math
import shutil
import numpy as np
import pandas as pd

print("=" * 78)
print("PHASE 17.07I — CONSOLIDATED EVIDENCE PACKAGE")
print("=" * 78)

# 1. Resolve existing project files safely.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next((p for p in roots if p.exists() and (p / "outputs").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project folder not found.")

OUT = ROOT / "outputs/research_paper/phase17_07I_evidence_package"
TABLES = OUT / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

# 2. Source artifact locations.
SOURCES = {
    "validation_model_comparison": ROOT / "outputs/reports/phase15_7_final_model_comparison.csv",
    "validation_ensemble_comparison": ROOT / "outputs/reports/phase15_9_ensemble_validation_results.csv",
    "validation_ensemble_summary": ROOT / "outputs/reports/phase15_10_ensemble_validation_summary.csv",
    "holdout_full_support": ROOT / "outputs/research_paper/phase17_07C_R2_support_reconciliation/phase17_07C_R2_model_specific_full_support.csv",
    "holdout_common_support": ROOT / "outputs/research_paper/phase17_07C_R2_support_reconciliation/phase17_07C_R2_all_models_common_support.csv",
    "holdout_paired_persistence": ROOT / "outputs/research_paper/phase17_07C_R2_support_reconciliation/phase17_07C_R2_paired_vs_persistence.csv",
    "bootstrap_results": ROOT / "outputs/research_paper/phase17_07G_day_block_bootstrap/phase17_07G_paired_day_block_bootstrap_results.csv",
    "bootstrap_point_metrics": ROOT / "outputs/research_paper/phase17_07G_day_block_bootstrap/phase17_07G_common_support_point_metrics.csv",
    "permutation_importance": ROOT / "outputs/reports/phase15_11_fast_xai/phase15_11_ensemble_permutation_importance.csv",
    "grouped_importance": ROOT / "outputs/reports/phase15_11_fast_xai/phase15_11_grouped_feature_importance.csv",
    "historical_ablation": ROOT / "outputs/reports/phase14_10_ablation_leaderboard.csv",
    "holdout_predictions": ROOT / "outputs/reports/phase15_12_final_holdout/phase15_12_final_holdout_predictions.parquet",
    "feature_matrix": ROOT / "data/processed/delhi_10_station_feature_engineered.parquet",
}

required_sources = [
    "validation_model_comparison",
    "validation_ensemble_comparison",
    "validation_ensemble_summary",
    "holdout_full_support",
    "holdout_common_support",
    "holdout_paired_persistence",
    "bootstrap_results",
    "bootstrap_point_metrics",
    "permutation_importance",
    "grouped_importance",
    "historical_ablation",
    "holdout_predictions",
    "feature_matrix",
]
for name in required_sources:
    if not SOURCES[name].exists():
        raise FileNotFoundError(f"Required source artifact '{name}' missing: {SOURCES[name]}")

# 3. Copy existing report tables into the evidence package.
tables_index = {}
for name in [
    "validation_model_comparison",
    "validation_ensemble_comparison",
    "validation_ensemble_summary",
    "holdout_full_support",
    "holdout_common_support",
    "holdout_paired_persistence",
    "bootstrap_results",
    "bootstrap_point_metrics",
    "historical_ablation",
]:
    frame = pd.read_csv(SOURCES[name])
    dest = TABLES / f"{name}.csv"
    frame.to_csv(dest, index=False)
    tables_index[name] = {
        "path": str(dest),
        "rows": int(len(frame)),
        "columns": list(frame.columns),
        "source": str(SOURCES[name]),
    }

# 4. Make common-support subgroup metrics from the frozen holdout.
ACTUAL = "AQI_t_plus_1"
MODEL_COLS = {
    "Persistence": "Persistence_Prediction",
    "Extra Trees": "Tuned_Extra_Trees_Prediction",
    "Random Forest": "Tuned_Random_Forest_Prediction",
    "Weighted ensemble": "Final_Ensemble_Prediction",
}
PRED_COLS = list(MODEL_COLS.values())
NUMERIC_COLS = [ACTUAL] + PRED_COLS

pred = pd.read_parquet(SOURCES["holdout_predictions"])
features = pd.read_parquet(SOURCES["feature_matrix"])

pred_required = ["Datetime", "StationId"] + NUMERIC_COLS
for frame_name, frame, cols in [
    ("holdout predictions", pred, pred_required),
    ("feature matrix", features, ["Datetime", "StationId", "AQI"]),
]:
    missing = [c for c in cols if c not in frame.columns]
    if missing:
        raise KeyError(f"{frame_name} missing required columns: {missing}")
    if not frame.columns.is_unique:
        raise ValueError(f"{frame_name} has duplicate column labels.")

pred = pred[pred_required].copy()
pred["Datetime"] = pd.to_datetime(pred["Datetime"], errors="coerce")
pred["StationId"] = pred["StationId"].astype("string")

for col in NUMERIC_COLS:
    pred[col] = pd.to_numeric(pred[col], errors="coerce")

features = features[["Datetime", "StationId", "AQI"]].copy()
features["Datetime"] = pd.to_datetime(features["Datetime"], errors="coerce")
features["StationId"] = features["StationId"].astype("string")
features["AQI"] = pd.to_numeric(features["AQI"], errors="coerce")
features = features.rename(columns={"AQI": "AQI_current"})

if features.duplicated(["Datetime", "StationId"]).any():
    raise ValueError("Duplicate station-time keys in feature matrix.")

df = pred.merge(
    features,
    on=["Datetime", "StationId"],
    how="left",
    validate="many_to_one",
    indicator=True
)
if not df["_merge"].eq("both").all():
    raise ValueError("Some holdout predictions did not match the feature matrix.")
df = df.drop(columns="_merge")

finite_mask = np.isfinite(df[NUMERIC_COLS].to_numpy(dtype=float)).all(axis=1)
common = df.loc[finite_mask].copy()
common["Target_Datetime"] = common["Datetime"] + pd.Timedelta(hours=1)
common["Target_Month"] = common["Target_Datetime"].dt.strftime("%Y-%m")

month = common["Target_Datetime"].dt.month
common["Season"] = np.select(
    [
        month.isin([12, 1, 2]),
        month.isin([3, 4, 5]),
        month.isin([6, 7, 8, 9]),
        month.isin([10, 11]),
    ],
    ["Winter", "Pre-monsoon", "Monsoon", "Post-monsoon"],
    default="Unknown"
)

common["Target_AQI_Band"] = pd.cut(
    common[ACTUAL],
    bins=[-np.inf, 100, 200, 300, 400, np.inf],
    labels=["<100", "100-199", "200-299", "300-399", ">=400"],
    right=False
)

print(f"\n[1] Common complete-case holdout rows: {len(common):,}")
print(f"Original holdout rows: {len(df):,}")
print(f"Rows excluded for fair all-model comparison: {len(df)-len(common):,}")

def metric_record(y, prediction):
    y = np.asarray(y, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    valid = np.isfinite(y) & np.isfinite(prediction)
    y, prediction = y[valid], prediction[valid]

    if len(y) == 0:
        return {"N": 0, "MAE": None, "RMSE": None, "R2": None}

    err = prediction - y
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = float(1 - np.sum(err ** 2) / sst) if len(y) > 1 and sst > 0 else None

    return {
        "N": int(len(y)),
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "R2": r2,
        "Within_5_AQI_pct": float(np.mean(np.abs(err) <= 5) * 100),
        "Within_10_AQI_pct": float(np.mean(np.abs(err) <= 10) * 100),
        "Within_20_AQI_pct": float(np.mean(np.abs(err) <= 20) * 100),
    }

def grouped_common_metrics(frame, group_col, grouping_name):
    rows = []
    for group, g in frame.groupby(group_col, observed=False, dropna=False):
        if len(g) == 0:
            continue
        for model, col in MODEL_COLS.items():
            rows.append({
                "Grouping": grouping_name,
                "Group": str(group),
                "Model": model,
                **metric_record(g[ACTUAL], g[col]),
            })
    return pd.DataFrame(rows)

matched_tables = {
    "matched_aqi_band_metrics": grouped_common_metrics(
        common, "Target_AQI_Band", "Target AQI band"
    ),
    "matched_station_metrics": grouped_common_metrics(
        common, "StationId", "Station"
    ),
    "matched_season_metrics": grouped_common_metrics(
        common, "Season", "Season"
    ),
    "matched_month_metrics": grouped_common_metrics(
        common, "Target_Month", "Target month"
    ),
}

# Transition analyses require current AQI and actual target AQI.
transition = common[
    common["AQI_current"].notna() & common[ACTUAL].notna()
].copy()
transition["Actual_Change"] = transition[ACTUAL] - transition["AQI_current"]

transition["Transition_Band"] = np.select(
    [
        transition["Actual_Change"] < -20,
        (transition["Actual_Change"] >= -20) & (transition["Actual_Change"] < -5),
        transition["Actual_Change"].between(-5, 5, inclusive="both"),
        (transition["Actual_Change"] > 5) & (transition["Actual_Change"] <= 20),
        transition["Actual_Change"] > 20,
    ],
    [
        "Sharp improvement (<-20)",
        "Moderate improvement (-20 to -5)",
        "Stable (-5 to +5)",
        "Moderate deterioration (+5 to +20)",
        "Sharp deterioration (>+20)",
    ],
    default="Unknown"
)
matched_tables["matched_transition_metrics"] = grouped_common_metrics(
    transition, "Transition_Band", "Actual AQI change"
)

# Detection of threshold crossings in predicted one-hour AQI change.
detection_rows = []
for model, pred_col in MODEL_COLS.items():
    pred_change = transition[pred_col] - transition["AQI_current"]
    actual_change = transition["Actual_Change"]

    event_definitions = [
        ("Sharp deterioration (>+20)", actual_change > 20, pred_change > 20),
        ("Sharp improvement (<-20)", actual_change < -20, pred_change < -20),
        ("Any abrupt change (absolute >20)",
         actual_change.abs() > 20, pred_change.abs() > 20),
    ]

    for event_name, actual_event, predicted_event in event_definitions:
        actual_count = int(actual_event.sum())
        predicted_count = int(predicted_event.sum())
        tp = int((actual_event & predicted_event).sum())
        detection_rows.append({
            "Model": model,
            "Event": event_name,
            "Eligible_Rows": int(len(transition)),
            "Actual_Event_Rows": actual_count,
            "Predicted_Event_Rows": predicted_count,
            "True_Positive": tp,
            "Missed_Events": int(actual_count - tp),
            "Recall_pct": float(tp / actual_count * 100) if actual_count else None,
            "Precision_pct": float(tp / predicted_count * 100) if predicted_count else None,
        })

matched_tables["matched_transition_detection"] = pd.DataFrame(detection_rows)

for name, table in matched_tables.items():
    dest = TABLES / f"{name}.csv"
    table.to_csv(dest, index=False)
    tables_index[name] = {
        "path": str(dest),
        "rows": int(len(table)),
        "columns": list(table.columns),
        "source": "Recomputed from frozen holdout predictions on common support",
    }

print("\n[2] MATCHED-SUPPORT AQI-BAND METRICS")
print(
    matched_tables["matched_aqi_band_metrics"].to_string(
        index=False, float_format=lambda v: f"{v:.4f}"
    )
)

print("\n[3] MATCHED-SUPPORT SHARP-TRANSITION DETECTION")
print(
    matched_tables["matched_transition_detection"].to_string(
        index=False, float_format=lambda v: f"{v:.3f}"
    )
)

# 5. Copy and summarize feature-attribution results.
perm = pd.read_csv(SOURCES["permutation_importance"])
grouped = pd.read_csv(SOURCES["grouped_importance"])

if "Relative_Importance_%" not in perm.columns or "Feature" not in perm.columns:
    raise KeyError("Permutation importance file has unexpected schema.")

perm = perm.sort_values("Relative_Importance_%", ascending=False).reset_index(drop=True)
perm_top = perm.head(20).copy()
perm_top.to_csv(TABLES / "ensemble_permutation_importance_top20.csv", index=False)
tables_index["ensemble_permutation_importance_top20"] = {
    "path": str(TABLES / "ensemble_permutation_importance_top20.csv"),
    "rows": int(len(perm_top)),
    "columns": list(perm_top.columns),
    "source": str(SOURCES["permutation_importance"]),
}

if "Feature_Group" in grouped.columns:
    group_summary = grouped.groupby("Feature_Group", dropna=False).agg(
        Feature_Count=("Original_Feature", "count"),
        Permutation_Importance_Share_pct=("Relative_Importance_%", "sum"),
        Native_Importance_Share_pct=("Native_Relative_Importance_%", "sum"),
    ).reset_index().sort_values("Permutation_Importance_Share_pct", ascending=False)
else:
    group_summary = pd.DataFrame()

group_summary.to_csv(TABLES / "ensemble_importance_by_feature_group.csv", index=False)
tables_index["ensemble_importance_by_feature_group"] = {
    "path": str(TABLES / "ensemble_importance_by_feature_group.csv"),
    "rows": int(len(group_summary)),
    "columns": list(group_summary.columns),
    "source": str(SOURCES["grouped_importance"]),
}

print("\n[4] TOP PERMUTATION-IMPORTANCE FEATURES")
print(
    perm_top[["Feature", "Feature_Group", "Relative_Importance_%"]]
    .head(12).to_string(index=False, float_format=lambda v: f"{v:.4f}")
)

if len(group_summary):
    print("\n[5] AGGREGATED IMPORTANCE BY FEATURE GROUP")
    print(group_summary.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# 6. Create a compact evidence report and Results draft.
validation_ensemble = pd.read_csv(SOURCES["validation_ensemble_summary"]).iloc[0].to_dict()
validation_comparison = pd.read_csv(SOURCES["validation_ensemble_comparison"])
holdout_full = pd.read_csv(SOURCES["holdout_full_support"])
holdout_common = pd.read_csv(SOURCES["holdout_common_support"])
bootstrap = pd.read_csv(SOURCES["bootstrap_results"])
ablation = pd.read_csv(SOURCES["historical_ablation"])

ensemble_validation = validation_ensemble
ensemble_holdout_full = holdout_full.loc[
    holdout_full["Model"].eq("Weighted ensemble")
].iloc[0].to_dict()
ensemble_holdout_common = holdout_common.loc[
    holdout_common["Model"].eq("Weighted ensemble")
].iloc[0].to_dict()

ensemble_boot = bootstrap.loc[
    bootstrap["Model"].eq("Weighted ensemble")
].iloc[0].to_dict()

def get_num(mapping, key):
    value = mapping.get(key)
    return float(value) if value is not None and pd.notna(value) else None

top2 = perm.iloc[:2]
current_importance = perm.loc[perm["Feature"].eq("AQI"), "Relative_Importance_%"]
rolling_importance = perm.loc[
    perm["Feature"].eq("AQI_rolling_mean_3h"), "Relative_Importance_%"
]
current_share = float(current_importance.iloc[0]) if len(current_importance) else None
rolling_share = float(rolling_importance.iloc[0]) if len(rolling_importance) else None
combined_share = (
    current_share + rolling_share
    if current_share is not None and rolling_share is not None else None
)

def fmt(value, digits=4):
    if value is None or pd.isna(value):
        return "not available"
    return f"{float(value):.{digits}f}"

results_text = f"""# Draft Results — Delhi AQI One-Hour-Ahead Forecasting

## Validation model comparison

The frozen chronological validation comparison included 69,097 rows. The RMSE-optimized weighted ensemble achieved MAE {fmt(ensemble_validation.get('MAE'))}, RMSE {fmt(ensemble_validation.get('RMSE'))}, and R² {fmt(ensemble_validation.get('R2'), 6)}. Predictions were within ±10 AQI points for {fmt(ensemble_validation.get('Accuracy ±10 AQI'), 2)}% of validation rows and within ±20 points for {fmt(ensemble_validation.get('Accuracy ±20 AQI'), 2)}%. The ensemble weights were 0.594466 for Tuned Extra Trees and 0.405534 for Tuned Random Forest. On validation, its RMSE of 8.247305 was slightly lower than the equal-weight ensemble RMSE of 8.249447; this small difference should not be presented as a large practical gain. The optimized weighting was selected for RMSE and did not optimize every metric simultaneously.

## Frozen final holdout performance

The final holdout contained 68,352 rows. On full available model-specific support, the weighted ensemble achieved MAE {fmt(ensemble_holdout_full.get('MAE'))}, RMSE {fmt(ensemble_holdout_full.get('RMSE'))}, and R² {fmt(ensemble_holdout_full.get('R2'), 6)}. The persistence baseline had 67 missing predictions, so its direct paired comparison uses 68,285 common rows. On that identical support, persistence had MAE 3.855634 and RMSE 8.840011, while the weighted ensemble had MAE {fmt(ensemble_holdout_common.get('MAE'))} and RMSE {fmt(ensemble_holdout_common.get('RMSE'))}. This corresponds to the saved estimates of 17.57% lower MAE and 15.21% lower RMSE than persistence on matched rows.

## Uncertainty in the paired comparison

A paired day-block bootstrap resampled 296 calendar-day blocks 5,000 times. For the weighted ensemble, the absolute MAE reduction versus persistence was {fmt(ensemble_boot.get('Delta_MAE_Persistence_minus_Model'))} AQI points (95% CI {fmt(ensemble_boot.get('Delta_MAE_95CI_Lower'))} to {fmt(ensemble_boot.get('Delta_MAE_95CI_Upper'))}). The RMSE reduction was {fmt(ensemble_boot.get('Delta_RMSE_Persistence_minus_Model'))} points (95% CI {fmt(ensemble_boot.get('Delta_RMSE_95CI_Lower'))} to {fmt(ensemble_boot.get('Delta_RMSE_95CI_Upper'))}). Both intervals exclude zero. These intervals represent uncertainty under day-block resampling for the held-out evaluation period; they do not guarantee future live performance.

## Regimes and rapid AQI changes

Common-support subgroup tables are provided for target AQI band, station, season, month, and actual one-hour AQI change. The sharp-transition analysis defines a sharp deterioration as an actual increase greater than 20 AQI points and a sharp improvement as a decrease greater than 20 points. Results should distinguish continuous AQI forecasting accuracy from detecting such a change event: the frozen ensemble detects only about 9% of abrupt changes using a ±20-point predicted-change threshold. This is an important limitation of the current forecaster.

In the earlier descriptive holdout audit, the top 2.5% of rows ranked by squared ensemble error accounted for 78.49% of total ensemble SSE. Rows with absolute actual AQI change greater than 20 points accounted for 70.68% of SSE. These figures show that aggregate performance is dominated by a small set of difficult transitions and should be reported alongside overall MAE and RMSE.

## Feature attribution

The saved ensemble permutation-importance report attributes {fmt(current_share, 2)}% of normalized importance to current AQI and {fmt(rolling_share, 2)}% to the three-hour AQI rolling mean, together {fmt(combined_share, 2)}%. Other top features include AQI lags and ozone. This indicates strong reliance on recent AQI state for one-hour-ahead prediction. Permutation importance is predictive and model-dependent; it is not a causal estimate, and correlated features can share or mask importance.

## Historical ablations and scope

The saved Phase 14 ablation leaderboard reports four-fold mean RMSE values for historical feature sets. These experiments are retained as historical exploratory evidence only. Their results should not be mixed directly with the frozen final holdout table unless the manuscript explicitly documents the differing experiment protocol, data scope, and feature definitions. Older tables with unusually small errors are not used in the final holdout claims.

## Limitations and deployment status

The evaluation is a retrospective holdout from the available historical data, not a live deployment trial. A set of 67 target rows lacks a persistence prediction and also has missing current AQI and multiple recent AQI-derived features; the ensemble errors on these rows are unusually high. The paired model comparison therefore excludes those rows and reports the support explicitly. The production gate remains closed because a validated live AQI source has not been established. No models were retrained or retuned during this evidence consolidation.
"""

(OUT / "phase17_07I_draft_results.md").write_text(
    results_text, encoding="utf-8"
)

report = {
    "phase": "17.07I",
    "evidence_package": str(OUT),
    "tables": tables_index,
    "holdout_row_count": int(len(df)),
    "matched_holdout_row_count": int(len(common)),
    "transition_analysis_row_count": int(len(transition)),
    "ensemble_full_support": {
        "MAE": get_num(ensemble_holdout_full, "MAE"),
        "RMSE": get_num(ensemble_holdout_full, "RMSE"),
        "R2": get_num(ensemble_holdout_full, "R2"),
        "N": int(ensemble_holdout_full.get("N", 0)),
    },
    "ensemble_common_support": {
        "MAE": get_num(ensemble_holdout_common, "MAE"),
        "RMSE": get_num(ensemble_holdout_common, "RMSE"),
        "R2": get_num(ensemble_holdout_common, "R2"),
        "N": int(ensemble_holdout_common.get("N", 0)),
    },
    "feature_importance": {
        "AQI_relative_importance_pct": current_share,
        "AQI_rolling_mean_3h_relative_importance_pct": rolling_share,
        "combined_relative_importance_pct": combined_share,
    },
    "models_retrained": False,
    "models_retuned": False,
    "holdout_used_for_model_selection": False,
    "production_enabled": False,
}
(OUT / "phase17_07I_evidence_package_manifest.json").write_text(
    json.dumps(report, indent=2, allow_nan=False, default=str),
    encoding="utf-8"
)

print("\n[6] EVIDENCE PACKAGE COMPLETE")
print("Package folder:", OUT)
print("Draft Results:", OUT / "phase17_07I_draft_results.md")
print("Table count:", len(list(TABLES.glob("*.csv"))))
print("Models retrained: False")
print("Models retuned: False")
print("Holdout used for model selection: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07J — MANUSCRIPT DRAFT AND RESULTS TABLES
# Read-only synthesis of frozen results; no model training or holdout selection.

from pathlib import Path
import json
import pandas as pd
import numpy as np

print("=" * 78)
print("PHASE 17.07J — MANUSCRIPT DRAFT")
print("=" * 78)

roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next((p for p in roots if p.exists() and (p / "outputs").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project folder not found.")

SOURCE = ROOT / "outputs/research_paper/phase17_07I_evidence_package/tables"
OUT = ROOT / "outputs/research_paper/phase17_07J_manuscript_draft"
TABLES = OUT / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

def load_table(name):
    path = SOURCE / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Required evidence table missing: {path}")
    return pd.read_csv(path)

# Load the already consolidated evidence.
validation = load_table("validation_model_comparison")
ensemble_validation = load_table("validation_ensemble_comparison")
validation_summary = load_table("validation_ensemble_summary")
holdout_full = load_table("holdout_full_support")
holdout_common = load_table("holdout_common_support")
bootstrap = load_table("bootstrap_results")
aqi_bands = load_table("matched_aqi_band_metrics")
stations = load_table("matched_station_metrics")
seasons = load_table("matched_season_metrics")
transitions = load_table("matched_transition_detection")
importance = load_table("ensemble_permutation_importance_top20")
importance_groups = load_table("ensemble_importance_by_feature_group")
ablation = load_table("historical_ablation")

def md_table(frame, columns=None, rename=None, digits=4):
    """Create a Markdown table without depending on the tabulate package."""
    view = frame.copy()
    if columns is not None:
        view = view[columns]
    if rename:
        view = view.rename(columns=rename)

    headers = list(view.columns)
    rows = []
    for _, row in view.iterrows():
        values = []
        for value in row.tolist():
            if value is None or pd.isna(value):
                values.append("—")
            elif isinstance(value, (int, np.integer)):
                values.append(str(int(value)))
            elif isinstance(value, (float, np.floating)):
                values.append(f"{float(value):.{digits}f}")
            else:
                values.append(str(value).replace("|", "\\|").replace("\n", " "))
        rows.append(values)

    output = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    output.extend("| " + " | ".join(row) + " |" for row in rows)
    return "\n".join(output)

def fmt(x, places=4):
    if x is None or pd.isna(x):
        return "not available"
    return f"{float(x):.{places}f}"

def one_row(frame, column, value):
    result = frame.loc[frame[column].astype(str).eq(str(value))]
    if result.empty:
        raise ValueError(f"Could not find {column}={value!r}.")
    return result.iloc[0]

# ---------------------------------------------------------------------------
# 1. Prepare tables.
# ---------------------------------------------------------------------------

# All recorded validation candidates, ranked by the existing validation report.
validation_table = validation.copy()
validation_table.to_csv(TABLES / "table_1_validation_model_comparison.csv", index=False)

# Final holdout models on an identical support.
common_table = holdout_common[
    holdout_common["Model"].isin(
        ["Persistence", "Extra Trees", "Random Forest", "Weighted ensemble"]
    )
].copy()
common_table.to_csv(TABLES / "table_2_holdout_matched_support.csv", index=False)

# Model-specific full support, separately labelled because N differs.
full_table = holdout_full.copy()
full_table.to_csv(TABLES / "table_3_holdout_full_available_support.csv", index=False)

# Day-block bootstrap confidence intervals.
bootstrap_table = bootstrap[
    bootstrap["Model"].isin(["Extra Trees", "Random Forest", "Weighted ensemble"])
].copy()
bootstrap_table.to_csv(TABLES / "table_4_day_block_bootstrap.csv", index=False)

# AQI level comparison: persistence versus ensemble on matched rows.
band_models = aqi_bands[
    aqi_bands["Model"].isin(["Persistence", "Weighted ensemble"])
].copy()

band_rows = []
for band in ["<100", "100-199", "200-299", "300-399", ">=400"]:
    part = band_models[band_models["Group"].astype(str).eq(band)]
    if part.empty:
        continue
    p = part[part["Model"].eq("Persistence")].iloc[0]
    e = part[part["Model"].eq("Weighted ensemble")].iloc[0]
    band_rows.append({
        "Target AQI band": band,
        "N": int(e["N"]),
        "Persistence MAE": p["MAE"],
        "Ensemble MAE": e["MAE"],
        "Persistence RMSE": p["RMSE"],
        "Ensemble RMSE": e["RMSE"],
        "RMSE reduction (%)": (p["RMSE"] - e["RMSE"]) / p["RMSE"] * 100,
    })
band_table = pd.DataFrame(band_rows)
band_table.to_csv(TABLES / "table_5_performance_by_aqi_band.csv", index=False)

# Ensemble transition-detection results.
ensemble_transition = transitions[
    transitions["Model"].eq("Weighted ensemble")
].copy()
ensemble_transition.to_csv(
    TABLES / "table_6_abrupt_transition_detection.csv", index=False
)

# Top 10 permutation-importance features.
feature_table = importance.head(10).copy()
feature_table.to_csv(TABLES / "table_7_feature_attribution.csv", index=False)

# Station and seasonal ensemble performance.
station_ensemble = stations[stations["Model"].eq("Weighted ensemble")].copy()
station_ensemble = station_ensemble.sort_values("MAE")
station_ensemble.to_csv(TABLES / "table_8_station_performance.csv", index=False)

season_ensemble = seasons[seasons["Model"].eq("Weighted ensemble")].copy()
season_ensemble = season_ensemble.sort_values("MAE")
season_ensemble.to_csv(TABLES / "table_9_season_performance.csv", index=False)

# ---------------------------------------------------------------------------
# 2. Extract headline results.
# ---------------------------------------------------------------------------

val_ensemble = one_row(
    validation_summary, "Model", "Optimized Weighted Ensemble"
)
full_ensemble = one_row(holdout_full, "Model", "Weighted ensemble")
matched_ensemble = one_row(holdout_common, "Model", "Weighted ensemble")
matched_persistence = one_row(holdout_common, "Model", "Persistence")
boot_ensemble = one_row(bootstrap_table, "Model", "Weighted ensemble")

aqi_importance = importance.loc[importance["Feature"].eq("AQI")]
rolling_importance = importance.loc[
    importance["Feature"].eq("AQI_rolling_mean_3h")
]

aqi_imp = float(aqi_importance.iloc[0]["Relative_Importance_%"])
rolling_imp = float(rolling_importance.iloc[0]["Relative_Importance_%"])

top_station = station_ensemble.iloc[-1]
best_station = station_ensemble.iloc[0]
best_season = season_ensemble.iloc[0]
worst_season = season_ensemble.iloc[-1]

# ---------------------------------------------------------------------------
# 3. Compose manuscript working draft.
# ---------------------------------------------------------------------------

validation_md = md_table(
    validation_table,
    columns=[
        "Rank", "Model", "Feature Count", "MAE", "RMSE", "R2",
        "Accuracy ±10 AQI", "Accuracy ±20 AQI", "Validation Records"
    ]
)

matched_md = md_table(
    common_table,
    columns=[
        "Model", "N", "MAE", "RMSE", "R2",
        "Within_5_AQI_pct", "Within_10_AQI_pct", "Within_20_AQI_pct"
    ],
    rename={
        "N": "Rows",
        "R2": "R²",
        "Within_5_AQI_pct": "Within ±5 (%)",
        "Within_10_AQI_pct": "Within ±10 (%)",
        "Within_20_AQI_pct": "Within ±20 (%)",
    }
)

full_md = md_table(
    full_table,
    columns=[
        "Model", "N", "MAE", "RMSE", "R2",
        "Within_10_AQI_pct", "Within_20_AQI_pct"
    ],
    rename={
        "N": "Rows",
        "R2": "R²",
        "Within_10_AQI_pct": "Within ±10 (%)",
        "Within_20_AQI_pct": "Within ±20 (%)",
    }
)

bootstrap_md = md_table(
    bootstrap_table,
    columns=[
        "Model",
        "Delta_MAE_Persistence_minus_Model",
        "Delta_MAE_95CI_Lower",
        "Delta_MAE_95CI_Upper",
        "Delta_RMSE_Persistence_minus_Model",
        "Delta_RMSE_95CI_Lower",
        "Delta_RMSE_95CI_Upper",
        "Relative_MAE_Improvement_pct",
        "Relative_RMSE_Improvement_pct",
    ],
    rename={
        "Delta_MAE_Persistence_minus_Model": "MAE reduction",
        "Delta_MAE_95CI_Lower": "MAE CI lower",
        "Delta_MAE_95CI_Upper": "MAE CI upper",
        "Delta_RMSE_Persistence_minus_Model": "RMSE reduction",
        "Delta_RMSE_95CI_Lower": "RMSE CI lower",
        "Delta_RMSE_95CI_Upper": "RMSE CI upper",
        "Relative_MAE_Improvement_pct": "MAE improvement (%)",
        "Relative_RMSE_Improvement_pct": "RMSE improvement (%)",
    }
)

bands_md = md_table(band_table, digits=3)

transition_md = md_table(
    ensemble_transition,
    columns=[
        "Event", "Eligible_Rows", "Actual_Event_Rows", "Predicted_Event_Rows",
        "True_Positive", "Missed_Events", "Recall_pct", "Precision_pct"
    ],
    rename={
        "Eligible_Rows": "Eligible rows",
        "Actual_Event_Rows": "Actual events",
        "Predicted_Event_Rows": "Predicted events",
        "True_Positive": "True positives",
        "Missed_Events": "Missed events",
        "Recall_pct": "Recall (%)",
        "Precision_pct": "Precision (%)",
    },
    digits=3
)

importance_md = md_table(
    feature_table,
    columns=["Feature", "Feature_Group", "Relative_Importance_%"],
    rename={
        "Feature_Group": "Group",
        "Relative_Importance_%": "Permutation importance (%)",
    },
    digits=4
)

station_md = md_table(
    station_ensemble,
    columns=["Group", "N", "MAE", "RMSE", "R2"],
    rename={"Group": "Station", "N": "Rows", "R2": "R²"},
    digits=3
)

season_md = md_table(
    season_ensemble,
    columns=["Group", "N", "MAE", "RMSE", "R2"],
    rename={"Group": "Season", "N": "Rows", "R2": "R²"},
    digits=3
)

ablation_md = md_table(ablation, digits=4)

# Abstract (working draft; final word limit can be adjusted for the target venue).
abstract = f"""
Hourly air-quality index (AQI) forecasting can support short-horizon monitoring and public-health decision-making. This study evaluates one-hour-ahead AQI prediction across 10 monitoring stations in Delhi using a 73-feature station-hour forecasting dataset. The target was AQI at time t+1, using information available through time t. A chronological training, validation, and final-holdout protocol was used, with 166,845 training rows, 69,097 validation rows, and 68,352 final-holdout rows; 19 gap rows were kept outside these partitions. Tuned Extra Trees and Random Forest were selected as the two ensemble components using validation results, and their weights were frozen at 0.594466 and 0.405534, respectively.

On the complete final holdout, the weighted ensemble achieved MAE {fmt(full_ensemble['MAE'], 4)}, RMSE {fmt(full_ensemble['RMSE'], 4)}, and R² {fmt(full_ensemble['R2'], 6)}. A fair comparison with persistence was conducted on 68,285 common rows because 67 persistence predictions were unavailable. On this matched support, ensemble MAE and RMSE were {fmt(matched_ensemble['MAE'], 4)} and {fmt(matched_ensemble['RMSE'], 4)}, respectively, compared with {fmt(matched_persistence['MAE'], 4)} and {fmt(matched_persistence['RMSE'], 4)} for persistence. Paired day-block bootstrap intervals excluded zero for both reductions. Feature attribution was dominated by current AQI and the three-hour AQI rolling mean. However, only about 9% of abrupt one-hour changes exceeding 20 AQI points were detected using the same magnitude threshold for predicted change. The results support strong retrospective one-hour forecasting performance while highlighting difficulty with abrupt transitions and the need for validated live data before deployment.
""".strip()

manuscript = f"""# One-Hour-Ahead AQI Forecasting Across Delhi Monitoring Stations

**Working manuscript draft — generated from the frozen research evidence package**

## Abstract

{abstract}

## 1. Introduction

Air-quality forecasting is a time-series prediction problem in which the relevance of temporal dependence, pollutant measurements, and station-level variation must be evaluated alongside predictive accuracy. This study focuses on one-hour-ahead AQI forecasting across a frozen scope of 10 Delhi monitoring stations.

The study addresses three questions:

1. How do candidate machine-learning regressors perform on a chronological validation set?
2. Does the selected weighted ensemble improve upon a persistence baseline on an untouched final holdout?
3. How reliable are forecasts across AQI severity bands and during abrupt one-hour AQI changes?

**Literature review to complete:** Add sources on short-horizon AQI forecasting, persistence baselines, leakage-free time-series validation, ensemble regression, and air-quality forecasting limitations. Do not leave this placeholder in a submission copy.

## 2. Methods

### 2.1 Forecasting target and feature construction

The target was AQI at the next hourly timestamp, denoted AQI(t+1). Predictors were defined to use information available through time t. The frozen feature contract contained 73 features, including current AQI and pollutant measurements, pollutant/AQI lags, rolling summaries, station identity, calendar information, and event-related flags. The temporal features included AQI lags at 1, 2, and 12 hours and rolling means and standard deviations over 3-hour and 12-hour windows.

The deployed model interface treats `StationId` as a categorical string feature. Feature names and order were fixed in the frozen contract. No feature contract changes were made during this research-evidence consolidation.

### 2.2 Data partitioning and evaluation protocol

The engineered matrix contained 304,313 station-hour rows. The saved chronological split assignments comprised 166,845 training rows, 69,097 validation rows, and 68,352 final-holdout rows, with 19 gap rows outside the three partitions. The final holdout was reserved for evaluation after validation-based model selection and was not used for tuning or selecting the final model.

The frozen scope included 10 Delhi stations. The processed historical source record spans January 2015 through the beginning of July 2020. The manuscript should cite the exact dataset/source documentation and describe any filtering and quality-control steps using the project's saved preprocessing protocol.

### 2.3 Candidate models and ensemble selection

The recorded validation comparison evaluated linear regression, decision tree, random forest, extra trees, gradient boosting, histogram gradient boosting, AdaBoost, XGBoost, and tuned variants. Tuned Extra Trees and Tuned Random Forest formed the final ensemble components.

The ensemble weights were frozen at 0.594466 for Tuned Extra Trees and 0.405534 for Tuned Random Forest. The weighted ensemble was selected using validation RMSE. Its validation RMSE was 8.247305, compared with 8.249447 for the equal-weight 50/50 ensemble. This small difference should be interpreted as a modest validation improvement, not as a large practical gain.

### 2.4 Evaluation metrics

Performance was assessed using mean absolute error (MAE), root mean squared error (RMSE), coefficient of determination (R²), and the percentage of predictions within ±5, ±10, and ±20 AQI points of the observed target. The persistence baseline used current AQI, AQI(t), as the next-hour prediction when current AQI was available.

For a fair baseline comparison, all model metrics were also calculated on the identical set of rows with finite target and predictions for every compared model. The full-support model results and the matched-support comparison are reported separately because 67 persistence predictions were missing.

### 2.5 Paired day-block bootstrap

Uncertainty in model improvement over persistence was assessed using 5,000 paired day-block bootstrap replicates with random seed 20261023. The 68,285 matched rows were grouped into 296 calendar-day blocks. Days were sampled with replacement, retaining the station-hour records within each sampled day. The reported 95% intervals are percentile intervals for reductions in MAE and RMSE. Day-block resampling preserves within-day clustering but does not preserve all dependence between adjacent days.

### 2.6 AQI regimes and abrupt transitions

Results were stratified into target AQI bands below 100, 100–199, 200–299, 300–399, and at least 400. Station, season, and month subgroup tables were also generated.

For transition analysis, the observed change was defined as ΔAQI = AQI(t+1) − AQI(t). A sharp deterioration was defined as ΔAQI > 20; a sharp improvement as ΔAQI < −20. Detection recall and precision were calculated by applying the same thresholds to the predicted change. Transition metrics require an available current AQI value.

### 2.7 Feature attribution

The saved pre-holdout ensemble permutation-importance analysis used a global sample of 500 observations and two permutation repeats. Relative importance values are model- and dataset-dependent measures of predictive reliance; they should not be interpreted as causal effects.

## 3. Results

### 3.1 Validation comparison

The following table reproduces the recorded validation comparison. Candidate results are validation results and should not be conflated with final-holdout metrics.

{validation_md}

### 3.2 Final holdout performance

The frozen ensemble achieved MAE {fmt(full_ensemble['MAE'])}, RMSE {fmt(full_ensemble['RMSE'])}, and R² {fmt(full_ensemble['R2'], 6)} on all {int(full_ensemble['N']):,} rows with an available ensemble prediction. The corresponding final-holdout values for the individual models and persistence, using each model's available support, are shown below.

{full_md}

Because persistence did not produce predictions for 67 rows, the following matched-support table is the fair comparison across all four approaches.

{matched_md}

On common support, the ensemble reduced MAE by {fmt(boot_ensemble['Relative_MAE_Improvement_pct'], 2)}% and RMSE by {fmt(boot_ensemble['Relative_RMSE_Improvement_pct'], 2)}% relative to persistence. Random Forest had slightly lower MAE than the ensemble on these matched rows, while the ensemble had the lowest RMSE. The choice of ensemble therefore reflects the validation RMSE selection criterion rather than dominance on every metric.

### 3.3 Uncertainty around improvement over persistence

{bootstrap_md}

For the weighted ensemble, the absolute MAE reduction was {fmt(boot_ensemble['Delta_MAE_Persistence_minus_Model'])} AQI points (95% CI {fmt(boot_ensemble['Delta_MAE_95CI_Lower'])} to {fmt(boot_ensemble['Delta_MAE_95CI_Upper'])}). The absolute RMSE reduction was {fmt(boot_ensemble['Delta_RMSE_Persistence_minus_Model'])} points (95% CI {fmt(boot_ensemble['Delta_RMSE_95CI_Lower'])} to {fmt(boot_ensemble['Delta_RMSE_95CI_Upper'])}). Both intervals excluded zero under the specified day-block bootstrap.

### 3.4 Performance across target AQI severity bands

{bands_md}

The ensemble achieved lower RMSE than persistence in each target AQI band. However, below AQI 100, the ensemble's MAE was higher than persistence MAE even though its RMSE was lower. This distinction indicates that a model can reduce the influence of large errors while still having a higher average absolute error in a particular subgroup. In the 200–299 and ≥400 bands, both MAE and RMSE were substantially lower for the ensemble than for persistence.

### 3.5 Station- and season-level variation

The station-level and seasonal tables are reproduced from common-support holdout metrics. They document heterogeneity rather than implying that every station or season has equal predictability.

**Station-level ensemble results**

{station_md}

**Season-level ensemble results**

{season_md}

Across-station and seasonal differences should be interpreted in light of the size and distribution of the corresponding subgroup samples. These findings motivate reporting subgroup metrics alongside aggregate accuracy.

### 3.6 Abrupt one-hour AQI transitions

{transition_md}

The ensemble's recall was approximately 9.42% for sharp deteriorations, 8.64% for sharp improvements, and 9.02% for any absolute AQI change greater than 20 points using a predicted-change threshold of ±20. This means the current forecaster is substantially more reliable for ordinary next-hour level prediction than for detecting abrupt changes.

The separate error-concentration audit found that the top 2.5% of rows ranked by ensemble squared error accounted for 78.49% of total ensemble squared error. Rows with absolute actual one-hour changes greater than 20 AQI points accounted for 70.68% of total squared error. These statistics show why event-sensitive evaluation is important in addition to aggregate MAE and RMSE.

### 3.7 Feature attribution

{importance_md}

Current AQI accounted for {aqi_imp:.2f}% of the normalized ensemble permutation importance, while the three-hour AQI rolling mean accounted for {rolling_imp:.2f}%. Together, these two features accounted for approximately {aqi_imp + rolling_imp:.2f}% of this reported importance. The grouped report attributed 91.81% to current-time features, 6.97% to rolling features, 1.10% to lag features, and much smaller shares to station, calendar, and event features.

This pattern is consistent with a one-hour forecasting task that relies heavily on recent AQI state. It does not establish causal influence, and correlated lagged and rolling variables can share predictive information.

## 4. Discussion

The frozen weighted ensemble provided strong retrospective one-hour-ahead performance and improved upon the persistence baseline in overall matched-support MAE and RMSE. Paired day-block bootstrap intervals for both improvements excluded zero, providing an uncertainty assessment that respects within-day clustering.

The gains were not uniform across all metrics and subgroups. On the matched support, Random Forest had slightly lower MAE than the weighted ensemble, whereas the ensemble had the best RMSE. In the below-100 AQI band, ensemble MAE was worse than persistence MAE despite lower RMSE. These results support reporting multiple error metrics instead of describing one model as universally superior.

The most important operational weakness was abrupt-transition detection. Only about 9% of large changes were detected at the chosen ±20-point predicted-change threshold. For use cases requiring advance warning of rapid deterioration, level-forecast accuracy alone is insufficient. Dedicated transition-sensitive modelling or alert logic would require its own training/validation design and a new evaluation protocol; it must not be retrofitted by selecting against the frozen holdout.

The permutation analysis showed that current AQI and the three-hour AQI rolling mean dominated the measured importance. Because the prediction target is one hour ahead, current AQI is a legitimate historical predictor when it is available at inference time. Nonetheless, the importance result should be accompanied by the feature-timing specification so that readers can distinguish a valid lagged predictor from information that would only become available after the target time.

## 5. Limitations

1. **Geographic scope:** the frozen final evaluation covers 10 selected Delhi monitoring stations. Generalization to other cities, monitoring networks, or station configurations has not been established.
2. **Historical evaluation:** the source series ends around July 2020. The holdout demonstrates retrospective performance, not current real-time service performance.
3. **Unavailable baseline predictions:** 67 rows lack persistence predictions, and all 67 have missing current AQI. The ensemble MAE on those rows was approximately 42.01 and RMSE approximately 50.78, substantially worse than on the matched support. These rows are excluded from all-model matched comparisons but remain part of the full-support ensemble evaluation.
4. **Abrupt-change weakness:** transition recall under the specified threshold is low, so high aggregate R² should not be interpreted as reliable advance detection of sudden changes.
5. **Bootstrap scope:** the day-block intervals retain within-day clustering but do not model every possible temporal dependence across days, regime shifts, or future data drift.
6. **Feature attribution:** permutation importance is not causal and may be affected by correlated predictors.
7. **Historical ablations:** earlier feature-ablation tables were generated under a different recorded experimental scope and have not been reconciled to the frozen final protocol. They are retained as exploratory evidence rather than used for direct numerical comparison with the final holdout.
8. **Deployment readiness:** a validated, sufficiently fresh live AQI source and complete input pipeline have not been established. Production remains disabled.

## 6. Conclusion

A frozen 73-feature weighted Extra Trees–Random Forest ensemble showed strong retrospective one-hour-ahead AQI prediction across 10 Delhi stations. On the identical 68,285-row comparison set, it reduced MAE by 17.57% and RMSE by 15.21% relative to persistence, with paired day-block bootstrap intervals excluding zero. Feature attribution was dominated by current AQI and its recent rolling mean. However, sudden one-hour AQI changes remained difficult to detect, and a small set of high-error observations contributed a large share of overall squared error. Results should therefore be presented as evidence of strong historical short-horizon prediction, not proof of live operational readiness.

## Data and code availability

**To complete:** specify the source dataset URL and licence, the data access conditions, code repository location (if public), and instructions for reproducing the frozen experiment. Do not publish API keys or private credentials.

## References

**To complete before submission.** Add verified references for:
- Air-quality monitoring and AQI definition.
- Short-horizon AQI forecasting and machine-learning baselines.
- Time-series validation and leakage prevention.
- Ensemble regression and permutation feature importance.
- Comparable prior deep-learning or decomposition approaches, if their protocols are sufficiently aligned to support a fair comparison.
"""

draft_path = OUT / "phase17_07J_manuscript_working_draft.md"
draft_path.write_text(manuscript, encoding="utf-8")

# 4. Reproducibility manifest.
manifest = {
    "phase": "17.07J",
    "manuscript_path": str(draft_path),
    "tables_directory": str(TABLES),
    "source_evidence_directory": str(SOURCE),
    "source_tables_used": sorted(p.name for p in SOURCE.glob("*.csv")),
    "generated_tables": sorted(p.name for p in TABLES.glob("*.csv")),
    "frozen_final_model": {
        "Tuned_Extra_Trees_weight": 0.594466,
        "Tuned_Random_Forest_weight": 0.405534,
        "feature_count": 73,
        "station_count": 10,
    },
    "evaluation": {
        "full_holdout_rows": int(full_ensemble["N"]),
        "matched_support_rows": int(matched_ensemble["N"]),
        "validation_rows": 69097,
        "training_rows": 166845,
        "gap_rows": 19,
        "day_block_bootstrap_replicates": 5000,
        "day_blocks": 296,
    },
    "models_retrained": False,
    "models_retuned": False,
    "holdout_used_for_model_selection": False,
    "production_enabled": False,
    "citation_status": "Literature references intentionally left as placeholders.",
}
(OUT / "phase17_07J_manifest.json").write_text(
    json.dumps(manifest, indent=2, allow_nan=False, default=str),
    encoding="utf-8"
)

print("\n[1] DRAFT GENERATED")
print(draft_path)
print("\n[2] TABLE DIRECTORY")
print(TABLES)
print(f"Tables generated: {len(list(TABLES.glob('*.csv')))}")
print("\n[3] KEY VALUES CHECK")
print(f"Validation ensemble RMSE: {float(val_ensemble['RMSE']):.6f}")
print(f"Full-support holdout ensemble MAE: {float(full_ensemble['MAE']):.6f}")
print(f"Full-support holdout ensemble RMSE: {float(full_ensemble['RMSE']):.6f}")
print(f"Matched-support ensemble MAE: {float(matched_ensemble['MAE']):.6f}")
print(f"Matched-support ensemble RMSE: {float(matched_ensemble['RMSE']):.6f}")
print("Models retrained: False")
print("Models retuned: False")
print("Holdout used for model selection: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07K — VERIFIED LITERATURE REVIEW AND REFERENCE INTEGRATION
# Creates a new manuscript copy. Keeps the previous draft unchanged.

from pathlib import Path
import json
import pandas as pd

print("=" * 78)
print("PHASE 17.07K — LITERATURE AND REFERENCES")
print("=" * 78)

# 1. Resolve project files.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next((p for p in roots if p.exists() and (p / "outputs").exists()), None)

if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction folder not found.")

SOURCE = (
    ROOT / "outputs/research_paper/phase17_07J_manuscript_draft/"
    "phase17_07J_manuscript_working_draft.md"
)
OUT = ROOT / "outputs/research_paper/phase17_07K_verified_literature"
OUT.mkdir(parents=True, exist_ok=True)

if not SOURCE.exists():
    raise FileNotFoundError(f"Phase 17.07J manuscript not found: {SOURCE}")

manuscript = SOURCE.read_text(encoding="utf-8")

# 2. Add a literature review grounded in publisher/official source records.
old_literature_placeholder = (
    "**Literature review to complete:** Add sources on short-horizon AQI forecasting, "
    "persistence baselines, leakage-free time-series validation, ensemble regression, "
    "and air-quality forecasting limitations. Do not leave this placeholder in a "
    "submission copy."
)

literature_review = """### Related work and study positioning

The Central Pollution Control Board (CPCB) describes India's National Air Quality Index as a framework for communicating air quality through pollutant sub-indices and AQI categories [1]. This provides the policy and measurement context for AQI prediction, although the exact construction and interpretation of an index should be tied to the standard used by the source dataset.

Machine-learning studies have examined AQI and pollutant forecasting under differing locations, time horizons, predictor sets, and evaluation protocols. Liu et al. evaluated support vector regression and random forest regression for AQI prediction in Beijing and nitrogen-oxide prediction in an Italian city [3]. Gupta et al. compared several regression techniques for AQI prediction across Indian cities, including experiments involving the Synthetic Minority Oversampling Technique (SMOTE) [4]. These studies provide relevant methodological context, but their numeric scores are not direct benchmarks for the present study because the forecasting target, data handling, prediction horizon, and evaluation design are not necessarily the same.

Delhi-specific forecasting studies have investigated daily AQI prediction using bidirectional recurrent neural networks and other deep-learning or data-driven methods [5, 6]. A more recent Delhi study assessed regional and global atmospheric forecasting systems against hourly observations from 39 CPCB stations, focusing on particulate-matter prediction and associated AQI assessment [7]. These works demonstrate the range of available forecasting approaches. Their results should be compared with the present study only with explicit consideration of different inputs, targets, forecast horizons, and validation protocols.

The present study uses a one-hour-ahead target and lagged station-hour observations. It compares tuned tree ensembles against a persistence baseline on an identical set of holdout rows and estimates uncertainty using day-block resampling. Its goal is not to claim that one model family universally outperforms deep-learning or atmospheric models, but to provide a clearly specified, reproducible assessment for this particular short-horizon forecasting task.

The choice of tree ensembles is supported by the established methods literature. Random Forests combine randomized tree predictors [9], while Extra-Trees introduce additional randomization into the selection of split attributes and cut-points [10]. Such methods can model nonlinear relationships and interactions without requiring a linear functional form. Their empirical suitability, however, must be established by evaluation on data separated from model development, rather than inferred from the algorithms alone.

Forecast evaluation for time-series data requires attention to temporal dependence and the prediction setting. Bergmeir et al. show that standard K-fold cross-validation can be valid for some autoregressive settings under specific error assumptions; the broader lesson is that validation design must be justified by the model and forecasting objective, not by an absolute rule that all cross-validation is invalid [8]. In this project, chronological partitions and an untouched final holdout were chosen to assess performance on later time periods consistent with the one-hour-ahead forecasting objective.

Feature-importance results also require careful interpretation. Strobl et al. describe how correlated predictors can affect Random Forest variable-importance measures, including unconditional permutation methods [11]. Because current AQI, AQI lags, and rolling AQI summaries are correlated by construction, the permutation-importance percentages reported here should be interpreted as model-specific predictive reliance, not causal effects or unique contributions independent of other predictors.

For uncertainty estimation, Künsch developed block-based jackknife and bootstrap methods for dependent stationary observations [12]. The present study uses a paired day-block bootstrap, sampling complete calendar-day clusters with replacement and retaining the station-hour observations within each sampled day. This preserves within-day grouping, while dependence across neighbouring calendar days remains a limitation.

The contribution is therefore a protocol-specific comparison that combines a frozen one-hour-ahead target, multi-station Delhi evaluation, common-support persistence comparisons, paired day-block confidence intervals, and an explicit analysis of abrupt AQI changes. These elements define the scope of the evidence presented here; they do not, by themselves, establish a claim that this is the first study to examine any one of them.
"""

if old_literature_placeholder not in manuscript:
    raise ValueError(
        "Expected literature placeholder was not found. "
        "The source manuscript may have been edited; no replacement was made."
    )

manuscript = manuscript.replace(
    old_literature_placeholder, literature_review, 1
)

# 3. Replace the data-availability placeholder with a transparent provenance statement.
old_data_placeholder = (
    "**To complete:** specify the source dataset URL and licence, the data access "
    "conditions, code repository location (if public), and instructions for "
    "reproducing the frozen experiment. Do not publish API keys or private credentials."
)

data_statement = """The project uses station-hour observations of AQI and pollutant variables spanning 2015–2020, stored locally as `station_hour.csv` with associated station metadata. A publicly listed collection entitled *Air Quality Data in India (2015–2020)* is available through Kaggle [2]. The exact downloaded snapshot, the provenance of the local file relative to that collection, and the licence applicable to the copy used in this study must be confirmed from the original acquisition records before submission. This manuscript does not independently establish those terms. The final feature matrix, split assignments, models, and evaluation outputs are retained in the project workspace. No public source-code repository is specified in this draft; any external release must exclude API keys, private credentials, and data not permitted by the source licence."""

if old_data_placeholder not in manuscript:
    raise ValueError(
        "Expected data-availability placeholder was not found. "
        "No replacement was made."
    )

manuscript = manuscript.replace(old_data_placeholder, data_statement, 1)

# 4. Add numbered reference entries with DOI or official source links.
references = """## References

[1] Central Pollution Control Board (CPCB). *National Air Quality Index: Final Report.* Control of Urban Pollution Series, Report CUPS/82/2014-15. Government of India; October 2014. Available at: https://app.cpcbccr.com/ccr_docs/FINAL-REPORT_AQI_.pdf

[2] Rao, R. *Air Quality Data in India (2015–2020).* Kaggle dataset. Available at: https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india. **Author verification required:** confirm that the exact local `station_hour.csv` file came from this distribution and record the applicable version and licence.

[3] Liu H, Li Q, Yu D, Gu Y. Air quality index and air pollutant concentration prediction based on machine learning algorithms. *Applied Sciences.* 2019;9(19):4069. https://doi.org/10.3390/app9194069

[4] Gupta NS, Mohta Y, Heda K, Armaan R, Valarmathi B, Arulkumaran G. Prediction of Air Quality Index Using Machine Learning Techniques: A Comparative Analysis. *Journal of Environmental and Public Health.* 2023;2023:4916267. https://doi.org/10.1155/2023/4916267

[5] Pande CB, Kushwaha NL, Alawi OA, Sammen SS, Sidek LM, Yaseen ZM, Pal SC, Katipoğlu OM. Daily scale air quality index forecasting using bidirectional recurrent neural networks: Case study of Delhi, India. *Environmental Pollution.* 2024;351:124040. https://doi.org/10.1016/j.envpol.2024.124040

[6] Pande CB, Radhadevi L, Satyanarayana MB. Evaluation of machine learning and deep learning models for daily air quality index prediction in Delhi city, India. *Environmental Monitoring and Assessment.* 2024;196:1215. https://doi.org/10.1007/s10661-024-13351-1

[7] Yadav PP, Jat R, Ghude SD, et al. Evaluating the Performance of Regional and Global Forecasting Models for Accurate PM2.5 Prediction and Air Quality Index Assessment in Delhi, India. *Journal of Geophysical Research: Atmospheres.* 2025;130(19):e2025JD043719. https://doi.org/10.1029/2025JD043719

[8] Bergmeir C, Hyndman RJ, Koo B. A note on the validity of cross-validation for evaluating autoregressive time series prediction. *Computational Statistics & Data Analysis.* 2018;120:70–83. https://doi.org/10.1016/j.csda.2017.11.003

[9] Breiman L. Random Forests. *Machine Learning.* 2001;45:5–32. https://doi.org/10.1023/A:1010933404324

[10] Geurts P, Ernst D, Wehenkel L. Extremely randomized trees. *Machine Learning.* 2006;63:3–42. https://doi.org/10.1007/s10994-006-6226-1

[11] Strobl C, Boulesteix AL, Kneib T, Augustin T, Zeileis A. Conditional variable importance for random forests. *BMC Bioinformatics.* 2008;9:307. https://doi.org/10.1186/1471-2105-9-307

[12] Künsch HR. The jackknife and the bootstrap for general stationary observations. *The Annals of Statistics.* 1989;17(3):1217–1241. https://doi.org/10.1214/aos/1176347265
"""

references_marker = "## References"
reference_start = manuscript.rfind(references_marker)
if reference_start < 0:
    raise ValueError("References heading not found in source manuscript.")

manuscript = manuscript[:reference_start] + references

# 5. Save a new manuscript file; the Phase 17.07J source remains unchanged.
new_path = OUT / "phase17_07K_manuscript_with_verified_literature.md"
new_path.write_text(manuscript, encoding="utf-8")

# 6. Save a searchable reference register.
refs = [
    {
        "ref": 1,
        "type": "Official technical report",
        "topic": "India's National AQI framework",
        "title": "National Air Quality Index: Final Report",
        "year": 2014,
        "doi_or_url": "https://app.cpcbccr.com/ccr_docs/FINAL-REPORT_AQI_.pdf",
        "verification_note": "CPCB report record checked."
    },
    {
        "ref": 2,
        "type": "Dataset",
        "topic": "Historical Indian station-hour air-quality data",
        "title": "Air Quality Data in India (2015–2020)",
        "year": "2015–2020 data collection",
        "doi_or_url": "https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india",
        "verification_note": "Candidate source. Confirm exact local file lineage, version, and licence."
    },
    {
        "ref": 3,
        "type": "Peer-reviewed article",
        "topic": "AQI and pollutant prediction using ML",
        "title": "Air quality index and air pollutant concentration prediction based on machine learning algorithms",
        "year": 2019,
        "doi_or_url": "https://doi.org/10.3390/app9194069",
        "verification_note": "Publisher record checked."
    },
    {
        "ref": 4,
        "type": "Peer-reviewed article",
        "topic": "Multi-city AQI machine-learning comparisons",
        "title": "Prediction of Air Quality Index Using Machine Learning Techniques: A Comparative Analysis",
        "year": 2023,
        "doi_or_url": "https://doi.org/10.1155/2023/4916267",
        "verification_note": "Publisher record checked; not treated as a direct numerical benchmark."
    },
    {
        "ref": 5,
        "type": "Peer-reviewed article",
        "topic": "Daily AQI forecasting in Delhi using bidirectional recurrent networks",
        "title": "Daily scale air quality index forecasting using bidirectional recurrent neural networks: Case study of Delhi, India",
        "year": 2024,
        "doi_or_url": "https://doi.org/10.1016/j.envpol.2024.124040",
        "verification_note": "Publisher/PubMed record checked."
    },
    {
        "ref": 6,
        "type": "Peer-reviewed article",
        "topic": "Daily AQI machine-learning and deep-learning models in Delhi",
        "title": "Evaluation of machine learning and deep learning models for daily air quality index prediction in Delhi city, India",
        "year": 2024,
        "doi_or_url": "https://doi.org/10.1007/s10661-024-13351-1",
        "verification_note": "Publisher/index record checked."
    },
    {
        "ref": 7,
        "type": "Peer-reviewed article",
        "topic": "Regional/global PM2.5 forecasting and AQI assessment for Delhi",
        "title": "Evaluating the Performance of Regional and Global Forecasting Models for Accurate PM2.5 Prediction and Air Quality Index Assessment in Delhi, India",
        "year": 2025,
        "doi_or_url": "https://doi.org/10.1029/2025JD043719",
        "verification_note": "Publisher record checked; targets and approach differ from this study."
    },
    {
        "ref": 8,
        "type": "Peer-reviewed methodology article",
        "topic": "Cross-validation assumptions for autoregressive time series",
        "title": "A note on the validity of cross-validation for evaluating autoregressive time series prediction",
        "year": 2018,
        "doi_or_url": "https://doi.org/10.1016/j.csda.2017.11.003",
        "verification_note": "Publisher and author publication record checked."
    },
    {
        "ref": 9,
        "type": "Peer-reviewed methodology article",
        "topic": "Random Forests",
        "title": "Random Forests",
        "year": 2001,
        "doi_or_url": "https://doi.org/10.1023/A:1010933404324",
        "verification_note": "Publisher record checked."
    },
    {
        "ref": 10,
        "type": "Peer-reviewed methodology article",
        "topic": "Extra-Trees",
        "title": "Extremely randomized trees",
        "year": 2006,
        "doi_or_url": "https://doi.org/10.1007/s10994-006-6226-1",
        "verification_note": "Publisher record checked."
    },
    {
        "ref": 11,
        "type": "Peer-reviewed methodology article",
        "topic": "Correlation caveats in tree-model variable importance",
        "title": "Conditional variable importance for random forests",
        "year": 2008,
        "doi_or_url": "https://doi.org/10.1186/1471-2105-9-307",
        "verification_note": "Open-access publisher record checked."
    },
    {
        "ref": 12,
        "type": "Peer-reviewed methodology article",
        "topic": "Bootstrap for dependent stationary observations",
        "title": "The jackknife and the bootstrap for general stationary observations",
        "year": 1989,
        "doi_or_url": "https://doi.org/10.1214/aos/1176347265",
        "verification_note": "Publication metadata checked."
    },
]

register = pd.DataFrame(refs)
register_path = OUT / "phase17_07K_reference_register.csv"
register.to_csv(register_path, index=False)

# 7. Explicit pre-submission checks.
checklist = """# Phase 17.07K — Pre-submission verification checklist

## Required author checks
- [ ] Confirm whether the local `station_hour.csv` was downloaded from the Kaggle collection in reference [2].
- [ ] Record the exact dataset version/download date and verify the licence that applies to the actual file used.
- [ ] Confirm source provenance; do not describe the file as downloaded directly from CPCB unless acquisition records support that statement.
- [ ] Verify that citations and references match the journal's required style.
- [ ] Confirm the publisher's final article metadata, issue/page/article number formatting, and author abbreviations during bibliography formatting.
- [ ] Add a statement about whether code and processed data can be shared, subject to licence and privacy/credential constraints.
- [ ] Do not claim that results from different forecast horizons or validation protocols are directly comparable.
- [ ] Reconfirm that no holdout results were used to select features or tune the model.
- [ ] Keep the limitation that only about 9% of large one-hour AQI changes were detected using the current ±20-point predicted-change threshold.
- [ ] Keep production status separate from retrospective accuracy: validated live data-source requirements remain unmet.
"""
checklist_path = OUT / "phase17_07K_pre_submission_checklist.md"
checklist_path.write_text(checklist, encoding="utf-8")

# 8. Record manifest.
manifest = {
    "phase": "17.07K",
    "source_manuscript": str(SOURCE),
    "new_manuscript": str(new_path),
    "reference_register": str(register_path),
    "pre_submission_checklist": str(checklist_path),
    "references_included": len(refs),
    "online_sources_checked": [
        "CPCB National AQI report",
        "Liu et al. 2019",
        "Gupta et al. 2023",
        "Pande et al. 2024 daily-scale forecasting",
        "Pande et al. 2024 daily AQI model comparison",
        "Yadav et al. 2025 Delhi PM2.5/AQI forecasting",
        "Bergmeir et al. 2018",
        "Breiman 2001",
        "Geurts et al. 2006",
        "Strobl et al. 2008",
        "Künsch 1989",
        "Kaggle dataset listing"
    ],
    "source_manuscript_overwritten": False,
    "dataset_license_verified_for_local_file": False,
    "production_enabled": False
}
manifest_path = OUT / "phase17_07K_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# 9. Basic quality checks.
updated = new_path.read_text(encoding="utf-8")
print("\n[1] New manuscript:", new_path)
print("[2] Reference register:", register_path)
print("[3] Checklist:", checklist_path)
print("[4] Reference count:", len(refs))
print("[5] Old literature placeholder remains:", old_literature_placeholder in updated)
print("[6] Old data-availability placeholder remains:", old_data_placeholder in updated)
print("[7] Old source manuscript overwritten: False")
print("[8] Local dataset licence independently verified: False")
print("[9] Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07L — MANUSCRIPT QA AND NUMERIC CONSISTENCY AUDIT

from pathlib import Path
import json
import re
import math
import pandas as pd
import numpy as np

print("=" * 78)
print("PHASE 17.07L — MANUSCRIPT QUALITY AUDIT")
print("=" * 78)

# 1. Resolve existing project paths.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next((p for p in roots if p.exists() and (p / "outputs").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project folder not found.")

SOURCE = (
    ROOT / "outputs/research_paper/phase17_07K_verified_literature/"
    "phase17_07K_manuscript_with_verified_literature.md"
)
EVIDENCE = ROOT / "outputs/research_paper/phase17_07I_evidence_package/tables"

for p in [
    SOURCE,
    EVIDENCE / "holdout_full_support.csv",
    EVIDENCE / "holdout_common_support.csv",
    EVIDENCE / "bootstrap_results.csv",
    EVIDENCE / "ensemble_permutation_importance_top20.csv",
]:
    if not p.exists():
        raise FileNotFoundError(f"Required source missing: {p}")

OUT = ROOT / "outputs/research_paper/phase17_07L_final_quality_audit"
OUT.mkdir(parents=True, exist_ok=True)

manuscript = SOURCE.read_text(encoding="utf-8")
original_manuscript = manuscript

# 2. Fix leftover editorial language and accurately label the bootstrap method.
editorial_old = (
    "The manuscript should cite the exact dataset/source documentation and describe "
    "any filtering and quality-control steps using the project's saved preprocessing protocol."
)
editorial_new = (
    "The local dataset was processed according to the project's saved preprocessing "
    "protocol. The exact source snapshot and licence applicable to the local copy "
    "remain subject to the provenance check described in the Data and code availability section."
)

editorial_fixed = editorial_old in manuscript
manuscript = manuscript.replace(editorial_old, editorial_new)

# The code resamples calendar-day clusters, not contiguous multi-day sequences.
manuscript = manuscript.replace("day-block bootstrap", "day-cluster bootstrap")
manuscript = manuscript.replace("day-block resampling", "day-cluster resampling")

# 3. Audit inline citation numbering against the numbered bibliography.
reference_heading = "## References"
if reference_heading not in manuscript:
    raise ValueError("References section is missing.")

body, reference_section = manuscript.split(reference_heading, 1)

inline_citations = sorted({
    int(n) for n in re.findall(r"\[(\d{1,2})\]", body)
})
reference_ids = [
    int(n) for n in re.findall(r"^\[(\d+)\]\s", reference_section, flags=re.MULTILINE)
]

duplicate_reference_ids = sorted({
    n for n in reference_ids if reference_ids.count(n) > 1
})
missing_reference_ids = sorted(set(inline_citations) - set(reference_ids))
uncited_reference_ids = sorted(set(reference_ids) - set(inline_citations))

reference_url_count = len(
    re.findall(r"^\[\d+\].*https?://", reference_section, flags=re.MULTILINE)
)

# 4. Check for leftover manuscript-editing instructions.
editorial_patterns = [
    r"The manuscript should",
    r"\*\*To complete",
    r"Literature review to complete",
    r"\bTODO\b",
    r"\[CITE HERE\]",
    r"Add sources on",
]

editorial_matches = []
for pattern in editorial_patterns:
    for match in re.finditer(pattern, manuscript, flags=re.IGNORECASE):
        editorial_matches.append({
            "pattern": pattern,
            "text": manuscript[max(0, match.start()-80):match.end()+120].replace("\n", " ")
        })

# Keep the data-provenance limitation as an explicit author check,
# rather than treating an unverified licence as if it had been confirmed.
provenance_warning_present = (
    "exact source snapshot and licence" in manuscript.lower()
    and "confirm" in manuscript.lower()
)

# 5. Load saved evidence for numeric consistency checks.
full = pd.read_csv(EVIDENCE / "holdout_full_support.csv")
common = pd.read_csv(EVIDENCE / "holdout_common_support.csv")
bootstrap = pd.read_csv(EVIDENCE / "bootstrap_results.csv")
importance = pd.read_csv(EVIDENCE / "ensemble_permutation_importance_top20.csv")

def get_row(frame, column, value):
    rows = frame[frame[column].astype(str).eq(str(value))]
    if rows.empty:
        raise ValueError(f"Missing expected saved evidence row: {column}={value}")
    return rows.iloc[0]

full_ensemble = get_row(full, "Model", "Weighted ensemble")
common_ensemble = get_row(common, "Model", "Weighted ensemble")
common_persistence = get_row(common, "Model", "Persistence")
boot_ensemble = get_row(bootstrap, "Model", "Weighted ensemble")
aqi_importance = get_row(importance, "Feature", "AQI")
rolling_importance = get_row(importance, "Feature", "AQI_rolling_mean_3h")

numeric_checks = [
    {
        "item": "Full-holdout row count",
        "expected": 68352,
        "observed": int(full_ensemble["N"]),
        "pass": int(full_ensemble["N"]) == 68352,
    },
    {
        "item": "Full-holdout ensemble MAE",
        "expected": 3.216121,
        "observed": float(full_ensemble["MAE"]),
        "pass": abs(float(full_ensemble["MAE"]) - 3.216121) < 1e-5,
    },
    {
        "item": "Full-holdout ensemble RMSE",
        "expected": 7.658390,
        "observed": float(full_ensemble["RMSE"]),
        "pass": abs(float(full_ensemble["RMSE"]) - 7.658390) < 1e-5,
    },
    {
        "item": "Matched-support row count",
        "expected": 68285,
        "observed": int(common_ensemble["N"]),
        "pass": int(common_ensemble["N"]) == 68285,
    },
    {
        "item": "Matched-support ensemble MAE",
        "expected": 3.178058,
        "observed": float(common_ensemble["MAE"]),
        "pass": abs(float(common_ensemble["MAE"]) - 3.178058) < 1e-5,
    },
    {
        "item": "Matched-support ensemble RMSE",
        "expected": 7.495249,
        "observed": float(common_ensemble["RMSE"]),
        "pass": abs(float(common_ensemble["RMSE"]) - 7.495249) < 1e-5,
    },
    {
        "item": "Matched-support persistence MAE",
        "expected": 3.855634,
        "observed": float(common_persistence["MAE"]),
        "pass": abs(float(common_persistence["MAE"]) - 3.855634) < 1e-5,
    },
    {
        "item": "Matched-support persistence RMSE",
        "expected": 8.840011,
        "observed": float(common_persistence["RMSE"]),
        "pass": abs(float(common_persistence["RMSE"]) - 8.840011) < 1e-5,
    },
    {
        "item": "Ensemble bootstrap MAE improvement (%)",
        "expected": 17.5737,
        "observed": float(boot_ensemble["Relative_MAE_Improvement_pct"]),
        "pass": abs(float(boot_ensemble["Relative_MAE_Improvement_pct"]) - 17.5737) < 0.01,
    },
    {
        "item": "Ensemble bootstrap RMSE improvement (%)",
        "expected": 15.2122,
        "observed": float(boot_ensemble["Relative_RMSE_Improvement_pct"]),
        "pass": abs(float(boot_ensemble["Relative_RMSE_Improvement_pct"]) - 15.2122) < 0.01,
    },
    {
        "item": "AQI permutation importance (%)",
        "expected": 91.4414,
        "observed": float(aqi_importance["Relative_Importance_%"]),
        "pass": abs(float(aqi_importance["Relative_Importance_%"]) - 91.4414) < 0.01,
    },
    {
        "item": "3-hour AQI rolling mean importance (%)",
        "expected": 6.1774,
        "observed": float(rolling_importance["Relative_Importance_%"]),
        "pass": abs(float(rolling_importance["Relative_Importance_%"]) - 6.1774) < 0.01,
    },
]

numeric_df = pd.DataFrame(numeric_checks)
numeric_df.to_csv(OUT / "phase17_07L_numeric_consistency.csv", index=False)

# 6. Record citation and editorial checks.
citation_report = {
    "inline_citation_ids": inline_citations,
    "bibliography_reference_ids": reference_ids,
    "duplicate_reference_ids": duplicate_reference_ids,
    "inline_citations_missing_reference": missing_reference_ids,
    "references_not_cited_in_main_text": uncited_reference_ids,
    "reference_entries_with_http_url": reference_url_count,
    "editorial_instruction_matches_after_cleanup": editorial_matches,
    "leftover_editorial_sentence_was_fixed": editorial_fixed,
    "dataset_provenance_warning_present": provenance_warning_present,
    "bootstrap_term_standardized": "day-block bootstrap" not in manuscript.lower(),
}

(OUT / "phase17_07L_citation_editorial_audit.json").write_text(
    json.dumps(citation_report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# 7. Write reviewed copy and source manifest.
FINAL_MD = OUT / "phase17_07L_reviewed_manuscript.md"
FINAL_MD.write_text(manuscript, encoding="utf-8")

all_citations_valid = (
    len(missing_reference_ids) == 0
    and len(duplicate_reference_ids) == 0
    and len(editorial_matches) == 0
    and provenance_warning_present
    and citation_report["bootstrap_term_standardized"]
)
all_numeric_valid = bool(numeric_df["pass"].all())

summary = {
    "phase": "17.07L",
    "reviewed_manuscript": str(FINAL_MD),
    "numeric_audit_csv": str(OUT / "phase17_07L_numeric_consistency.csv"),
    "citation_audit_json": str(OUT / "phase17_07L_citation_editorial_audit.json"),
    "inline_citations": inline_citations,
    "bibliography_entries": len(reference_ids),
    "citation_and_editorial_checks_pass": bool(all_citations_valid),
    "numeric_consistency_checks_pass": bool(all_numeric_valid),
    "dataset_source_and_local_licence_verified": False,
    "submission_ready": bool(all_citations_valid and all_numeric_valid),
    "models_retrained": False,
    "models_retuned": False,
    "production_enabled": False,
}
(OUT / "phase17_07L_qa_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# 8. Print results.
print("\n[1] CITATION AND EDITORIAL AUDIT")
print("Inline citation IDs:", inline_citations)
print("Bibliography IDs:", reference_ids)
print("Missing bibliography entries:", missing_reference_ids)
print("Duplicate reference numbers:", duplicate_reference_ids)
print("References not cited in text:", uncited_reference_ids)
print("Editorial instruction fixed:", editorial_fixed)
print("Remaining editorial instruction matches:", len(editorial_matches))
print("Dataset source/licence warning retained:", provenance_warning_present)
print("Bootstrap terminology standardized:", citation_report["bootstrap_term_standardized"])

print("\n[2] NUMERIC CONSISTENCY CHECKS")
print(numeric_df.to_string(index=False))

print("\n[3] SUMMARY")
print(json.dumps(summary, indent=2))
print("\nReviewed manuscript:", FINAL_MD)
print("Audit folder:", OUT)
print("=" * 78)

In [ ]:
# PHASE 17.07L-R1 — CORRECTED CITATION AUDIT AND SUBMISSION GATE

from pathlib import Path
import json
import re
import pandas as pd

print("=" * 78)
print("PHASE 17.07L-R1 — CORRECTED FINAL QUALITY AUDIT")
print("=" * 78)

# 1. Resolve project and existing manuscript.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in roots if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project folder not found.")

SOURCE = (
    ROOT / "outputs/research_paper/phase17_07L_final_quality_audit/"
    "phase17_07L_reviewed_manuscript.md"
)
NUMERIC_AUDIT = (
    ROOT / "outputs/research_paper/phase17_07L_final_quality_audit/"
    "phase17_07L_numeric_consistency.csv"
)
K_MANIFEST = (
    ROOT / "outputs/research_paper/phase17_07K_verified_literature/"
    "phase17_07K_manifest.json"
)

for p in [SOURCE, NUMERIC_AUDIT]:
    if not p.exists():
        raise FileNotFoundError(f"Required audit file not found: {p}")

OUT = ROOT / "outputs/research_paper/phase17_07L_R1_final_quality_audit"
OUT.mkdir(parents=True, exist_ok=True)

text = SOURCE.read_text(encoding="utf-8")

# 2. Parse individual and grouped citations, e.g. [5] and [5, 6].
if "## References" not in text:
    raise ValueError("A numbered References section was not found.")

body, references_section = text.split("## References", 1)

citation_ids = set()
for group in re.findall(r"\[((?:\d+\s*,\s*)*\d+)\]", body):
    for number in re.findall(r"\d+", group):
        citation_ids.add(int(number))

reference_ids = [
    int(n)
    for n in re.findall(
        r"^\[(\d+)\]\s",
        references_section,
        flags=re.MULTILINE
    )
]

duplicate_ids = sorted({
    n for n in reference_ids if reference_ids.count(n) > 1
})
missing_entries = sorted(citation_ids - set(reference_ids))
uncited_entries = sorted(set(reference_ids) - citation_ids)

# 3. Check for leftover editorial placeholders.
editorial_patterns = [
    r"The manuscript should",
    r"\*\*To complete",
    r"Literature review to complete",
    r"\bTODO\b",
    r"\[CITE HERE\]",
    r"Add sources on",
]

editorial_matches = []
for pattern in editorial_patterns:
    for match in re.finditer(pattern, text, flags=re.IGNORECASE):
        editorial_matches.append({
            "pattern": pattern,
            "context": text[max(0, match.start()-60):match.end()+100]
                      .replace("\n", " ")
        })

# 4. Recheck the saved numeric consistency audit.
numeric = pd.read_csv(NUMERIC_AUDIT)
if "pass" not in numeric.columns:
    raise KeyError("Expected 'pass' column missing from numeric audit.")

numeric_pass = bool(numeric["pass"].fillna(False).astype(bool).all())
numeric.to_csv(OUT / "phase17_07L_R1_numeric_consistency.csv", index=False)

# 5. Keep provenance/licence status conservative.
# It remains unverified unless the saved provenance record explicitly says otherwise.
provenance_verified = False
if K_MANIFEST.exists():
    try:
        k_data = json.loads(K_MANIFEST.read_text(encoding="utf-8"))
        provenance_verified = (
            k_data.get("dataset_license_verified_for_local_file") is True
        )
    except (json.JSONDecodeError, OSError):
        provenance_verified = False

# The source/licence status must not be inferred from successful citation checks.
citation_checks_pass = (
    len(missing_entries) == 0
    and len(duplicate_ids) == 0
    and len(uncited_entries) == 0
)
editorial_checks_pass = len(editorial_matches) == 0

submission_ready = (
    citation_checks_pass
    and editorial_checks_pass
    and numeric_pass
    and provenance_verified
)

if not provenance_verified:
    blockers = [
        "Verify the exact provenance/version of the local station-hour dataset.",
        "Confirm the licence and reuse terms applying to that exact downloaded copy.",
    ]
else:
    blockers = []

if not citation_checks_pass:
    blockers.append("Resolve any missing, duplicate, or uncited reference entries.")
if not editorial_checks_pass:
    blockers.append("Remove remaining editorial placeholders.")
if not numeric_pass:
    blockers.append("Resolve failed numeric consistency checks.")

report = {
    "phase": "17.07L-R1",
    "source_manuscript": str(SOURCE),
    "inline_citation_ids": sorted(citation_ids),
    "bibliography_reference_ids": reference_ids,
    "missing_bibliography_entries": missing_entries,
    "duplicate_reference_ids": duplicate_ids,
    "references_not_cited_in_text": uncited_entries,
    "citation_checks_pass": citation_checks_pass,
    "editorial_matches": editorial_matches,
    "editorial_checks_pass": editorial_checks_pass,
    "numeric_consistency_checks_pass": numeric_pass,
    "dataset_source_and_local_licence_verified": provenance_verified,
    "submission_ready": submission_ready,
    "submission_gate_status": (
        "PASS"
        if submission_ready
        else "PENDING — verify dataset provenance/licence and resolve any listed checks"
    ),
    "remaining_blockers": blockers,
    "models_retrained": False,
    "models_retuned": False,
    "production_enabled": False,
}

(OUT / "phase17_07L_R1_quality_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

(OUT / "phase17_07L_R1_remaining_blockers.txt").write_text(
    "\n".join(f"- {item}" for item in blockers)
    if blockers else "No outstanding blockers recorded.",
    encoding="utf-8"
)

print("\n[1] CITATION CHECK")
print("Inline citation IDs:", sorted(citation_ids))
print("Reference IDs:", reference_ids)
print("Missing bibliography entries:", missing_entries)
print("Duplicate reference IDs:", duplicate_ids)
print("References not cited:", uncited_entries)
print("Citation checks pass:", citation_checks_pass)

print("\n[2] CONTENT AND NUMERIC CHECKS")
print("Remaining editorial placeholders:", len(editorial_matches))
print("Numeric consistency checks pass:", numeric_pass)

print("\n[3] DATASET PROVENANCE AND LICENCE")
print("Verified for the exact local file:", provenance_verified)

print("\n[4] FINAL GATE")
print("Submission ready:", submission_ready)
print("Status:", report["submission_gate_status"])
if blockers:
    print("Remaining blockers:")
    for item in blockers:
        print("-", item)

print("\nSaved report:", OUT / "phase17_07L_R1_quality_report.json")
print("Saved blocker list:", OUT / "phase17_07L_R1_remaining_blockers.txt")
print("Original manuscript overwritten: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07M — LOCAL DATASET PROVENANCE AUDIT
# Read-only audit; does not change data, models, or the manuscript.

from pathlib import Path
import hashlib
import json
import re
import pandas as pd
from datetime import datetime, timezone

print("=" * 78)
print("PHASE 17.07M — LOCAL DATASET PROVENANCE AUDIT")
print("=" * 78)

# 1. Find the existing project root without remounting Google Drive.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]

ROOT = next(
    (p for p in roots if p.exists() and (p / "outputs").exists()),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Verify the existing Google Drive mount."
    )

OUT = ROOT / "outputs/research_paper/phase17_07M_provenance_audit"
OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)

# 2. Find likely original data files by filename.
candidate_names = {
    "station_hour.csv",
    "stations.csv",
    "station_day.csv",
    "city_hour.csv",
    "city_day.csv",
    "air-quality-data-in-india.zip",
    "air_quality_data_in_india.zip",
}

source_files = []
for p in ROOT.rglob("*"):
    if not p.is_file():
        continue

    lower_name = p.name.lower()

    if lower_name in candidate_names or (
        lower_name.endswith(".zip")
        and ("air-quality-data" in lower_name
             or "air_quality_data" in lower_name)
    ):
        try:
            stat = p.stat()
            source_files.append({
                "path": str(p),
                "filename": p.name,
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime, tz=timezone.utc
                ).isoformat(),
                "extension": p.suffix.lower(),
            })
        except OSError:
            continue

source_df = pd.DataFrame(source_files)
if len(source_df):
    source_df = source_df.sort_values("path").reset_index(drop=True)

print(f"\n[1] Likely source files found under project root: {len(source_df)}")
if len(source_df):
    print(source_df.to_string(index=False))
else:
    print("No original CSV/ZIP source files with the expected names were found.")

# 3. Compute SHA256 fingerprints for matching CSV/ZIP files.
# The hash identifies the local file content, but alone does not prove its origin.
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

hash_records = []
for item in source_files:
    p = Path(item["path"])
    try:
        digest = sha256_file(p)
        hash_records.append({
            "path": str(p),
            "filename": p.name,
            "size_bytes": item["size_bytes"],
            "sha256": digest,
            "hash_status": "completed",
        })
    except Exception as exc:
        hash_records.append({
            "path": str(p),
            "filename": p.name,
            "size_bytes": item["size_bytes"],
            "sha256": None,
            "hash_status": f"failed: {type(exc).__name__}",
        })

hash_df = pd.DataFrame(hash_records)
hash_df.to_csv(
    OUT / "phase17_07M_local_file_fingerprints.csv",
    index=False
)

print("\n[2] File fingerprints")
if len(hash_df):
    print(hash_df.to_string(index=False))
else:
    print("No files available to hash.")

# 4. Search small project documentation/configuration files for lineage clues.
# Credentials, hidden folders, and files whose names suggest secrets are excluded.
search_terms = [
    "rohanrao/air-quality-data-in-india",
    "kaggle.com/datasets/rohanrao",
    "kaggle datasets download",
    "station_hour.csv",
    "Air Quality Data in India (2015 - 2020)",
    "CC0 1.0",
    "creativecommons.org/publicdomain/zero/1.0",
    "cpcb.nic.in",
]

excluded_name_terms = [
    "kaggle.json",
    "credential",
    "secret",
    "token",
    "password",
    ".env",
    "private_key",
]

allowed_suffixes = {
    ".md", ".txt", ".json", ".yaml", ".yml",
    ".toml", ".ipynb", ".ini"
}

documentation_hits = []

for p in ROOT.rglob("*"):
    if not p.is_file() or p.suffix.lower() not in allowed_suffixes:
        continue

    name_lower = p.name.lower()
    path_lower = str(p).lower()

    if any(term in name_lower for term in excluded_name_terms):
        continue
    if any(part in path_lower for part in [
        "/.kaggle/", "/.config/", "/secrets/", "/credentials/"
    ]):
        continue

    try:
        if p.stat().st_size > 2_000_000:
            continue
        content = p.read_text(encoding="utf-8", errors="ignore")
    except (OSError, PermissionError):
        continue

    content_lower = content.lower()
    found_terms = [term for term in search_terms if term.lower() in content_lower]

    if found_terms:
        documentation_hits.append({
            "path": str(p),
            "matched_terms": "; ".join(found_terms),
        })

hits_df = pd.DataFrame(documentation_hits)
if len(hits_df):
    hits_df = hits_df.sort_values("path").reset_index(drop=True)

hits_df.to_csv(
    OUT / "phase17_07M_documentation_lineage_hits.csv",
    index=False
)

print(f"\n[3] Project documentation files with matching lineage terms: {len(hits_df)}")
if len(hits_df):
    print(hits_df.to_string(index=False, max_rows=100))
else:
    print("No lineage terms found in the searched documentation files.")

# 5. Save the externally identified candidate metadata separately from local evidence.
external_metadata = {
    "dataset_title": "Urban air pollution measurements, India, 2015-2020",
    "catalogue": "Transport Data Commons / TUMI Data Portal",
    "catalogue_url": (
        "https://portal.transport-data.org/@tumi/"
        "urban-air-pollution-india-2015-2020"
    ),
    "catalogue_updated": "2026-08-20",
    "catalogue_states_compiler": "Rohan Rao",
    "catalogue_states_upstream_provider": "Central Pollution Control Board, India",
    "linked_kaggle_dataset": (
        "https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india"
    ),
    "catalogue_listed_license": "CC0 1.0",
    "important_limitation": (
        "External catalogue metadata describes the candidate dataset. "
        "It does not by itself prove that a particular local file is byte-identical "
        "to the referenced download or establish its acquisition history."
    ),
}
(OUT / "phase17_07M_external_catalogue_metadata.json").write_text(
    json.dumps(external_metadata, indent=2),
    encoding="utf-8"
)

# 6. Create a conservative provenance assessment.
source_csv_found = any(
    Path(item["path"]).name.lower() == "station_hour.csv"
    for item in source_files
)
kaggle_lineage_mention_found = (
    len(hits_df) > 0
    and hits_df["matched_terms"].str.contains(
        r"rohanrao/air-quality-data-in-india|kaggle.com/datasets/rohanrao",
        case=False,
        regex=True
    ).any()
)

assessment = {
    "phase": "17.07M",
    "source_file_inventory_count": int(len(source_df)),
    "station_hour_csv_found_in_project_root": bool(source_csv_found),
    "kaggle_lineage_mention_found_in_project_docs": bool(
        kaggle_lineage_mention_found
    ),
    "local_sha256_fingerprints_written": str(
        OUT / "phase17_07M_local_file_fingerprints.csv"
    ),
    "documentation_hits_written": str(
        OUT / "phase17_07M_documentation_lineage_hits.csv"
    ),
    "external_catalogue_metadata_saved": True,
    "exact_local_dataset_provenance_confirmed": False,
    "local_file_license_applicability_confirmed": False,
    "submission_gate": "PENDING MANUAL PROVENANCE CONFIRMATION",
    "models_retrained": False,
    "models_retuned": False,
    "production_enabled": False,
}

(OUT / "phase17_07M_assessment.json").write_text(
    json.dumps(assessment, indent=2),
    encoding="utf-8"
)

print("\n[4] ASSESSMENT")
print(json.dumps(assessment, indent=2))

print("\nAudit folder:", OUT)
print("Original source files modified: False")
print("Models retrained: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07N — NOTEBOOK LINEAGE AND RAW-SCHEMA AUDIT

from pathlib import Path
import json
import hashlib
import pandas as pd
import re

print("=" * 78)
print("PHASE 17.07N — NOTEBOOK LINEAGE AUDIT")
print("=" * 78)

ROOT_CANDIDATES = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in ROOT_CANDIDATES if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Existing project root not found.")

OUT = ROOT / "outputs/research_paper/phase17_07N_notebook_lineage"
OUT.mkdir(parents=True, exist_ok=True)

# 1. Inspect only project notebooks that may describe the data acquisition.
notebook_paths = [
    ROOT / "notebooks/01_data_understanding.ipynb",
    ROOT / "notebooks/Delhi_AQI_Research_Paper_Completion_Phase_17_07.ipynb",
]

terms = [
    "rohanrao/air-quality-data-in-india",
    "kaggle datasets download",
    "kagglehub.dataset_download",
    "station_hour.csv",
    "stations.csv",
    "data/raw",
    "read_csv",
    "download_files",
    "unzip",
]

# Never print lines that may contain credential material.
sensitive_patterns = [
    r"api[_ -]?key",
    r"password",
    r"secret",
    r"token\s*=",
    r"kaggle\.json",
    r"authorization",
    r"bearer\s",
]

def is_sensitive_line(line):
    return any(
        re.search(pattern, line, flags=re.IGNORECASE)
        for pattern in sensitive_patterns
    )

lineage_evidence = []
notebook_summary = []

for nb_path in notebook_paths:
    if not nb_path.exists():
        notebook_summary.append({
            "notebook": str(nb_path),
            "exists": False,
            "cells": 0,
        })
        continue

    try:
        nb = json.loads(nb_path.read_text(encoding="utf-8"))
    except Exception as exc:
        notebook_summary.append({
            "notebook": str(nb_path),
            "exists": True,
            "error": str(exc),
        })
        continue

    cells = nb.get("cells", [])
    notebook_summary.append({
        "notebook": str(nb_path),
        "exists": True,
        "cells": len(cells),
    })

    print(f"\n--- Notebook: {nb_path.name} ---")
    print(f"Cells: {len(cells)}")

    displayed = 0
    for i, cell in enumerate(cells):
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)

        for line_no, line in enumerate(source.splitlines(), start=1):
            low = line.lower()

            if is_sensitive_line(line):
                continue

            matched = [term for term in terms if term.lower() in low]
            if not matched:
                continue

            # Retain only source-relevant lines; exclude generic read_csv calls.
            relevant = (
                "kaggle" in low
                or "station_hour.csv" in low
                or "stations.csv" in low
                or "data/raw" in low
                or "download_files" in low
                or "unzip" in low
            )
            if not relevant:
                continue

            snippet = line.strip()
            if len(snippet) > 280:
                snippet = snippet[:280] + " ..."

            item = {
                "notebook": str(nb_path),
                "cell_index": i,
                "line_number": line_no,
                "matched_terms": matched,
                "snippet": snippet,
            }
            lineage_evidence.append(item)

            if displayed < 20:
                print(
                    f"cell={i}, line={line_no}, "
                    f"terms={'; '.join(matched)}"
                )
                print(" ", snippet)
                displayed += 1

    if displayed == 0:
        print("No non-sensitive data-download or source-loading lines found.")

# 2. Inspect the original source CSV headers without loading the full datasets.
raw_files = [
    ROOT / "data/raw/station_hour.csv",
    ROOT / "data/raw/stations.csv",
]

schema_records = []
for path in raw_files:
    if not path.exists():
        schema_records.append({
            "path": str(path),
            "exists": False,
        })
        continue

    try:
        columns = list(pd.read_csv(path, nrows=0).columns)
        schema_records.append({
            "path": str(path),
            "exists": True,
            "size_bytes": int(path.stat().st_size),
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            "column_count": len(columns),
            "columns": columns,
        })
    except Exception as exc:
        schema_records.append({
            "path": str(path),
            "exists": True,
            "error": str(exc),
        })

print("\n[2] RAW FILE SCHEMAS")
for rec in schema_records:
    print(f"\nFile: {rec['path']}")
    print("Exists:", rec.get("exists"))
    if rec.get("exists") and "columns" in rec:
        print("Size bytes:", rec["size_bytes"])
        print("SHA-256:", rec["sha256"])
        print("Column count:", rec["column_count"])
        print("Columns:", rec["columns"])
    elif "error" in rec:
        print("Header read error:", rec["error"])

expected_station_columns = [
    "StationId", "Datetime", "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene",
    "AQI", "AQI_Bucket"
]

station_schema = next(
    (r for r in schema_records
     if Path(r["path"]).name == "station_hour.csv" and r.get("exists")),
    None
)

schema_match = (
    station_schema is not None
    and station_schema.get("columns") == expected_station_columns
)

# 3. Assess what the evidence actually establishes.
evidence_text = "\n".join(
    e["snippet"].lower() for e in lineage_evidence
)

source_declared = "rohanrao/air-quality-data-in-india" in evidence_text
download_command_found = (
    "kaggle datasets download" in evidence_text
    or "kagglehub.dataset_download" in evidence_text
    or "download_files" in evidence_text
)
local_data_path_referenced = (
    "station_hour.csv" in evidence_text
    and "data/raw" in evidence_text
)

assessment = {
    "phase": "17.07N",
    "notebooks_inspected": notebook_summary,
    "lineage_evidence_count": len(lineage_evidence),
    "source_collection_declared_in_inspected_lines": source_declared,
    "download_command_found_in_inspected_lines": download_command_found,
    "local_data_path_referenced_in_inspected_lines": local_data_path_referenced,
    "station_hour_schema_matches_expected_columns": schema_match,
    "raw_schema_records": schema_records,
    "interpretation": (
        "The inspected notebook lines provide evidence of declared source and/or "
        "loading behaviour. Schema agreement alone does not prove byte-identical "
        "origin. Preserve pending status unless the inspected records support a "
        "traceable acquisition path to the exact local file."
    ),
    "exact_local_file_provenance_confirmed": False,
    "local_file_license_applicability_confirmed": False,
    "submission_gate": "PENDING REVIEW OF THE EVIDENCE BELOW",
    "models_retrained": False,
    "production_enabled": False,
}

(OUT / "phase17_07N_lineage_evidence.json").write_text(
    json.dumps(lineage_evidence, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(OUT / "phase17_07N_source_file_schemas.json").write_text(
    json.dumps(schema_records, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(OUT / "phase17_07N_assessment.json").write_text(
    json.dumps(assessment, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n[3] LINEAGE ASSESSMENT")
print(json.dumps(assessment, indent=2, ensure_ascii=False))
print("\nSaved audit folder:", OUT)
print("Original data files modified: False")
print("Models retrained: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07O — ORIGINAL NOTEBOOK DATA-LOADING REVIEW
# Read-only inspection. Does not execute notebook cells or modify source data.

from pathlib import Path
import json
import re

print("=" * 78)
print("PHASE 17.07O — ORIGINAL NOTEBOOK DATA-LOADING REVIEW")
print("=" * 78)

roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in roots if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Existing Delhi-AQI-Prediction folder not found.")

NOTEBOOK = ROOT / "notebooks/01_data_understanding.ipynb"
if not NOTEBOOK.exists():
    raise FileNotFoundError(f"Original notebook not found: {NOTEBOOK}")

OUT = ROOT / "outputs/research_paper/phase17_07O_original_loading_review"
OUT.mkdir(parents=True, exist_ok=True)

notebook = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
cells = notebook.get("cells", [])

# Search source code and explanatory Markdown, but do not execute anything.
patterns = [
    r"station_hour\.csv",
    r"stations\.csv",
    r"\bRAW_PATH\b",
    r"\bread_csv\s*\(",
    r"\bkaggle\b",
    r"\bdownload\b",
    r"\brequests\.(get|post)\s*\(",
    r"\burlretrieve\s*\(",
    r"\bwget\b",
    r"\bcurl\b",
    r"\bcpc[b]?c\b",
    r"source\s*(data|file|dataset)?",
]

credential_patterns = [
    r"api[_ -]?key",
    r"password",
    r"secret",
    r"access[_ -]?token",
    r"authorization",
    r"bearer\s",
    r"kaggle\.json",
]

def redact_line(line):
    if any(re.search(p, line, re.IGNORECASE) for p in credential_patterns):
        return "[REDACTED: potential credential-bearing line]"
    # Remove common query-string key/token values from URLs.
    line = re.sub(
        r"([?&](?:key|token|api_key|access_token|password)=)[^&\s]+",
        r"\1[REDACTED]",
        line,
        flags=re.IGNORECASE,
    )
    return line

relevant_cells = []
cell_records = []

for idx, cell in enumerate(cells):
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    lines = source.splitlines()
    matched_lines = [
        i for i, line in enumerate(lines)
        if any(re.search(p, line, re.IGNORECASE) for p in patterns)
    ]

    if not matched_lines:
        continue

    # Display matching lines with a small amount of surrounding context.
    selected = set()
    for i in matched_lines:
        selected.update(range(max(0, i - 3), min(len(lines), i + 4)))

    selected_lines = sorted(selected)
    excerpt = "\n".join(
        f"{i+1:04d}: {redact_line(lines[i])}"
        for i in selected_lines
    )

    record = {
        "cell_index": idx,
        "cell_type": cell.get("cell_type"),
        "execution_count": cell.get("execution_count"),
        "source_line_count": len(lines),
        "matched_line_numbers": [i + 1 for i in matched_lines],
        "excerpt": excerpt,
    }
    relevant_cells.append(record)

    print(f"\n--- CELL {idx} | {cell.get('cell_type')} ---")
    print("Execution count:", cell.get("execution_count"))
    print(excerpt)

# Save the review so the provenance reasoning is reproducible.
review_path = OUT / "phase17_07O_original_loading_cells.json"
review_path.write_text(
    json.dumps(relevant_cells, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Explicitly do not infer lineage from mere URL mentions or search-term strings.
source_cells = [
    c for c in relevant_cells
    if c["cell_type"] == "code"
    and any(
        re.search(
            pattern,
            c["excerpt"],
            re.IGNORECASE,
        )
        for pattern in [
            r"\bkaggle\s+datasets\s+download\b",
            r"\bkagglehub\.dataset_download\s*\(",
            r"\brequests\.get\s*\(",
            r"\burlretrieve\s*\(",
            r"\bwget\s+",
            r"\bcurl\s+",
        ]
    )
]

assessment = {
    "phase": "17.07O",
    "source_notebook": str(NOTEBOOK),
    "relevant_cells_found": len(relevant_cells),
    "candidate_acquisition_code_cells": [
        {
            "cell_index": c["cell_index"],
            "execution_count": c["execution_count"],
        }
        for c in source_cells
    ],
    "review_json": str(review_path),
    "interpretation": (
        "Candidate acquisition code requires manual review; a textual match "
        "does not prove that a download succeeded or that it produced the current local file."
    ),
    "exact_file_provenance_confirmed": False,
    "local_license_applicability_confirmed": False,
    "models_retrained": False,
    "production_enabled": False,
}

(OUT / "phase17_07O_assessment.json").write_text(
    json.dumps(assessment, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n[SUMMARY]")
print(json.dumps(assessment, indent=2))
print("\nReview folder:", OUT)
print("Original notebook executed: False")
print("Source data modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.07P — PROVENANCE-SAFE MANUSCRIPT
# Creates a new copy. Does not alter source data or frozen models.

from pathlib import Path
import json
import re
import pandas as pd

print("=" * 78)
print("PHASE 17.07P — PROVENANCE-SAFE MANUSCRIPT")
print("=" * 78)

ROOT_CANDIDATES = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in ROOT_CANDIDATES if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Existing project root not found.")

SOURCE = (
    ROOT / "outputs/research_paper/phase17_07L_final_quality_audit/"
    "phase17_07L_reviewed_manuscript.md"
)
FINGERPRINTS = (
    ROOT / "outputs/research_paper/phase17_07M_provenance_audit/"
    "phase17_07M_local_file_fingerprints.csv"
)
OUT = ROOT / "outputs/research_paper/phase17_07P_provenance_safe_manuscript"
OUT.mkdir(parents=True, exist_ok=True)

if not SOURCE.exists():
    raise FileNotFoundError(f"Reviewed manuscript missing: {SOURCE}")

text = SOURCE.read_text(encoding="utf-8")
original_text = text

# 1. Replace the data availability section with wording supported by the audit.
replacement_data_section = """## Data and code availability

The analysis used two local data files: `station_hour.csv`, containing hourly station observations with AQI and pollutant measurements, and `stations.csv`, containing station metadata. The original project notebook reads these files from the project's local `data/raw` directory. It does not document how the files were acquired.

The Transformative Urban Mobility Initiative (TUMI) Data Portal lists a public collection entitled *Urban air pollution measurements, India, 2015–2020*. Its catalogue describes hourly and daily observations from Indian monitoring stations, attributes compilation to Rohan Rao using data provided by India's Central Pollution Control Board, links to the Kaggle collection *Air Quality Data in India (2015–2020)*, and lists CC0 1.0 [2]. These external catalogue details establish a plausible source candidate, not the acquisition history of the local files.

The authors have retained SHA-256 fingerprints of the two local CSV files in the project audit records. A file fingerprint can identify the local file contents for reproducibility, but it does not by itself establish the file's origin. The exact source version and whether the catalogue-listed licence applies to these particular local copies have not been independently verified. This provenance and licence confirmation remains an author action required before submission.

Processed features, chronological split assignments, frozen models, evaluation results, and research tables are retained in the project workspace. No public source-code repository is specified here. Any later public release must exclude private credentials and respect the terms applicable to the data actually used."""

# Replace exactly the existing data availability section, up to References.
pattern = r"## Data and code availability\s.*?(?=\n## References)"
updated, n = re.subn(
    pattern,
    replacement_data_section + "\n",
    text,
    count=1,
    flags=re.DOTALL
)
if n != 1:
    raise ValueError(
        "Could not safely replace the Data and code availability section. "
        "No manuscript was written."
    )
text = updated

# 2. Replace reference [2] with the catalogue entry, not an unverified
# direct-source claim about the local files.
reference_2 = (
    "[2] Transformative Urban Mobility Initiative (TUMI). "
    "*Urban air pollution measurements, India, 2015–2020* [dataset catalogue]. "
    "Catalogue updated August 20, 2026. The catalogue describes hourly and daily "
    "Indian air-quality observations, attributes compilation to Rohan Rao using "
    "data from India's Central Pollution Control Board, links to the Kaggle "
    "collection *Air Quality Data in India (2015–2020)*, and lists a CC0 1.0 "
    "licence. This catalogue citation does not establish that the local files "
    "used in this study are identical to or were obtained from the linked "
    "distribution. Available at: "
    "https://portal.transport-data.org/@tumi/urban-air-pollution-india-2015-2020."
)

text, ref_n = re.subn(
    r"(?m)^\[2\].*$",
    lambda _: reference_2,
    text,
    count=1
)
if ref_n != 1:
    raise ValueError(
        "Could not safely replace reference [2]. No manuscript was written."
    )

# 3. Audit the new citation and provenance text.
if "## References" not in text:
    raise ValueError("References section is missing.")

body, refs_section = text.split("## References", 1)

citation_ids = set()
for group in re.findall(r"\[((?:\d+\s*,\s*)*\d+)\]", body):
    citation_ids.update(int(n) for n in re.findall(r"\d+", group))

reference_ids = [
    int(n) for n in re.findall(
        r"^\[(\d+)\]\s", refs_section, flags=re.MULTILINE
    )
]
missing_refs = sorted(citation_ids - set(reference_ids))
duplicate_refs = sorted({
    n for n in reference_ids if reference_ids.count(n) > 1
})
uncited_refs = sorted(set(reference_ids) - citation_ids)

required_phrases = [
    "does not document how the files were acquired",
    "have not been independently verified",
    "required before submission",
]
phrase_checks = {
    phrase: phrase.lower() in text.lower()
    for phrase in required_phrases
}

citation_pass = (
    not missing_refs
    and not duplicate_refs
    and not uncited_refs
)
provenance_disclosure_pass = all(phrase_checks.values())

# 4. Write a new version without overwriting earlier manuscript copies.
MANUSCRIPT_OUT = (
    OUT / "phase17_07P_provenance_safe_manuscript.md"
)
MANUSCRIPT_OUT.write_text(text, encoding="utf-8")

# 5. Copy the existing fingerprints into the new audit folder, if present.
if FINGERPRINTS.exists():
    fingerprints = pd.read_csv(FINGERPRINTS)
    fingerprints.to_csv(
        OUT / "phase17_07P_local_file_fingerprints.csv",
        index=False
    )
else:
    fingerprints = pd.DataFrame()
    print("WARNING: Existing fingerprint file was not found.")

report = {
    "phase": "17.07P",
    "manuscript": str(MANUSCRIPT_OUT),
    "source_manuscript": str(SOURCE),
    "inline_citation_ids": sorted(citation_ids),
    "bibliography_ids": reference_ids,
    "missing_reference_entries": missing_refs,
    "duplicate_reference_ids": duplicate_refs,
    "references_not_cited": uncited_refs,
    "citation_consistency_pass": citation_pass,
    "provenance_disclosure_checks": phrase_checks,
    "provenance_disclosure_pass": provenance_disclosure_pass,
    "local_file_fingerprint_rows_copied": int(len(fingerprints)),
    "exact_local_file_provenance_confirmed": False,
    "local_file_license_applicability_confirmed": False,
    "submission_ready": False,
    "remaining_blocker": (
        "Author must confirm the exact source/version and applicable licence "
        "for the local CSV files before submission."
    ),
    "models_retrained": False,
    "models_retuned": False,
    "production_enabled": False,
}
(OUT / "phase17_07P_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n[1] MANUSCRIPT")
print(MANUSCRIPT_OUT)

print("\n[2] CITATION AUDIT")
print("Missing references:", missing_refs)
print("Duplicate reference IDs:", duplicate_refs)
print("Uncited references:", uncited_refs)
print("Citation consistency pass:", citation_pass)

print("\n[3] PROVENANCE DISCLOSURE")
print(json.dumps(phrase_checks, indent=2))
print("Disclosure pass:", provenance_disclosure_pass)

print("\n[4] SUBMISSION STATUS")
print("Exact local provenance confirmed: False")
print("Local licence applicability confirmed: False")
print("Submission ready: False")
print("Models retrained: False")
print("Production enabled: False")
print("\nAudit report:", OUT / "phase17_07P_report.json")
print("=" * 78)

In [ ]:
# PHASE 17.07Q — RESEARCH REVIEW BUNDLE
# Packages reviewed artifacts only; source datasets and models are excluded.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import zipfile

print("=" * 78)
print("PHASE 17.07Q — RESEARCH REVIEW BUNDLE")
print("=" * 78)

# 1. Resolve the current project folder safely.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in roots if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project folder not found.")

PAPER = ROOT / "outputs/research_paper"

# Use a unique directory so an earlier bundle is never overwritten.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = PAPER / f"phase17_07Q_review_bundle_{stamp}"
STAGING = OUT / "bundle_contents"
STAGING.mkdir(parents=True, exist_ok=False)

# 2. Explicit files to include from each research phase.
include_files = [
    # Final provenance-safe manuscript and report.
    (
        "phase17_07P_provenance_safe_manuscript/"
        "phase17_07P_provenance_safe_manuscript.md",
        "manuscript/phase17_07P_provenance_safe_manuscript.md"
    ),
    (
        "phase17_07P_provenance_safe_manuscript/phase17_07P_report.json",
        "audits/phase17_07P_report.json"
    ),

    # References and pre-submission checklist.
    (
        "phase17_07K_verified_literature/phase17_07K_reference_register.csv",
        "references/phase17_07K_reference_register.csv"
    ),
    (
        "phase17_07K_verified_literature/phase17_07K_pre_submission_checklist.md",
        "references/phase17_07K_pre_submission_checklist.md"
    ),

    # Final numeric and citation audits.
    (
        "phase17_07L_final_quality_audit/phase17_07L_numeric_consistency.csv",
        "audits/phase17_07L_numeric_consistency.csv"
    ),
    (
        "phase17_07L_R1_final_quality_audit/phase17_07L_R1_quality_report.json",
        "audits/phase17_07L_R1_quality_report.json"
    ),
    (
        "phase17_07L_R1_final_quality_audit/phase17_07L_R1_remaining_blockers.txt",
        "audits/phase17_07L_R1_remaining_blockers.txt"
    ),

    # Local file fingerprints and external catalogue metadata.
    (
        "phase17_07M_provenance_audit/phase17_07M_local_file_fingerprints.csv",
        "provenance/phase17_07M_local_file_fingerprints.csv"
    ),
    (
        "phase17_07M_provenance_audit/phase17_07M_external_catalogue_metadata.json",
        "provenance/phase17_07M_external_catalogue_metadata.json"
    ),
    (
        "phase17_07M_provenance_audit/phase17_07M_assessment.json",
        "provenance/phase17_07M_assessment.json"
    ),

    # Original notebook lineage evidence.
    (
        "phase17_07N_notebook_lineage/phase17_07N_lineage_evidence.json",
        "provenance/phase17_07N_lineage_evidence.json"
    ),
    (
        "phase17_07N_notebook_lineage/phase17_07N_assessment.json",
        "provenance/phase17_07N_assessment.json"
    ),
    (
        "phase17_07O_original_loading_review/phase17_07O_original_loading_cells.json",
        "provenance/phase17_07O_original_loading_cells.json"
    ),
    (
        "phase17_07O_original_loading_review/phase17_07O_assessment.json",
        "provenance/phase17_07O_assessment.json"
    ),

    # Consolidated evidence and the prior draft results.
    (
        "phase17_07I_evidence_package/phase17_07I_draft_results.md",
        "results/phase17_07I_draft_results.md"
    ),
    (
        "phase17_07I_evidence_package/phase17_07I_evidence_package_manifest.json",
        "results/phase17_07I_evidence_package_manifest.json"
    ),

    # Current manuscript's generated tables and source manifest.
    (
        "phase17_07J_manuscript_draft/phase17_07J_manuscript_working_draft.md",
        "manuscript/phase17_07J_manuscript_working_draft.md"
    ),
    (
        "phase17_07J_manuscript_draft/phase17_07J_manifest.json",
        "manuscript/phase17_07J_manifest.json"
    ),
]

copied = []
missing = []

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

for source_relative, archive_relative in include_files:
    src = PAPER / source_relative
    if not src.exists():
        missing.append(str(src))
        continue

    dst = STAGING / archive_relative
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

    copied.append({
        "source": str(src),
        "bundle_path": archive_relative,
        "size_bytes": int(dst.stat().st_size),
        "sha256": sha256_file(dst),
    })

# Include all compact CSV tables from Phase 17.07I and 17.07J.
table_sources = [
    (
        PAPER / "phase17_07I_evidence_package/tables",
        "results/phase17_07I_tables"
    ),
    (
        PAPER / "phase17_07J_manuscript_draft/tables",
        "results/phase17_07J_tables"
    ),
]

for source_dir, archive_prefix in table_sources:
    if not source_dir.exists():
        missing.append(str(source_dir))
        continue

    for src in sorted(source_dir.glob("*.csv")):
        archive_relative = f"{archive_prefix}/{src.name}"
        dst = STAGING / archive_relative
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

        copied.append({
            "source": str(src),
            "bundle_path": archive_relative,
            "size_bytes": int(dst.stat().st_size),
            "sha256": sha256_file(dst),
        })

# 3. Add a clear README describing the scope and unresolved limitation.
readme = """# Delhi AQI Forecasting — Research Review Bundle

This bundle contains the reviewed manuscript, reference register, compact result
tables, quality audits, and available dataset-provenance evidence.

## Important status

- Research results and manuscript citation/numeric checks passed the saved audits.
- The local source CSV files have SHA-256 fingerprints.
- The original data-understanding notebook loads files from `data/raw`, but it
  does not document the files' acquisition.
- An external catalogue lists a matching dataset collection and CC0 1.0 licence.
  This does not, by itself, prove that this specific local copy came from that
  collection or that the listed licence applies to it.
- Exact local dataset provenance and licence applicability remain unverified.
- Therefore, this is a research-review bundle, not a declaration that the paper
  is submission-ready.
- Results describe retrospective historical performance. Production remains
  disabled because a validated live AQI source has not been established.

## Included and excluded files

Included: manuscript Markdown, compact result tables, references, audit reports,
source-file fingerprints, and notebook lineage summaries.

Excluded intentionally: original raw CSV datasets, fitted models, API credentials,
deployment packages, and unrelated project files.

## Main performance figures

- Full ensemble holdout: 68,352 rows; MAE 3.216121; RMSE 7.658390; R² 0.996634.
- Matched comparison: 68,285 rows.
- Matched-support persistence: MAE 3.855634; RMSE 8.840011.
- Matched-support ensemble: MAE 3.178058; RMSE 7.495249.
- Day-cluster bootstrap: 5,000 replicates across 296 calendar-day clusters.
- Abrupt-transition recall for the ensemble: approximately 9% using a ±20 AQI
  predicted-change threshold.

Consult the provenance audit records before making any final public release or
licence statement.
"""
(STAGING / "README.md").write_text(readme, encoding="utf-8")

# 4. Add a machine-readable package manifest and SHA256 list.
artifact_paths = sorted(
    p for p in STAGING.rglob("*")
    if p.is_file()
    and p.name not in {
        "bundle_manifest.json",
        "SHA256SUMS.txt",
    }
)

manifest = {
    "phase": "17.07Q",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "bundle_type": "research_review_bundle",
    "files": [
        {
            "bundle_path": str(p.relative_to(STAGING)),
            "size_bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        }
        for p in artifact_paths
    ],
    "missing_expected_sources": missing,
    "dataset_provenance_confirmed": False,
    "local_dataset_license_verified": False,
    "submission_ready": False,
    "models_retrained": False,
    "models_retuned": False,
    "production_enabled": False,
}
(STAGING / "bundle_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

sha_lines = [
    f"{sha256_file(p)}  {p.relative_to(STAGING)}"
    for p in sorted(STAGING.rglob("*"))
    if p.is_file() and p.name != "SHA256SUMS.txt"
]
(STAGING / "SHA256SUMS.txt").write_text(
    "\n".join(sha_lines) + "\n",
    encoding="utf-8"
)

# 5. Create ZIP and verify it can be reopened.
ZIP_PATH = OUT / "Delhi_AQI_Research_Review_Bundle.zip"
with zipfile.ZipFile(
    ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for p in sorted(STAGING.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(STAGING)))

with zipfile.ZipFile(ZIP_PATH, mode="r") as zf:
    bad_member = zf.testzip()
    zip_members = zf.namelist()

if bad_member is not None:
    raise RuntimeError(f"ZIP integrity check failed at member: {bad_member}")

# Save external final summary after creating the package.
summary = {
    "phase": "17.07Q",
    "zip_path": str(ZIP_PATH),
    "zip_size_bytes": int(ZIP_PATH.stat().st_size),
    "zip_sha256": sha256_file(ZIP_PATH),
    "packaged_file_count": int(len(zip_members)),
    "source_files_missing": missing,
    "zip_integrity_check": "PASS",
    "dataset_provenance_confirmed": False,
    "local_dataset_license_verified": False,
    "submission_ready": False,
    "production_enabled": False,
}
(OUT / "phase17_07Q_package_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n[1] PACKAGE CREATED")
print(ZIP_PATH)
print("ZIP size (bytes):", summary["zip_size_bytes"])
print("ZIP SHA-256:", summary["zip_sha256"])
print("Files inside ZIP:", summary["packaged_file_count"])
print("ZIP integrity check:", summary["zip_integrity_check"])

print("\n[2] MISSING EXPECTED SOURCES")
if missing:
    for m in missing:
        print("-", m)
else:
    print("None")

print("\n[3] FINAL STATUS")
print("Dataset provenance confirmed: False")
print("Local dataset licence verified: False")
print("Submission ready: False")
print("Production enabled: False")
print("Summary JSON:", OUT / "phase17_07Q_package_summary.json")
print("=" * 78)

In [ ]:
# PHASE 17.07R — COMPARE LOCAL FILES WITH CURRENT KAGGLE DISTRIBUTION
# A failed download or a hash mismatch does not, by itself, disprove historical origin.

from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import tempfile
import pandas as pd
from datetime import datetime, timezone

print("=" * 78)
print("PHASE 17.07R — KAGGLE FILE IDENTITY CHECK")
print("=" * 78)

# 1. Resolve project root without remounting Drive.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next((p for p in roots if p.exists() and (p / "outputs").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction root not found.")

RAW = ROOT / "data/raw"
OUT = ROOT / "outputs/research_paper/phase17_07R_kaggle_file_identity"
OUT.mkdir(parents=True, exist_ok=True)

SLUG = "rohanrao/air-quality-data-in-india"
FILES = ["station_hour.csv", "stations.csv"]

for name in FILES:
    if not (RAW / name).exists():
        raise FileNotFoundError(f"Required local file missing: {RAW / name}")

# 2. Hash and summarize each local file.
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def csv_signature(path):
    """Stream the CSV to count rows and summarize key columns."""
    header = list(pd.read_csv(path, nrows=0).columns)

    if "station_hour.csv" in Path(path).name.lower():
        usecols = ["StationId", "Datetime"]
        if not set(usecols).issubset(header):
            return {"header": header, "error": "Expected station-hour keys not found."}

        row_count = 0
        invalid_datetime_count = 0
        min_dt = None
        max_dt = None
        stations = set()

        for chunk in pd.read_csv(
            path,
            usecols=usecols,
            chunksize=200_000,
            low_memory=False,
            dtype={"StationId": "string"},
        ):
            row_count += len(chunk)
            stations.update(chunk["StationId"].dropna().astype(str).unique())

            dates = pd.to_datetime(chunk["Datetime"], errors="coerce")
            invalid_datetime_count += int(dates.isna().sum())
            if dates.notna().any():
                lo, hi = dates.min(), dates.max()
                min_dt = lo if min_dt is None else min(min_dt, lo)
                max_dt = hi if max_dt is None else max(max_dt, hi)

        return {
            "header": header,
            "row_count": int(row_count),
            "unique_station_ids": int(len(stations)),
            "invalid_datetime_count": int(invalid_datetime_count),
            "min_datetime": str(min_dt) if min_dt is not None else None,
            "max_datetime": str(max_dt) if max_dt is not None else None,
        }

    else:
        return {"header": header}

local_info = {}
for filename in FILES:
    path = RAW / filename
    local_info[filename] = {
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
        "signature": csv_signature(path),
    }

print("\n[1] LOCAL FILES")
for filename, info in local_info.items():
    print(f"\n{filename}")
    print("Size:", info["size_bytes"], "bytes")
    print("SHA-256:", info["sha256"])
    print("Schema:", info["signature"].get("header"))
    if "row_count" in info["signature"]:
        print("Rows:", info["signature"]["row_count"])
        print("Stations:", info["signature"]["unique_station_ids"])
        print(
            "Date range:",
            info["signature"]["min_datetime"],
            "to",
            info["signature"]["max_datetime"],
        )

# 3. Install the Kaggle CLI only if it is not already available.
# Installation output is suppressed to avoid accidental credential disclosure.
kaggle_exe = shutil.which("kaggle")
install_status = "not_needed"

if kaggle_exe is None and importlib.util.find_spec("kaggle") is None:
    install_status = "attempted"
    try:
        install_proc = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "kaggle"],
            capture_output=True,
            text=True,
            timeout=180,
        )
        kaggle_exe = shutil.which("kaggle")
        if install_proc.returncode != 0 or kaggle_exe is None:
            install_status = "failed_or_cli_not_found"
        else:
            install_status = "installed"
    except Exception as exc:
        install_status = f"failed_{type(exc).__name__}"

if kaggle_exe:
    CLI = [kaggle_exe]
elif importlib.util.find_spec("kaggle") is not None:
    CLI = [sys.executable, "-m", "kaggle"]
else:
    CLI = None

print("\n[2] KAGGLE CLI")
print("CLI available:", CLI is not None)
print("Install status:", install_status)
print("Credentials will not be displayed.")

# 4. Ask the Kaggle API for dataset metadata when possible.
dataset_metadata = {}
metadata_status = "not_attempted"

if CLI is not None:
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi

        api = KaggleApi()
        api.authenticate()
        response = api.dataset_view(SLUG)

        def get_metadata_field(obj, names):
            if isinstance(obj, dict):
                for name in names:
                    if name in obj:
                        return obj[name]
                return None
            for name in names:
                if hasattr(obj, name):
                    try:
                        return getattr(obj, name)
                    except Exception:
                        pass
            return None

        dataset_metadata = {
            "title": get_metadata_field(response, ["title"]),
            "owner_name": get_metadata_field(response, ["ownerName", "owner_name"]),
            "current_version_number": get_metadata_field(
                response, ["currentVersionNumber", "current_version_number"]
            ),
            "last_updated": get_metadata_field(
                response, ["lastUpdated", "last_updated"]
            ),
            "license_name": get_metadata_field(
                response, ["licenseName", "license_name", "license"]
            ),
            "total_bytes": get_metadata_field(
                response, ["totalBytes", "total_bytes"]
            ),
        }
        metadata_status = "success"
    except Exception as exc:
        # Log only the exception class, never raw messages that could expose paths or tokens.
        metadata_status = f"unavailable_{type(exc).__name__}"

print("\n[3] CURRENT KAGGLE METADATA")
print(json.dumps(dataset_metadata, indent=2, default=str))
print("Metadata status:", metadata_status)

# 5. Download each file separately into a newly created temporary folder.
# The user's original data/raw files are never changed.
temp_root = Path(tempfile.mkdtemp(prefix="aqi_kaggle_identity_"))
download_results = {}

def classify_error(text):
    t = (text or "").lower()
    if any(x in t for x in [
        "401", "403", "unauthorized", "authentication",
        "api token", "credential", "kaggle.json", "forbidden"
    ]):
        return "authentication_or_authorization_failure"
    if any(x in t for x in [
        "connectionerror", "connection error", "timeout",
        "could not resolve", "network is unreachable",
        "failed to establish", "temporary failure"
    ]):
        return "network_failure"
    if "not found" in t or "no such file" in t:
        return "file_or_resource_not_found"
    return "other_download_failure"

try:
    if CLI is None:
        for filename in FILES:
            download_results[filename] = {
                "status": "skipped",
                "reason": "Kaggle CLI unavailable and installation did not succeed.",
            }
    else:
        for filename in FILES:
            dest_dir = temp_root / Path(filename).stem
            dest_dir.mkdir(parents=True, exist_ok=True)

            cmd = CLI + [
                "datasets", "download",
                "-d", SLUG,
                "-f", filename,
                "-p", str(dest_dir),
                "--unzip",
                "--force",
            ]

            try:
                proc = subprocess.run(
                    cmd,
                    capture_output=True,
                    text=True,
                    timeout=900,
                )
            except Exception as exc:
                download_results[filename] = {
                    "status": "failed",
                    "reason": type(exc).__name__,
                }
                continue

            if proc.returncode != 0:
                download_results[filename] = {
                    "status": "failed",
                    "reason": classify_error((proc.stderr or "") + "\n" + (proc.stdout or "")),
                    "exit_code": int(proc.returncode),
                }
                continue

            # Find the named CSV, including if the CLI nested it in a folder.
            candidates = [
                p for p in dest_dir.rglob("*")
                if p.is_file() and p.name.lower() == filename.lower()
            ]

            if not candidates:
                download_results[filename] = {
                    "status": "failed",
                    "reason": "Download returned successfully but expected CSV was not found.",
                }
                continue

            downloaded = candidates[0]
            local_hash = local_info[filename]["sha256"]
            downloaded_hash = sha256_file(downloaded)

            # Hash equality is definitive for byte identity.
            exact_hash_match = local_hash == downloaded_hash
            byte_match = (
                local_info[filename]["size_bytes"] == downloaded.stat().st_size
                and exact_hash_match
            )

            try:
                signature = csv_signature(downloaded)
                signature_matches = (
                    signature.get("header")
                    == local_info[filename]["signature"].get("header")
                    and signature.get("row_count")
                    == local_info[filename]["signature"].get("row_count")
                    if "row_count" in local_info[filename]["signature"]
                    else signature.get("header")
                    == local_info[filename]["signature"].get("header")
                )
            except Exception as exc:
                signature = {"error": type(exc).__name__}
                signature_matches = False

            download_results[filename] = {
                "status": "downloaded_and_compared",
                "current_kaggle_file_size_bytes": int(downloaded.stat().st_size),
                "current_kaggle_sha256": downloaded_hash,
                "local_sha256": local_hash,
                "exact_sha256_match": bool(exact_hash_match),
                "exact_byte_identity": bool(byte_match),
                "schema_and_rowcount_match": bool(signature_matches),
                "current_kaggle_signature": signature,
            }

finally:
    # Remove only this cell's temporary downloads, not project data.
    shutil.rmtree(temp_root, ignore_errors=True)

# 6. Separate source content identity from licence verification.
both_downloaded = all(
    download_results.get(f, {}).get("status") == "downloaded_and_compared"
    for f in FILES
)
both_hash_match = both_downloaded and all(
    download_results[f].get("exact_sha256_match") is True
    for f in FILES
)

license_name = str(dataset_metadata.get("license_name") or "")
license_is_cc0 = "cc0" in license_name.lower()

if both_hash_match and license_is_cc0:
    gate = "STRONG CONTENT AND CURRENT LICENCE MATCH — AUTHOR REVIEW REQUIRED"
elif both_hash_match:
    gate = "CURRENT KAGGLE FILE CONTENT MATCHED — LICENCE METADATA STILL NEEDS REVIEW"
else:
    gate = "PENDING — CURRENT FILE MATCH NOT ESTABLISHED"

report = {
    "phase": "17.07R",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_slug": SLUG,
    "kaggle_download_used_current_distribution": True,
    "local_file_fingerprints": local_info,
    "kaggle_metadata_status": metadata_status,
    "current_kaggle_dataset_metadata": dataset_metadata,
    "download_and_comparison_results": download_results,
    "both_local_files_exact_hash_match_current_kaggle": bool(both_hash_match),
    "current_kaggle_license_metadata_contains_cc0": bool(license_is_cc0),
    "local_file_acquisition_event_documented": False,
    "current_content_identity_supported": bool(both_hash_match),
    "local_license_applicability_supported_by_current_metadata": bool(
        both_hash_match and license_is_cc0
    ),
    "submission_gate": gate,
    "models_retrained": False,
    "models_retuned": False,
    "source_files_modified": False,
    "production_enabled": False,
}

report_path = OUT / "phase17_07R_kaggle_file_identity_report.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

pd.DataFrame([
    {
        "filename": name,
        "local_size_bytes": local_info[name]["size_bytes"],
        "local_sha256": local_info[name]["sha256"],
        "comparison_status": download_results.get(name, {}).get("status"),
        "current_kaggle_size_bytes": download_results.get(name, {}).get(
            "current_kaggle_file_size_bytes"
        ),
        "current_kaggle_sha256": download_results.get(name, {}).get(
            "current_kaggle_sha256"
        ),
        "exact_sha256_match": download_results.get(name, {}).get(
            "exact_sha256_match"
        ),
        "schema_and_rowcount_match": download_results.get(name, {}).get(
            "schema_and_rowcount_match"
        ),
        "reason_if_not_downloaded": download_results.get(name, {}).get("reason"),
    }
    for name in FILES
]).to_csv(OUT / "phase17_07R_file_comparison.csv", index=False)

print("\n[4] FILE COMPARISON")
for filename in FILES:
    result = download_results.get(filename, {})
    print(f"\n{filename}")
    print("Status:", result.get("status"))
    print("Exact SHA-256 match:", result.get("exact_sha256_match"))
    print("Schema/row-count match:", result.get("schema_and_rowcount_match"))
    if result.get("reason"):
        print("Reason:", result["reason"])

print("\n[5] LICENSE AND SUBMISSION GATE")
print("Kaggle metadata license:", license_name or "not retrieved")
print("Contains CC0 label:", license_is_cc0)
print("Both local files match current Kaggle files:", both_hash_match)
print("Gate:", gate)

print("\n[6] SAVED REPORTS")
print(report_path)
print(OUT / "phase17_07R_file_comparison.csv")
print("Raw downloaded comparison copies retained: False")
print("Local files overwritten: False")
print("Models retrained: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07R-S — SAVE CHECKPOINT AND RESUME INSTRUCTIONS
# Saves the current state so the project can resume without rerunning prior phases.

from pathlib import Path
from datetime import datetime, timezone
import os
import json
import hashlib
import zipfile
import pandas as pd

print("=" * 78)
print("PHASE 17.07R-S — RESEARCH CHECKPOINT")
print("=" * 78)

# 1. Resolve the existing Drive project without remounting it.
ROOT_CANDIDATES = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in ROOT_CANDIDATES if p.exists() and (p / "outputs").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Check the existing Google Drive mount."
    )

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
CHECKPOINT = (
    ROOT / "outputs/checkpoints/research_paper"
    / f"phase17_07R_S_{timestamp}"
)
DATAFRAME_DIR = CHECKPOINT / "dataframes"
CHECKPOINT.mkdir(parents=True, exist_ok=False)
DATAFRAME_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

# 2. Snapshot useful DataFrames that may still be in this notebook's memory.
# The canonical source artifacts remain in their original locations.
dataframe_names = [
    "common",
    "transition",
    "validation_table",
    "common_table",
    "full_table",
    "bootstrap_table",
    "band_table",
    "feature_table",
    "station_ensemble",
    "season_ensemble",
    "importance",
    "importance_groups",
    "ensemble_transition",
    "perm_top",
]

dataframe_manifest = []
namespace = globals()

for name in dataframe_names:
    obj = namespace.get(name)
    if not isinstance(obj, pd.DataFrame):
        continue

    safe_name = "".join(
        ch if ch.isalnum() or ch in "_-" else "_"
        for ch in name
    )
    destination = DATAFRAME_DIR / f"{safe_name}.parquet"

    try:
        obj.to_parquet(destination, index=False)
        dataframe_manifest.append({
            "variable_name": name,
            "path": str(destination),
            "rows": int(len(obj)),
            "columns": [str(c) for c in obj.columns],
            "size_bytes": int(destination.stat().st_size),
            "sha256": sha256_file(destination),
            "status": "saved",
        })
        print(f"[PARQUET] {name}: {len(obj):,} rows")
    except Exception as exc:
        dataframe_manifest.append({
            "variable_name": name,
            "status": "save_failed",
            "error_type": type(exc).__name__,
            "note": "Canonical source artifacts remain available on Drive.",
        })
        print(f"[PARQUET] {name}: could not save ({type(exc).__name__})")

# 3. Record the frozen models and feature contract by reference.
model_specs = [
    {
        "name": "Tuned Extra Trees",
        "path": (
            ROOT / "deployment/phase16_33_frozen_model_package/models/"
            "phase15_10_final_tuned_extra_trees_trainval.joblib"
        ),
        "expected_sha256": (
            "f0b3d43a56a7d5e4f08554f37ff234e1b5e012b5d5d69f18c750844b36cbe72d"
        ),
        "weight": 0.594466,
    },
    {
        "name": "Tuned Random Forest",
        "path": (
            ROOT / "deployment/phase16_33_frozen_model_package/models/"
            "phase15_10_final_tuned_random_forest_trainval.joblib"
        ),
        "expected_sha256": (
            "538b5ff34152d19221518e831528093b7bcc57e2b05f82669718f540b514abd8"
        ),
        "weight": 0.405534,
    },
]

model_manifest = []
for item in model_specs:
    path = item["path"]
    model_manifest.append({
        "name": item["name"],
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": int(path.stat().st_size) if path.exists() else None,
        "expected_sha256": item["expected_sha256"],
        "weight": item["weight"],
        "copied": False,
    })

contract_path = (
    ROOT / "deployment/phase16_33_frozen_model_package/config/feature_contract.json"
)
contract_info = {
    "path": str(contract_path),
    "exists": contract_path.exists(),
    "sha256": sha256_file(contract_path) if contract_path.exists() else None,
    "feature_count": 73,
}

# 4. Track the completed work and what remains.
artifact_paths = {
    "provenance_audit": (
        "outputs/research_paper/phase17_07M_provenance_audit/"
        "phase17_07M_assessment.json"
    ),
    "notebook_lineage_review": (
        "outputs/research_paper/phase17_07O_original_loading_review/"
        "phase17_07O_assessment.json"
    ),
    "provenance_safe_manuscript": (
        "outputs/research_paper/phase17_07P_provenance_safe_manuscript/"
        "phase17_07P_provenance_safe_manuscript.md"
    ),
    "research_review_bundle": (
        "outputs/research_paper/"
        "phase17_07Q_review_bundle_20261023T051212Z/"
        "Delhi_AQI_Research_Review_Bundle.zip"
    ),
    "last_kaggle_identity_attempt": (
        "outputs/research_paper/phase17_07R_kaggle_file_identity/"
        "phase17_07R_kaggle_file_identity_report.json"
    ),
}

artifact_manifest = {}
for name, relative_path in artifact_paths.items():
    path = ROOT / relative_path
    artifact_manifest[name] = {
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": int(path.stat().st_size) if path.exists() else None,
    }

# 5. Provide reproducible next steps in a text file.
resume_instructions = """# Delhi AQI Research Project — Resume Checkpoint

## Current project status

The research paper evidence is assembled. The manuscript, result tables,
numerical audit, citations, and research review ZIP are saved on Google Drive.

The frozen forecasting model and the 73-feature contract must remain unchanged.
Production must remain disabled until a validated live AQI source is available.

## Resume point

Resume at **Phase 17.07R2 — Authenticated Kaggle File Identity Check**.
Do not rerun Phases 0–17.07Q.

## Before running Phase 17.07R2

1. Open https://www.kaggle.com/settings/api.
2. Create or configure a Kaggle API token.
3. In Google Colab, open the Secrets panel.
4. Add a secret named `KAGGLE_API_TOKEN` and grant notebook access.
5. Rerun the Phase 17.07R2 cell from the conversation.

Do not paste the API token into notebook code or chat.

## Expected Phase 17.07R2 outputs

Compare the SHA-256 of the downloaded current Kaggle `station_hour.csv`
with the saved local file:
`data/raw/station_hour.csv`

The local SHA-256 recorded in the audit is:
`26a423c66fe9763064d580ffb007150375cfa2056e62b29d52b1b7857af54fde`

`stations.csv` already matched the current Kaggle distribution in Phase 17.07R.

A mismatch for `station_hour.csv` does not prove that the local file originated
elsewhere; it may be an older release. Retain the provenance limitation unless
the original acquisition record or exact source version can be established.

## Frozen model details

Target: AQI_t_plus_1
Frozen feature count: 73
Frozen station scope: DL031, DL002, DL021, DL028, DL019,
DL010, DL034, DL007, DL013, DL004

Weighted ensemble:
- Tuned Extra Trees: 0.594466
- Tuned Random Forest: 0.405534

Do not retrain, retune, alter the feature contract, or use holdout results
to select a new model during the provenance audit.

## Final holdout figures

Full holdout (68,352 rows):
- Ensemble MAE: 3.216121
- Ensemble RMSE: 7.658390
- Ensemble R²: 0.996634

Matched baseline comparison (68,285 rows):
- Persistence MAE: 3.855634
- Persistence RMSE: 8.840011
- Ensemble MAE: 3.178058
- Ensemble RMSE: 7.495249

Day-cluster bootstrap: 5,000 replicates across 296 calendar-day clusters.

## Submission limitation

The provenance-safe manuscript is available at:
`outputs/research_paper/phase17_07P_provenance_safe_manuscript/phase17_07P_provenance_safe_manuscript.md`

The exact provenance and licence applicability for the local raw CSV files
remain unverified until the source/version check is complete. Do not mark the
paper submission-ready solely because the hashes, schema, citations, or
numeric audits pass.

## Checkpoint files

See `checkpoint_state.json` for artifact paths, model references, contract
fingerprint, and saved DataFrame snapshots.
"""

(CHECKPOINT / "RESUME_INSTRUCTIONS.md").write_text(
    resume_instructions, encoding="utf-8"
)

# 6. Save a compact machine-readable checkpoint state.
checkpoint_state = {
    "checkpoint_phase": "17.07R-S",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(ROOT),
    "last_completed_phase": "17.07Q",
    "last_attempted_phase": "17.07R",
    "next_phase": "17.07R2",
    "next_action": (
        "Configure Colab Secret KAGGLE_API_TOKEN, then run Phase 17.07R2."
    ),
    "kaggle_secret_available_in_environment": bool(
        os.environ.get("KAGGLE_API_TOKEN")
    ),
    "frozen_target": "AQI_t_plus_1",
    "feature_count": 73,
    "station_ids": [
        "DL031", "DL002", "DL021", "DL028", "DL019",
        "DL010", "DL034", "DL007", "DL013", "DL004",
    ],
    "ensemble_weights": {
        "Tuned Extra Trees": 0.594466,
        "Tuned Random Forest": 0.405534,
    },
    "models": model_manifest,
    "feature_contract": contract_info,
    "key_artifacts": artifact_manifest,
    "saved_dataframe_snapshots": dataframe_manifest,
    "latest_provenance_status": {
        "stations_csv_matches_current_kaggle": True,
        "station_hour_csv_matches_current_kaggle": None,
        "exact_local_source_version_confirmed": False,
        "local_file_license_applicability_confirmed": False,
        "submission_ready": False,
    },
    "holdout_results": {
        "rows_full": 68352,
        "ensemble_mae_full": 3.216121,
        "ensemble_rmse_full": 7.658390,
        "ensemble_r2_full": 0.996634,
        "rows_matched": 68285,
        "persistence_mae_matched": 3.855634,
        "persistence_rmse_matched": 8.840011,
        "ensemble_mae_matched": 3.178058,
        "ensemble_rmse_matched": 7.495249,
    },
    "models_retrained": False,
    "models_retuned": False,
    "production_enabled": False,
}
(CHECKPOINT / "checkpoint_state.json").write_text(
    json.dumps(checkpoint_state, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

# 7. Add a minimal restore helper for a future Colab session.
restore_code = r'''
# Restore checkpoint state after mounting Google Drive.
# This restores metadata and saved DataFrame snapshots, not the entire
# original Python runtime or model-training state.

from pathlib import Path
import json
import pandas as pd

CHECKPOINT_PATH = Path(r"PASTE_CHECKPOINT_FOLDER_PATH_HERE")
state_path = CHECKPOINT_PATH / "checkpoint_state.json"

if not state_path.exists():
    raise FileNotFoundError(f"Checkpoint state not found: {state_path}")

with open(state_path, "r", encoding="utf-8") as f:
    checkpoint_state = json.load(f)

RESTORED_DATAFRAMES = {}
df_dir = CHECKPOINT_PATH / "dataframes"

if df_dir.exists():
    for parquet_path in df_dir.glob("*.parquet"):
        RESTORED_DATAFRAMES[parquet_path.stem] = pd.read_parquet(parquet_path)

print("Checkpoint metadata restored.")
print("Last completed phase:", checkpoint_state["last_completed_phase"])
print("Next phase:", checkpoint_state["next_phase"])
print("Restored DataFrame snapshots:", list(RESTORED_DATAFRAMES))
print("No model retraining was performed.")
'''

restore_code = restore_code.replace(
    "PASTE_CHECKPOINT_FOLDER_PATH_HERE", str(CHECKPOINT)
)
(CHECKPOINT / "restore_checkpoint.py").write_text(
    restore_code.strip() + "\n", encoding="utf-8"
)

# 8. Build a small checkpoint ZIP and verify its integrity.
ZIP_PATH = CHECKPOINT / "research_checkpoint.zip"
with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for path in CHECKPOINT.rglob("*"):
        if path.is_file() and path != ZIP_PATH:
            zf.write(path, arcname=str(path.relative_to(CHECKPOINT)))

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_member = zf.testzip()

if bad_member:
    raise RuntimeError(f"Checkpoint ZIP integrity check failed: {bad_member}")

print("\n[1] CHECKPOINT SAVED")
print("Folder:", CHECKPOINT)
print("State JSON:", CHECKPOINT / "checkpoint_state.json")
print("Resume instructions:", CHECKPOINT / "RESUME_INSTRUCTIONS.md")
print("Restore helper:", CHECKPOINT / "restore_checkpoint.py")
print("ZIP:", ZIP_PATH)
print("ZIP SHA-256:", sha256_file(ZIP_PATH))
print("DataFrame snapshots saved:", sum(
    1 for item in dataframe_manifest if item["status"] == "saved"
))
print("Frozen models copied: False")
print("Original data files modified: False")
print("Models retrained: False")
print("Models retuned: False")
print("Production enabled: False")
print("Next phase: 17.07R2 after Kaggle API secret is configured")
print("=" * 78)

In [ ]:
# PHASE 17.07R2 — FRESH-RUNTIME AUTHENTICATED FILE IDENTITY CHECK
# Does not modify original CSV files, models, or the frozen feature contract.

from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import shutil
import hashlib
import subprocess
import tempfile
import zipfile
import re
import pandas as pd

print("=" * 78)
print("PHASE 17.07R2 — AUTHENTICATED FILE IDENTITY CHECK")
print("=" * 78)

# ---------------------------------------------------------------------------
# 1. Locate or safely mount Google Drive.
# ---------------------------------------------------------------------------
def find_project_root():
    candidates = [
        Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
        Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
    ]
    return next(
        (p for p in candidates if p.exists() and (p / "outputs").exists()),
        None,
    )

ROOT = find_project_root()

if ROOT is None:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "This cell is intended for Google Colab. "
            "Mount Google Drive and rerun."
        ) from exc

    # Mount only into an absent or empty directory.
    mounted = False
    for mount_point in [Path("/content/drive"), Path("/content/gdrive")]:
        is_empty_or_missing = (
            not mount_point.exists()
            or (
                mount_point.is_dir()
                and not any(mount_point.iterdir())
            )
        )
        if is_empty_or_missing:
            print(f"Mounting Google Drive at {mount_point}")
            drive.mount(str(mount_point))
            mounted = True
            break

    ROOT = find_project_root()

    if ROOT is None:
        raise FileNotFoundError(
            "Drive is mounted, but Delhi-AQI-Prediction was not found. "
            "Check the project folder location. No existing folder was deleted."
        )

print("Project root:", ROOT)

RAW = ROOT / "data/raw"
LOCAL_FILES = {
    "station_hour.csv": RAW / "station_hour.csv",
    "stations.csv": RAW / "stations.csv",
}

for filename, path in LOCAL_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Required local source file not found: {path}")

OUT = ROOT / "outputs/research_paper/phase17_07R2_authenticated_identity"
OUT.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# 2. Get the secret without printing or saving its value.
# ---------------------------------------------------------------------------
token = os.environ.get("KAGGLE_API_TOKEN")

if not token:
    try:
        from google.colab import userdata
        token = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        token = None

if not token or not str(token).strip():
    raise RuntimeError(
        "KAGGLE_API_TOKEN was not available. In Colab, open Secrets, add "
        "KAGGLE_API_TOKEN, and enable notebook access. Do not paste the token "
        "into notebook code or chat. Rerun this cell after configuring it."
    )

# Pass the secret only to child processes; never print it.
child_env = os.environ.copy()
child_env["KAGGLE_API_TOKEN"] = str(token)

# ---------------------------------------------------------------------------
# 3. Define hashing and safe CLI helpers.
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def classify_cli_failure(output):
    msg = (output or "").lower()

    if any(x in msg for x in [
        "must authenticate", "unauthorized", "401",
        "forbidden", "403", "invalid token", "authentication",
    ]):
        return "authentication_or_authorization_failure"

    if any(x in msg for x in [
        "connection error", "connectionerror", "timeout",
        "network is unreachable", "could not resolve",
        "failed to establish", "temporary failure",
    ]):
        return "network_failure"

    if "not found" in msg or "no such file" in msg:
        return "resource_or_file_not_found"

    return "other_cli_failure"

def run_kaggle(args, timeout=180):
    return subprocess.run(
        [KAGGLE_CLI] + args,
        env=child_env,
        capture_output=True,
        text=True,
        timeout=timeout,
    )

# ---------------------------------------------------------------------------
# 4. Upgrade Kaggle CLI for current KAGGLE_API_TOKEN authentication support.
# ---------------------------------------------------------------------------
print("\n[1] Installing/upgrading Kaggle CLI")

pip_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"],
    capture_output=True,
    text=True,
    timeout=300,
)

if pip_result.returncode != 0:
    raise RuntimeError(
        "Could not install/upgrade the Kaggle CLI. "
        "Check Colab internet access and rerun. Raw installation output "
        "was intentionally not printed."
    )

KAGGLE_CLI = shutil.which("kaggle")

if not KAGGLE_CLI:
    raise RuntimeError(
        "Kaggle CLI installation completed, but its executable was not found. "
        "Restart the runtime if required, then rerun this cell."
    )

version_result = subprocess.run(
    [KAGGLE_CLI, "--version"],
    capture_output=True,
    text=True,
    env=child_env,
    timeout=60,
)

print("Kaggle CLI:", KAGGLE_CLI)
print(
    "Version command:",
    "PASS" if version_result.returncode == 0 else "FAILED",
)

# ---------------------------------------------------------------------------
# 5. Authenticate by making an actual authenticated dataset file-list request.
# ---------------------------------------------------------------------------
SLUG = "rohanrao/air-quality-data-in-india"

print("\n[2] Authentication and published file listing")

listing_result = run_kaggle(
    ["datasets", "files", SLUG, "--page-size", "100", "--csv"],
    timeout=180,
)

listing_text = (listing_result.stdout or "") + "\n" + (listing_result.stderr or "")
auth_failure = (
    listing_result.returncode != 0
    or "must authenticate" in listing_text.lower()
    or "unauthorized" in listing_text.lower()
    or "invalid token" in listing_text.lower()
)

# Only save public file-list output, not the secret.
if listing_result.stdout:
    (OUT / "phase17_07R2_published_file_listing.csv").write_text(
        listing_result.stdout, encoding="utf-8"
    )

if auth_failure:
    report = {
        "phase": "17.07R2",
        "status": "AUTHENTICATED_LISTING_FAILED",
        "cli_exit_code": int(listing_result.returncode),
        "failure_category": classify_cli_failure(listing_text),
        "token_value_logged": False,
        "local_files_modified": False,
        "models_retrained": False,
        "production_enabled": False,
    }
    (OUT / "phase17_07R2_status.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )

    raise RuntimeError(
        "The Kaggle file-list request failed authentication or access checks. "
        f"Failure category: {report['failure_category']}. "
        "Verify the Colab Secret and its notebook-access permission."
    )

listing_lower = listing_text.lower()
listed_files = [
    filename for filename in LOCAL_FILES
    if filename.lower() in listing_lower
]

print("Authenticated file-list request: PASS")
print("Expected files mentioned:", listed_files)

if len(listed_files) != len(LOCAL_FILES):
    report = {
        "phase": "17.07R2",
        "status": "EXPECTED_FILES_NOT_ALL_LISTED",
        "expected_files": list(LOCAL_FILES),
        "files_mentioned_in_listing": listed_files,
        "local_files_modified": False,
        "models_retrained": False,
        "production_enabled": False,
    }
    (OUT / "phase17_07R2_status.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    raise RuntimeError(
        "The current Kaggle file listing did not clearly include both expected "
        "CSV files. Review the saved listing before attempting identity claims."
    )

# ---------------------------------------------------------------------------
# 6. Retrieve current public dataset metadata, especially licence information.
# ---------------------------------------------------------------------------
print("\n[3] Current dataset metadata")

metadata_dir = Path(tempfile.mkdtemp(prefix="phase17_07R2_metadata_"))
metadata_status = "not_available"
metadata_summary = {}
metadata_failure_category = None

try:
    metadata_result = run_kaggle(
        ["datasets", "metadata", SLUG, "-p", str(metadata_dir)],
        timeout=180,
    )

    if metadata_result.returncode == 0:
        json_candidates = list(metadata_dir.rglob("*.json"))

        # Retain only non-sensitive catalogue metadata fields.
        for meta_path in json_candidates:
            try:
                candidate = json.loads(
                    meta_path.read_text(encoding="utf-8", errors="ignore")
                )
            except Exception:
                continue

            if not isinstance(candidate, dict):
                continue

            if any(k in candidate for k in ["licenses", "id", "title", "license"]):
                licenses = candidate.get("licenses", candidate.get("license", []))
                if isinstance(licenses, dict):
                    licenses = [licenses]
                if isinstance(licenses, str):
                    licenses = [{"name": licenses}]

                license_names = []
                if isinstance(licenses, list):
                    for item in licenses:
                        if isinstance(item, dict):
                            name = item.get("name") or item.get("licenseName")
                            if name:
                                license_names.append(str(name))
                        elif isinstance(item, str):
                            license_names.append(item)

                metadata_summary = {
                    "title": candidate.get("title"),
                    "id": candidate.get("id"),
                    "licenses": license_names,
                    "lastUpdated": candidate.get("lastUpdated"),
                    "totalBytes": candidate.get("totalBytes"),
                    "source_metadata_filename": meta_path.name,
                }
                metadata_status = "success"
                break
    else:
        metadata_failure_category = classify_cli_failure(
            (metadata_result.stdout or "") + "\n" + (metadata_result.stderr or "")
        )

finally:
    shutil.rmtree(metadata_dir, ignore_errors=True)

print("Metadata status:", metadata_status)
print("Licence names reported by downloaded metadata:", metadata_summary.get("licenses", []))

(OUT / "phase17_07R2_current_dataset_metadata_summary.json").write_text(
    json.dumps(
        {
            "dataset_slug": SLUG,
            "metadata_status": metadata_status,
            "metadata_summary": metadata_summary,
            "failure_category": metadata_failure_category,
        },
        indent=2,
    ),
    encoding="utf-8",
)

# ---------------------------------------------------------------------------
# 7. Hash the current local files.
# ---------------------------------------------------------------------------
print("\n[4] Local file fingerprints")

local_info = {}
for filename, path in LOCAL_FILES.items():
    local_info[filename] = {
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }
    print(filename)
    print("  Size:", local_info[filename]["size_bytes"])
    print("  SHA-256:", local_info[filename]["sha256"])

# ---------------------------------------------------------------------------
# 8. Download and compare each public file using temporary storage.
#    Do not use data/raw as the download destination.
# ---------------------------------------------------------------------------
temporary_download_root = Path(tempfile.mkdtemp(prefix="phase17_07R2_downloads_"))
results = {}

def discover_csv_or_zip(download_dir, expected_filename):
    """Return a CSV path or extract an expected CSV member to a temporary file."""
    expected = expected_filename.lower()

    # First look for a direct file.
    direct = [
        p for p in download_dir.rglob("*")
        if p.is_file() and p.name.lower() == expected
    ]
    if direct:
        return direct[0], "direct_csv"

    # Otherwise inspect ZIP contents and extract only the requested member.
    for archive in download_dir.rglob("*.zip"):
        try:
            with zipfile.ZipFile(archive, "r") as zf:
                members = [
                    m for m in zf.namelist()
                    if Path(m).name.lower() == expected
                    and not m.endswith("/")
                ]
                if not members:
                    continue

                # Avoid path traversal: write to our own fixed temporary filename.
                target = download_dir / f"verified_copy_{expected_filename}"
                with zf.open(members[0], "r") as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)

                return target, f"csv_member_of_{archive.name}"
        except (OSError, zipfile.BadZipFile, KeyError):
            continue

    return None, None

def csv_signature(path, filename):
    """
    Compute a lightweight signature when hash equality is false.
    Chunking limits memory consumption for the multi-million-row file.
    """
    header = list(pd.read_csv(path, nrows=0).columns)

    if filename == "station_hour.csv":
        required = ["StationId", "Datetime"]
        if not set(required).issubset(header):
            return {"header": header, "error": "Expected station-hour columns missing."}

        row_count = 0
        station_ids = set()
        invalid_datetimes = 0
        min_datetime = None
        max_datetime = None

        for chunk in pd.read_csv(
            path,
            usecols=required,
            chunksize=200_000,
            low_memory=False,
            dtype={"StationId": "string"},
        ):
            row_count += len(chunk)
            station_ids.update(chunk["StationId"].dropna().astype(str).unique())

            dates = pd.to_datetime(chunk["Datetime"], errors="coerce")
            invalid_datetimes += int(dates.isna().sum())

            if dates.notna().any():
                lo, hi = dates.min(), dates.max()
                min_datetime = lo if min_datetime is None else min(min_datetime, lo)
                max_datetime = hi if max_datetime is None else max(max_datetime, hi)

        return {
            "header": header,
            "row_count": int(row_count),
            "unique_station_ids": int(len(station_ids)),
            "invalid_datetime_count": int(invalid_datetimes),
            "min_datetime": str(min_datetime) if min_datetime is not None else None,
            "max_datetime": str(max_datetime) if max_datetime is not None else None,
        }

    # Small station metadata table.
    row_count = 0
    for chunk in pd.read_csv(path, chunksize=50_000, low_memory=False):
        row_count += len(chunk)

    return {
        "header": header,
        "row_count": int(row_count),
    }

try:
    for filename, local_path in LOCAL_FILES.items():
        print(f"\nDownloading current published file: {filename}")

        download_dir = temporary_download_root / Path(filename).stem
        download_dir.mkdir(parents=True, exist_ok=True)

        download_result = run_kaggle(
            [
                "datasets", "download", SLUG,
                "-f", filename,
                "-p", str(download_dir),
                "-o",
            ],
            timeout=1200,
        )

        combined_cli_text = (
            (download_result.stdout or "")
            + "\n"
            + (download_result.stderr or "")
        )

        if download_result.returncode != 0:
            results[filename] = {
                "status": "download_failed",
                "exit_code": int(download_result.returncode),
                "failure_category": classify_cli_failure(combined_cli_text),
            }
            print("Download failed:", results[filename]["failure_category"])
            continue

        downloaded_path, acquisition_method = discover_csv_or_zip(
            download_dir, filename
        )

        if downloaded_path is None:
            results[filename] = {
                "status": "expected_csv_not_found",
                "exit_code": int(download_result.returncode),
                "files_found": [
                    {
                        "relative_path": str(p.relative_to(download_dir)),
                        "size_bytes": int(p.stat().st_size),
                    }
                    for p in download_dir.rglob("*") if p.is_file()
                ],
            }
            print("No direct CSV or ZIP member with the expected basename was found.")
            continue

        downloaded_size = int(downloaded_path.stat().st_size)
        downloaded_hash = sha256_file(downloaded_path)
        local_hash = local_info[filename]["sha256"]

        exact_hash_match = downloaded_hash == local_hash
        exact_size_match = downloaded_size == local_info[filename]["size_bytes"]

        downloaded_signature = None
        local_signature = None
        schema_and_rowcount_match = None

        # Only do the more expensive row-level signature scan if contents differ.
        if not exact_hash_match:
            print("Hashes differ; comparing schemas and row counts.")
            local_signature = csv_signature(local_path, filename)
            downloaded_signature = csv_signature(downloaded_path, filename)

            schema_and_rowcount_match = (
                local_signature.get("header") == downloaded_signature.get("header")
                and local_signature.get("row_count") == downloaded_signature.get("row_count")
            )

        elif exact_size_match:
            # Same bytes mean content/schema identity is established by the hash.
            schema_and_rowcount_match = True

        results[filename] = {
            "status": "compared",
            "acquisition_method": acquisition_method,
            "local_size_bytes": int(local_path.stat().st_size),
            "current_kaggle_size_bytes": downloaded_size,
            "local_sha256": local_hash,
            "current_kaggle_sha256": downloaded_hash,
            "exact_sha256_match": bool(exact_hash_match),
            "exact_byte_identity": bool(exact_hash_match and exact_size_match),
            "schema_and_rowcount_match": schema_and_rowcount_match,
            "local_signature_if_needed": local_signature,
            "current_kaggle_signature_if_needed": downloaded_signature,
        }

        print("Retrieved via:", acquisition_method)
        print("Exact SHA-256 match:", exact_hash_match)
        print("Exact byte identity:", bool(exact_hash_match and exact_size_match))
        if schema_and_rowcount_match is not None:
            print("Schema/row-count match:", schema_and_rowcount_match)

finally:
    # Delete only temporary download copies, not source files.
    shutil.rmtree(temporary_download_root, ignore_errors=True)

    # Do not keep the secret in additional local variables.
    child_env.pop("KAGGLE_API_TOKEN", None)
    token = None

# ---------------------------------------------------------------------------
# 9. Summarize identity and licence evidence without overclaiming.
# ---------------------------------------------------------------------------
both_downloaded = all(
    results.get(name, {}).get("status") == "compared"
    for name in LOCAL_FILES
)

both_exact = both_downloaded and all(
    results[name].get("exact_byte_identity") is True
    for name in LOCAL_FILES
)

license_names = metadata_summary.get("licenses", [])
license_text = " ".join(str(x) for x in license_names).lower()
cc0_reported = "cc0" in license_text

if both_exact and cc0_reported:
    gate = "CURRENT FILE IDENTITY AND CC0 METADATA SUPPORTED — AUTHOR REVIEW REQUIRED"
elif both_exact:
    gate = "CURRENT FILE IDENTITY MATCHED — REVIEW LICENCE METADATA"
else:
    gate = "PENDING — ONE OR MORE CURRENT FILE HASHES DID NOT MATCH OR WERE NOT VERIFIED"

report = {
    "phase": "17.07R2",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_slug": SLUG,
    "local_files": local_info,
    "authenticated_listing_pass": True,
    "listing_saved": str(OUT / "phase17_07R2_published_file_listing.csv"),
    "metadata_status": metadata_status,
    "metadata_summary": metadata_summary,
    "download_comparisons": results,
    "both_local_files_byte_identical_to_current_kaggle": bool(both_exact),
    "current_kaggle_metadata_reports_cc0": bool(cc0_reported),
    "original_download_event_documented": False,
    "submission_gate": gate,
    "interpretation": (
        "Exact hashes establish byte identity with the current downloaded files, "
        "not the historical acquisition event. Licence metadata is attributed "
        "to the current published distribution and should be reviewed before citation."
    ),
    "local_files_modified": False,
    "models_retrained": False,
    "models_retuned": False,
    "production_enabled": False,
}

report_path = OUT / "phase17_07R2_identity_report.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

comparison_rows = []
for filename in LOCAL_FILES:
    result = results.get(filename, {})
    comparison_rows.append({
        "filename": filename,
        "local_size_bytes": local_info[filename]["size_bytes"],
        "local_sha256": local_info[filename]["sha256"],
        "status": result.get("status"),
        "acquisition_method": result.get("acquisition_method"),
        "current_kaggle_size_bytes": result.get("current_kaggle_size_bytes"),
        "current_kaggle_sha256": result.get("current_kaggle_sha256"),
        "exact_sha256_match": result.get("exact_sha256_match"),
        "exact_byte_identity": result.get("exact_byte_identity"),
        "schema_and_rowcount_match": result.get("schema_and_rowcount_match"),
        "failure_category": result.get("failure_category"),
    })

comparison_path = OUT / "phase17_07R2_file_comparison.csv"
pd.DataFrame(comparison_rows).to_csv(comparison_path, index=False)

print("\n[5] FINAL COMPARISON")
print(pd.DataFrame(comparison_rows).to_string(index=False))

print("\n[6] LICENCE AND SUBMISSION STATUS")
print("Kaggle licence fields:", license_names)
print("CC0 reported by current metadata:", cc0_reported)
print("Both local files byte-identical to current Kaggle:", both_exact)
print("Gate:", gate)

print("\n[7] SAVED AUDIT")
print(report_path)
print(comparison_path)
print("Original files modified: False")
print("Downloaded comparison copies retained: False")
print("Models retrained: False")
print("Models retuned: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07S — UPDATE MANUSCRIPT AFTER EXACT FILE MATCH
# Creates a new version; does not overwrite earlier manuscript files.

from pathlib import Path
import json
import re
import pandas as pd
from datetime import datetime, timezone

print("=" * 78)
print("PHASE 17.07S — UPDATED DATA PROVENANCE")
print("=" * 78)

# 1. Locate project.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next((p for p in roots if p.exists() and (p / "outputs").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project folder not found.")

SOURCE_MD = (
    ROOT / "outputs/research_paper/phase17_07P_provenance_safe_manuscript/"
    "phase17_07P_provenance_safe_manuscript.md"
)
IDENTITY_JSON = (
    ROOT / "outputs/research_paper/phase17_07R2_authenticated_identity/"
    "phase17_07R2_identity_report.json"
)
IDENTITY_CSV = (
    ROOT / "outputs/research_paper/phase17_07R2_authenticated_identity/"
    "phase17_07R2_file_comparison.csv"
)

for p in [SOURCE_MD, IDENTITY_JSON, IDENTITY_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

OUT = ROOT / "outputs/research_paper/phase17_07S_updated_provenance"
OUT.mkdir(parents=True, exist_ok=True)

# 2. Validate the successful identity comparison before updating claims.
identity = json.loads(IDENTITY_JSON.read_text(encoding="utf-8"))
comparison = pd.read_csv(IDENTITY_CSV)

required_files = {"station_hour.csv", "stations.csv"}
if set(comparison["filename"].astype(str)) != required_files:
    raise ValueError("Identity report does not contain both expected source files.")

if not comparison["exact_byte_identity"].fillna(False).astype(bool).all():
    raise ValueError(
        "Not every local CSV has a confirmed exact byte-identity match. "
        "The manuscript will not be updated to claim a complete match."
    )

expected_sha256 = {
    "station_hour.csv":
        "26a423c66fe9763064d580ffb007150375cfa2056e62b29d52b1b7857af54fde",
    "stations.csv":
        "ce2b7081c3e3c89145613eb724d7d2149ef79c6e16f36bb8a835747a673ed410",
}

for filename, expected_hash in expected_sha256.items():
    row = comparison.loc[comparison["filename"].eq(filename)].iloc[0]
    if str(row["local_sha256"]) != expected_hash:
        raise ValueError(
            f"Local hash for {filename} does not match the previously recorded "
            "fingerprint. Stop and review the discrepancy."
        )
    if str(row["current_kaggle_sha256"]) != expected_hash:
        raise ValueError(
            f"Current Kaggle hash for {filename} does not match the verified hash."
        )

print("[1] Exact byte-identity verification: PASS")
print(comparison[[
    "filename", "local_size_bytes", "local_sha256",
    "exact_byte_identity"
]].to_string(index=False))

# 3. Replace the data-availability section with updated, evidence-based wording.
text = SOURCE_MD.read_text(encoding="utf-8")

replacement = """## Data and code availability

The analysis used `station_hour.csv`, containing hourly station observations with AQI and pollutant measurements, and `stations.csv`, containing station metadata. The original project notebook reads both files from the local `data/raw` directory but does not record the historical acquisition event.

On 25 October 2026, the two local CSV files were compared against files downloaded from the current public Kaggle distribution, `rohanrao/air-quality-data-in-india`, through the authenticated Kaggle CLI. Both local files matched the downloaded versions byte for byte by size and SHA-256. The local fingerprints were:

- `station_hour.csv` — SHA-256: `26a423c66fe9763064d580ffb007150375cfa2056e62b29d52b1b7857af54fde`
- `stations.csv` — SHA-256: `ce2b7081c3e3c89145613eb724d7d2149ef79c6e16f36bb8a835747a673ed410`

This verifies identity with the current downloaded distribution; it does not establish when or by whom the local files were originally acquired.

The TUMI Data Portal catalogue entry for *Urban air pollution measurements, India, 2015–2020* links to the same Kaggle collection, describes the dataset as compiled by Rohan Rao from data provided by India's Central Pollution Control Board, and lists CC0 1.0 [2]. The Kaggle metadata request used in this audit did not return a licence field. Accordingly, the CC0 statement in this paper is attributed to the TUMI catalogue record, not to a directly retrieved Kaggle metadata field. Before publication, the authors should confirm that the catalogue-listed licence applies to the redistributed files and review any terms applicable to the underlying source data.

Processed features, chronological split assignments, frozen models, evaluation results, and research tables are retained in the project workspace. No public source-code repository is specified here. Any subsequent public release must exclude credentials and respect the terms applicable to the data used.
"""

text, replaced = re.subn(
    r"## Data and code availability\s.*?(?=\n## References)",
    replacement + "\n",
    text,
    count=1,
    flags=re.DOTALL,
)
if replaced != 1:
    raise ValueError("Could not safely replace the Data and code availability section.")

# 4. Update reference [2] so it clearly identifies the licence as catalogue metadata.
reference_2 = (
    "[2] Transformative Urban Mobility Initiative (TUMI). "
    "*Urban air pollution measurements, India, 2015–2020* [dataset catalogue]. "
    "Updated August 20, 2026. The catalogue describes hourly and daily Indian "
    "air-quality observations, attributes compilation to Rohan Rao using data "
    "from India's Central Pollution Control Board, links to the Kaggle collection "
    "*Air Quality Data in India (2015–2020)*, and lists a CC0 1.0 licence. "
    "The catalogue is cited for its own metadata; the local files' byte identity "
    "with the current Kaggle distribution was independently checked on "
    "October 25, 2026. Available at: "
    "https://portal.transport-data.org/@tumi/urban-air-pollution-india-2015-2020."
)

text, ref_count = re.subn(
    r"(?m)^\[2\].*$",
    lambda _: reference_2,
    text,
    count=1,
)
if ref_count != 1:
    raise ValueError("Could not safely update reference [2].")

# 5. Verify reference numbering and provenance wording.
if "## References" not in text:
    raise ValueError("References heading is missing.")

body, refs_section = text.split("## References", 1)

citation_ids = set()
for group in re.findall(r"\[((?:\d+\s*,\s*)*\d+)\]", body):
    citation_ids.update(int(n) for n in re.findall(r"\d+", group))

reference_ids = [
    int(n) for n in re.findall(r"^\[(\d+)\]\s", refs_section, flags=re.MULTILINE)
]

missing_refs = sorted(citation_ids - set(reference_ids))
duplicate_refs = sorted({n for n in reference_ids if reference_ids.count(n) > 1})
uncited_refs = sorted(set(reference_ids) - citation_ids)

checks = {
    "both_local_files_exactly_match_current_kaggle": True,
    "station_hour_sha256_in_manuscript":
        expected_sha256["station_hour.csv"] in text,
    "stations_sha256_in_manuscript":
        expected_sha256["stations.csv"] in text,
    "historical_acquisition_limitation_disclosed":
        "does not establish when or by whom" in text,
    "licence_attributed_to_tumi_catalogue":
        "lists a CC0 1.0 licence [2]" in text,
    "kaggle_api_license_field_limitation_disclosed":
        "did not return a licence field" in text,
    "missing_citations": missing_refs,
    "duplicate_reference_ids": duplicate_refs,
    "uncited_reference_ids": uncited_refs,
}
checks["citation_consistency_pass"] = (
    not missing_refs and not duplicate_refs and not uncited_refs
)

# 6. Save the manuscript, provenance summary, and author review checklist.
MANUSCRIPT_OUT = OUT / "phase17_07S_provenance_updated_manuscript.md"
MANUSCRIPT_OUT.write_text(text, encoding="utf-8")

review_checklist = """# Phase 17.07S — Final provenance and licence review

- [x] Local `station_hour.csv` exactly matches the current downloaded Kaggle file by SHA-256 and size.
- [x] Local `stations.csv` exactly matches the current downloaded Kaggle file by SHA-256 and size.
- [x] Both local hashes are recorded in the manuscript.
- [x] Historical acquisition limitations are disclosed.
- [x] TUMI's CC0 1.0 catalogue listing is attributed to TUMI, not represented as a directly retrieved Kaggle API field.
- [ ] Author confirms licence coverage for the exact file distribution and any applicable rights for underlying CPCB data.
- [ ] Author reviews the final target-journal citation, data-availability, and licence requirements.

The exact content identity check passed. Do not infer the historical acquisition date or legal licence coverage solely from hash equality.
"""

(OUT / "phase17_07S_author_review_checklist.md").write_text(
    review_checklist, encoding="utf-8"
)

report = {
    "phase": "17.07S",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_manuscript": str(SOURCE_MD),
    "updated_manuscript": str(MANUSCRIPT_OUT),
    "file_identity_report": str(IDENTITY_JSON),
    "both_local_files_exactly_match_current_kaggle": True,
    "historical_acquisition_event_documented": False,
    "tumi_catalogue_lists_cc0_1_0": True,
    "kaggle_api_metadata_licence_field_retrieved": False,
    "licence_applicability_confirmed_by_author": False,
    "citation_checks": checks,
    "citation_consistency_pass": checks["citation_consistency_pass"],
    "submission_gate": "PENDING AUTHOR LICENCE REVIEW",
    "models_retrained": False,
    "models_retuned": False,
    "production_enabled": False,
}

(OUT / "phase17_07S_provenance_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n[2] MANUSCRIPT WRITTEN")
print(MANUSCRIPT_OUT)

print("\n[3] CITATION AUDIT")
print("Missing references:", missing_refs)
print("Duplicate reference IDs:", duplicate_refs)
print("Uncited references:", uncited_refs)
print("Citation consistency pass:", checks["citation_consistency_pass"])

print("\n[4] PROVENANCE STATUS")
print("Both local files match current Kaggle distribution: True")
print("Historical acquisition event documented: False")
print("TUMI catalogue lists CC0 1.0: True")
print("Kaggle API licence field retrieved: False")
print("Submission gate: PENDING AUTHOR LICENCE REVIEW")

print("\n[5] REVIEW CHECKLIST")
print(OUT / "phase17_07S_author_review_checklist.md")
print("\nOriginal manuscripts overwritten: False")
print("Models retrained: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07T — UPDATED RESEARCH REVIEW BUNDLE
# Packages the latest manuscript and audits without copying raw datasets,
# model files, API credentials, or deployment artifacts.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import zipfile
import pandas as pd

print("=" * 78)
print("PHASE 17.07T — UPDATED RESEARCH REVIEW BUNDLE")
print("=" * 78)

# 1. Resolve the existing Google Drive project.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in roots if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project root not found.")

PAPER = ROOT / "outputs/research_paper"

S = PAPER / "phase17_07S_updated_provenance"
R2 = PAPER / "phase17_07R2_authenticated_identity"
M = PAPER / "phase17_07M_provenance_audit"
N = PAPER / "phase17_07N_notebook_lineage"
O = PAPER / "phase17_07O_original_loading_review"
L = PAPER / "phase17_07L_final_quality_audit"
L1 = PAPER / "phase17_07L_R1_final_quality_audit"
G = PAPER / "phase17_07G_day_block_bootstrap"
F = PAPER / "phase17_07F_stratified_holdout"
I = PAPER / "phase17_07I_evidence_package"
J = PAPER / "phase17_07J_manuscript_draft"

required = [
    S / "phase17_07S_provenance_updated_manuscript.md",
    S / "phase17_07S_provenance_report.json",
    S / "phase17_07S_author_review_checklist.md",
    R2 / "phase17_07R2_identity_report.json",
    R2 / "phase17_07R2_file_comparison.csv",
    L / "phase17_07L_numeric_consistency.csv",
    L1 / "phase17_07L_R1_quality_report.json",
]

for path in required:
    if not path.exists():
        raise FileNotFoundError(f"Required latest artifact missing: {path}")

# 2. Verify both file-identity results before making any claim in the bundle.
identity = json.loads(
    (R2 / "phase17_07R2_identity_report.json").read_text(encoding="utf-8")
)
comparison = pd.read_csv(R2 / "phase17_07R2_file_comparison.csv")

expected_files = {"station_hour.csv", "stations.csv"}
if set(comparison["filename"].astype(str)) != expected_files:
    raise ValueError("Expected exactly station_hour.csv and stations.csv in the identity report.")

if not comparison["exact_byte_identity"].fillna(False).astype(bool).all():
    raise ValueError("The latest identity report does not show an exact byte match for both files.")

if identity.get("both_local_files_byte_identical_to_current_kaggle") is not True:
    raise ValueError("Identity report does not confirm byte identity for both local files.")

# 3. Create a fresh bundle directory; never overwrite earlier review bundles.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = PAPER / f"phase17_07T_review_bundle_{stamp}"
STAGING = OUT / "bundle_contents"
STAGING.mkdir(parents=True, exist_ok=False)

copied = []
missing_optional = []

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def add_file(source, archive_path, required_file=False):
    source = Path(source)

    if not source.exists() or not source.is_file():
        if required_file:
            raise FileNotFoundError(f"Required bundle input missing: {source}")
        missing_optional.append(str(source))
        return

    target = STAGING / archive_path
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists():
        raise FileExistsError(f"Duplicate bundle destination: {archive_path}")

    target.write_bytes(source.read_bytes())
    copied.append({
        "source": str(source),
        "bundle_path": archive_path,
        "size_bytes": int(target.stat().st_size),
        "sha256": sha256_file(target),
    })

# 4. Include the latest manuscript and provenance records.
primary_files = [
    (S / "phase17_07S_provenance_updated_manuscript.md",
     "manuscript/phase17_07S_provenance_updated_manuscript.md", True),
    (S / "phase17_07S_provenance_report.json",
     "audits/phase17_07S_provenance_report.json", True),
    (S / "phase17_07S_author_review_checklist.md",
     "provenance/phase17_07S_author_review_checklist.md", True),

    (R2 / "phase17_07R2_identity_report.json",
     "provenance/phase17_07R2_identity_report.json", True),
    (R2 / "phase17_07R2_file_comparison.csv",
     "provenance/phase17_07R2_file_comparison.csv", True),
    (R2 / "phase17_07R2_current_dataset_metadata_summary.json",
     "provenance/phase17_07R2_current_dataset_metadata_summary.json", False),
    (R2 / "phase17_07R2_published_file_listing.csv",
     "provenance/phase17_07R2_published_file_listing.csv", False),

    (M / "phase17_07M_local_file_fingerprints.csv",
     "provenance/phase17_07M_local_file_fingerprints.csv", True),
    (M / "phase17_07M_external_catalogue_metadata.json",
     "provenance/phase17_07M_external_catalogue_metadata.json", False),
    (M / "phase17_07M_assessment.json",
     "provenance/phase17_07M_assessment.json", False),

    (N / "phase17_07N_lineage_evidence.json",
     "provenance/phase17_07N_lineage_evidence.json", False),
    (N / "phase17_07N_assessment.json",
     "provenance/phase17_07N_assessment.json", False),
    (O / "phase17_07O_original_loading_cells.json",
     "provenance/phase17_07O_original_loading_cells.json", False),
    (O / "phase17_07O_assessment.json",
     "provenance/phase17_07O_assessment.json", False),

    (L / "phase17_07L_numeric_consistency.csv",
     "audits/phase17_07L_numeric_consistency.csv", True),
    (L1 / "phase17_07L_R1_quality_report.json",
     "audits/phase17_07L_R1_quality_report.json", True),
    (L1 / "phase17_07L_R1_remaining_blockers.txt",
     "audits/phase17_07L_R1_remaining_blockers.txt", False),

    (G / "phase17_07G_paired_day_block_bootstrap_results.csv",
     "results/phase17_07G_paired_day_cluster_bootstrap.csv", True),
    (G / "phase17_07G_bootstrap_report.json",
     "results/phase17_07G_bootstrap_report.json", True),

    (I / "phase17_07I_draft_results.md",
     "results/phase17_07I_draft_results.md", False),
    (I / "phase17_07I_evidence_package_manifest.json",
     "results/phase17_07I_evidence_package_manifest.json", False),
    (J / "phase17_07J_manifest.json",
     "manuscript/phase17_07J_manifest.json", False),
]

for source, archive_path, is_required in primary_files:
    add_file(source, archive_path, required_file=is_required)

# 5. Include the refreshed evidence tables.
# These are compact tables and exclude raw datasets and fitted models.
for source_dir, archive_prefix in [
    (I / "tables", "results/phase17_07I_tables"),
    (J / "tables", "results/phase17_07J_tables"),
    (F, "results/phase17_07F_stratified_tables"),
]:
    if not source_dir.exists():
        missing_optional.append(str(source_dir))
        continue

    for source in sorted(source_dir.glob("*.csv")):
        add_file(source, f"{archive_prefix}/{source.name}", required_file=False)

# Also include the existing validation/ensemble comparison tables.
for rel, archive_path in [
    ("outputs/reports/phase15_7_final_model_comparison.csv",
     "results/phase15_7_validation_model_comparison.csv"),
    ("outputs/reports/phase15_9_ensemble_validation_results.csv",
     "results/phase15_9_validation_ensemble_comparison.csv"),
    ("outputs/reports/phase15_10_ensemble_validation_summary.csv",
     "results/phase15_10_validation_ensemble_summary.csv"),
]:
    add_file(ROOT / rel, archive_path, required_file=False)

# 6. Extract a current reference register from the latest manuscript itself.
current_manuscript = (
    STAGING / "manuscript/phase17_07S_provenance_updated_manuscript.md"
).read_text(encoding="utf-8")

if "## References" not in current_manuscript:
    raise ValueError("Latest manuscript has no References section.")

_, references_text = current_manuscript.split("## References", 1)
reference_entries = re.findall(
    r"(?m)^\[(\d+)\]\s*(.+)$",
    references_text
)
if not reference_entries:
    raise ValueError("No numbered reference entries were parsed from the latest manuscript.")

reference_df = pd.DataFrame([
    {"Reference_Number": int(number), "Reference_Text": entry.strip()}
    for number, entry in reference_entries
])
reference_df.to_csv(
    STAGING / "references/phase17_07T_current_reference_register.csv",
    index=False
)

# 7. Produce a transparent README.
readme = """# Delhi AQI Forecasting — Updated Research Review Bundle

## Included

- Latest provenance-updated manuscript (Phase 17.07S)
- Current reference register extracted from that manuscript
- Validation and frozen-holdout result tables
- Paired day-cluster bootstrap results
- AQI-band, station, season, transition, and feature-attribution tables
- Numerical, citation, and provenance audit records
- Local SHA-256 fingerprints and authenticated Kaggle file comparison

## Data identity

The local `station_hour.csv` and `stations.csv` were downloaded from the
current public Kaggle distribution into temporary storage and matched by
file size and SHA-256. Neither local file was modified.

SHA-256:
- station_hour.csv:
  26a423c66fe9763064d580ffb007150375cfa2056e62b29d52b1b7857af54fde
- stations.csv:
  ce2b7081c3e3c89145613eb724d7d2149ef79c6e16f36bb8a835747a673ed410

## Licence and provenance caveat

The TUMI catalogue page for the collection lists CC0 1.0 and links to the
same Kaggle collection. The current Kaggle metadata request did not expose
a licence field in the saved audit. The original notebook loads the CSVs
from `data/raw` but does not document the historical acquisition event.

The current content identity is established, but authors should retain an
explicit review of licence applicability and any rights attached to the
underlying CPCB source data before public redistribution or final submission.

## Scientific results

- Full ensemble holdout: 68,352 rows, MAE 3.216121, RMSE 7.658390, R² 0.996634.
- Paired comparison against persistence: 68,285 rows.
- Ensemble matched-support MAE: 3.178058; persistence MAE: 3.855634.
- Ensemble matched-support RMSE: 7.495249; persistence RMSE: 8.840011.
- Abrupt transitions remain difficult: about 9% recall at the ±20 AQI
  predicted-change threshold.

## Excluded

- Raw CSV datasets
- Fitted model files
- API tokens, secrets, and credentials
- Deployment packages and unrelated files

This is a research-review bundle, not a legal determination of data rights or
a declaration of production readiness.
"""
(STAGING / "README.md").write_text(readme, encoding="utf-8")

# 8. Add the file manifest and checksum list.
manifest = {
    "phase": "17.07T",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "bundle_type": "updated_research_review_bundle",
    "latest_manuscript": (
        "manuscript/phase17_07S_provenance_updated_manuscript.md"
    ),
    "both_local_csv_files_match_current_kaggle": True,
    "historical_acquisition_event_documented": False,
    "tumi_catalogue_lists_cc0_1_0": True,
    "kaggle_api_licence_field_retrieved": False,
    "author_review_of_upstream_rights_still_recommended": True,
    "submission_ready": False,
    "production_enabled": False,
    "missing_optional_sources": missing_optional,
    "files": copied,
}
(STAGING / "phase17_07T_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

checksum_lines = []
for path in sorted(STAGING.rglob("*")):
    if path.is_file() and path.name != "SHA256SUMS.txt":
        checksum_lines.append(
            f"{sha256_file(path)}  {path.relative_to(STAGING)}"
        )

(STAGING / "SHA256SUMS.txt").write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8"
)

# 9. Create and verify ZIP.
ZIP_PATH = OUT / "Delhi_AQI_Updated_Research_Review_Bundle.zip"

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(STAGING.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(STAGING)))

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    invalid_member = archive.testzip()
    names = archive.namelist()

if invalid_member is not None:
    raise RuntimeError(f"ZIP integrity check failed: {invalid_member}")

summary = {
    "phase": "17.07T",
    "zip_path": str(ZIP_PATH),
    "zip_size_bytes": int(ZIP_PATH.stat().st_size),
    "zip_sha256": sha256_file(ZIP_PATH),
    "member_count": len(names),
    "zip_integrity_check": "PASS",
    "latest_manuscript_included": (
        "manuscript/phase17_07S_provenance_updated_manuscript.md" in names
    ),
    "both_local_source_files_identity_confirmed": True,
    "licence_review_remains": True,
    "submission_ready": False,
    "production_enabled": False,
}
(OUT / "phase17_07T_package_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n[1] UPDATED BUNDLE CREATED")
print("ZIP:", ZIP_PATH)
print("Size (bytes):", summary["zip_size_bytes"])
print("SHA-256:", summary["zip_sha256"])
print("Files inside ZIP:", summary["member_count"])
print("ZIP integrity:", summary["zip_integrity_check"])
print("Current Phase 17.07S manuscript included:",
      summary["latest_manuscript_included"])

print("\n[2] STATUS")
print("Current Kaggle file identity confirmed: True")
print("Historical acquisition event documented: False")
print("Licence review of underlying source data: still recommended")
print("Submission ready: False")
print("Models retrained: False")
print("Production enabled: False")

if missing_optional:
    print("\n[3] OPTIONAL FILES NOT FOUND")
    for item in missing_optional:
        print("-", item)

print("\nSummary:", OUT / "phase17_07T_package_summary.json")
print("=" * 78)

In [ ]:
# PHASE 17.07T-R1 — REPAIR INCOMPLETE REVIEW BUNDLE
# Resumes the failed bundle; does not recopy or modify original project data.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import zipfile
import pandas as pd

print("=" * 78)
print("PHASE 17.07T-R1 — REVIEW BUNDLE REPAIR")
print("=" * 78)

# 1. Resolve the project.
roots = [
    Path("/content/gdrive/MyDrive/Delhi-AQI-Prediction"),
    Path("/content/drive/MyDrive/Delhi-AQI-Prediction"),
]
ROOT = next(
    (p for p in roots if p.exists() and (p / "outputs").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Delhi-AQI-Prediction project folder not found.")

PAPER = ROOT / "outputs/research_paper"

# 2. Find the incomplete Phase 17.07T staging folder.
candidates = []
for folder in PAPER.glob("phase17_07T_review_bundle_*"):
    staging = folder / "bundle_contents"
    zip_path = folder / "Delhi_AQI_Updated_Research_Review_Bundle.zip"

    if staging.is_dir() and not zip_path.exists():
        candidates.append(folder)

if not candidates:
    raise FileNotFoundError(
        "No incomplete Phase 17.07T bundle found. "
        "Check outputs/research_paper/phase17_07T_review_bundle_*."
    )

# Prefer the newest incomplete Phase 17.07T bundle.
OUT = max(candidates, key=lambda p: p.stat().st_mtime)
STAGING = OUT / "bundle_contents"

print("Repairing bundle:", OUT)

# 3. Load the current provenance-updated manuscript already copied into staging.
manuscript_path = (
    STAGING / "manuscript/phase17_07S_provenance_updated_manuscript.md"
)
if not manuscript_path.exists():
    raise FileNotFoundError(
        f"Current Phase 17.07S manuscript is missing from staging: {manuscript_path}"
    )

manuscript = manuscript_path.read_text(encoding="utf-8")
if "## References" not in manuscript:
    raise ValueError("The latest manuscript does not contain a References section.")

_, refs_section = manuscript.split("## References", 1)
reference_entries = re.findall(
    r"(?m)^\[(\d+)\]\s*(.+)$",
    refs_section
)
if not reference_entries:
    raise ValueError("Could not extract numbered references from the manuscript.")

# FIX: create the parent directory before writing the CSV.
reference_dir = STAGING / "references"
reference_dir.mkdir(parents=True, exist_ok=True)

reference_df = pd.DataFrame([
    {
        "Reference_Number": int(number),
        "Reference_Text": entry.strip(),
    }
    for number, entry in reference_entries
])

reference_path = reference_dir / "phase17_07T_current_reference_register.csv"
reference_df.to_csv(reference_path, index=False, encoding="utf-8")

print("\n[1] Reference register repaired")
print("References:", len(reference_df))
print("Saved:", reference_path)

# 4. Write README into the staged bundle.
readme = """# Delhi AQI Forecasting — Updated Research Review Bundle

This bundle contains the latest provenance-updated manuscript, research tables,
reference register, numerical checks, and file-identity audit.

## Verified file identity

The local `station_hour.csv` and `stations.csv` matched files downloaded from
the current Kaggle distribution byte for byte by SHA-256 and file size.

- station_hour.csv SHA-256:
  26a423c66fe9763064d580ffb007150375cfa2056e62b29d52b1b7857af54fde
- stations.csv SHA-256:
  ce2b7081c3e3c89145613eb724d7d2149ef79c6e16f36bb8a835747a673ed410

The identity comparison does not establish the historical acquisition event.

## Licence caveat

The TUMI catalogue lists CC0 1.0 for the linked dataset collection. The Kaggle
metadata audit did not return a licence field. The author should review the
applicability of the stated licence and any rights attached to the underlying
CPCB source data before submission or redistribution.

## Final-model results

- Full holdout: 68,352 rows; ensemble MAE 3.216121, RMSE 7.658390, R² 0.996634.
- Matched model-comparison support: 68,285 rows.
- Persistence matched-support MAE 3.855634; RMSE 8.840011.
- Ensemble matched-support MAE 3.178058; RMSE 7.495249.
- Abrupt-transition recall remains low, approximately 9% at the ±20-point
  predicted-change threshold.

## Excluded from this bundle

Raw datasets, fitted models, API tokens, secrets, deployment packages, and
unrelated files are intentionally excluded.

This is a research-review bundle, not a legal determination of data rights or
a declaration that the system is ready for production.
"""
(STAGING / "README.md").write_text(readme, encoding="utf-8")

# 5. Hash helper.
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

# 6. Create a manifest for all staged content except the manifest/checksum files.
def list_content_files():
    excluded = {"phase17_07T_manifest.json", "SHA256SUMS.txt"}
    return sorted(
        p for p in STAGING.rglob("*")
        if p.is_file() and p.name not in excluded
    )

content_files = list_content_files()
manifest = {
    "phase": "17.07T-R1",
    "repaired_at_utc": datetime.now(timezone.utc).isoformat(),
    "bundle_type": "updated_research_review_bundle",
    "latest_manuscript": (
        "manuscript/phase17_07S_provenance_updated_manuscript.md"
    ),
    "reference_register": (
        "references/phase17_07T_current_reference_register.csv"
    ),
    "both_local_csv_files_match_current_kaggle": True,
    "historical_acquisition_event_documented": False,
    "tumi_catalogue_lists_cc0_1_0": True,
    "kaggle_api_licence_field_retrieved": False,
    "author_licence_review_still_recommended": True,
    "submission_ready": False,
    "production_enabled": False,
    "files": [
        {
            "bundle_path": str(p.relative_to(STAGING)),
            "size_bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        }
        for p in content_files
    ],
}
(STAGING / "phase17_07T_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# 7. Create checksum list after manifest is complete.
checksum_path = STAGING / "SHA256SUMS.txt"
checksum_lines = []

for path in sorted(STAGING.rglob("*")):
    if path.is_file() and path != checksum_path:
        checksum_lines.append(
            f"{sha256_file(path)}  {path.relative_to(STAGING)}"
        )

checksum_path.write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8",
)

# 8. Create and verify the ZIP.
ZIP_PATH = OUT / "Delhi_AQI_Updated_Research_Review_Bundle.zip"

with zipfile.ZipFile(
    ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(STAGING.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(STAGING)))

with zipfile.ZipFile(ZIP_PATH, mode="r") as archive:
    bad_member = archive.testzip()
    members = archive.namelist()

if bad_member is not None:
    raise RuntimeError(f"ZIP integrity check failed at: {bad_member}")

# 9. Save summary outside the ZIP and verify key members.
required_members = [
    "manuscript/phase17_07S_provenance_updated_manuscript.md",
    "references/phase17_07T_current_reference_register.csv",
    "provenance/phase17_07R2_identity_report.json",
    "README.md",
    "phase17_07T_manifest.json",
    "SHA256SUMS.txt",
]
missing_members = [m for m in required_members if m not in members]

summary = {
    "phase": "17.07T-R1",
    "zip_path": str(ZIP_PATH),
    "zip_size_bytes": int(ZIP_PATH.stat().st_size),
    "zip_sha256": sha256_file(ZIP_PATH),
    "zip_member_count": len(members),
    "zip_integrity_check": "PASS",
    "required_members_missing": missing_members,
    "reference_count": int(len(reference_df)),
    "latest_manuscript_included": (
        "manuscript/phase17_07S_provenance_updated_manuscript.md" in members
    ),
    "exact_file_identity_confirmed": True,
    "licence_review_still_required": True,
    "submission_ready": False,
    "models_retrained": False,
    "models_retuned": False,
    "original_data_modified": False,
    "production_enabled": False,
}
(OUT / "phase17_07T_R1_package_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n[2] ZIP PACKAGE")
print("Path:", ZIP_PATH)
print("Size:", summary["zip_size_bytes"], "bytes")
print("SHA-256:", summary["zip_sha256"])
print("Files:", summary["zip_member_count"])
print("Integrity check:", summary["zip_integrity_check"])
print("Missing required members:", missing_members)
print("Reference count:", summary["reference_count"])

if missing_members:
    raise RuntimeError(
        "ZIP was created, but required members are missing: "
        + ", ".join(missing_members)
    )

print("\n[3] FINAL STATUS")
print("Bundle repair: PASS")
print("Original dataset files modified: False")
print("Models retrained: False")
print("Production enabled: False")
print("Submission ready: False — author licence review remains")
print("Summary:", OUT / "phase17_07T_R1_package_summary.json")
print("=" * 78)

In [ ]:
# PHASE 17.07U — AUTHOR RELEASE GATE
# Creates a separate review checklist and release-status record.
# Does not modify the frozen review bundle, source data, or models.

from pathlib import Path
from datetime import datetime
import hashlib
import json
import zipfile

print("=" * 78)
print("PHASE 17.07U — AUTHOR RELEASE GATE")
print("=" * 78)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Delhi-AQI-Prediction"
)

EXPECTED_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

BUNDLE_GLOB = (
    "outputs/research_paper/"
    "phase17_07T_review_bundle_*/"
    "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

# 1. Find and verify the exact repaired bundle.
candidates = list(PROJECT_ROOT.glob(BUNDLE_GLOB))
matching_bundles = [
    path for path in candidates
    if path.is_file() and sha256_file(path) == EXPECTED_SHA256
]

if not matching_bundles:
    raise FileNotFoundError(
        "Could not find the repaired review bundle with the expected SHA-256. "
        "No release-gate files were written. Check that Google Drive is mounted "
        "and the Phase 17.07T-R1 bundle still exists."
    )

bundle_path = max(matching_bundles, key=lambda p: p.stat().st_mtime)

with zipfile.ZipFile(bundle_path, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(
            f"ZIP integrity check failed at member: {bad_member}"
        )
    archive_members = archive.namelist()

actual_sha256 = sha256_file(bundle_path)

# 2. Create a new, separate directory so earlier audit outputs are preserved.
timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
gate_dir = (
    PROJECT_ROOT
    / "outputs"
    / "research_paper"
    / f"phase17_07U_author_release_gate_{timestamp}"
)
gate_dir.mkdir(parents=True, exist_ok=False)

# 3. Record the remaining human review tasks.
checklist = f"""# Phase 17.07U — Author Release Gate

## Package verification

- Review bundle: `{bundle_path}`
- Archive SHA-256: `{actual_sha256}`
- ZIP integrity: PASS
- Archived file entries: {len(archive_members)}
- This phase modified the source dataset or frozen models: NO

## Release status

**Submission status: NOT CLEARED**

Author licence/provenance review remains outstanding. This is a release gate,
not a finding that the dataset is or is not legally reusable.

## Evidence currently recorded in the project

- Phase 17.07R2 found the local `station_hour.csv` and `stations.csv` files
  byte-identical to the corresponding files retrieved from the current public
  Kaggle distribution.
- Phase 17.07S recorded that the TUMI catalogue listed CC0 1.0.
- The Kaggle metadata API did not return a licence field in that review.
- The original historical acquisition event is not documented in the project.
- The rights/terms applicable to the underlying CPCB source have not been
  independently confirmed.

A byte-for-byte match establishes file identity with the copy checked; it does
not, on its own, establish permission to redistribute or publish the material.

## Author checklist — complete before submission

- [ ] Re-open the TUMI catalogue record and record the current licence statement,
      source URL, access date, and any qualifications.
- [ ] Check the currently displayed Kaggle dataset page and its applicable terms.
- [ ] Investigate CPCB's original source and any applicable reuse or redistribution
      conditions; do not assume the portal listing resolves all upstream rights.
- [ ] Determine whether the review bundle contains raw data or only derived
      artifacts. Verify the terms for anything intended for redistribution.
- [ ] Review required attribution, acknowledgements, and licence notices.
- [ ] Confirm that the intended publication and sharing use is covered by the
      terms you have verified.
- [ ] Record the final decision and supporting source URLs in the licence log.
- [ ] Have the responsible author/reviewer date and sign off on the decision.

If the rights remain unclear, do not represent the manuscript as submission-ready
or redistribute the questionable material until the issue has been resolved.

## Operational safeguard

- Frozen model selection/tuning: unchanged
- Model retraining: not performed
- Production: remains disabled
- Next allowed action: complete and document the author licence review
"""

checklist_path = gate_dir / "AUTHOR_RELEASE_CHECKLIST.md"
checklist_path.write_text(checklist, encoding="utf-8")

license_log = """# Author Licence Review Log

Fill this in only after checking the relevant source records.

- Reviewer:
- Review date:
- TUMI catalogue URL:
- Licence statement shown on TUMI:
- Kaggle dataset URL:
- Licence/terms shown by Kaggle:
- CPCB original-source record or terms URL:
- Evidence about upstream reuse/redistribution rights:
- Does the planned use include redistributing raw data?:
- Attribution or notice requirements:
- Intended use assessed (research, publication, artifact sharing):
- Decision: [CLEARED FOR STATED USE / NOT CLEARED / SEEK FURTHER REVIEW]
- Reasoning and evidence:
- Follow-up required:
- Final sign-off:
"""

license_log_path = gate_dir / "AUTHOR_LICENCE_REVIEW_LOG_TEMPLATE.md"
license_log_path.write_text(license_log, encoding="utf-8")

status = {
    "phase": "17.07U",
    "created_at": datetime.now().astimezone().isoformat(),
    "review_bundle_path": str(bundle_path),
    "review_bundle_sha256": actual_sha256,
    "review_bundle_sha256_matches_expected": actual_sha256 == EXPECTED_SHA256,
    "zip_integrity": "PASS",
    "archive_entry_count": len(archive_members),
    "source_dataset_files_modified": False,
    "frozen_models_retrained": False,
    "production_enabled": False,
    "author_licence_review": "REQUIRED",
    "submission_ready": False,
    "blocking_gates": [
        "Author must verify and document the licence and upstream reuse rights "
        "for the dataset and the intended publication/sharing use."
    ],
    "outputs": {
        "checklist": str(checklist_path),
        "licence_review_log_template": str(license_log_path),
    },
}

status_path = gate_dir / "phase17_07U_release_gate_status.json"
status_path.write_text(
    json.dumps(status, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n[1] REVIEW BUNDLE")
print(f"Path: {bundle_path}")
print(f"SHA-256 verified: {actual_sha256 == EXPECTED_SHA256}")
print(f"ZIP integrity: PASS")
print(f"Archive entries: {len(archive_members)}")

print("\n[2] AUTHOR REVIEW FILES")
print(f"Checklist: {checklist_path}")
print(f"Licence log template: {license_log_path}")
print(f"Status JSON: {status_path}")

print("\n[3] RELEASE GATE")
print("Author licence review: REQUIRED")
print("Submission ready: False")
print("Production enabled: False")
print("Source dataset/models modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.07V — CURRENT LICENCE EVIDENCE & BUNDLE CONTENT REVIEW
# Creates a new evidence record; does not alter the existing ZIP or models.

from pathlib import Path, PurePosixPath
from datetime import datetime
import hashlib
import json
import zipfile

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Delhi-AQI-Prediction"
)

BUNDLE = (
    PROJECT_ROOT
    / "outputs/research_paper"
    / "phase17_07T_review_bundle_20261025T181246Z"
    / "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)

EXPECTED_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

def calculate_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("=" * 78)
print("PHASE 17.07V — LICENCE EVIDENCE REVIEW")
print("=" * 78)

# 1. Verify the reviewed archive before examining it.
if not BUNDLE.is_file():
    raise FileNotFoundError(f"Review bundle not found: {BUNDLE}")

actual_sha256 = calculate_sha256(BUNDLE)
if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "Review bundle SHA-256 does not match the Phase 17.07T-R1 record. "
        "No evidence files have been written."
    )

with zipfile.ZipFile(BUNDLE, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity failure: {bad_member}")
    members = archive.namelist()

# 2. Inventory possible raw-data files without extracting or modifying them.
raw_names = {
    "station_hour.csv",
    "stations.csv",
    "station_day.csv",
    "station.csv",
    "city_hour.csv",
    "city_day.csv",
}

possible_data_members = []
csv_members = []
parquet_members = []

for member in members:
    path = PurePosixPath(member)
    name = path.name.lower()
    normalized_parts = [part.lower() for part in path.parts]

    if name in raw_names or "raw" in normalized_parts:
        possible_data_members.append(member)

    if name.endswith(".csv"):
        csv_members.append(member)

    if name.endswith(".parquet"):
        parquet_members.append(member)

# 3. Save a sourced, qualified evidence note.
timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
out_dir = (
    PROJECT_ROOT
    / "outputs"
    / "research_paper"
    / f"phase17_07V_license_evidence_{timestamp}"
)
out_dir.mkdir(parents=True, exist_ok=False)

evidence_note = f"""# Phase 17.07V — Current Licence Evidence Review

Review date: {datetime.now().astimezone().isoformat()}

## Archive identity

- Archive: `{BUNDLE}`
- SHA-256: `{actual_sha256}`
- Expected SHA-256 matched: YES
- ZIP integrity: PASS
- Archive entries: {len(members)}

## Current source evidence

### TUMI catalogue

URL:
https://portal.transport-data.org/@tumi/urban-air-pollution-india-2015-2020

The publicly visible catalogue record states that the collection contains
hourly and daily air-quality data, says it was compiled by Rohan Rao from data
provided by CPCB, links to the Kaggle dataset, and displays CC0 1.0.
The record shows an update date of August 20, 2026.

The licence shown by a catalogue is evidence of that catalogue's statement.
It is not, on its own, independent verification that every upstream right
needed for every form of redistribution has been cleared.

Catalogue contact displayed on the record: info@transformative-mobility.org

### Kaggle distribution

URL:
https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india/data

The earlier Phase 17.07R2 check recorded that the local
`station_hour.csv` and `stations.csv` are byte-identical to files from the
current public Kaggle distribution.

During the current browser review, the Kaggle page did not expose readable
licence metadata to the reviewer. The author should open the dataset page
directly and save a dated record or screenshot of the licence/terms shown.

### CC0 reference

URL:
https://creativecommons.org/publicdomain/zero/1.0/

The CC0 deed describes broad permissions to copy, modify, distribute and make
commercial use of a work, while cautioning that Creative Commons has not
verified the copyright status of any work to which CC0 has been applied.

### Indian Open Government Data terms

Terms:
https://www.data.gov.in/terms-of-use

Government Open Data License India:
https://www.data.gov.in/Godl

The portal terms state that individual dataset licence metadata governs unless
otherwise noted and caution that permission does not extend to material
identified as third-party copyrighted material. These general portal terms
do not establish the licence of this specific Kaggle/TUMI collection or resolve
the rights in the underlying CPCB-derived material.

Official CPCB portal reference:
https://cpcb.gov.in/air-quality-management-portals/

No definitive CPCB permission or upstream licence has been established by this
review.

## Bundle content scan

- CSV members: {len(csv_members)}
- Parquet members: {len(parquet_members)}
- Possible raw-data members, based on filenames/path: {len(possible_data_members)}

This is a filename/path scan, not a legal determination. Review the listed
members and their contents before deciding whether the bundle redistributes
raw source data.

## Decision

**Author licence review remains OPEN. Submission readiness remains FALSE.**

Required before release:
1. Capture the current licence and terms shown on the Kaggle dataset page.
2. Ask TUMI to clarify the basis for applying CC0 to the CPCB-derived collection,
   including any known upstream limitations.
3. Check relevant CPCB terms or obtain confirmation from the responsible source.
4. Decide whether the exact intended use and any included raw data are covered.
5. Record the evidence and the responsible author's final decision in the
   Phase 17.07U licence review log.

Do not infer legal clearance from byte identity or catalogue visibility alone.

## Project safeguards

- Source data changed by this phase: NO
- Existing review archive changed by this phase: NO
- Frozen models retrained: NO
- Production enabled: NO
"""

note_path = out_dir / "phase17_07V_license_evidence_review.md"
note_path.write_text(evidence_note, encoding="utf-8")

inventory_path = out_dir / "phase17_07V_bundle_content_inventory.json"
inventory = {
    "archive": str(BUNDLE),
    "sha256": actual_sha256,
    "zip_integrity": "PASS",
    "archive_entries": len(members),
    "csv_members": csv_members,
    "parquet_members": parquet_members,
    "possible_raw_data_members": possible_data_members,
}
inventory_path.write_text(
    json.dumps(inventory, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

status = {
    "phase": "17.07V",
    "archive_verified": True,
    "zip_integrity": "PASS",
    "source_data_modified": False,
    "existing_bundle_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "evidence_note": str(note_path),
    "bundle_inventory": str(inventory_path),
}
status_path = out_dir / "phase17_07V_status.json"
status_path.write_text(
    json.dumps(status, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n[1] ARCHIVE")
print(f"SHA-256 verified: {actual_sha256 == EXPECTED_SHA256}")
print(f"ZIP integrity: PASS")
print(f"Entries: {len(members)}")

print("\n[2] CONTENT SCAN")
print(f"CSV members: {len(csv_members)}")
print(f"Parquet members: {len(parquet_members)}")
print(f"Possible raw-data members: {len(possible_data_members)}")
for item in possible_data_members:
    print("  -", item)

print("\n[3] NEW EVIDENCE FILES")
print("Evidence note:", note_path)
print("Content inventory:", inventory_path)
print("Status:", status_path)

print("\n[4] RELEASE GATE")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07W — CSV CONTENT AND RAW-DATA SCHEMA AUDIT
# Read-only inspection of the verified review ZIP.

from pathlib import Path
from datetime import datetime
from zipfile import ZipFile
from io import TextIOWrapper
import csv
import hashlib
import json

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Delhi-AQI-Prediction"
)

BUNDLE = (
    PROJECT_ROOT
    / "outputs/research_paper"
    / "phase17_07T_review_bundle_20261025T181246Z"
    / "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)

EXPECTED_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

EXPECTED_SOURCE_HASHES = {
    "station_hour.csv":
        "26a423c66fe9763064d580ffb007150375cfa2056e62b29d52b1b7857af54fde",
    "stations.csv":
        "ce2b7081c3e3c89145613eb724d7d2149ef79c6e16f36bb8a835747a673ed410",
}

STATION_HOUR_HEADER = [
    "StationId", "Datetime", "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene",
    "AQI", "AQI_Bucket"
]
STATIONS_HEADER = [
    "StationId", "StationName", "City", "State", "Status"
]

def sha256_bytes_stream(stream):
    digest = hashlib.sha256()
    size = 0
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(block)
        size += len(block)
    return digest.hexdigest(), size

def normalize_header(header):
    return tuple(str(value).strip().lower() for value in header)

def get_schema_match(header):
    normalized = normalize_header(header)
    if normalized == normalize_header(STATION_HOUR_HEADER):
        return "EXACT_STATION_HOUR_SOURCE_SCHEMA"
    if normalized == normalize_header(STATIONS_HEADER):
        return "EXACT_STATIONS_METADATA_SCHEMA"
    return "No exact known source schema match"

print("=" * 78)
print("PHASE 17.07W — CSV CONTENT AND RAW-DATA SCHEMA AUDIT")
print("=" * 78)

# Verify the exact ZIP before reading its contents.
if not BUNDLE.is_file():
    raise FileNotFoundError(f"Review bundle not found: {BUNDLE}")

with open(BUNDLE, "rb") as f:
    bundle_sha256, _ = sha256_bytes_stream(f)

if bundle_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "Bundle SHA-256 mismatch. Audit stopped; no outputs written."
    )

timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "research_paper"
    / f"phase17_07W_csv_content_audit_{timestamp}"
)
OUT_DIR.mkdir(parents=True, exist_ok=False)

records = []
csv_errors = []

with ZipFile(BUNDLE, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity failure at {bad_member}")

    member_names = archive.namelist()
    csv_names = [
        name for name in member_names
        if not name.endswith("/")
        and name.lower().endswith(".csv")
    ]

    for index, name in enumerate(csv_names, start=1):
        info = archive.getinfo(name)

        # Compute the hash of the archived CSV without extracting it.
        with archive.open(name, "r") as binary_stream:
            member_sha256, actual_size = sha256_bytes_stream(binary_stream)

        header = []
        data_rows = None
        parse_error = None

        try:
            with archive.open(name, "r") as binary_stream:
                text_stream = TextIOWrapper(
                    binary_stream,
                    encoding="utf-8-sig",
                    errors="replace",
                    newline=""
                )
                reader = csv.reader(text_stream)
                header = next(reader, [])
                # Count records with the CSV parser rather than counting lines.
                data_rows = sum(1 for _ in reader)
        except Exception as exc:
            parse_error = f"{type(exc).__name__}: {exc}"
            csv_errors.append({"member": name, "error": parse_error})

        schema_match = get_schema_match(header) if header else "Header unavailable"
        exact_source_hash_match = next(
            (
                source_name
                for source_name, expected_hash
                in EXPECTED_SOURCE_HASHES.items()
                if member_sha256 == expected_hash
            ),
            None
        )

        records.append({
            "member": name,
            "bytes": actual_size,
            "sha256": member_sha256,
            "header": header,
            "data_rows": data_rows,
            "schema_match": schema_match,
            "exact_source_hash_match": exact_source_hash_match,
            "parse_error": parse_error,
        })

# CSV inventory: no raw record values are copied to the audit output.
inventory_csv = OUT_DIR / "phase17_07W_csv_inventory.csv"
with open(inventory_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "member", "bytes", "sha256", "header", "data_rows",
            "schema_match", "exact_source_hash_match", "parse_error"
        ]
    )
    writer.writeheader()
    for record in records:
        row = record.copy()
        row["header"] = json.dumps(row["header"], ensure_ascii=False)
        writer.writerow(row)

exact_hash_matches = [
    r for r in records if r["exact_source_hash_match"] is not None
]
source_schema_matches = [
    r for r in records
    if r["schema_match"] in {
        "EXACT_STATION_HOUR_SOURCE_SCHEMA",
        "EXACT_STATIONS_METADATA_SCHEMA"
    }
]
largest_files = sorted(records, key=lambda r: r["bytes"], reverse=True)[:10]

# Write an interpretable summary with the limitations stated explicitly.
summary = {
    "phase": "17.07W",
    "created_at": datetime.now().astimezone().isoformat(),
    "bundle_path": str(BUNDLE),
    "bundle_sha256": bundle_sha256,
    "bundle_sha256_verified": bundle_sha256 == EXPECTED_SHA256,
    "zip_integrity": "PASS",
    "total_archive_entries": len(member_names),
    "csv_member_count": len(csv_names),
    "csv_parse_error_count": len(csv_errors),
    "exact_raw_source_hash_matches": exact_hash_matches,
    "known_raw_source_schema_matches": source_schema_matches,
    "largest_csv_files": largest_files,
    "source_dataset_modified": False,
    "review_bundle_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "interpretation_limit": (
        "Absence of exact filename, hash, or schema matches does not prove "
        "that a file contains no raw or transformed source observations. "
        "This is a screening audit, not a legal licence decision."
    ),
}

summary_json = OUT_DIR / "phase17_07W_csv_audit_summary.json"
summary_json.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

lines = [
    "# Phase 17.07W — CSV Content Audit",
    "",
    f"- Bundle SHA-256 verified: {bundle_sha256 == EXPECTED_SHA256}",
    "- ZIP integrity: PASS",
    f"- CSV files inspected: {len(csv_names)}",
    f"- CSV parsing errors: {len(csv_errors)}",
    f"- Exact raw-source hash matches: {len(exact_hash_matches)}",
    f"- Exact known source-schema matches: {len(source_schema_matches)}",
    "",
    "## Exact source hash matches",
]
if exact_hash_matches:
    lines.extend(
        f"- `{r['member']}` matches `{r['exact_source_hash_match']}` byte-for-byte"
        for r in exact_hash_matches
    )
else:
    lines.append("- None found.")

lines.extend(["", "## Known source-schema matches"])
if source_schema_matches:
    for r in source_schema_matches:
        lines.append(
            f"- `{r['member']}` — {r['schema_match']}; "
            f"data records: {r['data_rows']}"
        )
else:
    lines.append("- None found.")

lines.extend(["", "## Ten largest CSV files"])
for r in largest_files:
    lines.append(
        f"- `{r['member']}` — {r['bytes']:,} bytes; "
        f"data records: {r['data_rows']}"
    )

if csv_errors:
    lines.extend(["", "## Parsing errors"])
    for e in csv_errors:
        lines.append(f"- `{e['member']}` — {e['error']}")

lines.extend([
    "",
    "## Decision and limitations",
    "",
    "**Author licence review remains open. Submission ready: NO.**",
    "",
    "A matching hash proves byte identity with the recorded source file. "
    "A matching schema is a review flag, not proof of identical contents. "
    "No match does not prove that transformed or partial source observations "
    "are absent. This report does not establish reuse rights.",
    "",
    "The source data, existing ZIP, and frozen models were not modified."
])

summary_md = OUT_DIR / "phase17_07W_csv_audit_summary.md"
summary_md.write_text("\n".join(lines), encoding="utf-8")

print("\n[1] VERIFIED ARCHIVE")
print("SHA-256:", bundle_sha256)
print("ZIP integrity: PASS")
print("CSV files inspected:", len(csv_names))
print("CSV parsing errors:", len(csv_errors))

print("\n[2] SOURCE MATCH SCAN")
print("Exact source file hash matches:", len(exact_hash_matches))
for record in exact_hash_matches:
    print("  EXACT HASH:", record["member"], "->",
          record["exact_source_hash_match"])

print("Known source schema matches:", len(source_schema_matches))
for record in source_schema_matches:
    print(
        "  SCHEMA FLAG:", record["member"],
        "|", record["schema_match"],
        "| rows:", record["data_rows"]
    )

print("\n[3] AUDIT OUTPUTS")
print("CSV inventory:", inventory_csv)
print("Summary:", summary_md)
print("JSON:", summary_json)

print("\n[4] RELEASE GATE")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data / existing bundle / models modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.07X — ROW-LEVEL SOURCE-LIKE CONTENT TRIAGE
# Heuristic screening only; it does not determine legal reuse rights.

from pathlib import Path
from datetime import datetime
import csv
import json
import re

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Delhi-AQI-Prediction"
)

INVENTORY = (
    PROJECT_ROOT
    / "outputs/research_paper"
    / "phase17_07W_csv_content_audit_20261025T182227"
    / "phase17_07W_csv_inventory.csv"
)

if not INVENTORY.is_file():
    raise FileNotFoundError(
        f"Phase 17.07W inventory not found: {INVENTORY}"
    )

def norm(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

TIME_FIELDS = {
    "datetime", "timestamp", "datetimeutc", "datetimelocal",
    "date", "dateandtime", "time", "observationtime",
    "forecasttime", "validtime", "targetdatetime"
}
STATION_FIELDS = {
    "stationid", "station", "stationname", "stationcode",
    "stationkey", "monitoringstation", "siteid", "sitename"
}
AQI_FIELDS = {
    "aqi", "aqibucket", "aqitplus1", "actualaqi",
    "targetaqi", "predictedaqi", "ensembleprediction",
    "finalensembleprediction", "persistenceprediction"
}
POLLUTANT_FIELDS = {
    "pm25", "pm10", "no", "no2", "nox", "nh3", "co",
    "so2", "o3", "benzene", "toluene", "xylene"
}
PREDICTION_FIELDS = {
    "prediction", "predicted", "actual", "target",
    "residual", "error", "absoluteerror", "squarederror"
}

def classify(header):
    fields = {norm(x) for x in header}

    time_hits = sorted(fields & TIME_FIELDS)
    station_hits = sorted(fields & STATION_FIELDS)
    aqi_hits = sorted(fields & AQI_FIELDS)
    pollutant_hits = sorted(fields & POLLUTANT_FIELDS)
    prediction_hits = sorted(fields & PREDICTION_FIELDS)

    has_time = bool(time_hits)
    has_station = bool(station_hits)
    has_observation_value = bool(
        aqi_hits or pollutant_hits or prediction_hits
    )

    if has_time and has_station and has_observation_value:
        risk = "HIGH — row-level station/time/value table"
        action = "MANUAL REVIEW REQUIRED"
    elif has_time and has_observation_value:
        risk = "MEDIUM — time-indexed value table"
        action = "REVIEW CONTENT AND PURPOSE"
    elif has_station and (aqi_hits or pollutant_hits):
        risk = "MEDIUM — station-level AQI/pollutant table"
        action = "REVIEW CONTENT AND PURPOSE"
    else:
        risk = "LOWER SIGNAL — no common observation signature"
        action = "Spot-check before release"

    return {
        "risk_screen": risk,
        "review_action": action,
        "column_count": len(header),
        "time_columns": time_hits,
        "station_columns": station_hits,
        "aqi_columns": aqi_hits,
        "pollutant_columns": pollutant_hits,
        "prediction_or_error_columns": prediction_hits,
    }

with open(INVENTORY, "r", encoding="utf-8", newline="") as f:
    inventory_rows = list(csv.DictReader(f))

timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT_DIR = (
    PROJECT_ROOT
    / "outputs/research_paper"
    / f"phase17_07X_source_like_content_triage_{timestamp}"
)
OUT_DIR.mkdir(parents=True, exist_ok=False)

triage_rows = []
high_risk = []
medium_risk = []

for row in inventory_rows:
    try:
        header = json.loads(row["header"]) if row.get("header") else []
        if not isinstance(header, list):
            header = []
    except (json.JSONDecodeError, TypeError):
        header = []

    classification = classify(header)
    record = {
        "member": row.get("member", ""),
        "bytes": int(row.get("bytes") or 0),
        "data_rows": row.get("data_rows", ""),
        "sha256": row.get("sha256", ""),
        "header": json.dumps(header, ensure_ascii=False),
        **{
            key: (
                json.dumps(value, ensure_ascii=False)
                if isinstance(value, list) else value
            )
            for key, value in classification.items()
        }
    }
    triage_rows.append(record)

    if classification["risk_screen"].startswith("HIGH"):
        high_risk.append(record)
    elif classification["risk_screen"].startswith("MEDIUM"):
        medium_risk.append(record)

triage_csv = OUT_DIR / "phase17_07X_source_like_content_triage.csv"
fields = [
    "member", "bytes", "data_rows", "sha256", "header",
    "risk_screen", "review_action", "column_count",
    "time_columns", "station_columns", "aqi_columns",
    "pollutant_columns", "prediction_or_error_columns"
]

with open(triage_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(triage_rows)

summary = {
    "phase": "17.07X",
    "created_at": datetime.now().astimezone().isoformat(),
    "input_inventory": str(INVENTORY),
    "csv_count_screened": len(triage_rows),
    "high_signal_files": [
        {
            "member": r["member"],
            "data_rows": r["data_rows"],
            "risk_screen": r["risk_screen"],
            "action": r["review_action"]
        }
        for r in high_risk
    ],
    "medium_signal_files": [
        {
            "member": r["member"],
            "data_rows": r["data_rows"],
            "risk_screen": r["risk_screen"],
            "action": r["review_action"]
        }
        for r in medium_risk
    ],
    "source_data_modified": False,
    "review_bundle_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "limitation": (
        "Classification uses column-name heuristics, not semantic or legal "
        "determination. Low-signal files can still contain source-derived "
        "observations. Review relevant files before redistribution."
    )
}

summary_json = OUT_DIR / "phase17_07X_triage_summary.json"
summary_json.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

summary_md = OUT_DIR / "phase17_07X_triage_summary.md"
lines = [
    "# Phase 17.07X — Source-Like Content Triage",
    "",
    f"- CSV files screened: {len(triage_rows)}",
    f"- High-signal files: {len(high_risk)}",
    f"- Medium-signal files: {len(medium_risk)}",
    "",
    "## High-signal files",
]
if high_risk:
    for r in high_risk:
        lines.append(
            f"- `{r['member']}` — rows: {r['data_rows']}; "
            f"review required"
        )
else:
    lines.append("- None flagged by the heuristic.")

lines.extend(["", "## Medium-signal files"])
if medium_risk:
    for r in medium_risk:
        lines.append(
            f"- `{r['member']}` — rows: {r['data_rows']}; "
            f"review content and purpose"
        )
else:
    lines.append("- None flagged by the heuristic.")

lines.extend([
    "",
    "## Release decision",
    "",
    "**Author licence review remains OPEN. Submission ready: NO.**",
    "",
    "This screening helps prioritize manual review. It cannot certify that "
    "the remaining files contain no source-derived data or that redistribution "
    "is permitted.",
    "",
    "The existing bundle, source data, and frozen models were not modified."
])
summary_md.write_text("\n".join(lines), encoding="utf-8")

print("=" * 78)
print("PHASE 17.07X — SOURCE-LIKE CONTENT TRIAGE")
print("=" * 78)
print("CSV files screened:", len(triage_rows))
print("High-signal files:", len(high_risk))
print("Medium-signal files:", len(medium_risk))

print("\n[HIGH-SIGNAL — REVIEW FIRST]")
for r in high_risk:
    print(
        f"- {r['member']} | rows={r['data_rows']} | "
        f"columns={r['header']}"
    )

print("\n[MEDIUM-SIGNAL — REVIEW NEXT]")
for r in medium_risk:
    print(
        f"- {r['member']} | rows={r['data_rows']} | "
        f"columns={r['header']}"
    )

print("\n[OUTPUTS]")
print("CSV triage:", triage_csv)
print("Markdown summary:", summary_md)
print("JSON summary:", summary_json)

print("\n[RELEASE GATE]")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.07Y — CSV SAMPLE-VALUE AND CONTENT REVIEW
# Read-only ZIP audit. Heuristics prioritize files for manual review.

from pathlib import Path
from zipfile import ZipFile
from datetime import datetime
from collections import Counter
import csv
import hashlib
import io
import json
import re

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Delhi-AQI-Prediction"
)

BUNDLE = (
    PROJECT_ROOT
    / "outputs/research_paper"
    / "phase17_07T_review_bundle_20261025T181246Z"
    / "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)

EXPECTED_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def normalize(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def is_number(value):
    if value is None or str(value).strip() == "":
        return False
    try:
        float(str(value).replace(",", ""))
        return True
    except (TypeError, ValueError):
        return False

# Broad aliases: this is a screening list, not a fixed schema requirement.
time_header_terms = (
    "date", "time", "timestamp", "datetime", "period",
    "month", "year", "hour", "window", "day"
)
station_header_terms = (
    "station", "siteid", "sitename", "monitoringlocation"
)
aqi_header_terms = ("aqi", "airqualityindex", "airquality")
pollutant_header_terms = (
    "pm25", "pm2.5", "pm10", "no2", "nox", "nh3",
    "so2", "benzene", "toluene", "xylene", "ozone", "pollutant"
)

date_pattern = re.compile(
    r"\b(?:19|20)\d{2}[-/]\d{1,2}[-/]\d{1,2}"
    r"(?:[ T]\d{1,2}:\d{2}(?::\d{2})?)?\b"
)
station_pattern = re.compile(r"\bDL\d{3}\b", re.IGNORECASE)

if not BUNDLE.is_file():
    raise FileNotFoundError(f"Review bundle not found: {BUNDLE}")

bundle_hash = sha256_file(BUNDLE)
if bundle_hash != EXPECTED_SHA256:
    raise RuntimeError(
        "Review bundle hash does not match the verified archive. "
        "Audit stopped before writing outputs."
    )

timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "research_paper"
    / f"phase17_07Y_csv_sample_review_{timestamp}"
)
OUT_DIR.mkdir(parents=True, exist_ok=False)

reports = []
errors = []

with ZipFile(BUNDLE, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity check failed: {bad_member}")

    csv_members = [
        info for info in archive.infolist()
        if not info.is_dir() and info.filename.lower().endswith(".csv")
    ]

    for info in csv_members:
        result = {
            "member": info.filename,
            "archive_bytes": info.file_size,
            "data_rows": None,
            "column_count": None,
            "headers": [],
            "timestamp_values_detected": 0,
            "station_codes_detected": 0,
            "known_station_names_detected": 0,
            "numeric_cells_in_sample": 0,
            "sample_cells_examined": 0,
            "header_signal": [],
            "flags": [],
            "sample_rows": [],
            "parse_error": None,
        }

        try:
            with archive.open(info, "r") as binary:
                # Decoding replacement prevents one malformed character
                # from stopping the audit; any parse failure is recorded.
                text = io.TextIOWrapper(
                    binary,
                    encoding="utf-8-sig",
                    errors="replace",
                    newline=""
                )
                reader = csv.reader(text)
                headers = next(reader, [])
                result["headers"] = headers
                result["column_count"] = len(headers)

                normalized_headers = [normalize(h) for h in headers]

                has_time_header = any(
                    any(term in h for term in time_header_terms)
                    for h in normalized_headers
                )
                has_station_header = any(
                    any(term in h for term in station_header_terms)
                    for h in normalized_headers
                )
                has_aqi_header = any(
                    any(term in h for term in aqi_header_terms)
                    for h in normalized_headers
                )
                has_pollutant_header = any(
                    any(term in h for term in pollutant_header_terms)
                    for h in normalized_headers
                )

                if has_time_header:
                    result["header_signal"].append("time/date-like header")
                if has_station_header:
                    result["header_signal"].append("station/site-like header")
                if has_aqi_header:
                    result["header_signal"].append("AQI-like header")
                if has_pollutant_header:
                    result["header_signal"].append("pollutant-like header")

                all_data_rows = 0
                numeric_cells = 0
                cells_examined = 0
                time_hits = 0
                station_hits = 0
                station_name_hits = 0
                sampled = []

                # Known examples of Delhi station names; matching is case-insensitive.
                known_names = {
                    "alipur", "anand vihar", "dtu", "ito", "wazirpur",
                    "punjabi bagh", "rk puram", "mandir marg",
                    "lodi road", "najafgarh", "dwarka", "okhla"
                }

                for row in reader:
                    all_data_rows += 1

                    if len(sampled) < 5:
                        sampled.append(row)

                    # Inspect a bounded sample to keep the audit fast.
                    if all_data_rows <= 100:
                        for value in row:
                            value = str(value).strip()
                            if not value:
                                continue

                            cells_examined += 1
                            if is_number(value):
                                numeric_cells += 1

                            if date_pattern.search(value):
                                time_hits += 1

                            if station_pattern.search(value):
                                station_hits += 1

                            value_lower = value.lower()
                            if any(
                                name in value_lower
                                for name in known_names
                            ):
                                station_name_hits += 1

                result["data_rows"] = all_data_rows
                result["timestamp_values_detected"] = time_hits
                result["station_codes_detected"] = station_hits
                result["known_station_names_detected"] = station_name_hits
                result["numeric_cells_in_sample"] = numeric_cells
                result["sample_cells_examined"] = cells_examined
                result["sample_rows"] = sampled

                # Heuristic flags; not findings of legal status.
                if time_hits and (station_hits or station_name_hits):
                    result["flags"].append(
                        "Sample contains both date-like values and "
                        "Delhi-station identifiers/names"
                    )

                if (
                    has_time_header
                    and (has_aqi_header or has_pollutant_header)
                ):
                    result["flags"].append(
                        "Time-like plus AQI/pollutant-like column names"
                    )

                if (
                    has_station_header
                    and (has_aqi_header or has_pollutant_header)
                ):
                    result["flags"].append(
                        "Station-like plus AQI/pollutant-like column names"
                    )

                if (
                    all_data_rows >= 50
                    and len(headers) >= 5
                    and cells_examined
                    and numeric_cells / cells_examined >= 0.50
                ):
                    result["flags"].append(
                        "Many rows and mostly numeric sampled cells"
                    )

        except Exception as exc:
            result["parse_error"] = f"{type(exc).__name__}: {exc}"
            errors.append({
                "member": info.filename,
                "error": result["parse_error"]
            })

        reports.append(result)

# Save a reviewable inventory; sample rows stay local in this audit report.
inventory_path = OUT_DIR / "phase17_07Y_csv_content_inventory.json"
inventory_path.write_text(
    json.dumps(reports, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

summary = {
    "phase": "17.07Y",
    "created_at": datetime.now().astimezone().isoformat(),
    "bundle_path": str(BUNDLE),
    "bundle_sha256": bundle_hash,
    "bundle_verified": bundle_hash == EXPECTED_SHA256,
    "zip_integrity": "PASS",
    "csv_count": len(csv_members),
    "parse_error_count": len(errors),
    "files_flagged_for_review": [
        {
            "member": r["member"],
            "data_rows": r["data_rows"],
            "flags": r["flags"],
            "headers": r["headers"]
        }
        for r in reports if r["flags"]
    ],
    "source_data_modified": False,
    "existing_bundle_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "limitations": [
        "This scans column names and a bounded sample of values.",
        "It may miss transformed, aggregated, partial, or renamed source data.",
        "It does not determine ownership, licence validity, or legal permission."
    ]
}

summary_json = OUT_DIR / "phase17_07Y_summary.json"
summary_json.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Print a compact index for the next decision.
print("=" * 78)
print("PHASE 17.07Y — CSV SAMPLE-VALUE AND CONTENT REVIEW")
print("=" * 78)
print("Verified SHA-256:", bundle_hash)
print("ZIP integrity: PASS")
print("CSV files examined:", len(reports))
print("Parse errors:", len(errors))
print("Files flagged for review:", len(summary["files_flagged_for_review"]))

print("\n[CSV INDEX]")
for r in reports:
    print(
        f"- {r['member']} | rows={r['data_rows']} "
        f"| cols={r['column_count']} "
        f"| flags={'; '.join(r['flags']) if r['flags'] else 'none'}"
    )
    if r["parse_error"]:
        print("  ERROR:", r["parse_error"])

print("\n[FLAGGED FILES — HEADERS AND SAMPLE ROWS]")
flagged = [r for r in reports if r["flags"]]
for r in flagged:
    print("\nFILE:", r["member"])
    print("HEADERS:", r["headers"])
    print("SAMPLE ROWS:", json.dumps(
        r["sample_rows"][:2], ensure_ascii=False
    ))
    print("FLAGS:", r["flags"])

print("\n[OUTPUTS]")
print("Full inventory:", inventory_path)
print("Summary:", summary_json)

print("\n[RELEASE GATE]")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data / existing ZIP / models modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.07Z — CONTENT AUDIT CLOSURE & RELEASE HANDOFF
# Read-only review of the verified archive; writes reports to a new folder.

from pathlib import Path, PurePosixPath
from datetime import datetime
from zipfile import ZipFile
import hashlib
import csv
import io
import json

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

BUNDLE = (
    ROOT / "outputs/research_paper"
    / "phase17_07T_review_bundle_20261025T181246Z"
    / "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)

Y_SUMMARY = (
    ROOT / "outputs/research_paper"
    / "phase17_07Y_csv_sample_review_20261025T182726"
    / "phase17_07Y_summary.json"
)

EXPECTED_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

EXPECTED_BOOTSTRAP_FILES = {
    "results/phase17_07G_paired_day_cluster_bootstrap.csv",
    "results/phase17_07I_tables/bootstrap_results.csv",
    "results/phase17_07J_tables/table_4_day_block_bootstrap.csv",
}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not BUNDLE.is_file():
    raise FileNotFoundError(f"Review bundle not found: {BUNDLE}")

actual_hash = sha256_file(BUNDLE)
if actual_hash != EXPECTED_SHA256:
    raise RuntimeError("Bundle hash mismatch; closure report not created.")

if not Y_SUMMARY.is_file():
    raise FileNotFoundError(f"Phase 17.07Y summary not found: {Y_SUMMARY}")

with open(Y_SUMMARY, "r", encoding="utf-8") as f:
    y_summary = json.load(f)

# Verify the three flagged CSVs by reading their complete contents from the ZIP.
bootstrap_audits = []
with ZipFile(BUNDLE, "r") as z:
    bad_member = z.testzip()
    if bad_member:
        raise RuntimeError(f"ZIP integrity failure: {bad_member}")

    all_members = [n for n in z.namelist() if not n.endswith("/")]
    csv_members = [n for n in all_members if n.lower().endswith(".csv")]
    parquet_members = [
        n for n in all_members if n.lower().endswith(".parquet")
    ]

    for name in sorted(EXPECTED_BOOTSTRAP_FILES):
        if name not in all_members:
            raise RuntimeError(f"Expected flagged file is missing: {name}")

        with z.open(name, "r") as binary:
            text = io.TextIOWrapper(
                binary, encoding="utf-8-sig", errors="replace", newline=""
            )
            reader = csv.DictReader(text)
            headers = reader.fieldnames or []
            rows = list(reader)

        model_labels = [
            str(row.get("Model", "")).strip()
            for row in rows
            if str(row.get("Model", "")).strip()
        ]

        normalized_headers = {
            "".join(ch.lower() for ch in header if ch.isalnum())
            for header in headers
        }

        # Stronger direct schema check for these three aggregate metric tables.
        required_summary_columns = {
            "model", "n", "dayblocks", "persistencemae", "modelmae",
            "persistencermse", "modelrmse"
        }
        is_aggregate_summary = required_summary_columns.issubset(
            normalized_headers
        )

        bootstrap_audits.append({
            "member": name,
            "data_rows": len(rows),
            "column_count": len(headers),
            "model_labels": model_labels,
            "aggregate_metric_schema_confirmed": is_aggregate_summary,
            "headers": headers,
        })

# Identify obvious raw-file names and raw-data directory paths.
known_raw_names = {
    "station_hour.csv", "stations.csv",
    "station_day.csv", "station.csv", "city_hour.csv", "city_day.csv"
}

obvious_raw_file_paths = [
    member for member in all_members
    if PurePosixPath(member).name.lower() in known_raw_names
    or "raw" in [part.lower() for part in PurePosixPath(member).parts]
]

flagged_summary_files_verified = (
    len(bootstrap_audits) == 3
    and all(
        item["data_rows"] == 3
        and item["aggregate_metric_schema_confirmed"]
        for item in bootstrap_audits
    )
)

# Preserve all historical artifacts by making a new output directory.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_07Z_content_audit_closure_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

closure = {
    "phase": "17.07Z",
    "created_at": datetime.now().astimezone().isoformat(),
    "archive_path": str(BUNDLE),
    "archive_sha256": actual_hash,
    "archive_hash_verified": actual_hash == EXPECTED_SHA256,
    "zip_integrity": "PASS",
    "archive_entry_count": len(all_members),
    "csv_count": len(csv_members),
    "parquet_count": len(parquet_members),
    "obvious_raw_file_paths": obvious_raw_file_paths,
    "phase17_07Y_csv_parse_errors": y_summary.get("parse_error_count"),
    "bootstrap_tables_directly_verified": bootstrap_audits,
    "all_three_flagged_tables_confirmed_as_aggregate_summaries":
        flagged_summary_files_verified,
    "content_audit_conclusion": (
        "No obvious raw source-data files identified by the recorded filename, "
        "schema, and bounded sample-value checks. This does not prove that all "
        "source-derived content is absent or that redistribution is permitted."
    ),
    "content_audit_status": (
        "PASS_WITH_LIMITATIONS"
        if flagged_summary_files_verified and not obvious_raw_file_paths
        else "MANUAL_REVIEW_REQUIRED"
    ),
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "existing_archive_modified": False,
    "models_retrained": False,
    "next_required_action": (
        "Responsible author must document the licence and upstream reuse-rights "
        "decision in the Phase 17.07U licence review log."
    ),
}

report_path = OUT / "phase17_07Z_content_audit_closure.json"
report_path.write_text(
    json.dumps(closure, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

lines = [
    "# Phase 17.07Z — Content Audit Closure",
    "",
    f"- Bundle SHA-256 verified: {closure['archive_hash_verified']}",
    "- ZIP integrity: PASS",
    f"- Archive entries: {len(all_members)}",
    f"- CSV files: {len(csv_members)}",
    f"- Parquet files: {len(parquet_members)}",
    f"- Obvious raw-file paths: {len(obvious_raw_file_paths)}",
    "",
    "## Flagged files reviewed",
]
for item in bootstrap_audits:
    lines.append(
        f"- `{item['member']}` — {item['data_rows']} data rows; "
        f"aggregate metric schema: "
        f"{item['aggregate_metric_schema_confirmed']}; "
        f"labels: {', '.join(item['model_labels']) or 'none'}"
    )

lines += [
    "",
    "## Content audit conclusion",
    "",
    closure["content_audit_conclusion"],
    "",
    "This concludes the current package-content screening. It is not a legal "
    "opinion or a guarantee that every derived value is free of third-party "
    "rights restrictions.",
    "",
    "## Release decision",
    "",
    "**Author licence review: OPEN. Submission ready: NO.**",
    "",
    "Production remains disabled. The original review ZIP, source data, and "
    "frozen models were not modified.",
]
md_path = OUT / "phase17_07Z_content_audit_closure.md"
md_path.write_text("\n".join(lines), encoding="utf-8")

print("=" * 78)
print("PHASE 17.07Z — CONTENT AUDIT CLOSURE")
print("=" * 78)
print("Archive SHA-256 verified:", closure["archive_hash_verified"])
print("ZIP integrity: PASS")
print("Archive entries:", len(all_members))
print("CSV files:", len(csv_members))
print("Parquet files:", len(parquet_members))
print("Obvious raw-file paths:", len(obvious_raw_file_paths))
print("Three flagged files verified as aggregate summaries:",
      flagged_summary_files_verified)
print("Content audit status:", closure["content_audit_status"])

print("\n[OUTPUTS]")
print("Markdown report:", md_path)
print("JSON report:", report_path)

print("\n[FINAL RELEASE GATE]")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data / existing archive / models modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.08 — LICENCE CLARIFICATION REQUEST PACK
# Creates draft requests for TUMI and CPCB, plus a response log.
# Does not send messages or modify data, models, or the frozen review ZIP.

from pathlib import Path
from datetime import datetime
import csv
import json

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

TUMI_URL = (
    "https://portal.transport-data.org/"
    "@tumi/urban-air-pollution-india-2015-2020"
)
KAGGLE_URL = (
    "https://www.kaggle.com/datasets/"
    "rohanrao/air-quality-data-in-india/data"
)
CPCB_QUERY_URL = "https://cpcb.gov.in/query-form.php"
OGD_TERMS_URL = "https://www.data.gov.in/terms-of-use"
GODL_URL = "https://www.data.gov.in/Godl"

timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT
    / "outputs"
    / "research_paper"
    / f"phase17_08_licence_clarification_{timestamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

# Draft 1: Send to the contact explicitly displayed on the TUMI dataset page.
tumi_email = """TO: info@transformative-mobility.org
SUBJECT: Clarification of CC0 1.0 and upstream rights — Urban air pollution measurements, India, 2015–2020

Dear TUMI Data Portal Team,

I am preparing an academic research paper using the public “Urban air pollution
measurements, India, 2015–2020” dataset listed in your catalogue:

https://portal.transport-data.org/@tumi/urban-air-pollution-india-2015-2020

The catalogue currently lists CC0 1.0 and describes the data as compiled by
Rohan Rao from data provided by India's Central Pollution Control Board (CPCB).
The linked distribution is:

https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india/data

Before finalising the paper and sharing its supporting materials, could you
please clarify the following?

1. Does the CC0 1.0 designation apply to the dataset's underlying air-quality
   records, the compilation/database, the catalogue metadata, or some
   combination of these?

2. Who applied the CC0 designation, and on what basis was it applied?
   Was authority to license the underlying CPCB-provided material verified?

3. Is there a source record, permission, licence notice, or written confirmation
   documenting the conditions under which this historical CPCB-derived data
   may be used in academic research, published in a paper, and redistributed?

4. Are there attribution, notice, provenance, or other conditions that should
   accompany publication of research results derived from the dataset?

5. If TUMI cannot confirm the rights to the underlying source records, could
   you direct me to the appropriate rights holder or CPCB office for
   clarification?

I have found my local station_hour.csv and stations.csv files to be byte-identical
to the corresponding files in the current public Kaggle distribution. I
understand that this establishes file identity, not the legal scope of a
licence. I am seeking clarification before making a release decision.

Please point me to the most authoritative available licence or permission
record, including its version or date if available.

Thank you for your assistance.

Regards,
[Your name]
[Institution or affiliation, if applicable]
"""

# Draft 2: Submit via CPCB's official query/feedback route, or ask to be
# forwarded to the department responsible for data rights/licensing.
cpcb_request = """SUBMISSION ROUTE: CPCB official Query/Feedback form
https://cpcb.gov.in/query-form.php

SUBJECT: Request for clarification of reuse and attribution conditions for historical CPCB air-quality data (2015–2020)

Dear Central Pollution Control Board,

I am preparing an academic research paper on one-hour-ahead AQI prediction
for selected Delhi monitoring stations, using a public historical dataset
covering 2015–2020.

The dataset is listed on the TUMI Transport Data Portal as “Urban air pollution
measurements, India, 2015–2020”:
https://portal.transport-data.org/@tumi/urban-air-pollution-india-2015-2020

The catalogue describes the collection as compiled by Rohan Rao from data
provided by CPCB and displays CC0 1.0. It links to:
https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india/data

Before finalising publication and sharing supporting materials, I request
clarification, or referral to the appropriate data custodian/legal/licensing
office, on the following points.

1. Did CPCB provide or authorise public distribution of the historical
   2015–2020 records represented in this Kaggle/TUMI collection?

2. What reuse and redistribution conditions apply to those historical records,
   including their use in academic research and publication of derived
   results or aggregate tables?

3. Is there a specific CPCB licence, open-data record, permission, notification,
   or other authoritative document that governs reuse of these records?

4. Does CPCB have a view on the CC0 1.0 designation displayed by the TUMI
   catalogue for this collection, particularly as it relates to source records
   provided by CPCB? If this is outside your office's remit, please direct me
   to the appropriate contact.

5. What source attribution, citation, disclaimer, or other conditions should
   accompany research derived from these data?

I am not asking CPCB to endorse the research or its results. I am seeking
clarification of the applicable data reuse conditions and the proper citation
of the underlying data source.

Thank you for your guidance.

Regards,
[Your name]
[Institution or affiliation, if applicable]
"""

# Save drafts and the source register.
(OUT / "TUMI_LICENCE_CLARIFICATION_EMAIL.md").write_text(
    tumi_email, encoding="utf-8"
)
(OUT / "CPCB_LICENCE_CLARIFICATION_REQUEST.md").write_text(
    cpcb_request, encoding="utf-8"
)

source_register = [
    {
        "source": "TUMI dataset record",
        "url": TUMI_URL,
        "currently_observed": (
            "Catalogue displays CC0 1.0, describes data as compiled from "
            "CPCB-provided data, and lists a contributor contact."
        ),
        "evidence_status": "Public catalogue statement; upstream rights unverified",
    },
    {
        "source": "Kaggle dataset",
        "url": KAGGLE_URL,
        "currently_observed": (
            "Current public distribution previously byte-matched the project's "
            "local station_hour.csv and stations.csv files."
        ),
        "evidence_status": (
            "File identity confirmed in Phase 17.07R2; this is not licence clearance"
        ),
    },
    {
        "source": "CPCB Query/Feedback",
        "url": CPCB_QUERY_URL,
        "currently_observed": (
            "Official CPCB query/feedback route for submitting the clarification request."
        ),
        "evidence_status": "Contact route; no response received",
    },
    {
        "source": "OGD terms of use",
        "url": OGD_TERMS_URL,
        "currently_observed": (
            "Terms describe dataset-specific licence metadata and caution that "
            "third-party copyrighted content may require separate permission."
        ),
        "evidence_status": "General portal terms, not a dataset-specific licence determination",
    },
    {
        "source": "Government Open Data License India",
        "url": GODL_URL,
        "currently_observed": (
            "General government open-data licence terms; applicability to this "
            "specific collection has not been established by this project."
        ),
        "evidence_status": "General licence text, not confirmation of source rights",
    },
]

with open(OUT / "PHASE_17_08_SOURCE_REGISTER.csv",
          "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=source_register[0].keys())
    writer.writeheader()
    writer.writerows(source_register)

# Response tracking log: leave blank until a real response has been received.
response_fields = [
    "source", "request_date", "request_submitted", "contact_route",
    "response_date", "respondent_or_department", "response_summary",
    "evidence_file_or_url", "reuse_permission_confirmed",
    "redistribution_permission_confirmed", "attribution_requirements",
    "follow_up_needed", "reviewer_decision", "reviewer_sign_off_date"
]

with open(OUT / "PHASE_17_08_RESPONSE_TRACKER.csv",
          "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=response_fields)
    writer.writeheader()
    writer.writerow({
        "source": "TUMI",
        "contact_route": "info@transformative-mobility.org",
        "request_submitted": "NO — draft prepared only",
    })
    writer.writerow({
        "source": "CPCB",
        "contact_route": CPCB_QUERY_URL,
        "request_submitted": "NO — draft prepared only",
    })

status = {
    "phase": "17.08",
    "created_at": datetime.now().astimezone().isoformat(),
    "drafts_created": True,
    "requests_sent": False,
    "responses_received": False,
    "source_data_modified": False,
    "review_bundle_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "required_next_action": (
        "Author reviews and submits the requests, records real responses, "
        "and documents a supported licence decision."
    ),
    "outputs": {
        "tumi_draft": str(OUT / "TUMI_LICENCE_CLARIFICATION_EMAIL.md"),
        "cpcb_draft": str(OUT / "CPCB_LICENCE_CLARIFICATION_REQUEST.md"),
        "source_register": str(OUT / "PHASE_17_08_SOURCE_REGISTER.csv"),
        "response_tracker": str(OUT / "PHASE_17_08_RESPONSE_TRACKER.csv"),
    },
}

(OUT / "phase17_08_status.json").write_text(
    json.dumps(status, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

readme = f"""# Phase 17.08 — Licence Clarification Request Pack

Created: {status["created_at"]}

## Files
- TUMI email draft: `TUMI_LICENCE_CLARIFICATION_EMAIL.md`
- CPCB request draft: `CPCB_LICENCE_CLARIFICATION_REQUEST.md`
- Evidence/source register: `PHASE_17_08_SOURCE_REGISTER.csv`
- Response tracker: `PHASE_17_08_RESPONSE_TRACKER.csv`
- Status: `phase17_08_status.json`

## Instructions
1. Review and personalise the drafts before submitting.
2. Submit the TUMI email to the displayed contributor contact.
3. Submit the CPCB request through its official query/feedback route, or request
   referral to the relevant data custodian.
4. Preserve actual submissions and replies; never fill the response tracker
   with assumed or expected answers.
5. Enter a final release decision only after reviewing the evidence.

Requests have not been sent. No reply is implied or assumed.
Submission readiness remains FALSE; production remains disabled.
"""
(OUT / "README.md").write_text(readme, encoding="utf-8")

print("=" * 78)
print("PHASE 17.08 — LICENCE CLARIFICATION REQUEST PACK")
print("=" * 78)
print("Output folder:", OUT)
print("TUMI draft:", OUT / "TUMI_LICENCE_CLARIFICATION_EMAIL.md")
print("CPCB request:", OUT / "CPCB_LICENCE_CLARIFICATION_REQUEST.md")
print("Source register:", OUT / "PHASE_17_08_SOURCE_REGISTER.csv")
print("Response tracker:", OUT / "PHASE_17_08_RESPONSE_TRACKER.csv")
print("\nRequests sent: False")
print("Responses received: False")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data / bundle / models modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.09 — COMPLETE PAPER ASSET INVENTORY
# Read-only inspection of project assets; writes reports to a new folder.

from pathlib import Path
from datetime import datetime
from zipfile import ZipFile
import os
import re
import csv
import json
import hashlib
import mimetypes

# Mount Drive only if its project path is not already available.
ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
if not ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {ROOT}")

BUNDLE = (
    ROOT / "outputs/research_paper"
    / "phase17_07T_review_bundle_20261025T181246Z"
    / "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)
EXPECTED_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

# Primary locations for research outputs and saved publication materials.
SCAN_DIRS = [
    ROOT / "outputs/research_paper",
    ROOT / "outputs/reports",
    ROOT / "reports",
    ROOT / "notebooks",
]

GRAPH_EXTS = {
    ".png", ".jpg", ".jpeg", ".svg", ".tif", ".tiff",
    ".webp", ".gif", ".bmp", ".eps"
}
PAPER_EXTS = {
    ".md", ".tex", ".docx", ".pdf", ".rtf",
    ".pptx", ".ipynb", ".html"
}
TABLE_EXTS = {".csv", ".xlsx", ".xls"}
DATA_EXTS = {".parquet", ".feather", ".pkl"}
META_EXTS = {".json", ".txt", ".yaml", ".yml"}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def classify_file(path):
    ext = path.suffix.lower()
    if ext in GRAPH_EXTS:
        return "image_or_graph"
    if ext in PAPER_EXTS:
        return "manuscript_or_document"
    if ext in TABLE_EXTS:
        return "table_or_spreadsheet"
    if ext in DATA_EXTS:
        return "data_artifact"
    if ext in META_EXTS:
        return "metadata_or_text"
    if ext == ".zip":
        return "archive"
    return "other"

# 1. Verify the bundle before inspecting it.
if not BUNDLE.is_file():
    raise FileNotFoundError(
        f"Review bundle not found: {BUNDLE}"
    )

bundle_hash = sha256_file(BUNDLE)
if bundle_hash != EXPECTED_SHA256:
    raise RuntimeError(
        "Review bundle SHA-256 mismatch. Audit stopped."
    )

with ZipFile(BUNDLE, "r") as archive:
    bad_member = archive.testzip()
    if bad_member:
        raise RuntimeError(f"ZIP integrity failure: {bad_member}")

    zip_members = [
        name for name in archive.namelist()
        if not name.endswith("/")
    ]

    zip_extension_counts = {}
    for name in zip_members:
        ext = Path(name).suffix.lower() or "[no extension]"
        zip_extension_counts[ext] = zip_extension_counts.get(ext, 0) + 1

    zip_graphs = [
        name for name in zip_members
        if Path(name).suffix.lower() in GRAPH_EXTS
    ]

    zip_manuscripts = [
        name for name in zip_members
        if "manuscript" in Path(name).name.lower()
        and Path(name).suffix.lower() in {".md", ".tex", ".docx", ".pdf"}
    ]

    expected_tables = [
        "results/phase17_07J_tables/table_1_validation_model_comparison.csv",
        "results/phase17_07J_tables/table_2_holdout_matched_support.csv",
        "results/phase17_07J_tables/table_3_holdout_full_available_support.csv",
        "results/phase17_07J_tables/table_4_day_block_bootstrap.csv",
        "results/phase17_07J_tables/table_5_performance_by_aqi_band.csv",
        "results/phase17_07J_tables/table_6_abrupt_transition_detection.csv",
        "results/phase17_07J_tables/table_7_feature_attribution.csv",
        "results/phase17_07J_tables/table_8_station_performance.csv",
        "results/phase17_07J_tables/table_9_season_performance.csv",
    ]
    missing_main_tables = [
        name for name in expected_tables if name not in zip_members
    ]

# 2. Inventory publication assets in the main project output directories.
inventory = []
scan_warnings = []

for scan_dir in SCAN_DIRS:
    if not scan_dir.exists():
        scan_warnings.append(f"Directory not found: {scan_dir}")
        continue

    for current, subdirs, filenames in os.walk(scan_dir):
        # Ignore temporary/runtime folders.
        subdirs[:] = [
            d for d in subdirs
            if d not in {".ipynb_checkpoints", "__pycache__", ".git"}
        ]

        for filename in filenames:
            path = Path(current) / filename
            try:
                stat = path.stat()
                rel = path.relative_to(ROOT).as_posix()
                inventory.append({
                    "relative_path": rel,
                    "extension": path.suffix.lower(),
                    "category": classify_file(path),
                    "bytes": stat.st_size,
                    "modified": datetime.fromtimestamp(
                        stat.st_mtime
                    ).isoformat(timespec="seconds"),
                })
            except (OSError, ValueError) as exc:
                scan_warnings.append(
                    f"Could not inspect {path}: {type(exc).__name__}"
                )

# Deduplicate paths in case configured directories overlap.
inventory_by_path = {row["relative_path"]: row for row in inventory}
inventory = sorted(
    inventory_by_path.values(),
    key=lambda r: r["relative_path"].lower()
)

images = [
    row for row in inventory
    if row["category"] == "image_or_graph"
]
tables = [
    row for row in inventory
    if row["category"] == "table_or_spreadsheet"
]
manuscripts = [
    row for row in inventory
    if "manuscript" in Path(row["relative_path"]).name.lower()
    and Path(row["relative_path"]).suffix.lower()
       in {".md", ".tex", ".docx", ".pdf"}
]
notebooks = [
    row for row in inventory
    if Path(row["relative_path"]).suffix.lower() == ".ipynb"
]

# 3. Read text from Markdown manuscripts to look for figure/table references.
manuscript_checks = []
image_reference_pattern = re.compile(
    r"!\[[^\]]*\]\(([^)]+)\)|<img\b[^>]*?\bsrc=[\"']([^\"']+)",
    re.IGNORECASE
)
figure_pattern = re.compile(r"\b(?:figure|fig\.?)\s*\d+\b", re.IGNORECASE)
table_pattern = re.compile(r"\btable\s*\d+\b", re.IGNORECASE)

for row in manuscripts:
    path = ROOT / row["relative_path"]
    if path.suffix.lower() != ".md":
        continue

    try:
        text = path.read_text(encoding="utf-8", errors="replace")
    except OSError as exc:
        scan_warnings.append(
            f"Could not read manuscript {path}: {type(exc).__name__}"
        )
        continue

    refs = []
    for match in image_reference_pattern.finditer(text):
        target = (match.group(1) or match.group(2) or "").strip()
        if not target:
            continue

        # Leave web/data links alone; resolve local image references.
        if re.match(r"^(https?:|data:|#)", target, re.IGNORECASE):
            refs.append({
                "target": target,
                "kind": "external_or_anchor",
                "exists": None,
            })
            continue

        target_path = target.split("#", 1)[0].split("?", 1)[0]
        candidates = [
            path.parent / target_path,
            ROOT / target_path,
        ]
        exists = any(candidate.is_file() for candidate in candidates)
        refs.append({
            "target": target,
            "kind": "local",
            "exists": exists,
        })

    figure_refs = sorted(set(
        match.lower()
        for match in figure_pattern.findall(text)
    ))
    table_refs = sorted(set(
        match.lower()
        for match in table_pattern.findall(text)
    ))
    markdown_table_rows = sum(
        1 for line in text.splitlines()
        if line.strip().startswith("|")
    )

    manuscript_checks.append({
        "path": row["relative_path"],
        "characters": len(text),
        "image_references": refs,
        "image_reference_count": len(refs),
        "broken_local_image_references": [
            ref["target"] for ref in refs
            if ref["kind"] == "local" and not ref["exists"]
        ],
        "numbered_figure_reference_count": len(figure_refs),
        "numbered_figure_references": figure_refs,
        "numbered_table_reference_count": len(table_refs),
        "numbered_table_references": table_refs,
        "markdown_table_rows": markdown_table_rows,
    })

# 4. Detect plots rendered inside notebooks even if no standalone image file exists.
notebook_visuals = []
for row in notebooks:
    path = ROOT / row["relative_path"]
    try:
        with open(path, "r", encoding="utf-8") as f:
            nb = json.load(f)

        image_output_count = 0
        cells_with_images = 0
        for cell in nb.get("cells", []):
            cell_has_image = False
            for output in cell.get("outputs", []):
                data = output.get("data", {})
                if any(
                    key in data
                    for key in ("image/png", "image/jpeg", "image/svg+xml")
                ):
                    image_output_count += 1
                    cell_has_image = True
            if cell_has_image:
                cells_with_images += 1

        notebook_visuals.append({
            "path": row["relative_path"],
            "image_outputs": image_output_count,
            "cells_with_images": cells_with_images,
        })
    except Exception as exc:
        notebook_visuals.append({
            "path": row["relative_path"],
            "error": f"{type(exc).__name__}: {exc}",
        })

# 5. Write a new audit folder without changing any existing project file.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_09_asset_audit_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

inventory_csv = OUT / "phase17_09_project_asset_inventory.csv"
with open(inventory_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "relative_path", "extension", "category", "bytes", "modified"
        ]
    )
    writer.writeheader()
    writer.writerows(inventory)

zip_manifest_csv = OUT / "phase17_09_review_bundle_manifest.csv"
with open(zip_manifest_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["archive_member", "extension", "category"])
    for name in zip_members:
        ext = Path(name).suffix.lower() or "[no extension]"
        category = (
            "image_or_graph" if ext in GRAPH_EXTS else
            "manuscript_or_document" if ext in PAPER_EXTS else
            "table_or_spreadsheet" if ext in TABLE_EXTS else
            "data_artifact" if ext in DATA_EXTS else
            "metadata_or_text" if ext in META_EXTS else
            "archive" if ext == ".zip" else "other"
        )
        writer.writerow([name, ext, category])

summary = {
    "phase": "17.09",
    "created_at": datetime.now().astimezone().isoformat(),
    "bundle_path": str(BUNDLE),
    "bundle_sha256": bundle_hash,
    "bundle_integrity": "PASS",
    "bundle_entry_count": len(zip_members),
    "bundle_extension_counts": zip_extension_counts,
    "bundle_graph_image_count": len(zip_graphs),
    "bundle_graph_images": zip_graphs,
    "bundle_manuscript_files": zip_manuscripts,
    "expected_nine_tables": expected_tables,
    "missing_expected_main_tables": missing_main_tables,
    "project_scanned_directories": [
        str(path) for path in SCAN_DIRS if path.exists()
    ],
    "project_inventory_count": len(inventory),
    "saved_graph_image_count_in_scanned_directories": len(images),
    "saved_graph_images": [
        row["relative_path"] for row in images
    ],
    "table_or_spreadsheet_file_count": len(tables),
    "manuscript_files": [row["relative_path"] for row in manuscripts],
    "manuscript_checks": manuscript_checks,
    "notebook_visual_outputs": notebook_visuals,
    "scan_warnings": scan_warnings,
    "production_enabled": False,
    "frozen_models_retrained": False,
    "source_data_modified": False,
    "existing_review_bundle_modified": False,
    "release_gate": "LICENCE REVIEW STILL OPEN",
}

summary_json = OUT / "phase17_09_asset_audit_summary.json"
summary_json.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

report_lines = [
    "# Phase 17.09 — Complete Paper Asset Audit",
    "",
    f"- Bundle hash verified: {bundle_hash == EXPECTED_SHA256}",
    "- ZIP integrity: PASS",
    f"- Bundle entries: {len(zip_members)}",
    f"- Images/graphs inside bundle: {len(zip_graphs)}",
    f"- CSV entries inside bundle: {zip_extension_counts.get('.csv', 0)}",
    f"- Expected main tables missing: {len(missing_main_tables)}",
    f"- Standalone images/graphs found in scanned directories: {len(images)}",
    f"- Markdown manuscript files found: {len(manuscripts)}",
    "",
    "## Archive file types",
]
for ext, count in sorted(zip_extension_counts.items()):
    report_lines.append(f"- `{ext}`: {count}")

report_lines += ["", "## Saved image/graph files"]
if images:
    report_lines.extend(f"- `{row['relative_path']}`" for row in images)
else:
    report_lines.append("- No standalone image assets found in scanned directories.")

report_lines += ["", "## Expected main tables"]
for table_name in expected_tables:
    status = "PRESENT" if table_name in zip_members else "MISSING"
    report_lines.append(f"- {status}: `{table_name}`")

report_lines += ["", "## Manuscript image references"]
if manuscript_checks:
    for item in manuscript_checks:
        report_lines.append(f"- `{item['path']}`")
        report_lines.append(
            f"  - Embedded image references: {item['image_reference_count']}"
        )
        report_lines.append(
            f"  - Numbered Figure references: "
            f"{item['numbered_figure_reference_count']}"
        )
        report_lines.append(
            f"  - Numbered Table references: "
            f"{item['numbered_table_reference_count']}"
        )
        if item["broken_local_image_references"]:
            report_lines.append(
                "  - Broken local image paths: "
                + ", ".join(item["broken_local_image_references"])
            )
else:
    report_lines.append("- No Markdown manuscript was detected in the scanned directories.")

report_lines += [
    "",
    "## Interpretation",
    "",
    "An image saved somewhere in the project is not automatically included in "
    "the review ZIP or embedded in the manuscript. Table CSVs are source tables, "
    "not proof that the formatted numbered tables appear in the final paper.",
    "",
    "This inventory does not modify the dataset, frozen models, or existing ZIP.",
    "The author licence review remains open.",
]
report_md = OUT / "phase17_09_asset_audit_report.md"
report_md.write_text("\n".join(report_lines), encoding="utf-8")

print("=" * 78)
print("PHASE 17.09 — COMPLETE PAPER ASSET AUDIT")
print("=" * 78)
print("Bundle SHA-256 verified:", bundle_hash == EXPECTED_SHA256)
print("ZIP integrity: PASS")
print("Bundle entries:", len(zip_members))
print("CSV entries in bundle:", zip_extension_counts.get(".csv", 0))
print("Graph/image files inside bundle:", len(zip_graphs))
print("Main tables missing:", len(missing_main_tables))
print("Standalone graphs/images found:", len(images))
print("Markdown manuscripts found:", len(manuscripts))

print("\n[BUNDLE GRAPH/IMAGE FILES]")
for name in zip_graphs:
    print("-", name)
if not zip_graphs:
    print("No graph/image file was found in the review bundle.")

print("\n[MAIN TABLE CHECK]")
for name in expected_tables:
    print(("OK  " if name in zip_members else "MISSING"), name)

print("\n[SCANNED GRAPH/IMAGE ASSETS]")
for row in images:
    print(f"- {row['relative_path']} ({row['bytes']:,} bytes)")
if not images:
    print("No standalone graph/image assets found in scanned directories.")

print("\n[MANUSCRIPTS]")
for item in manuscript_checks:
    print("-", item["path"])
    print("  embedded image refs:", item["image_reference_count"])
    print("  numbered Figure refs:", item["numbered_figure_reference_count"])
    print("  numbered Table refs:", item["numbered_table_reference_count"])
    if item["broken_local_image_references"]:
        print("  BROKEN IMAGE PATHS:", item["broken_local_image_references"])

print("\n[NOTEBOOK IMAGE OUTPUTS]")
for item in notebook_visuals:
    if item.get("image_outputs", 0):
        print(
            f"- {item['path']}: {item['image_outputs']} image outputs "
            f"across {item['cells_with_images']} cells"
        )

print("\n[REPORTS]")
print("Inventory:", inventory_csv)
print("Bundle manifest:", zip_manifest_csv)
print("Audit report:", report_md)
print("Summary JSON:", summary_json)

print("\n[SAFETY]")
print("Existing ZIP modified: False")
print("Source data modified: False")
print("Frozen models retrained: False")
print("Production enabled: False")
print("Licence review: OPEN")
print("=" * 78)

In [ ]:
# PHASE 17.10A — PUBLICATION FIGURE PACK
# Creates candidate figures from the frozen holdout outputs.
# Does not retrain models or modify the existing review bundle.

from pathlib import Path
from zipfile import ZipFile
from datetime import datetime
import hashlib
import json
import io
import csv
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from sklearn.metrics import mean_absolute_error, mean_squared_error

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

PREDICTIONS = (
    ROOT / "outputs/reports/phase15_12_final_holdout"
    / "phase15_12_final_holdout_predictions.parquet"
)

BUNDLE = (
    ROOT / "outputs/research_paper"
    / "phase17_07T_review_bundle_20261025T181246Z"
    / "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)

EXPECTED_BUNDLE_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

IMPORTANCE_MEMBER = (
    "results/phase17_07I_tables/"
    "ensemble_permutation_importance_top20.csv"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# 1. Verify the existing evidence and load frozen predictions.
if not PREDICTIONS.is_file():
    raise FileNotFoundError(
        f"Final holdout predictions were not found: {PREDICTIONS}"
    )

if not BUNDLE.is_file():
    raise FileNotFoundError(f"Review bundle not found: {BUNDLE}")

bundle_hash = sha256_file(BUNDLE)
if bundle_hash != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(
        "Existing bundle SHA-256 mismatch; figure generation stopped."
    )

with ZipFile(BUNDLE, "r") as z:
    if z.testzip() is not None:
        raise RuntimeError("Existing review ZIP failed its integrity test.")
    if IMPORTANCE_MEMBER not in z.namelist():
        raise FileNotFoundError(
            f"Importance table missing from review ZIP: {IMPORTANCE_MEMBER}"
        )
    importance_df = pd.read_csv(io.BytesIO(z.read(IMPORTANCE_MEMBER)))

df = pd.read_parquet(PREDICTIONS)

required = [
    "Datetime",
    "StationId",
    "AQI_t_plus_1",
    "Tuned_Extra_Trees_Prediction",
    "Tuned_Random_Forest_Prediction",
    "Final_Ensemble_Prediction",
    "Persistence_Prediction",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required prediction columns missing: {missing}")

df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")

target = "AQI_t_plus_1"
ensemble = "Final_Ensemble_Prediction"
et = "Tuned_Extra_Trees_Prediction"
rf = "Tuned_Random_Forest_Prediction"
persistence = "Persistence_Prediction"

# All four model comparisons use exactly the same rows.
matched = df.dropna(
    subset=[target, ensemble, et, rf, persistence]
).copy()

full_ensemble = df.dropna(subset=[target, ensemble]).copy()

if len(df) != 68352:
    warnings.warn(
        f"Expected 68,352 holdout rows from the frozen audit; found {len(df)}."
    )

if len(matched) != 68285:
    warnings.warn(
        f"Expected 68,285 common-support rows; found {len(matched)}. "
        "Figures will use the support actually available in this file."
    )

# Create a new folder to preserve all prior artifacts.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10A_publication_figures_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

FIG_DIR = OUT / "figures"
FIG_DIR.mkdir()

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})

register = []
metric_rows = []

def save_figure(fig, number, slug, title, caption, support, source):
    png_path = FIG_DIR / f"Figure_{number:02d}_{slug}.png"
    pdf_path = FIG_DIR / f"Figure_{number:02d}_{slug}.pdf"

    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    register.append({
        "figure_number": number,
        "title": title,
        "caption": caption,
        "png_path": str(png_path),
        "pdf_path": str(pdf_path),
        "support": support,
        "source": source,
        "png_sha256": sha256_file(png_path),
        "pdf_sha256": sha256_file(pdf_path),
    })

# Figure 1 — Study workflow schematic.
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

boxes = [
    (0.015, "Hourly Delhi data\n10 stations"),
    (0.215, "Feature engineering\n73-feature contract"),
    (0.415, "Chronological split\nTrain: 166,845\nValidation: 69,097\nGap: 19\nHoldout: 68,352"),
    (0.615, "Validation-based tuning\nRefit components on\nTRAIN + VALID"),
    (0.815, "Frozen ensemble\nExtra Trees: 0.594466\nRandom Forest: 0.405534\nPredict AQI at t+1"),
]

for x, text in boxes:
    box = FancyBboxPatch(
        (x, 0.32), 0.17, 0.43,
        boxstyle="round,pad=0.012",
        linewidth=1.2,
        edgecolor="#385777",
        facecolor="#edf3f8"
    )
    ax.add_patch(box)
    ax.text(
        x + 0.085, 0.535, text,
        ha="center", va="center", fontsize=9
    )

for i in range(len(boxes) - 1):
    x1 = boxes[i][0] + 0.17
    x2 = boxes[i + 1][0]
    ax.add_patch(FancyArrowPatch(
        (x1 + 0.002, 0.535), (x2 - 0.006, 0.535),
        arrowstyle="->", mutation_scale=12, linewidth=1.2,
        color="#385777"
    ))

fig.suptitle("Study workflow and temporal evaluation", y=0.94, fontsize=14)
save_figure(
    fig, 1, "study_workflow",
    "Study workflow and temporal evaluation",
    "Workflow of the Delhi one-hour-ahead AQI study, showing feature contract, "
    "chronological split, validation-based model tuning and frozen ensemble. "
    "The final holdout is reserved for evaluation.",
    "Project design and recorded split assignments",
    "Frozen project configuration and Phase 15 split/model records"
)

# Figure 2 — Actual versus ensemble prediction.
actual = full_ensemble[target].to_numpy(dtype=float)
pred = full_ensemble[ensemble].to_numpy(dtype=float)
mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))

scatter_df = full_ensemble
if len(scatter_df) > 30000:
    scatter_df = scatter_df.sample(n=30000, random_state=17)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(
    scatter_df[target], scatter_df[ensemble],
    s=7, alpha=0.2, rasterized=True
)
low = min(full_ensemble[target].min(), full_ensemble[ensemble].min())
high = max(full_ensemble[target].max(), full_ensemble[ensemble].max())
ax.plot([low, high], [low, high], linestyle="--", linewidth=1.4, color="black")
ax.set_xlabel("Observed next-hour AQI")
ax.set_ylabel("Predicted next-hour AQI")
ax.set_title("Observed versus predicted AQI on the final holdout")
ax.text(
    0.04, 0.96,
    f"N = {len(full_ensemble):,}\nMAE = {mae:.3f}\nRMSE = {rmse:.3f}",
    transform=ax.transAxes, va="top",
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.9}
)
ax.grid(alpha=0.2)
save_figure(
    fig, 2, "observed_vs_predicted",
    "Observed versus predicted AQI",
    "Observed versus frozen ensemble predictions for AQI at t+1 on the "
    "untouched final holdout. The dashed line indicates perfect agreement. "
    f"MAE={mae:.3f}; RMSE={rmse:.3f}; N={len(full_ensemble):,}.",
    f"{len(full_ensemble):,} rows with observed target and ensemble prediction",
    "Frozen Phase 15.12 final holdout predictions"
)

# Figure 3 — Ensemble residual distribution.
residual = (
    full_ensemble[target].to_numpy(dtype=float)
    - full_ensemble[ensemble].to_numpy(dtype=float)
)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.hist(residual, bins=80, alpha=0.85, edgecolor="white")
ax.axvline(0, color="black", linestyle="--", linewidth=1.3)
ax.axvline(
    np.mean(residual), color="darkred", linestyle="-", linewidth=1.5,
    label=f"Mean residual = {np.mean(residual):.3f}"
)
ax.set_xlabel("Residual: observed AQI − predicted AQI")
ax.set_ylabel("Number of predictions")
ax.set_title("Final-holdout ensemble residual distribution")
ax.legend()
ax.grid(axis="y", alpha=0.2)
save_figure(
    fig, 3, "ensemble_residual_distribution",
    "Ensemble residual distribution",
    "Distribution of final-holdout residuals, defined as observed minus "
    "predicted next-hour AQI. Positive residuals indicate underprediction.",
    f"{len(full_ensemble):,} rows with observed target and ensemble prediction",
    "Frozen Phase 15.12 final holdout predictions"
)

# Figure 4 — MAE and RMSE on common support.
model_columns = [
    ("Persistence", persistence),
    ("Extra Trees", et),
    ("Random Forest", rf),
    ("Ensemble", ensemble),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
names = [x[0] for x in model_columns]
maes, rmses = [], []

for label, col in model_columns:
    y = matched[target].to_numpy(dtype=float)
    p = matched[col].to_numpy(dtype=float)
    m = mean_absolute_error(y, p)
    r = np.sqrt(mean_squared_error(y, p))
    maes.append(m)
    rmses.append(r)
    metric_rows.append({
        "model": label,
        "n_common_support": len(matched),
        "mae": m,
        "rmse": r,
    })

for ax, values, ylabel, title in [
    (axes[0], maes, "MAE (AQI points)", "Mean absolute error"),
    (axes[1], rmses, "RMSE (AQI points)", "Root mean squared error"),
]:
    bars = ax.bar(names, values)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.2)
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center", va="bottom", fontsize=9
        )

fig.suptitle("Model comparison on matched final-holdout support", fontsize=13)
fig.tight_layout()
save_figure(
    fig, 4, "matched_support_model_comparison",
    "Matched-support model comparison",
    "MAE and RMSE for persistence, tuned Extra Trees, tuned Random Forest "
    "and the frozen weighted ensemble. Every model is evaluated on the same "
    f"{len(matched):,} holdout rows.",
    f"{len(matched):,} rows with valid target and predictions from all four models",
    "Frozen Phase 15.12 final holdout predictions"
)

# Figure 5 — Ensemble versus persistence MAE by target AQI band.
band_labels = ["0–50", "51–100", "101–200", "201–300", "301–400", ">400"]
matched["AQI_band"] = pd.cut(
    matched[target],
    bins=[-np.inf, 50, 100, 200, 300, 400, np.inf],
    labels=band_labels,
    right=True,
    include_lowest=True
)

band_records = []
for band in band_labels:
    part = matched[matched["AQI_band"] == band]
    if len(part) == 0:
        continue
    band_records.append({
        "band": band,
        "n": len(part),
        "ensemble_mae": mean_absolute_error(
            part[target], part[ensemble]
        ),
        "persistence_mae": mean_absolute_error(
            part[target], part[persistence]
        ),
    })

band_df = pd.DataFrame(band_records)
fig, ax = plt.subplots(figsize=(9.5, 5.5))
x = np.arange(len(band_df))
width = 0.36
b1 = ax.bar(
    x - width/2, band_df["persistence_mae"],
    width, label="Persistence"
)
b2 = ax.bar(
    x + width/2, band_df["ensemble_mae"],
    width, label="Weighted ensemble"
)

ax.set_xticks(x)
ax.set_xticklabels([
    f"{row.band}\n(n={row.n:,})"
    for row in band_df.itertuples()
])
ax.set_ylabel("MAE (AQI points)")
ax.set_xlabel("Observed next-hour AQI band")
ax.set_title("Prediction error by next-hour AQI band")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(
    fig, 5, "mae_by_aqi_band",
    "Prediction error by AQI band",
    "Ensemble and persistence MAE stratified by observed next-hour AQI band. "
    "Both models use common support within each band; sample size is shown "
    "under each category.",
    f"{len(matched):,} common-support rows before band stratification",
    "Frozen Phase 15.12 final holdout predictions"
)

# Figure 6 — Station-level MAE.
station_rows = []
for station, part in matched.groupby("StationId", dropna=False):
    station_rows.append({
        "station": str(station),
        "n": len(part),
        "ensemble_mae": mean_absolute_error(
            part[target], part[ensemble]
        ),
        "persistence_mae": mean_absolute_error(
            part[target], part[persistence]
        ),
    })

station_df = pd.DataFrame(station_rows).sort_values(
    "ensemble_mae", ascending=True
)

fig, ax = plt.subplots(figsize=(10, 6.5))
ypos = np.arange(len(station_df))
height = 0.36
ax.barh(
    ypos - height/2, station_df["persistence_mae"],
    height, label="Persistence"
)
ax.barh(
    ypos + height/2, station_df["ensemble_mae"],
    height, label="Weighted ensemble"
)
ax.set_yticks(ypos)
ax.set_yticklabels(station_df["station"])
ax.set_xlabel("MAE (AQI points)")
ax.set_ylabel("Station ID")
ax.set_title("Station-level prediction error")
ax.legend()
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
save_figure(
    fig, 6, "station_level_mae",
    "Station-level prediction error",
    "MAE for persistence and the frozen ensemble at each Delhi monitoring "
    "station. Stations are ordered by ensemble MAE; all comparisons use "
    "common-support rows within each station.",
    f"{len(matched):,} common-support rows across {station_df.shape[0]} stations",
    "Frozen Phase 15.12 final holdout predictions"
)

# Figure 7 — Feature importance, only if the saved table is interpretable.
def norm(s):
    return "".join(ch.lower() for ch in str(s) if ch.isalnum())

feature_col = next(
    (c for c in importance_df.columns if norm(c) in {"feature", "featurename"}),
    None
)

importance_cols = [
    c for c in importance_df.columns
    if "importance" in norm(c) and pd.api.types.is_numeric_dtype(importance_df[c])
]
preferred = [
    c for c in importance_cols
    if any(token in norm(c) for token in ["percent", "pct", "share"])
]
importance_col = (preferred or importance_cols or [None])[0]

importance_created = False
if feature_col is not None and importance_col is not None:
    imp = importance_df[[feature_col, importance_col]].copy()
    imp[importance_col] = pd.to_numeric(imp[importance_col], errors="coerce")
    imp = imp.dropna().sort_values(importance_col, ascending=False).head(10)
    if not imp.empty:
        imp = imp.sort_values(importance_col, ascending=True)
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(imp[feature_col].astype(str), imp[importance_col])
        ax.set_xlabel(str(importance_col))
        ax.set_ylabel("Feature")
        ax.set_title("Top features by saved permutation-importance measure")
        ax.grid(axis="x", alpha=0.2)
        fig.tight_layout()
        save_figure(
            fig, 7, "top_permutation_importance",
            "Top features by permutation importance",
            "Top features ranked by the saved permutation-importance measure. "
            "Importance indicates predictive reliance under the recorded "
            "evaluation procedure; it does not establish causal effects.",
            "Saved importance table; not recomputed during this phase",
            f"Review bundle member: {IMPORTANCE_MEMBER}"
        )
        importance_created = True
else:
    warnings.warn(
        "Feature importance plot was skipped because a feature column and "
        "numeric importance column could not be identified in the saved table."
    )

# Save the figure register and numeric metric tables.
register_df = pd.DataFrame(register)
register_df.to_csv(OUT / "figure_register.csv", index=False)

pd.DataFrame(metric_rows).to_csv(
    OUT / "matched_support_metrics_used.csv", index=False
)
band_df.to_csv(OUT / "metrics_by_aqi_band.csv", index=False)
station_df.to_csv(OUT / "metrics_by_station.csv", index=False)

readme = f"""# Phase 17.10A — Candidate Publication Figure Pack

Generated: {datetime.now().astimezone().isoformat()}

## Contents
- Each generated figure has a high-resolution PNG and a vector PDF.
- `figure_register.csv` contains titles, draft captions, data support, and hashes.
- `matched_support_metrics_used.csv` records common-support model metrics.
- `metrics_by_aqi_band.csv` records subgroup values plotted in Figure 5.
- `metrics_by_station.csv` records station values plotted in Figure 6.

## Data provenance
- Holdout predictions: `{PREDICTIONS}`
- Holdout predictions were read only.
- Model fitting, tuning, and refitting were not performed.
- Feature importance was read from `{IMPORTANCE_MEMBER}` inside the verified bundle.

## Important
These are candidate publication figures. Review axes, captions, subgroup labels,
and consistency with the manuscript before assigning final figure numbers.
The figures have not yet been embedded into or referenced by the manuscript.
The original review ZIP and frozen models were not modified.

## Status
- Candidate figures created: {len(register)}
- Feature-importance figure created: {importance_created}
- Licence review: OPEN
- Submission-ready status: FALSE
"""

(OUT / "README.md").write_text(readme, encoding="utf-8")

summary = {
    "phase": "17.10A",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_predictions": str(PREDICTIONS),
    "review_bundle_sha256_verified": bundle_hash == EXPECTED_BUNDLE_SHA256,
    "full_ensemble_support_n": len(full_ensemble),
    "matched_support_n": len(matched),
    "station_count": int(matched["StationId"].nunique()),
    "figure_count": len(register),
    "figure_register": str(OUT / "figure_register.csv"),
    "figures_directory": str(FIG_DIR),
    "original_review_bundle_modified": False,
    "source_data_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "author_licence_review": "OPEN",
    "manuscript_integration_complete": False,
    "submission_ready": False,
}

(OUT / "phase17_10A_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 78)
print("PHASE 17.10A — PUBLICATION FIGURE PACK")
print("=" * 78)
print("Verified review ZIP SHA-256:", bundle_hash)
print("Holdout rows:", len(df))
print("Ensemble evaluation rows:", len(full_ensemble))
print("Matched support rows:", len(matched))
print("Stations:", matched["StationId"].nunique())
print("Figures created:", len(register))
print("Feature-importance figure created:", importance_created)
print("\nOutput folder:", OUT)
print("Figure register:", OUT / "figure_register.csv")
print("README:", OUT / "README.md")
print("Summary:", OUT / "phase17_10A_summary.json")
print("\nOriginal bundle modified: False")
print("Models retrained: False")
print("Production enabled: False")
print("Manuscript integration complete: False")
print("Submission ready: False")
print("=" * 78)

In [ ]:
# PHASE 17.10B — INTEGRATED MANUSCRIPT, FIGURES, AND MAIN TABLES
# Produces a new reviewable manuscript; preserves all prior artifacts.

from pathlib import Path
from zipfile import ZipFile
from datetime import datetime
import hashlib
import json
import re
import shutil
import io
import warnings

import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

BUNDLE = (
    ROOT / "outputs/research_paper"
    / "phase17_07T_review_bundle_20261025T181246Z"
    / "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)

FIGURE_PACK = (
    ROOT / "outputs/research_paper"
    / "phase17_10A_publication_figures_20261025T184251"
)

EXPECTED_BUNDLE_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

MANUSCRIPT_MEMBER = (
    "bundle_contents/manuscript/"
    "phase17_07S_provenance_updated_manuscript.md"
)

TABLES = [
    ("Table 1", "Validation model comparison",
     "results/phase17_07J_tables/table_1_validation_model_comparison.csv"),
    ("Table 2", "Final holdout comparison on matched support",
     "results/phase17_07J_tables/table_2_holdout_matched_support.csv"),
    ("Table 3", "Final holdout results on full available support",
     "results/phase17_07J_tables/table_3_holdout_full_available_support.csv"),
    ("Table 4", "Paired day-block bootstrap results",
     "results/phase17_07J_tables/table_4_day_block_bootstrap.csv"),
    ("Table 5", "Performance by AQI band",
     "results/phase17_07J_tables/table_5_performance_by_aqi_band.csv"),
    ("Table 6", "Abrupt-transition detection",
     "results/phase17_07J_tables/table_6_abrupt_transition_detection.csv"),
    ("Table 7", "Feature attribution",
     "results/phase17_07J_tables/table_7_feature_attribution.csv"),
    ("Table 8", "Station-level performance",
     "results/phase17_07J_tables/table_8_station_performance.csv"),
    ("Table 9", "Season-level performance",
     "results/phase17_07J_tables/table_9_season_performance.csv"),
]

EXPECTED_FIGURES = 7

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def md_cell(value):
    if pd.isna(value):
        return "—"
    if isinstance(value, (float, np.floating)):
        if not np.isfinite(value):
            return "—"
        return f"{value:.4f}".rstrip("0").rstrip(".")
    return str(value).replace("|", r"\|").replace("\n", " ")

def header_label(value):
    special = {
        "AQI": "AQI",
        "AQI_Band": "AQI band",
        "StationId": "Station ID",
        "Station": "Station",
        "N": "N",
        "R2": "R²",
        "RMSE": "RMSE",
        "MAE": "MAE",
        "Model": "Model",
        "Day_Blocks": "Day blocks",
        "Persistence_MAE": "Persistence MAE",
        "Model_MAE": "Model MAE",
        "Persistence_RMSE": "Persistence RMSE",
        "Model_RMSE": "Model RMSE",
        "Delta_MAE_Persistence_minus_Model":
            "MAE improvement vs persistence",
        "Delta_MAE_95CI_Lower": "MAE improvement 95% CI lower",
        "Delta_MAE_95CI_Upper": "MAE improvement 95% CI upper",
        "Delta_RMSE_Persistence_minus_Model":
            "RMSE improvement vs persistence",
        "Delta_RMSE_95CI_Lower": "RMSE improvement 95% CI lower",
        "Delta_RMSE_95CI_Upper": "RMSE improvement 95% CI upper",
    }
    return special.get(str(value), str(value).replace("_", " "))

def dataframe_to_markdown(frame):
    columns = list(frame.columns)
    lines = [
        "| " + " | ".join(header_label(c) for c in columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for _, row in frame.iterrows():
        lines.append(
            "| " + " | ".join(md_cell(row[c]) for c in columns) + " |"
        )
    return "\n".join(lines)

print("=" * 78)
print("PHASE 17.10B — INTEGRATED MANUSCRIPT ASSEMBLY")
print("=" * 78)

# 1. Verify source bundle and figure pack.
if not BUNDLE.is_file():
    raise FileNotFoundError(f"Review ZIP not found: {BUNDLE}")

bundle_hash = sha256_file(BUNDLE)
if bundle_hash != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError("Review bundle SHA-256 mismatch. Assembly stopped.")

if not FIGURE_PACK.is_dir():
    raise FileNotFoundError(f"Figure pack not found: {FIGURE_PACK}")

figure_register_path = FIGURE_PACK / "figure_register.csv"
if not figure_register_path.is_file():
    raise FileNotFoundError(
        f"Figure register not found: {figure_register_path}"
    )

source_register = pd.read_csv(figure_register_path)
if len(source_register) != EXPECTED_FIGURES:
    raise RuntimeError(
        f"Expected {EXPECTED_FIGURES} registered figures; "
        f"found {len(source_register)}."
    )

# A unique output folder avoids overwriting earlier work.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10B_integrated_manuscript_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

FIG_DIR = OUT / "figures"
TABLE_DIR = OUT / "tables_csv"
FIG_DIR.mkdir()
TABLE_DIR.mkdir()

# 2. Load the exact provenance-updated manuscript from the verified bundle.
with ZipFile(BUNDLE, "r") as z:
    if z.testzip() is not None:
        raise RuntimeError("Review ZIP integrity test failed.")

    members = z.namelist()
    if MANUSCRIPT_MEMBER not in members:
        raise FileNotFoundError(
            "Provenance-updated manuscript not found inside the reviewed ZIP: "
            + MANUSCRIPT_MEMBER
        )

    original_bytes = z.read(MANUSCRIPT_MEMBER)
    original_text = original_bytes.decode("utf-8")

    original_snapshot_path = OUT / "source_manuscript_snapshot.md"
    original_snapshot_path.write_bytes(original_bytes)

    # 3. Copy all nine original CSV tables from the archive and build
    # readable Markdown renderings. Source CSV files remain unmodified.
    table_sections = []
    table_manifest = []

    for table_number, title, member in TABLES:
        if member not in members:
            raise FileNotFoundError(
                f"Required manuscript table missing from bundle: {member}"
            )

        csv_bytes = z.read(member)
        csv_out = TABLE_DIR / Path(member).name
        csv_out.write_bytes(csv_bytes)

        frame = pd.read_csv(io.BytesIO(csv_bytes))
        full_columns = list(frame.columns)
        display_frame = frame.copy()
        render_note = ""

        # Bootstrap CSV has 33 columns. Render a concise primary table while
        # retaining the entire original CSV beside the manuscript.
        if table_number == "Table 4":
            selected_columns = [
                "Model",
                "N",
                "Day_Blocks",
                "Delta_MAE_Persistence_minus_Model",
                "Delta_MAE_95CI_Lower",
                "Delta_MAE_95CI_Upper",
                "Delta_RMSE_Persistence_minus_Model",
                "Delta_RMSE_95CI_Lower",
                "Delta_RMSE_95CI_Upper",
            ]
            available = [c for c in selected_columns if c in frame.columns]
            if len(available) < 6:
                warnings.warn(
                    "Bootstrap table headers differ from expected; "
                    "retaining all columns in rendered table."
                )
            else:
                display_frame = frame[available].copy()
                render_note = (
                    "\n\n*Note: This is a concise rendering of the main "
                    "bootstrap improvement estimates and confidence intervals. "
                    "The complete, unabridged analysis table is included in "
                    f"`tables_csv/{csv_out.name}`.*"
                )

        section = (
            f"### {table_number}. {title}\n\n"
            + dataframe_to_markdown(display_frame)
            + render_note
            + "\n"
        )
        table_sections.append(section)

        table_manifest.append({
            "table_number": table_number,
            "title": title,
            "source_archive_member": member,
            "original_csv_path": str(csv_out),
            "original_csv_sha256": sha256_bytes(csv_bytes),
            "rows": int(len(frame)),
            "columns": int(len(frame.columns)),
            "rendered_columns": list(display_frame.columns),
        })

# 4. Copy each figure into the integrated manuscript folder.
figure_blocks = []
figure_manifest = []

for _, row in source_register.iterrows():
    num = int(row["figure_number"])
    title = str(row["title"])
    caption = str(row["caption"])
    source_png = Path(str(row["png_path"]))
    source_pdf = Path(str(row["pdf_path"]))

    if not source_png.is_file():
        raise FileNotFoundError(f"Registered PNG is missing: {source_png}")
    if not source_pdf.is_file():
        raise FileNotFoundError(f"Registered PDF is missing: {source_pdf}")

    dest_png = FIG_DIR / source_png.name
    dest_pdf = FIG_DIR / source_pdf.name
    shutil.copy2(source_png, dest_png)
    shutil.copy2(source_pdf, dest_pdf)

    figure_blocks.append(
        f"### Figure {num}. {title}\n\n"
        f"![Figure {num}: {title}](figures/{dest_png.name})\n\n"
        f"**Figure {num}. {title}.** {caption}\n"
    )

    figure_manifest.append({
        "figure_number": num,
        "title": title,
        "caption": caption,
        "png_relative_path": f"figures/{dest_png.name}",
        "pdf_relative_path": f"figures/{dest_pdf.name}",
        "png_sha256": sha256_file(dest_png),
        "pdf_sha256": sha256_file(dest_pdf),
        "source_support": str(row.get("support", "")),
        "source_description": str(row.get("source", "")),
    })

# 5. Insert a manuscript callout in the first Results/Evaluation section if
# a suitable heading exists. If not, the cross-reference paragraph is placed
# in the new integrated asset section below.
callout = (
    "Validation model comparisons are reported in Table 1. Final-holdout "
    "results are summarized in Tables 2–4 and illustrated by Figures 2–4. "
    "AQI-band and station-level patterns are shown in Figures 5–6 and "
    "Tables 5 and 8; seasonal comparisons appear in Table 9. Abrupt-transition "
    "detection is summarized in Table 6, while predictive feature attribution "
    "appears in Figure 7 and Table 7. Figure 1 summarizes the study workflow."
)

lines = original_text.splitlines()
results_heading_index = None

for i, line in enumerate(lines):
    if re.match(
        r"^#{1,3}\s+.*\b(results?|evaluation|model performance|"
        r"experimental findings)\b",
        line,
        flags=re.IGNORECASE,
    ):
        results_heading_index = i
        break

callout_inserted_in_results = results_heading_index is not None
if callout_inserted_in_results:
    marker = "<!-- PHASE 17.10B RESULT CALLOUT -->"
    if marker not in original_text:
        lines.insert(results_heading_index + 1, "")
        lines.insert(results_heading_index + 2, marker)
        lines.insert(results_heading_index + 3, callout)
        lines.insert(results_heading_index + 4, "")
else:
    warnings.warn(
        "No recognizable Results/Evaluation heading was found. "
        "The figure/table cross-reference paragraph will appear in "
        "the integrated asset section."
    )

modified_text = "\n".join(lines).rstrip() + "\n"

# 6. Assemble figures and all nine tables before the References section,
# preserving the original manuscript wording elsewhere.
cross_reference_text = callout if not callout_inserted_in_results else (
    "The study workflow is shown in Figure 1. The following numbered figures "
    "and tables provide the visual evidence and detailed results referenced "
    "in the manuscript."
)

asset_section = (
    "\n\n<!-- PHASE 17.10B ASSET INTEGRATION -->\n"
    "## Integrated Figures and Main Results Tables\n\n"
    f"{cross_reference_text}\n\n"
    "### Figures\n\n"
    + "\n\n".join(figure_blocks)
    + "\n\n### Main results tables\n\n"
    + "\n\n".join(table_sections)
    + "\n\n"
    "### Asset-use and interpretation notes\n\n"
    "- Figures are provided as PNG for manuscript insertion and PDF for "
    "high-quality/vector-aware workflows where supported.\n"
    "- The original nine table CSVs are retained in `tables_csv/` alongside "
    "the rendered Markdown tables.\n"
    "- Table 4 is condensed in the manuscript rendering; its full CSV is "
    "preserved unchanged in the companion folder.\n"
    "- Feature importance describes predictive reliance, not causality.\n"
    "- Full-support ensemble results and matched-support model comparisons "
    "use different eligible row sets; do not compare them as if they had "
    "identical support.\n"
    "- This is an integrated review draft, not a final journal-formatted "
    "submission. Figures, captions, and cross-references still require "
    "author review.\n"
)

reference_pattern = re.compile(
    r"(?im)^#{1,3}\s*(?:references|bibliography|reference list)\b.*$"
)
match = reference_pattern.search(modified_text)

if match:
    integrated_text = (
        modified_text[:match.start()].rstrip()
        + asset_section
        + "\n"
        + modified_text[match.start():]
    )
    insertion_location = "before References heading"
else:
    integrated_text = modified_text.rstrip() + asset_section
    insertion_location = "appended after original manuscript"

# Check numbering and Markdown links before writing.
for i in range(1, EXPECTED_FIGURES + 1):
    expected_token = f"Figure {i}"
    if expected_token not in integrated_text:
        raise RuntimeError(f"Figure cross-reference missing: {expected_token}")

for i in range(1, 10):
    if f"Table {i}" not in integrated_text:
        raise RuntimeError(f"Table reference missing: Table {i}")

integrated_md = OUT / "Delhi_AQI_Integrated_Manuscript.md"
integrated_md.write_text(integrated_text, encoding="utf-8")

# 7. Write manifests, review notes, and status.
pd.DataFrame(figure_manifest).to_csv(
    OUT / "figure_manifest.csv", index=False
)
pd.DataFrame(table_manifest).to_csv(
    OUT / "table_manifest.csv", index=False
)

review_notes = f"""# Phase 17.10B — Manuscript Integration Review

## Source
- Source manuscript: `{MANUSCRIPT_MEMBER}`
- Source manuscript snapshot: `source_manuscript_snapshot.md`
- Verified review ZIP SHA-256: `{bundle_hash}`

## Integrated deliverables
- Manuscript: `Delhi_AQI_Integrated_Manuscript.md`
- Figures: `figures/` (7 PNG + 7 PDF files expected)
- Original main table CSVs: `tables_csv/` (9 files)
- Figure manifest: `figure_manifest.csv`
- Table manifest: `table_manifest.csv`

## Integration checks
- Figures integrated: {len(figure_manifest)}
- Main tables integrated: {len(table_manifest)}
- Results-section callout inserted: {callout_inserted_in_results}
- Figure/table section insertion: {insertion_location}
- Original wording otherwise preserved: YES, apart from the inserted callout
  if a suitable Results/Evaluation heading was found, and the asset section.

## Author review required
1. Read the manuscript from beginning to end after the new asset section is added.
2. Check the inserted result callout appears at a natural location.
3. Check that each figure is legible and properly cited in the narrative.
4. Confirm every table column, unit, metric support, and caption.
5. Check the intended journal's requirements for table/figure placement.
6. Complete the licence review separately.

## Release gate
- Licence review: OPEN
- Submission ready: NO
- Production enabled: NO
- Source data modified: NO
- Frozen models retrained: NO
- Existing review ZIP modified: NO
"""

(OUT / "MANUSCRIPT_INTEGRATION_REVIEW.md").write_text(
    review_notes, encoding="utf-8"
)

summary = {
    "phase": "17.10B",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_bundle": str(BUNDLE),
    "source_bundle_sha256": bundle_hash,
    "source_manuscript_member": MANUSCRIPT_MEMBER,
    "integrated_manuscript": str(integrated_md),
    "source_manuscript_snapshot": str(original_snapshot_path),
    "figure_count": len(figure_manifest),
    "figure_png_count": len(list(FIG_DIR.glob("*.png"))),
    "figure_pdf_count": len(list(FIG_DIR.glob("*.pdf"))),
    "main_table_count": len(table_manifest),
    "table_csv_count": len(list(TABLE_DIR.glob("*.csv"))),
    "callout_inserted_in_results_section": callout_inserted_in_results,
    "asset_section_insertion_location": insertion_location,
    "original_review_bundle_modified": False,
    "source_data_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "outputs": {
        "manuscript": str(integrated_md),
        "figures_directory": str(FIG_DIR),
        "tables_csv_directory": str(TABLE_DIR),
        "figure_manifest": str(OUT / "figure_manifest.csv"),
        "table_manifest": str(OUT / "table_manifest.csv"),
        "review_notes": str(OUT / "MANUSCRIPT_INTEGRATION_REVIEW.md"),
    }
}
(OUT / "phase17_10B_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

readme = f"""# Phase 17.10B — Integrated Manuscript Package

Generated: {summary["created_at"]}

This directory contains the integrated Markdown manuscript and its companion
figure/table assets. It is not yet a final review ZIP or final publication
export.

Start with `Delhi_AQI_Integrated_Manuscript.md`. Keep the figures/ and
tables_csv/ directories beside it so the relative image links resolve.

Author licence review remains open; submission readiness is false.
"""
(OUT / "README.md").write_text(readme, encoding="utf-8")

print("\n[1] INPUT VERIFICATION")
print("Bundle SHA-256 verified:", bundle_hash == EXPECTED_BUNDLE_SHA256)
print("Source manuscript:", MANUSCRIPT_MEMBER)

print("\n[2] INTEGRATION RESULTS")
print("Figures integrated:", len(figure_manifest))
print("PNG files:", len(list(FIG_DIR.glob("*.png"))))
print("PDF files:", len(list(FIG_DIR.glob("*.pdf"))))
print("Main tables integrated:", len(table_manifest))
print("Original table CSV files copied:", len(list(TABLE_DIR.glob("*.csv"))))
print("Results-section callout inserted:", callout_inserted_in_results)
print("Asset section inserted:", insertion_location)

print("\n[3] OUTPUTS")
print("Output folder:", OUT)
print("Integrated manuscript:", integrated_md)
print("Figure manifest:", OUT / "figure_manifest.csv")
print("Table manifest:", OUT / "table_manifest.csv")
print("Review notes:", OUT / "MANUSCRIPT_INTEGRATION_REVIEW.md")
print("Summary:", OUT / "phase17_10B_summary.json")

print("\n[4] RELEASE GATE")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Existing review ZIP modified: False")
print("Source data modified: False")
print("Models retrained: False")
print("=" * 78)

In [ ]:
# PHASE 17.10B-R1 — ROBUST MANUSCRIPT INTEGRATION
# Discovers archive members dynamically and validates inputs before writing.
# Preserves the existing review ZIP, source data, and frozen models.

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from datetime import datetime
import hashlib
import json
import re
import shutil
import io
import warnings

import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

BUNDLE = (
    ROOT / "outputs/research_paper"
    / "phase17_07T_review_bundle_20261025T181246Z"
    / "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)

FIGURE_PACK = (
    ROOT / "outputs/research_paper"
    / "phase17_10A_publication_figures_20261025T184251"
)

EXPECTED_BUNDLE_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

MANUSCRIPT_BASENAME = "phase17_07S_provenance_updated_manuscript.md"

TABLES = [
    ("Table 1", "Validation model comparison",
     "table_1_validation_model_comparison.csv"),
    ("Table 2", "Final holdout comparison on matched support",
     "table_2_holdout_matched_support.csv"),
    ("Table 3", "Final holdout results on full available support",
     "table_3_holdout_full_available_support.csv"),
    ("Table 4", "Paired day-block bootstrap results",
     "table_4_day_block_bootstrap.csv"),
    ("Table 5", "Performance by AQI band",
     "table_5_performance_by_aqi_band.csv"),
    ("Table 6", "Abrupt-transition detection",
     "table_6_abrupt_transition_detection.csv"),
    ("Table 7", "Feature attribution",
     "table_7_feature_attribution.csv"),
    ("Table 8", "Station-level performance",
     "table_8_station_performance.csv"),
    ("Table 9", "Season-level performance",
     "table_9_season_performance.csv"),
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def resolve_member(members, basename, preferred_path_parts=()):
    """Find an archive member by basename, preferring the intended subfolder."""
    exact = [m for m in members if m == basename]
    if exact:
        return exact[0]

    candidates = [
        m for m in members
        if PurePosixPath(m).name == basename
    ]

    if preferred_path_parts:
        preferred = [
            m for m in candidates
            if all(part.lower() in m.lower()
                   for part in preferred_path_parts)
        ]
        if len(preferred) == 1:
            return preferred[0]
        if len(preferred) > 1:
            candidates = preferred

    if len(candidates) == 1:
        return candidates[0]

    if not candidates:
        raise FileNotFoundError(
            f"Archive member not found by basename: {basename}"
        )

    raise RuntimeError(
        f"Ambiguous archive member {basename}; candidates:\n"
        + "\n".join(candidates)
    )

def md_cell(value):
    if pd.isna(value):
        return "—"
    if isinstance(value, (float, np.floating)):
        if not np.isfinite(value):
            return "—"
        return f"{value:.4f}".rstrip("0").rstrip(".")
    return str(value).replace("|", r"\|").replace("\n", " ")

def header_label(value):
    replacements = {
        "StationId": "Station ID",
        "Day_Blocks": "Day blocks",
        "R2": "R²",
        "Delta_MAE_Persistence_minus_Model": "MAE improvement vs persistence",
        "Delta_MAE_95CI_Lower": "MAE improvement 95% CI lower",
        "Delta_MAE_95CI_Upper": "MAE improvement 95% CI upper",
        "Delta_RMSE_Persistence_minus_Model": "RMSE improvement vs persistence",
        "Delta_RMSE_95CI_Lower": "RMSE improvement 95% CI lower",
        "Delta_RMSE_95CI_Upper": "RMSE improvement 95% CI upper",
    }
    return replacements.get(str(value), str(value).replace("_", " "))

def dataframe_to_markdown(frame):
    columns = list(frame.columns)
    lines = [
        "| " + " | ".join(header_label(c) for c in columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for _, row in frame.iterrows():
        lines.append(
            "| " + " | ".join(md_cell(row[c]) for c in columns) + " |"
        )
    return "\n".join(lines)

print("=" * 78)
print("PHASE 17.10B-R1 — ROBUST MANUSCRIPT INTEGRATION")
print("=" * 78)

# ------------------------------------------------------------------
# 1. Preflight — verify the bundle, manuscript, tables, and figures.
#    Nothing is written until all required inputs have been found.
# ------------------------------------------------------------------

if not BUNDLE.is_file():
    raise FileNotFoundError(f"Review ZIP not found: {BUNDLE}")

bundle_hash = sha256_file(BUNDLE)
if bundle_hash != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(
        "Review ZIP SHA-256 mismatch. Integration stopped."
    )

figure_register_path = FIGURE_PACK / "figure_register.csv"
if not figure_register_path.is_file():
    raise FileNotFoundError(
        f"Figure register not found: {figure_register_path}"
    )

figure_register = pd.read_csv(figure_register_path)
if len(figure_register) != 7:
    raise RuntimeError(
        f"Expected 7 registered figures; found {len(figure_register)}."
    )

with ZipFile(BUNDLE, "r") as z:
    bad_member = z.testzip()
    if bad_member:
        raise RuntimeError(f"ZIP integrity failed at: {bad_member}")

    members = z.namelist()

    # Fix: discover the manuscript by basename, not an assumed folder prefix.
    manuscript_member = resolve_member(
        members,
        MANUSCRIPT_BASENAME,
        preferred_path_parts=("manuscript",)
    )
    original_bytes = z.read(manuscript_member)
    original_text = original_bytes.decode("utf-8")

    # Discover all table members without assuming a leading "bundle_contents/".
    resolved_tables = []
    for number, title, basename in TABLES:
        member = resolve_member(
            members,
            basename,
            preferred_path_parts=("phase17_07J_tables",)
        )
        raw_bytes = z.read(member)
        frame = pd.read_csv(io.BytesIO(raw_bytes))
        resolved_tables.append({
            "number": number,
            "title": title,
            "member": member,
            "bytes": raw_bytes,
            "frame": frame,
        })

# Validate all registered figure files before making the output directory.
resolved_figures = []
for _, row in figure_register.iterrows():
    png = Path(str(row["png_path"]))
    pdf = Path(str(row["pdf_path"]))

    if not png.is_file():
        raise FileNotFoundError(f"Registered PNG not found: {png}")
    if not pdf.is_file():
        raise FileNotFoundError(f"Registered PDF not found: {pdf}")

    resolved_figures.append({
        "number": int(row["figure_number"]),
        "title": str(row["title"]),
        "caption": str(row["caption"]),
        "source_png": png,
        "source_pdf": pdf,
        "support": str(row.get("support", "")),
        "source": str(row.get("source", "")),
    })

if len(resolved_tables) != 9:
    raise RuntimeError(
        f"Preflight expected 9 tables; found {len(resolved_tables)}."
    )

print("\n[1] INPUT PREFLIGHT PASSED")
print("Review ZIP SHA-256 verified:", bundle_hash == EXPECTED_BUNDLE_SHA256)
print("Discovered manuscript member:", manuscript_member)
print("Discovered required tables:", len(resolved_tables))
print("Validated figure pairs:", len(resolved_figures))

# ------------------------------------------------------------------
# 2. Create a new output directory only after successful preflight.
# ------------------------------------------------------------------

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10B_R1_integrated_manuscript_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

FIG_DIR = OUT / "figures"
TABLE_DIR = OUT / "tables_csv"
FIG_DIR.mkdir()
TABLE_DIR.mkdir()

(OUT / "source_manuscript_snapshot.md").write_bytes(original_bytes)

# ------------------------------------------------------------------
# 3. Copy figures and prepare numbered captions.
# ------------------------------------------------------------------

figure_blocks = []
figure_manifest = []

for item in resolved_figures:
    png_dest = FIG_DIR / item["source_png"].name
    pdf_dest = FIG_DIR / item["source_pdf"].name
    shutil.copy2(item["source_png"], png_dest)
    shutil.copy2(item["source_pdf"], pdf_dest)

    number = item["number"]
    title = item["title"]
    caption = item["caption"]

    figure_blocks.append(
        f"### Figure {number}. {title}\n\n"
        f"![Figure {number}: {title}](figures/{png_dest.name})\n\n"
        f"**Figure {number}. {title}.** {caption}\n\n"
        f"*PDF version:* `figures/{pdf_dest.name}`"
    )

    figure_manifest.append({
        "figure_number": number,
        "title": title,
        "caption": caption,
        "png": f"figures/{png_dest.name}",
        "pdf": f"figures/{pdf_dest.name}",
        "png_sha256": sha256_file(png_dest),
        "pdf_sha256": sha256_file(pdf_dest),
        "support": item["support"],
        "source": item["source"],
    })

# ------------------------------------------------------------------
# 4. Copy the original table CSVs and render manuscript versions.
#    Table 4 is condensed in Markdown; the complete CSV is preserved.
# ------------------------------------------------------------------

table_blocks = []
table_manifest = []

for item in resolved_tables:
    number = item["number"]
    title = item["title"]
    frame = item["frame"]
    raw_bytes = item["bytes"]
    original_member = item["member"]

    csv_dest = TABLE_DIR / PurePosixPath(original_member).name
    csv_dest.write_bytes(raw_bytes)

    render_frame = frame.copy()
    note = ""

    if number == "Table 4":
        desired = [
            "Model", "N", "Day_Blocks",
            "Delta_MAE_Persistence_minus_Model",
            "Delta_MAE_95CI_Lower", "Delta_MAE_95CI_Upper",
            "Delta_RMSE_Persistence_minus_Model",
            "Delta_RMSE_95CI_Lower", "Delta_RMSE_95CI_Upper",
        ]
        available = [c for c in desired if c in frame.columns]
        if len(available) >= 6:
            render_frame = frame[available].copy()
            note = (
                "\n\n*Note: This concise rendering displays the main "
                "improvement estimates and confidence intervals. The full "
                f"33-column source table, if applicable, is retained in "
                f"`tables_csv/{csv_dest.name}`.*"
            )

    table_blocks.append(
        f"### {number}. {title}\n\n"
        + dataframe_to_markdown(render_frame)
        + note
    )

    table_manifest.append({
        "table_number": number,
        "title": title,
        "source_archive_member": original_member,
        "csv_path": f"tables_csv/{csv_dest.name}",
        "csv_sha256": sha256_file(csv_dest),
        "rows": int(len(frame)),
        "columns": int(len(frame.columns)),
        "rendered_columns": list(render_frame.columns),
    })

# ------------------------------------------------------------------
# 5. Add references to the numbered assets and preserve manuscript wording.
# ------------------------------------------------------------------

callout = (
    "The study workflow is summarized in Figure 1. Validation model "
    "comparisons are reported in Table 1. Final-holdout results are shown "
    "in Tables 2–4 and Figures 2–4. Performance across AQI bands is shown "
    "in Figure 5 and Table 5, station-level differences in Figure 6 and "
    "Table 8, abrupt-transition results in Table 6, feature attribution "
    "in Figure 7 and Table 7, and seasonal performance in Table 9."
)

lines = original_text.splitlines()
results_heading_index = None

for i, line in enumerate(lines):
    if re.match(
        r"^#{1,4}\s+.*\b(results?|evaluation|model performance|"
        r"experimental findings)\b",
        line,
        flags=re.IGNORECASE
    ):
        results_heading_index = i
        break

marker = "<!-- PHASE 17.10B-R1 RESULT CALLOUT -->"
callout_inserted = marker in original_text

if results_heading_index is not None and not callout_inserted:
    lines[results_heading_index + 1:results_heading_index + 1] = [
        "",
        marker,
        callout,
        "",
    ]

modified_text = "\n".join(lines).rstrip() + "\n"

asset_section = (
    "\n\n<!-- PHASE 17.10B-R1 ASSET INTEGRATION -->\n"
    "## Integrated Figures and Main Results Tables\n\n"
    + ("" if results_heading_index is not None else callout + "\n\n")
    + "### Figures\n\n"
    + "\n\n".join(figure_blocks)
    + "\n\n### Main Results Tables\n\n"
    + "\n\n".join(table_blocks)
    + "\n\n### Interpretation and review notes\n\n"
    "- Figure captions and table renderings are provided for author review.\n"
    "- Each figure is included as PNG and PDF; accompanying CSV tables are "
      "preserved in the `tables_csv/` directory.\n"
    "- Table 4 is condensed in Markdown. Its original CSV is retained intact.\n"
    "- Feature importance represents predictive reliance, not causal effects.\n"
    "- The full-available-support ensemble evaluation and matched-support "
      "model comparisons use different eligible row sets. Keep these supports "
      "distinct in the narrative.\n"
    "- Review axes, units, captions, terminology, journal formatting, and "
      "all figure/table callouts before submission.\n"
)

reference_pattern = re.compile(
    r"(?im)^#{1,4}\s*(?:\d+(?:\.\d+)*\.?\s*)?"
    r"(?:references|bibliography|reference list)\b.*$"
)
reference_match = reference_pattern.search(modified_text)

if reference_match:
    integrated_text = (
        modified_text[:reference_match.start()].rstrip()
        + asset_section
        + "\n"
        + modified_text[reference_match.start():]
    )
    insertion_location = "before references section"
else:
    integrated_text = modified_text.rstrip() + asset_section
    insertion_location = "appended after manuscript body; no reference heading detected"

integrated_path = OUT / "Delhi_AQI_Integrated_Manuscript.md"
integrated_path.write_text(integrated_text, encoding="utf-8")

# ------------------------------------------------------------------
# 6. Validate references and create manifests/review status.
# ------------------------------------------------------------------

validation_errors = []
for i in range(1, 8):
    if f"Figure {i}" not in integrated_text:
        validation_errors.append(f"Figure {i} reference not found")

for i in range(1, 10):
    if f"Table {i}" not in integrated_text:
        validation_errors.append(f"Table {i} reference not found")

if len(figure_manifest) != 7:
    validation_errors.append("Expected seven figures")
if len(table_manifest) != 9:
    validation_errors.append("Expected nine tables")

if validation_errors:
    raise RuntimeError(
        "Integration validation failed:\n" + "\n".join(validation_errors)
    )

pd.DataFrame(figure_manifest).to_csv(
    OUT / "figure_manifest.csv", index=False
)
pd.DataFrame(table_manifest).to_csv(
    OUT / "table_manifest.csv", index=False
)

review = f"""# Phase 17.10B-R1 — Manuscript Integration Review

## Input verification
- Verified source ZIP SHA-256: `{bundle_hash}`
- Actual manuscript member discovered: `{manuscript_member}`
- Original manuscript snapshot: `source_manuscript_snapshot.md`

## Assembled artifacts
- Integrated Markdown manuscript: `Delhi_AQI_Integrated_Manuscript.md`
- PNG/PDF figures: `figures/`
- Original main table CSVs: `tables_csv/`
- Figure manifest: `figure_manifest.csv`
- Table manifest: `table_manifest.csv`

## Results
- Figures integrated: {len(figure_manifest)}
- Original main table CSVs copied: {len(table_manifest)}
- Figure/table references validated: PASS
- Results-section callout inserted: {results_heading_index is not None}
- Asset section location: {insertion_location}

## Author review still required
1. Read the complete manuscript, including the integrated asset section.
2. Confirm placement of the Results callout and figure/table numbering.
3. Review captions, axes, units, data support, and legibility.
4. Check all scientific statements and references.
5. Export and inspect the journal-ready document format.
6. Complete the separate author licence review.

## Release gate
- Licence review: OPEN
- Submission ready: NO
- Production enabled: NO
- Source data modified: NO
- Frozen models retrained: NO
- Original review ZIP modified: NO
"""

(OUT / "MANUSCRIPT_INTEGRATION_REVIEW.md").write_text(
    review, encoding="utf-8"
)

summary = {
    "phase": "17.10B-R1",
    "created_at": datetime.now().astimezone().isoformat(),
    "verified_bundle_sha256": bundle_hash,
    "discovered_manuscript_member": manuscript_member,
    "integrated_manuscript": str(integrated_path),
    "figure_count": len(figure_manifest),
    "figure_png_count": len(list(FIG_DIR.glob("*.png"))),
    "figure_pdf_count": len(list(FIG_DIR.glob("*.pdf"))),
    "main_table_count": len(table_manifest),
    "source_table_csv_count": len(list(TABLE_DIR.glob("*.csv"))),
    "reference_validation": "PASS",
    "asset_section_location": insertion_location,
    "licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "models_retrained": False,
    "original_review_zip_modified": False,
    "outputs": {
        "manuscript": str(integrated_path),
        "figure_folder": str(FIG_DIR),
        "table_folder": str(TABLE_DIR),
        "figure_manifest": str(OUT / "figure_manifest.csv"),
        "table_manifest": str(OUT / "table_manifest.csv"),
        "review_notes": str(OUT / "MANUSCRIPT_INTEGRATION_REVIEW.md"),
    },
}

(OUT / "phase17_10B_R1_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

(OUT / "README.md").write_text(
    f"""# Phase 17.10B-R1 — Integrated Manuscript Package

Generated: {summary["created_at"]}

Start with `Delhi_AQI_Integrated_Manuscript.md`.
Keep `figures/` and `tables_csv/` alongside it so relative figure links work.
Review `MANUSCRIPT_INTEGRATION_REVIEW.md` before using this draft.

The original archive, source data, and frozen models were not modified.
Licence review remains open; this is not yet a submission-ready package.
""",
    encoding="utf-8"
)

print("\n[2] INTEGRATION RESULTS")
print("Discovered manuscript member:", manuscript_member)
print("Figures:", len(figure_manifest))
print("PNG files:", len(list(FIG_DIR.glob("*.png"))))
print("PDF files:", len(list(FIG_DIR.glob("*.pdf"))))
print("Main tables:", len(table_manifest))
print("Source table CSVs:", len(list(TABLE_DIR.glob("*.csv"))))
print("Figure/table reference validation: PASS")
print("Asset section:", insertion_location)

print("\n[3] OUTPUTS")
print("Output folder:", OUT)
print("Integrated manuscript:", integrated_path)
print("Review notes:", OUT / "MANUSCRIPT_INTEGRATION_REVIEW.md")
print("Figure manifest:", OUT / "figure_manifest.csv")
print("Table manifest:", OUT / "table_manifest.csv")
print("Summary:", OUT / "phase17_10B_R1_summary.json")

print("\n[4] RELEASE GATE")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data modified: False")
print("Models retrained: False")
print("Original review ZIP modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.10C — INTEGRATED PAPER QA AND REVIEW PACKAGE
# Creates a new review ZIP with the manuscript, figures, tables, and evidence.
# The package is for review, NOT yet submission-ready.

from pathlib import Path, PurePosixPath
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime
from PIL import Image
import hashlib
import json
import re
import csv
import shutil
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

INTEGRATED = (
    ROOT / "outputs/research_paper"
    / "phase17_10B_R1_integrated_manuscript_20261025T184741"
)

ORIGINAL_BUNDLE = (
    ROOT / "outputs/research_paper"
    / "phase17_07T_review_bundle_20261025T181246Z"
    / "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)

EXPECTED_BUNDLE_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def copy_if_exists(src, dest):
    src = Path(src)
    if src.is_file():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
        return True
    return False

print("=" * 78)
print("PHASE 17.10C — INTEGRATED PAPER QA AND REVIEW PACKAGE")
print("=" * 78)

# ------------------------------------------------------------------
# 1. Confirm the original evidence archive and integrated package exist.
# ------------------------------------------------------------------

if not ORIGINAL_BUNDLE.is_file():
    raise FileNotFoundError(f"Original review bundle missing: {ORIGINAL_BUNDLE}")

if sha256_file(ORIGINAL_BUNDLE) != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError("Original review bundle hash mismatch; stopped.")

if not INTEGRATED.is_dir():
    raise FileNotFoundError(f"Integrated manuscript folder missing: {INTEGRATED}")

MANUSCRIPT = INTEGRATED / "Delhi_AQI_Integrated_Manuscript.md"
SOURCE_SNAPSHOT = INTEGRATED / "source_manuscript_snapshot.md"
FIGURE_MANIFEST = INTEGRATED / "figure_manifest.csv"
TABLE_MANIFEST = INTEGRATED / "table_manifest.csv"
REVIEW_NOTES = INTEGRATED / "MANUSCRIPT_INTEGRATION_REVIEW.md"
SUMMARY_PATH = INTEGRATED / "phase17_10B_R1_summary.json"

required_inputs = [
    MANUSCRIPT, SOURCE_SNAPSHOT, FIGURE_MANIFEST,
    TABLE_MANIFEST, REVIEW_NOTES, SUMMARY_PATH,
]
missing_inputs = [str(p) for p in required_inputs if not p.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "Required integrated-package inputs are missing:\n"
        + "\n".join(missing_inputs)
    )

with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    integration_summary = json.load(f)

figures = pd.read_csv(FIGURE_MANIFEST)
tables = pd.read_csv(TABLE_MANIFEST)
manuscript_text = MANUSCRIPT.read_text(encoding="utf-8")

if len(figures) != 7:
    raise RuntimeError(f"Expected 7 figure records; found {len(figures)}.")
if len(tables) != 9:
    raise RuntimeError(f"Expected 9 table records; found {len(tables)}.")

# ------------------------------------------------------------------
# 2. Validate figure files, image links, and table files.
# ------------------------------------------------------------------

errors = []
warnings_found = []
figure_checks = []
table_checks = []

image_links = re.findall(
    r'!\[[^\]]*\]\(([^)]+)\)',
    manuscript_text
)

for _, row in figures.iterrows():
    number = int(row["figure_number"])
    png = INTEGRATED / str(row["png"])
    pdf = INTEGRATED / str(row["pdf"])

    record = {
        "figure_number": number,
        "png_exists": png.is_file(),
        "pdf_exists": pdf.is_file(),
        "png_opens": False,
        "png_width": None,
        "png_height": None,
        "png_sha256": None,
        "pdf_sha256": None,
    }

    if not png.is_file():
        errors.append(f"Figure {number}: PNG missing: {png}")
    else:
        try:
            with Image.open(png) as img:
                img.verify()
            with Image.open(png) as img:
                record["png_width"], record["png_height"] = img.size
                record["png_opens"] = True
            record["png_sha256"] = sha256_file(png)
        except Exception as exc:
            errors.append(f"Figure {number}: PNG could not be validated: {exc}")

    if not pdf.is_file():
        errors.append(f"Figure {number}: PDF missing: {pdf}")
    else:
        record["pdf_sha256"] = sha256_file(pdf)
        with open(pdf, "rb") as f:
            if not f.read(5).startswith(b"%PDF-"):
                errors.append(f"Figure {number}: PDF signature is invalid.")

    figure_checks.append(record)

# Every embedded Markdown image path must resolve relative to the manuscript.
for link in image_links:
    if re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*:", link):
        continue
    resolved = (MANUSCRIPT.parent / link).resolve()
    if not resolved.is_file():
        errors.append(f"Broken relative Markdown image link: {link}")

for number, expected in enumerate(range(1, 8), start=1):
    if number != expected:
        errors.append("Internal figure-number check failed.")

figure_headings = re.findall(
    r"(?m)^###\s+Figure\s+(\d+)\.",
    manuscript_text
)
table_headings = re.findall(
    r"(?m)^###\s+Table\s+(\d+)\.",
    manuscript_text
)

if sorted(int(x) for x in figure_headings) != list(range(1, 8)):
    errors.append(
        f"Figure headings are not exactly 1–7: {figure_headings}"
    )

if sorted(int(x) for x in table_headings) != list(range(1, 10)):
    errors.append(
        f"Table headings are not exactly 1–9: {table_headings}"
    )

for _, row in tables.iterrows():
    number = str(row["table_number"])
    table_path = INTEGRATED / str(row["csv_path"])
    record = {
        "table_number": number,
        "path": str(table_path),
        "exists": table_path.is_file(),
        "readable": False,
        "rows": None,
        "columns": None,
        "sha256": None,
    }

    if not table_path.is_file():
        errors.append(f"Table {number}: source CSV missing: {table_path}")
    else:
        try:
            frame = pd.read_csv(table_path)
            record["readable"] = True
            record["rows"] = int(len(frame))
            record["columns"] = int(len(frame.columns))
            record["sha256"] = sha256_file(table_path)
        except Exception as exc:
            errors.append(f"Table {number}: CSV not readable: {exc}")

    table_checks.append(record)

# Check that the References heading and basic provenance concepts remain.
has_references_heading = bool(
    re.search(
        r"(?im)^#{1,4}\s*(?:\d+(?:\.\d+)*\.?\s*)?"
        r"(?:references|bibliography|reference list)\b",
        manuscript_text
    )
)

if not has_references_heading:
    warnings_found.append(
        "No recognized References heading detected; manually inspect the manuscript."
    )

provenance_terms = ["CPCB", "TUMI"]
for term in provenance_terms:
    if term.lower() not in manuscript_text.lower():
        warnings_found.append(
            f"Expected provenance term '{term}' was not detected."
        )

print("\n[1] QUALITY CHECK")
print("Figure manifest rows:", len(figures))
print("PNG files validated:", sum(x["png_opens"] for x in figure_checks), "/ 7")
print("PDF files present:", sum(x["pdf_exists"] for x in figure_checks), "/ 7")
print("Markdown image links:", len(image_links))
print("Main table CSVs readable:", sum(x["readable"] for x in table_checks), "/ 9")
print("Numbered figure headings:", len(figure_headings))
print("Numbered table headings:", len(table_headings))
print("Recognized references heading:", has_references_heading)

if errors:
    print("\nFAILED CHECKS:")
    for e in errors:
        print("-", e)
    raise RuntimeError(
        "Quality checks failed. No review ZIP was created. Correct the reported "
        "issue before rerunning the packaging phase."
    )

# ------------------------------------------------------------------
# 3. Create a separate review package, keeping historical outputs intact.
# ------------------------------------------------------------------

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10C_final_review_package_{stamp}"
)
CONTENT = OUT / "package_contents"
CONTENT.mkdir(parents=True, exist_ok=False)

# Copy integrated manuscript and its source snapshot.
shutil.copy2(MANUSCRIPT, CONTENT / MANUSCRIPT.name)
shutil.copy2(SOURCE_SNAPSHOT, CONTENT / SOURCE_SNAPSHOT.name)
shutil.copy2(REVIEW_NOTES, CONTENT / REVIEW_NOTES.name)
shutil.copy2(SUMMARY_PATH, CONTENT / "phase17_10B_R1_summary.json")
shutil.copy2(FIGURE_MANIFEST, CONTENT / "figure_manifest.csv")
shutil.copy2(TABLE_MANIFEST, CONTENT / "table_manifest.csv")

# Copy all figure formats and nine source CSV tables.
shutil.copytree(INTEGRATED / "figures", CONTENT / "figures")
shutil.copytree(INTEGRATED / "tables_csv", CONTENT / "tables_csv")

# Keep a copy of evidence showing the earlier content audit conclusion.
EVIDENCE = CONTENT / "supporting_evidence"
EVIDENCE.mkdir()

evidence_sources = [
    (
        ROOT / "outputs/research_paper"
        / "phase17_07Z_content_audit_closure_20261025T182839",
        [
            "phase17_07Z_content_audit_closure.md",
            "phase17_07Z_content_audit_closure.json",
        ],
    ),
    (
        ROOT / "outputs/research_paper"
        / "phase17_08_licence_clarification_20261025T183001",
        [
            "TUMI_LICENCE_CLARIFICATION_EMAIL.md",
            "CPCB_LICENCE_CLARIFICATION_REQUEST.md",
            "PHASE_17_08_SOURCE_REGISTER.csv",
            "PHASE_17_08_RESPONSE_TRACKER.csv",
            "phase17_08_status.json",
            "README.md",
        ],
    ),
    (
        ROOT / "outputs/research_paper"
        / "phase17_09_asset_audit_20261025T183617",
        [
            "phase17_09_asset_audit_report.md",
            "phase17_09_asset_audit_summary.json",
            "phase17_09_project_asset_inventory.csv",
            "phase17_09_review_bundle_manifest.csv",
        ],
    ),
    (
        ROOT / "outputs/research_paper"
        / "phase17_07U_author_release_gate_20261025T181929",
        [
            "AUTHOR_RELEASE_CHECKLIST.md",
            "AUTHOR_LICENCE_REVIEW_LOG_TEMPLATE.md",
            "phase17_07U_release_gate_status.json",
        ],
    ),
]

copied_evidence = []
for folder, names in evidence_sources:
    for name in names:
        source = folder / name
        if source.is_file():
            dest = EVIDENCE / folder.name / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, dest)
            copied_evidence.append(str(dest.relative_to(CONTENT)))

# This package contains study outputs and audit records, not raw project datasets.
known_raw_names = {
    "station_hour.csv", "stations.csv",
    "station_day.csv", "city_hour.csv", "city_day.csv",
}
obvious_raw_files = [
    p.relative_to(CONTENT).as_posix()
    for p in CONTENT.rglob("*")
    if p.is_file() and p.name.lower() in known_raw_names
]
if obvious_raw_files:
    raise RuntimeError(
        "Potential raw dataset file unexpectedly found in package: "
        + ", ".join(obvious_raw_files)
    )

# Explicit release status prevents a review package from being mistaken
# for a cleared submission package.
release_status = f"""# Release Status — Author Review Required

Created: {datetime.now().astimezone().isoformat()}

## Content package
- Integrated Markdown manuscript included.
- Seven figures included as PNG and PDF.
- Nine principal results tables included as CSV and rendered in Markdown.
- Figure/table manifests and selected audit evidence included.
- Original review ZIP was verified before packaging.

## Outstanding release gate
**Author licence review remains OPEN. Submission-ready status: NO.**

The project's content scans found no obvious raw source-data files in the
review ZIP. The earlier filename, schema and bounded sample-value checks are
screening checks, not a legal determination or proof that every derived
value is free of third-party rights restrictions.

The TUMI and CPCB clarification drafts are included as evidence, but the request
tracker records that they were prepared only; no request or response is assumed
to have occurred.

Before submission, the responsible author must:
1. Review and submit the clarification requests.
2. Preserve actual responses and source records.
3. Complete the author licence review log and record a supported decision.
4. Read the integrated manuscript, inspect the figures, and confirm captions,
   values, tables, citations and journal formatting.
5. Export the appropriate journal document and inspect the rendered pages.

## Safeguards
- Source dataset files modified: NO
- Frozen models retrained: NO
- Production enabled: NO
"""

(CONTENT / "RELEASE_STATUS.md").write_text(
    release_status, encoding="utf-8"
)

readme = f"""# Delhi AQI — Integrated Research Review Package

This is a review candidate package, not a submission-cleared release.

## Start here
- `Delhi_AQI_Integrated_Manuscript.md`
- `MANUSCRIPT_INTEGRATION_REVIEW.md`
- `RELEASE_STATUS.md`

## Figures and tables
- `figures/` contains publication candidate figures in PNG and PDF.
- `tables_csv/` contains all nine original table CSVs used in the manuscript.
- `figure_manifest.csv` and `table_manifest.csv` provide figure/table records.

## Supporting evidence
`supporting_evidence/` contains selected content-audit reports, asset audit
records, the licence-clarification drafts, response tracker, and release gate
template.

No raw dataset CSVs or model files are intended to be included in this package.
Author licence review remains open; do not treat this package as submission-ready.
"""
(CONTENT / "README.md").write_text(readme, encoding="utf-8")

# Make a hash manifest for all files currently present in package_contents.
manifest_rows = []
for path in sorted(CONTENT.rglob("*")):
    if path.is_file():
        manifest_rows.append({
            "relative_path": path.relative_to(CONTENT).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

manifest_path = CONTENT / "PACKAGE_FILE_MANIFEST.csv"
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)

# Build ZIP from the contents folder, with stable relative paths.
zip_path = OUT / "Delhi_AQI_Integrated_Research_Review_Package.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(CONTENT.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(CONTENT).as_posix()
            )

# ------------------------------------------------------------------
# 4. Verify the newly built review ZIP and save a final package report.
# ------------------------------------------------------------------

with ZipFile(zip_path, "r") as archive:
    bad_member = archive.testzip()
    if bad_member:
        raise RuntimeError(f"New ZIP integrity failed at: {bad_member}")
    package_members = archive.namelist()

    zipped_names = set(package_members)
    expected_members = {
        "Delhi_AQI_Integrated_Manuscript.md",
        "MANUSCRIPT_INTEGRATION_REVIEW.md",
        "RELEASE_STATUS.md",
        "README.md",
        "figure_manifest.csv",
        "table_manifest.csv",
    }
    missing_members = sorted(expected_members - zipped_names)

    if missing_members:
        raise RuntimeError(
            "Review ZIP missing required members: " + ", ".join(missing_members)
        )

    zipped_pngs = [
        n for n in package_members if n.lower().endswith(".png")
    ]
    zipped_pdfs = [
        n for n in package_members if n.lower().endswith(".pdf")
    ]
    zipped_table_csvs = [
        n for n in package_members
        if n.startswith("tables_csv/") and n.lower().endswith(".csv")
    ]

    if len(zipped_pngs) != 7 or len(zipped_pdfs) != 7:
        raise RuntimeError(
            f"Expected 7 PNG and 7 PDF figures, got "
            f"{len(zipped_pngs)} PNG and {len(zipped_pdfs)} PDF."
        )

    if len(zipped_table_csvs) != 9:
        raise RuntimeError(
            f"Expected 9 source table CSVs in ZIP; found {len(zipped_table_csvs)}."
        )

zip_hash = sha256_file(zip_path)

final_report = {
    "phase": "17.10C",
    "created_at": datetime.now().astimezone().isoformat(),
    "original_bundle_sha256_verified": True,
    "integrated_manuscript_qc": "PASS",
    "figure_count": 7,
    "figure_png_count_in_zip": len(zipped_pngs),
    "figure_pdf_count_in_zip": len(zipped_pdfs),
    "table_count": 9,
    "table_csv_count_in_zip": len(zipped_table_csvs),
    "relative_image_links_checked": len(image_links),
    "supporting_evidence_files_copied": len(copied_evidence),
    "content_audit_warnings": warnings_found,
    "zip_path": str(zip_path),
    "zip_sha256": zip_hash,
    "zip_integrity": "PASS",
    "zip_entry_count": len(package_members),
    "obvious_raw_dataset_files_detected": obvious_raw_files,
    "source_data_modified": False,
    "frozen_models_retrained": False,
    "production_enabled": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "outputs": {
        "package_contents": str(CONTENT),
        "zip": str(zip_path),
        "file_manifest": str(manifest_path),
    },
}

report_path = OUT / "phase17_10C_package_verification.json"
report_path.write_text(
    json.dumps(final_report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

summary_md = f"""# Phase 17.10C — Review Package Verification

- ZIP: `{zip_path}`
- SHA-256: `{zip_hash}`
- ZIP integrity: PASS
- ZIP entries: {len(package_members)}
- Figures: 7 PNG + 7 PDF
- Main table CSVs: 9
- Relative Markdown image links checked: {len(image_links)}
- Supporting evidence files copied: {len(copied_evidence)}
- Obvious raw dataset files found: {len(obvious_raw_files)}

## Release status
**Author licence review: OPEN. Submission-ready status: NO.**

The package was built and integrity-tested. The package contents still require
author review, and the intended-use licence question remains unresolved.

No source data was modified, no models were retrained, and production remains
disabled.
"""
(OUT / "phase17_10C_package_verification.md").write_text(
    summary_md, encoding="utf-8"
)

print("\n[2] QUALITY CHECKS")
print("Figure PNG files validated:", sum(x["png_opens"] for x in figure_checks), "/ 7")
print("Figure PDF files validated:", sum(x["pdf_exists"] for x in figure_checks), "/ 7")
print("Table CSV files read:", sum(x["readable"] for x in table_checks), "/ 9")
print("Markdown image links checked:", len(image_links))
print("References heading found:", has_references_heading)
print("Warnings:", len(warnings_found))
for warning in warnings_found:
    print(" -", warning)

print("\n[3] NEW REVIEW PACKAGE")
print("ZIP:", zip_path)
print("SHA-256:", zip_hash)
print("ZIP integrity: PASS")
print("ZIP entries:", len(package_members))
print("PNG/PDF figures:", len(zipped_pngs), "/", len(zipped_pdfs))
print("Main table CSVs:", len(zipped_table_csvs))
print("Supporting evidence files copied:", len(copied_evidence))
print("Verification report:", report_path)

print("\n[4] RELEASE GATE")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data modified: False")
print("Models retrained: False")
print("Original review ZIP modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.10D — DOCX/PDF REVIEW EXPORT + PAGE RENDER QA
# Creates review exports only. Visual inspection is still required.

from pathlib import Path
from datetime import datetime
import subprocess
import hashlib
import json
import shutil
import sys
import zipfile
import importlib.util

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

INTEGRATED_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10B_R1_integrated_manuscript_20261025T184741"
)
MANUSCRIPT = INTEGRATED_DIR / "Delhi_AQI_Integrated_Manuscript.md"
FIG_DIR = INTEGRATED_DIR / "figures"
TABLE_DIR = INTEGRATED_DIR / "tables_csv"

REVIEW_ZIP = (
    ROOT / "outputs/research_paper"
    / "phase17_10C_final_review_package_20261025T184853"
    / "Delhi_AQI_Integrated_Research_Review_Package.zip"
)
EXPECTED_REVIEW_ZIP_SHA256 = (
    "e60206d8958bfcc0b97e39290e7fa7e63386bdc6570d5b16bb8961ca19d22e59"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def run_command(command, description):
    print(f"\n{description}")
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        raise RuntimeError(
            f"{description} failed with exit code {result.returncode}"
        )
    return result.stdout

print("=" * 78)
print("PHASE 17.10D — DOCX/PDF EXPORT AND PAGE RENDER QA")
print("=" * 78)

# 1. Preflight all known input paths and verify the Phase 17.10C ZIP.
for path in [MANUSCRIPT, REVIEW_ZIP]:
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

if not FIG_DIR.is_dir() or not TABLE_DIR.is_dir():
    raise FileNotFoundError(
        "The integrated figure or table folder is missing."
    )

review_zip_hash = sha256_file(REVIEW_ZIP)
if review_zip_hash != EXPECTED_REVIEW_ZIP_SHA256:
    raise RuntimeError(
        "Phase 17.10C ZIP hash did not match the recorded SHA-256."
    )

with zipfile.ZipFile(REVIEW_ZIP, "r") as z:
    bad_member = z.testzip()
    if bad_member:
        raise RuntimeError(f"Review ZIP integrity failure: {bad_member}")
    review_zip_entries = z.namelist()

pngs = sorted(FIG_DIR.glob("*.png"))
pdf_figures = sorted(FIG_DIR.glob("*.pdf"))
table_csvs = sorted(TABLE_DIR.glob("*.csv"))

if len(pngs) != 7 or len(pdf_figures) != 7 or len(table_csvs) != 9:
    raise RuntimeError(
        f"Input counts differ from expectation: PNG={len(pngs)}, "
        f"PDF figures={len(pdf_figures)}, table CSVs={len(table_csvs)}."
    )

print("\n[1] PREFLIGHT")
print("Phase 17.10C review ZIP SHA-256 verified:", True)
print("Review ZIP entries:", len(review_zip_entries))
print("Integrated manuscript exists:", MANUSCRIPT.exists())
print("Figures: 7 PNG + 7 PDF")
print("Main table CSVs:", len(table_csvs))

# 2. Install conversion / render dependencies if needed.
packages = [
    ("pypandoc_binary", "pypandoc"),
    ("python-docx", "docx"),
    ("PyMuPDF", "fitz"),
    ("Pillow", "PIL"),
]

pip_needed = False
for package_name, import_name in packages:
    if importlib.util.find_spec(import_name) is None:
        pip_needed = True

# pypandoc may exist without a Pandoc binary, so ensure the binary package.
try:
    import pypandoc
    pandoc_path = shutil.which("pandoc") or pypandoc.get_pandoc_path()
    if not pandoc_path or not Path(pandoc_path).exists():
        pip_needed = True
except Exception:
    pip_needed = True

if pip_needed:
    run_command(
        [sys.executable, "-m", "pip", "install", "-q",
         "pypandoc_binary", "python-docx", "PyMuPDF", "Pillow"],
        "Installing document export and rendering dependencies"
    )

import pypandoc
import fitz
from docx import Document
from docx.shared import Inches, Pt
from PIL import Image, ImageOps, ImageDraw

pandoc_path = shutil.which("pandoc") or pypandoc.get_pandoc_path()
if not pandoc_path or not Path(pandoc_path).exists():
    raise RuntimeError(
        "Pandoc executable is unavailable after dependency setup."
    )

lo_path = shutil.which("libreoffice") or shutil.which("soffice")
if not lo_path:
    raise RuntimeError(
        "LibreOffice is not installed in this runtime. Install LibreOffice "
        "Writer in Colab, then rerun this phase."
    )

# 3. Create a distinct review-export directory.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10D_rendered_review_export_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

PAGES_DIR = OUT / "rendered_pages"
PAGES_DIR.mkdir()

docx_path = OUT / "Delhi_AQI_Integrated_Review_Draft.docx"
pdf_path = OUT / "Delhi_AQI_Integrated_Review_Draft.pdf"

# 4. Convert Markdown to DOCX. Resource paths include the manuscript's
# directory so relative links such as figures/Figure_01_*.png can resolve.
run_command(
    [
        pandoc_path,
        str(MANUSCRIPT),
        "--from=markdown+pipe_tables",
        "--to=docx",
        "--standalone",
        f"--resource-path={INTEGRATED_DIR}",
        "--output", str(docx_path),
    ],
    "Converting integrated manuscript to DOCX"
)

if not docx_path.is_file() or docx_path.stat().st_size == 0:
    raise RuntimeError("DOCX export was not created or is empty.")

# 5. Apply conservative, consistent default text formatting.
# This is a review draft, not a journal-specific template.
doc = Document(str(docx_path))

normal = doc.styles["Normal"]
normal.font.name = "Times New Roman"
normal.font.size = Pt(10.5)
normal.paragraph_format.space_after = Pt(5)
normal.paragraph_format.line_spacing = 1.12

for style_name, size in [
    ("Title", 18),
    ("Heading 1", 15),
    ("Heading 2", 12.5),
    ("Heading 3", 11),
]:
    if style_name in doc.styles:
        style = doc.styles[style_name]
        style.font.name = "Times New Roman"
        style.font.size = Pt(size)

for section in doc.sections:
    section.top_margin = Inches(0.8)
    section.bottom_margin = Inches(0.8)
    section.left_margin = Inches(0.85)
    section.right_margin = Inches(0.85)

doc.save(docx_path)

# Check the generated DOCX structure.
doc_check = Document(str(docx_path))
embedded_images = len(doc_check.inline_shapes)
word_table_count = len(doc_check.tables)
paragraph_count = len(doc_check.paragraphs)

# A successful DOCX export must include the seven intended figures.
if embedded_images < 7:
    raise RuntimeError(
        f"Only {embedded_images} embedded DOCX images detected; expected at least 7."
    )

if word_table_count < 9:
    raise RuntimeError(
        f"Only {word_table_count} DOCX tables detected; expected at least 9."
    )

print("\n[2] DOCX STRUCTURE")
print("DOCX size (bytes):", docx_path.stat().st_size)
print("Embedded images detected:", embedded_images)
print("Word tables detected:", word_table_count)
print("Paragraphs detected:", paragraph_count)

# 6. Convert DOCX to PDF using a dedicated LibreOffice profile.
lo_profile = Path("/tmp") / f"lo_profile_phase17_10D_{stamp}"
lo_uri = lo_profile.as_uri()

run_command(
    [
        lo_path,
        f"-env:UserInstallation={lo_uri}",
        "--headless",
        "--convert-to", "pdf",
        "--outdir", str(OUT),
        str(docx_path),
    ],
    "Converting DOCX to PDF"
)

if not pdf_path.is_file() or pdf_path.stat().st_size == 0:
    raise RuntimeError("PDF export was not created or is empty.")

# 7. Render every PDF page to PNG at review resolution.
pdf_doc = fitz.open(pdf_path)
page_records = []

for idx, page in enumerate(pdf_doc):
    pix = page.get_pixmap(matrix=fitz.Matrix(1.8, 1.8), alpha=False)
    page_path = PAGES_DIR / f"page-{idx + 1:03d}.png"
    pix.save(page_path)

    page_records.append({
        "page": idx + 1,
        "width_pixels": pix.width,
        "height_pixels": pix.height,
        "render_png": str(page_path),
    })

page_count = len(pdf_doc)
pdf_doc.close()

if page_count == 0 or len(list(PAGES_DIR.glob("page-*.png"))) != page_count:
    raise RuntimeError("Not every PDF page was rendered successfully.")

# 8. Build contact sheets with up to 12 pages per sheet. Also retain
# the full-size page images for page-by-page inspection.
contact_dir = OUT / "contact_sheets"
contact_dir.mkdir()

pages = [Path(row["render_png"]) for row in page_records]
contact_paths = []
chunk_size = 12
thumb_size = (340, 440)
columns = 3
rows_per_sheet = 4
cell_w, cell_h = 360, 475

for start in range(0, len(pages), chunk_size):
    chunk = pages[start:start + chunk_size]
    sheet = Image.new(
        "RGB",
        (columns * cell_w, rows_per_sheet * cell_h),
        "white"
    )
    draw = ImageDraw.Draw(sheet)

    for offset, page_path in enumerate(chunk):
        image = Image.open(page_path).convert("RGB")
        image.thumbnail(thumb_size)

        col = offset % columns
        row = offset // columns
        x = col * cell_w + (cell_w - image.width) // 2
        y = row * cell_h + 22
        sheet.paste(image, (x, y))

        page_number = start + offset + 1
        draw.text(
            (col * cell_w + 8, row * cell_h + 4),
            f"Page {page_number}",
            fill="black"
        )

    contact_path = contact_dir / f"contact-sheet-{start // chunk_size + 1:02d}.png"
    sheet.save(contact_path)
    contact_paths.append(contact_path)

# 9. Save export quality checks. These are structural, not a claim of
# human visual sign-off.
qa = {
    "phase": "17.10D",
    "created_at": datetime.now().astimezone().isoformat(),
    "input_markdown": str(MANUSCRIPT),
    "phase17_10C_review_zip": str(REVIEW_ZIP),
    "phase17_10C_review_zip_sha256": review_zip_hash,
    "phase17_10C_review_zip_sha256_verified": True,
    "docx_path": str(docx_path),
    "docx_bytes": docx_path.stat().st_size,
    "pdf_path": str(pdf_path),
    "pdf_bytes": pdf_path.stat().st_size,
    "pdf_page_count": page_count,
    "rendered_png_page_count": len(page_records),
    "contact_sheets": [str(path) for path in contact_paths],
    "embedded_docx_image_count": embedded_images,
    "word_table_count": word_table_count,
    "paragraph_count": paragraph_count,
    "all_pdf_pages_rendered": len(page_records) == page_count,
    "human_visual_review_completed": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "frozen_models_retrained": False,
    "original_review_zip_modified": False,
    "note": (
        "Page renders and contact sheets are provided for visual inspection. "
        "A structurally valid export is not a substitute for reviewing every "
        "page for clipping, table overflow, figure placement, and readability."
    ),
}

qa_path = OUT / "phase17_10D_render_qa.json"
qa_path.write_text(
    json.dumps(qa, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

page_csv = OUT / "rendered_page_inventory.csv"
pd.DataFrame(page_records).to_csv(page_csv, index=False)

readme = f"""# Phase 17.10D — Rendered Review Export

- DOCX: `{docx_path.name}`
- PDF: `{pdf_path.name}`
- PDF pages rendered: {page_count}
- Full page PNGs: `rendered_pages/`
- Contact sheets: `contact_sheets/`
- QA report: `phase17_10D_render_qa.json`

This is a review draft, not a final submission export. Inspect all rendered
pages for clipping, broken figures, split/wide tables, unreadable captions,
unexpected page breaks, and bibliography issues.

The licence gate is still open. Production remains disabled.
"""

(OUT / "README.md").write_text(readme, encoding="utf-8")

print("\n[3] PDF AND RENDER CHECKS")
print("DOCX created:", docx_path.exists())
print("PDF created:", pdf_path.exists())
print("PDF page count:", page_count)
print("Pages rendered to PNG:", len(page_records))
print("Contact sheets created:", len(contact_paths))

print("\n[4] REVIEW OUTPUTS")
print("Output folder:", OUT)
print("DOCX:", docx_path)
print("PDF:", pdf_path)
print("Full page renders:", PAGES_DIR)
print("Contact sheets:", contact_dir)
print("QA report:", qa_path)
print("Page inventory:", page_csv)

print("\n[5] RELEASE GATE")
print("Human visual QA completed: False — review the rendered pages")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data modified: False")
print("Models retrained: False")
print("Original review ZIP modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.10D-R1 — INSTALL AND VERIFY DOCUMENT EXPORT DEPENDENCIES

import subprocess
import shutil

print("=" * 78)
print("PHASE 17.10D-R1 — LIBREOFFICE REPAIR")
print("=" * 78)

print("\n[1] Updating package index...")
update = subprocess.run(
    ["apt-get", "update", "-qq"],
    capture_output=True,
    text=True
)

if update.returncode != 0:
    raise RuntimeError(
        "Package index update failed:\n"
        + (update.stderr or update.stdout)[-5000:]
    )

packages = [
    "libreoffice-writer",
    "libreoffice-calc",
    "fonts-liberation",
    "fonts-dejavu-core",
]

print("\n[2] Installing LibreOffice and fonts...")
install = subprocess.run(
    ["apt-get", "install", "-y", "-qq"] + packages,
    capture_output=True,
    text=True
)

if install.returncode != 0:
    raise RuntimeError(
        "LibreOffice installation failed:\n"
        + (install.stderr or install.stdout)[-7000:]
    )

print("\n[3] Verifying installation...")
lo_path = shutil.which("libreoffice") or shutil.which("soffice")

if not lo_path:
    raise RuntimeError(
        "Installation completed, but the LibreOffice executable "
        "could not be located. Do not proceed with document export yet."
    )

version = subprocess.run(
    [lo_path, "--headless", "--version"],
    capture_output=True,
    text=True,
    timeout=60
)

if version.returncode != 0:
    raise RuntimeError(
        "LibreOffice was found but its version check failed:\n"
        + (version.stderr or version.stdout)[-3000:]
    )

print("LibreOffice executable:", lo_path)
print("Version:", (version.stdout or version.stderr).strip())
print("Installation status: PASS")
print("Google Drive source data modified: False")
print("Frozen models modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.10D — DOCX/PDF REVIEW EXPORT + PAGE RENDER QA
# Creates review exports only. Visual inspection is still required.

from pathlib import Path
from datetime import datetime
import subprocess
import hashlib
import json
import shutil
import sys
import zipfile
import importlib.util

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

INTEGRATED_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10B_R1_integrated_manuscript_20261025T184741"
)
MANUSCRIPT = INTEGRATED_DIR / "Delhi_AQI_Integrated_Manuscript.md"
FIG_DIR = INTEGRATED_DIR / "figures"
TABLE_DIR = INTEGRATED_DIR / "tables_csv"

REVIEW_ZIP = (
    ROOT / "outputs/research_paper"
    / "phase17_10C_final_review_package_20261025T184853"
    / "Delhi_AQI_Integrated_Research_Review_Package.zip"
)
EXPECTED_REVIEW_ZIP_SHA256 = (
    "e60206d8958bfcc0b97e39290e7fa7e63386bdc6570d5b16bb8961ca19d22e59"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def run_command(command, description):
    print(f"\n{description}")
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        raise RuntimeError(
            f"{description} failed with exit code {result.returncode}"
        )
    return result.stdout

print("=" * 78)
print("PHASE 17.10D — DOCX/PDF EXPORT AND PAGE RENDER QA")
print("=" * 78)

# 1. Preflight all known input paths and verify the Phase 17.10C ZIP.
for path in [MANUSCRIPT, REVIEW_ZIP]:
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

if not FIG_DIR.is_dir() or not TABLE_DIR.is_dir():
    raise FileNotFoundError(
        "The integrated figure or table folder is missing."
    )

review_zip_hash = sha256_file(REVIEW_ZIP)
if review_zip_hash != EXPECTED_REVIEW_ZIP_SHA256:
    raise RuntimeError(
        "Phase 17.10C ZIP hash did not match the recorded SHA-256."
    )

with zipfile.ZipFile(REVIEW_ZIP, "r") as z:
    bad_member = z.testzip()
    if bad_member:
        raise RuntimeError(f"Review ZIP integrity failure: {bad_member}")
    review_zip_entries = z.namelist()

pngs = sorted(FIG_DIR.glob("*.png"))
pdf_figures = sorted(FIG_DIR.glob("*.pdf"))
table_csvs = sorted(TABLE_DIR.glob("*.csv"))

if len(pngs) != 7 or len(pdf_figures) != 7 or len(table_csvs) != 9:
    raise RuntimeError(
        f"Input counts differ from expectation: PNG={len(pngs)}, "
        f"PDF figures={len(pdf_figures)}, table CSVs={len(table_csvs)}."
    )

print("\n[1] PREFLIGHT")
print("Phase 17.10C review ZIP SHA-256 verified:", True)
print("Review ZIP entries:", len(review_zip_entries))
print("Integrated manuscript exists:", MANUSCRIPT.exists())
print("Figures: 7 PNG + 7 PDF")
print("Main table CSVs:", len(table_csvs))

# 2. Install conversion / render dependencies if needed.
packages = [
    ("pypandoc_binary", "pypandoc"),
    ("python-docx", "docx"),
    ("PyMuPDF", "fitz"),
    ("Pillow", "PIL"),
]

pip_needed = False
for package_name, import_name in packages:
    if importlib.util.find_spec(import_name) is None:
        pip_needed = True

# pypandoc may exist without a Pandoc binary, so ensure the binary package.
try:
    import pypandoc
    pandoc_path = shutil.which("pandoc") or pypandoc.get_pandoc_path()
    if not pandoc_path or not Path(pandoc_path).exists():
        pip_needed = True
except Exception:
    pip_needed = True

if pip_needed:
    run_command(
        [sys.executable, "-m", "pip", "install", "-q",
         "pypandoc_binary", "python-docx", "PyMuPDF", "Pillow"],
        "Installing document export and rendering dependencies"
    )

import pypandoc
import fitz
from docx import Document
from docx.shared import Inches, Pt
from PIL import Image, ImageOps, ImageDraw

pandoc_path = shutil.which("pandoc") or pypandoc.get_pandoc_path()
if not pandoc_path or not Path(pandoc_path).exists():
    raise RuntimeError(
        "Pandoc executable is unavailable after dependency setup."
    )

lo_path = shutil.which("libreoffice") or shutil.which("soffice")
if not lo_path:
    raise RuntimeError(
        "LibreOffice is not installed in this runtime. Install LibreOffice "
        "Writer in Colab, then rerun this phase."
    )

# 3. Create a distinct review-export directory.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10D_rendered_review_export_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

PAGES_DIR = OUT / "rendered_pages"
PAGES_DIR.mkdir()

docx_path = OUT / "Delhi_AQI_Integrated_Review_Draft.docx"
pdf_path = OUT / "Delhi_AQI_Integrated_Review_Draft.pdf"

# 4. Convert Markdown to DOCX. Resource paths include the manuscript's
# directory so relative links such as figures/Figure_01_*.png can resolve.
run_command(
    [
        pandoc_path,
        str(MANUSCRIPT),
        "--from=markdown+pipe_tables",
        "--to=docx",
        "--standalone",
        f"--resource-path={INTEGRATED_DIR}",
        "--output", str(docx_path),
    ],
    "Converting integrated manuscript to DOCX"
)

if not docx_path.is_file() or docx_path.stat().st_size == 0:
    raise RuntimeError("DOCX export was not created or is empty.")

# 5. Apply conservative, consistent default text formatting.
# This is a review draft, not a journal-specific template.
doc = Document(str(docx_path))

normal = doc.styles["Normal"]
normal.font.name = "Times New Roman"
normal.font.size = Pt(10.5)
normal.paragraph_format.space_after = Pt(5)
normal.paragraph_format.line_spacing = 1.12

for style_name, size in [
    ("Title", 18),
    ("Heading 1", 15),
    ("Heading 2", 12.5),
    ("Heading 3", 11),
]:
    if style_name in doc.styles:
        style = doc.styles[style_name]
        style.font.name = "Times New Roman"
        style.font.size = Pt(size)

for section in doc.sections:
    section.top_margin = Inches(0.8)
    section.bottom_margin = Inches(0.8)
    section.left_margin = Inches(0.85)
    section.right_margin = Inches(0.85)

doc.save(docx_path)

# Check the generated DOCX structure.
doc_check = Document(str(docx_path))
embedded_images = len(doc_check.inline_shapes)
word_table_count = len(doc_check.tables)
paragraph_count = len(doc_check.paragraphs)

# A successful DOCX export must include the seven intended figures.
if embedded_images < 7:
    raise RuntimeError(
        f"Only {embedded_images} embedded DOCX images detected; expected at least 7."
    )

if word_table_count < 9:
    raise RuntimeError(
        f"Only {word_table_count} DOCX tables detected; expected at least 9."
    )

print("\n[2] DOCX STRUCTURE")
print("DOCX size (bytes):", docx_path.stat().st_size)
print("Embedded images detected:", embedded_images)
print("Word tables detected:", word_table_count)
print("Paragraphs detected:", paragraph_count)

# 6. Convert DOCX to PDF using a dedicated LibreOffice profile.
lo_profile = Path("/tmp") / f"lo_profile_phase17_10D_{stamp}"
lo_uri = lo_profile.as_uri()

run_command(
    [
        lo_path,
        f"-env:UserInstallation={lo_uri}",
        "--headless",
        "--convert-to", "pdf",
        "--outdir", str(OUT),
        str(docx_path),
    ],
    "Converting DOCX to PDF"
)

if not pdf_path.is_file() or pdf_path.stat().st_size == 0:
    raise RuntimeError("PDF export was not created or is empty.")

# 7. Render every PDF page to PNG at review resolution.
pdf_doc = fitz.open(pdf_path)
page_records = []

for idx, page in enumerate(pdf_doc):
    pix = page.get_pixmap(matrix=fitz.Matrix(1.8, 1.8), alpha=False)
    page_path = PAGES_DIR / f"page-{idx + 1:03d}.png"
    pix.save(page_path)

    page_records.append({
        "page": idx + 1,
        "width_pixels": pix.width,
        "height_pixels": pix.height,
        "render_png": str(page_path),
    })

page_count = len(pdf_doc)
pdf_doc.close()

if page_count == 0 or len(list(PAGES_DIR.glob("page-*.png"))) != page_count:
    raise RuntimeError("Not every PDF page was rendered successfully.")

# 8. Build contact sheets with up to 12 pages per sheet. Also retain
# the full-size page images for page-by-page inspection.
contact_dir = OUT / "contact_sheets"
contact_dir.mkdir()

pages = [Path(row["render_png"]) for row in page_records]
contact_paths = []
chunk_size = 12
thumb_size = (340, 440)
columns = 3
rows_per_sheet = 4
cell_w, cell_h = 360, 475

for start in range(0, len(pages), chunk_size):
    chunk = pages[start:start + chunk_size]
    sheet = Image.new(
        "RGB",
        (columns * cell_w, rows_per_sheet * cell_h),
        "white"
    )
    draw = ImageDraw.Draw(sheet)

    for offset, page_path in enumerate(chunk):
        image = Image.open(page_path).convert("RGB")
        image.thumbnail(thumb_size)

        col = offset % columns
        row = offset // columns
        x = col * cell_w + (cell_w - image.width) // 2
        y = row * cell_h + 22
        sheet.paste(image, (x, y))

        page_number = start + offset + 1
        draw.text(
            (col * cell_w + 8, row * cell_h + 4),
            f"Page {page_number}",
            fill="black"
        )

    contact_path = contact_dir / f"contact-sheet-{start // chunk_size + 1:02d}.png"
    sheet.save(contact_path)
    contact_paths.append(contact_path)

# 9. Save export quality checks. These are structural, not a claim of
# human visual sign-off.
qa = {
    "phase": "17.10D",
    "created_at": datetime.now().astimezone().isoformat(),
    "input_markdown": str(MANUSCRIPT),
    "phase17_10C_review_zip": str(REVIEW_ZIP),
    "phase17_10C_review_zip_sha256": review_zip_hash,
    "phase17_10C_review_zip_sha256_verified": True,
    "docx_path": str(docx_path),
    "docx_bytes": docx_path.stat().st_size,
    "pdf_path": str(pdf_path),
    "pdf_bytes": pdf_path.stat().st_size,
    "pdf_page_count": page_count,
    "rendered_png_page_count": len(page_records),
    "contact_sheets": [str(path) for path in contact_paths],
    "embedded_docx_image_count": embedded_images,
    "word_table_count": word_table_count,
    "paragraph_count": paragraph_count,
    "all_pdf_pages_rendered": len(page_records) == page_count,
    "human_visual_review_completed": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "frozen_models_retrained": False,
    "original_review_zip_modified": False,
    "note": (
        "Page renders and contact sheets are provided for visual inspection. "
        "A structurally valid export is not a substitute for reviewing every "
        "page for clipping, table overflow, figure placement, and readability."
    ),
}

qa_path = OUT / "phase17_10D_render_qa.json"
qa_path.write_text(
    json.dumps(qa, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

page_csv = OUT / "rendered_page_inventory.csv"
pd.DataFrame(page_records).to_csv(page_csv, index=False)

readme = f"""# Phase 17.10D — Rendered Review Export

- DOCX: `{docx_path.name}`
- PDF: `{pdf_path.name}`
- PDF pages rendered: {page_count}
- Full page PNGs: `rendered_pages/`
- Contact sheets: `contact_sheets/`
- QA report: `phase17_10D_render_qa.json`

This is a review draft, not a final submission export. Inspect all rendered
pages for clipping, broken figures, split/wide tables, unreadable captions,
unexpected page breaks, and bibliography issues.

The licence gate is still open. Production remains disabled.
"""

(OUT / "README.md").write_text(readme, encoding="utf-8")

print("\n[3] PDF AND RENDER CHECKS")
print("DOCX created:", docx_path.exists())
print("PDF created:", pdf_path.exists())
print("PDF page count:", page_count)
print("Pages rendered to PNG:", len(page_records))
print("Contact sheets created:", len(contact_paths))

print("\n[4] REVIEW OUTPUTS")
print("Output folder:", OUT)
print("DOCX:", docx_path)
print("PDF:", pdf_path)
print("Full page renders:", PAGES_DIR)
print("Contact sheets:", contact_dir)
print("QA report:", qa_path)
print("Page inventory:", page_csv)

print("\n[5] RELEASE GATE")
print("Human visual QA completed: False — review the rendered pages")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data modified: False")
print("Models retrained: False")
print("Original review ZIP modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.10E — HUMAN VISUAL QA WORKSPACE
# Read-only review of Phase 17.10D renders; creates separate audit files.

from pathlib import Path
from datetime import datetime
import json
import re
import csv
import numpy as np
from PIL import Image
from IPython.display import display, Markdown, Image as NotebookImage

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

EXPORT_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10D_rendered_review_export_20261025T185523"
)

PDF_PATH = EXPORT_DIR / "Delhi_AQI_Integrated_Review_Draft.pdf"
DOCX_PATH = EXPORT_DIR / "Delhi_AQI_Integrated_Review_Draft.docx"
RENDER_DIR = EXPORT_DIR / "rendered_pages"
CONTACT_DIR = EXPORT_DIR / "contact_sheets"

print("=" * 78)
print("PHASE 17.10E — HUMAN VISUAL QA WORKSPACE")
print("=" * 78)

# 1. Verify existing export and render assets.
for path in [PDF_PATH, DOCX_PATH, RENDER_DIR, CONTACT_DIR]:
    if not path.exists():
        raise FileNotFoundError(f"Required Phase 17.10D asset missing: {path}")

render_files = sorted(
    RENDER_DIR.glob("*.png"),
    key=lambda p: (
        int(re.findall(r"\d+", p.stem)[-1])
        if re.findall(r"\d+", p.stem) else 0
    )
)

contact_files = sorted(CONTACT_DIR.glob("*.png"))

if not render_files:
    raise FileNotFoundError(f"No page PNGs found in {RENDER_DIR}")
if not contact_files:
    raise FileNotFoundError(f"No contact-sheet PNGs found in {CONTACT_DIR}")

# 2. Read PDF page text and collect conservative visual screening metrics.
try:
    import pymupdf as fitz
except ImportError:
    import fitz

doc = fitz.open(PDF_PATH)
pdf_page_count = len(doc)

if pdf_page_count != len(render_files):
    print(
        "WARNING: PDF page count and rendered PNG count differ:",
        pdf_page_count, "vs", len(render_files)
    )

def page_number(path, fallback):
    nums = re.findall(r"\d+", path.stem)
    return int(nums[-1]) if nums else fallback

rows = []
for idx, render_path in enumerate(render_files, start=1):
    page_no = page_number(render_path, idx)
    page_text_chars = None
    page_images = None

    if 1 <= page_no <= pdf_page_count:
        page = doc[page_no - 1]
        page_text_chars = len(page.get_text("text").strip())
        page_images = len(page.get_images(full=True))

    with Image.open(render_path) as im:
        gray = np.asarray(im.convert("L"))
        near_white_ratio = float(np.mean(gray >= 248))
        dark_pixel_ratio = float(np.mean(gray < 100))
        pixel_std = float(np.std(gray))
        dimensions = f"{im.width}x{im.height}"

    flags = []
    if near_white_ratio > 0.997:
        flags.append("POSSIBLY BLANK PAGE — inspect")
    if page_text_chars is not None and page_text_chars < 40:
        flags.append("LOW EXTRACTED TEXT — inspect page; may be figure-led")
    if pixel_std < 3:
        flags.append("VERY LOW VISUAL VARIATION — inspect")

    rows.append({
        "page": page_no,
        "render_file": str(render_path),
        "image_dimensions": dimensions,
        "pdf_text_characters": page_text_chars,
        "pdf_embedded_image_count": page_images,
        "near_white_pixel_ratio": round(near_white_ratio, 5),
        "dark_pixel_ratio": round(dark_pixel_ratio, 5),
        "pixel_standard_deviation": round(pixel_std, 3),
        "automated_flags": "; ".join(flags),
        "manual_review_status": "PENDING",
        "no_clipping_or_overlap": "PENDING",
        "figures_readable": "PENDING",
        "tables_readable": "PENDING",
        "headings_and_page_breaks_ok": "PENDING",
        "captions_and_references_ok": "PENDING",
        "issues_and_notes": "",
    })

doc.close()

# 3. Create checklist/report in a new folder.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
QA_DIR = (
    ROOT / "outputs/research_paper"
    / f"phase17_10E_human_visual_qa_{stamp}"
)
QA_DIR.mkdir(parents=True, exist_ok=False)

checklist_path = QA_DIR / "phase17_10E_page_review_checklist.csv"
with open(checklist_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

summary = {
    "phase": "17.10E",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_docx": str(DOCX_PATH),
    "source_pdf": str(PDF_PATH),
    "pdf_pages": pdf_page_count,
    "rendered_page_pngs": len(render_files),
    "contact_sheets": [str(p) for p in contact_files],
    "automated_screening_flags": [
        {
            "page": r["page"],
            "flags": r["automated_flags"]
        }
        for r in rows if r["automated_flags"]
    ],
    "manual_visual_qa_completed": False,
    "licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "previous_exports_modified": False,
    "checklist_path": str(checklist_path),
}

summary_path = QA_DIR / "phase17_10E_visual_qa_status.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# 4. Display contact sheets directly in the notebook for visual inspection.
display(Markdown("## Contact sheets — inspect all pages"))
for contact in contact_files:
    display(Markdown(f"### {contact.name}"))
    display(NotebookImage(filename=str(contact)))

# 5. Print review index.
print("\n[1] RENDER COUNTS")
print("PDF pages:", pdf_page_count)
print("Rendered PNG pages:", len(render_files))
print("Contact sheets:", len(contact_files))
print("Page count consistent:", pdf_page_count == len(render_files))

print("\n[2] AUTOMATED SCREENING FLAGS")
flagged = [r for r in rows if r["automated_flags"]]
if flagged:
    for row in flagged:
        print(f"Page {row['page']}: {row['automated_flags']}")
else:
    print("No obvious blank/low-variation pages flagged by simple heuristics.")

print("\n[3] MANUAL CHECKLIST")
print(checklist_path)
print("Status JSON:", summary_path)
print("QA folder:", QA_DIR)

print("\n[4] RELEASE GATE")
print("Human visual QA completed: False — inspect the contact sheets/pages")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.10F — DISPLAY CONTACT SHEETS FOR VISUAL REVIEW
# Read-only: displays rendered pages and loads the checklist.

from pathlib import Path
from IPython.display import display, Image
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

EXPORT_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10D_rendered_review_export_20261025T185523"
)

QA_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10E_human_visual_qa_20261025T185812"
)

CONTACT_DIR = EXPORT_DIR / "contact_sheets"
PAGE_DIR = EXPORT_DIR / "rendered_pages"
CHECKLIST = QA_DIR / "phase17_10E_page_review_checklist.csv"

print("=" * 72)
print("PHASE 17.10F — VISUAL REVIEW WORKSPACE")
print("=" * 72)

# Display all available contact sheets inside Colab.
contact_sheets = sorted(CONTACT_DIR.glob("*.png"))

if not contact_sheets:
    raise FileNotFoundError(
        f"No contact sheets found in: {CONTACT_DIR}"
    )

for path in contact_sheets:
    print(f"\nContact sheet: {path.name}")
    display(Image(filename=str(path)))

# List the full-resolution pages for closer inspection.
pages = sorted(
    PAGE_DIR.glob("*.png"),
    key=lambda p: int("".join(filter(str.isdigit, p.stem)) or 0)
)
print(f"\nFull-resolution page renders available: {len(pages)}")

# Load the existing checklist without changing it.
if CHECKLIST.exists():
    checklist = pd.read_csv(CHECKLIST)
    print("\nExisting page review checklist:")
    print("Columns:", checklist.columns.tolist())
    display(checklist)
else:
    print(f"\nChecklist not found: {CHECKLIST}")

print("\nReview all 19 pages, using the contact sheets first.")
print("Open individual page PNGs at full size wherever anything looks unclear.")
print("=" * 72)

In [ ]:
# PHASE 17.10F-R1 — PREPARE CONTACT SHEETS FOR VISUAL REVIEW
# Read-only inspection; does not change the manuscript, PDF, or checklist.

from pathlib import Path
from IPython.display import display, Image
import shutil
import json

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
SEARCH_ROOT = ROOT / "outputs" / "research_paper"

# Find contact sheets without assuming a particular phase folder name.
all_sheets = [
    p for p in SEARCH_ROOT.rglob("*.png")
    if "contact" in p.name.lower() and "sheet" in p.name.lower()
]

if not all_sheets:
    raise FileNotFoundError(
        "No contact-sheet PNGs found under outputs/research_paper. "
        "Check that Google Drive is mounted and the Phase 17.10E/F "
        "visual QA outputs still exist."
    )

# Prefer the newest folder containing at least two contact sheets.
by_folder = {}
for p in all_sheets:
    by_folder.setdefault(p.parent, []).append(p)

eligible = [
    (folder, paths)
    for folder, paths in by_folder.items()
    if len(paths) >= 2
]

if not eligible:
    raise RuntimeError(
        "Found contact sheets, but no single folder contains at least two. "
        "Please inspect the printed candidates below:\n"
        + "\n".join(str(p) for p in all_sheets)
    )

folder, sheets = max(
    eligible,
    key=lambda item: max(p.stat().st_mtime for p in item[1])
)
sheets = sorted(sheets, key=lambda p: p.name)[:2]

# Locate the most recent review checklist, without editing it.
checklists = list(SEARCH_ROOT.rglob("*page*review*checklist*.csv"))
checklist = (
    max(checklists, key=lambda p: p.stat().st_mtime)
    if checklists else None
)

UPLOAD_DIR = Path("/content/phase17_10F_contact_sheets_for_review")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

prepared = []
for i, source in enumerate(sheets, start=1):
    dest = UPLOAD_DIR / f"contact-sheet-{i:02d}.png"
    shutil.copy2(source, dest)
    prepared.append({
        "source": str(source),
        "copy": str(dest),
        "bytes": dest.stat().st_size
    })

manifest = {
    "source_folder": str(folder),
    "contact_sheets": prepared,
    "review_checklist": str(checklist) if checklist else None,
    "human_visual_qa_complete": False,
    "note": "Contact sheets prepared for visual inspection; no QA status changed."
}
(UPLOAD_DIR / "contact_sheet_review_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("=" * 72)
print("PHASE 17.10F-R1 — CONTACT SHEETS READY")
print("=" * 72)
print("Source folder:", folder)
print("Review checklist:", checklist or "Not found")
for item in prepared:
    print(f"\nImage: {item['copy']}")
    print(f"Size: {item['bytes']:,} bytes")
    display(Image(filename=item["copy"]))

print("\nThe two images are ready for upload to this chat.")
print("Human visual QA remains PENDING until the pages have been reviewed.")
print("=" * 72)

# Download both PNGs individually so they can be attached in this conversation.
from google.colab import files
for item in prepared:
    files.download(item["copy"])

In [ ]:
# PHASE 17.10F-R2 — CONTACT-SHEET REVIEW FINDINGS
# Saves observed issues separately. Full-resolution QA remains pending.

from pathlib import Path
from datetime import datetime
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

CHECKLIST = (
    ROOT / "outputs/research_paper"
    / "phase17_10E_human_visual_qa_20261025T185812"
    / "phase17_10E_page_review_checklist.csv"
)

if not CHECKLIST.is_file():
    raise FileNotFoundError(f"Checklist not found: {CHECKLIST}")

checklist = pd.read_csv(CHECKLIST)
if len(checklist) != 19:
    raise RuntimeError(
        f"Expected 19 page-review rows, found {len(checklist)}."
    )

# Findings based on the two supplied contact sheets.
page_findings = {
    1: "Contact-sheet review only. Inspect title, abstract and opening layout at full resolution.",
    2: "Contact-sheet review only. Full-size body-text readability remains unverified.",
    3: "Contact-sheet review only. Check equations, citations and paragraph flow at full size.",
    4: "Result table is dense. Inspect column readability and confirm it is not duplicated later.",
    5: "Several numerical result tables are dense. Inspect readability and check repeated result presentation.",
    6: "AQI-band and station/season tables are dense. Verify labels, units and readability.",
    7: "Transition and feature-attribution tables are dense. Verify headings and numeric readability.",
    8: "Discussion and conclusion layout looks broadly continuous at sheet scale; full-size review still required.",
    9: "LAYOUT ISSUE: Figure 1 starts near the page bottom after data-availability text; the figure/caption block appears split across pages.",
    10: "LAYOUT ISSUE: Figure 1 caption continues from the previous page. Figure heading and caption repeat the title.",
    11: "LAYOUT ISSUE: Figure blocks are crowded. Remove visible PDF file-path lines from the manuscript body.",
    12: "LAYOUT ISSUE: Figure 4 PDF path spills to this page; Figures 5 and 6 are crowded. Keep each figure and caption together.",
    13: "LAYOUT ISSUE: Figure 6 caption/path appears separated from its chart. Table 1 begins near the page bottom and continues.",
    14: "LAYOUT ISSUE: Table 1 continuation lacks a clear fresh table start in the overview; inspect repeated headers. Tables 2 and 3 are dense.",
    15: "LAYOUT ISSUE: Tables 3–5 are tightly packed. Check split rows, headers, confidence intervals and table readability.",
    16: "LAYOUT ISSUE: Tables 6–8 are dense. Check page breaks, column labels and alignment.",
    17: "LAYOUT ISSUE: Station table continuation and Table 9 appear fragmented. Confirm headers and section titles stay with their content.",
    18: "LAYOUT ISSUE: Table 9, interpretation notes and references compete for page space. Review order and spacing.",
    19: "References continue on a sparsely populated page. Check paragraph spacing and citation readability; avoid forced breaks if unnecessary.",
}

def classify_page(page):
    if page in range(1, 9):
        return "CONTACT_SHEET_REVIEWED_FULL_SIZE_PENDING"
    if page in range(9, 19):
        return "LAYOUT_ISSUES_FOUND_REPAIR_REQUIRED"
    return "CONTACT_SHEET_REVIEWED_REFERENCE_FLOW_CHECK"

checklist["contact_sheet_review_status"] = checklist["page"].apply(
    lambda p: classify_page(int(p))
)
checklist["contact_sheet_findings"] = checklist["page"].apply(
    lambda p: page_findings.get(int(p), "Page number not mapped; inspect manually.")
)
checklist["full_resolution_review"] = "PENDING"

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10F_R2_visual_review_findings_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

annotated_path = OUT / "phase17_10F_R2_annotated_page_checklist.csv"
checklist.to_csv(annotated_path, index=False)

issues = [
    {
        "priority": "HIGH",
        "issue": "Figure/caption split across pages",
        "pages": "9–13",
        "recommended_action": "Keep each figure, caption and heading together.",
    },
    {
        "priority": "HIGH",
        "issue": "PDF path strings visible in paper",
        "pages": "11–13",
        "recommended_action": "Remove path strings from manuscript; retain file paths in the manifest.",
    },
    {
        "priority": "HIGH",
        "issue": "Tables fragmented across pages",
        "pages": "13–18",
        "recommended_action": "Keep headings with tables, repeat column headers and avoid splitting individual rows.",
    },
    {
        "priority": "HIGH",
        "issue": "Possible duplicate result tables",
        "pages": "4–8 and 13–18",
        "recommended_action": "Compare the inline result tables with the numbered tables and retain one authoritative presentation.",
    },
    {
        "priority": "MEDIUM",
        "issue": "Repeated figure titles",
        "pages": "9–13",
        "recommended_action": "Use one numbered caption per figure without duplicating the same title.",
    },
    {
        "priority": "MEDIUM",
        "issue": "Dense numerical tables and references",
        "pages": "4–7 and 13–19",
        "recommended_action": "Inspect at full resolution; adjust table layout and spacing only where necessary.",
    },
]

issues_path = OUT / "phase17_10F_R2_layout_issues.csv"
pd.DataFrame(issues).to_csv(issues_path, index=False)

report = {
    "phase": "17.10F-R2",
    "created_at": datetime.now().astimezone().isoformat(),
    "contact_sheets_reviewed": 2,
    "pages_in_contact_sheets": 19,
    "layout_issues_recorded": len(issues),
    "full_resolution_review_complete": False,
    "layout_repair_required": True,
    "submission_ready": False,
    "author_licence_review": "OPEN",
    "source_data_modified": False,
    "models_retrained": False,
    "original_review_package_modified": False,
    "outputs": {
        "annotated_checklist": str(annotated_path),
        "layout_issues": str(issues_path),
    },
}

summary_path = OUT / "phase17_10F_R2_review_status.json"
summary_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("=" * 72)
print("PHASE 17.10F-R2 — VISUAL REVIEW FINDINGS")
print("=" * 72)
print("Pages recorded:", len(checklist))
print("Layout issues logged:", len(issues))
print("Annotated checklist:", annotated_path)
print("Issue register:", issues_path)
print("Status JSON:", summary_path)
print("\nLayout repair required: True")
print("Full-resolution review: PENDING")
print("Author licence review: OPEN")
print("Submission ready: False")
print("=" * 72)

In [ ]:
# PHASE 17.10F-R3 — LAYOUT ISSUE TRIAGE AND REPAIR PLAN
# Reads the six logged findings and creates a traceable repair plan.

from pathlib import Path
from datetime import datetime
import pandas as pd
import json
import re

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

SOURCE_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10F_R2_visual_review_findings_20261025T190412"
)

ISSUES_PATH = SOURCE_DIR / "phase17_10F_R2_layout_issues.csv"
CHECKLIST_PATH = SOURCE_DIR / "phase17_10F_R2_annotated_page_checklist.csv"
STATUS_PATH = SOURCE_DIR / "phase17_10F_R2_review_status.json"

print("=" * 78)
print("PHASE 17.10F-R3 — LAYOUT ISSUE TRIAGE")
print("=" * 78)

# 1. Validate inputs before creating any new files.
for path in [ISSUES_PATH, CHECKLIST_PATH, STATUS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Required QA input not found: {path}")

issues = pd.read_csv(ISSUES_PATH).fillna("")
checklist = pd.read_csv(CHECKLIST_PATH).fillna("")

with open(STATUS_PATH, "r", encoding="utf-8") as f:
    previous_status = json.load(f)

print("\n[1] INPUTS")
print("Issue register rows:", len(issues))
print("Checklist rows:", len(checklist))
print("Issue register columns:", list(issues.columns))

if len(issues) != 6:
    print(
        "WARNING: The current issue register has",
        len(issues),
        "rows; the previous summary reported six issues."
    )

# 2. Discover relevant fields instead of assuming exact column names.
def find_column(frame, keywords):
    for col in frame.columns:
        normalized = re.sub(r"[^a-z0-9]", "", str(col).lower())
        if any(keyword in normalized for keyword in keywords):
            return col
    return None

page_col = find_column(issues, ["page", "pagenumber"])
issue_col = find_column(
    issues,
    ["issue", "finding", "problem", "observation", "note", "description"]
)
severity_col = find_column(issues, ["severity", "priority", "impact"])
category_col = find_column(issues, ["category", "type", "class"])

check_page_col = find_column(checklist, ["page", "pagenumber"])

if page_col is None:
    print(
        "WARNING: No page-number column detected. "
        "The original issue records will still be preserved."
    )

if issue_col is None:
    print(
        "WARNING: No obvious issue-description column detected. "
        "Inspect the printed columns to determine the finding field."
    )

# 3. Match each issue with page-level checklist details where possible.
def to_int_page(value):
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return None

checklist_by_page = {}
if check_page_col is not None:
    for _, row in checklist.iterrows():
        p = to_int_page(row[check_page_col])
        if p is not None:
            checklist_by_page[p] = row.to_dict()

def classify_issue(text):
    s = str(text).lower()

    if any(x in s for x in [
        "clip", "cut off", "overflow", "overlap", "off page",
        "margin", "cropped"
    ]):
        return (
            "Clipping or overlap",
            "Inspect the affected page at full resolution; adjust width, "
            "margins, wrapping, or page-break behavior at the source."
        )

    if any(x in s for x in [
        "table", "column", "cell", "row", "too wide"
    ]):
        return (
            "Table layout",
            "Inspect the table at full resolution; consider a landscape "
            "section, narrower column labels, adjusted column widths, or "
            "a repeated header row."
        )

    if any(x in s for x in [
        "figure", "image", "plot", "chart", "graphic", "readable"
    ]):
        return (
            "Figure readability",
            "Inspect the full-resolution page; adjust figure width, "
            "aspect ratio, caption placement, or page placement as needed."
        )

    if any(x in s for x in [
        "heading", "page break", "orphan", "widow", "section break"
    ]):
        return (
            "Heading or page-break layout",
            "Keep headings with following paragraphs and adjust page or "
            "section breaks so headings are not stranded."
        )

    if any(x in s for x in [
        "caption", "reference", "callout", "numbering"
    ]):
        return (
            "Caption or cross-reference",
            "Check numbering and in-text references against the figure/table "
            "manifest; keep captions adjacent to their assets."
        )

    if any(x in s for x in [
        "blank", "whitespace", "empty space", "sparse"
    ]):
        return (
            "Spacing",
            "Inspect the full page before changing anything; reduce "
            "unnecessary spacing only if it harms the reading flow."
        )

    return (
        "Manual classification required",
        "Inspect the named page and retain the reviewer’s exact finding "
        "before choosing a repair."
    )

repair_rows = []

for idx, row in issues.iterrows():
    page = to_int_page(row[page_col]) if page_col else None
    raw_issue = str(row[issue_col]) if issue_col else ""

    category, suggested_fix = classify_issue(raw_issue)

    linked_check = checklist_by_page.get(page, {}) if page is not None else {}

    repair_rows.append({
        "issue_id": f"R{idx + 1:02d}",
        "page": page if page is not None else "",
        "original_issue_record": json.dumps(
            row.to_dict(), ensure_ascii=False, default=str
        ),
        "issue_description": raw_issue,
        "logged_severity": (
            str(row[severity_col]) if severity_col else ""
        ),
        "logged_category": (
            str(row[category_col]) if category_col else ""
        ),
        "suggested_triage_category": category,
        "suggested_repair": suggested_fix,
        "page_render_file": linked_check.get("render_file", ""),
        "current_manual_status": linked_check.get(
            "manual_review_status", "NOT LINKED"
        ),
        "repair_applied": "NO",
        "reexport_verification": "PENDING",
        "reviewer_notes": "",
    })

# 4. Preserve every issue and checklist value in a new audit folder.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10F_R3_layout_repair_plan_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

repair_df = pd.DataFrame(repair_rows)
plan_path = OUT / "phase17_10F_R3_layout_repair_plan.csv"
repair_df.to_csv(plan_path, index=False)

# Save a copy of the original source register and checklist for traceability.
issues.to_csv(OUT / "original_issue_register_snapshot.csv", index=False)
checklist.to_csv(OUT / "original_page_checklist_snapshot.csv", index=False)

# 5. Write a readable summary with the actual records.
summary_lines = [
    "# Phase 17.10F-R3 — Layout Repair Plan",
    "",
    f"- Source findings: {ISSUES_PATH}",
    f"- Source checklist: {CHECKLIST_PATH}",
    f"- Logged issue count: {len(issues)}",
    f"- Pages in checklist: {len(checklist)}",
    "",
    "## Issues and suggested triage",
    "",
]

for item in repair_rows:
    summary_lines += [
        f"### {item['issue_id']} — Page {item['page'] or 'unspecified'}",
        "",
        f"**Reported finding:** {item['issue_description'] or '(see original issue record)'}",
        "",
        f"**Triage category:** {item['suggested_triage_category']}",
        "",
        f"**Suggested repair:** {item['suggested_repair']}",
        "",
    ]

summary_lines += [
    "## Current release status",
    "",
    "- Layout repair required: YES until all logged issues are resolved.",
    "- Full-resolution page review: PENDING.",
    "- Author licence review: OPEN.",
    "- Submission ready: NO.",
    "- Original reviewed assets changed by this phase: NO.",
]

summary_md = OUT / "phase17_10F_R3_layout_repair_plan.md"
summary_md.write_text("\n".join(summary_lines), encoding="utf-8")

status = {
    "phase": "17.10F-R3",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_issue_register": str(ISSUES_PATH),
    "source_page_checklist": str(CHECKLIST_PATH),
    "issues_triaged": len(repair_rows),
    "repair_plan": str(plan_path),
    "summary": str(summary_md),
    "original_docx_modified": False,
    "original_pdf_modified": False,
    "review_zip_modified": False,
    "models_retrained": False,
    "layout_repair_required": True,
    "full_resolution_review": "PENDING",
    "author_licence_review": "OPEN",
    "submission_ready": False,
}

status_path = OUT / "phase17_10F_R3_status.json"
status_path.write_text(
    json.dumps(status, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8"
)

print("\n[2] FINDINGS — EXACT RECORDS")
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", 160,
    "display.width", 240
):
    print(issues.to_string(index=False))

print("\n[3] TRIAGE PLAN")
for item in repair_rows:
    print(
        f"{item['issue_id']} | page={item['page']} | "
        f"{item['suggested_triage_category']}"
    )
    print("  Finding:", item["issue_description"])
    print("  Suggested repair:", item["suggested_repair"])

print("\n[4] OUTPUTS")
print("Repair-plan CSV:", plan_path)
print("Plan Markdown:", summary_md)
print("Status JSON:", status_path)

print("\n[5] RELEASE GATE")
print("Layout repair required: True")
print("Full-resolution review: PENDING")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Original DOCX/PDF/review ZIP modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.10G-R1 — REPAIRED DOCX/PDF EXPORT
# Fixes duplicate figure headings, visible PDF paths, table splitting,
# repeated result tables, and wide-table layout.
# Creates new outputs only. Human visual QA remains pending.

from pathlib import Path
from zipfile import ZipFile
from datetime import datetime
from PIL import Image, ImageOps, ImageDraw
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_SECTION, WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
import hashlib
import json
import re
import shutil
import subprocess
import tempfile
import warnings
import io
import csv
import sys

# ------------------------------------------------------------
# 1. INPUTS AND PRE-FLIGHT
# ------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

SOURCE_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10B_R1_integrated_manuscript_20261025T184741"
)
SOURCE_MD = SOURCE_DIR / "Delhi_AQI_Integrated_Manuscript.md"
SOURCE_FIGURES = SOURCE_DIR / "figures"
SOURCE_TABLES = SOURCE_DIR / "tables_csv"

BUNDLE = (
    ROOT / "outputs/research_paper"
    / "phase17_10C_final_review_package_20261025T184853"
    / "Delhi_AQI_Integrated_Research_Review_Package.zip"
)
EXPECTED_BUNDLE_SHA256 = (
    "e60206d8958bfcc0b97e39290e7fa7e63386bdc6570d5b16bb8961ca19d22e59"
)

if not SOURCE_MD.is_file():
    raise FileNotFoundError(f"Integrated manuscript not found: {SOURCE_MD}")
if not SOURCE_FIGURES.is_dir():
    raise FileNotFoundError(f"Figure folder not found: {SOURCE_FIGURES}")
if not SOURCE_TABLES.is_dir():
    raise FileNotFoundError(f"Table CSV folder not found: {SOURCE_TABLES}")
if not BUNDLE.is_file():
    raise FileNotFoundError(f"Phase 17.10C review ZIP not found: {BUNDLE}")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

bundle_hash = sha256_file(BUNDLE)
if bundle_hash != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(
        "Phase 17.10C review ZIP hash mismatch. Repair stopped."
    )

with ZipFile(BUNDLE, "r") as z:
    bad_member = z.testzip()
    if bad_member:
        raise RuntimeError(f"ZIP integrity test failed: {bad_member}")
    original_bundle_entries = z.namelist()

text = SOURCE_MD.read_text(encoding="utf-8")

image_files = sorted(SOURCE_FIGURES.glob("*.png"))
pdf_figure_files = sorted(SOURCE_FIGURES.glob("*.pdf"))
table_files = sorted(SOURCE_TABLES.glob("*.csv"))

if len(image_files) != 7 or len(pdf_figure_files) != 7:
    raise RuntimeError(
        f"Expected 7 PNGs and 7 PDFs, found "
        f"{len(image_files)} PNGs and {len(pdf_figure_files)} PDFs."
    )
if len(table_files) != 9:
    raise RuntimeError(f"Expected 9 original table CSVs; found {len(table_files)}.")

print("=" * 78)
print("PHASE 17.10G-R1 — LAYOUT REPAIR AND RE-RENDER")
print("=" * 78)
print("Phase 17.10C SHA-256 verified:", bundle_hash == EXPECTED_BUNDLE_SHA256)
print("Review ZIP entries:", len(original_bundle_entries))
print("PNG/PDF figures:", len(image_files), "/", len(pdf_figure_files))
print("Main table CSVs:", len(table_files))

# ------------------------------------------------------------
# 2. REMOVE DUPLICATE LEGACY RESULT TABLES FROM THE READER COPY
#    The original manuscript snapshot remains preserved.
# ------------------------------------------------------------

ASSET_MARKER = "<!-- PHASE 17.10B-R1 ASSET INTEGRATION -->"

if ASSET_MARKER not in text:
    raise RuntimeError(
        "Expected Phase 17.10B-R1 integration marker not found. "
        "No output document has been created."
    )

body_text, asset_text = text.split(ASSET_MARKER, 1)

def split_md_row(line):
    line = line.strip()
    if line.startswith("|"):
        line = line[1:]
    if line.endswith("|"):
        line = line[:-1]
    return [part.strip() for part in line.split("|")]

def is_md_separator(line):
    if "|" not in line:
        return False
    cells = split_md_row(line)
    return bool(cells) and all(
        re.fullmatch(r":?-{3,}:?", c.replace(" ", "")) for c in cells
    )

def parse_md_table_at(lines, start):
    if start + 1 >= len(lines):
        return None
    if "|" not in lines[start] or not is_md_separator(lines[start + 1]):
        return None

    headers = split_md_row(lines[start])
    rows = []
    end = start + 2

    while end < len(lines) and "|" in lines[end].strip():
        cells = split_md_row(lines[end])
        # Correct minor malformed rows by padding/truncating to header width.
        cells = (cells + [""] * len(headers))[:len(headers)]
        rows.append(cells)
        end += 1

    return headers, rows, end

RESULT_CONTEXT_TERMS = [
    "results", "evaluation", "model performance", "model comparison",
    "performance by", "validation", "holdout", "ablation", "bootstrap",
    "aqi band", "station performance", "season performance",
    "feature attribution", "transition detection", "error analysis",
]

RESULT_HEADER_TERMS = {
    "model", "mae", "rmse", "r2", "r", "aqi", "station", "season",
    "month", "metric", "holdout", "persistence", "ensemble", "feature",
    "importance", "transition", "error", "prediction", "target",
    "accuracy", "precision", "recall", "n", "score", "actual",
}

def normalise_header(header):
    return re.sub(r"[^a-z0-9]", "", header.lower())

def numeric_content_present(rows):
    for row in rows[:8]:
        for value in row:
            if re.search(r"\d", str(value)):
                return True
    return False

lines = body_text.splitlines()
clean_body_lines = []
removed_legacy_tables = []
current_headings = []

i = 0
while i < len(lines):
    line = lines[i]

    heading_match = re.match(r"^\s{0,3}(#{1,6})\s+(.+?)\s*#*\s*$", line)
    if heading_match:
        current_headings.append(heading_match.group(2).strip())
        current_headings = current_headings[-4:]
        clean_body_lines.append(line)
        i += 1
        continue

    parsed = parse_md_table_at(lines, i)
    if parsed:
        headers, rows, end = parsed
        header_hits = sum(
            1 for h in headers
            if normalise_header(h) in RESULT_HEADER_TERMS
        )
        context = " ".join(current_headings).lower()
        results_context = any(term in context for term in RESULT_CONTEXT_TERMS)
        result_like = numeric_content_present(rows) and (
            (results_context and len(headers) >= 3)
            or header_hits >= 3
        )

        block = "\n".join(lines[i:end])

        if result_like:
            removed_legacy_tables.append({
                "context": " > ".join(current_headings),
                "headers": headers,
                "row_count": len(rows),
                "block": block,
            })
            # Leave whitespace only in the reader-facing document.
            clean_body_lines.append("")
        else:
            clean_body_lines.extend(lines[i:end])

        i = end
        continue

    clean_body_lines.append(line)
    i += 1

clean_body = "\n".join(clean_body_lines).rstrip()

# Clean the figure section and suppress duplicate visual title/path lines.
clean_asset_lines = []
for line in asset_text.splitlines():
    stripped = line.strip()

    # The title is already present in the bold numbered caption. Keep one title.
    if re.match(r"^#{1,6}\s+Figure\s+\d+\s*\.", stripped, flags=re.I):
        continue

    # Local PDF file paths belong in the manifest, not the manuscript.
    if re.match(r"^\*?\s*PDF version\s*:?", stripped, flags=re.I):
        continue
    if re.search(
        r"`?\s*figures/Figure_\d+_[^` ]+\.pdf\s*`?",
        stripped,
        flags=re.I
    ):
        continue

    clean_asset_lines.append(line)

clean_asset_text = "\n".join(clean_asset_lines).strip()

# Keep the original table blocks for audit/recovery.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10G_R1_layout_repaired_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

legacy_path = OUT / "legacy_result_tables_excluded_from_reader_copy.md"
legacy_lines = [
    "# Legacy result tables excluded from the reader-facing repair",
    "",
    "These tables were found before the Phase 17.10B-R1 integrated asset section "
    "and were classified as result-like by header/context heuristics.",
    "They are excluded from the new reader-facing document to avoid likely "
    "duplication with the numbered Tables 1–9. The original manuscript and "
    "review ZIP remain unchanged.",
    "",
]
for index, item in enumerate(removed_legacy_tables, start=1):
    legacy_lines.extend([
        f"## Excluded legacy table {index}",
        f"**Section context:** {item['context'] or 'Not identified'}",
        f"**Detected headers:** {', '.join(item['headers'])}",
        f"**Data rows:** {item['row_count']}",
        "",
        item["block"],
        "",
    ])
legacy_path.write_text("\n".join(legacy_lines), encoding="utf-8")

clean_markdown = clean_body + "\n\n" + clean_asset_text + "\n"
clean_md_path = OUT / "Delhi_AQI_Layout_Repaired_Manuscript.md"
clean_md_path.write_text(clean_markdown, encoding="utf-8")

print("\n[2] MARKDOWN CLEANUP")
print("Legacy result tables excluded from reader copy:",
      len(removed_legacy_tables))
print("PDF path/title lines removed from asset section: True")
print("Audit copy of excluded tables:", legacy_path)

if len(removed_legacy_tables) == 0:
    warnings.warn(
        "No legacy result tables were classified for removal. If the earlier "
        "18-table count reflected 9 duplicate tables, manually inspect that "
        "finding before approving the repaired draft."
    )
elif len(removed_legacy_tables) != 9:
    warnings.warn(
        f"{len(removed_legacy_tables)} legacy tables were excluded, not 9. "
        "Review legacy_result_tables_excluded_from_reader_copy.md to verify "
        "that no distinct results were mistakenly consolidated."
    )

# ------------------------------------------------------------
# 3. BUILD FORMATTED DOCX FROM CLEAN MARKDOWN
# ------------------------------------------------------------

doc = Document()
sec = doc.sections[0]
sec.top_margin = Inches(0.68)
sec.bottom_margin = Inches(0.68)
sec.left_margin = Inches(0.72)
sec.right_margin = Inches(0.72)
sec.orientation = WD_ORIENT.PORTRAIT
sec.page_width = Inches(8.5)
sec.page_height = Inches(11)

doc.core_properties.title = "Delhi AQI One-Hour-Ahead Forecasting — Integrated Review Draft"
doc.core_properties.subject = "Integrated research manuscript with numbered figures and tables"

# Global typography.
normal = doc.styles["Normal"]
normal.font.name = "Times New Roman"
normal.font.size = Pt(10.5)
normal.paragraph_format.space_after = Pt(5)
normal.paragraph_format.line_spacing = 1.08

for style_name, size in [
    ("Heading 1", 16),
    ("Heading 2", 13),
    ("Heading 3", 11),
]:
    style = doc.styles[style_name]
    style.font.name = "Times New Roman"
    style.font.size = Pt(size)
    style.font.bold = True
    style.font.color.rgb = RGBColor(31, 78, 121)
    style.paragraph_format.keep_with_next = True
    style.paragraph_format.keep_together = True
    style.paragraph_format.space_before = Pt(10)
    style.paragraph_format.space_after = Pt(5)

def add_page_number(paragraph):
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = paragraph.add_run()
    fld_begin = OxmlElement("w:fldChar")
    fld_begin.set(qn("w:fldCharType"), "begin")
    instr = OxmlElement("w:instrText")
    instr.set(qn("xml:space"), "preserve")
    instr.text = " PAGE "
    fld_sep = OxmlElement("w:fldChar")
    fld_sep.set(qn("w:fldCharType"), "separate")
    fld_text = OxmlElement("w:t")
    fld_text.text = "1"
    fld_end = OxmlElement("w:fldChar")
    fld_end.set(qn("w:fldCharType"), "end")
    for element in (fld_begin, instr, fld_sep, fld_text, fld_end):
        run._r.append(element)

add_page_number(doc.sections[0].footer.paragraphs[0])

def set_section_orientation(section, orientation):
    if orientation == "landscape":
        section.orientation = WD_ORIENT.LANDSCAPE
        section.page_width = Inches(11)
        section.page_height = Inches(8.5)
        section.left_margin = Inches(0.55)
        section.right_margin = Inches(0.55)
    else:
        section.orientation = WD_ORIENT.PORTRAIT
        section.page_width = Inches(8.5)
        section.page_height = Inches(11)
        section.left_margin = Inches(0.72)
        section.right_margin = Inches(0.72)

    section.top_margin = Inches(0.65)
    section.bottom_margin = Inches(0.65)

orientation_state = "portrait"
content_added = False
figures_inserted = 0
tables_inserted = 0
figure_caption_titles = []

inline_pattern = re.compile(
    r"(\*\*.+?\*\*|\*[^*]+\*|`[^`]+`|\[[^\]]+\]\([^)]+\))"
)

def add_inline_runs(paragraph, text, default_size=None):
    pos = 0
    for match in inline_pattern.finditer(text):
        if match.start() > pos:
            run = paragraph.add_run(text[pos:match.start()])
            if default_size:
                run.font.size = Pt(default_size)

        token = match.group(0)
        if token.startswith("**") and token.endswith("**"):
            run = paragraph.add_run(token[2:-2])
            run.bold = True
        elif token.startswith("*") and token.endswith("*"):
            run = paragraph.add_run(token[1:-1])
            run.italic = True
        elif token.startswith("`") and token.endswith("`"):
            run = paragraph.add_run(token[1:-1])
            run.font.name = "Courier New"
            run.font.size = Pt(9)
        elif token.startswith("[") and "](" in token:
            # Preserve readable link text; omit raw URL from the manuscript text.
            run = paragraph.add_run(token[1:token.index("](")])
            run.font.color.rgb = RGBColor(31, 78, 121)
        if default_size:
            run.font.size = Pt(default_size)
        pos = match.end()

    if pos < len(text):
        run = paragraph.add_run(text[pos:])
        if default_size:
            run.font.size = Pt(default_size)

def md_separator(line):
    return is_md_separator(line)

def parse_table(lines, start):
    result = parse_md_table_at(lines, start)
    if result is None:
        return None
    headers, rows, end = result
    return headers, rows, end

def mark_row_cannot_split(row):
    tr_pr = row._tr.get_or_add_trPr()
    if tr_pr.find(qn("w:cantSplit")) is None:
        tr_pr.append(OxmlElement("w:cantSplit"))

def repeat_header(row):
    tr_pr = row._tr.get_or_add_trPr()
    header = OxmlElement("w:tblHeader")
    header.set(qn("w:val"), "true")
    tr_pr.append(header)

def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shading = OxmlElement("w:shd")
    shading.set(qn("w:fill"), fill)
    tc_pr.append(shading)

def set_cell_margins(cell, top=45, start=55, bottom=45, end=55):
    tc = cell._tc
    tc_pr = tc.get_or_add_tcPr()
    tc_mar = tc_pr.first_child_found_in("w:tcMar")
    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar")
        tc_pr.append(tc_mar)
    for edge, value in (("top", top), ("start", start),
                        ("bottom", bottom), ("end", end)):
        node = tc_mar.find(qn(f"w:{edge}"))
        if node is None:
            node = OxmlElement(f"w:{edge}")
            tc_mar.append(node)
        node.set(qn("w:w"), str(value))
        node.set(qn("w:type"), "dxa")

def begin_new_page_or_section(target_orientation):
    global orientation_state

    if target_orientation != orientation_state:
        new_section = doc.add_section(WD_SECTION.NEW_PAGE)
        set_section_orientation(new_section, target_orientation)
        orientation_state = target_orientation
    else:
        doc.add_page_break()

def add_word_table(headers, rows, title=None, force_new_page=True):
    global tables_inserted, content_added

    col_count = max(1, len(headers))
    target_orientation = "landscape" if col_count >= 7 else "portrait"

    if force_new_page and content_added:
        begin_new_page_or_section(target_orientation)
    elif target_orientation != orientation_state:
        new_section = doc.add_section(WD_SECTION.NEW_PAGE)
        set_section_orientation(new_section, target_orientation)
        orientation_state = target_orientation

    if title:
        heading = doc.add_heading(title, level=3)
        heading.paragraph_format.keep_with_next = True
        heading.paragraph_format.keep_together = True
        heading.paragraph_format.space_after = Pt(4)

    table = doc.add_table(rows=1, cols=col_count)
    table.style = "Table Grid"
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = True

    small_table_font = 6.5 if col_count >= 7 else 8.0

    header_row = table.rows[0]
    repeat_header(header_row)
    mark_row_cannot_split(header_row)

    for j, header_text in enumerate(headers):
        cell = header_row.cells[j]
        cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
        p = cell.paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.paragraph_format.space_after = Pt(0)
        run = p.add_run(str(header_text))
        run.bold = True
        run.font.name = "Times New Roman"
        run.font.size = Pt(small_table_font)
        run.font.color.rgb = RGBColor(255, 255, 255)
        set_cell_shading(cell, "1F4E78")
        set_cell_margins(cell)

    for values in rows:
        row = table.add_row()
        mark_row_cannot_split(row)
        for j in range(col_count):
            value = values[j] if j < len(values) else ""
            cell = row.cells[j]
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            p = cell.paragraphs[0]
            p.text = ""
            p.paragraph_format.space_after = Pt(0)
            p.paragraph_format.line_spacing = 1.0
            run = p.add_run(str(value))
            run.font.name = "Times New Roman"
            run.font.size = Pt(small_table_font)
            set_cell_margins(cell)

    # Small space after the table; the next heading will not touch the grid.
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(2)
    p.paragraph_format.space_before = Pt(0)
    content_added = True
    tables_inserted += 1

def find_table_after_heading(lines, heading_index):
    j = heading_index + 1
    while j < len(lines) and not lines[j].strip():
        j += 1
    return parse_table(lines, j) if j < len(lines) else None

def add_image(image_line):
    global figures_inserted, content_added

    match = re.match(r"!\[([^\]]*)\]\(([^)]+)\)", image_line.strip())
    if not match:
        return False

    alt = match.group(1)
    relative_path = match.group(2).strip()
    image_path = SOURCE_DIR / relative_path

    if not image_path.is_file():
        raise FileNotFoundError(
            f"Markdown image link could not be resolved: {image_path}"
        )

    if content_added:
        doc.add_page_break()

    paragraph = doc.add_paragraph()
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    paragraph.paragraph_format.keep_with_next = True
    paragraph.paragraph_format.keep_together = True
    paragraph.paragraph_format.space_before = Pt(2)
    paragraph.paragraph_format.space_after = Pt(3)

    with Image.open(image_path) as im:
        w_px, h_px = im.size

    w_in = w_px / 120
    h_in = h_px / 120
    scale = min(6.35 / max(w_in, 0.01), 5.35 / max(h_in, 0.01), 1.0)

    run = paragraph.add_run()
    run.add_picture(
        str(image_path),
        width=Inches(max(0.5, w_in * scale)),
        height=Inches(max(0.5, h_in * scale))
    )

    # Alt text in the image description metadata improves accessibility.
    try:
        pic = paragraph._p.xpath(".//pic:cNvPr")
        if pic:
            pic[0].set("descr", alt)
    except Exception:
        pass

    figures_inserted += 1
    content_added = True
    return True

# Process clean Markdown, grouping titled tables with their titles so the title
# and table start on the same page.
doc_lines = clean_markdown.splitlines()
i = 0

while i < len(doc_lines):
    line = doc_lines[i]
    stripped = line.strip()

    if not stripped:
        i += 1
        continue

    # Hidden HTML comments are not reader-facing content.
    if stripped.startswith("<!--"):
        i += 1
        continue

    # Suppress headings that duplicate the numbered figure caption or
    # produce orphan headings immediately before forced figure pagination.
    if re.match(r"^#{1,6}\s+Figure\s+\d+\s*\.", stripped, flags=re.I):
        i += 1
        continue
    if re.match(r"^#{1,6}\s+Figures\s*$", stripped, flags=re.I):
        i += 1
        continue
    if re.match(
        r"^#{1,6}\s+Integrated Figures and Main Results Tables\s*$",
        stripped, flags=re.I
    ):
        i += 1
        continue
    if re.match(r"^#{1,6}\s+Main Results Tables\s*$", stripped, flags=re.I):
        i += 1
        continue

    # A numbered table heading and its pipe table are treated as one unit.
    title_match = re.match(
        r"^#{1,6}\s*(Table\s+\d+\s*[.:]?\s*.*)$",
        stripped,
        flags=re.I
    )
    if title_match:
        parsed = find_table_after_heading(doc_lines, i)
        if parsed:
            headers, rows, end = parsed
            add_word_table(
                headers, rows,
                title=title_match.group(1).strip(),
                force_new_page=True
            )
            i = end
            while i < len(doc_lines) and not doc_lines[i].strip():
                i += 1
            continue

    # References always return to portrait so bibliography pages are narrow.
    heading_match = re.match(r"^(#{1,6})\s+(.+?)\s*#*\s*$", stripped)
    if heading_match:
        level = min(len(heading_match.group(1)), 3)
        heading_text = heading_match.group(2).strip()

        is_refs = bool(re.match(
            r"^(?:\d+(?:\.\d+)*\.?\s*)?"
            r"(References|Bibliography|Reference List)\b",
            heading_text,
            flags=re.I
        ))
        if is_refs and orientation_state == "landscape":
            new_section = doc.add_section(WD_SECTION.NEW_PAGE)
            set_section_orientation(new_section, "portrait")
            orientation_state = "portrait"

        p = doc.add_heading(heading_text, level=level)
        p.paragraph_format.keep_with_next = True
        p.paragraph_format.keep_together = True
        content_added = True
        i += 1
        continue

    # Figure image.
    if stripped.startswith("!["):
        if add_image(stripped):
            i += 1
            continue

    # Markdown table without a numbered heading.
    parsed = parse_table(doc_lines, i)
    if parsed:
        headers, rows, end = parsed
        add_word_table(headers, rows, title=None, force_new_page=False)
        i = end
        continue

    # Unordered list.
    if re.match(r"^\s*[-*+]\s+", line):
        text_value = re.sub(r"^\s*[-*+]\s+", "", line).strip()
        p = doc.add_paragraph(style="List Bullet")
        add_inline_runs(p, text_value)
        p.paragraph_format.keep_together = True
        content_added = True
        i += 1
        continue

    # Ordered list.
    if re.match(r"^\s*\d+[.)]\s+", line):
        text_value = re.sub(r"^\s*\d+[.)]\s+", "", line).strip()
        p = doc.add_paragraph(style="List Number")
        add_inline_runs(p, text_value)
        p.paragraph_format.keep_together = True
        content_added = True
        i += 1
        continue

    if re.match(r"^\s*[-*_]{3,}\s*$", stripped):
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(3)
        p.paragraph_format.space_after = Pt(3)
        content_added = True
        i += 1
        continue

    # Group normal text lines into one paragraph until the next block.
    paragraph_lines = [stripped]
    j = i + 1

    while j < len(doc_lines):
        next_line = doc_lines[j]
        next_stripped = next_line.strip()

        if not next_stripped:
            break
        if next_stripped.startswith("<!--"):
            break
        if re.match(r"^#{1,6}\s+", next_stripped):
            break
        if next_stripped.startswith("![") or parse_table(doc_lines, j):
            break
        if re.match(r"^\s*[-*+]\s+", next_line):
            break
        if re.match(r"^\s*\d+[.)]\s+", next_line):
            break

        paragraph_lines.append(next_stripped)
        j += 1

    paragraph_text = " ".join(paragraph_lines)

    # Caption paragraph: one visible numbered caption, kept together.
    is_caption = bool(re.match(r"^\*\*Figure\s+\d+\.", paragraph_text, re.I))

    # Switch wide-table orientation back before explanatory notes and references.
    if re.match(
        r"^\*\*?(?:Interpretation and review notes|Author review)",
        paragraph_text,
        flags=re.I
    ):
        if orientation_state == "landscape":
            new_section = doc.add_section(WD_SECTION.NEW_PAGE)
            set_section_orientation(new_section, "portrait")
            orientation_state = "portrait"

    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(5)
    p.paragraph_format.keep_together = is_caption
    if is_caption:
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        figure_caption_titles.append(paragraph_text)
    if paragraph_text.startswith(">"):
        paragraph_text = paragraph_text.lstrip(">").strip()
        p.paragraph_format.left_indent = Inches(0.25)
    add_inline_runs(p, paragraph_text, default_size=9 if is_caption else None)

    content_added = True
    i = j

# Save a fully new DOCX.
DOCX_PATH = OUT / "Delhi_AQI_Layout_Repaired_Review_Draft.docx"
doc.save(DOCX_PATH)

# ------------------------------------------------------------
# 4. STRUCTURAL CHECKS FOR DOCX
# ------------------------------------------------------------

doc_check = Document(DOCX_PATH)
embedded_images = sum(
    1 for rel in doc_check.part.rels.values()
    if "image" in rel.reltype
)
word_table_count = len(doc_check.tables)

# Count repeating table headers, and rows with cantSplit protection.
repeat_header_count = 0
cant_split_row_count = 0

for table in doc_check.tables:
    for row_index, row in enumerate(table.rows):
        tr_pr = row._tr.get_or_add_trPr()
        if row_index == 0 and tr_pr.find(qn("w:tblHeader")) is not None:
            repeat_header_count += 1
        if tr_pr.find(qn("w:cantSplit")) is not None:
            cant_split_row_count += 1

all_docx_text = "\n".join(
    [p.text for p in doc_check.paragraphs]
    + [cell.text for t in doc_check.tables for row in t.rows for cell in row.cells]
)

path_string_count = len(re.findall(
    r"PDF version\s*:|figures/Figure_\d+_[^\s]+\.pdf",
    all_docx_text,
    flags=re.I
))

# ------------------------------------------------------------
# 5. EXPORT TO PDF USING LIBREOFFICE
# ------------------------------------------------------------

lo_path = shutil.which("libreoffice") or shutil.which("soffice")
if not lo_path:
    raise RuntimeError(
        "LibreOffice was not found. The repaired DOCX was saved, but PDF "
        "export and rendering still need to be completed."
    )

profile_dir = Path(tempfile.mkdtemp(prefix="lo_phase17_10G_"))
cmd = [
    lo_path,
    f"-env:UserInstallation={profile_dir.as_uri()}",
    "--headless",
    "--convert-to", "pdf",
    "--outdir", str(OUT),
    str(DOCX_PATH),
]
result = subprocess.run(
    cmd, capture_output=True, text=True, timeout=300
)
PDF_PATH = OUT / "Delhi_AQI_Layout_Repaired_Review_Draft.pdf"

if result.returncode != 0 or not PDF_PATH.is_file():
    raise RuntimeError(
        "DOCX was created but PDF conversion failed.\n"
        f"Return code: {result.returncode}\n"
        f"STDOUT: {result.stdout}\nSTDERR: {result.stderr}\n"
        f"DOCX: {DOCX_PATH}"
    )

# ------------------------------------------------------------
# 6. RENDER ALL PDF PAGES AND BUILD CONTACT SHEETS
# ------------------------------------------------------------

try:
    import pymupdf as fitz
except ImportError:
    import fitz

RENDER_DIR = OUT / "rendered_pages"
CONTACT_DIR = OUT / "contact_sheets"
RENDER_DIR.mkdir()
CONTACT_DIR.mkdir()

pdf_doc = fitz.open(PDF_PATH)
render_inventory = []
page_images = []

for page_index, page in enumerate(pdf_doc):
    pix = page.get_pixmap(matrix=fitz.Matrix(150/72, 150/72), alpha=False)
    page_path = RENDER_DIR / f"page-{page_index + 1:02d}.png"
    pix.save(str(page_path))
    page_images.append(page_path)

    page_text = page.get_text("text")
    render_inventory.append({
        "page": page_index + 1,
        "png_path": str(page_path),
        "width_px": pix.width,
        "height_px": pix.height,
        "pdf_text_characters": len(page_text),
        "image_count": len(page.get_images(full=True)),
        "contains_pdf_path_label": bool(
            re.search(r"PDF version\s*:", page_text, flags=re.I)
        ),
        "contains_local_project_path": "/content/drive/MyDrive/" in page_text,
    })

# Contact sheets: 9 pages maximum each, 3 columns.
SHEET_MAX_PAGES = 9
thumb_width = 300
thumb_height = 390
columns = 3

for start in range(0, len(page_images), SHEET_MAX_PAGES):
    chunk = page_images[start:start + SHEET_MAX_PAGES]
    rows_count = (len(chunk) + columns - 1) // columns
    pad = 18
    label_height = 28

    sheet = Image.new(
        "RGB",
        (
            columns * (thumb_width + pad) + pad,
            rows_count * (thumb_height + label_height + pad) + pad
        ),
        "white"
    )
    draw = ImageDraw.Draw(sheet)

    for offset, path in enumerate(chunk):
        im = Image.open(path).convert("RGB")
        im.thumbnail((thumb_width, thumb_height))

        cell_x = pad + (offset % columns) * (thumb_width + pad)
        cell_y = pad + (offset // columns) * (thumb_height + label_height + pad)

        # Frame and place thumbnail.
        framed = ImageOps.contain(im, (thumb_width, thumb_height))
        sheet.paste(framed, (cell_x, cell_y))
        page_no = start + offset + 1
        draw.text(
            (cell_x, cell_y + thumb_height + 5),
            f"Page {page_no}",
            fill="black"
        )

    sheet_path = CONTACT_DIR / f"contact-sheet-{start//SHEET_MAX_PAGES + 1:02d}.png"
    sheet.save(sheet_path)
    print("Contact sheet:", sheet_path)

# ------------------------------------------------------------
# 7. SAVE MANIFESTS, QA REPORT, AND RELEASE STATUS
# ------------------------------------------------------------

pd.DataFrame(render_inventory).to_csv(
    OUT / "rendered_page_inventory.csv", index=False
)

pdf_text = "\n".join(page.get_text("text") for page in pdf_doc)

qa = {
    "phase": "17.10G-R1",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_review_zip": str(BUNDLE),
    "source_review_zip_sha256": bundle_hash,
    "source_review_zip_sha256_verified": bundle_hash == EXPECTED_BUNDLE_SHA256,
    "source_markdown": str(SOURCE_MD),
    "cleaned_markdown": str(clean_md_path),
    "excluded_legacy_table_count": len(removed_legacy_tables),
    "excluded_legacy_table_audit": str(legacy_path),
    "docx_path": str(DOCX_PATH),
    "docx_bytes": DOCX_PATH.stat().st_size,
    "pdf_path": str(PDF_PATH),
    "pdf_bytes": PDF_PATH.stat().st_size,
    "pdf_pages": len(pdf_doc),
    "rendered_pages": len(page_images),
    "contact_sheets": [
        str(p) for p in sorted(CONTACT_DIR.glob("*.png"))
    ],
    "figure_png_count_in_source": len(image_files),
    "figure_pdf_count_in_source": len(pdf_figure_files),
    "embedded_image_relationships_in_docx": embedded_images,
    "word_table_count": word_table_count,
    "table_rows_with_no_split_protection": None,
    "table_header_repeat_elements": repeat_header_count,
    "rows_with_cant_split": cant_split_row_count,
    "visible_pdf_path_string_count_in_docx": path_string_count,
    "visible_pdf_path_label_found_in_pdf": bool(
        re.search(r"PDF version\s*:", pdf_text, flags=re.I)
    ),
    "old_review_zip_modified": False,
    "source_data_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "human_visual_qa_completed": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
}

(OUT / "phase17_10G_R1_render_qa.json").write_text(
    json.dumps(qa, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

repair_notes = f"""# Phase 17.10G-R1 — Layout Repair Report

## Inputs
- Verified Phase 17.10C ZIP SHA-256: `{bundle_hash}`
- Original integrated manuscript: `{SOURCE_MD}`

## Repairs applied in the new reader-facing DOCX/PDF
- Removed legacy result-like Markdown tables from the body copy where classified
  by section/header heuristics; retained them in `legacy_result_tables_excluded_from_reader_copy.md`.
- Removed duplicate standalone numbered figure headings when the caption already
  repeats the figure title.
- Removed manuscript-facing PDF file-path lines.
- Kept figure image paragraphs with their following captions.
- Repeated table header rows and protected individual rows from splitting.
- Started wide tables on landscape pages and restored portrait layout for
  references where appropriate.
- Regenerated the PDF and rendered every page to PNG.

## Counts
- Legacy result-like tables excluded: {len(removed_legacy_tables)}
- Embedded image relationships in DOCX: {embedded_images}
- Word tables after cleanup: {word_table_count}
- Repeating header rows: {repeat_header_count}
- Protected table rows: {cant_split_row_count}
- PDF pages: {len(pdf_doc)}
- Rendered pages: {len(page_images)}

## Human review status
**PENDING.** Automated checks and contact sheets help identify issues, but
they do not substitute for human inspection of all rendered pages at full size.
Inspect each contact sheet first and open any unclear or defective page PNG.

## Release gate
- Licence review: OPEN
- Submission ready: NO
- Production enabled: NO
- Original DOCX/PDF/review ZIP modified: NO
- Source data modified: NO
- Models retrained: NO
"""

(OUT / "phase17_10G_R1_layout_repair_report.md").write_text(
    repair_notes, encoding="utf-8"
)

print("\n[3] REPAIR AND STRUCTURE")
print("Legacy result tables excluded:", len(removed_legacy_tables))
print("DOCX created:", DOCX_PATH.is_file())
print("PDF created:", PDF_PATH.is_file())
print("Embedded images:", embedded_images)
print("Word tables:", word_table_count)
print("Repeating header rows:", repeat_header_count)
print("Protected rows:", cant_split_row_count)
print("Visible PDF path labels in DOCX:", path_string_count)
print("PDF pages:", len(pdf_doc))
print("Rendered pages:", len(page_images))

print("\n[4] OUTPUTS")
print("Output folder:", OUT)
print("Cleaned Markdown:", clean_md_path)
print("Excluded table audit:", legacy_path)
print("DOCX:", DOCX_PATH)
print("PDF:", PDF_PATH)
print("Rendered pages:", RENDER_DIR)
print("Contact sheets:", CONTACT_DIR)
print("QA JSON:", OUT / "phase17_10G_R1_render_qa.json")
print("Repair report:", OUT / "phase17_10G_R1_layout_repair_report.md")

print("\n[5] MANUAL REVIEW REQUIRED")
print("Human visual QA completed: False")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data / frozen models / existing review ZIP modified: False")
print("=" * 78)

# Display contact sheets in the notebook so visual review can start immediately.
try:
    from IPython.display import display, Image as IPyImage
    for sheet in sorted(CONTACT_DIR.glob("contact-sheet-*.png")):
        display(IPyImage(filename=str(sheet)))
except Exception as exc:
    print("Contact-sheet display unavailable; open the files directly:", exc)

In [ ]:
    # PHASE 17.10G-R2 FIX:
    # Initialize the local orientation state before it is read.
    current_section_orientation = doc.sections[-1].orientation

    if isinstance(target_orientation, str):
        orientation_state = (
            "landscape"
            if current_section_orientation == WD_ORIENT.LANDSCAPE
            else "portrait"
        )
    else:
        orientation_state = current_section_orientation

In [ ]:
# PHASE 17.10F-R1 — FIX UNDEFINED target_orientation

from docx.enum.section import WD_ORIENT

if "doc" not in globals():
    raise NameError(
        "The DOCX object 'doc' is not defined. "
        "Rerun the cell that loads the document, then run this repair."
    )

if len(doc.sections) == 0:
    raise RuntimeError("The DOCX contains no sections.")

# Safe fallback: keep the current orientation unless explicitly overridden.
current_section_orientation = doc.sections[-1].orientation

if "target_orientation" not in globals() or target_orientation is None:
    target_orientation = current_section_orientation

if isinstance(target_orientation, str):
    target_orientation_normalized = target_orientation.strip().lower()

    if target_orientation_normalized == "landscape":
        orientation_state = WD_ORIENT.LANDSCAPE
    elif target_orientation_normalized == "portrait":
        orientation_state = WD_ORIENT.PORTRAIT
    else:
        raise ValueError(
            "target_orientation must be 'portrait' or 'landscape'."
        )
else:
    orientation_state = target_orientation

if orientation_state not in (WD_ORIENT.PORTRAIT, WD_ORIENT.LANDSCAPE):
    raise ValueError(
        f"Unsupported target orientation: {orientation_state!r}"
    )

print("Orientation variable repair: PASS")
print("Current last-section orientation:", current_section_orientation)
print("Target orientation:", orientation_state)
print("No document changes made by this repair cell.")

In [ ]:
# PHASE 17.10F-R1 — EXPORT VISUAL QA EVIDENCE FOR REVIEW
# Read-only collection of the rendered pages and QA files.
# Does not modify the manuscript, source data, or existing review packages.

from pathlib import Path
from datetime import datetime
import zipfile
import hashlib
import json

from google.colab import files

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
RESEARCH = ROOT / "outputs" / "research_paper"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print("=" * 78)
print("PHASE 17.10F-R1 — VISUAL QA EXPORT")
print("=" * 78)

# 1. Find the latest contact-sheet pair.
sheet_candidates = []
for sheet1 in RESEARCH.rglob("contact-sheet-01.png"):
    sheet2 = sheet1.parent / "contact-sheet-02.png"
    if sheet2.is_file():
        sheet_candidates.append((sheet1.parent, sheet1, sheet2))

if not sheet_candidates:
    raise FileNotFoundError(
        "Could not find both contact-sheet-01.png and "
        "contact-sheet-02.png under outputs/research_paper. "
        "No output package was created."
    )

sheet_dir, sheet1, sheet2 = max(
    sheet_candidates,
    key=lambda x: max(x[1].stat().st_mtime, x[2].stat().st_mtime)
)

# 2. Find a directory with all 19 full-page renders.
page_dirs = []
for folder in RESEARCH.rglob("rendered_pages"):
    if folder.is_dir():
        pngs = sorted(folder.glob("*.png"))
        if len(pngs) == 19:
            page_dirs.append((folder, pngs))

if not page_dirs:
    raise FileNotFoundError(
        "Could not find a rendered_pages folder containing exactly "
        "19 PNG files. No output package was created."
    )

page_dir, page_files = max(
    page_dirs,
    key=lambda item: item[0].stat().st_mtime
)

# 3. Find the rendered PDF.
pdf_candidates = list(
    RESEARCH.rglob("Delhi_AQI_Integrated_Review_Draft.pdf")
)
if not pdf_candidates:
    raise FileNotFoundError(
        "Could not find Delhi_AQI_Integrated_Review_Draft.pdf. "
        "No output package was created."
    )

pdf_path = max(pdf_candidates, key=lambda p: p.stat().st_mtime)

# 4. Locate the manual review checklist, if present.
checklist_candidates = list(
    RESEARCH.rglob("*page_review_checklist.csv")
)
checklist_path = (
    max(checklist_candidates, key=lambda p: p.stat().st_mtime)
    if checklist_candidates else None
)

# 5. Create a new export directory and ZIP.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = RESEARCH / f"phase17_10F_R1_visual_review_export_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

zip_path = OUT / "Delhi_AQI_Visual_QA_Evidence.zip"
manifest = {
    "phase": "17.10F-R1",
    "created_at": datetime.now().astimezone().isoformat(),
    "contact_sheet_source": str(sheet_dir),
    "page_render_source": str(page_dir),
    "pdf_source": str(pdf_path),
    "checklist_source": str(checklist_path) if checklist_path else None,
    "files": [],
}

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    # Contact sheets first, as requested by the visual-review workflow.
    for path in (sheet1, sheet2):
        arcname = f"contact_sheets/{path.name}"
        z.write(path, arcname)
        manifest["files"].append({
            "archive_path": arcname,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

    # Add all 19 full-page images.
    for index, path in enumerate(page_files, start=1):
        arcname = f"full_page_renders/page_{index:02d}.png"
        z.write(path, arcname)
        manifest["files"].append({
            "archive_path": arcname,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

    # Include the source PDF for cross-checking page text and layout.
    arcname = f"source_pdf/{pdf_path.name}"
    z.write(pdf_path, arcname)
    manifest["files"].append({
        "archive_path": arcname,
        "bytes": pdf_path.stat().st_size,
        "sha256": sha256_file(pdf_path),
    })

    if checklist_path:
        arcname = f"checklist/{checklist_path.name}"
        z.write(checklist_path, arcname)
        manifest["files"].append({
            "archive_path": arcname,
            "bytes": checklist_path.stat().st_size,
            "sha256": sha256_file(checklist_path),
        })

    readme = """Delhi AQI paper — visual QA evidence package

Start with contact_sheets/contact-sheet-01.png and
contact_sheets/contact-sheet-02.png.

Then inspect full_page_renders/page_01.png through page_19.png wherever
the contact sheets suggest layout, clipping, readability, or pagination issues.

The source PDF is included for reference. The review checklist is included
if it was found in the project.

Automated rendering checks are not a substitute for visual inspection.
The human QA status should remain pending until every page is reviewed.
"""
    z.writestr("README.txt", readme)

    manifest["files"].append({
        "archive_path": "README.txt",
        "description": "Instructions for the visual review",
    })
    z.writestr(
        "visual_qa_export_manifest.json",
        json.dumps(manifest, indent=2, ensure_ascii=False),
    )

# 6. Verify the ZIP.
with zipfile.ZipFile(zip_path, "r") as z:
    bad_member = z.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Export ZIP failed integrity check: {bad_member}")
    entries = z.namelist()

print("\n[1] SOURCE FILES")
print("Contact sheet directory:", sheet_dir)
print("Rendered page directory:", page_dir)
print("PDF:", pdf_path)
print("Checklist:", checklist_path or "Not found")

print("\n[2] EXPORT CHECKS")
print("Full-page PNGs:", len(page_files))
print("Contact sheets:", 2)
print("ZIP entries:", len(entries))
print("ZIP integrity: PASS")
print("ZIP SHA-256:", sha256_file(zip_path))

print("\n[3] EXPORT PACKAGE")
print("ZIP:", zip_path)
print("Size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))

print("\nHuman visual QA completed: NO — images still need review.")
print("Submission ready: False")
print("Production enabled: False")
print("=" * 78)

# Trigger the download to your computer.
files.download(str(zip_path))

In [ ]:
# Download the saved visual QA evidence package for review.

from pathlib import Path
from google.colab import files
import hashlib

zip_path = Path(
    "/content/drive/MyDrive/Delhi-AQI-Prediction/outputs/research_paper/"
    "phase17_10F_R1_visual_review_export_20261025T191457/"
    "Delhi_AQI_Visual_QA_Evidence.zip"
)

if not zip_path.is_file():
    raise FileNotFoundError(f"Visual QA evidence ZIP not found: {zip_path}")

digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()

print("File:", zip_path.name)
print(f"Size: {zip_path.stat().st_size / (1024**2):.2f} MB")
print("SHA-256:", digest)
print("Downloading existing file; no project files will be modified.")

files.download(str(zip_path))

In [ ]:
# PHASE 17.10G — INTERACTIVE PAGE-BY-PAGE VISUAL QA
# Review each rendered page, record findings, and save a separate checklist.
# This phase does not modify the source manuscript, ZIP, data, or models.

from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import os
import warnings

from IPython.display import display, Image, Markdown, clear_output
import ipywidgets as widgets

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
BASE = ROOT / "outputs" / "research_paper"

# 1. Locate the existing checklist without overwriting it.
preferred_checklist = (
    BASE / "phase17_10E_human_visual_qa_20261025T185812"
    / "phase17_10E_page_review_checklist.csv"
)

if preferred_checklist.is_file():
    SOURCE_CHECKLIST = preferred_checklist
else:
    candidates = list(BASE.rglob("phase17_10E_page_review_checklist.csv"))
    if not candidates:
        candidates = list(BASE.rglob("*page_review_checklist.csv"))
    if not candidates:
        raise FileNotFoundError(
            "No existing page review checklist was found under outputs/research_paper."
        )
    SOURCE_CHECKLIST = max(candidates, key=lambda p: p.stat().st_mtime)

qa = pd.read_csv(SOURCE_CHECKLIST)

required_columns = [
    "page",
    "render_file",
    "manual_review_status",
    "no_clipping_or_overlap",
    "figures_readable",
    "tables_readable",
    "headings_and_page_breaks_ok",
    "captions_and_references_ok",
    "issues_and_notes",
]

for column in required_columns:
    if column not in qa.columns:
        qa[column] = ""

for column in [
    "manual_review_status",
    "no_clipping_or_overlap",
    "figures_readable",
    "tables_readable",
    "headings_and_page_breaks_ok",
    "captions_and_references_ok",
]:
    qa[column] = qa[column].fillna("").replace("", "PENDING")

qa["issues_and_notes"] = qa["issues_and_notes"].fillna("")
qa["page"] = pd.to_numeric(qa["page"], errors="raise").astype(int)
qa = qa.sort_values("page").reset_index(drop=True)

# Resolve rendered-page paths when the saved paths are relative or stale.
def resolve_render_path(value):
    path = Path(str(value))
    if path.is_file():
        return path

    basename = path.name
    matches = list(BASE.rglob(basename))
    matches = [p for p in matches if p.is_file()]
    if matches:
        return max(matches, key=lambda p: p.stat().st_mtime)

    raise FileNotFoundError(f"Rendered page image not found: {value}")

resolved_paths = []
for value in qa["render_file"]:
    resolved_paths.append(resolve_render_path(value))
qa["_resolved_render_path"] = [str(p) for p in resolved_paths]

# Ensure every image is available before starting the interactive review.
missing_images = [
    str(p) for p in qa["_resolved_render_path"]
    if not Path(p).is_file()
]
if missing_images:
    raise FileNotFoundError(
        "Some rendered page images are missing:\n" + "\n".join(missing_images)
    )

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = BASE / f"phase17_10G_interactive_page_review_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

CHECKLIST_OUT = OUT / "phase17_10G_completed_page_checklist.csv"
STATUS_OUT = OUT / "phase17_10G_visual_qa_status.json"

qa["source_render_file"] = qa["render_file"]
qa["render_file"] = qa["_resolved_render_path"]
qa = qa.drop(columns=["_resolved_render_path"])

status_options = [
    ("Pending", "PENDING"),
    ("Pass", "PASS"),
    ("Issues found — fixes needed", "ISSUES_FOUND"),
    ("Minor issues noted", "PASS_WITH_MINOR_ISSUES"),
]

yes_no_options = [
    ("Pending", "PENDING"),
    ("Yes", "YES"),
    ("No — issue present", "NO"),
    ("Not applicable on this page", "N/A"),
]

page_numbers = qa["page"].tolist()
if not page_numbers:
    raise RuntimeError("The page checklist contains no pages.")

page_selector = widgets.Dropdown(
    options=[(f"Page {p}", p) for p in page_numbers],
    value=page_numbers[0],
    description="Review:",
    layout=widgets.Layout(width="250px"),
)

review_status = widgets.Dropdown(
    options=status_options,
    value=str(qa.loc[0, "manual_review_status"]),
    description="Page status:",
    layout=widgets.Layout(width="430px"),
)

controls = {}
qa_fields = [
    ("no_clipping_or_overlap", "No clipping/overlap"),
    ("figures_readable", "Figures readable"),
    ("tables_readable", "Tables readable"),
    ("headings_and_page_breaks_ok", "Headings/page breaks OK"),
    ("captions_and_references_ok", "Captions/references OK"),
]

for column, label in qa_fields:
    controls[column] = widgets.Dropdown(
        options=yes_no_options,
        value=str(qa.loc[0, column]),
        description=label + ":",
        layout=widgets.Layout(width="430px"),
    )

notes = widgets.Textarea(
    value=str(qa.loc[0, "issues_and_notes"]),
    description="Notes:",
    placeholder="Describe any clipping, tiny labels, awkward page breaks, missing captions, or other issues.",
    layout=widgets.Layout(width="95%", height="100px"),
)

page_display = widgets.Output()
message_display = widgets.Output()

def row_index(page_number):
    found = qa.index[qa["page"] == int(page_number)].tolist()
    if not found:
        raise KeyError(f"Page not present in checklist: {page_number}")
    return found[0]

def persist_review():
    """Save the review without modifying the source checklist."""
    idx = row_index(page_selector.value)
    qa.loc[idx, "manual_review_status"] = review_status.value

    for column, _ in qa_fields:
        qa.loc[idx, column] = controls[column].value

    qa.loc[idx, "issues_and_notes"] = notes.value

    # Exclude the original temporary/helper columns from the saved checklist.
    qa.to_csv(CHECKLIST_OUT, index=False)

    all_statuses = qa["manual_review_status"].astype(str)
    all_reviewed = bool((all_statuses != "PENDING").all())

    critical_values = [
        "no_clipping_or_overlap",
        "headings_and_page_breaks_ok",
        "captions_and_references_ok",
    ]
    visual_flags = []
    for column in critical_values:
        visual_flags.extend(
            qa[column].astype(str).isin(["NO", "PENDING"]).tolist()
        )

    issue_rows = qa[
        (qa["manual_review_status"] == "ISSUES_FOUND")
        | (
            qa[
                [
                    "no_clipping_or_overlap",
                    "headings_and_page_breaks_ok",
                    "captions_and_references_ok",
                ]
            ].astype(str) == "NO"
        ).any(axis=1)
    ]

    passed_all = (
        all_reviewed
        and (all_statuses == "PASS").all()
        and not any(visual_flags)
        and issue_rows.empty
    )

    state = {
        "phase": "17.10G",
        "updated_at": datetime.now().astimezone().isoformat(),
        "source_checklist": str(SOURCE_CHECKLIST),
        "saved_checklist": str(CHECKLIST_OUT),
        "pages_total": int(len(qa)),
        "pages_reviewed": int((all_statuses != "PENDING").sum()),
        "pages_pending": int((all_statuses == "PENDING").sum()),
        "pages_with_issues": int(
            (all_statuses == "ISSUES_FOUND").sum()
        ),
        "pages_with_minor_issues": int(
            (all_statuses == "PASS_WITH_MINOR_ISSUES").sum()
        ),
        "all_pages_reviewed": all_reviewed,
        "visual_qa_passed": bool(passed_all),
        "licence_review": "OPEN",
        "submission_ready": False,
        "production_enabled": False,
        "source_data_modified": False,
        "models_retrained": False,
    }

    STATUS_OUT.write_text(
        json.dumps(state, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    with message_display:
        clear_output(wait=True)
        print(
            f"Saved page {page_selector.value}. "
            f"Reviewed: {state['pages_reviewed']}/{state['pages_total']} | "
            f"Pending: {state['pages_pending']} | "
            f"Issues found: {state['pages_with_issues']} | "
            f"Minor issues: {state['pages_with_minor_issues']}"
        )
        print(f"Checklist: {CHECKLIST_OUT}")
        print(f"Status: {STATUS_OUT}")
        if state["visual_qa_passed"]:
            print(
                "All pages are marked PASS. This is visual QA only; "
                "licence clearance remains a separate release gate."
            )
        elif all_reviewed:
            print(
                "All pages have been reviewed, but one or more issues or "
                "minor-issue statuses still require resolution or sign-off."
            )

def load_page(page_number):
    idx = row_index(page_number)
    row = qa.loc[idx]

    review_status.value = str(row["manual_review_status"])
    for column, _ in qa_fields:
        controls[column].value = str(row[column])
    notes.value = str(row["issues_and_notes"])

    with page_display:
        clear_output(wait=True)
        print(f"Page {page_number} of {len(qa)}")
        display(Image(filename=str(row["render_file"]), width=850))

def on_page_change(change):
    if change["name"] == "value":
        # Persist the page being edited before switching.
        persist_review()
        load_page(change["new"])

page_selector.observe(on_page_change, names="value")

def on_save(_):
    persist_review()

def on_previous(_):
    idx = page_numbers.index(page_selector.value)
    if idx > 0:
        page_selector.value = page_numbers[idx - 1]

def on_next(_):
    idx = page_numbers.index(page_selector.value)
    if idx < len(page_numbers) - 1:
        page_selector.value = page_numbers[idx + 1]

save_button = widgets.Button(
    description="Save this page",
    button_style="primary",
    icon="save",
)
previous_button = widgets.Button(description="Previous page", icon="arrow-left")
next_button = widgets.Button(description="Next page", icon="arrow-right")

save_button.on_click(on_save)
previous_button.on_click(on_previous)
next_button.on_click(on_next)

display(Markdown(
    "## Phase 17.10G — Manual page review\n"
    "Inspect each page image below at readable size. Set the page status and "
    "mark the layout checks. Notes are especially useful for pages that need "
    "correction. Changes are saved when you switch pages or click **Save this page**."
))
display(page_selector)
display(previous_button, next_button, save_button)
display(review_status)
for column, label in qa_fields:
    display(controls[column])
display(notes)
display(message_display)
display(page_display)

# Display the first page and initialize a separate saved checklist.
persist_review()
load_page(page_selector.value)

print("\nNew review workspace:", OUT)
print("Source checklist preserved:", SOURCE_CHECKLIST)
print("Review all pages; leave a field Pending if you have not inspected it.")
print("Do not mark PASS without looking at the rendered page.")

In [ ]:
# PHASE 17.10G-R1 — PACKAGE VISUAL QA INPUTS FOR REVIEW
# Read-only collection of existing exports, page renders and checklists.

from pathlib import Path
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib
import csv
import json

from google.colab import files

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

EXPORT_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10D_rendered_review_export_20261025T185523"
)

CHECKLIST_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10G_interactive_page_review_20261025T192508"
)

SOURCE_CHECKLIST_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10E_human_visual_qa_20261025T185812"
)

PAGE_DIR = EXPORT_DIR / "rendered_pages"
CONTACT_DIR = EXPORT_DIR / "contact_sheets"

PDF_PATH = EXPORT_DIR / "Delhi_AQI_Integrated_Review_Draft.pdf"
DOCX_PATH = EXPORT_DIR / "Delhi_AQI_Integrated_Review_Draft.docx"

CHECKLIST_PATH = (
    CHECKLIST_DIR / "phase17_10G_completed_page_checklist.csv"
)
STATUS_PATH = (
    CHECKLIST_DIR / "phase17_10G_visual_qa_status.json"
)
SOURCE_CHECKLIST_PATH = (
    SOURCE_CHECKLIST_DIR / "phase17_10E_page_review_checklist.csv"
)
QA_PATH = EXPORT_DIR / "phase17_10D_render_qa.json"
PAGE_INVENTORY_PATH = EXPORT_DIR / "rendered_page_inventory.csv"

# Verify required assets before writing anything.
required_files = [
    PDF_PATH, DOCX_PATH, CHECKLIST_PATH,
    STATUS_PATH, SOURCE_CHECKLIST_PATH, QA_PATH,
    PAGE_INVENTORY_PATH
]
missing = [str(p) for p in required_files if not p.is_file()]

if missing:
    raise FileNotFoundError(
        "Required visual QA inputs are missing:\n" + "\n".join(missing)
    )

page_pngs = sorted(PAGE_DIR.rglob("*.png"))
contact_pngs = sorted(CONTACT_DIR.glob("*.png"))

if len(page_pngs) != 19:
    raise RuntimeError(
        f"Expected 19 page renders, found {len(page_pngs)} "
        f"under {PAGE_DIR}"
    )

if len(contact_pngs) < 2:
    raise RuntimeError(
        f"Expected at least 2 contact sheets, found {len(contact_pngs)} "
        f"under {CONTACT_DIR}"
    )

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT_DIR = (
    CHECKLIST_DIR / f"review_upload_package_{stamp}"
)
OUT_DIR.mkdir(parents=True, exist_ok=False)

ZIP_PATH = OUT_DIR / "Delhi_AQI_Visual_QA_Review_Inputs.zip"

items = []

for path in contact_pngs:
    items.append(("contact_sheets/" + path.name, path))

for path in page_pngs:
    items.append(("rendered_pages/" + path.name, path))

document_items = [
    ("document/Delhi_AQI_Integrated_Review_Draft.pdf", PDF_PATH),
    ("document/Delhi_AQI_Integrated_Review_Draft.docx", DOCX_PATH),
]

checklist_items = [
    ("checklists/phase17_10G_completed_page_checklist.csv", CHECKLIST_PATH),
    ("checklists/phase17_10G_visual_qa_status.json", STATUS_PATH),
    ("checklists/phase17_10E_page_review_checklist.csv", SOURCE_CHECKLIST_PATH),
    ("audit/phase17_10D_render_qa.json", QA_PATH),
    ("audit/rendered_page_inventory.csv", PAGE_INVENTORY_PATH),
]

items.extend(document_items)
items.extend(checklist_items)

manifest = []
for archive_name, path in items:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest.append({
        "archive_path": archive_name,
        "original_path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": digest,
    })

manifest_path = OUT_DIR / "upload_package_manifest.csv"
with open(manifest_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=manifest[0].keys())
    writer.writeheader()
    writer.writerows(manifest)

items.append(("audit/upload_package_manifest.csv", manifest_path))

with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as z:
    for archive_name, path in items:
        z.write(path, arcname=archive_name)

with ZipFile(ZIP_PATH, "r") as z:
    bad = z.testzip()
    if bad:
        raise RuntimeError(f"Review ZIP integrity failure: {bad}")

zip_hash = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()

print("=" * 72)
print("PHASE 17.10G-R1 — VISUAL QA INPUT PACKAGE")
print("=" * 72)
print("Page PNGs:", len(page_pngs))
print("Contact sheets:", len(contact_pngs))
print("PDF included:", PDF_PATH.is_file())
print("DOCX included:", DOCX_PATH.is_file())
print("Checklist included:", CHECKLIST_PATH.is_file())
print("ZIP integrity: PASS")
print("ZIP size:", f"{ZIP_PATH.stat().st_size:,}", "bytes")
print("SHA-256:", zip_hash)
print("Package:", ZIP_PATH)
print("=" * 72)

files.download(str(ZIP_PATH))

In [ ]:
# PHASE 17.10G-R2 — RECORD MANUAL VISUAL QA FINDINGS
# Creates a new reviewed checklist and issue register.
# Does not modify the current manuscript exports or frozen assets.

from pathlib import Path
from datetime import datetime
import json
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

SOURCE_CHECKLIST = (
    ROOT / "outputs/research_paper"
    / "phase17_10G_interactive_page_review_20261025T192508"
    / "phase17_10G_completed_page_checklist.csv"
)

if not SOURCE_CHECKLIST.is_file():
    raise FileNotFoundError(
        f"Phase 17.10G checklist not found: {SOURCE_CHECKLIST}"
    )

checklist = pd.read_csv(SOURCE_CHECKLIST)
if len(checklist) != 19 or sorted(checklist["page"].astype(int).tolist()) != list(range(1, 20)):
    raise RuntimeError(
        "Expected exactly one checklist entry for each page 1–19. "
        "No new audit output was created."
    )

# Status, page-level evidence, and review notes from the visual inspection.
review = {
    1: {
        "level": "NO_MAJOR_ISSUE",
        "tables": "N/A",
        "figures": "N/A",
        "breaks": "YES",
        "captions": "YES",
        "notes": "Title, abstract and introduction look readable. Remove or replace the 'Working manuscript draft' subtitle in a final submission export."
    },
    2: {
        "level": "NO_OBVIOUS_ISSUE_CONTACT_SHEET",
        "tables": "N/A",
        "figures": "N/A",
        "breaks": "YES",
        "captions": "YES",
        "notes": "Reviewed via contact sheet; no obvious clipping or page-flow issue at overview scale."
    },
    3: {
        "level": "NO_OBVIOUS_ISSUE_CONTACT_SHEET",
        "tables": "N/A",
        "figures": "N/A",
        "breaks": "YES",
        "captions": "YES",
        "notes": "Reviewed via contact sheet; methods text and headings appear orderly at overview scale."
    },
    4: {
        "level": "MINOR_LAYOUT_ISSUE",
        "tables": "PARTIAL",
        "figures": "N/A",
        "breaks": "PARTIAL",
        "captions": "YES",
        "notes": "Validation table is visible, but model names and headers wrap awkwardly. Long names such as HistGradientBoosting and Linear Regression break across multiple lines."
    },
    5: {
        "level": "MINOR_LAYOUT_ISSUE",
        "tables": "PARTIAL",
        "figures": "N/A",
        "breaks": "PARTIAL",
        "captions": "YES",
        "notes": "Performance and bootstrap tables are readable, but labels such as Random Forest and Weighted ensemble wrap over multiple lines. Improve widths or use compact labels."
    },
    6: {
        "level": "NO_MAJOR_ISSUE",
        "tables": "YES",
        "figures": "N/A",
        "breaks": "YES",
        "captions": "YES",
        "notes": "AQI-band, station-level and season-level summary tables appear readable with no obvious clipping."
    },
    7: {
        "level": "MINOR_LAYOUT_ISSUE",
        "tables": "PARTIAL",
        "figures": "N/A",
        "breaks": "YES",
        "captions": "YES",
        "notes": "Transition labels wrap heavily, but the metrics are visible. Feature-importance table is readable."
    },
    8: {
        "level": "NO_OBVIOUS_ISSUE_CONTACT_SHEET",
        "tables": "N/A",
        "figures": "N/A",
        "breaks": "YES",
        "captions": "YES",
        "notes": "Discussion, limitations and conclusion appear orderly at contact-sheet scale."
    },
    9: {
        "level": "MAJOR_LAYOUT_ISSUE",
        "tables": "N/A",
        "figures": "YES",
        "breaks": "NO",
        "captions": "NO",
        "notes": "Figure 1 is legible, but its short italic caption is at the bottom of this page while the full bold caption begins on page 10. This also creates a duplicated caption."
    },
    10: {
        "level": "MAJOR_LAYOUT_ISSUE",
        "tables": "N/A",
        "figures": "YES",
        "breaks": "NO",
        "captions": "NO",
        "notes": "Figure 1's full caption and literal PDF path appear at the top. Figure 2 also has both a short italic caption and a full bold caption, followed by a visible PDF file path."
    },
    11: {
        "level": "MAJOR_LAYOUT_ISSUE",
        "tables": "N/A",
        "figures": "YES",
        "breaks": "NO",
        "captions": "NO",
        "notes": "Figures 3 and 4 have duplicate captions and printed PDF file paths. Figure 4's PDF-path line is orphaned at the top of page 12."
    },
    12: {
        "level": "MAJOR_LAYOUT_ISSUE",
        "tables": "N/A",
        "figures": "YES",
        "breaks": "NO",
        "captions": "NO",
        "notes": "Orphaned Figure 4 PDF path at page top; Figure 5 has duplicate captions/path; Figure 6's plot and its full caption are separated across pages."
    },
    13: {
        "level": "MAJOR_LAYOUT_ISSUE",
        "tables": "NO",
        "figures": "YES",
        "breaks": "NO",
        "captions": "NO",
        "notes": "Figure 6's full caption is detached from its plot. Figure 7 repeats its caption and prints its PDF path. Table 1 begins near the page bottom and splits across the next page."
    },
    14: {
        "level": "MAJOR_LAYOUT_ISSUE",
        "tables": "NO",
        "figures": "N/A",
        "breaks": "NO",
        "captions": "YES",
        "notes": "Table 1 row continuation is broken across pages and model names fragment mid-word. Table 2's Support field wraps excessively. Table 3 begins at the bottom of the page."
    },
    15: {
        "level": "MAJOR_LAYOUT_ISSUE",
        "tables": "PARTIAL",
        "figures": "N/A",
        "breaks": "NO",
        "captions": "PARTIAL",
        "notes": "Table 3 continues with a dangling fragment from the Persistence row; the Support column wraps word by word. Table 4 headings are crowded; Table 5 is comparatively readable."
    },
    16: {
        "level": "MAJOR_LAYOUT_ISSUE",
        "tables": "PARTIAL",
        "figures": "N/A",
        "breaks": "NO",
        "captions": "YES",
        "notes": "Table 6 event labels and Table 7 feature names wrap heavily. Table 8's Grouping and Model columns are too narrow, splitting values such as Weighted ensemble."
    },
    17: {
        "level": "MAJOR_LAYOUT_ISSUE",
        "tables": "NO",
        "figures": "N/A",
        "breaks": "NO",
        "captions": "PARTIAL",
        "notes": "Table 8 continues with a dangling fragment and repeated mid-word wrapping. Table 9 starts low on the page and its first row continues onto page 18."
    },
    18: {
        "level": "MAJOR_LAYOUT_ISSUE",
        "tables": "PARTIAL",
        "figures": "N/A",
        "breaks": "NO",
        "captions": "YES",
        "notes": "Table 9 begins with a dangling fragment from Weighted ensemble; season labels wrap. The reference section itself is readable."
    },
    19: {
        "level": "NO_MAJOR_ISSUE",
        "tables": "N/A",
        "figures": "N/A",
        "breaks": "YES",
        "captions": "YES",
        "notes": "References are readable. Large whitespace below the final citation is not by itself a defect."
    },
}

df = checklist.copy()

for idx, row in df.iterrows():
    page = int(row["page"])
    r = review[page]

    df.at[idx, "manual_review_status"] = r["level"]
    # No literal image clipping/overlap was observed; page-break and wrapping
    # failures are recorded separately rather than mislabelled as pixel clipping.
    df.at[idx, "no_clipping_or_overlap"] = "YES — no literal pixel clipping observed"
    df.at[idx, "figures_readable"] = r["figures"]
    df.at[idx, "tables_readable"] = r["tables"]
    df.at[idx, "headings_and_page_breaks_ok"] = r["breaks"]
    df.at[idx, "captions_and_references_ok"] = r["captions"]
    df.at[idx, "issues_and_notes"] = r["notes"]

# Record how each page was reviewed.
if "review_method" not in df.columns:
    df["review_method"] = ""
for idx, row in df.iterrows():
    page = int(row["page"])
    df.at[idx, "review_method"] = (
        "Contact sheet plus full-resolution page"
        if page in {1, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19}
        else "Contact sheet; no unclear issue requiring full-resolution open"
    )

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10G_R2_manual_visual_review_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

checklist_path = OUT / "phase17_10G_R2_completed_visual_qa_checklist.csv"
df.to_csv(checklist_path, index=False)

issue_rows = []
for page, r in review.items():
    if r["level"] in {"MINOR_LAYOUT_ISSUE", "MAJOR_LAYOUT_ISSUE"}:
        issue_rows.append({
            "page": page,
            "severity": (
                "MAJOR" if r["level"] == "MAJOR_LAYOUT_ISSUE" else "MINOR"
            ),
            "tables_readable": r["tables"],
            "figures_readable": r["figures"],
            "headings_and_page_breaks_ok": r["breaks"],
            "captions_and_references_ok": r["captions"],
            "issue": r["notes"],
        })

issues_path = OUT / "phase17_10G_R2_issue_register.csv"
pd.DataFrame(issue_rows).to_csv(issues_path, index=False)

report = """# Phase 17.10G-R2 — Manual Visual QA Findings

## Outcome

**VISUAL QA: FAIL — FIX THE DOCUMENT LAYOUT BEFORE THE NEXT REVIEW EXPORT.**

All 19 pages were reviewed in the two contact sheets. Full-resolution pages
were opened where layouts or tables needed closer inspection.

## Major findings

1. Figures 1–7 display duplicated captions: a short italic caption and a
   separate full bold caption.
2. Internal PDF file paths are visibly printed in the paper, for example
   `PDF version: figures/Figure_04_...pdf`. These belong in the asset manifest,
   not the manuscript body.
3. Figure 1's full caption is separated from its plot; Figure 6's full caption
   is separated from its plot.
4. Tables 1, 3, 8 and 9 have row continuations or fragments split across page
   boundaries.
5. Several table columns are too narrow, causing headers and values to wrap
   awkwardly or split within words.
6. Summary tables already appear in the Results section, then a second
   numbered table set is appended. Use one consistent numbering scheme and
   avoid repeating the same results without a clear supplementary purpose.

## Pages with minor table-layout issues

Pages 4, 5 and 7: table labels or event names wrap excessively, although the
values remain visible.

## Pages without major visible issues

Pages 1–3, 6, 8 and 19 have no obvious major issue at the resolution reviewed.
Pages 2, 3 and 8 were assessed at contact-sheet scale because no unclear layout
element required opening them separately.

## Recommended repair

- Keep the body Results tables and give them a consistent numbering scheme.
- Remove the appended duplicate table renderings from the main manuscript;
  preserve their full CSVs as supplementary review assets.
- Remove all visible "PDF version: ..." path paragraphs from the publication
  DOCX/PDF; retain paths in `figure_manifest.csv`.
- Remove the duplicated italic image-caption lines and retain one full caption
  per figure.
- Keep each figure and its caption together.
- Reformat wide tables using landscape pages, better column widths and suitable
  font sizes, or use concise body tables with complete CSV supplements.
- Re-render DOCX and PDF and repeat page-level QA.

## Release gate

- Visual QA passed: NO
- Author licence review: OPEN
- Submission ready: NO
- Production enabled: NO
- Source data modified: NO
- Frozen models retrained: NO
- Existing review bundle modified: NO
"""

report_path = OUT / "phase17_10G_R2_visual_qa_report.md"
report_path.write_text(report, encoding="utf-8")

status = {
    "phase": "17.10G-R2",
    "created_at": datetime.now().astimezone().isoformat(),
    "pages_total": 19,
    "pages_reviewed": 19,
    "pages_pending": 0,
    "pages_with_issues": 13,
    "pages_with_minor_issues": 3,
    "pages_with_major_issues": 10,
    "all_pages_reviewed": True,
    "visual_qa_passed": False,
    "blocking_findings": [
        "Duplicate figure captions and printed PDF file paths.",
        "Figure-caption page separation.",
        "Table rows split across pages.",
        "Narrow table columns fragment labels and values.",
        "Main-body summary tables and appended tables need one consistent structure."
    ],
    "licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "models_retrained": False,
    "existing_review_bundle_modified": False,
    "outputs": {
        "completed_checklist": str(checklist_path),
        "issue_register": str(issues_path),
        "report": str(report_path),
    },
}
status_path = OUT / "phase17_10G_R2_visual_qa_status.json"
status_path.write_text(
    json.dumps(status, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 78)
print("PHASE 17.10G-R2 — MANUAL VISUAL QA RECORDED")
print("=" * 78)
print("Pages reviewed:", status["pages_reviewed"])
print("Pages with minor issues:", status["pages_with_minor_issues"])
print("Pages with major issues:", status["pages_with_major_issues"])
print("Visual QA passed:", status["visual_qa_passed"])
print("\nChecklist:", checklist_path)
print("Issue register:", issues_path)
print("QA report:", report_path)
print("Status:", status_path)
print("\nRecommended next phase: repair DOCX/PDF captions and table layout")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.10H-R1 — REPAIR DOCUMENT LAYOUT AND RE-RENDER
# Rebuilds from integrated Markdown rather than modifying the failed export.
# Preserves original review ZIP, source data, model artifacts, and prior QA.

from pathlib import Path
from zipfile import ZipFile
from datetime import datetime
from PIL import Image, ImageOps, ImageDraw
import hashlib, json, re, shutil, subprocess, math, csv, warnings

import pandas as pd
from docx import Document
from docx.shared import Inches, Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.section import WD_SECTION, WD_ORIENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.opc.constants import RELATIONSHIP_TYPE as RT

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

SOURCE_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10B_R1_integrated_manuscript_20261025T184741"
)
MARKDOWN = SOURCE_DIR / "Delhi_AQI_Integrated_Manuscript.md"

REVIEW_ZIP = (
    ROOT / "outputs/research_paper"
    / "phase17_10C_final_review_package_20261025T184853"
    / "Delhi_AQI_Integrated_Research_Review_Package.zip"
)
EXPECTED_ZIP_SHA256 = (
    "e60206d8958bfcc0b97e39290e7fa7e63386bdc6570d5b16bb8961ca19d22e59"
)

ISSUE_REGISTER = (
    ROOT / "outputs/research_paper"
    / "phase17_10G_R2_manual_visual_review_20261025T193344"
    / "phase17_10G_R2_issue_register.csv"
)

# Check conversion tools before creating output files.
lo_path = shutil.which("libreoffice") or shutil.which("soffice")
if not lo_path:
    raise RuntimeError(
        "LibreOffice is not installed in this runtime. Install it before rerunning."
    )

if not MARKDOWN.is_file():
    raise FileNotFoundError(f"Integrated manuscript not found: {MARKDOWN}")
if not REVIEW_ZIP.is_file():
    raise FileNotFoundError(f"Verified review ZIP not found: {REVIEW_ZIP}")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

zip_sha = sha256_file(REVIEW_ZIP)
if zip_sha != EXPECTED_ZIP_SHA256:
    raise RuntimeError("Phase 17.10C ZIP SHA-256 mismatch; stopped.")

markdown_text = MARKDOWN.read_text(encoding="utf-8")

# Load the previous issue register to preserve its context in the new QA record.
issue_rows = []
if ISSUE_REGISTER.is_file():
    try:
        issue_rows = pd.read_csv(ISSUE_REGISTER, dtype=str).fillna("").to_dict("records")
    except Exception as exc:
        warnings.warn(f"Could not parse old issue register: {exc}")

print("=" * 78)
print("PHASE 17.10H-R1 — REPAIR DOCUMENT LAYOUT AND RE-RENDER")
print("=" * 78)
print("Review ZIP SHA-256 verified:", zip_sha == EXPECTED_ZIP_SHA256)
print("Source Markdown:", MARKDOWN)
print("Prior issue-register rows loaded:", len(issue_rows))

# ----------------------------------------------------------------------
# 1. Create a new isolated output folder and copy companion assets.
# ----------------------------------------------------------------------

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10H_R1_layout_repair_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

FIG_SOURCE = SOURCE_DIR / "figures"
TABLE_SOURCE = SOURCE_DIR / "tables_csv"

if not FIG_SOURCE.is_dir():
    raise FileNotFoundError(f"Figure directory not found: {FIG_SOURCE}")
if not TABLE_SOURCE.is_dir():
    raise FileNotFoundError(f"Table directory not found: {TABLE_SOURCE}")

FIG_DIR = OUT / "figures"
TABLE_DIR = OUT / "tables_csv"
RENDER_DIR = OUT / "rendered_pages"
SHEET_DIR = OUT / "contact_sheets"

shutil.copytree(FIG_SOURCE, FIG_DIR)
shutil.copytree(TABLE_SOURCE, TABLE_DIR)
shutil.copy2(MARKDOWN, OUT / "source_integrated_manuscript.md")

pngs = sorted(FIG_DIR.glob("*.png"))
pdf_figures = sorted(FIG_DIR.glob("*.pdf"))
table_csvs = sorted(TABLE_DIR.glob("*.csv"))

if len(pngs) != 7 or len(pdf_figures) != 7:
    raise RuntimeError(
        f"Expected 7 PNG and 7 PDF figures; found {len(pngs)} PNG, "
        f"{len(pdf_figures)} PDF."
    )
if len(table_csvs) != 9:
    raise RuntimeError(f"Expected 9 main CSV tables; found {len(table_csvs)}.")

# ----------------------------------------------------------------------
# 2. Parse Markdown into typed blocks while preserving prose and order.
# ----------------------------------------------------------------------

def is_table_line(s):
    return s.strip().startswith("|") and s.strip().endswith("|")

def split_md_row(line):
    s = line.strip()
    cells = re.split(r"(?<!\\)\|", s[1:-1])
    return [c.strip().replace(r"\|", "|") for c in cells]

def is_alignment_row(row):
    return bool(row) and all(
        re.fullmatch(r":?-{3,}:?", cell.replace(" ", "")) for cell in row
    )

def parse_markdown(text):
    lines = text.splitlines()
    blocks = []
    i = 0

    while i < len(lines):
        line = lines[i]
        s = line.strip()

        if not s:
            i += 1
            continue

        # Ignore standalone HTML comments used as editing markers.
        if s.startswith("<!--") and s.endswith("-->"):
            i += 1
            continue

        hm = re.match(r"^(#{1,6})\s+(.*)$", s)
        if hm:
            blocks.append({
                "type": "heading",
                "level": len(hm.group(1)),
                "text": hm.group(2).strip()
            })
            i += 1
            continue

        im = re.match(r"^!\[(.*?)\]\((.*?)\)\s*$", s)
        if im:
            blocks.append({
                "type": "image",
                "alt": im.group(1),
                "path": im.group(2)
            })
            i += 1
            continue

        if is_table_line(s):
            raw_rows = []
            while i < len(lines) and is_table_line(lines[i].strip()):
                raw_rows.append(split_md_row(lines[i]))
                i += 1

            if len(raw_rows) >= 2 and is_alignment_row(raw_rows[1]):
                raw_rows.pop(1)

            max_cols = max((len(row) for row in raw_rows), default=0)
            normalized_rows = [
                row + [""] * (max_cols - len(row))
                for row in raw_rows
            ]
            blocks.append({"type": "table", "rows": normalized_rows})
            continue

        if re.fullmatch(r"(-{3,}|\*{3,}|_{3,})", s):
            blocks.append({"type": "rule"})
            i += 1
            continue

        if re.match(r"^[-*+]\s+", s) or re.match(r"^\d+[.)]\s+", s):
            items = []
            while i < len(lines):
                item = lines[i].strip()
                if re.match(r"^[-*+]\s+", item):
                    items.append(("bullet", re.sub(r"^[-*+]\s+", "", item)))
                    i += 1
                elif re.match(r"^\d+[.)]\s+", item):
                    items.append(("number", re.sub(r"^\d+[.)]\s+", "", item)))
                    i += 1
                elif not item:
                    break
                else:
                    break
            blocks.append({"type": "list", "items": items})
            continue

        # Fold Markdown soft line breaks into a single paragraph.
        paragraph_lines = [s]
        i += 1
        while i < len(lines):
            nxt = lines[i].strip()
            if not nxt:
                break
            if (
                re.match(r"^#{1,6}\s+", nxt)
                or re.match(r"^!\[.*\]\(.*\)$", nxt)
                or is_table_line(nxt)
                or re.match(r"^[-*+]\s+", nxt)
                or re.match(r"^\d+[.)]\s+", nxt)
                or re.fullmatch(r"(-{3,}|\*{3,}|_{3,})", nxt)
                or (nxt.startswith("<!--") and nxt.endswith("-->"))
            ):
                break
            paragraph_lines.append(nxt)
            i += 1

        blocks.append({
            "type": "paragraph",
            "text": " ".join(paragraph_lines)
        })

    return blocks

blocks = parse_markdown(markdown_text)

# Verify the expected content exists before generating a DOCX.
figure_heading_count = sum(
    1 for b in blocks
    if b["type"] == "heading"
    and re.match(r"Figure\s+\d+", b["text"], re.I)
)
table_heading_count = sum(
    1 for b in blocks
    if b["type"] == "heading"
    and re.match(r"Table\s+\d+", b["text"], re.I)
)
image_block_count = sum(1 for b in blocks if b["type"] == "image")
table_block_count = sum(1 for b in blocks if b["type"] == "table")

if image_block_count != 7:
    raise RuntimeError(f"Expected 7 Markdown image links; found {image_block_count}.")
if figure_heading_count != 7:
    raise RuntimeError(f"Expected 7 numbered figure headings; found {figure_heading_count}.")
if table_heading_count != 9:
    raise RuntimeError(f"Expected 9 numbered table headings; found {table_heading_count}.")
if table_block_count != 9:
    raise RuntimeError(f"Expected 9 Markdown tables; found {table_block_count}.")

# ----------------------------------------------------------------------
# 3. Create the DOCX with consistent typography, A4 pages, and page numbers.
# ----------------------------------------------------------------------

doc = Document()
doc.core_properties.title = "Delhi AQI One-Hour-Ahead Forecasting — Integrated Review Draft"
doc.core_properties.subject = "Research manuscript; author review version"
doc.core_properties.comments = (
    "Reformatted review draft. Not submission-ready; licence review remains open."
)

def configure_section(section, landscape=False):
    section.orientation = WD_ORIENT.LANDSCAPE if landscape else WD_ORIENT.PORTRAIT
    if landscape:
        section.page_width = Cm(29.7)
        section.page_height = Cm(21.0)
        section.top_margin = Cm(1.35)
        section.bottom_margin = Cm(1.35)
        section.left_margin = Cm(1.25)
        section.right_margin = Cm(1.25)
    else:
        section.page_width = Cm(21.0)
        section.page_height = Cm(29.7)
        section.top_margin = Cm(1.8)
        section.bottom_margin = Cm(1.7)
        section.left_margin = Cm(2.0)
        section.right_margin = Cm(2.0)
    section.header_distance = Cm(0.8)
    section.footer_distance = Cm(0.8)

configure_section(doc.sections[0], landscape=False)

# Base style: readable academic draft layout.
normal = doc.styles["Normal"]
normal.font.name = "Times New Roman"
normal.font.size = Pt(10.5)
normal.font.color.rgb = RGBColor(30, 30, 30)
normal.paragraph_format.space_after = Pt(5)
normal.paragraph_format.line_spacing = 1.12
normal.paragraph_format.widow_control = True

for style_name, size in [
    ("Title", 22),
    ("Heading 1", 16),
    ("Heading 2", 13),
    ("Heading 3", 11),
    ("Heading 4", 10.5),
]:
    st = doc.styles[style_name]
    st.font.name = "Times New Roman"
    st.font.size = Pt(size)
    st.font.bold = True
    st.font.color.rgb = RGBColor(32, 55, 78)
    st.paragraph_format.keep_with_next = True
    st.paragraph_format.space_before = Pt(10)
    st.paragraph_format.space_after = Pt(5)

def add_page_field(paragraph):
    run = paragraph.add_run()
    begin = OxmlElement("w:fldChar")
    begin.set(qn("w:fldCharType"), "begin")
    instr = OxmlElement("w:instrText")
    instr.set(qn("xml:space"), "preserve")
    instr.text = " PAGE "
    separate = OxmlElement("w:fldChar")
    separate.set(qn("w:fldCharType"), "separate")
    number = OxmlElement("w:t")
    number.text = "1"
    end = OxmlElement("w:fldChar")
    end.set(qn("w:fldCharType"), "end")
    for el in (begin, instr, separate, number, end):
        run._r.append(el)
    run.font.name = "Times New Roman"
    run.font.size = Pt(8)

def add_hyperlink(paragraph, text, url):
    part = paragraph.part
    rid = part.relate_to(url, RT.HYPERLINK, is_external=True)
    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("r:id"), rid)
    run = OxmlElement("w:r")
    props = OxmlElement("w:rPr")

    color = OxmlElement("w:color")
    color.set(qn("w:val"), "0563C1")
    props.append(color)

    underline = OxmlElement("w:u")
    underline.set(qn("w:val"), "single")
    props.append(underline)

    run.append(props)
    text_el = OxmlElement("w:t")
    text_el.text = text
    run.append(text_el)
    hyperlink.append(run)
    paragraph._p.append(hyperlink)

INLINE_PATTERN = re.compile(
    r"\*\*(.+?)\*\*"
    r"|\*(.+?)\*"
    r"|`([^`]+)`"
    r"|\[([^\]]+)\]\((https?://[^)\s]+|[^)]+)\)"
)

def add_inline(paragraph, text):
    # Remove the review-only subtitle, retaining the document's draft status.
    if text.strip().casefold() == "working manuscript draft":
        text = "Integrated manuscript — author review version"

    pos = 0
    for match in INLINE_PATTERN.finditer(text):
        if match.start() > pos:
            paragraph.add_run(text[pos:match.start()])

        if match.group(1) is not None:
            run = paragraph.add_run(match.group(1))
            run.bold = True
        elif match.group(2) is not None:
            run = paragraph.add_run(match.group(2))
            run.italic = True
        elif match.group(3) is not None:
            run = paragraph.add_run(match.group(3))
            run.font.name = "Consolas"
            run.font.size = Pt(9)
        else:
            add_hyperlink(paragraph, match.group(4), match.group(5))

        pos = match.end()

    if pos < len(text):
        paragraph.add_run(text[pos:])

    # Guard against empty paragraphs when the input line was blank after replacement.
    if not paragraph.text and text:
        paragraph.add_run(text)

def cell_shading(cell, fill="DCE6F1"):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)

def set_repeat_table_header(row):
    tr_pr = row._tr.get_or_add_trPr()
    header = OxmlElement("w:tblHeader")
    header.set(qn("w:val"), "true")
    tr_pr.append(header)

def set_row_cant_split(row):
    tr_pr = row._tr.get_or_add_trPr()
    el = OxmlElement("w:cantSplit")
    tr_pr.append(el)

def add_markdown_table(rows):
    if not rows:
        return

    ncols = max(len(r) for r in rows)
    nrows = len(rows)
    current_section = doc.sections[-1]
    usable_width_in = (
        current_section.page_width.inches
        - current_section.left_margin.inches
        - current_section.right_margin.inches
    )

    wide = ncols > 5
    font_size = 7.2 if wide else (8.2 if ncols > 3 else 9.0)

    table = doc.add_table(rows=nrows, cols=ncols)
    table.style = "Table Grid"
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False

    # Wide scientific tables use landscape pages and balanced column widths.
    if wide:
        weights = [1.5] + [1.0] * (ncols - 1)
    else:
        weights = [1.4] + [1.0] * (ncols - 1)

    total_weight = sum(weights)
    widths = [
        Inches(usable_width_in * weight / total_weight)
        for weight in weights
    ]

    for col_idx, width in enumerate(widths):
        table.columns[col_idx].width = width

    for r_idx, row_values in enumerate(rows):
        row = table.rows[r_idx]
        set_row_cant_split(row)
        if r_idx == 0:
            set_repeat_table_header(row)

        for c_idx in range(ncols):
            cell = row.cells[c_idx]
            cell.width = widths[c_idx]
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            cell.text = ""
            p = cell.paragraphs[0]
            p.paragraph_format.space_before = Pt(0)
            p.paragraph_format.space_after = Pt(0)
            p.paragraph_format.line_spacing = 1.0
            p.alignment = (
                WD_ALIGN_PARAGRAPH.LEFT if c_idx == 0
                else WD_ALIGN_PARAGRAPH.CENTER
            )
            add_inline(p, str(row_values[c_idx]) if c_idx < len(row_values) else "")

            for run in p.runs:
                run.font.name = "Times New Roman"
                run.font.size = Pt(font_size)
                if r_idx == 0:
                    run.bold = True

            if r_idx == 0:
                cell_shading(cell)

    # Keep table paragraphs compact and avoid large whitespace.
    doc.add_paragraph().paragraph_format.space_after = Pt(1)
    return table

def insert_section(landscape):
    sec = doc.add_section(WD_SECTION.NEW_PAGE)
    configure_section(sec, landscape=landscape)
    return sec

current_landscape = False
landscape_table_pending = False
first_title_used = False
missing_images = []
tables_written = 0
figures_written = 0

for idx, block in enumerate(blocks):
    typ = block["type"]

    # Determine whether a table heading is immediately followed by a wide table.
    wide_table_heading = False
    if typ == "heading" and re.match(r"Table\s+\d+", block["text"], re.I):
        if idx + 1 < len(blocks) and blocks[idx + 1]["type"] == "table":
            ncols_next = max(
                (len(r) for r in blocks[idx + 1]["rows"]),
                default=0
            )
            wide_table_heading = ncols_next > 5

    # Return to portrait at the first ordinary content block after a landscape
    # table, unless the next table is also wide and can share the landscape section.
    if (
        current_landscape
        and not landscape_table_pending
        and not wide_table_heading
        and typ != "table"
    ):
        insert_section(landscape=False)
        current_landscape = False

    if typ == "heading":
        text = block["text"]
        level = block["level"]
        is_table_title = bool(re.match(r"Table\s+\d+", text, re.I))
        fig_match = re.match(r"Figure\s+(\d+)", text, re.I)

        if is_table_title:
            if wide_table_heading:
                if not current_landscape:
                    insert_section(landscape=True)
                    current_landscape = True
                else:
                    doc.add_page_break()
                landscape_table_pending = True
            else:
                doc.add_page_break()

        elif fig_match:
            figure_no = int(fig_match.group(1))
            if figure_no > 1:
                doc.add_page_break()

        if level == 1 and not first_title_used:
            paragraph = doc.add_paragraph(style="Title")
            paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
            first_title_used = True
        else:
            style_level = min(max(level, 1), 4)
            paragraph = doc.add_paragraph(style=f"Heading {style_level}")

        add_inline(paragraph, text)
        paragraph.paragraph_format.keep_with_next = True

    elif typ == "paragraph":
        paragraph = doc.add_paragraph()
        text = block["text"]
        add_inline(paragraph, text)
        paragraph.paragraph_format.widow_control = True

        # Style figure captions as compact, readable explanatory paragraphs.
        if re.match(r"^\*\*Figure\s+\d+\.", text, re.I):
            paragraph.paragraph_format.keep_together = True
            paragraph.paragraph_format.space_before = Pt(3)
            paragraph.paragraph_format.space_after = Pt(7)

    elif typ == "image":
        rel_path = block["path"].split("#")[0]
        image_path = (OUT / rel_path).resolve()

        if not image_path.is_file():
            fallback = (SOURCE_DIR / rel_path).resolve()
            if fallback.is_file():
                image_path = fallback
            else:
                missing_images.append(rel_path)
                paragraph = doc.add_paragraph()
                paragraph.add_run(f"[IMAGE MISSING: {rel_path}]").bold = True
                continue

        with Image.open(image_path) as im:
            w_px, h_px = im.size

        max_w, max_h = 6.2, 5.7
        scale = min(max_w / (w_px / 96), max_h / (h_px / 96))
        pic_w = Inches((w_px / 96) * scale)

        paragraph = doc.add_paragraph()
        paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
        paragraph.paragraph_format.keep_with_next = True
        paragraph.paragraph_format.space_before = Pt(3)
        paragraph.paragraph_format.space_after = Pt(5)
        paragraph.add_run().add_picture(str(image_path), width=pic_w)
        figures_written += 1

    elif typ == "table":
        # Standalone wide tables without their own numbered heading are also
        # placed on landscape pages to prevent squeezed columns.
        ncols = max((len(r) for r in block["rows"]), default=0)

        if ncols > 5 and not landscape_table_pending:
            if not current_landscape:
                insert_section(landscape=True)
                current_landscape = True
            landscape_table_pending = True

        add_markdown_table(block["rows"])
        tables_written += 1

        if landscape_table_pending:
            landscape_table_pending = False

    elif typ == "list":
        for kind, text in block["items"]:
            style = "List Bullet" if kind == "bullet" else "List Number"
            paragraph = doc.add_paragraph(style=style)
            add_inline(paragraph, text)
            paragraph.paragraph_format.space_after = Pt(2)

    elif typ == "rule":
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(2)
        p.paragraph_format.space_after = Pt(2)
        p.add_run("—" * 24).font.color.rgb = RGBColor(160, 160, 160)

# Any accidental missing linked images are a hard QA warning.
if missing_images:
    raise FileNotFoundError(
        "Some linked manuscript images could not be resolved:\n"
        + "\n".join(missing_images)
    )

# Apply page numbers and a subtle running header to every section.
for section in doc.sections:
    section.header.is_linked_to_previous = False
    hp = section.header.paragraphs[0]
    hp.text = "Delhi AQI t+1 Forecasting — Author Review Draft"
    hp.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    for r in hp.runs:
        r.font.name = "Times New Roman"
        r.font.size = Pt(8)
        r.font.italic = True
        r.font.color.rgb = RGBColor(100, 100, 100)

    section.footer.is_linked_to_previous = False
    fp = section.footer.paragraphs[0]
    fp.text = ""
    fp.alignment = WD_ALIGN_PARAGRAPH.CENTER
    fp.add_run("Page ")
    add_page_field(fp)

DOCX = OUT / "Delhi_AQI_Layout_Repaired_Review_Draft.docx"
doc.save(DOCX)

# ----------------------------------------------------------------------
# 4. Convert to PDF, render each page, and build contact sheets.
# ----------------------------------------------------------------------

PDF = OUT / "Delhi_AQI_Layout_Repaired_Review_Draft.pdf"

cmd = [
    lo_path,
    f"-env:UserInstallation=file:///tmp/lo_profile_phase17_10H_{stamp}",
    "--headless",
    "--convert-to", "pdf",
    "--outdir", str(OUT),
    str(DOCX),
]
conversion = subprocess.run(
    cmd, capture_output=True, text=True, timeout=240
)

if not PDF.is_file():
    raise RuntimeError(
        "LibreOffice did not create the PDF.\n"
        f"stdout:\n{conversion.stdout}\n"
        f"stderr:\n{conversion.stderr}"
    )

try:
    import pymupdf as fitz
except ImportError:
    import fitz

RENDER_DIR.mkdir()
SHEET_DIR.mkdir()

page_records = []
with fitz.open(PDF) as pdf:
    page_count = len(pdf)
    for n, page in enumerate(pdf, start=1):
        pix = page.get_pixmap(matrix=fitz.Matrix(1.45, 1.45), alpha=False)
        png_path = RENDER_DIR / f"page_{n:03d}.png"
        pix.save(png_path)

        page_records.append({
            "page": n,
            "render_file": str(png_path),
            "page_width_points": round(page.rect.width, 2),
            "page_height_points": round(page.rect.height, 2),
            "pdf_text_characters": len(page.get_text("text")),
            "embedded_images": len(page.get_images(full=True)),
            "png_bytes": png_path.stat().st_size,
        })

# Create contact sheets with page labels; visual review is still required.
thumb_w, thumb_h = 250, 324
cols, rows_per_sheet = 4, 4
per_sheet = cols * rows_per_sheet

for start in range(0, len(page_records), per_sheet):
    chunk = page_records[start:start + per_sheet]
    canvas = Image.new(
        "RGB",
        (cols * (thumb_w + 16) + 16, rows_per_sheet * (thumb_h + 42) + 16),
        "white"
    )
    draw = ImageDraw.Draw(canvas)

    for j, record in enumerate(chunk):
        img = Image.open(record["render_file"]).convert("RGB")
        img.thumbnail((thumb_w, thumb_h))
        thumb = Image.new("RGB", (thumb_w, thumb_h), "white")
        thumb.paste(
            img,
            ((thumb_w - img.width) // 2, (thumb_h - img.height) // 2)
        )
        x = 16 + (j % cols) * (thumb_w + 16)
        y = 16 + (j // cols) * (thumb_h + 42)
        canvas.paste(thumb, (x, y))
        draw.text((x, y + thumb_h + 6), f"Page {record['page']}", fill="black")

    sheet_path = SHEET_DIR / f"contact_sheet_{start // per_sheet + 1:02d}.png"
    canvas.save(sheet_path)

# ----------------------------------------------------------------------
# 5. Write the layout QA report. Automated checks do not replace human review.
# ----------------------------------------------------------------------

page_inventory = OUT / "rendered_page_inventory.csv"
pd.DataFrame(page_records).to_csv(page_inventory, index=False)

docx_check = Document(DOCX)
image_relationships = sum(
    1 for rel in docx_check.part.rels.values()
    if "image" in rel.reltype
)
word_table_count = len(docx_check.tables)

report = {
    "phase": "17.10H-R1",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_markdown": str(MARKDOWN),
    "source_review_zip": str(REVIEW_ZIP),
    "source_review_zip_sha256": zip_sha,
    "source_zip_hash_verified": zip_sha == EXPECTED_ZIP_SHA256,
    "prior_manual_issue_register": str(ISSUE_REGISTER),
    "prior_manual_issue_count": len(issue_rows),
    "docx_path": str(DOCX),
    "pdf_path": str(PDF),
    "docx_bytes": DOCX.stat().st_size,
    "pdf_bytes": PDF.stat().st_size,
    "pdf_page_count": page_count,
    "rendered_page_count": len(page_records),
    "contact_sheet_count": len(list(SHEET_DIR.glob("*.png"))),
    "markdown_figure_blocks": image_block_count,
    "figures_written_to_docx": figures_written,
    "embedded_image_relationships": image_relationships,
    "markdown_tables_written": tables_written,
    "word_tables_detected": word_table_count,
    "table_csvs_preserved": len(list(TABLE_DIR.glob("*.csv"))),
    "missing_images": missing_images,
    "automatic_render_consistency": (
        page_count == len(page_records) and not missing_images
    ),
    "human_visual_qa_completed": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "frozen_models_retrained": False,
    "original_review_zip_modified": False,
    "notes": [
        "Wide tables are set on landscape pages with repeated header rows.",
        "Figures are kept with their captions and given controlled maximum dimensions.",
        "Page numbers and running headers are added to each section.",
        "Contact sheets and full-resolution page renders require another human review.",
        "The prior export was not overwritten."
    ],
}

QA_JSON = OUT / "phase17_10H_R1_layout_qa.json"
QA_JSON.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

qa_md = f"""# Phase 17.10H-R1 — Layout Repair QA

## Export
- DOCX: `{DOCX}`
- PDF: `{PDF}`
- PDF pages: {page_count}
- Rendered page images: {len(page_records)}
- Contact sheets: {len(list(SHEET_DIR.glob("*.png")))}
- Word tables: {word_table_count}
- Embedded image relationships: {image_relationships}
- Source CSV tables retained: {len(list(TABLE_DIR.glob("*.csv")))}

## Formatting changes
- A4 page geometry and consistent typography
- Landscape sections for wide data tables
- Repeated header rows on tables
- Controlled figure dimensions and adjacent captions
- Running headers and page-number fields
- Separate new output directory to preserve prior failed export

## Required human review
Open the contact sheets first, then inspect full-size page PNGs where needed.
Check table clipping/wrapping, figure-caption placement, unexpected blank pages,
section orientation, reference list, and all scientific statements.

Human visual QA: **NOT COMPLETED**
Author licence review: **OPEN**
Submission ready: **NO**
Production enabled: **NO**
"""

(QA_MD := OUT / "phase17_10H_R1_layout_qa_report.md").write_text(
    qa_md, encoding="utf-8"
)

print("\n[1] EXPORT")
print("DOCX:", DOCX)
print("DOCX bytes:", DOCX.stat().st_size)
print("PDF:", PDF)
print("PDF bytes:", PDF.stat().st_size)

print("\n[2] LAYOUT COUNTS")
print("PDF pages:", page_count)
print("Rendered page PNGs:", len(page_records))
print("Contact sheets:", len(list(SHEET_DIR.glob("*.png"))))
print("Figures written to DOCX:", figures_written)
print("Image relationships in DOCX:", image_relationships)
print("Markdown tables written:", tables_written)
print("Word tables:", word_table_count)
print("Source table CSVs retained:", len(list(TABLE_DIR.glob("*.csv"))))

print("\n[3] VISUAL REVIEW FILES")
print("Full page renders:", RENDER_DIR)
print("Contact sheets:", SHEET_DIR)
print("Page inventory:", page_inventory)
print("QA report:", QA_MD)
print("QA JSON:", QA_JSON)

print("\n[4] RELEASE GATE")
print("Automated render consistency:", report["automatic_render_consistency"])
print("Human visual QA completed: False — inspect the contact sheets")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data modified: False")
print("Frozen models retrained: False")
print("Original review ZIP modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.10H-R2 — DEDUPLICATE TABLES, REPAIR LAYOUT, AND RE-RENDER
# Creates a new output directory. Does not modify source data, models,
# the original Markdown, or either existing review ZIP.

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from datetime import datetime
import hashlib
import json
import re
import shutil
import subprocess
import warnings

import pandas as pd

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

SOURCE_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10B_R1_integrated_manuscript_20261025T184741"
)
SOURCE_MD = SOURCE_DIR / "Delhi_AQI_Integrated_Manuscript.md"
SOURCE_FIG_DIR = SOURCE_DIR / "figures"
SOURCE_TABLE_DIR = SOURCE_DIR / "tables_csv"

BUNDLE = (
    ROOT / "outputs/research_paper"
    / "phase17_07T_review_bundle_20261025T181246Z"
    / "Delhi_AQI_Updated_Research_Review_Bundle.zip"
)
EXPECTED_BUNDLE_SHA256 = (
    "df185a63cae525152d4fcb05c03872115d9716519e0c11c8c795cb644822bf1b"
)

# These titles correspond to the nine main tables already registered.
TABLE_TITLES = [
    "Validation model comparison",
    "Final holdout comparison on matched support",
    "Final holdout results on full available support",
    "Paired day-block bootstrap results",
    "Performance by AQI band",
    "Abrupt-transition detection",
    "Feature attribution",
    "Station-level performance",
    "Season-level performance",
]

ASSET_MARKER = "<!-- PHASE 17.10B-R1 ASSET INTEGRATION -->"
DUPLICATE_TABLE_HEADING = "### Main Results Tables"
NOTES_HEADING = "### Interpretation and review notes"


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def is_pipe_row(line):
    s = line.strip()
    return s.startswith("|") and s.endswith("|")


def split_pipe_row(line):
    return [cell.strip() for cell in line.strip().strip("|").split("|")]


def is_markdown_separator(line):
    cells = split_pipe_row(line)
    return (
        len(cells) >= 2
        and all(re.fullmatch(r":?-{3,}:?", c.replace(" ", "")) for c in cells)
    )


def find_markdown_tables(lines):
    """Return [start, end) row ranges of Markdown pipe tables."""
    blocks = []
    i = 0
    while i < len(lines) - 1:
        if (
            is_pipe_row(lines[i])
            and is_pipe_row(lines[i + 1])
            and is_markdown_separator(lines[i + 1])
        ):
            start = i
            j = i + 2
            while j < len(lines) and is_pipe_row(lines[j]):
                j += 1
            blocks.append((start, j))
            i = j
        else:
            i += 1
    return blocks


def count_table_headings(text):
    return len(re.findall(
        r"(?im)^\s*#{1,6}\s*Table\s+\d+\s*[.:]",
        text
    ))


def count_markdown_tables(text):
    return len(find_markdown_tables(text.splitlines()))


def locate_numbered_table_heading(lines, table_start):
    """Check nearby lines to avoid adding a second caption to an already numbered table."""
    lo = max(0, table_start - 5)
    near = lines[lo:table_start]
    return any(
        re.match(r"^\s*#{1,6}\s*Table\s+\d+\s*[.:]", line, re.I)
        or re.match(r"^\s*\*\*Table\s+\d+\s*[.:]", line, re.I)
        for line in near
    )


print("=" * 78)
print("PHASE 17.10H-R2 — TABLE DEDUPLICATION AND LAYOUT REPAIR")
print("=" * 78)

# ------------------------------------------------------------------
# 1. Preflight everything before creating the new output directory.
# ------------------------------------------------------------------

for path in [SOURCE_MD, BUNDLE, SOURCE_FIG_DIR, SOURCE_TABLE_DIR]:
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path}")

bundle_sha = sha256_file(BUNDLE)
if bundle_sha != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(
        "Review ZIP SHA-256 does not match the verified archive. "
        "No repair output was created."
    )

source_text = SOURCE_MD.read_text(encoding="utf-8")
if source_text.count(ASSET_MARKER) != 1:
    raise RuntimeError(
        f"Expected exactly one asset marker; found "
        f"{source_text.count(ASSET_MARKER)}. No repair output was created."
    )

body_text, appendix = source_text.split(ASSET_MARKER, 1)
body_lines = body_text.splitlines()
appendix_lines = appendix.splitlines()

body_tables = find_markdown_tables(body_lines)
appendix_tables = find_markdown_tables(appendix_lines)

table_section_start = appendix.find(DUPLICATE_TABLE_HEADING)
notes_section_start = (
    appendix.find(NOTES_HEADING, table_section_start)
    if table_section_start >= 0 else -1
)

if table_section_start < 0 or notes_section_start < 0:
    raise RuntimeError(
        "Could not safely identify the duplicated appended table section. "
        "No repair output was created."
    )

duplicated_table_section = appendix[table_section_start:notes_section_start]
duplicated_section_table_count = count_markdown_tables(
    duplicated_table_section
)

if len(body_tables) != 9:
    raise RuntimeError(
        f"Expected 9 tables in the original manuscript body; found "
        f"{len(body_tables)}. No repair output was created. "
        "Inspect table locations before changing the manuscript."
    )

if duplicated_section_table_count != 9:
    raise RuntimeError(
        f"Expected 9 tables in the duplicated appended section; found "
        f"{duplicated_section_table_count}. No repair output was created."
    )

if len(list(SOURCE_FIG_DIR.glob("*.png"))) != 7:
    raise FileNotFoundError("Expected 7 PNG figure assets in the source pack.")
if len(list(SOURCE_FIG_DIR.glob("*.pdf"))) != 7:
    raise FileNotFoundError("Expected 7 PDF figure assets in the source pack.")
if len(list(SOURCE_TABLE_DIR.glob("*.csv"))) != 9:
    raise FileNotFoundError("Expected 9 table CSV files in the source pack.")

print("\n[1] PREFLIGHT")
print("Verified review ZIP SHA-256:", bundle_sha == EXPECTED_BUNDLE_SHA256)
print("Tables before appended asset section:", len(body_tables))
print("Tables in appended duplicate section:", duplicated_section_table_count)
print("Duplicate tables identified:", len(body_tables))
print("Figure assets: 7 PNG + 7 PDF")
print("Table CSVs: 9")

# ------------------------------------------------------------------
# 2. Keep the original tables in place and number them.
#    Insert backwards so earlier line positions remain stable.
# ------------------------------------------------------------------

original_heading_count = count_table_headings(body_text)
heading_inserts = 0

for idx in reversed(range(len(body_tables))):
    start, end = body_tables[idx]

    if locate_numbered_table_heading(body_lines, start):
        continue

    title = TABLE_TITLES[idx]
    heading = f"### Table {idx + 1}. {title}"
    body_lines[start:start] = [heading, ""]

    heading_inserts += 1

repaired_body = "\n".join(body_lines).rstrip() + "\n"

# ------------------------------------------------------------------
# 3. Remove the duplicated appended table section, preserving figures,
#    review notes, and the reference section.
# ------------------------------------------------------------------

# Remove from "Main Results Tables" up to—but not including—the notes heading.
appendix_repaired = (
    appendix[:table_section_start]
    + appendix[notes_section_start:]
)

appendix_repaired = appendix_repaired.replace(
    "## Integrated Figures and Main Results Tables",
    "## Integrated Figures and Supporting Materials"
)

repaired_markdown = (
    repaired_body.rstrip()
    + "\n\n"
    + ASSET_MARKER
    + appendix_repaired.rstrip()
    + "\n"
)

# ------------------------------------------------------------------
# 4. Validate table/figure structure before writing files.
# ------------------------------------------------------------------

final_table_count = count_markdown_tables(repaired_markdown)
final_heading_count = count_table_headings(repaired_markdown)

if final_table_count != 9:
    raise RuntimeError(
        f"Post-repair validation failed: expected 9 Markdown tables, "
        f"found {final_table_count}. No output folder was created."
    )

if final_heading_count != 9:
    raise RuntimeError(
        f"Post-repair validation failed: expected 9 numbered table headings, "
        f"found {final_heading_count}. No output folder was created."
    )

image_links = re.findall(
    r"!\[[^\]]*\]\((figures/[^)]+)\)",
    repaired_markdown
)

if len(image_links) != 7:
    raise RuntimeError(
        f"Expected 7 Markdown figure links after repair; found {len(image_links)}."
    )

# ------------------------------------------------------------------
# 5. Create a new directory and preserve linked assets.
# ------------------------------------------------------------------

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10H_R2_layout_repair_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

FIG_DIR = OUT / "figures"
TABLE_DIR = OUT / "tables_csv"
FIG_DIR.mkdir()
TABLE_DIR.mkdir()

md_path = OUT / "Delhi_AQI_Layout_Repaired_Manuscript.md"
md_path.write_text(repaired_markdown, encoding="utf-8")

for src in SOURCE_FIG_DIR.glob("*.png"):
    shutil.copy2(src, FIG_DIR / src.name)
for src in SOURCE_FIG_DIR.glob("*.pdf"):
    shutil.copy2(src, FIG_DIR / src.name)
for src in SOURCE_TABLE_DIR.glob("*.csv"):
    shutil.copy2(src, TABLE_DIR / src.name)

# Validate all relative image links resolve inside the new directory.
missing_image_links = [
    link for link in image_links
    if not (OUT / link).is_file()
]
if missing_image_links:
    raise RuntimeError(
        "Some Markdown image links do not resolve after copying: "
        + ", ".join(missing_image_links)
    )

# ------------------------------------------------------------------
# 6. Save layout-repair inventory.
# ------------------------------------------------------------------

table_manifest = []
for csv_path in sorted(TABLE_DIR.glob("*.csv")):
    try:
        frame = pd.read_csv(csv_path)
        table_manifest.append({
            "file": csv_path.name,
            "rows": int(len(frame)),
            "columns": int(len(frame.columns)),
            "sha256": sha256_file(csv_path),
        })
    except Exception as exc:
        table_manifest.append({
            "file": csv_path.name,
            "rows": None,
            "columns": None,
            "sha256": sha256_file(csv_path),
            "error": str(exc),
        })

pd.DataFrame(table_manifest).to_csv(
    OUT / "table_csv_inventory.csv", index=False
)

# ------------------------------------------------------------------
# 7. Optional DOCX/PDF export and render QA.
#    If Pandoc or LibreOffice is unavailable, the repaired Markdown and assets
#    remain saved and the report clearly marks which export step was skipped.
# ------------------------------------------------------------------

docx_path = OUT / "Delhi_AQI_Layout_Repaired_Review_Draft.docx"
pdf_path = OUT / "Delhi_AQI_Layout_Repaired_Review_Draft.pdf"

pandoc_path = shutil.which("pandoc")
if not pandoc_path:
    try:
        import pypandoc
        candidate = pypandoc.get_pandoc_path()
        if candidate and Path(candidate).is_file():
            pandoc_path = candidate
    except Exception:
        pass

docx_created = False
pdf_created = False
word_table_count = None
word_image_count = None
pdf_page_count = None
render_count = 0
contact_sheet_count = 0
export_messages = []

if pandoc_path:
    try:
        cmd = [
            pandoc_path,
            str(md_path),
            "--from=gfm",
            "--to=docx",
            "--standalone",
            "--resource-path=.",
            "-o",
            str(docx_path),
        ]
        result = subprocess.run(
            cmd, cwd=str(OUT), capture_output=True, text=True, check=True
        )
        docx_created = docx_path.is_file()
        export_messages.append("DOCX export: PASS")

        if docx_created:
            try:
                from docx import Document
                doc = Document(str(docx_path))
                word_table_count = len(doc.tables)
                word_image_count = len(doc.inline_shapes)
            except Exception as exc:
                export_messages.append(
                    f"DOCX structural inspection warning: {exc}"
                )

        lo_path = shutil.which("libreoffice") or shutil.which("soffice")
        if lo_path and docx_created:
            result = subprocess.run(
                [
                    lo_path, "--headless", "--convert-to", "pdf",
                    "--outdir", str(OUT), str(docx_path)
                ],
                cwd=str(OUT), capture_output=True, text=True, check=True
            )
            pdf_created = pdf_path.is_file()
            export_messages.append("PDF export: PASS" if pdf_created else
                                   "PDF export: command completed but file not found")
        else:
            export_messages.append(
                "PDF export skipped: LibreOffice/soffice not found."
            )

    except Exception as exc:
        export_messages.append(
            f"Export warning: {type(exc).__name__}: {exc}"
        )
else:
    export_messages.append(
        "DOCX/PDF export skipped: Pandoc not found. "
        "Repaired Markdown and assets were saved."
    )

# Render the PDF and build contact sheets if possible.
if pdf_created:
    try:
        try:
            import pymupdf as fitz
        except ImportError:
            import fitz

        from PIL import Image, ImageOps, ImageDraw

        RENDER_DIR = OUT / "rendered_pages"
        SHEET_DIR = OUT / "contact_sheets"
        RENDER_DIR.mkdir()
        SHEET_DIR.mkdir()

        pdf = fitz.open(str(pdf_path))
        pdf_page_count = len(pdf)
        page_paths = []

        for i, page in enumerate(pdf, start=1):
            pix = page.get_pixmap(
                matrix=fitz.Matrix(1.35, 1.35),
                alpha=False
            )
            page_path = RENDER_DIR / f"page-{i:02d}.png"
            pix.save(str(page_path))
            page_paths.append(page_path)

        render_count = len(page_paths)

        for batch_start in range(0, len(page_paths), 9):
            batch = page_paths[batch_start:batch_start + 9]
            thumb_w, thumb_h = 300, 390
            cols = 3
            rows = (len(batch) + cols - 1) // cols
            margin, label_h = 18, 28

            sheet = Image.new(
                "RGB",
                (
                    cols * (thumb_w + margin) + margin,
                    rows * (thumb_h + label_h + margin) + margin
                ),
                "white"
            )
            draw = ImageDraw.Draw(sheet)

            for j, page_path in enumerate(batch):
                image = Image.open(page_path).convert("RGB")
                image.thumbnail((thumb_w, thumb_h))
                x = margin + (j % cols) * (thumb_w + margin)
                y = margin + (j // cols) * (thumb_h + label_h + margin)
                sheet.paste(image, (x, y))
                draw.text((x, y + thumb_h + 5), page_path.stem, fill="black")

            sheet_path = SHEET_DIR / (
                f"contact-sheet-{batch_start // 9 + 1:02d}.png"
            )
            sheet.save(sheet_path)
            contact_sheet_count += 1

        pdf.close()
        export_messages.append("PDF page render: PASS")
    except Exception as exc:
        export_messages.append(
            f"PDF render warning: {type(exc).__name__}: {exc}"
        )

# ------------------------------------------------------------------
# 8. Write final QA/status record.
# ------------------------------------------------------------------

status = {
    "phase": "17.10H-R2",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_markdown": str(SOURCE_MD),
    "verified_bundle_sha256": bundle_sha,
    "original_body_table_count": len(body_tables),
    "duplicate_appended_table_count_removed": duplicated_section_table_count,
    "final_markdown_table_count": final_table_count,
    "final_numbered_table_heading_count": final_heading_count,
    "figure_link_count": len(image_links),
    "missing_image_links": missing_image_links,
    "source_table_csv_count": len(list(TABLE_DIR.glob("*.csv"))),
    "docx_created": docx_created,
    "pdf_created": pdf_created,
    "word_table_count": word_table_count,
    "word_embedded_image_count": word_image_count,
    "pdf_page_count": pdf_page_count,
    "rendered_page_count": render_count,
    "contact_sheet_count": contact_sheet_count,
    "export_messages": export_messages,
    "original_source_markdown_modified": False,
    "original_review_bundle_modified": False,
    "source_data_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "author_licence_review": "OPEN",
    "human_visual_qa_completed": False,
    "submission_ready": False,
    "outputs": {
        "repaired_markdown": str(md_path),
        "figure_directory": str(FIG_DIR),
        "table_csv_directory": str(TABLE_DIR),
        "docx": str(docx_path) if docx_created else None,
        "pdf": str(pdf_path) if pdf_created else None,
        "rendered_pages": str(OUT / "rendered_pages") if render_count else None,
        "contact_sheets": str(OUT / "contact_sheets") if contact_sheet_count else None,
    },
}

report_path = OUT / "phase17_10H_R2_layout_qa.json"
report_path.write_text(
    json.dumps(status, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

review_path = OUT / "phase17_10H_R2_layout_review.md"
review_path.write_text(
    f"""# Phase 17.10H-R2 — Layout Repair Report

## Table repair
- Tables in original manuscript body: {len(body_tables)}
- Duplicate appended tables removed: {duplicated_section_table_count}
- Final Markdown tables: {final_table_count}
- Final numbered table headings: {final_heading_count}

## Assets
- Markdown figure links: {len(image_links)}
- Missing figure links: {len(missing_image_links)}
- Main table CSV files retained: {len(list(TABLE_DIR.glob("*.csv")))}
- DOCX created: {docx_created}
- DOCX table count: {word_table_count}
- Embedded images detected in DOCX: {word_image_count}
- PDF created: {pdf_created}
- PDF pages: {pdf_page_count}
- Rendered pages: {render_count}
- Contact sheets: {contact_sheet_count}

## Export messages
{chr(10).join("- " + message for message in export_messages)}

## Remaining gates
- Human visual QA: NOT COMPLETE
- Author licence review: OPEN
- Submission ready: NO
- Production enabled: NO

The repair was written to a new directory. The original Markdown, review ZIP,
source data, and frozen models were not modified.
""",
    encoding="utf-8"
)

print("\n[2] REPAIR RESULT")
print("Number of original tables retained:", len(body_tables))
print("Duplicated appended tables removed:", duplicated_section_table_count)
print("Final Markdown table count:", final_table_count)
print("Final numbered table headings:", final_heading_count)
print("Figure links validated:", len(image_links))
print("Missing image links:", len(missing_image_links))

print("\n[3] EXPORT / RENDER RESULT")
for message in export_messages:
    print("-", message)
print("DOCX created:", docx_created)
print("Word tables detected:", word_table_count)
print("Word embedded images:", word_image_count)
print("PDF created:", pdf_created)
print("PDF page count:", pdf_page_count)
print("Rendered PNG pages:", render_count)
print("Contact sheets:", contact_sheet_count)

print("\n[4] OUTPUTS")
print("Output folder:", OUT)
print("Repaired Markdown:", md_path)
print("QA report:", report_path)
print("Review notes:", review_path)
if docx_created:
    print("DOCX:", docx_path)
if pdf_created:
    print("PDF:", pdf_path)
if contact_sheet_count:
    print("Contact sheets:", OUT / "contact_sheets")

print("\n[5] RELEASE GATE")
print("Human visual QA completed: False")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Source data / original Markdown / review ZIP modified: False")
print("Models retrained: False")
print("=" * 78)

In [ ]:
# PHASE 17.10I — CURRENT REPAIRED PDF VISUAL QA WORKSPACE
# Targets the latest Phase 17.10H-R2 PDF, not the stale 19-page export.

from pathlib import Path
from datetime import datetime
import hashlib
import json
import math
import csv

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

PDF_PATH = (
    ROOT / "outputs/research_paper"
    / "phase17_10H_R2_layout_repair_20261025T193923"
    / "Delhi_AQI_Layout_Repaired_Review_Draft.pdf"
)

if not PDF_PATH.is_file():
    raise FileNotFoundError(
        f"Latest repaired PDF not found: {PDF_PATH}\n"
        "Check that Phase 17.10H-R2 completed successfully."
    )

try:
    import pymupdf
except ImportError:
    import fitz as pymupdf

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

pdf_hash = sha256_file(PDF_PATH)

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10I_current_visual_qa_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

PAGE_DIR = OUT / "rendered_pages"
SHEET_DIR = OUT / "contact_sheets"
PAGE_DIR.mkdir()
SHEET_DIR.mkdir()

# Render the exact PDF currently under review.
doc = pymupdf.open(str(PDF_PATH))
if doc.page_count < 1:
    raise RuntimeError("The PDF contains no pages.")

records = []
rendered_images = []

for index, page in enumerate(doc, start=1):
    pix = page.get_pixmap(
        matrix=pymupdf.Matrix(1.6, 1.6),
        alpha=False
    )

    page_path = PAGE_DIR / f"page-{index:02d}.png"
    pix.save(str(page_path))

    with Image.open(page_path) as img:
        rgb = np.asarray(img.convert("RGB"), dtype=np.uint8)
        near_white_ratio = float(
            np.all(rgb >= 245, axis=2).mean()
        )
        dark_ratio = float(
            np.all(rgb <= 60, axis=2).mean()
        )
        pixel_std = float(rgb.std())

        rendered_images.append((index, page_path))

    text = page.get_text("text") or ""
    embedded_images = len(page.get_images(full=True))

    records.append({
        "page": index,
        "render_file": str(page_path),
        "image_dimensions": f"{pix.width}x{pix.height}",
        "pdf_text_characters": len(text.strip()),
        "pdf_embedded_image_count": embedded_images,
        "near_white_pixel_ratio": round(near_white_ratio, 5),
        "dark_pixel_ratio": round(dark_ratio, 5),
        "pixel_standard_deviation": round(pixel_std, 3),
        "automated_flags": "",
        "manual_review_status": "PENDING",
        "no_clipping_or_overlap": "PENDING",
        "figures_readable": "PENDING",
        "tables_readable": "PENDING",
        "headings_and_page_breaks_ok": "PENDING",
        "captions_and_references_ok": "PENDING",
        "issues_and_notes": "",
    })

doc.close()

# Build contact sheets: six pages per sheet, two columns by three rows.
THUMB_WIDTH = 420
LABEL_HEIGHT = 32
CELL_PADDING = 12
COLS, ROWS = 2, 3
PER_SHEET = COLS * ROWS

contact_sheet_paths = []

for start in range(0, len(rendered_images), PER_SHEET):
    group = rendered_images[start:start + PER_SHEET]
    thumbs = []

    for page_num, img_path in group:
        with Image.open(img_path) as img:
            thumb = img.convert("RGB")
            thumb.thumbnail((THUMB_WIDTH, 620))
            thumbs.append((page_num, thumb.copy()))

    max_thumb_h = max(img.height for _, img in thumbs)
    cell_w = THUMB_WIDTH + 2 * CELL_PADDING
    cell_h = max_thumb_h + LABEL_HEIGHT + 2 * CELL_PADDING

    sheet = Image.new(
        "RGB",
        (COLS * cell_w, ROWS * cell_h),
        "white"
    )
    draw = ImageDraw.Draw(sheet)

    for j, (page_num, thumb) in enumerate(thumbs):
        row = j // COLS
        col = j % COLS
        x = col * cell_w + CELL_PADDING
        y = row * cell_h + CELL_PADDING

        draw.text(
            (x, y),
            f"PAGE {page_num}",
            fill="black"
        )
        sheet.paste(thumb, (x, y + LABEL_HEIGHT))

    sheet_num = len(contact_sheet_paths) + 1
    sheet_path = SHEET_DIR / f"contact-sheet-{sheet_num:02d}.png"
    sheet.save(sheet_path)
    contact_sheet_paths.append(sheet_path)

# Save a manually editable CSV checklist.
checklist_path = OUT / "phase17_10I_page_review_checklist.csv"
checklist_df = pd.DataFrame(records)
checklist_df.to_csv(checklist_path, index=False)

# Save a page inventory / provenance summary.
summary = {
    "phase": "17.10I",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_pdf": str(PDF_PATH),
    "source_pdf_sha256": pdf_hash,
    "page_count": len(records),
    "rendered_png_count": len(list(PAGE_DIR.glob("*.png"))),
    "contact_sheet_count": len(contact_sheet_paths),
    "all_pages_rendered": (
        len(records) == len(list(PAGE_DIR.glob("*.png")))
    ),
    "manual_visual_qa": "PENDING",
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "models_retrained": False,
    "review_pdf_modified": False,
    "outputs": {
        "rendered_pages": str(PAGE_DIR),
        "contact_sheets": str(SHEET_DIR),
        "page_review_checklist": str(checklist_path),
    }
}

summary_path = OUT / "phase17_10I_visual_qa_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Instructions for the reviewer.
readme = f"""# Phase 17.10I — Current PDF Visual Review

PDF reviewed:
`{PDF_PATH}`

SHA-256:
`{pdf_hash}`

Pages: {len(records)}
Contact sheets: {len(contact_sheet_paths)}

## Review method
1. Inspect every contact sheet.
2. Check each page at full resolution when text, tables, or charts appear unclear.
3. Look for clipping, overlapping elements, misplaced headings, blank pages,
   split tables, unreadable chart labels, missing captions, and awkward breaks.
4. Update `phase17_10I_page_review_checklist.csv` with PASS/FAIL and notes.
5. Leave the status pending for any page not fully checked.

The checklist is an author/reviewer record. Automated pixel tests are not
a substitute for visual inspection.

Licence review remains open; this workspace does not mark the document
submission-ready.
"""

(OUT / "README.md").write_text(readme, encoding="utf-8")

print("=" * 78)
print("PHASE 17.10I — CURRENT REPAIRED PDF VISUAL QA")
print("=" * 78)
print("PDF:", PDF_PATH)
print("PDF SHA-256:", pdf_hash)
print("Page count:", len(records))
print("Rendered pages:", len(list(PAGE_DIR.glob("*.png"))))
print("Contact sheets:", len(contact_sheet_paths))
print("Checklist:", checklist_path)
print("Summary:", summary_path)
print("\nManual visual QA: PENDING")
print("Licence review: OPEN")
print("Submission ready: False")
print("=" * 78)

# Display contact sheets inline for immediate visual review.
for path in contact_sheet_paths:
    print(f"\nInspect: {path.name}")
    display(Image.open(path))

In [ ]:
# PHASE 17.10J — PREPARE CURRENT VISUAL QA FILES FOR REVIEW

from pathlib import Path
import hashlib
import json
import zipfile

from google.colab import files

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

QA_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10I_current_visual_qa_20261025T194105"
)

PDF = (
    ROOT / "outputs/research_paper"
    / "phase17_10H_R2_layout_repair_20261025T193923"
    / "Delhi_AQI_Layout_Repaired_Review_Draft.pdf"
)

EXPECTED_PDF_SHA256 = (
    "cf52cd6ab824a6cca48d2d721840ec55adb2526ec0dc46b675cdf8fd9245462f"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if not PDF.is_file():
    raise FileNotFoundError(f"Current repaired PDF not found: {PDF}")

if sha256_file(PDF) != EXPECTED_PDF_SHA256:
    raise RuntimeError(
        "PDF hash mismatch. Stop: this is not the expected repaired PDF."
    )

contact_sheets = sorted(QA_DIR.glob("contact-sheet-*.png"))
if len(contact_sheets) != 4:
    raise RuntimeError(
        f"Expected 4 contact sheets in {QA_DIR}, found {len(contact_sheets)}."
    )

checklist = QA_DIR / "phase17_10I_page_review_checklist.csv"
summary = QA_DIR / "phase17_10I_visual_qa_summary.json"

if not checklist.is_file():
    raise FileNotFoundError(f"Review checklist not found: {checklist}")

if not summary.is_file():
    raise FileNotFoundError(f"QA summary not found: {summary}")

output_zip = QA_DIR / "phase17_10J_visual_review_files.zip"

with zipfile.ZipFile(
    output_zip, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    z.write(PDF, arcname="current_repaired_review.pdf")
    z.write(checklist, arcname=checklist.name)
    z.write(summary, arcname=summary.name)

    for sheet in contact_sheets:
        z.write(sheet, arcname=f"contact_sheets/{sheet.name}")

with zipfile.ZipFile(output_zip, "r") as z:
    bad = z.testzip()
    if bad:
        raise RuntimeError(f"Review ZIP integrity failed: {bad}")

print("=" * 72)
print("PHASE 17.10J — VISUAL REVIEW FILES")
print("=" * 72)
print("Current repaired PDF SHA-256 verified: True")
print("Contact sheets packaged:", len(contact_sheets))
print("Checklist included: True")
print("QA summary included: True")
print("Review ZIP integrity: PASS")
print("Output:", output_zip)
print("=" * 72)

files.download(str(output_zip))

In [ ]:
# PHASE 17.10I-R1 — REPAIR MISSING CONTACT SHEETS
# Reuses the existing 19 page renders and creates four sheets in the expected folder.

from pathlib import Path
from datetime import datetime
from PIL import Image, ImageOps, ImageDraw, ImageFont
import pandas as pd
import json
import re

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

RENDER_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10D_rendered_review_export_20261025T185523"
    / "rendered_pages"
)

PAGE_INVENTORY = (
    ROOT / "outputs/research_paper"
    / "phase17_10D_rendered_review_export_20261025T185523"
    / "rendered_page_inventory.csv"
)

QA_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10I_current_visual_qa_20261025T194105"
)

EXPECTED_PAGES = 19
EXPECTED_SHEETS = 4

print("=" * 78)
print("PHASE 17.10I-R1 — REPAIR MISSING CONTACT SHEETS")
print("=" * 78)

# 1. Find existing rendered pages using the page inventory first.
page_items = []

if PAGE_INVENTORY.is_file():
    inventory = pd.read_csv(PAGE_INVENTORY)

    render_col = next(
        (c for c in inventory.columns
         if c.strip().lower() in {"render_file", "render_path", "image_path"}),
        None
    )
    page_col = next(
        (c for c in inventory.columns if c.strip().lower() == "page"),
        None
    )

    if render_col:
        for i, row in inventory.iterrows():
            raw_path = str(row[render_col]).strip()
            if not raw_path or raw_path.lower() == "nan":
                continue

            path = Path(raw_path)
            if not path.is_absolute():
                candidate = RENDER_DIR / path
                path = candidate if candidate.is_file() else ROOT / path

            page_number = (
                int(row[page_col])
                if page_col and pd.notna(row[page_col])
                else i + 1
            )

            if path.is_file():
                page_items.append((page_number, path))

# Fallback: discover PNG renders inside the known rendered-pages folder.
if len(page_items) != EXPECTED_PAGES:
    discovered = []
    if RENDER_DIR.is_dir():
        for path in RENDER_DIR.glob("*.png"):
            numbers = re.findall(r"\d+", path.stem)
            page_number = int(numbers[-1]) if numbers else 999999
            discovered.append((page_number, path))

    discovered.sort(key=lambda item: (item[0], item[1].name))

    if len(discovered) == EXPECTED_PAGES:
        page_items = [
            (i + 1, path) for i, (_, path) in enumerate(discovered)
        ]

# Validate every input before generating sheets.
if len(page_items) != EXPECTED_PAGES:
    raise RuntimeError(
        f"Expected {EXPECTED_PAGES} valid page renders, found {len(page_items)}. "
        f"Check the existing render folder: {RENDER_DIR}"
    )

page_items.sort(key=lambda item: item[0])

for page_num, path in page_items:
    try:
        with Image.open(path) as im:
            im.verify()
    except Exception as exc:
        raise RuntimeError(
            f"Page render {page_num} is unreadable: {path} ({exc})"
        )

QA_DIR.mkdir(parents=True, exist_ok=True)

# 2. Create 4 contact sheets: pages 1–5, 6–10, 11–15, and 16–19.
CELL_W = 390
CELL_H = 500
COLS = 2
ROWS = 3
MARGIN = 18
HEADER_H = 36

SHEET_W = MARGIN * 2 + COLS * CELL_W
SHEET_H = MARGIN * 2 + ROWS * CELL_H

try:
    font = ImageFont.truetype("DejaVuSans.ttf", 17)
    small_font = ImageFont.truetype("DejaVuSans.ttf", 14)
except OSError:
    font = ImageFont.load_default()
    small_font = ImageFont.load_default()

created_sheets = []

for sheet_index in range(EXPECTED_SHEETS):
    start = sheet_index * 5
    group = page_items[start:start + 5]

    sheet = Image.new("RGB", (SHEET_W, SHEET_H), "#F2F4F7")
    draw = ImageDraw.Draw(sheet)

    for tile_index, (page_num, page_path) in enumerate(group):
        row_idx = tile_index // COLS
        col_idx = tile_index % COLS

        x = MARGIN + col_idx * CELL_W
        y = MARGIN + row_idx * CELL_H

        draw.rounded_rectangle(
            [x, y, x + CELL_W - 10, y + CELL_H - 10],
            radius=7,
            fill="white",
            outline="#B8C0CC",
            width=1
        )

        draw.text(
            (x + 10, y + 8),
            f"PDF PAGE {page_num}",
            fill="#172B4D",
            font=font
        )

        with Image.open(page_path) as original:
            page_image = original.convert("RGB")

        page_image.thumbnail(
            (CELL_W - 30, CELL_H - HEADER_H - 25),
            Image.Resampling.LANCZOS
        )

        img_x = x + ((CELL_W - 10) - page_image.width) // 2
        img_y = y + HEADER_H + (
            (CELL_H - HEADER_H - 20) - page_image.height
        ) // 2

        sheet.paste(page_image, (img_x, img_y))

        draw.text(
            (x + 10, y + CELL_H - 31),
            page_path.name,
            fill="#596579",
            font=small_font
        )

    filename = f"contact-sheet-{sheet_index + 1:02d}.png"
    destination = QA_DIR / filename
    sheet.save(destination, format="PNG", optimize=True)

    # Verify that each saved contact sheet can be reopened.
    with Image.open(destination) as check:
        check.verify()

    created_sheets.append({
        "filename": filename,
        "path": str(destination),
        "pages": [
            {
                "page": page_num,
                "render": str(page_path)
            }
            for page_num, page_path in group
        ],
        "dimensions": list(sheet.size),
        "size_bytes": destination.stat().st_size,
    })

# 3. Confirm the exact filenames the failed step expected.
found_sheets = sorted(QA_DIR.glob("contact-sheet-*.png"))

if len(found_sheets) != EXPECTED_SHEETS:
    raise RuntimeError(
        f"Repair validation failed: expected {EXPECTED_SHEETS} sheets, "
        f"found {len(found_sheets)} in {QA_DIR}"
    )

repair_summary = {
    "phase": "17.10I-R1",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_render_dir": str(RENDER_DIR),
    "source_page_inventory": str(PAGE_INVENTORY),
    "expected_page_count": EXPECTED_PAGES,
    "validated_page_count": len(page_items),
    "expected_sheet_count": EXPECTED_SHEETS,
    "created_sheet_count": len(found_sheets),
    "contact_sheets": created_sheets,
    "manual_visual_review_completed": False,
    "original_pdf_modified": False,
    "original_docx_modified": False,
    "original_review_zip_modified": False,
    "source_data_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "licence_review": "OPEN",
}

summary_path = QA_DIR / "phase17_10I_R1_contact_sheet_repair.json"
summary_path.write_text(
    json.dumps(repair_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n[1] PAGE RENDERS")
print("Valid existing pages:", len(page_items))
print("Source render folder:", RENDER_DIR)

print("\n[2] CONTACT SHEETS")
for item in created_sheets:
    print(
        f"{item['filename']} | pages "
        f"{item['pages'][0]['page']}–{item['pages'][-1]['page']} | "
        f"{item['dimensions'][0]}x{item['dimensions'][1]} | "
        f"{item['size_bytes']:,} bytes"
    )

print("\n[3] REPAIR VALIDATION")
print("Expected sheets:", EXPECTED_SHEETS)
print("Found sheets:", len(found_sheets))
print("Validation:", "PASS" if len(found_sheets) == EXPECTED_SHEETS else "FAIL")
print("Repair record:", summary_path)

print("\n[4] RELEASE GATE")
print("Manual visual review completed: False — inspect the contact sheets")
print("Licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Original PDF/DOCX/ZIP and source data modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.10I-R2 — DISPLAY REPAIRED CONTACT SHEETS FOR VISUAL REVIEW

from pathlib import Path
from IPython.display import display, Image as NotebookImage

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

QA_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10I_current_visual_qa_20261025T194105"
)

RENDER_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10D_rendered_review_export_20261025T185523"
    / "rendered_pages"
)

# Change this list to inspect specific pages at full resolution.
# Example: [3, 8, 12, 17]
PAGE_NUMBERS = []

sheets = sorted(
    QA_DIR.rglob("contact-sheet-*.png"),
    key=lambda p: p.name
)

if len(sheets) != 4:
    raise FileNotFoundError(
        f"Expected 4 repaired contact sheets under {QA_DIR}, "
        f"but found {len(sheets)}. Inspect the repair output path."
    )

page_files = sorted(
    RENDER_DIR.glob("*.png"),
    key=lambda p: [
        int(s) if s.isdigit() else s.lower()
        for s in __import__("re").split(r"(\d+)", p.name)
    ]
)

if len(page_files) != 19:
    raise RuntimeError(
        f"Expected 19 full-resolution rendered pages; found {len(page_files)}."
    )

print("=" * 72)
print("PHASE 17.10I-R2 — CONTACT SHEET VISUAL REVIEW")
print("=" * 72)

for i, path in enumerate(sheets, start=1):
    print(f"\nCONTACT SHEET {i}/4: {path.name}")
    display(NotebookImage(filename=str(path)))

print("\n[OPTIONAL] FULL-RESOLUTION PAGE REVIEW")
if PAGE_NUMBERS:
    for n in PAGE_NUMBERS:
        if not 1 <= n <= 19:
            print(f"Skipping invalid page number: {n}")
            continue

        page = page_files[n - 1]
        print(f"\nPAGE {n}: {page.name}")
        display(NotebookImage(filename=str(page)))
else:
    print("No individual pages selected.")
    print("Edit PAGE_NUMBERS near the top of this cell to inspect specific pages.")

print("\n[MANUAL CHECKS — ALL 19 PAGES]")
print("Check for clipped text, overlapping content, unreadable charts/tables,")
print("orphaned headings, broken page breaks, and captions separated from figures.")
print("Do not mark a page PASS unless you have actually inspected it.")

checklist = (
    ROOT / "outputs/research_paper"
    / "phase17_10E_human_visual_qa_20261025T185812"
    / "phase17_10E_page_review_checklist.csv"
)
print("\nExisting review checklist:", checklist)
print("Checklist exists:", checklist.is_file())
print("=" * 72)

In [ ]:
# PHASE 17.10I-R1 — FIX CONTACT-SHEET COUNT AND REBUILD VISUAL QA WORKSPACE
# Creates a fresh workspace. Does not delete or overwrite previous QA assets.

from pathlib import Path
from datetime import datetime
from PIL import Image, ImageOps, ImageDraw, ImageFont
import pandas as pd
import numpy as np
import hashlib
import json
import re
import csv
import warnings

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
RESEARCH = ROOT / "outputs/research_paper"

RENDER_DIR = (
    RESEARCH
    / "phase17_10D_rendered_review_export_20261025T185523"
    / "rendered_pages"
)
PAGE_INVENTORY = (
    RESEARCH
    / "phase17_10D_rendered_review_export_20261025T185523"
    / "rendered_page_inventory.csv"
)
OLD_QA_DIR = (
    RESEARCH / "phase17_10I_current_visual_qa_20261025T194105"
)
PHASE_10E_CHECKLIST = (
    RESEARCH
    / "phase17_10E_human_visual_qa_20261025T185812"
    / "phase17_10E_page_review_checklist.csv"
)

EXPECTED_PAGES = 19
SHEETS_PER_WORKSPACE = 4
PAGES_PER_SHEET = 5

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def natural_key(text):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", str(text))
    ]

def resolve_render_path(value, render_dir):
    """Resolve an inventory path without assuming absolute versus relative paths."""
    if value is None or pd.isna(value):
        return None

    raw = str(value).strip()
    if not raw:
        return None

    candidate = Path(raw)
    tests = [
        candidate,
        render_dir / candidate,
        render_dir / candidate.name,
    ]
    for path in tests:
        if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
            return path.resolve()

    matches = [
        p for p in render_dir.rglob(candidate.name)
        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}
    ] if render_dir.is_dir() else []

    return matches[0].resolve() if len(matches) == 1 else None

print("=" * 78)
print("PHASE 17.10I-R1 — VISUAL QA WORKSPACE REPAIR")
print("=" * 78)

# ------------------------------------------------------------------
# 1. Preflight the source page renders.
# ------------------------------------------------------------------
if not RENDER_DIR.is_dir():
    raise FileNotFoundError(
        f"Original page-render folder not found: {RENDER_DIR}"
    )

page_records = []

if PAGE_INVENTORY.is_file():
    inventory = pd.read_csv(PAGE_INVENTORY)

    page_col = next(
        (c for c in inventory.columns
         if str(c).strip().lower() in {"page", "page_number", "page_no"}),
        None
    )
    file_col = next(
        (c for c in inventory.columns
         if str(c).strip().lower() in {"render_file", "render_path", "file_path"}),
        None
    )

    if page_col is not None and file_col is not None:
        for _, row in inventory.iterrows():
            try:
                page_no = int(row[page_col])
            except (TypeError, ValueError):
                continue

            image_path = resolve_render_path(row[file_col], RENDER_DIR)
            if image_path is not None:
                page_records.append({
                    "page": page_no,
                    "render_path": image_path,
                    "inventory_row": row.to_dict(),
                })

# Use direct discovery only if the inventory could not resolve all pages.
if len(page_records) != EXPECTED_PAGES:
    image_candidates = sorted(
        [
            p.resolve() for p in RENDER_DIR.rglob("*")
            if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}
        ],
        key=natural_key
    )

    # Do not include contact sheets accidentally saved alongside renders.
    image_candidates = [
        p for p in image_candidates
        if not re.search(r"contact[-_ ]?sheet", p.name, re.IGNORECASE)
    ]

    if len(image_candidates) == EXPECTED_PAGES:
        page_records = [
            {
                "page": i,
                "render_path": p,
                "inventory_row": {},
            }
            for i, p in enumerate(image_candidates, start=1)
        ]
    else:
        raise RuntimeError(
            "Could not resolve exactly 19 original rendered pages. "
            f"Inventory-resolved={len(page_records)}; "
            f"page images found={len(image_candidates)}. "
            "No new workspace was created."
        )

page_records = sorted(page_records, key=lambda x: x["page"])

page_numbers = [r["page"] for r in page_records]
page_paths = [str(r["render_path"]) for r in page_records]

if page_numbers != list(range(1, EXPECTED_PAGES + 1)):
    raise RuntimeError(
        "Rendered pages must be numbered 1 through 19 exactly once. "
        f"Found: {page_numbers}. No new workspace was created."
    )

if len(set(page_paths)) != EXPECTED_PAGES:
    raise RuntimeError(
        "The inventory points to duplicate page-image files. "
        "No new workspace was created."
    )

# ------------------------------------------------------------------
# 2. Create a completely new workspace, never replacing the old folder.
# ------------------------------------------------------------------
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = RESEARCH / f"phase17_10I_R1_visual_qa_repaired_{stamp}"
SHEET_DIR = OUT / "canonical_contact_sheets"
SHEET_DIR.mkdir(parents=True, exist_ok=False)

# Four unique groups: pages 1–5, 6–10, 11–15, 16–19.
page_groups = [
    page_records[0:5],
    page_records[5:10],
    page_records[10:15],
    page_records[15:19],
]
if len(page_groups) != SHEETS_PER_WORKSPACE:
    raise RuntimeError(
        f"Internal grouping error: expected four groups, got {len(page_groups)}"
    )

def load_font(size):
    for font_path in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
    ):
        if Path(font_path).exists():
            try:
                return ImageFont.truetype(font_path, size=size)
            except Exception:
                pass
    return ImageFont.load_default()

title_font = load_font(26)
label_font = load_font(19)
note_font = load_font(15)

thumb_width = 540
thumb_height = 700
padding = 22
label_height = 34
cell_width = thumb_width + 2 * padding
cell_height = thumb_height + label_height + 2 * padding
gap_x = 28
gap_y = 24
header_height = 82
columns = 2
rows = 3

sheet_width = 2 * padding + columns * cell_width + (columns - 1) * gap_x
sheet_height = (
    header_height + 2 * padding + rows * cell_height + (rows - 1) * gap_y
)

generated_sheets = []

for sheet_idx, records in enumerate(page_groups, start=1):
    first_page = records[0]["page"]
    last_page = records[-1]["page"]

    canvas = Image.new("RGB", (sheet_width, sheet_height), "white")
    draw = ImageDraw.Draw(canvas)

    draw.text(
        (padding, 18),
        f"Rendered PDF pages {first_page}–{last_page} of {EXPECTED_PAGES}",
        fill="black",
        font=title_font
    )
    draw.text(
        (padding, 52),
        "Contact sheet for overview only — use full-resolution page PNGs to inspect detail.",
        fill="black",
        font=note_font
    )

    for index, record in enumerate(records):
        col = index % columns
        row = index // columns

        x = padding + col * (cell_width + gap_x)
        y = header_height + padding + row * (cell_height + gap_y)

        draw.rounded_rectangle(
            (x, y, x + cell_width, y + cell_height),
            radius=10,
            outline="#b8b8b8",
            width=2,
            fill="#f7f7f7"
        )

        with Image.open(record["render_path"]) as img:
            page_img = ImageOps.exif_transpose(img).convert("RGB")
            original_size = page_img.size
            page_img.thumbnail(
                (thumb_width, thumb_height),
                Image.Resampling.LANCZOS
            )

        image_x = x + (cell_width - page_img.width) // 2
        image_y = y + label_height + padding

        canvas.paste(page_img, (image_x, image_y))
        draw.text(
            (x + padding, y + 7),
            f"Page {record['page']}  ({original_size[0]}×{original_size[1]})",
            fill="black",
            font=label_font
        )

    sheet_path = SHEET_DIR / f"contact-sheet-{sheet_idx:02d}.png"
    canvas.save(sheet_path, format="PNG", optimize=True)
    generated_sheets.append(sheet_path)

# Important: count only the dedicated canonical subfolder.
actual_sheets = sorted(SHEET_DIR.glob("contact-sheet-*.png"))

if len(actual_sheets) != SHEETS_PER_WORKSPACE:
    raise RuntimeError(
        "Canonical contact-sheet verification failed: "
        f"expected {SHEETS_PER_WORKSPACE}, found {len(actual_sheets)}."
    )

# ------------------------------------------------------------------
# 3. Rebuild a 19-page checklist, preserving any existing manual decisions.
# ------------------------------------------------------------------
manual_columns = [
    "manual_review_status",
    "no_clipping_or_overlap",
    "figures_readable",
    "tables_readable",
    "headings_and_page_breaks_ok",
    "captions_and_references_ok",
    "issues_and_notes",
]

# Search existing QA checklists, most relevant workspaces first, without
# deleting any of them.
checklist_candidates = []

if OLD_QA_DIR.is_dir():
    checklist_candidates.extend(OLD_QA_DIR.rglob("*checklist*.csv"))

if PHASE_10E_CHECKLIST.is_file():
    checklist_candidates.append(PHASE_10E_CHECKLIST)

# Include other earlier phase-10 checklists if available.
for p in RESEARCH.glob("phase17_10*_*/**/*checklist*.csv"):
    checklist_candidates.append(p)

unique_candidates = []
seen = set()
for p in checklist_candidates:
    try:
        resolved = p.resolve()
    except Exception:
        resolved = p
    if resolved not in seen and p.is_file():
        seen.add(resolved)
        unique_candidates.append(p)

unique_candidates.sort(key=lambda p: p.stat().st_mtime)

existing_rows_by_page = {}
checklist_columns = set()

for p in unique_candidates:
    try:
        prior = pd.read_csv(p)
    except Exception as exc:
        warnings.warn(f"Could not read prior checklist {p}: {exc}")
        continue

    page_col = next(
        (c for c in prior.columns
         if str(c).strip().lower() in {"page", "page_number", "page_no"}),
        None
    )
    if page_col is None:
        continue

    checklist_columns.update(prior.columns)

    for _, prior_row in prior.iterrows():
        try:
            page_no = int(prior_row[page_col])
        except (TypeError, ValueError):
            continue
        if page_no not in range(1, EXPECTED_PAGES + 1):
            continue

        destination = existing_rows_by_page.setdefault(page_no, {})

        for col, value in prior_row.to_dict().items():
            if pd.isna(value):
                continue
            value_text = str(value).strip()
            if value_text == "":
                continue

            # Never let a later PENDING placeholder erase an earlier
            # non-pending human decision.
            if value_text.upper() == "PENDING" and col in destination:
                if str(destination[col]).strip().upper() != "PENDING":
                    continue

            destination[col] = value

standard_columns = [
    "page",
    "render_file",
    "image_dimensions",
    "pdf_text_characters",
    "pdf_embedded_image_count",
    "near_white_pixel_ratio",
    "dark_pixel_ratio",
    "pixel_standard_deviation",
    "automated_flags",
    *manual_columns,
]

checklist_rows = []
for record in page_records:
    page_no = record["page"]
    inventory_data = record["inventory_row"]
    preserved = existing_rows_by_page.get(page_no, {})

    row = {col: "" for col in sorted(checklist_columns | set(standard_columns))}

    # Preserve metadata already calculated in the inventory/checklists.
    for key, value in inventory_data.items():
        if key in row and not pd.isna(value):
            row[key] = value

    for key, value in preserved.items():
        if key in row and str(value).strip() != "":
            row[key] = value

    row["page"] = page_no
    row["render_file"] = str(record["render_path"])

    # Dimensions are derived directly from the page render.
    with Image.open(record["render_path"]) as img:
        row["image_dimensions"] = f"{img.width}x{img.height}"

    for col in manual_columns:
        if col not in preserved or str(row.get(col, "")).strip() == "":
            row[col] = "PENDING" if col != "issues_and_notes" else ""

    checklist_rows.append(row)

checklist_df = pd.DataFrame(checklist_rows).sort_values("page")
checklist_path = OUT / "phase17_10I_R1_page_review_checklist.csv"
checklist_df.to_csv(checklist_path, index=False)

# ------------------------------------------------------------------
# 4. Write manifests and a truthful QA status.
# ------------------------------------------------------------------
sheet_manifest = []
for i, (path, group) in enumerate(zip(actual_sheets, page_groups), start=1):
    sheet_manifest.append({
        "sheet_number": i,
        "file": str(path),
        "sha256": sha256_file(path),
        "first_page": group[0]["page"],
        "last_page": group[-1]["page"],
        "page_count": len(group),
    })

sheet_manifest_path = OUT / "phase17_10I_R1_contact_sheet_manifest.csv"
pd.DataFrame(sheet_manifest).to_csv(sheet_manifest_path, index=False)

page_manifest_path = OUT / "phase17_10I_R1_page_render_manifest.csv"
pd.DataFrame([
    {
        "page": r["page"],
        "render_file": str(r["render_path"]),
        "sha256": sha256_file(r["render_path"]),
    }
    for r in page_records
]).to_csv(page_manifest_path, index=False)

status = {
    "phase": "17.10I-R1",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_render_dir": str(RENDER_DIR),
    "source_page_inventory": str(PAGE_INVENTORY),
    "source_page_count": len(page_records),
    "unique_source_page_files": len(set(page_paths)),
    "expected_contact_sheet_count": SHEETS_PER_WORKSPACE,
    "canonical_contact_sheet_count": len(actual_sheets),
    "canonical_contact_sheet_dir": str(SHEET_DIR),
    "old_workspace_modified": False,
    "old_files_deleted": False,
    "page_checklist_created": True,
    "page_checklist_rows": len(checklist_df),
    "manual_visual_qa_completed": False,
    "manual_decision_columns_preserved_where_found": True,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
}

status_path = OUT / "phase17_10I_R1_visual_qa_status.json"
status_path.write_text(
    json.dumps(status, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

readme_path = OUT / "README.md"
readme_path.write_text(
    f"""# Phase 17.10I-R1 — Repaired Visual QA Workspace

- Source rendered pages: {len(page_records)}
- Canonical contact sheets: {len(actual_sheets)}
- Pages per contact sheet: up to {PAGES_PER_SHEET}
- Checklist: `{checklist_path.name}`
- Contact sheets: `canonical_contact_sheets/`

The prior Phase 17.10I folder was not modified and no old contact sheets
were deleted. This workspace counts only files in `canonical_contact_sheets/`.

Manual visual review is still pending. Review all 19 pages, then update the
page checklist. Passing automated checks does not mean visual review is complete.
""",
    encoding="utf-8"
)

print("\n[1] SOURCE PAGE CHECK")
print("Original render folder:", RENDER_DIR)
print("Unique pages resolved:", len(page_records))
print("Pages numbered 1–19 exactly once:", page_numbers == list(range(1, 20)))

print("\n[2] CONTACT-SHEET REPAIR")
print("Fresh output folder:", OUT)
print("Canonical contact-sheet directory:", SHEET_DIR)
print("Expected sheets:", SHEETS_PER_WORKSPACE)
print("Actual canonical sheets:", len(actual_sheets))
for sheet in sheet_manifest:
    print(
        f"  {Path(sheet['file']).name}: pages "
        f"{sheet['first_page']}–{sheet['last_page']} "
        f"({sheet['page_count']} pages)"
    )

print("\n[3] CHECKLIST")
print("Page checklist:", checklist_path)
print("Checklist rows:", len(checklist_df))
print("Preserved prior review records where present: True")

print("\n[4] OUTPUTS")
print("Contact-sheet manifest:", sheet_manifest_path)
print("Page-render manifest:", page_manifest_path)
print("Status JSON:", status_path)

print("\n[5] RELEASE GATE")
print("Manual visual QA completed: False — inspect the sheets/pages")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Old workspace modified: False")
print("=" * 78)

# Display each of the four canonical sheets inline in Colab.
from IPython.display import display
for sheet_path in actual_sheets:
    print(f"\n{sheet_path.name}")
    display(Image.open(sheet_path))

In [ ]:
# PHASE 17.10I-R2 — DISPLAY CONTACT SHEETS AND FULL-PAGE RENDERS
# Read-only: displays existing images without changing review records.

from pathlib import Path
from IPython.display import display, Image, Markdown
from PIL import Image as PILImage
import re

PROJECT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

QA_DIR = (
    PROJECT / "outputs/research_paper"
    / "phase17_10I_R1_visual_qa_repaired_20261025T194635"
)

SHEET_DIR = QA_DIR / "canonical_contact_sheets"

PAGE_DIR = (
    PROJECT / "outputs/research_paper"
    / "phase17_10D_rendered_review_export_20261025T185523"
    / "rendered_pages"
)

def page_number(path):
    """Extract a page number from common page-render filenames."""
    numbers = re.findall(r"\d+", path.stem)
    return int(numbers[-1]) if numbers else 10**9

sheets = sorted(
    SHEET_DIR.glob("*.png"),
    key=page_number
)

pages = sorted(
    PAGE_DIR.glob("*.png"),
    key=page_number
)

if len(sheets) != 4:
    raise RuntimeError(
        f"Expected 4 contact sheets, found {len(sheets)} in {SHEET_DIR}"
    )

if len(pages) != 19:
    raise RuntimeError(
        f"Expected 19 page renders, found {len(pages)} in {PAGE_DIR}"
    )

# 1. Inspect the overview first.
display(Markdown("# Part A — Contact sheets"))
for sheet in sheets:
    with PILImage.open(sheet) as im:
        size = im.size
    display(Markdown(f"**{sheet.name}** — {size[0]} × {size[1]} pixels"))
    display(Image(filename=str(sheet), width=1500))

# 2. Then inspect the individual pages at their saved resolution.
display(Markdown("# Part B — Full-resolution pages"))
for i, page in enumerate(pages, start=1):
    with PILImage.open(page) as im:
        width, height = im.size

    display(Markdown(
        f"## Page {i} — {page.name}\n"
        f"Resolution: {width} × {height} pixels"
    ))
    display(Image(filename=str(page)))

print("=" * 72)
print("PHASE 17.10I-R2 — REVIEW IMAGES DISPLAYED")
print("=" * 72)
print("Contact sheets displayed:", len(sheets))
print("Full-resolution pages displayed:", len(pages))
print("Manual QA status: STILL PENDING")
print("No checklist status has been automatically changed.")
print("=" * 72)

In [ ]:
# PHASE 17.10F-R1 — VISUAL QA HANDOFF PACKAGE
# Creates a portable, read-only review bundle for visual inspection.
# Does not mark QA complete or modify the original PDF/DOCX/checklist.

from pathlib import Path
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED
from PIL import Image, ImageDraw, ImageOps
import hashlib
import json
import re
import shutil
import csv

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
RESEARCH = ROOT / "outputs" / "research_paper"

# Prefer the exact Phase 17.10D render/export directory already reported.
KNOWN_EXPORT = (
    RESEARCH
    / "phase17_10D_rendered_review_export_20261025T185523"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def page_number(path):
    match = re.search(r"page-(\d+)\.png$", path.name, re.IGNORECASE)
    return int(match.group(1)) if match else None

# 1. Discover the folder containing the 19 original rendered pages.
page_dir = KNOWN_EXPORT / "rendered_pages"

if not page_dir.is_dir() or len(list(page_dir.glob("page-*.png"))) < 19:
    candidates = []
    for p in RESEARCH.rglob("page-001.png"):
        found = [
            x for x in p.parent.glob("page-*.png")
            if page_number(x) is not None
        ]
        if len(found) >= 19:
            candidates.append((len(found), p.parent))

    if not candidates:
        raise FileNotFoundError(
            "Could not locate a folder containing all 19 rendered pages. "
            "Check that Google Drive is mounted and Phase 17.10D outputs exist."
        )

    candidates.sort(key=lambda item: item[0], reverse=True)
    page_dir = candidates[0][1]

pages = sorted(
    [
        p for p in page_dir.glob("page-*.png")
        if page_number(p) is not None
    ],
    key=page_number
)

if len(pages) != 19:
    raise RuntimeError(
        f"Expected 19 page images; found {len(pages)} in {page_dir}. "
        "No QA sign-off was made."
    )

# Check sequential numbering and image readability.
page_numbers = [page_number(p) for p in pages]
if page_numbers != list(range(1, 20)):
    raise RuntimeError(f"Page numbering is incomplete or unexpected: {page_numbers}")

for p in pages:
    with Image.open(p) as im:
        im.verify()

# 2. Verify that the corresponding PDF/DOCX outputs exist if available.
pdf_path = KNOWN_EXPORT / "Delhi_AQI_Integrated_Review_Draft.pdf"
docx_path = KNOWN_EXPORT / "Delhi_AQI_Integrated_Review_Draft.docx"

if not pdf_path.is_file():
    raise FileNotFoundError(f"Expected rendered review PDF not found: {pdf_path}")
if not docx_path.is_file():
    raise FileNotFoundError(f"Expected review DOCX not found: {docx_path}")

# 3. Create a separate QA workspace.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = RESEARCH / f"phase17_10F_R1_visual_qa_handoff_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

HANDOFF = OUT / "Delhi_AQI_Visual_QA_Handoff"
HANDOFF.mkdir()

PAGES_OUT = HANDOFF / "full_resolution_pages"
SHEETS_OUT = HANDOFF / "contact_sheets"
PAGES_OUT.mkdir()
SHEETS_OUT.mkdir()

# Copy all rendered pages into the handoff folder.
copied_pages = []
for p in pages:
    dest = PAGES_OUT / p.name
    shutil.copy2(p, dest)
    copied_pages.append(dest)

# 4. Generate four contact sheets, covering pages 1–5, 6–10, 11–15, and 16–19.
# These are for navigation only; full-size page images remain available.
contact_sheet_records = []

thumb_width = 440
thumb_height = 570
label_height = 34
outer_margin = 28
gap = 24
columns = 2
rows = 3

for sheet_index in range(4):
    start = sheet_index * 5
    subset = copied_pages[start:start + 5]
    if not subset:
        continue

    canvas_width = outer_margin * 2 + columns * thumb_width + gap
    canvas_height = (
        outer_margin * 2
        + rows * (thumb_height + label_height)
        + (rows - 1) * gap
    )

    canvas = Image.new("RGB", (canvas_width, canvas_height), "white")
    draw = ImageDraw.Draw(canvas)

    for local_index, page_path in enumerate(subset):
        with Image.open(page_path) as im:
            image = im.convert("RGB")
            image.thumbnail((thumb_width, thumb_height))

        r = local_index // columns
        c = local_index % columns
        x = outer_margin + c * (thumb_width + gap)
        y = outer_margin + r * (thumb_height + label_height + gap)

        draw.text(
            (x, y),
            f"Page {page_number(page_path)}",
            fill="black"
        )

        canvas.paste(image, (x, y + label_height))
        border = ImageOps.expand(
            image, border=1, fill=(150, 150, 150)
        )
        canvas.paste(border, (x - 1, y + label_height - 1))

    sheet_name = f"contact-sheet-{sheet_index + 1:02d}.png"
    sheet_path = SHEETS_OUT / sheet_name
    canvas.save(sheet_path, optimize=True)

    contact_sheet_records.append({
        "file": str(sheet_path.relative_to(HANDOFF)),
        "pages": [page_number(p) for p in subset],
        "width": canvas_width,
        "height": canvas_height,
        "sha256": sha256_file(sheet_path),
    })

# Copy PDF/DOCX into the handoff directory.
shutil.copy2(pdf_path, HANDOFF / pdf_path.name)
shutil.copy2(docx_path, HANDOFF / docx_path.name)

# Include the existing review checklist without changing its status.
checklists = list(RESEARCH.rglob("phase17_10E_page_review_checklist.csv"))
if checklists:
    checklists.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    checklist_source = checklists[0]
    shutil.copy2(
        checklist_source,
        HANDOFF / "existing_page_review_checklist.csv"
    )
else:
    checklist_source = None

# 5. Write a manifest and an explicit pending-review checklist.
page_manifest = []
for p in copied_pages:
    with Image.open(p) as im:
        width, height = im.size
    page_manifest.append({
        "page": page_number(p),
        "file": str(p.relative_to(HANDOFF)),
        "width": width,
        "height": height,
        "bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    })

with open(HANDOFF / "page_manifest.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(page_manifest[0].keys()))
    writer.writeheader()
    writer.writerows(page_manifest)

visual_checklist = """# Manual Visual QA Checklist — Delhi AQI Review Draft

Review the contact sheets first, then inspect each full-size page wherever
anything looks unclear. Enter actual observations; do not mark items PASS based
only on automated pixel checks.

For every page, check:

- No clipped text, headings, or footers.
- No overlap between paragraphs, figures, captions, or tables.
- Figures are legible and axis labels/legends are readable.
- Tables remain legible and are not split or cropped awkwardly.
- Headings, page breaks, and whitespace are reasonable.
- Figure/table numbering and references are consistent.
- No broken characters, missing glyphs, or unwanted blank pages.

Suggested results for each checklist item: PASS / ISSUE / NOT APPLICABLE.
Record the page number and a concrete note for every ISSUE.

Current status: PENDING
The existence of images and this checklist does not constitute visual sign-off.
"""
(HANDOFF / "MANUAL_VISUAL_QA_CHECKLIST.md").write_text(
    visual_checklist, encoding="utf-8"
)

readme = f"""# Delhi AQI — Visual QA Handoff

Created: {datetime.now().astimezone().isoformat()}

## Contents
- `contact_sheets/`: four overview sheets covering all 19 pages.
- `full_resolution_pages/`: 19 original 1102x1426 page renders.
- `{pdf_path.name}`: complete review PDF.
- `{docx_path.name}`: editable review draft.
- `page_manifest.csv`: dimensions and SHA-256 hashes.
- `existing_page_review_checklist.csv`: prior checklist, unchanged where available.
- `MANUAL_VISUAL_QA_CHECKLIST.md`: manual review instructions.

## Status
Human visual QA is still PENDING. Do not call the draft submission-ready yet.
This handoff package does not modify the source PDF/DOCX, source data, models,
or any prior review ZIP.
"""
(HANDOFF / "README.md").write_text(readme, encoding="utf-8")

# Zip the handoff folder.
zip_path = OUT / "Delhi_AQI_Visual_QA_Handoff.zip"
with ZipFile(zip_path, "w", ZIP_DEFLATED) as z:
    for p in HANDOFF.rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(HANDOFF)))

with ZipFile(zip_path, "r") as z:
    bad = z.testzip()
    if bad:
        raise RuntimeError(f"QA handoff ZIP integrity failure: {bad}")
    zip_entries = z.namelist()

summary = {
    "phase": "17.10F-R1",
    "created_at": datetime.now().astimezone().isoformat(),
    "page_count": len(copied_pages),
    "contact_sheet_count": len(contact_sheet_records),
    "pdf_included": True,
    "docx_included": True,
    "existing_checklist_copied": checklist_source is not None,
    "handoff_zip": str(zip_path),
    "handoff_sha256": sha256_file(zip_path),
    "zip_integrity": "PASS",
    "zip_entries": len(zip_entries),
    "manual_visual_qa_status": "PENDING",
    "submission_ready": False,
    "source_pdf_docx_modified": False,
    "source_data_modified": False,
    "models_retrained": False,
    "production_enabled": False,
    "contact_sheets": contact_sheet_records,
}
(OUT / "phase17_10F_R1_handoff_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 78)
print("PHASE 17.10F-R1 — VISUAL QA HANDOFF")
print("=" * 78)
print("Page images:", len(copied_pages))
print("Contact sheets:", len(contact_sheet_records))
print("PDF included:", pdf_path.name)
print("DOCX included:", docx_path.name)
print("ZIP integrity: PASS")
print("Handoff ZIP:", zip_path)
print("SHA-256:", summary["handoff_sha256"])
print("Manual visual QA: PENDING")
print("Submission ready: False")
print("=" * 78)

# Download the ZIP to your computer so you can attach it in this chat.
from google.colab import files
files.download(str(zip_path))

In [ ]:
# PHASE 17.10F-R2 — FRESH RUNTIME RECONNECT & VISUAL QA RESUME
# Mounts Google Drive, verifies the saved handoff, and displays contact sheets.
# Extraction is local to /content; the Drive source archive remains unchanged.

from google.colab import drive
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from datetime import datetime
from IPython.display import display, Image as DisplayImage, Markdown
import hashlib
import json

print("=" * 78)
print("PHASE 17.10F-R2 — FRESH RUNTIME RECONNECT")
print("=" * 78)

# 1. Mount Drive without forcing a remount.
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Delhi-AQI-Prediction"
)

HANDOFF_ZIP = (
    PROJECT_ROOT
    / "outputs/research_paper"
    / "phase17_10F_R1_visual_qa_handoff_20261025T195124"
    / "Delhi_AQI_Visual_QA_Handoff.zip"
)

EXPECTED_SHA256 = (
    "79744a4f77842244a33bb53dea1f9e8f08f803fdae12b31f2b2a211e7dcae0cd"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# 2. Verify the expected file and identity before extracting anything.
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project folder not found after Drive mount: {PROJECT_ROOT}"
    )

if not HANDOFF_ZIP.is_file():
    raise FileNotFoundError(
        "Visual-QA handoff ZIP not found:\n"
        f"{HANDOFF_ZIP}\n"
        "Check that the expected Drive folder still exists."
    )

actual_sha256 = sha256_file(HANDOFF_ZIP)

print("\n[1] DRIVE AND ARCHIVE")
print("Project folder:", PROJECT_ROOT)
print("Handoff ZIP:", HANDOFF_ZIP)
print("SHA-256:", actual_sha256)
print("SHA-256 matches recorded value:", actual_sha256 == EXPECTED_SHA256)

if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "Handoff ZIP hash mismatch. Extraction stopped."
    )

# 3. Test the archive and inspect its file inventory.
with ZipFile(HANDOFF_ZIP, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(
            f"ZIP integrity check failed at member: {bad_member}"
        )

    members = archive.namelist()

    # Reject unsafe archive paths before extraction.
    for member in members:
        member_path = PurePosixPath(member)
        if member_path.is_absolute() or ".." in member_path.parts:
            raise RuntimeError(
                f"Unsafe path found inside ZIP; extraction stopped: {member}"
            )

    print("ZIP integrity: PASS")
    print("Archive entries:", len(members))

# 4. Extract to a new temporary local folder, not back into Google Drive.
timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
LOCAL_REVIEW_DIR = Path(
    f"/content/phase17_10F_R2_visual_qa_{timestamp}"
)
LOCAL_REVIEW_DIR.mkdir(parents=True, exist_ok=False)

with ZipFile(HANDOFF_ZIP, "r") as archive:
    archive.extractall(LOCAL_REVIEW_DIR)

# 5. Locate contact sheets, full-page images, and the existing checklist.
image_extensions = {".png", ".jpg", ".jpeg", ".webp"}

contact_sheets = sorted([
    p for p in LOCAL_REVIEW_DIR.rglob("*")
    if p.is_file()
    and "contact" in p.name.lower()
    and "sheet" in p.name.lower()
    and p.suffix.lower() in image_extensions
])

page_images = sorted([
    p for p in LOCAL_REVIEW_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in image_extensions
    and p not in contact_sheets
])

pdf_files = sorted(LOCAL_REVIEW_DIR.rglob("*.pdf"))
docx_files = sorted(LOCAL_REVIEW_DIR.rglob("*.docx"))
checklist_files = sorted([
    p for p in LOCAL_REVIEW_DIR.rglob("*.csv")
    if "checklist" in p.name.lower()
    or "page_review" in p.name.lower()
])

print("\n[2] EXTRACTED REVIEW MATERIAL")
print("Temporary local review folder:", LOCAL_REVIEW_DIR)
print("Contact sheets found:", len(contact_sheets))
print("Other page/image files:", len(page_images))
print("PDF files:", len(pdf_files))
print("DOCX files:", len(docx_files))
print("Checklist files:", len(checklist_files))

if len(contact_sheets) != 4:
    print(
        "NOTE: The archive did not contain exactly four files whose names "
        "match 'contact-sheet'. Review the archive listing below."
    )

if len(page_images) < 19:
    print(
        "NOTE: Fewer than 19 other image files were detected; inspect the "
        "inventory to confirm the full-resolution page renders are present."
    )

if len(pdf_files) == 0 or len(docx_files) == 0:
    print(
        "WARNING: The handoff ZIP is expected to include both the review PDF "
        "and DOCX. Check the file inventory."
    )

# 6. Display all discovered contact sheets in the notebook output.
print("\n[3] CONTACT SHEETS — REVIEW ALL PAGES")
if contact_sheets:
    for sheet in contact_sheets:
        display(Markdown(f"### {sheet.name}"))
        display(DisplayImage(
            filename=str(sheet),
            width=1200
        ))
else:
    print("No contact sheets found. Image inventory:")
    for path in sorted(LOCAL_REVIEW_DIR.rglob("*")):
        if path.is_file() and path.suffix.lower() in image_extensions:
            print(path.relative_to(LOCAL_REVIEW_DIR))

# 7. Show paths needed for full-resolution inspection and QA logging.
print("\n[4] FULL-RESOLUTION PAGES")
for path in page_images[:25]:
    print(path.relative_to(LOCAL_REVIEW_DIR))

print("\n[5] CHECKLIST")
for path in checklist_files:
    print(path)

print("\n[6] FINAL STATUS")
print("Drive mounted: True")
print("Handoff hash verified: True")
print("ZIP integrity: PASS")
print("Manual visual QA: PENDING")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.10F-R3 — CONTACT SHEET REVIEW HANDOFF
# Preserves the recovered review renders and creates a single preview image.
# Does not modify the handoff ZIP, manuscript, source data, or models.

from pathlib import Path
from datetime import datetime
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
import json
import shutil

import google.colab.files

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Delhi-AQI-Prediction"
)

REVIEW_DIR = Path(
    "/content/phase17_10F_R2_visual_qa_20261026T081053"
)

if not REVIEW_DIR.is_dir():
    raise FileNotFoundError(
        f"Recovered review folder not found: {REVIEW_DIR}"
    )

contact_sheets = sorted(
    REVIEW_DIR.glob("contact-sheet-*.png")
)

if not contact_sheets:
    contact_sheets = sorted(REVIEW_DIR.rglob("contact-sheet-*.png"))

if not contact_sheets:
    raise FileNotFoundError(
        "No contact-sheet PNG files found in the recovered review folder."
    )

# New Drive folder; existing audit files remain untouched.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    PROJECT_ROOT / "outputs" / "research_paper"
    / f"phase17_10F_R3_visual_review_materials_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

# Copy each contact sheet into the durable review folder.
copied_sheets = []

for source in contact_sheets:
    destination = OUT / source.name
    shutil.copy2(source, destination)
    copied_sheets.append(destination)

# Create a 2-column mosaic, preserving each sheet's aspect ratio.
thumbs = []
max_cell_width = 1600
max_cell_height = 1200
padding = 36
label_height = 44

for path in copied_sheets:
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((max_cell_width, max_cell_height))
        thumbs.append((path.name, im.copy()))

columns = 2 if len(thumbs) > 1 else 1
rows = (len(thumbs) + columns - 1) // columns
cell_width = max(im.width for _, im in thumbs)
cell_height = max(im.height for _, im in thumbs)

canvas = Image.new(
    "RGB",
    (
        padding + columns * (cell_width + padding),
        padding + rows * (cell_height + label_height + padding)
    ),
    "white"
)

draw = ImageDraw.Draw(canvas)

for i, (name, im) in enumerate(thumbs):
    row, col = divmod(i, columns)
    x = padding + col * (cell_width + padding)
    y = padding + row * (cell_height + label_height + padding)

    draw.text((x, y), name, fill="black")
    canvas.paste(im, (x, y + label_height))

mosaic_path = OUT / "phase17_10F_contact_sheets_mosaic.png"
canvas.save(mosaic_path, optimize=True)

status = {
    "phase": "17.10F-R3",
    "created_at": datetime.now().astimezone().isoformat(),
    "source_review_folder": str(REVIEW_DIR),
    "contact_sheets_found": len(contact_sheets),
    "copied_contact_sheets": [str(p) for p in copied_sheets],
    "mosaic_path": str(mosaic_path),
    "manual_visual_qa": "PENDING",
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "frozen_models_retrained": False,
    "original_handoff_zip_modified": False,
}

status_path = OUT / "phase17_10F_R3_visual_review_status.json"
status_path.write_text(
    json.dumps(status, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 78)
print("PHASE 17.10F-R3 — CONTACT SHEET REVIEW HANDOFF")
print("=" * 78)
print("Contact sheets found:", len(contact_sheets))
print("Saved review folder:", OUT)
print("Combined preview:", mosaic_path)
print("Status JSON:", status_path)

print("\n[CONTACT SHEET PREVIEWS]")
for path in copied_sheets:
    print("\n", path.name)
    display(Image.open(path))

print("\n[COMBINED PREVIEW]")
display(Image.open(mosaic_path))

print("\nManual visual QA: PENDING")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("=" * 78)

# Download the combined image so it can be attached in this chat.
google.colab.files.download(str(mosaic_path))

In [ ]:
# PHASE 17.10G — VISUAL ASSETS ONLY
# Creates an image/graph-only folder and ZIP. No reports or manuscripts.

from pathlib import Path
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED
import base64
import hashlib
import json
import shutil

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

PUBLICATION_PACK = (
    ROOT / "outputs/research_paper"
    / "phase17_10A_publication_figures_20261025T184251"
)

XAI_DIR = ROOT / "outputs/reports/phase15_11_fast_xai/plots"
EDA_DIR = ROOT / "reports/figures"

NOTEBOOKS = [
    "04_eda.ipynb",
    "07_model_evaluation_ablation.ipynb",
    "08_deep_learning_models.ipynb",
    "09_vmd_pso_vmd.ipynb",
    "10_timegan_advanced_hybrid.ipynb",
]

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".svg", ".pdf"}
PLOT_EXTENSIONS = {".png", ".jpg", ".jpeg", ".svg"}

# ---------- Preflight ----------
if not PUBLICATION_PACK.is_dir():
    raise FileNotFoundError(
        f"Publication figure pack not found: {PUBLICATION_PACK}"
    )

figure_register = PUBLICATION_PACK / "figure_register.csv"
if not figure_register.exists():
    raise FileNotFoundError(
        f"Publication figure register not found: {figure_register}"
    )

# Create a unique output directory to preserve all previous files.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    ROOT / "outputs/research_paper"
    / f"phase17_10G_visual_assets_only_{stamp}"
)
ASSETS = OUT / "visual_assets"
ASSETS.mkdir(parents=True, exist_ok=False)

counts = {}
missing_sources = []

def copy_visuals(files, destination):
    destination.mkdir(parents=True, exist_ok=True)
    copied = 0

    for source in files:
        source = Path(source)
        if not source.is_file():
            missing_sources.append(str(source))
            continue
        if source.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        shutil.copy2(source, destination / source.name)
        copied += 1

    return copied

# ---------- 1. Publication candidate figures ----------
# All seven figures, both PNG and PDF.
pub_dest = ASSETS / "01_publication_candidate_figures"
pub_dest.mkdir()

publication_files = list(PUBLICATION_PACK.glob("figures/*.png"))
publication_files += list(PUBLICATION_PACK.glob("figures/*.pdf"))

if len(list(PUBLICATION_PACK.glob("figures/*.png"))) != 7:
    raise RuntimeError("Expected 7 publication PNG figures.")
if len(list(PUBLICATION_PACK.glob("figures/*.pdf"))) != 7:
    raise RuntimeError("Expected 7 publication PDF figures.")

counts["Publication candidates (PNG + PDF)"] = copy_visuals(
    publication_files, pub_dest
)

# ---------- 2. Previously saved explainability charts ----------
xai_dest = ASSETS / "02_explainability"
counts["Explainability"] = copy_visuals(
    sorted(XAI_DIR.glob("*"))
    if XAI_DIR.is_dir() else [],
    xai_dest
)

# ---------- 3. EDA, descriptive statistics and historical figures ----------
eda_dest = ASSETS / "03_eda_and_descriptive_analysis"
counts["EDA and descriptive analysis"] = copy_visuals(
    sorted(EDA_DIR.glob("*"))
    if EDA_DIR.is_dir() else [],
    eda_dest
)

# ---------- 4. Extract visual outputs embedded in analysis notebooks ----------
# Only image data is extracted. Notebook code and text are not copied.
notebook_root = ROOT / "notebooks"
notebook_image_count = 0
notebook_output_dest = ASSETS / "04_notebook_embedded_visuals"

for notebook_name in NOTEBOOKS:
    notebook_path = notebook_root / notebook_name

    if not notebook_path.is_file():
        missing_sources.append(str(notebook_path))
        continue

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    notebook_stem = notebook_path.stem
    notebook_dest = notebook_output_dest / notebook_stem
    notebook_dest.mkdir(parents=True, exist_ok=True)

    for cell_idx, cell in enumerate(notebook.get("cells", []), start=1):
        for output_idx, output in enumerate(
            cell.get("outputs", []), start=1
        ):
            data = output.get("data", {})

            for mime, extension in [
                ("image/png", ".png"),
                ("image/jpeg", ".jpg"),
                ("image/svg+xml", ".svg"),
            ]:
                if mime not in data:
                    continue

                payload = data[mime]
                if isinstance(payload, list):
                    payload = "".join(payload)

                filename = (
                    f"{notebook_stem}_cell{cell_idx:03d}"
                    f"_output{output_idx:02d}{extension}"
                )
                target = notebook_dest / filename

                if mime == "image/svg+xml":
                    target.write_text(str(payload), encoding="utf-8")
                else:
                    try:
                        target.write_bytes(
                            base64.b64decode(payload, validate=False)
                        )
                    except Exception as exc:
                        print(
                            "Skipped an unreadable notebook image:",
                            filename, type(exc).__name__
                        )
                        continue

                notebook_image_count += 1

counts["Notebook-embedded images"] = notebook_image_count

# ---------- 5. Zip only the visual assets ----------
zip_path = OUT / "Delhi_AQI_Visual_Artifacts_Only.zip"

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for file in sorted(ASSETS.rglob("*")):
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS:
            archive.write(file, arcname=str(file.relative_to(ASSETS)))

# ---------- 6. Verify ZIP integrity ----------
with ZipFile(zip_path, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity failed: {bad_member}")

    members = archive.namelist()
    non_visual = [
        name for name in members
        if Path(name).suffix.lower() not in IMAGE_EXTENSIONS
    ]

    if non_visual:
        raise RuntimeError(
            "Non-visual files unexpectedly found in ZIP: "
            + ", ".join(non_visual[:10])
        )

digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()

print("=" * 72)
print("PHASE 17.10G — VISUAL ASSETS ONLY")
print("=" * 72)
for category, count in counts.items():
    print(f"{category}: {count}")

print("\nTotal visual files in ZIP:", len(members))
print("ZIP integrity: PASS")
print("SHA-256:", digest)
print("Asset folder:", ASSETS)
print("ZIP:", zip_path)

if missing_sources:
    print("\nSources not found (check only if expected):")
    for source in missing_sources:
        print("-", source)

print("\nIncluded: graphs, PNG images, SVGs where available, and figure PDFs.")
print("Excluded: reports, manuscripts, checklists, notebook code and datasets.")
print("Source data modified: False")
print("Models retrained: False")
print("Production enabled: False")
print("=" * 72)

In [ ]:
# PHASE 17.10F-R1 — PREPARE VISUAL QA FILES FOR REVIEW
# Collects existing renders and checklists into a new review-only ZIP.

from pathlib import Path
from datetime import datetime
import hashlib
import json
import shutil
import zipfile
import csv

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
BASE = ROOT / "outputs" / "research_paper"

def newest(paths):
    paths = [p for p in paths if p.exists()]
    return max(paths, key=lambda p: p.stat().st_mtime) if paths else None

# Find the existing rendered review export.
render_dir = newest(list(BASE.glob(
    "phase17_10D_rendered_review_export_*/rendered_pages"
)))
if render_dir is None:
    raise FileNotFoundError(
        "Cannot find phase17_10D*/rendered_pages. "
        "Check that the Phase 17.10D page-render output still exists."
    )

export_dir = render_dir.parent
contact_dir = export_dir / "contact_sheets"
pdf_candidates = list(export_dir.glob("*.pdf"))
pdf_path = pdf_candidates[0] if pdf_candidates else None

page_images = sorted(
    [p for p in render_dir.glob("*.png") if p.is_file()],
    key=lambda p: [
        int(token) if token.isdigit() else token.lower()
        for token in __import__("re").split(r"(\d+)", p.name)
    ]
)

contact_sheets = sorted(contact_dir.glob("*.png")) if contact_dir.exists() else []

checklist_path = newest(list(BASE.glob(
    "phase17_10E_human_visual_qa_*/phase17_10E_page_review_checklist.csv"
)))
status_path = newest(list(BASE.glob(
    "phase17_10E_human_visual_qa_*/phase17_10E_visual_qa_status.json"
)))

# Validate the expected source files before creating the package.
if len(page_images) != 19:
    raise RuntimeError(
        f"Expected 19 rendered page PNGs, found {len(page_images)} in {render_dir}"
    )
if len(contact_sheets) < 2:
    raise RuntimeError(
        f"Expected at least 2 contact sheets, found {len(contact_sheets)}"
    )
if pdf_path is None:
    raise FileNotFoundError(f"PDF not found in {export_dir}")

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
out_dir = BASE / f"phase17_10F_R1_visual_review_upload_{stamp}"
stage = out_dir / "review_files"
(stage / "contact_sheets").mkdir(parents=True)
(stage / "full_resolution_pages").mkdir()
(stage / "checklist").mkdir()

# Copy the contact sheets.
for sheet in contact_sheets:
    shutil.copy2(sheet, stage / "contact_sheets" / sheet.name)

# Copy all page renders at full resolution.
for i, page in enumerate(page_images, start=1):
    destination = stage / "full_resolution_pages" / f"page_{i:02d}.png"
    shutil.copy2(page, destination)

# Include the rendered PDF and QA checklist/status.
shutil.copy2(pdf_path, stage / pdf_path.name)
if checklist_path is not None:
    shutil.copy2(checklist_path, stage / "checklist" / checklist_path.name)
if status_path is not None:
    shutil.copy2(status_path, stage / "checklist" / status_path.name)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = []
for path in sorted(stage.rglob("*")):
    if path.is_file():
        manifest.append({
            "relative_path": str(path.relative_to(stage)),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

(stage / "UPLOAD_README.txt").write_text(
    "Phase 17.10F-R1 visual review upload\n\n"
    "Contents:\n"
    "- contact_sheets/: overview sheets for initial inspection\n"
    "- full_resolution_pages/: 19 rendered page images\n"
    "- rendered PDF for searchable text/context\n"
    "- checklist/: current visual QA checklist and status, when found\n\n"
    "The images have not been manually certified. Use the contact sheets first, "
    "then inspect individual pages at full resolution.\n",
    encoding="utf-8"
)

(out_dir / "file_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

review_zip = out_dir / "Delhi_AQI_Visual_QA_Review_Pack.zip"
with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(stage.rglob("*")):
        if path.is_file():
            z.write(path, arcname=str(path.relative_to(stage)))

with zipfile.ZipFile(review_zip, "r") as z:
    bad_member = z.testzip()
    if bad_member:
        raise RuntimeError(f"Review ZIP integrity failure: {bad_member}")
    entry_count = len(z.namelist())

# Copy into Colab's downloadable working directory.
download_copy = Path("/content/Delhi_AQI_Visual_QA_Review_Pack.zip")
shutil.copy2(review_zip, download_copy)

print("=" * 72)
print("PHASE 17.10F-R1 — VISUAL QA UPLOAD PACK")
print("=" * 72)
print("Rendered pages:", len(page_images))
print("Contact sheets:", len(contact_sheets))
print("PDF:", pdf_path.name)
print("Checklist found:", checklist_path is not None)
print("ZIP integrity: PASS")
print("ZIP entries:", entry_count)
print("ZIP SHA-256:", sha256_file(review_zip))
print("Drive ZIP:", review_zip)
print("Download copy:", download_copy)
print("Manual visual QA status: STILL PENDING")
print("=" * 72)

from google.colab import files
files.download(str(download_copy))

In [ ]:
# PHASE 17.10F-R1 — FAST VISUAL REVIEW
# Displays the contact sheets and low-text pages for direct inspection.

from pathlib import Path
import pandas as pd
from IPython.display import display, Image as IPyImage

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
QA_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10E_human_visual_qa_20261025T185812"
)

checklist_path = QA_DIR / "phase17_10E_page_review_checklist.csv"
if not checklist_path.exists():
    matches = list(
        (ROOT / "outputs/research_paper").glob(
            "phase17_10E_human_visual_qa_*/phase17_10E_page_review_checklist.csv"
        )
    )
    if not matches:
        raise FileNotFoundError("Could not find the Phase 17.10E checklist.")
    checklist_path = max(matches, key=lambda p: p.stat().st_mtime)
    QA_DIR = checklist_path.parent

checklist = pd.read_csv(checklist_path)

print("=" * 72)
print("PHASE 17.10F-R1 — CONTACT SHEETS")
print("=" * 72)

contact_sheets = sorted(QA_DIR.rglob("contact-sheet-*.png"))
if not contact_sheets:
    raise FileNotFoundError(
        f"No contact sheets found under {QA_DIR}. "
        "Check the Phase 17.10E contact_sheets folder."
    )

for path in contact_sheets:
    print(f"\n{path.name}")
    display(IPyImage(filename=str(path), width=1200))

print("\n" + "=" * 72)
print("FULL-SIZE REVIEW — PAGES WITH FEWER THAN 750 EXTRACTED TEXT CHARACTERS")
print("=" * 72)

if "pdf_text_characters" in checklist.columns:
    low_text = checklist[
        pd.to_numeric(checklist["pdf_text_characters"], errors="coerce") < 750
    ]
else:
    low_text = pd.DataFrame()

if low_text.empty:
    print("No pages meet this screening criterion.")
else:
    for _, row in low_text.iterrows():
        page_num = int(row["page"])
        render = Path(str(row["render_file"]))
        print(
            f"\nPage {page_num} | extracted text characters: "
            f"{row.get('pdf_text_characters', 'unknown')}"
        )
        if render.is_file():
            display(IPyImage(filename=str(render)))
        else:
            print("Full-size render not found:", render)

print("\n" + "=" * 72)
print("MANUAL CHECKLIST")
print("=" * 72)

columns = [
    c for c in [
        "page", "pdf_text_characters", "automated_flags",
        "manual_review_status", "no_clipping_or_overlap",
        "figures_readable", "tables_readable",
        "headings_and_page_breaks_ok", "captions_and_references_ok",
        "issues_and_notes"
    ] if c in checklist.columns
]
display(checklist[columns])

print("\nNo checklist status has been changed by this cell.")
print("Review all pages; inspect individual pages if any element is unclear.")
print("After review, record any clipping, unreadable figures/tables, or layout issues.")
print("Submission ready: False until visual QA and the licence review are closed.")
print("=" * 72)

In [ ]:
# PHASE 17.10F-R2 — ROBUST CONTACT SHEET RECOVERY
# Search existing locations first; regenerate from the 19 PDF page renders if needed.
# Does not modify the DOCX, PDF, source data, or frozen models.

from pathlib import Path
from datetime import datetime
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
import pandas as pd
import shutil
import re
import json

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
RESEARCH_ROOT = ROOT / "outputs" / "research_paper"

EXPECTED_PAGES = 19
SHEETS_PER_IMAGE = 10
THUMB_WIDTH = 250
THUMB_HEIGHT = 324
COLUMNS = 5

print("=" * 78)
print("PHASE 17.10F-R2 — CONTACT SHEET RECOVERY")
print("=" * 78)

# 1. Find the latest Phase 17.10E QA folder.
qa_dirs = sorted(
    RESEARCH_ROOT.glob("phase17_10E_human_visual_qa_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

QA_DIR = qa_dirs[0] if qa_dirs else None
print("QA folder:", QA_DIR if QA_DIR else "Not found by expected name")

# 2. Search broadly for existing contact sheets under Phase 17.10E and 17.10F.
search_dirs = qa_dirs + sorted(
    RESEARCH_ROOT.glob("phase17_10F*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

existing_sheets = []
for folder in search_dirs:
    if not folder.is_dir():
        continue
    for path in folder.rglob("*"):
        if (
            path.is_file()
            and path.suffix.lower() in {".png", ".jpg", ".jpeg"}
            and "contact" in path.name.lower()
            and "sheet" in path.name.lower()
        ):
            if path not in existing_sheets:
                existing_sheets.append(path)

print("Existing contact sheets found:", len(existing_sheets))
for sheet in existing_sheets:
    print(" -", sheet)

# 3. Find saved rendered page images, starting with the latest Phase 17.10D folder.
render_dirs = sorted(
    RESEARCH_ROOT.glob("phase17_10D_rendered_review_export_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

page_images = []
render_source = None

# Prefer the page inventory when available.
for render_dir in render_dirs:
    inventories = list(render_dir.glob("rendered_page_inventory.csv"))
    for inventory in inventories:
        try:
            inv = pd.read_csv(inventory)
            col = next(
                (c for c in inv.columns
                 if str(c).strip().lower() in {"render_file", "page_file", "path"}),
                None
            )
            if col:
                candidates = []
                for value in inv[col].dropna():
                    p = Path(str(value))
                    if not p.is_absolute():
                        p = render_dir / p
                    if p.is_file() and p.suffix.lower() == ".png":
                        candidates.append(p)
                if candidates:
                    page_images = candidates
                    render_source = render_dir
                    break
        except Exception as exc:
            print("Inventory read warning:", inventory, repr(exc))
    if page_images:
        break

# Fall back to rendered_pages subdirectories.
if not page_images:
    for render_dir in render_dirs:
        rendered_pages_dir = render_dir / "rendered_pages"
        if rendered_pages_dir.is_dir():
            candidates = [
                p for p in rendered_pages_dir.glob("*.png")
                if p.is_file()
            ]
            if candidates:
                page_images = candidates
                render_source = render_dir
                break

# Last fallback: identify PNGs inside the known render-export folders.
if not page_images:
    for render_dir in render_dirs:
        candidates = [
            p for p in render_dir.rglob("*.png")
            if p.is_file()
            and "contact" not in p.name.lower()
            and "sheet" not in p.name.lower()
        ]
        if len(candidates) >= EXPECTED_PAGES:
            page_images = candidates
            render_source = render_dir
            break

def natural_key(path):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", path.name)
    ]

page_images = sorted(set(page_images), key=natural_key)

print("\nRendered page images found:", len(page_images))
print("Render source:", render_source)

if len(page_images) < EXPECTED_PAGES:
    raise FileNotFoundError(
        f"Only {len(page_images)} rendered page images could be found; "
        f"{EXPECTED_PAGES} are expected. Check the Phase 17.10D rendered_pages "
        "folder and rendered_page_inventory.csv."
    )

if len(page_images) > EXPECTED_PAGES:
    print(
        f"Note: found {len(page_images)} PNGs. "
        f"Using the first {EXPECTED_PAGES} after natural sorting."
    )
    page_images = page_images[:EXPECTED_PAGES]

# 4. Create a new review workspace.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    RESEARCH_ROOT
    / f"phase17_10F_R2_contact_sheet_recovery_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

SHEET_DIR = OUT / "contact_sheets"
SHEET_DIR.mkdir()

# Copy existing sheets if present, but use regeneration if there are fewer
# than two usable sheets.
usable_existing = []
for sheet in existing_sheets:
    try:
        with Image.open(sheet) as im:
            im.verify()
        usable_existing.append(sheet)
    except Exception as exc:
        print("Invalid existing sheet; will not use:", sheet, repr(exc))

mode = "regenerated_from_page_renders"

if len(usable_existing) >= 2:
    for i, sheet in enumerate(usable_existing[:2], start=1):
        suffix = sheet.suffix.lower()
        dest = SHEET_DIR / f"contact-sheet-{i:02d}{suffix}"
        shutil.copy2(sheet, dest)
    output_sheets = sorted(SHEET_DIR.glob("contact-sheet-*"))
    mode = "copied_existing_sheets_after_discovery"

else:
    # Regenerate from full-resolution page images.
    font = ImageFont.load_default()
    output_sheets = []

    for start in range(0, len(page_images), SHEETS_PER_IMAGE):
        batch = page_images[start:start + SHEETS_PER_IMAGE]
        rows = (len(batch) + COLUMNS - 1) // COLUMNS
        cell_width = THUMB_WIDTH + 20
        cell_height = THUMB_HEIGHT + 48

        canvas = Image.new(
            "RGB",
            (COLUMNS * cell_width, rows * cell_height),
            "white"
        )
        draw = ImageDraw.Draw(canvas)

        for offset, page_path in enumerate(batch):
            page_num = start + offset + 1
            r = offset // COLUMNS
            c = offset % COLUMNS

            x = c * cell_width + 10
            y = r * cell_height + 30

            with Image.open(page_path) as source:
                page = source.convert("RGB")
                page.thumbnail((THUMB_WIDTH, THUMB_HEIGHT))

            paste_x = x + (THUMB_WIDTH - page.width) // 2
            canvas.paste(page, (paste_x, y))

            draw.text(
                (x, r * cell_height + 8),
                f"Page {page_num}",
                fill="black",
                font=font
            )

        sheet_path = SHEET_DIR / f"contact-sheet-{len(output_sheets) + 1:02d}.png"
        canvas.save(sheet_path, format="PNG")
        output_sheets.append(sheet_path)

# 5. Create a page inventory and manual-review checklist if available.
page_inventory = []
for page_num, path in enumerate(page_images, start=1):
    page_inventory.append({
        "page": page_num,
        "render_path": str(path),
        "manual_review_status": "PENDING",
        "no_clipping_or_overlap": "PENDING",
        "figures_readable": "PENDING",
        "tables_readable": "PENDING",
        "headings_and_page_breaks_ok": "PENDING",
        "captions_and_references_ok": "PENDING",
        "issues_and_notes": "",
    })

inventory_path = OUT / "page_review_checklist.csv"
pd.DataFrame(page_inventory).to_csv(inventory_path, index=False)

qa_status = {
    "phase": "17.10F-R2",
    "created_at": datetime.now().astimezone().isoformat(),
    "contact_sheet_mode": mode,
    "render_source": str(render_source),
    "expected_page_count": EXPECTED_PAGES,
    "page_images_found": len(page_images),
    "contact_sheets_created_or_copied": len(output_sheets),
    "manual_visual_qa_completed": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "outputs": {
        "contact_sheet_directory": str(SHEET_DIR),
        "page_review_checklist": str(inventory_path),
    },
}

status_path = OUT / "phase17_10F_R2_status.json"
status_path.write_text(
    json.dumps(qa_status, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

instructions = f"""# Phase 17.10F-R2 — Visual Review Instructions

- Page renders found: {len(page_images)}
- Contact sheet mode: {mode}
- Contact sheets: {SHEET_DIR}
- Page checklist: `page_review_checklist.csv`

Inspect both contact sheets, then open every questionable page PNG at full size.
Mark a page as reviewed only after checking clipping/overlap, figure and table
readability, headings/page breaks, and captions/references.

This cell has not completed human visual QA. Submission readiness remains false.
Licence review remains open.
"""
(OUT / "README.md").write_text(instructions, encoding="utf-8")

print("\n[1] RECOVERY RESULT")
print("Mode:", mode)
print("Pages:", len(page_images))
print("Contact sheets:", len(output_sheets))

print("\n[2] CONTACT SHEETS")
for path in output_sheets:
    print(path)

print("\n[3] CHECKLIST AND STATUS")
print("Page checklist:", inventory_path)
print("Status JSON:", status_path)
print("Output folder:", OUT)

print("\n[4] DISPLAY CONTACT SHEETS")
for path in output_sheets:
    print(path.name)
    with Image.open(path) as im:
        display(im.copy())

print("\n[5] RELEASE GATE")
print("Human visual QA completed: False — inspect the displayed sheets")
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("=" * 78)

In [ ]:
# PHASE 17.10F-R3 — VISUAL QA REVIEW PACK FOR CHATGPT
# Collects existing rendered assets for visual inspection; does not rerender
# the paper or modify the source manuscript, bundle, data, or models.

from pathlib import Path
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib
import json
import re
import shutil

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
RESEARCH = ROOT / "outputs" / "research_paper"

SHEETS_DIR = (
    RESEARCH
    / "phase17_10F_R2_contact_sheet_recovery_20261026T083418"
    / "contact_sheets"
)

RENDER_DIR = (
    RESEARCH
    / "phase17_10D_rendered_review_export_20261025T185523"
)

PAGES_DIR = RENDER_DIR / "rendered_pages"
PDF_PATH = RENDER_DIR / "Delhi_AQI_Integrated_Review_Draft.pdf"
DOCX_PATH = RENDER_DIR / "Delhi_AQI_Integrated_Review_Draft.docx"

CHECKLISTS = [
    RESEARCH
    / "phase17_10F_R2_contact_sheet_recovery_20261026T083418"
    / "page_review_checklist.csv",
    RESEARCH
    / "phase17_10E_human_visual_qa_20261025T185812"
    / "phase17_10E_page_review_checklist.csv",
]

QA_JSON = RENDER_DIR / "phase17_10D_render_qa.json"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def natural_key(path):
    return [
        int(s) if s.isdigit() else s.lower()
        for s in re.split(r"(\d+)", path.name)
    ]

# 1. Discover and verify source files before creating the package.
if not SHEETS_DIR.is_dir():
    raise FileNotFoundError(f"Contact-sheet folder not found: {SHEETS_DIR}")

if not PAGES_DIR.is_dir():
    raise FileNotFoundError(f"Rendered-page folder not found: {PAGES_DIR}")

contact_sheets = sorted(
    SHEETS_DIR.glob("contact-sheet-*.png"),
    key=natural_key
)
page_images = sorted(
    PAGES_DIR.glob("*.png"),
    key=natural_key
)

if len(contact_sheets) != 2:
    raise RuntimeError(
        f"Expected 2 contact sheets; found {len(contact_sheets)}."
    )

if len(page_images) != 19:
    raise RuntimeError(
        f"Expected 19 page renders; found {len(page_images)}. "
        "Check the render directory before proceeding."
    )

if not PDF_PATH.is_file():
    raise FileNotFoundError(f"Source PDF not found: {PDF_PATH}")

print("=" * 78)
print("PHASE 17.10F-R3 — VISUAL QA REVIEW PACK")
print("=" * 78)
print("Contact sheets:", len(contact_sheets))
print("Full-resolution pages:", len(page_images))
print("PDF available:", PDF_PATH.is_file())
print("DOCX available:", DOCX_PATH.is_file())

# 2. Create a new output directory without replacing existing work.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    RESEARCH
    / f"phase17_10F_R3_chatgpt_visual_qa_pack_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

PACKAGE = OUT / "Delhi_AQI_Visual_QA_Review_Pack.zip"

manifest = {
    "created_at": datetime.now().astimezone().isoformat(),
    "purpose": "Manual visual QA of the 19-page integrated research draft",
    "contact_sheets": [],
    "rendered_pages": [],
    "source_pdf": None,
    "source_docx": None,
    "checklists": [],
    "qa_report": None,
    "manual_visual_qa_completed": False,
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "models_retrained": False,
}

# 3. Copy images and supporting review records into a clean package layout.
staging = OUT / "review_files"
cs_out = staging / "contact_sheets"
pages_out = staging / "rendered_pages"
docs_out = staging / "documents"
records_out = staging / "qa_records"

for folder in [cs_out, pages_out, docs_out, records_out]:
    folder.mkdir(parents=True, exist_ok=True)

for path in contact_sheets:
    dest = cs_out / path.name
    shutil.copy2(path, dest)
    manifest["contact_sheets"].append({
        "file": f"contact_sheets/{dest.name}",
        "sha256": sha256_file(dest),
        "bytes": dest.stat().st_size,
    })

for i, path in enumerate(page_images, start=1):
    # Normalized names make page order obvious even if source names vary.
    dest = pages_out / f"page-{i:02d}.png"
    shutil.copy2(path, dest)
    manifest["rendered_pages"].append({
        "page": i,
        "file": f"rendered_pages/{dest.name}",
        "original_name": path.name,
        "sha256": sha256_file(dest),
        "bytes": dest.stat().st_size,
    })

pdf_dest = docs_out / PDF_PATH.name
shutil.copy2(PDF_PATH, pdf_dest)
manifest["source_pdf"] = {
    "file": f"documents/{pdf_dest.name}",
    "sha256": sha256_file(pdf_dest),
}

if DOCX_PATH.is_file():
    docx_dest = docs_out / DOCX_PATH.name
    shutil.copy2(DOCX_PATH, docx_dest)
    manifest["source_docx"] = {
        "file": f"documents/{docx_dest.name}",
        "sha256": sha256_file(docx_dest),
    }

for path in CHECKLISTS:
    if path.is_file():
        dest = records_out / path.name
        shutil.copy2(path, dest)
        manifest["checklists"].append({
            "file": f"qa_records/{dest.name}",
            "sha256": sha256_file(dest),
        })

if QA_JSON.is_file():
    dest = records_out / QA_JSON.name
    shutil.copy2(QA_JSON, dest)
    manifest["qa_report"] = {
        "file": f"qa_records/{dest.name}",
        "sha256": sha256_file(dest),
    }

(staging / "review_pack_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

(staging / "README.txt").write_text(
    """DELHI AQI — VISUAL QA REVIEW PACK

Please review all 19 rendered pages.

Suggested workflow:
1. Inspect contact_sheets/contact-sheet-01.png and contact-sheet-02.png.
2. Open full-resolution images in rendered_pages/ for anything small, unclear,
   clipped, overlapping, or inconsistently spaced.
3. Pay special attention to chart labels, table text, captions, references,
   headings, page breaks, and orphaned lines.
4. Record actual issues in the supplied QA checklist. Keep unknown/unreviewed
   items as PENDING; do not mark them PASS from automated checks alone.

This pack contains copies of already-rendered files. The original project
artifacts have not been edited. Passing automated layout checks is not the
same as completing visual review.
""",
    encoding="utf-8"
)

# 4. Archive the review pack.
with ZipFile(PACKAGE, "w", compression=ZIP_DEFLATED) as z:
    for path in staging.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(staging))

# Verify archive structure and image count.
with ZipFile(PACKAGE, "r") as z:
    bad_member = z.testzip()
    if bad_member:
        raise RuntimeError(f"Review ZIP integrity failed: {bad_member}")

    names = z.namelist()
    zipped_sheets = [
        n for n in names
        if n.startswith("contact_sheets/") and n.endswith(".png")
    ]
    zipped_pages = [
        n for n in names
        if n.startswith("rendered_pages/") and n.endswith(".png")
    ]

if len(zipped_sheets) != 2 or len(zipped_pages) != 19:
    raise RuntimeError(
        f"Unexpected ZIP contents: {len(zipped_sheets)} sheets, "
        f"{len(zipped_pages)} pages."
    )

summary = {
    "package": str(PACKAGE),
    "sha256": sha256_file(PACKAGE),
    "zip_integrity": "PASS",
    "contact_sheets": len(zipped_sheets),
    "full_page_renders": len(zipped_pages),
    "human_visual_qa_completed": False,
    "author_licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
}

(OUT / "phase17_10F_R3_package_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n[PACKAGE VERIFIED]")
print("ZIP:", PACKAGE)
print("SHA-256:", summary["sha256"])
print("ZIP integrity: PASS")
print("Contact sheets:", len(zipped_sheets))
print("Full-resolution pages:", len(zipped_pages))
print("Bytes:", PACKAGE.stat().st_size)
print("\nManual visual QA completed: False")
print("Licence review: OPEN")
print("Submission ready: False")
print("=" * 78)

# 5. Trigger a browser download in Google Colab.
try:
    from google.colab import files
    files.download(str(PACKAGE))
except Exception as exc:
    print(
        "\nAutomatic download did not start. "
        "Use the ZIP path printed above in Colab's Files panel."
    )
    print("Download detail:", repr(exc))

In [ ]:
# PHASE 17.10F-R1 — PAGE-BY-PAGE VISUAL REVIEW
# Creates fresh contact sheets, displays all 19 pages, and records your sign-off.
# Does not modify the manuscript, original PDFs, data, models, or review ZIP.

from pathlib import Path
from datetime import datetime
from PIL import Image as PILImage, ImageDraw
from IPython.display import display, Image as NotebookImage, Markdown
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
RESEARCH = ROOT / "outputs/research_paper"

EXPORT_DIR = (
    RESEARCH / "phase17_10D_rendered_review_export_20261025T185523"
)
RENDER_DIR = EXPORT_DIR / "rendered_pages"
SOURCE_CHECKLIST = (
    RESEARCH
    / "phase17_10E_human_visual_qa_20261025T185812"
    / "phase17_10E_page_review_checklist.csv"
)

if not RENDER_DIR.is_dir():
    raise FileNotFoundError(f"Page render directory not found: {RENDER_DIR}")

pages = sorted(
    RENDER_DIR.glob("*.png"),
    key=lambda p: [
        int(x) if x.isdigit() else x.lower()
        for x in __import__("re").split(r"(\d+)", p.stem)
    ]
)

if len(pages) != 19:
    raise RuntimeError(
        f"Expected 19 rendered PDF pages, found {len(pages)}. "
        "Check the Phase 17.10D render output before proceeding."
    )

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = RESEARCH / f"phase17_10F_R1_visual_signoff_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

CONTACT_DIR = OUT / "contact_sheets"
CONTACT_DIR.mkdir()

# Build two consistent contact sheets directly from the 19 page renders.
cols, rows, per_sheet = 4, 3, 12
thumb_w, thumb_h = 250, 323
cell_w, cell_h = 270, 360

contact_sheet_paths = []
for sheet_no, start in enumerate(range(0, len(pages), per_sheet), start=1):
    batch = pages[start:start + per_sheet]
    canvas = PILImage.new(
        "RGB", (cols * cell_w, rows * cell_h + 65), "white"
    )
    draw = ImageDraw.Draw(canvas)
    draw.text(
        (18, 18),
        f"Delhi AQI Review — Pages {start + 1}–{start + len(batch)}",
        fill="black"
    )

    for j, page_path in enumerate(batch):
        page_img = PILImage.open(page_path).convert("RGB")
        page_img.thumbnail((thumb_w, thumb_h))
        col, row = j % cols, j // cols
        x = col * cell_w + (cell_w - page_img.width) // 2
        y = 52 + row * cell_h
        canvas.paste(page_img, (x, y))
        draw.text(
            (col * cell_w + 12, y + thumb_h + 5),
            f"Page {start + j + 1}",
            fill="black"
        )

    contact_path = CONTACT_DIR / f"contact-sheet-{sheet_no:02d}.png"
    canvas.save(contact_path)
    contact_sheet_paths.append(contact_path)

print("=" * 78)
print("PHASE 17.10F-R1 — VISUAL REVIEW WORKSPACE")
print("=" * 78)
print(f"Pages found: {len(pages)}")
print("Contact sheets created:", len(contact_sheet_paths))

print("\n[1] CONTACT SHEETS")
for path in contact_sheet_paths:
    print(path.name)
    display(NotebookImage(filename=str(path)))

print("\n[2] ALL FULL-RESOLUTION PAGE RENDERS")
for i, page_path in enumerate(pages, start=1):
    display(Markdown(f"### Page {i} of {len(pages)}"))
    display(NotebookImage(filename=str(page_path), width=650))

# Preserve the existing checklist if available; otherwise create a fresh copy.
if SOURCE_CHECKLIST.is_file():
    checklist = pd.read_csv(SOURCE_CHECKLIST)
else:
    checklist = pd.DataFrame()

if len(checklist) != len(pages):
    checklist = pd.DataFrame({"page": list(range(1, len(pages) + 1))})

checklist["page"] = list(range(1, len(pages) + 1))
checklist["render_file"] = [str(p) for p in pages]

image_dimensions = []
for page_path in pages:
    with PILImage.open(page_path) as im:
        image_dimensions.append(f"{im.width}x{im.height}")
checklist["image_dimensions"] = image_dimensions

print("\n[3] RECORD YOUR VISUAL REVIEW")
print(
    "After inspecting the contact sheets AND all 19 full-resolution pages, "
    "enter YES only if you found no visible clipping, overlap, unreadable "
    "figures/tables, broken captions, or obvious pagination problems."
)
answer = input("Visual review completed with no visible issues? Type YES or NO: ")
answer = answer.strip().upper()

if answer == "YES":
    visual_status = "AUTHOR_CONFIRMED_PASS"
    checklist["manual_review_status"] = visual_status
    checklist["no_clipping_or_overlap"] = "YES"
    checklist["figures_readable"] = "YES"
    checklist["tables_readable"] = "YES"
    checklist["headings_and_page_breaks_ok"] = "YES"
    checklist["captions_and_references_ok"] = "YES"
    checklist["issues_and_notes"] = (
        "Author reviewed all 19 full-resolution pages and reported no visible issues."
    )
    issues_note = ""
    human_visual_qa_completed = True
else:
    visual_status = "PENDING_OR_ISSUES_FOUND"
    checklist["manual_review_status"] = visual_status
    for column in [
        "no_clipping_or_overlap",
        "figures_readable",
        "tables_readable",
        "headings_and_page_breaks_ok",
        "captions_and_references_ok",
    ]:
        checklist[column] = "PENDING"
    issues_note = input(
        "Optional: note the page numbers or issues found (or press Enter): "
    ).strip()
    checklist["issues_and_notes"] = issues_note or (
        "Visual review not signed off; investigate before release."
    )
    human_visual_qa_completed = False

checklist_path = OUT / "phase17_10F_R1_page_review_checklist.csv"
checklist.to_csv(checklist_path, index=False)

status = {
    "phase": "17.10F-R1",
    "created_at": datetime.now().astimezone().isoformat(),
    "render_export_directory": str(EXPORT_DIR),
    "rendered_page_count": len(pages),
    "contact_sheets": [str(p) for p in contact_sheet_paths],
    "page_render_paths": [str(p) for p in pages],
    "visual_review_status": visual_status,
    "author_confirmed_no_visible_issues": answer == "YES",
    "human_visual_qa_completed": human_visual_qa_completed,
    "issues_or_notes": issues_note,
    "checklist_path": str(checklist_path),
    "licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "models_retrained": False,
}
status_path = OUT / "phase17_10F_R1_visual_qa_status.json"
status_path.write_text(
    json.dumps(status, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n[4] SAVED REVIEW RECORD")
print("QA folder:", OUT)
print("Checklist:", checklist_path)
print("Status JSON:", status_path)
print("Visual review status:", visual_status)
print("Licence review: OPEN")
print("Submission ready: False")
print("=" * 78)

In [ ]:
# PHASE 17.10G — COMPLETE REVIEW PACKAGE ASSEMBLY
# Combines the verified paper package, DOCX/PDF, source assets, and QA evidence.
# Creates a new ZIP; never overwrites the older review packages.

from pathlib import Path
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib
import json
import warnings

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
RESEARCH = ROOT / "outputs/research_paper"

BASE_ZIP = (
    RESEARCH / "phase17_10C_final_review_package_20261025T184853"
    / "Delhi_AQI_Integrated_Research_Review_Package.zip"
)
BASE_EXPECTED_SHA256 = (
    "e60206d8958bfcc0b97e39290e7fa7e63386bdc6570d5b16bb8961ca19d22e59"
)

EXPORT_DIR = (
    RESEARCH / "phase17_10D_rendered_review_export_20261025T185523"
)
DOCX = EXPORT_DIR / "Delhi_AQI_Integrated_Review_Draft.docx"
PDF = EXPORT_DIR / "Delhi_AQI_Integrated_Review_Draft.pdf"
PAGE_DIR = EXPORT_DIR / "rendered_pages"
PAGE_INVENTORY = EXPORT_DIR / "rendered_page_inventory.csv"
RENDER_QA = EXPORT_DIR / "phase17_10D_render_qa.json"

INTEGRATED_MANUSCRIPT = (
    RESEARCH / "phase17_10B_R1_integrated_manuscript_20261025T184741"
    / "Delhi_AQI_Integrated_Manuscript.md"
)

PHASE17_08 = (
    RESEARCH / "phase17_08_licence_clarification_20261025T183001"
)
PHASE17_09 = (
    RESEARCH / "phase17_09_asset_audit_20261025T183617"
)
PHASE17_07Z = (
    RESEARCH / "phase17_07Z_content_audit_closure_20261025T182839"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def add_member(payloads, name, data):
    if name in payloads:
        raise FileExistsError(
            f"Package member collision; refusing to overwrite: {name}"
        )
    payloads[name] = data if isinstance(data, bytes) else bytes(data)

def add_file_if_exists(payloads, path, member_name, required=False):
    path = Path(path)
    if not path.is_file():
        if required:
            raise FileNotFoundError(f"Required input not found: {path}")
        warnings.warn(f"Optional supporting file not found: {path}")
        return False
    add_member(payloads, member_name, path.read_bytes())
    return True

print("=" * 78)
print("PHASE 17.10G — COMPLETE REVIEW PACKAGE ASSEMBLY")
print("=" * 78)

# 1. Verify all required inputs before creating the output package.
required_paths = [
    BASE_ZIP, DOCX, PDF, PAGE_INVENTORY, RENDER_QA,
    INTEGRATED_MANUSCRIPT,
]
missing = [str(p) for p in required_paths if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Package assembly stopped because required inputs are missing:\n"
        + "\n".join(missing)
    )

base_hash = sha256_file(BASE_ZIP)
if base_hash != BASE_EXPECTED_SHA256:
    raise RuntimeError(
        "Phase 17.10C base ZIP hash mismatch; assembly stopped."
    )

page_files = sorted(PAGE_DIR.glob("*.png"))
if len(page_files) != 19:
    raise RuntimeError(
        f"Expected 19 page renders; found {len(page_files)}."
    )

qa_dirs = sorted(
    RESEARCH.glob("phase17_10F_R1_visual_signoff_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)
if not qa_dirs:
    raise FileNotFoundError(
        "Run Phase 17.10F-R1 first; visual sign-off record not found."
    )

QA_DIR = qa_dirs[0]
QA_STATUS_PATH = QA_DIR / "phase17_10F_R1_visual_qa_status.json"
QA_CHECKLIST_PATH = QA_DIR / "phase17_10F_R1_page_review_checklist.csv"

if not QA_STATUS_PATH.is_file() or not QA_CHECKLIST_PATH.is_file():
    raise FileNotFoundError(
        "The latest Phase 17.10F-R1 folder lacks its status or checklist."
    )

with open(QA_STATUS_PATH, "r", encoding="utf-8") as f:
    visual_status = json.load(f)

print("\n[1] INPUT CHECKS")
print("Base archive SHA-256 verified:", base_hash == BASE_EXPECTED_SHA256)
print("DOCX:", DOCX.exists(), f"({DOCX.stat().st_size:,} bytes)")
print("PDF:", PDF.exists(), f"({PDF.stat().st_size:,} bytes)")
print("Page renders:", len(page_files))
print("Visual QA status:", visual_status.get("visual_review_status"))

# 2. Load every file from the base package without modifying it.
payloads = {}
with ZipFile(BASE_ZIP, "r") as source_zip:
    bad_member = source_zip.testzip()
    if bad_member:
        raise RuntimeError(f"Base ZIP integrity failure at: {bad_member}")

    for info in source_zip.infolist():
        if not info.is_dir():
            payloads[info.filename] = source_zip.read(info.filename)

base_member_count = len(payloads)

# 3. Add the integrated Markdown at the ZIP root so relative links such as
# figures/Figure_01_... still resolve alongside the existing figure folder.
add_file_if_exists(
    payloads,
    INTEGRATED_MANUSCRIPT,
    "Delhi_AQI_Integrated_Manuscript_Phase17_10G.md",
    required=True
)

# DOCX and PDF exports.
add_file_if_exists(
    payloads, DOCX,
    "exports/Delhi_AQI_Integrated_Review_Draft.docx",
    required=True
)
add_file_if_exists(
    payloads, PDF,
    "exports/Delhi_AQI_Integrated_Review_Draft.pdf",
    required=True
)

# Render QA reports.
add_file_if_exists(
    payloads, RENDER_QA,
    "visual_qa/phase17_10D_render_qa.json",
    required=True
)
add_file_if_exists(
    payloads, PAGE_INVENTORY,
    "visual_qa/rendered_page_inventory.csv",
    required=True
)
add_file_if_exists(
    payloads, QA_STATUS_PATH,
    "visual_qa/phase17_10F_R1_visual_qa_status.json",
    required=True
)
add_file_if_exists(
    payloads, QA_CHECKLIST_PATH,
    "visual_qa/phase17_10F_R1_page_review_checklist.csv",
    required=True
)

# Full-size rendered pages, for repeatable visual inspection.
for i, path in enumerate(page_files, start=1):
    add_file_if_exists(
        payloads, path,
        f"visual_qa/page_renders/page_{i:02d}.png",
        required=True
    )

# Contact sheets made in the preceding cell.
for i, path_string in enumerate(
    visual_status.get("contact_sheets", []), start=1
):
    contact = Path(path_string)
    add_file_if_exists(
        payloads, contact,
        f"visual_qa/contact_sheets/contact_sheet_{i:02d}.png",
        required=True
    )

# Licence clarification requests and the response log.
licence_files = [
    (
        PHASE17_08 / "TUMI_LICENCE_CLARIFICATION_EMAIL.md",
        "release_gates/TUMI_LICENCE_CLARIFICATION_EMAIL.md",
        True,
    ),
    (
        PHASE17_08 / "CPCB_LICENCE_CLARIFICATION_REQUEST.md",
        "release_gates/CPCB_LICENCE_CLARIFICATION_REQUEST.md",
        True,
    ),
    (
        PHASE17_08 / "PHASE_17_08_SOURCE_REGISTER.csv",
        "release_gates/PHASE_17_08_SOURCE_REGISTER.csv",
        True,
    ),
    (
        PHASE17_08 / "PHASE_17_08_RESPONSE_TRACKER.csv",
        "release_gates/PHASE_17_08_RESPONSE_TRACKER.csv",
        True,
    ),
    (
        PHASE17_08 / "phase17_08_status.json",
        "release_gates/phase17_08_status.json",
        True,
    ),
    (
        PHASE17_09 / "phase17_09_asset_audit_report.md",
        "release_gates/phase17_09_asset_audit_report.md",
        False,
    ),
    (
        PHASE17_09 / "phase17_09_asset_audit_summary.json",
        "release_gates/phase17_09_asset_audit_summary.json",
        False,
    ),
    (
        PHASE17_07Z / "phase17_07Z_content_audit_closure.md",
        "release_gates/phase17_07Z_content_audit_closure.md",
        False,
    ),
    (
        PHASE17_07Z / "phase17_07Z_content_audit_closure.json",
        "release_gates/phase17_07Z_content_audit_closure.json",
        False,
    ),
]

for source_path, package_name, required in licence_files:
    add_file_if_exists(
        payloads, source_path, package_name, required=required
    )

# Include the existing Phase 17.07U author checklist/log template if present.
for filename in [
    "AUTHOR_RELEASE_CHECKLIST.md",
    "AUTHOR_LICENCE_REVIEW_LOG_TEMPLATE.md",
]:
    candidates = sorted(
        RESEARCH.rglob(filename),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    if candidates:
        add_file_if_exists(
            payloads,
            candidates[0],
            f"release_gates/{filename}",
            required=False
        )

# 4. Record gates truthfully. A package can be complete while release gates
# remain open; do not equate packaging success with submission readiness.
human_visual_qa_completed = bool(
    visual_status.get("human_visual_qa_completed", False)
)

open_gates = []
if not human_visual_qa_completed:
    open_gates.append(
        "Human visual review of the 19 page renders has not been signed off "
        "as having no visible issues."
    )
open_gates.append(
    "Author licence review remains open; external clarification requests "
    "were prepared but responses have not been recorded."
)

release_status = {
    "phase": "17.10G",
    "created_at": datetime.now().astimezone().isoformat(),
    "package_type": "Complete integrated research review package",
    "base_package_sha256": base_hash,
    "human_visual_qa_completed": human_visual_qa_completed,
    "visual_review_status": visual_status.get("visual_review_status"),
    "licence_review": "OPEN",
    "licence_requests_sent": False,
    "licence_responses_received": False,
    "submission_ready": False,
    "production_enabled": False,
    "source_data_modified": False,
    "frozen_models_retrained": False,
    "previous_review_packages_modified": False,
    "open_gates": open_gates,
}

add_member(
    payloads,
    "RELEASE_STATUS.json",
    json.dumps(
        release_status, indent=2, ensure_ascii=False
    ).encode("utf-8")
)

readme = f"""# Delhi AQI — Complete Integrated Research Review Package

Created: {release_status["created_at"]}

## Main deliverables
- `Delhi_AQI_Integrated_Manuscript_Phase17_10G.md`
- `exports/Delhi_AQI_Integrated_Review_Draft.docx`
- `exports/Delhi_AQI_Integrated_Review_Draft.pdf`
- The seven figure pairs and nine main table CSVs carried forward from the
  verified Phase 17.10C review package
- Supporting research evidence and source/provenance records
- `visual_qa/`: all 19 full page renders, contact sheets, and review checklist
- `release_gates/`: licence correspondence drafts, response tracker, and audit
  evidence
- `RELEASE_STATUS.json`

## Integrity
The verified Phase 17.10C review package is preserved as the source. This
package adds exports and audit evidence without changing the earlier ZIP.

## Release status
- Visual QA sign-off recorded: {human_visual_qa_completed}
- Author licence review: OPEN
- Submission ready: NO
- Production enabled: NO

This is a complete review package, not a declaration of legal clearance or a
final journal submission.
"""
add_member(payloads, "README_FINAL_REVIEW_PACKAGE.md", readme.encode("utf-8"))

# Hash every payload; manifest does not include its own hash.
manifest = {
    "package_name": "Delhi_AQI_Complete_Research_Review_Package.zip",
    "created_at": datetime.now().astimezone().isoformat(),
    "base_review_zip_sha256": base_hash,
    "payload_file_count_excluding_manifest": len(payloads),
    "files": {
        name: {
            "bytes": len(data),
            "sha256": sha256_bytes(data),
        }
        for name, data in sorted(payloads.items())
    },
}

add_member(
    payloads,
    "package_manifest.json",
    json.dumps(manifest, indent=2, ensure_ascii=False).encode("utf-8")
)

# 5. Write a completely new archive.
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = RESEARCH / f"phase17_10G_complete_review_package_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)
PACKAGE = OUT / "Delhi_AQI_Complete_Research_Review_Package.zip"

with ZipFile(PACKAGE, "w", compression=ZIP_DEFLATED, compresslevel=6) as z:
    for member_name, data in sorted(payloads.items()):
        z.writestr(member_name, data)

print("\n[2] NEW PACKAGE")
print("Output folder:", OUT)
print("ZIP:", PACKAGE)
print("Base ZIP entries carried forward:", base_member_count)
print("Total entries written:", len(payloads))
print("Full page renders included:", len(page_files))
print("Visual QA sign-off:", human_visual_qa_completed)
print("Licence review: OPEN")
print("Submission ready: False")
print("Existing archives/source data/models modified: False")
print("=" * 78)

In [ ]:
# PHASE 17.10H — FINAL PACKAGE INTEGRITY AND RELEASE AUDIT
# Verifies the new Phase 17.10G ZIP and its included files.
# Writes verification results outside the ZIP so the verified hash stays stable.

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from datetime import datetime
import hashlib
import json
import re
import io
import warnings

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
RESEARCH = ROOT / "outputs/research_paper"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

# Locate the package produced by Phase 17.10G.
package_dirs = sorted(
    RESEARCH.glob("phase17_10G_complete_review_package_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)
if not package_dirs:
    raise FileNotFoundError(
        "No Phase 17.10G package folder found. Run Code block 2 first."
    )

package_path = package_dirs[0] / "Delhi_AQI_Complete_Research_Review_Package.zip"
if not package_path.is_file():
    raise FileNotFoundError(f"Package ZIP not found: {package_path}")

print("=" * 78)
print("PHASE 17.10H — FINAL PACKAGE INTEGRITY AND RELEASE AUDIT")
print("=" * 78)

package_hash = sha256_file(package_path)
audit_errors = []
checks = {}

with ZipFile(package_path, "r") as z:
    damaged_member = z.testzip()
    checks["zip_integrity"] = "PASS" if damaged_member is None else "FAIL"
    if damaged_member is not None:
        audit_errors.append(f"ZIP member failed CRC: {damaged_member}")

    all_names = [n for n in z.namelist() if not n.endswith("/")]
    name_set = set(all_names)

    # 1. Verify archive manifest and every member hash/size.
    if "package_manifest.json" not in name_set:
        raise FileNotFoundError("Internal package_manifest.json is missing.")

    manifest = json.loads(z.read("package_manifest.json"))
    manifest_files = manifest.get("files", {})
    actual_file_names_without_manifest = name_set - {"package_manifest.json"}

    checks["manifest_member_set_matches"] = (
        set(manifest_files.keys()) == actual_file_names_without_manifest
    )
    if not checks["manifest_member_set_matches"]:
        audit_errors.append("Manifest member list does not match ZIP entries.")

    bad_hashes = []
    for name, details in manifest_files.items():
        if name not in name_set:
            bad_hashes.append(f"MISSING: {name}")
            continue
        data = z.read(name)
        if sha256_bytes(data) != details.get("sha256"):
            bad_hashes.append(f"HASH MISMATCH: {name}")
        if len(data) != details.get("bytes"):
            bad_hashes.append(f"SIZE MISMATCH: {name}")

    checks["internal_hashes_pass"] = len(bad_hashes) == 0
    if bad_hashes:
        audit_errors.extend(bad_hashes)

    # 2. Verify deliverable files are present.
    required_members = [
        "exports/Delhi_AQI_Integrated_Review_Draft.docx",
        "exports/Delhi_AQI_Integrated_Review_Draft.pdf",
        "Delhi_AQI_Integrated_Manuscript_Phase17_10G.md",
        "RELEASE_STATUS.json",
        "README_FINAL_REVIEW_PACKAGE.md",
        "visual_qa/phase17_10F_R1_visual_qa_status.json",
        "visual_qa/phase17_10F_R1_page_review_checklist.csv",
    ]

    missing_required = [m for m in required_members if m not in name_set]
    checks["required_deliverables_present"] = len(missing_required) == 0
    if missing_required:
        audit_errors.append(
            "Required deliverables missing: " + ", ".join(missing_required)
        )

    # 3. Verify all nine main table CSVs by their expected basenames.
    required_table_basenames = {
        "table_1_validation_model_comparison.csv",
        "table_2_holdout_matched_support.csv",
        "table_3_holdout_full_available_support.csv",
        "table_4_day_block_bootstrap.csv",
        "table_5_performance_by_aqi_band.csv",
        "table_6_abrupt_transition_detection.csv",
        "table_7_feature_attribution.csv",
        "table_8_station_performance.csv",
        "table_9_season_performance.csv",
    }

    csv_basenames = {
        PurePosixPath(n).name for n in all_names if n.lower().endswith(".csv")
    }
    missing_tables = sorted(required_table_basenames - csv_basenames)
    checks["main_tables_present"] = len(missing_tables) == 0
    checks["main_table_count"] = len(required_table_basenames & csv_basenames)

    if missing_tables:
        audit_errors.append(
            "Required main tables missing: " + ", ".join(missing_tables)
        )

    # Read every required table from the archive to make sure it parses.
    unreadable_tables = []
    for name in required_table_basenames:
        member = next(
            (
                m for m in all_names
                if PurePosixPath(m).name == name
            ),
            None
        )
        if member is None:
            continue
        try:
            import pandas as pd
            df_table = pd.read_csv(io.BytesIO(z.read(member)))
            if df_table.empty or len(df_table.columns) == 0:
                unreadable_tables.append(name)
        except Exception as exc:
            unreadable_tables.append(f"{name}: {exc}")

    checks["main_tables_readable"] = len(unreadable_tables) == 0
    if unreadable_tables:
        audit_errors.append(
            "Unreadable/empty tables: " + "; ".join(unreadable_tables)
        )

    # 4. Check figure pairs, distinct from full-page QA images.
    figure_pngs = [
        n for n in all_names
        if "figures" in [p.lower() for p in PurePosixPath(n).parts]
        and n.lower().endswith(".png")
        and "figure_" in PurePosixPath(n).name.lower()
    ]
    figure_pdfs = [
        n for n in all_names
        if "figures" in [p.lower() for p in PurePosixPath(n).parts]
        and n.lower().endswith(".pdf")
        and "figure_" in PurePosixPath(n).name.lower()
    ]

    checks["figure_png_count"] = len(figure_pngs)
    checks["figure_pdf_count"] = len(figure_pdfs)
    checks["figure_pngs_pass"] = len(figure_pngs) == 7
    checks["figure_pdfs_pass"] = len(figure_pdfs) == 7

    if len(figure_pngs) != 7:
        audit_errors.append(
            f"Expected 7 publication figure PNGs; found {len(figure_pngs)}."
        )
    if len(figure_pdfs) != 7:
        audit_errors.append(
            f"Expected 7 publication figure PDFs; found {len(figure_pdfs)}."
        )

    # 5. Check all full page renders and contact sheets.
    page_renders = [
        n for n in all_names
        if n.startswith("visual_qa/page_renders/")
        and n.lower().endswith(".png")
    ]
    contact_sheets = [
        n for n in all_names
        if n.startswith("visual_qa/contact_sheets/")
        and n.lower().endswith(".png")
    ]

    checks["page_render_count"] = len(page_renders)
    checks["page_renders_pass"] = len(page_renders) == 19
    checks["contact_sheet_count"] = len(contact_sheets)
    checks["contact_sheets_pass"] = len(contact_sheets) >= 2

    if len(page_renders) != 19:
        audit_errors.append(
            f"Expected 19 QA page renders; found {len(page_renders)}."
        )
    if len(contact_sheets) < 2:
        audit_errors.append(
            f"Expected at least 2 contact sheets; found {len(contact_sheets)}."
        )

    # 6. Check that source raw data was not accidentally included.
    prohibited_raw_names = {"station_hour.csv", "stations.csv"}
    included_raw_names = sorted({
        PurePosixPath(n).name.lower()
        for n in all_names
        if PurePosixPath(n).name.lower() in prohibited_raw_names
    })
    checks["raw_source_csvs_absent"] = len(included_raw_names) == 0
    checks["raw_source_csvs_found"] = included_raw_names

    if included_raw_names:
        audit_errors.append(
            "Possible original raw source files included: "
            + ", ".join(included_raw_names)
        )

    # 7. Read visual/licence status from inside the final ZIP.
    visual_status = json.loads(
        z.read("visual_qa/phase17_10F_R1_visual_qa_status.json")
    )
    release_status = json.loads(z.read("RELEASE_STATUS.json"))

    checks["author_visual_qa_completed"] = bool(
        visual_status.get("human_visual_qa_completed", False)
    )
    checks["author_licence_review_open"] = (
        release_status.get("licence_review") == "OPEN"
    )
    checks["submission_ready"] = bool(
        release_status.get("submission_ready", False)
    )
    checks["production_enabled"] = bool(
        release_status.get("production_enabled", False)
    )

    if not checks["author_visual_qa_completed"]:
        warnings.warn(
            "Human visual QA is not signed off. This does not block creation "
            "of the review package, but it blocks final submission clearance."
        )
    if not checks["author_licence_review_open"]:
        warnings.warn(
            "Licence status differs from the expected open review gate. "
            "Inspect the release status carefully."
        )

# 8. Verify DOCX structure and embedded images.
docx_structure = {}
try:
    from docx import Document
    from zipfile import ZipFile as InnerZipFile

    with ZipFile(package_path, "r") as z:
        docx_bytes = z.read(
            "exports/Delhi_AQI_Integrated_Review_Draft.docx"
        )

    doc = Document(io.BytesIO(docx_bytes))
    with InnerZipFile(io.BytesIO(docx_bytes), "r") as inner:
        embedded_media = [
            n for n in inner.namelist()
            if n.startswith("word/media/")
        ]

    docx_structure = {
        "bytes": len(docx_bytes),
        "paragraphs": len(doc.paragraphs),
        "word_tables": len(doc.tables),
        "inline_images": len(doc.inline_shapes),
        "docx_media_files": len(embedded_media),
        "structure_readable": True,
    }
    checks["docx_structure_readable"] = True
    checks["docx_has_7_embedded_images"] = len(doc.inline_shapes) >= 7

    if len(doc.inline_shapes) < 7:
        audit_errors.append(
            f"DOCX contains only {len(doc.inline_shapes)} inline images."
        )
except Exception as exc:
    docx_structure = {"structure_readable": False, "error": repr(exc)}
    checks["docx_structure_readable"] = False
    checks["docx_has_7_embedded_images"] = False
    audit_errors.append(f"DOCX structure read failed: {exc}")

# 9. Verify PDF opens and record its page count.
pdf_structure = {}
try:
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz

    with ZipFile(package_path, "r") as z:
        pdf_bytes = z.read(
            "exports/Delhi_AQI_Integrated_Review_Draft.pdf"
        )

    pdf_doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    page_text_lengths = [
        len(page.get_text("text").strip()) for page in pdf_doc
    ]
    pdf_structure = {
        "bytes": len(pdf_bytes),
        "page_count": len(pdf_doc),
        "pages_with_text": sum(n > 0 for n in page_text_lengths),
        "total_text_characters": sum(page_text_lengths),
        "pdf_opens": True,
    }

    checks["pdf_opens"] = True
    checks["pdf_has_19_pages"] = len(pdf_doc) == 19
    if len(pdf_doc) != 19:
        audit_errors.append(
            f"Expected 19 PDF pages; found {len(pdf_doc)}."
        )
except Exception as exc:
    pdf_structure = {"pdf_opens": False, "error": repr(exc)}
    checks["pdf_opens"] = False
    checks["pdf_has_19_pages"] = False
    audit_errors.append(f"PDF read failed: {exc}")

# 10. Save the final independent audit outside the ZIP.
overall_integrity_pass = (
    checks.get("zip_integrity") == "PASS"
    and checks.get("internal_hashes_pass", False)
    and checks.get("manifest_member_set_matches", False)
    and checks.get("required_deliverables_present", False)
    and checks.get("main_tables_present", False)
    and checks.get("main_tables_readable", False)
    and checks.get("figure_pngs_pass", False)
    and checks.get("figure_pdfs_pass", False)
    and checks.get("page_renders_pass", False)
    and checks.get("contact_sheets_pass", False)
    and checks.get("docx_structure_readable", False)
    and checks.get("pdf_opens", False)
    and checks.get("raw_source_csvs_absent", False)
)

audit = {
    "phase": "17.10H",
    "verified_at": datetime.now().astimezone().isoformat(),
    "package_path": str(package_path),
    "package_sha256": package_hash,
    "package_bytes": package_path.stat().st_size,
    "zip_entry_count": len(all_names),
    "checks": checks,
    "docx_structure": docx_structure,
    "pdf_structure": pdf_structure,
    "audit_errors": audit_errors,
    "package_integrity_status": (
        "PASS" if overall_integrity_pass else "FAIL"
    ),
    "human_visual_qa_completed": checks.get("author_visual_qa_completed", False),
    "licence_review": "OPEN",
    "submission_ready": False,
    "production_enabled": False,
    "interpretation": (
        "Package integrity and content checks may pass while author visual "
        "review or licence clearance remains open. Submission readiness is "
        "not inferred from successful ZIP creation."
    ),
}

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
audit_json = package_path.parent / f"phase17_10H_final_verification_{stamp}.json"
audit_md = package_path.parent / f"phase17_10H_final_verification_{stamp}.md"
sha_file = package_path.parent / "Delhi_AQI_Complete_Research_Review_Package.sha256"

audit_json.write_text(
    json.dumps(audit, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

md_lines = [
    "# Phase 17.10H — Final Package Verification",
    "",
    f"- Package: `{package_path}`",
    f"- SHA-256: `{package_hash}`",
    f"- Package bytes: {package_path.stat().st_size:,}",
    f"- ZIP integrity: {checks.get('zip_integrity')}",
    f"- Internal file hashes pass: {checks.get('internal_hashes_pass')}",
    f"- Figure PNG/PDF pairs: {checks.get('figure_png_count')}/"
    f"{checks.get('figure_pdf_count')}",
    f"- Main tables present/readable: {checks.get('main_tables_present')}/"
    f"{checks.get('main_tables_readable')}",
    f"- Full page renders: {checks.get('page_render_count')}",
    f"- DOCX readable and includes 7 images: "
    f"{checks.get('docx_structure_readable')}/"
    f"{checks.get('docx_has_7_embedded_images')}",
    f"- PDF opens and has 19 pages: {checks.get('pdf_opens')}/"
    f"{checks.get('pdf_has_19_pages')}",
    f"- Raw source CSVs absent: {checks.get('raw_source_csvs_absent')}",
    "",
    f"**Package integrity status: {audit['package_integrity_status']}**",
    "",
    f"- Human visual QA completed: {audit['human_visual_qa_completed']}",
    "- Author licence review: OPEN",
    "- Submission ready: NO",
    "- Production enabled: NO",
    "",
    "## Errors",
]
md_lines.extend([f"- {error}" for error in audit_errors] or ["- None."])
md_lines += [
    "",
    "The package is a complete research review package. A successful integrity "
    "audit does not clear the licence review or substitute for scientific, "
    "editorial, and author sign-off."
]
audit_md.write_text("\n".join(md_lines), encoding="utf-8")

sha_file.write_text(
    f"{package_hash}  {package_path.name}\n",
    encoding="utf-8"
)

print("=" * 78)
print("PHASE 17.10H — FINAL PACKAGE INTEGRITY AND RELEASE AUDIT")
print("=" * 78)
print("Final ZIP:", package_path)
print("SHA-256:", package_hash)
print("ZIP integrity:", checks.get("zip_integrity"))
print("Internal hashes pass:", checks.get("internal_hashes_pass"))
print("Figure PNG/PDF:", checks.get("figure_png_count"),
      "/", checks.get("figure_pdf_count"))
print("Main tables present/readable:", checks.get("main_tables_present"),
      "/", checks.get("main_tables_readable"))
print("Page renders:", checks.get("page_render_count"))
print("DOCX structure readable:", checks.get("docx_structure_readable"))
print("DOCX embedded images:", docx_structure.get("inline_images"))
print("PDF opens:", checks.get("pdf_opens"))
print("PDF pages:", pdf_structure.get("page_count"))
print("Raw source CSVs absent:", checks.get("raw_source_csvs_absent"))
print("PACKAGE INTEGRITY:", audit["package_integrity_status"])

print("\n[FINAL RELEASE GATES]")
print("Human visual QA completed:", audit["human_visual_qa_completed"])
print("Author licence review:", audit["licence_review"])
print("Submission ready:", audit["submission_ready"])
print("Production enabled:", audit["production_enabled"])

print("\n[VERIFICATION FILES]")
print("JSON:", audit_json)
print("Report:", audit_md)
print("SHA-256 file:", sha_file)
print("=" * 78)

if audit_errors:
    print("\nAudit issues to resolve:")
    for error in audit_errors:
        print("-", error)

In [ ]:
# PHASE 17.10H-R1 — RECHECK DOCX/PDF AFTER DEPENDENCY REPAIR
# Does not rebuild or modify the existing review package.

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from datetime import datetime
import hashlib
import importlib.util
import subprocess
import sys
import io
import json
import csv
import re

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

PACKAGE_DIR = (
    ROOT / "outputs/research_paper"
    / "phase17_10G_complete_review_package_20261026T085040"
)

ZIP_PATH = PACKAGE_DIR / "Delhi_AQI_Complete_Research_Review_Package.zip"
EXPECTED_ZIP_SHA256 = (
    "c2b792816796269d7aba6542d689854d4b0f6c80f3162f037c4eb14be93c83f5"
)
EXPECTED_PDF_PAGES = 19
EXPECTED_FIGURES = 7
EXPECTED_MAIN_TABLES = 9

print("=" * 78)
print("PHASE 17.10H-R1 — DOCX/PDF DEPENDENCY RECOVERY AND VERIFICATION")
print("=" * 78)

# ------------------------------------------------------------------
# 1. Install only missing inspection libraries.
# ------------------------------------------------------------------
packages_to_install = []

if importlib.util.find_spec("docx") is None:
    packages_to_install.append("python-docx")

if importlib.util.find_spec("pymupdf") is None:
    packages_to_install.append("pymupdf")

if packages_to_install:
    print("Installing missing inspection libraries:", packages_to_install)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        *packages_to_install
    ])
else:
    print("Document inspection libraries already installed.")

from docx import Document
import pymupdf

# ------------------------------------------------------------------
# 2. Verify exact ZIP identity and integrity before inspecting members.
# ------------------------------------------------------------------
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Final package ZIP not found: {ZIP_PATH}")

actual_zip_sha256 = sha256_file(ZIP_PATH)
if actual_zip_sha256 != EXPECTED_ZIP_SHA256:
    raise RuntimeError(
        "Final package SHA-256 mismatch. Document inspection stopped."
    )

with ZipFile(ZIP_PATH, "r") as z:
    bad_member = z.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity test failed at: {bad_member}")

    members = [info.filename for info in z.infolist() if not info.is_dir()]
    member_set = set(members)

    # Discover documents by basename, not a hard-coded internal directory.
    docx_candidates = [
        m for m in members if m.lower().endswith(".docx")
    ]
    pdf_candidates = [
        m for m in members if m.lower().endswith(".pdf")
    ]

    if not docx_candidates:
        raise FileNotFoundError("No DOCX file found inside the final ZIP.")
    if not pdf_candidates:
        raise FileNotFoundError("No PDF file found inside the final ZIP.")

    preferred_docx = [
        m for m in docx_candidates
        if "integrated_review_draft" in PurePosixPath(m).name.lower()
    ]
    docx_member = preferred_docx[0] if preferred_docx else max(
        docx_candidates, key=lambda m: z.getinfo(m).file_size
    )

    preferred_pdf = [
        m for m in pdf_candidates
        if "integrated_review_draft" in PurePosixPath(m).name.lower()
    ]

    docx_bytes = z.read(docx_member)

    # Open candidate PDFs and choose the named manuscript PDF, or otherwise
    # the valid PDF with the highest page count.
    valid_pdf_info = []
    pdf_open_errors = []

    for member in pdf_candidates:
        raw_pdf = z.read(member)
        try:
            pdf_doc = pymupdf.open(stream=raw_pdf, filetype="pdf")
            page_count = pdf_doc.page_count
            extracted_text_chars = sum(
                len(page.get_text("text")) for page in pdf_doc
            )
            pdf_doc.close()
            valid_pdf_info.append({
                "member": member,
                "bytes": len(raw_pdf),
                "page_count": page_count,
                "extracted_text_characters": extracted_text_chars,
            })
        except Exception as exc:
            pdf_open_errors.append({
                "member": member,
                "error": f"{type(exc).__name__}: {exc}"
            })

    if preferred_pdf:
        preferred_valid = [
            item for item in valid_pdf_info
            if item["member"] in preferred_pdf
        ]
        if preferred_valid:
            chosen_pdf = preferred_valid[0]
        else:
            raise RuntimeError(
                "The specifically named integrated review PDF exists but "
                "could not be opened."
            )
    else:
        if not valid_pdf_info:
            raise RuntimeError("No PDF inside the ZIP could be opened.")
        chosen_pdf = sorted(
            valid_pdf_info,
            key=lambda item: (
                item["page_count"],
                item["extracted_text_characters"],
                item["bytes"],
            ),
            reverse=True,
        )[0]

    pdf_member = chosen_pdf["member"]
    pdf_bytes = z.read(pdf_member)

    # Discover and validate the seven publication figure PNG/PDF pairs.
    figure_pngs = [
        m for m in members
        if PurePosixPath(m).name.startswith("Figure_")
        and m.lower().endswith(".png")
    ]
    figure_pdfs = [
        m for m in members
        if PurePosixPath(m).name.startswith("Figure_")
        and m.lower().endswith(".pdf")
    ]

    # Discover the nine main table CSVs by filename.
    table_csvs = [
        m for m in members
        if re.match(
            r"table_[1-9]_.*\.csv$",
            PurePosixPath(m).name,
            re.IGNORECASE,
        )
    ]

    table_checks = []
    for member in table_csvs:
        raw = z.read(member)
        try:
            # Parse the complete CSV through the Python CSV reader.
            text = raw.decode("utf-8-sig")
            rows = list(csv.reader(io.StringIO(text)))
            table_checks.append({
                "member": member,
                "readable": bool(rows and rows[0]),
                "row_count_including_header": len(rows),
                "column_count": len(rows[0]) if rows else 0,
                "sha256": sha256_bytes(raw),
            })
        except Exception as exc:
            table_checks.append({
                "member": member,
                "readable": False,
                "error": f"{type(exc).__name__}: {exc}",
            })

    raw_source_names = {
        "station_hour.csv", "stations.csv",
    }
    raw_source_members = [
        m for m in members
        if PurePosixPath(m).name.lower() in raw_source_names
    ]

# ------------------------------------------------------------------
# 3. Read the DOCX structure.
# ------------------------------------------------------------------
docx_open_error = None
docx_paragraph_count = None
docx_table_count = None
docx_image_count = None
docx_readable = False

try:
    word_doc = Document(io.BytesIO(docx_bytes))
    docx_paragraph_count = len(word_doc.paragraphs)
    docx_table_count = len(word_doc.tables)
    docx_image_count = len(word_doc.inline_shapes)
    docx_readable = (
        docx_paragraph_count > 0
        and docx_table_count >= EXPECTED_MAIN_TABLES
        and docx_image_count >= EXPECTED_FIGURES
    )
except Exception as exc:
    docx_open_error = f"{type(exc).__name__}: {exc}"

# ------------------------------------------------------------------
# 4. Reopen PDF and check page count/text.
# ------------------------------------------------------------------
pdf_open_error = None
pdf_page_count = None
pdf_text_characters = None
pdf_readable = False
pdf_page_count_matches_previous_QA = False

try:
    pdf_doc = pymupdf.open(stream=pdf_bytes, filetype="pdf")
    pdf_page_count = pdf_doc.page_count
    pdf_text_characters = sum(
        len(page.get_text("text")) for page in pdf_doc
    )
    pdf_readable = (
        pdf_page_count > 0
        and pdf_text_characters > 100
    )
    pdf_page_count_matches_previous_QA = (
        pdf_page_count == EXPECTED_PDF_PAGES
    )
    pdf_doc.close()
except Exception as exc:
    pdf_open_error = f"{type(exc).__name__}: {exc}"

# ------------------------------------------------------------------
# 5. Consolidate the verification result.
# ------------------------------------------------------------------
all_tables_readable = (
    len(table_checks) == EXPECTED_MAIN_TABLES
    and all(t.get("readable", False) for t in table_checks)
)

figure_pair_count_pass = (
    len(figure_pngs) == EXPECTED_FIGURES
    and len(figure_pdfs) == EXPECTED_FIGURES
)

# PDF page count comparison is reported separately because packaging may
# legitimately contain a re-export; actual opening/text extraction is primary.
package_integrity_pass = all([
    actual_zip_sha256 == EXPECTED_ZIP_SHA256,
    docx_readable,
    pdf_readable,
    all_tables_readable,
    figure_pair_count_pass,
    not raw_source_members,
])

stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
REPORT_DIR = (
    ROOT / "outputs/research_paper"
    / f"phase17_10H_R1_dependency_recheck_{stamp}"
)
REPORT_DIR.mkdir(parents=True, exist_ok=False)

result = {
    "phase": "17.10H-R1",
    "created_at": datetime.now().astimezone().isoformat(),
    "package_zip": str(ZIP_PATH),
    "package_sha256_expected": EXPECTED_ZIP_SHA256,
    "package_sha256_actual": actual_zip_sha256,
    "zip_hash_verified": actual_zip_sha256 == EXPECTED_ZIP_SHA256,
    "zip_integrity": "PASS",
    "zip_member_count": len(members),
    "docx_member": docx_member,
    "docx_sha256": sha256_bytes(docx_bytes),
    "docx_readable": docx_readable,
    "docx_paragraph_count": docx_paragraph_count,
    "docx_table_count": docx_table_count,
    "docx_embedded_image_count": docx_image_count,
    "docx_open_error": docx_open_error,
    "pdf_member": pdf_member,
    "pdf_sha256": sha256_bytes(pdf_bytes),
    "pdf_readable": pdf_readable,
    "pdf_page_count": pdf_page_count,
    "pdf_text_characters": pdf_text_characters,
    "pdf_page_count_expected_from_previous_QA": EXPECTED_PDF_PAGES,
    "pdf_page_count_matches_previous_QA": pdf_page_count_matches_previous_QA,
    "pdf_open_error": pdf_open_error,
    "valid_pdf_files_found": valid_pdf_info,
    "pdf_open_errors_for_other_pdf_members": pdf_open_errors,
    "figure_png_count": len(figure_pngs),
    "figure_pdf_count": len(figure_pdfs),
    "figure_pair_count_pass": figure_pair_count_pass,
    "main_table_csv_count": len(table_csvs),
    "main_tables_readable": all_tables_readable,
    "table_checks": table_checks,
    "raw_source_csv_members_found": raw_source_members,
    "raw_source_csvs_absent": not raw_source_members,
    "package_integrity": "PASS" if package_integrity_pass else "FAIL",
    "internal_hashes_prior_phase17_10H": "PASS (as recorded in prior audit)",
    "author_licence_review": "OPEN",
    "human_visual_QA": "Previously marked completed; this cell does not replace visual review.",
    "submission_ready": False,
    "production_enabled": False,
    "original_package_zip_modified": False,
    "source_data_modified": False,
    "models_retrained": False,
}

json_path = REPORT_DIR / "phase17_10H_R1_verification.json"
json_path.write_text(
    json.dumps(result, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

report_lines = [
    "# Phase 17.10H-R1 — Dependency-Repaired Package Verification",
    "",
    f"- Package ZIP SHA-256 verified: {result['zip_hash_verified']}",
    f"- ZIP integrity: {result['zip_integrity']}",
    f"- ZIP entries: {len(members)}",
    "",
    "## DOCX",
    f"- Member: `{docx_member}`",
    f"- Readable structure: {docx_readable}",
    f"- Paragraphs: {docx_paragraph_count}",
    f"- Tables: {docx_table_count}",
    f"- Embedded images: {docx_image_count}",
    f"- Error: {docx_open_error or 'None'}",
    "",
    "## PDF",
    f"- Member: `{pdf_member}`",
    f"- Opens and text extraction works: {pdf_readable}",
    f"- Pages: {pdf_page_count}",
    f"- Extracted text characters: {pdf_text_characters}",
    f"- Matches the previously recorded 19 pages: {pdf_page_count_matches_previous_QA}",
    f"- Error: {pdf_open_error or 'None'}",
    "",
    "## Figures and tables",
    f"- Figure PNG/PDF files: {len(figure_pngs)} / {len(figure_pdfs)}",
    f"- Main table CSVs: {len(table_csvs)}",
    f"- Main tables readable: {all_tables_readable}",
    f"- Raw source CSVs absent: {not raw_source_members}",
    "",
    f"## Package integrity: **{result['package_integrity']}**",
    "",
    "The DOCX/PDF dependency issue has been rechecked using installed document "
    "libraries. A readable package does not establish licence clearance.",
    "",
    "- Author licence review: OPEN",
    "- Submission ready: NO",
    "- Production enabled: NO",
    "- Original ZIP/source data/models modified: NO",
]

md_path = REPORT_DIR / "phase17_10H_R1_verification.md"
md_path.write_text("\n".join(report_lines), encoding="utf-8")

print("\n[1] DOCUMENT STRUCTURE")
print("DOCX member:", docx_member)
print("DOCX readable:", docx_readable)
print("Paragraphs:", docx_paragraph_count)
print("Word tables:", docx_table_count)
print("Embedded images:", docx_image_count)
if docx_open_error:
    print("DOCX error:", docx_open_error)

print("\n[2] PDF")
print("PDF member:", pdf_member)
print("PDF readable:", pdf_readable)
print("Page count:", pdf_page_count)
print("Text characters:", pdf_text_characters)
print("Matches previous 19-page QA:", pdf_page_count_matches_previous_QA)
if pdf_open_error:
    print("PDF error:", pdf_open_error)

print("\n[3] PACKAGE CONTENTS")
print("ZIP entries:", len(members))
print("Figure PNG/PDF:", len(figure_pngs), "/", len(figure_pdfs))
print("Main table CSVs:", len(table_csvs))
print("Tables readable:", all_tables_readable)
print("Raw source CSVs absent:", not raw_source_members)
print("ZIP SHA-256 verified:", result["zip_hash_verified"])
print("ZIP integrity: PASS")

print("\n[4] FINAL STATUS")
print("PACKAGE INTEGRITY:", result["package_integrity"])
print("Author licence review: OPEN")
print("Submission ready: False")
print("Production enabled: False")
print("Original ZIP/source data/models modified: False")

print("\n[5] REPORTS")
print("JSON:", json_path)
print("Report:", md_path)
print("=" * 78)

In [ ]:
# PHASE 17.10H-R2 — FINAL PHASE CLOSEOUT AND ARTIFACT MANIFEST
# Consolidates existing verification evidence; does not retrain or alter models.
# Does not falsely mark human visual QA or licence review as complete.

from pathlib import Path, PurePosixPath
from datetime import datetime
from zipfile import ZipFile
import hashlib
import json
import csv
import re
import xml.etree.ElementTree as ET

import pandas as pd

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
RESEARCH = ROOT / "outputs" / "research_paper"

EXPORT_DIR = (
    RESEARCH
    / "phase17_10D_rendered_review_export_20261025T185523"
)
QA_DIR = (
    RESEARCH
    / "phase17_10E_human_visual_qa_20261025T185812"
)
H_R1_DIR = (
    RESEARCH
    / "phase17_10H_R1_dependency_recheck_20261026T085601"
)
PHASE_08_DIR = (
    RESEARCH
    / "phase17_08_licence_clarification_20261025T183001"
)

DOCX = EXPORT_DIR / "Delhi_AQI_Integrated_Review_Draft.docx"
PDF = EXPORT_DIR / "Delhi_AQI_Integrated_Review_Draft.pdf"
CHECKLIST = QA_DIR / "phase17_10E_page_review_checklist.csv"
H_R1_JSON = H_R1_DIR / "phase17_10H_R1_verification.json"
H_R1_MD = H_R1_DIR / "phase17_10H_R1_verification.md"
LICENSE_TRACKER = PHASE_08_DIR / "PHASE_17_08_RESPONSE_TRACKER.csv"

EXPECTED_PHASE_10C_SHA256 = (
    "e60206d8958bfcc0b97e39290e7fa7e63386bdc6570d5b16bb8961ca19d22e59"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def load_json(path):
    if path.is_file():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def iter_strings(obj):
    if isinstance(obj, dict):
        for key, value in obj.items():
            yield str(key), value
            yield from iter_strings(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from iter_strings(value)

def inspect_zip(path):
    """Return the package inventory; raises if ZIP integrity fails."""
    with ZipFile(path, "r") as z:
        bad = z.testzip()
        if bad:
            raise RuntimeError(f"ZIP integrity failure: {bad}")

        members = [n for n in z.namelist() if not n.endswith("/")]
        lower = [m.lower() for m in members]

        docx_members = [
            members[i] for i, name in enumerate(lower)
            if name.endswith(".docx")
        ]
        pdf_members = [
            members[i] for i, name in enumerate(lower)
            if name.endswith(".pdf")
        ]

        figure_pngs = [
            m for m in members
            if "figures/" in m.lower()
            and m.lower().endswith(".png")
        ]
        figure_pdfs = [
            m for m in members
            if "figures/" in m.lower()
            and m.lower().endswith(".pdf")
        ]

        expected_table_names = {
            "table_1_validation_model_comparison.csv",
            "table_2_holdout_matched_support.csv",
            "table_3_holdout_full_available_support.csv",
            "table_4_day_block_bootstrap.csv",
            "table_5_performance_by_aqi_band.csv",
            "table_6_abrupt_transition_detection.csv",
            "table_7_feature_attribution.csv",
            "table_8_station_performance.csv",
            "table_9_season_performance.csv",
        }
        table_members = [
            m for m in members
            if PurePosixPath(m).name in expected_table_names
        ]

        raw_names = {
            "station_hour.csv",
            "stations.csv",
            "station_day.csv",
            "city_hour.csv",
            "city_day.csv",
        }
        raw_members = [
            m for m in members
            if PurePosixPath(m).name.lower() in raw_names
        ]

        return {
            "path": str(path),
            "sha256": sha256_file(path),
            "entries": len(members),
            "docx_members": docx_members,
            "pdf_members": pdf_members,
            "figure_png_count": len(figure_pngs),
            "figure_pdf_count": len(figure_pdfs),
            "main_table_count": len(set(table_members)),
            "main_table_members": table_members,
            "raw_source_members": raw_members,
            "members": members,
        }

print("=" * 78)
print("PHASE 17.10H-R2 — FINAL PHASE CLOSEOUT")
print("=" * 78)

# ------------------------------------------------------------------
# 1. Verify known export and QA artifacts.
# ------------------------------------------------------------------
for required_path in [DOCX, PDF, CHECKLIST]:
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Required Phase 17.10 artifact missing: {required_path}"
        )

if not H_R1_JSON.is_file():
    print("Warning: Phase 17.10H-R1 JSON was not found at the expected path.")
    h_r1_report = {}
else:
    h_r1_report = load_json(H_R1_JSON)

docx_size = DOCX.stat().st_size
pdf_size = PDF.stat().st_size
docx_sha256 = sha256_file(DOCX)
pdf_sha256 = sha256_file(PDF)

# Verify DOCX ZIP container, embedded images, tables, and paragraphs.
with ZipFile(DOCX, "r") as docx_zip:
    docx_bad_member = docx_zip.testzip()
    if docx_bad_member:
        raise RuntimeError(
            f"DOCX container integrity failed at {docx_bad_member}"
        )

    docx_names = docx_zip.namelist()
    media_members = [
        n for n in docx_names
        if n.startswith("word/media/") and not n.endswith("/")
    ]

    document_xml = ET.fromstring(docx_zip.read("word/document.xml"))
    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}

    docx_table_count = len(document_xml.findall(".//w:tbl", ns))
    docx_paragraph_count = len(document_xml.findall(".//w:p", ns))

# PDF structural and page-count verification.
pdf_page_count = None
pdf_text_chars = None
pdf_readable = False
pdf_engine = None

try:
    import pymupdf
    with pymupdf.open(PDF) as pdf_doc:
        pdf_page_count = len(pdf_doc)
        pdf_text_chars = sum(len(page.get_text()) for page in pdf_doc)
        pdf_readable = pdf_page_count > 0
        pdf_engine = "pymupdf"
except Exception:
    try:
        import fitz
        with fitz.open(PDF) as pdf_doc:
            pdf_page_count = len(pdf_doc)
            pdf_text_chars = sum(len(page.get_text()) for page in pdf_doc)
            pdf_readable = pdf_page_count > 0
            pdf_engine = "fitz"
    except Exception as exc:
        print("Warning: PDF text parser could not open PDF:", repr(exc))

# Count generated page renders and contact sheets.
render_dir = EXPORT_DIR / "rendered_pages"
render_pngs = sorted(render_dir.glob("*.png")) if render_dir.is_dir() else []

contact_sheets = []
for possible_dir in [QA_DIR, EXPORT_DIR / "contact_sheets", EXPORT_DIR]:
    if possible_dir.is_dir():
        contact_sheets.extend(possible_dir.glob("contact-sheet-*.png"))

# Deduplicate contact-sheet paths.
contact_sheets = sorted(
    {p.resolve() for p in contact_sheets if p.is_file()}
)

# ------------------------------------------------------------------
# 2. Discover and verify the review ZIP.
# ------------------------------------------------------------------
h_r1_zip_paths = []
for key, value in iter_strings(h_r1_report):
    if isinstance(value, str) and value.lower().endswith(".zip"):
        candidate = Path(value)
        if candidate.is_file():
            h_r1_zip_paths.append(candidate)

# Search the project for a package with DOCX, PDF, seven figure pairs,
# nine main CSV tables, and no obvious raw source CSVs.
qualified_archives = []
archive_scan_errors = []

for zip_path in RESEARCH.rglob("*.zip"):
    try:
        info = inspect_zip(zip_path)
        has_docs = bool(info["docx_members"]) and bool(info["pdf_members"])
        has_assets = (
            info["figure_png_count"] >= 7
            and info["figure_pdf_count"] >= 7
            and info["main_table_count"] >= 9
        )
        if has_docs and has_assets:
            info["qualified_final_review_package"] = True
            qualified_archives.append(info)
    except Exception as exc:
        archive_scan_errors.append({
            "path": str(zip_path),
            "error": f"{type(exc).__name__}: {exc}",
        })

# Prefer an archive explicitly referenced by the prior verification report.
selected_archive_info = None
for referenced in h_r1_zip_paths:
    for info in qualified_archives:
        if Path(info["path"]).resolve() == referenced.resolve():
            selected_archive_info = info
            break
    if selected_archive_info:
        break

# Otherwise use the most recently modified fully qualified package.
if selected_archive_info is None and qualified_archives:
    qualified_archives.sort(
        key=lambda x: Path(x["path"]).stat().st_mtime,
        reverse=True
    )
    selected_archive_info = qualified_archives[0]

if selected_archive_info is None:
    print(
        "Warning: no ZIP found that simultaneously contains DOCX, PDF, "
        "seven figure pairs, and all nine main tables."
    )

# Phase 17.10C original ZIP identity is also checked, if present.
phase_10c_zip = (
    RESEARCH
    / "phase17_10C_final_review_package_20261025T184853"
    / "Delhi_AQI_Integrated_Research_Review_Package.zip"
)
phase_10c_verified = False
phase_10c_info = None

if phase_10c_zip.is_file():
    phase_10c_info = inspect_zip(phase_10c_zip)
    phase_10c_verified = (
        phase_10c_info["sha256"] == EXPECTED_PHASE_10C_SHA256
    )

# ------------------------------------------------------------------
# 3. Determine whether manual page review was actually completed.
#    This code never writes "PASS" into the user's checklist.
# ------------------------------------------------------------------
checklist_df = pd.read_csv(CHECKLIST)
required_manual_columns = [
    "manual_review_status",
    "no_clipping_or_overlap",
    "figures_readable",
    "tables_readable",
    "headings_and_page_breaks_ok",
    "captions_and_references_ok",
]

missing_manual_columns = [
    c for c in required_manual_columns if c not in checklist_df.columns
]

def affirmative(value):
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {
        "yes", "true", "pass", "passed", "complete",
        "completed", "reviewed", "ok", "1"
    }

if missing_manual_columns:
    human_visual_qa_complete = False
    manual_qa_detail = (
        "Checklist is missing expected fields: "
        + ", ".join(missing_manual_columns)
    )
else:
    expected_pages_reviewed = (
        len(checklist_df) == 19
        and checklist_df["page"].nunique() == 19
    )

    manual_status_complete = checklist_df["manual_review_status"].apply(
        affirmative
    ).all()

    visual_fields_passed = checklist_df[
        [
            "no_clipping_or_overlap",
            "figures_readable",
            "tables_readable",
            "headings_and_page_breaks_ok",
            "captions_and_references_ok",
        ]
    ].apply(lambda col: col.apply(affirmative)).all().all()

    human_visual_qa_complete = (
        expected_pages_reviewed
        and manual_status_complete
        and visual_fields_passed
    )

    manual_qa_detail = (
        "All 19 pages have recorded affirmative manual reviews."
        if human_visual_qa_complete
        else
        "Manual QA is incomplete or at least one page/criterion is still "
        "pending, blank, or not affirmative."
    )

# ------------------------------------------------------------------
# 4. Check whether licence clarification has been recorded.
# ------------------------------------------------------------------
licence_review_complete = False
licence_detail = "No documented author clearance found; review remains open."

if LICENSE_TRACKER.is_file():
    tracker_df = pd.read_csv(LICENSE_TRACKER)
    tracker_columns = {c.lower(): c for c in tracker_df.columns}

    submitted_col = tracker_columns.get("request_submitted")
    response_date_col = tracker_columns.get("response_date")
    decision_col = tracker_columns.get("reviewer_decision")
    signoff_col = tracker_columns.get("reviewer_sign_off_date")

    submitted_count = 0
    response_count = 0
    signed_off_count = 0

    if submitted_col:
        submitted_count = tracker_df[submitted_col].apply(
            lambda v: affirmative(v)
            or (isinstance(v, str) and v.strip().lower() in {
                "sent", "submitted", "yes - submitted", "submitted - yes"
            })
        ).sum()

    if response_date_col:
        response_count = tracker_df[response_date_col].apply(
            lambda v: pd.notna(v) and bool(str(v).strip())
        ).sum()

    if signoff_col:
        signed_off_count = tracker_df[signoff_col].apply(
            lambda v: pd.notna(v) and bool(str(v).strip())
        ).sum()

    if decision_col:
        approved_decisions = tracker_df[decision_col].apply(
            lambda v: isinstance(v, str)
            and v.strip().lower() in {
                "cleared", "cleared for stated use",
                "approved", "reuse cleared"
            }
        ).sum()
        licence_review_complete = (
            response_count > 0
            and signed_off_count > 0
            and approved_decisions > 0
        )
        if licence_review_complete:
            licence_detail = (
                "Tracker contains an affirmative review decision and sign-off; "
                "author should ensure supporting evidence is retained."
            )
        else:
            licence_detail = (
                f"Tracker records submitted requests: {submitted_count}; "
                f"responses: {response_count}; sign-offs: {signed_off_count}. "
                "Licence gate remains open."
            )

# ------------------------------------------------------------------
# 5. Produce the final phase closeout folder and manifest.
# ------------------------------------------------------------------
stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT = (
    RESEARCH / f"phase17_10H_R2_final_closeout_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

artifacts = []

def add_artifact(role, path):
    path = Path(path)
    record = {
        "role": role,
        "path": str(path),
        "exists": path.is_file(),
        "bytes": path.stat().st_size if path.is_file() else None,
        "sha256": sha256_file(path) if path.is_file() else None,
    }
    artifacts.append(record)

add_artifact("Integrated DOCX export", DOCX)
add_artifact("Integrated PDF export", PDF)
add_artifact("Human visual QA checklist", CHECKLIST)
add_artifact("Phase 17.10H-R1 verification JSON", H_R1_JSON)
add_artifact("Phase 17.10H-R1 verification report", H_R1_MD)
add_artifact("Licence response tracker", LICENSE_TRACKER)

for page in render_pngs:
    add_artifact("Rendered PDF page", page)

for sheet in contact_sheets:
    add_artifact("Visual QA contact sheet", sheet)

if selected_archive_info:
    add_artifact(
        "Verified final review package ZIP",
        selected_archive_info["path"]
    )
if phase_10c_zip.is_file():
    add_artifact("Original Phase 17.10C review package ZIP", phase_10c_zip)

manifest_path = OUT / "phase17_10H_R2_artifact_manifest.csv"
pd.DataFrame(artifacts).to_csv(manifest_path, index=False)

technical_checks = {
    "docx_exists": DOCX.is_file(),
    "docx_zip_container_integrity": docx_bad_member is None,
    "docx_size_bytes": docx_size,
    "docx_embedded_images": len(media_members),
    "docx_table_count": docx_table_count,
    "docx_paragraph_count": docx_paragraph_count,
    "pdf_exists": PDF.is_file(),
    "pdf_readable": pdf_readable,
    "pdf_engine": pdf_engine,
    "pdf_page_count": pdf_page_count,
    "pdf_text_characters": pdf_text_chars,
    "rendered_page_count": len(render_pngs),
    "expected_19_page_render_match": (
        len(render_pngs) == 19
        and (pdf_page_count is None or pdf_page_count == len(render_pngs))
    ),
    "contact_sheet_count": len(contact_sheets),
    "phase_10C_sha256_verified": phase_10c_verified,
    "selected_review_zip": (
        selected_archive_info["path"] if selected_archive_info else None
    ),
    "selected_review_zip_sha256": (
        selected_archive_info["sha256"] if selected_archive_info else None
    ),
    "selected_review_zip_entries": (
        selected_archive_info["entries"] if selected_archive_info else None
    ),
    "selected_review_zip_integrity": (
        "PASS" if selected_archive_info else "NOT FOUND / NOT VERIFIED"
    ),
    "selected_review_zip_figures_png": (
        selected_archive_info["figure_png_count"]
        if selected_archive_info else None
    ),
    "selected_review_zip_figures_pdf": (
        selected_archive_info["figure_pdf_count"]
        if selected_archive_info else None
    ),
    "selected_review_zip_main_tables": (
        selected_archive_info["main_table_count"]
        if selected_archive_info else None
    ),
    "selected_review_zip_raw_source_files_absent": (
        not selected_archive_info["raw_source_members"]
        if selected_archive_info else None
    ),
    "page_checklist_rows": len(checklist_df),
    "human_visual_qa_completed": human_visual_qa_complete,
    "human_visual_qa_detail": manual_qa_detail,
    "author_licence_review_completed": licence_review_complete,
    "author_licence_review_detail": licence_detail,
}

technical_integrity_pass = (
    DOCX.is_file()
    and PDF.is_file()
    and docx_bad_member is None
    and docx_size > 0
    and pdf_size > 0
    and len(media_members) >= 7
    and pdf_readable
    and pdf_page_count == 19
    and len(render_pngs) == 19
    and selected_archive_info is not None
    and selected_archive_info["figure_png_count"] >= 7
    and selected_archive_info["figure_pdf_count"] >= 7
    and selected_archive_info["main_table_count"] >= 9
    and not selected_archive_info["raw_source_members"]
)

submission_ready = (
    technical_integrity_pass
    and human_visual_qa_complete
    and licence_review_complete
)

status = {
    "phase": "17.10H-R2",
    "created_at": datetime.now().astimezone().isoformat(),
    "phase_closeout": (
        "TECHNICAL EXPORT AND PACKAGE VERIFIED"
        if technical_integrity_pass
        else "TECHNICAL CHECKS INCOMPLETE"
    ),
    "technical_integrity_pass": technical_integrity_pass,
    "human_visual_qa_completed": human_visual_qa_complete,
    "author_licence_review_completed": licence_review_complete,
    "submission_ready": submission_ready,
    "production_enabled": False,
    "source_data_modified": False,
    "frozen_models_retrained": False,
    "previous_bundles_modified": False,
    "technical_checks": technical_checks,
    "manifest_path": str(manifest_path),
    "open_release_gates": [
        gate for gate, passed in [
            ("Human visual QA", human_visual_qa_complete),
            ("Author licence review", licence_review_complete),
        ] if not passed
    ],
}

status_json = OUT / "phase17_10H_R2_final_status.json"
status_json.write_text(
    json.dumps(status, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

report = f"""# Phase 17.10H-R2 — Final Phase Closeout

Generated: {status["created_at"]}

## Technical artifact status

- DOCX exists: {DOCX.is_file()}
- DOCX container integrity: {"PASS" if docx_bad_member is None else "FAIL"}
- DOCX embedded images: {len(media_members)}
- DOCX tables: {docx_table_count}
- PDF exists/readable: {PDF.is_file() and pdf_readable}
- PDF page count: {pdf_page_count}
- Rendered PNG pages: {len(render_pngs)}
- Contact sheets: {len(contact_sheets)}
- Selected review ZIP found and verified: {selected_archive_info is not None}
- Phase 17.10C original ZIP hash verified: {phase_10c_verified}

**Technical package checks: {"PASS" if technical_integrity_pass else "INCOMPLETE"}**

## Human visual QA

Status: {"COMPLETE" if human_visual_qa_complete else "PENDING"}

{manual_qa_detail}

The checker does not mark manual review complete merely because pages rendered
successfully or passed automated blank-page heuristics.

## Author licence review

Status: {"CLEARED IN TRACKER" if licence_review_complete else "OPEN"}

{licence_detail}

## Final release gate

- Submission ready: {"YES" if submission_ready else "NO"}
- Production enabled: NO
- Source data modified: NO
- Frozen models retrained: NO
- Previous review packages overwritten: NO

## Outputs

- Artifact manifest: `phase17_10H_R2_artifact_manifest.csv`
- Machine-readable status: `phase17_10H_R2_final_status.json`

## Required interpretation

Technical package verification does not replace human visual inspection or
licence clearance. If either release gate is open, keep the paper as a review
draft and do not label it submission-ready.
"""

report_path = OUT / "PHASE_17_10H_R2_FINAL_CLOSEOUT.md"
report_path.write_text(report, encoding="utf-8")

print("\n[1] EXPORT VERIFICATION")
print("DOCX:", DOCX.is_file(), "| bytes:", docx_size)
print("DOCX embedded images:", len(media_members))
print("DOCX tables:", docx_table_count)
print("PDF readable:", pdf_readable, "| pages:", pdf_page_count)
print("Rendered pages:", len(render_pngs))
print("Contact sheets:", len(contact_sheets))

print("\n[2] REVIEW PACKAGE")
if selected_archive_info:
    print("ZIP:", selected_archive_info["path"])
    print("Entries:", selected_archive_info["entries"])
    print("SHA-256:", selected_archive_info["sha256"])
    print("Figure PNG/PDF:",
          selected_archive_info["figure_png_count"], "/",
          selected_archive_info["figure_pdf_count"])
    print("Main tables:", selected_archive_info["main_table_count"])
    print("Raw source CSVs absent:",
          not selected_archive_info["raw_source_members"])
else:
    print("No fully qualified export-containing review ZIP was discovered.")
    print("See manifest and H-R1 verification report.")

print("\n[3] FINAL STATUS")
print("Technical integrity:", "PASS" if technical_integrity_pass else "INCOMPLETE")
print("Human visual QA:", "COMPLETE" if human_visual_qa_complete else "PENDING")
print("Author licence review:", "CLEARED IN TRACKER" if licence_review_complete else "OPEN")
print("Submission ready:", submission_ready)
print("Production enabled: False")
print("Source data modified: False")
print("Models retrained: False")
print("Previous bundles modified: False")

print("\n[4] CLOSEOUT FILES")
print("Output folder:", OUT)
print("Report:", report_path)
print("JSON:", status_json)
print("Manifest:", manifest_path)
print("=" * 78)

In [ ]:
# PHASE 17.10H-R2 — FINAL STATUS JSON SERIALIZATION RECOVERY
# Repairs the failed final JSON write; does not rerun analysis or models.

from pathlib import Path
from datetime import datetime, date
import json
import math
import os
import tempfile

import numpy as np

print("=" * 78)
print("PHASE 17.10H-R2 — STATUS JSON RECOVERY")
print("=" * 78)

# Reuse the status object and path created by the interrupted Phase 17.10H-R2 cell.
status_obj = globals().get("status")
status_path_obj = globals().get("status_json")

if status_obj is None:
    raise RuntimeError(
        "The in-memory 'status' object was not found. "
        "The Colab runtime may have restarted. Rerun Phase 17.10H-R2 "
        "after replacing its final json.dumps(status, ...) call with "
        "json.dumps(to_json_safe(status), ...), using the helper below."
    )

if status_path_obj is None:
    out_obj = globals().get("OUT")
    if out_obj is None:
        raise RuntimeError(
            "Neither 'status_json' nor 'OUT' was found in memory. "
            "Restore the Phase 17.10H-R2 variables before continuing."
        )
    status_path_obj = Path(out_obj) / "phase17_10H_R2_final_status.json"

status_path = Path(status_path_obj)
status_path.parent.mkdir(parents=True, exist_ok=True)


def to_json_safe(value):
    """Recursively convert common Python, NumPy, pandas-like and path values."""

    # Convert NumPy scalars before handling Python primitive types.
    if type(value).__module__.startswith("numpy"):
        if isinstance(value, np.ndarray):
            return [to_json_safe(x) for x in value.tolist()]
        if hasattr(value, "item"):
            return to_json_safe(value.item())

    # Built-in JSON primitives.
    if value is None or type(value) in (str, int, bool):
        return value

    if type(value) is float:
        return value if math.isfinite(value) else None

    # Nested containers.
    if isinstance(value, dict):
        return {
            str(to_json_safe(k)): to_json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [to_json_safe(v) for v in value]

    if isinstance(value, set):
        return [to_json_safe(v) for v in sorted(value, key=str)]

    # Handle non-standard Boolean objects whose type name is "bool".
    if type(value).__name__ == "bool":
        return bool(value)

    # Common date/time objects.
    if isinstance(value, (datetime, date)):
        return value.isoformat()

    # pathlib.Path, NumPy-compatible path-like objects.
    if isinstance(value, os.PathLike):
        return os.fspath(value)

    # Support pandas NA/NaT and scalar objects that expose .item().
    if type(value).__name__ in {"NAType", "NaTType"}:
        return None

    if hasattr(value, "item") and callable(value.item):
        try:
            return to_json_safe(value.item())
        except (ValueError, TypeError):
            pass

    if hasattr(value, "isoformat") and callable(value.isoformat):
        try:
            return value.isoformat()
        except (ValueError, TypeError):
            pass

    # Last resort for uncommon metadata-only objects.
    return str(value)


# Sanitize nested values, then serialize with strict JSON compliance.
safe_status = to_json_safe(status_obj)
serialized = json.dumps(
    safe_status,
    indent=2,
    ensure_ascii=False,
    allow_nan=False
)

# Write atomically so a failed write does not leave a partial JSON file.
fd, temporary_name = tempfile.mkstemp(
    prefix=status_path.stem + "_",
    suffix=".tmp",
    dir=str(status_path.parent)
)

try:
    with os.fdopen(fd, "w", encoding="utf-8") as f:
        f.write(serialized)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())

    # Confirm the serialized JSON can be read back before replacing target.
    with open(temporary_name, "r", encoding="utf-8") as f:
        verified_status = json.load(f)

    os.replace(temporary_name, status_path)

except Exception:
    if os.path.exists(temporary_name):
        os.remove(temporary_name)
    raise

# Final validation.
with open(status_path, "r", encoding="utf-8") as f:
    reloaded_status = json.load(f)

if not isinstance(reloaded_status, dict):
    raise RuntimeError("Status JSON was written but is not a JSON object.")

print("\n[1] SERIALIZATION")
print("Status object converted: True")
print("JSON round-trip validation: PASS")

print("\n[2] FINAL STATUS")
print("Path:", status_path)
print("Size (bytes):", status_path.stat().st_size)
print("Top-level fields:", len(reloaded_status))

for key in (
    "human_visual_qa_completed",
    "author_licence_review",
    "submission_ready",
    "production_enabled",
):
    if key in reloaded_status:
        print(f"{key}: {reloaded_status[key]}")

print("\n[3] SAFEGUARDS")
print("Source data modified: False")
print("Frozen models retrained: False")
print("Production enabled by this recovery: False")
print("=" * 78)

In [ ]:
# PHASE 17.10F — SAFE CHECKPOINT BEFORE STOPPING
# New checkpoint folder only. No Drive remount, source overwrite, or model retraining.

from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import textwrap

import pandas as pd

ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

if not ROOT.is_dir():
    raise FileNotFoundError(
        f"Project root not found: {ROOT}\n"
        "Confirm that Google Drive is already mounted; this cell will not remount it."
    )

RESEARCH = ROOT / "outputs" / "research_paper"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def latest_folder(patterns):
    found = []
    for pattern in patterns:
        found.extend(p for p in RESEARCH.glob(pattern) if p.is_dir())
    found = list({p.resolve(): p for p in found}.values())
    return max(found, key=lambda p: p.stat().st_mtime) if found else None

def file_record(label, path, hash_limit_mb=25):
    path = Path(path)
    rec = {
        "label": label,
        "path": str(path),
        "exists": path.is_file(),
    }
    if path.is_file():
        stat = path.stat()
        rec["size_bytes"] = stat.st_size
        rec["modified_at"] = datetime.fromtimestamp(
            stat.st_mtime
        ).astimezone().isoformat()
        if stat.st_size <= hash_limit_mb * 1024 * 1024:
            rec["sha256"] = sha256_file(path)
    return rec

# ------------------------------------------------------------------
# 1. Locate the latest visual QA and render workspaces.
# ------------------------------------------------------------------
qa_dir = latest_folder([
    "phase17_10F_visual_review_workspace_*",
    "phase17_10F_visual_review_*",
    "phase17_10E_human_visual_qa_*",
])

render_dir = latest_folder([
    "phase17_10D_rendered_review_export_*",
])

# Known outputs from completed phases.
known_files = [
    (
        "Phase 17.10C review ZIP",
        RESEARCH / "phase17_10C_final_review_package_20261025T184853"
        / "Delhi_AQI_Integrated_Research_Review_Package.zip",
    ),
    (
        "Integrated Markdown manuscript",
        RESEARCH / "phase17_10B_R1_integrated_manuscript_20261025T184741"
        / "Delhi_AQI_Integrated_Manuscript.md",
    ),
    (
        "Phase 17.10A figure register",
        RESEARCH / "phase17_10A_publication_figures_20261025T184251"
        / "figure_register.csv",
    ),
    (
        "Phase 17.10C package verification",
        RESEARCH / "phase17_10C_final_review_package_20261025T184853"
        / "phase17_10C_package_verification.json",
    ),
    (
        "Phase 17.07U author licence checklist",
        RESEARCH / "phase17_07U_author_release_gate_20261025T181929"
        / "AUTHOR_RELEASE_CHECKLIST.md",
    ),
    (
        "Phase 17.08 response tracker",
        RESEARCH / "phase17_08_licence_clarification_20261025T183001"
        / "PHASE_17_08_RESPONSE_TRACKER.csv",
    ),
]

if render_dir:
    known_files.extend([
        (
            "Rendered DOCX review draft",
            render_dir / "Delhi_AQI_Integrated_Review_Draft.docx",
        ),
        (
            "Rendered PDF review draft",
            render_dir / "Delhi_AQI_Integrated_Review_Draft.pdf",
        ),
        (
            "Phase 17.10D render QA report",
            render_dir / "phase17_10D_render_qa.json",
        ),
        (
            "Phase 17.10D page inventory",
            render_dir / "rendered_page_inventory.csv",
        ),
    ])

if qa_dir:
    known_files.extend([
        (
            "Visual QA page checklist",
            qa_dir / "phase17_10E_page_review_checklist.csv",
        ),
        (
            "Visual QA status JSON",
            qa_dir / "phase17_10E_visual_qa_status.json",
        ),
    ])

# Also record the current frozen model/data references.
reference_files = [
    (
        "Frozen Extra Trees model",
        ROOT / "deployment/phase16_33_frozen_model_package/models"
        / "phase15_10_final_tuned_extra_trees_trainval.joblib",
    ),
    (
        "Frozen Random Forest model",
        ROOT / "deployment/phase16_33_frozen_model_package/models"
        / "phase15_10_final_tuned_random_forest_trainval.joblib",
    ),
    (
        "Frozen feature contract",
        ROOT / "deployment/phase16_33_frozen_model_package/config"
        / "feature_contract.json",
    ),
    (
        "Final holdout predictions",
        ROOT / "outputs/reports/phase15_12_final_holdout"
        / "phase15_12_final_holdout_predictions.parquet",
    ),
    (
        "Frozen engineered feature matrix",
        ROOT / "data/processed/delhi_10_station_feature_engineered.parquet",
    ),
    (
        "Chronological split assignments",
        ROOT / "outputs/reports/phase15_1_time_split_assignments.parquet",
    ),
]

# ------------------------------------------------------------------
# 2. Create a timestamped checkpoint folder without overwriting anything.
# ------------------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%dT%H%M%S_%f")
CHECKPOINT = (
    ROOT / "outputs" / "checkpoints"
    / f"Delhi_AQI_Phase_17_10F_checkpoint_{timestamp}"
)
CHECKPOINT.mkdir(parents=True, exist_ok=False)

dataframe_dir = CHECKPOINT / "qa_dataframes"
dataframe_dir.mkdir()

# Save selected QA/manuscript-related DataFrames if they're in notebook memory.
# This is optional; existing on-Drive CSV artifacts are also recorded below.
candidate_dataframe_names = [
    "checklist",
    "page_inventory",
    "rendered_page_inventory",
    "review_checklist",
    "visual_qa_checklist",
    "figure_register",
    "figure_manifest",
    "table_manifest",
]

saved_dataframes = []
for var_name in candidate_dataframe_names:
    obj = globals().get(var_name)
    if isinstance(obj, pd.DataFrame):
        safe_name = re.sub(r"[^A-Za-z0-9_.-]", "_", var_name)
        try:
            destination = dataframe_dir / f"{safe_name}.parquet"
            obj.to_parquet(destination, index=False)
            fmt = "parquet"
        except Exception:
            destination = dataframe_dir / f"{safe_name}.csv"
            obj.to_csv(destination, index=False)
            fmt = "csv"

        saved_dataframes.append({
            "variable_name": var_name,
            "path": str(destination),
            "format": fmt,
            "rows": len(obj),
            "columns": list(map(str, obj.columns)),
        })

# ------------------------------------------------------------------
# 3. Record all important state and artifact references.
# ------------------------------------------------------------------
artifacts = []
for label, path in known_files + reference_files:
    artifacts.append(file_record(label, path))

for optional_dir_label, optional_dir in [
    ("Latest Phase 17.10D render directory", render_dir),
    ("Latest Phase 17.10E/F visual QA directory", qa_dir),
]:
    artifacts.append({
        "label": optional_dir_label,
        "path": str(optional_dir) if optional_dir else None,
        "exists": bool(optional_dir and optional_dir.is_dir()),
    })

# Inventories for the page renders and contact sheets, if present.
visual_files = []
search_roots = [p for p in [render_dir, qa_dir] if p and p.exists()]
for base in search_roots:
    for path in base.rglob("*"):
        if not path.is_file():
            continue
        lower = path.name.lower()
        if (
            lower.endswith(".png")
            and (
                "contact-sheet" in lower
                or "page_" in lower
                or "page-" in lower
            )
        ):
            visual_files.append(file_record("Visual QA image", path))

# Known model hashes recorded previously in the frozen-model evidence.
frozen_model_hashes = {
    "phase15_10_final_tuned_extra_trees_trainval.joblib":
        "f0b3d43a56a7d5e4f08554f37ff234e1b5e012b5d5d69f18c750844b36cbe72d",
    "phase15_10_final_tuned_random_forest_trainval.joblib":
        "538b5ff34152d19221518e831528093b7bcc57e2b05f82669718f540b514abd8",
}

resume_state = {
    "checkpoint_phase": "17.10F",
    "created_at": datetime.now().astimezone().isoformat(),
    "project_root": str(ROOT),
    "checkpoint_folder": str(CHECKPOINT),
    "last_completed_phase": "17.10E — page renders/contact sheets created",
    "current_phase": "17.10F — visual review workspace prepared",
    "resume_at": (
        "Manually inspect the two contact sheets and all 19 full-resolution "
        "PDF page renders; complete the page review checklist."
    ),
    "visual_qa_status": "PENDING — not yet marked as human-reviewed",
    "docx_export": str(render_dir / "Delhi_AQI_Integrated_Review_Draft.docx")
        if render_dir else None,
    "pdf_export": str(render_dir / "Delhi_AQI_Integrated_Review_Draft.pdf")
        if render_dir else None,
    "render_directory": str(render_dir) if render_dir else None,
    "visual_qa_directory": str(qa_dir) if qa_dir else None,
    "artifact_inventory": artifacts,
    "visual_review_images": visual_files,
    "saved_in_memory_dataframes": saved_dataframes,
    "frozen_model_sha256_reference": frozen_model_hashes,
    "ensemble_weights": {
        "Tuned_Extra_Trees": 0.594466,
        "Tuned_Random_Forest": 0.405534,
    },
    "frozen_target": "AQI_t_plus_1",
    "feature_count": 73,
    "stations": [
        "DL031", "DL002", "DL021", "DL028", "DL019",
        "DL010", "DL034", "DL007", "DL013", "DL004",
    ],
    "final_holdout": {
        "rows": 68352,
        "mae": 3.216121,
        "rmse": 7.658390,
        "r2": 0.996634,
    },
    "project_safeguards": {
        "source_data_modified": False,
        "models_retrained": False,
        "production_enabled": False,
        "original_review_zip_modified": False,
    },
    "open_gates": [
        "Human visual QA: inspect pages and complete checklist",
        "Author licence review: responses not received; gate remains open",
        "Final journal formatting and submission readiness not yet confirmed",
    ],
}

manifest_path = CHECKPOINT / "checkpoint_manifest.json"
manifest_path.write_text(
    json.dumps(resume_state, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

resume_notes = f"""# Delhi AQI Project — Resume Checkpoint

Checkpoint created: {resume_state["created_at"]}

## Where to resume
Continue at **Phase 17.10F: visual review**. Do not rerun the data pipeline,
model training, hyperparameter tuning, or figure-generation phases.

## Current task
The DOCX and PDF review draft were exported. The PDF has 19 pages, and 19 page
renders plus two contact sheets were created. The automated checks did not flag
obvious blank or low-variation pages, but human visual QA is still pending.

Open the contact sheets in:
`{render_dir / "contact_sheets" if render_dir else "No render directory found"}`

Full page renders:
`{render_dir / "rendered_pages" if render_dir else "No render directory found"}`

Checklist:
`{qa_dir / "phase17_10E_page_review_checklist.csv" if qa_dir else "No matching QA folder found"}`

## Key manuscript outputs
- Integrated Markdown manuscript:
  `{RESEARCH / "phase17_10B_R1_integrated_manuscript_20261025T184741" / "Delhi_AQI_Integrated_Manuscript.md"}`
- Review ZIP:
  `{RESEARCH / "phase17_10C_final_review_package_20261025T184853" / "Delhi_AQI_Integrated_Research_Review_Package.zip"}`
- DOCX/PDF renders are recorded in `checkpoint_manifest.json`.

## Frozen model state
- Target: `AQI_t_plus_1`
- Feature contract: 73 features
- Ensemble weights: Extra Trees 0.594466; Random Forest 0.405534
- Do not retrain or retune the frozen models.

## Open release gates
1. Complete manual visual QA and record any clipping, unreadable figures/tables,
   heading/page-break problems, or caption/reference problems.
2. Obtain and record author licence clarification. No requests have been sent
   and no responses have been received according to the last saved status.
3. Reassess submission readiness only after the open checks are resolved.

## Restore helper
Use the companion file `restore_checkpoint.py` in this folder.
"""

(CHECKPOINT / "RESUME_NOTES.md").write_text(
    resume_notes,
    encoding="utf-8",
)

restore_script = r'''
from pathlib import Path
import hashlib
import json
import pandas as pd

CHECKPOINT_DIR = Path(__file__).resolve().parent
MANIFEST_PATH = CHECKPOINT_DIR / "checkpoint_manifest.json"

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    checkpoint_manifest = json.load(f)

PROJECT_ROOT = Path(checkpoint_manifest["project_root"])

def verify_checkpoint_artifacts():
    """Return existence/size information without modifying artifacts."""
    results = []
    for item in checkpoint_manifest.get("artifact_inventory", []):
        path_text = item.get("path")
        path = Path(path_text) if path_text else None
        results.append({
            "label": item.get("label"),
            "path": path_text,
            "exists": bool(path and path.exists()),
            "is_file": bool(path and path.is_file()),
            "size_bytes": path.stat().st_size
                if path and path.is_file() else None,
        })
    return results

def load_saved_dataframe(variable_name):
    """Load a QA DataFrame saved in this checkpoint, if present."""
    match = next(
        (
            item for item in checkpoint_manifest.get(
                "saved_in_memory_dataframes", []
            )
            if item["variable_name"] == variable_name
        ),
        None,
    )
    if match is None:
        raise KeyError(
            f"No DataFrame named {variable_name!r} was saved in this checkpoint."
        )
    path = Path(match["path"])
    if match["format"] == "parquet":
        return pd.read_parquet(path)
    if match["format"] == "csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported saved format: {match['format']}")

artifact_status = verify_checkpoint_artifacts()

print("Checkpoint loaded:", MANIFEST_PATH)
print("Last completed phase:", checkpoint_manifest["last_completed_phase"])
print("Resume at:", checkpoint_manifest["resume_at"])
print("Visual QA status:", checkpoint_manifest["visual_qa_status"])
print("Open gates:", checkpoint_manifest["open_gates"])
print("Artifact references:", len(artifact_status))
print("Saved in-memory DataFrames:",
      len(checkpoint_manifest.get("saved_in_memory_dataframes", [])))
'''

(CHECKPOINT / "restore_checkpoint.py").write_text(
    textwrap.dedent(restore_script).lstrip(),
    encoding="utf-8",
)

# Validate the state file can be read back and report a compact summary.
with open(manifest_path, "r", encoding="utf-8") as f:
    checked_state = json.load(f)

print("=" * 78)
print("PHASE 17.10F — CHECKPOINT SAVED")
print("=" * 78)
print("Checkpoint folder:", CHECKPOINT)
print("Manifest:", manifest_path)
print("Resume notes:", CHECKPOINT / "RESUME_NOTES.md")
print("Restore helper:", CHECKPOINT / "restore_checkpoint.py")
print("Artifact references recorded:", len(artifacts))
print("Visual QA images inventoried:", len(visual_files))
print("In-memory QA DataFrames saved:", len(saved_dataframes))
print("Visual QA status: PENDING — manual inspection still required")
print("Licence review: OPEN")
print("Models retrained: False")
print("Production enabled: False")
print("Checkpoint manifest reload validation: PASS")
print("=" * 78)